In [ ]:
# Copyright (c) 2026 Daniyar Kuzekov, Li Yang, Ercan Engin Kuruoğlu, Wai Kin (Victor) Chan
# Licensed under the GNU Affero General Public License v3.0 (AGPL‑3.0)

In [1]:
# ============================================================
# OFFLINE DeepSeek-MoE Ws Builder (no HF access required)
# - Discovers MoE experts from model.safetensors.index.json
# - Loads expert weights from local shards with progress bar
# - Builds effective matrices W_eff = down @ up (or down @ gate)
# - Optional int8 quant error sanity check
# - Saves outputs to OUTPUT_DIR
# ============================================================

import os, re, json, math, gc, sys
from pathlib import Path
from typing import Dict, Tuple, List, Optional
import numpy as np

# ---------- USER CONFIG ----------
MODEL_DIR   = os.path.expanduser("~/deepseek-model")     # <-- set to your server path
OUTPUT_DIR  = os.path.expanduser("~/moe_ws_outputs")     # where to write Ws outputs
LAYER       = None  # e.g. 1, or None = auto-pick first layer with MoE experts
MAX_EXPERTS = 8     # keep modest for quick tests; raise if you want more
PREFER_UP   = "up_proj"   # "up_proj" or "gate_proj"
DTYPE_OUT   = "float16"   # "float16" or "float32"
USE_TORCH_MATMUL = True   # torch matmul can be faster; falls back to numpy
RUN_INT8_SANITY = True    # quick symmetric int8 quantization error stats
# --------------------------------

# ---------- lightweight deps ----------
def _pip_install(pkgs: List[str]) -> None:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU"] + pkgs)

try:
    from tqdm import tqdm
except Exception:
    _pip_install(["tqdm>=4.66"])
    from tqdm import tqdm

try:
    from safetensors import safe_open as st_safe_open
except Exception:
    _pip_install(["safetensors>=0.4.5"])
    from safetensors import safe_open as st_safe_open

TORCH_OK = False
torch = None
if USE_TORCH_MATMUL:
    try:
        import torch
        TORCH_OK = True
    except Exception:
        TORCH_OK = False

np.set_printoptions(suppress=True, linewidth=160)

# ---------- helpers ----------
def die(msg: str):
    raise RuntimeError(msg)

def human_gb(nbytes: int) -> str:
    return f"{nbytes / (1024**3):.2f} GB"

def load_index(model_dir: str) -> Dict[str, str]:
    idx_path = Path(model_dir) / "model.safetensors.index.json"
    if not idx_path.exists():
        die(f"Missing index file: {idx_path}\nMake sure you copied model.safetensors.index.json into MODEL_DIR.")
    with open(idx_path, "r", encoding="utf-8") as f:
        j = json.load(f)
    wm = j.get("weight_map", {})
    if not wm:
        die(f"Index has no weight_map entries: {idx_path}")
    return wm

def list_local_shards(model_dir: str) -> Dict[str, str]:
    """Map shard filename -> absolute path"""
    model_dir = Path(model_dir)
    out = {}
    for p in sorted(model_dir.glob("model-*-of-*.safetensors")):
        out[p.name] = str(p)
    # also allow single-shard
    for p in sorted(model_dir.glob("model.safetensors")):
        out[p.name] = str(p)
    return out

# DeepSeek expert MLP has gate_proj, up_proj, down_proj weights (module is DeepseekMLP inside DeepseekMoE) :contentReference[oaicite:1]{index=1}
EXPERT_RE = re.compile(
    r"^model\.layers\.(\d+)\.mlp\.experts\.(\d+)\.(gate_proj|up_proj|down_proj)\.weight$"
)

def discover_experts_from_weight_map(weight_map: Dict[str, str]) -> Dict[int, Dict[int, Dict[str, str]]]:
    """
    Returns:
      layers[layer_idx][expert_id][proj_name] = tensor_key
    where proj_name in {"gate_proj","up_proj","down_proj"} and tensor_key is the exact key.
    """
    layers: Dict[int, Dict[int, Dict[str, str]]] = {}
    for k in weight_map.keys():
        m = EXPERT_RE.match(k)
        if not m:
            continue
        L = int(m.group(1))
        eid = int(m.group(2))
        proj = m.group(3)
        layers.setdefault(L, {}).setdefault(eid, {})[proj] = k
    return layers

def pick_layer(layers: Dict[int, Dict[int, Dict[str, str]]], preferred: Optional[int]) -> Tuple[int, List[int]]:
    """Pick a layer with at least 1 expert having down + (prefer_up or gate)."""
    if preferred is not None:
        if preferred not in layers:
            die(f"LAYER={preferred} not found in index (no experts matched pattern). Available layers: {sorted(layers)}")
        eids = sorted(layers[preferred].keys())
        return preferred, eids

    # auto-pick first valid
    for L in sorted(layers.keys()):
        eids = sorted(layers[L].keys())
        # ensure overlap down + something
        ok = []
        for eid in eids:
            d = layers[L][eid]
            if "down_proj" in d and ("up_proj" in d or "gate_proj" in d):
                ok.append(eid)
        if ok:
            return L, ok
    die("No MoE expert weights discovered in index (pattern mismatch).")

def resolve_shards_needed(weight_map: Dict[str, str], keys: List[str]) -> List[str]:
    needed = sorted({weight_map[k] for k in keys})
    return needed

class ShardStore:
    """Open shards once, reuse handles."""
    def __init__(self, shard_paths: Dict[str, str]):
        self.shard_paths = shard_paths
        self.handles_np = {}
        self.handles_pt = {}

    def get_tensor_np32(self, shard_fn: str, key: str) -> np.ndarray:
        # NumPy path first
        if shard_fn not in self.handles_np:
            self.handles_np[shard_fn] = st_safe_open(self.shard_paths[shard_fn], framework="np")
        try:
            arr = self.handles_np[shard_fn].get_tensor(key)
            return arr.astype(np.float32, copy=False)
        except TypeError:
            # some envs might not like bf16 -> fallback torch
            return self.get_tensor_torch_fp32(shard_fn, key).cpu().numpy()

    def get_tensor_torch_fp32(self, shard_fn: str, key: str):
        if not TORCH_OK:
            die("Torch fallback requested but torch is not available.")
        if shard_fn not in self.handles_pt:
            self.handles_pt[shard_fn] = st_safe_open(self.shard_paths[shard_fn], framework="pt")
        t = self.handles_pt[shard_fn].get_tensor(key)
        return t.detach().to(dtype=torch.float32, device="cpu")

    def close(self):
        # safe_open objects close on GC; explicitly drop refs
        self.handles_np.clear()
        self.handles_pt.clear()

def matmul_down_up(down: np.ndarray, up: np.ndarray) -> np.ndarray:
    """
    down: (hidden, inter)
    up:   (inter, hidden)
    returns (hidden, hidden)
    """
    if TORCH_OK and USE_TORCH_MATMUL:
        td = torch.from_numpy(down)
        tu = torch.from_numpy(up)
        out = (td @ tu).contiguous()
        return out.cpu().numpy()
    else:
        return down @ up

def to_dtype(arr: np.ndarray, dtype_out: str) -> np.ndarray:
    if dtype_out == "float16":
        return arr.astype(np.float16, copy=False)
    if dtype_out == "float32":
        return arr.astype(np.float32, copy=False)
    die(f"Unsupported DTYPE_OUT={dtype_out}")

def int8_symmetric_quant_error(x: np.ndarray) -> Dict[str, float]:
    """Simple per-tensor symmetric int8 quant error summary."""
    x = x.astype(np.float32, copy=False)
    maxabs = float(np.max(np.abs(x)))
    if maxabs == 0.0:
        return {"maxabs": 0.0, "scale": 1.0, "mse": 0.0, "rmse": 0.0, "rel_rmse": 0.0}
    scale = maxabs / 127.0
    q = np.clip(np.round(x / scale), -127, 127).astype(np.int8)
    xhat = (q.astype(np.float32) * scale)
    err = xhat - x
    mse = float(np.mean(err * err))
    rmse = float(math.sqrt(mse))
    rel = float(rmse / (np.sqrt(float(np.mean(x * x))) + 1e-12))
    return {"maxabs": maxabs, "scale": float(scale), "mse": mse, "rmse": rmse, "rel_rmse": rel}

# ---------- main ----------
print("== OFFLINE DeepSeek-MoE Ws Builder ==")
print(f"MODEL_DIR:  {MODEL_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"Torch available: {TORCH_OK} | USE_TORCH_MATMUL={USE_TORCH_MATMUL}")

model_dir = Path(MODEL_DIR)
if not model_dir.exists():
    die(f"MODEL_DIR does not exist: {MODEL_DIR}")

weight_map = load_index(MODEL_DIR)
layers = discover_experts_from_weight_map(weight_map)
layer_ok, eids_all = pick_layer(layers, LAYER)

# Choose experts that have required projections
valid_eids = []
for eid in sorted(eids_all):
    d = layers[layer_ok][eid]
    if "down_proj" not in d:
        continue
    if PREFER_UP in d:
        valid_eids.append(eid)
    elif PREFER_UP == "up_proj" and "gate_proj" in d:
        valid_eids.append(eid)  # fallback will happen per expert
    elif PREFER_UP == "gate_proj" and "up_proj" in d:
        valid_eids.append(eid)
if not valid_eids:
    die(f"No valid experts found at layer {layer_ok} with down_proj + ({PREFER_UP} or fallback).")

picked_eids = valid_eids[:MAX_EXPERTS]
print(f"\n[FOUND] layer={layer_ok} total_experts={len(valid_eids)} using_first={len(picked_eids)} eids={picked_eids}")

# Collect tensor keys to load
expert_specs = []
all_keys = []
for eid in picked_eids:
    d = layers[layer_ok][eid]
    down_k = d["down_proj"]
    up_k = d.get(PREFER_UP, None)
    used_up = PREFER_UP
    if up_k is None:
        # fallback
        alt = "gate_proj" if PREFER_UP == "up_proj" else "up_proj"
        up_k = d.get(alt, None)
        used_up = alt
    if up_k is None:
        continue
    expert_specs.append((eid, used_up, down_k, up_k))
    all_keys += [down_k, up_k]

if not expert_specs:
    die("After fallback logic, no experts remain to load.")

# Make sure shards exist locally
local_shards = list_local_shards(MODEL_DIR)
if not local_shards:
    die(f"No shard files found in MODEL_DIR.\nExpected model-00001-of-00007.safetensors ...\nGot empty.\nMODEL_DIR={MODEL_DIR}")

needed_shards = resolve_shards_needed(weight_map, all_keys)
missing = [fn for fn in needed_shards if fn not in local_shards]
if missing:
    die("Missing shard(s) in MODEL_DIR:\n" + "\n".join("  - " + m for m in missing) +
        f"\n\nPut the missing shards into: {MODEL_DIR}")

# Open shard handles once
store = ShardStore({fn: local_shards[fn] for fn in needed_shards})

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
out_npz = Path(OUTPUT_DIR) / f"deepseek_layer{layer_ok}_Ws_first{len(expert_specs)}.npz"

stats_rows = []
Ws = {}  # keep small set in memory; still saved to disk

print("\n=== Loading experts & building W_eff (with progress) ===")
for (eid, used_up, down_k, up_k) in tqdm(expert_specs, desc="experts"):
    down_shard = weight_map[down_k]
    up_shard   = weight_map[up_k]

    down = store.get_tensor_np32(down_shard, down_k)  # (hidden, inter)
    up   = store.get_tensor_np32(up_shard, up_k)      # (inter, hidden)

    # Build effective
    weff = matmul_down_up(down, up)

    # Convert dtype for saving
    down_s = to_dtype(down, DTYPE_OUT)
    up_s   = to_dtype(up, DTYPE_OUT)
    weff_s = to_dtype(weff, DTYPE_OUT)

    # Basic norms
    row = {
        "expert": eid,
        "used_up": used_up,
        "down_shape": tuple(down.shape),
        "up_shape": tuple(up.shape),
        "weff_shape": tuple(weff.shape),
        "down_frob": float(np.linalg.norm(down)),
        "up_frob": float(np.linalg.norm(up)),
        "weff_frob": float(np.linalg.norm(weff)),
    }

    # Optional int8 sanity
    if RUN_INT8_SANITY:
        q = int8_symmetric_quant_error(weff)
        row.update({
            "weff_int8_rmse": q["rmse"],
            "weff_int8_rel_rmse": q["rel_rmse"],
            "weff_int8_scale": q["scale"],
        })

    stats_rows.append(row)

    # Store arrays under unique names
    Ws[f"expert{eid}.down"] = down_s
    Ws[f"expert{eid}.up_{used_up}"] = up_s
    Ws[f"expert{eid}.W_eff"] = weff_s

    # free big temporaries
    del down, up, weff
    gc.collect()

store.close()

# Save packed result
np.savez_compressed(out_npz, **Ws)

# Print summary table
print("\n=== SUMMARY ===")
print(f"Saved: {out_npz}  ({out_npz.stat().st_size/1024/1024:.2f} MB)")
print(f"Layer: {layer_ok} | Experts saved: {len(expert_specs)} | DTYPE_OUT={DTYPE_OUT}")
print()

# Pretty print a few rows
cols = ["expert","used_up","down_shape","up_shape","weff_shape","weff_frob"]
if RUN_INT8_SANITY:
    cols += ["weff_int8_rel_rmse","weff_int8_scale"]

for r in stats_rows[:min(12, len(stats_rows))]:
    line = " | ".join([f"{c}={r[c]}" for c in cols])
    print(line)

print("\n✅ Done. You can now load the .npz and feed `expert*.W_eff` (or `down/up`) into your quantization pipeline.")
print("Tip: If you want a different layer, set LAYER=<int> and rerun.")


== OFFLINE DeepSeek-MoE Ws Builder ==
MODEL_DIR:  /home/daniyar/deepseek-model
OUTPUT_DIR: /home/daniyar/moe_ws_outputs
Torch available: True | USE_TORCH_MATMUL=True

[FOUND] layer=1 total_experts=64 using_first=8 eids=[0, 1, 2, 3, 4, 5, 6, 7]

=== Loading experts & building W_eff (with progress) ===


experts: 100%|███████████████████████████████████████████████████████████████████████████████| 8/8 [00:02<00:00,  2.80it/s]



=== SUMMARY ===
Saved: /home/daniyar/moe_ws_outputs/deepseek_layer1_Ws_first8.npz  (126.82 MB)
Layer: 1 | Experts saved: 8 | DTYPE_OUT=float16

expert=0 | used_up=up_proj | down_shape=(2048, 1408) | up_shape=(1408, 2048) | weff_shape=(2048, 2048) | weff_frob=81.77774047851562 | weff_int8_rel_rmse=0.019156362185399702 | weff_int8_scale=0.002649810839825728
expert=1 | used_up=up_proj | down_shape=(2048, 1408) | up_shape=(1408, 2048) | weff_shape=(2048, 2048) | weff_frob=79.79796600341797 | weff_int8_rel_rmse=0.013997053062172226 | weff_int8_scale=0.001889506193596547
expert=2 | used_up=up_proj | down_shape=(2048, 1408) | up_shape=(1408, 2048) | weff_shape=(2048, 2048) | weff_frob=68.22835540771484 | weff_int8_rel_rmse=0.01573123310351635 | weff_int8_scale=0.0018155933834436372
expert=3 | used_up=up_proj | down_shape=(2048, 1408) | up_shape=(1408, 2048) | weff_shape=(2048, 2048) | weff_frob=87.83383178710938 | weff_int8_rel_rmse=0.012141378951348071 | weff_int8_scale=0.001803937507426644

In [9]:
# ============================================================
# DeepSeek KT++-X Optimized (offline) — "everything included" v7
#
# Goals:
#   - Runs end-to-end even when CALIB_PATH is missing (no hard crash)
#   - Still supports "strict" mode for meaningful ridge-based Ws
#   - Better residual model: FULL per-expert r×r coefficients (RES_COEF_MODE=full)
#   - Faster sparse residual: vectorized top-k blocks by energy
#   - Compression-realistic basis option: BASIS_MODE=hadamard_perm (implicit, tiny)
#
# IMPORTANT (accuracy reality check):
#   - 99–100% accuracy on real DeepSeek routing requires REAL CALIB_PATH (true layer inputs)
#     and usually ROUTER_PATH (RIDGE_WEIGHTED=1). Without them, any metric is at best proxy.
#
# Dependencies: torch, safetensors, numpy (tqdm optional)
# Optional: transformers (only if you enable CAPTURE_CALIB=1)
# ============================================================

import os, re, json, math, time, random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except Exception as e:
    raise RuntimeError("This script requires PyTorch.") from e

try:
    from safetensors import safe_open
except Exception as e:
    raise RuntimeError("This script requires safetensors (pip install safetensors).") from e

try:
    from tqdm import tqdm
except Exception:
    def tqdm(x, **kwargs):  # type: ignore
        return x

# ----------------------------
# Threading / determinism
# ----------------------------
NTHREADS = int(os.environ.get("KTXX_THREADS", "8"))
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(NTHREADS))
try:
    torch.set_num_threads(NTHREADS)
except Exception:
    pass

SEED = int(os.environ.get("SEED", "1234"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

DEVICE = torch.device(os.environ.get("DEVICE", "cpu"))

DTYPE_W   = torch.float16   # stored blocks/bases dtype (unless int8 quant)
DTYPE_ACC = torch.float32   # compute dtype

def now():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def log(msg: str):
    print(msg, flush=True)

# ----------------------------
# Config
# ----------------------------
@dataclass
class Cfg:
    # Paths
    MODEL_DIR: str = os.environ.get("MODEL_DIR", "/home/daniyar/deepseek-model")
    OUTPUT_DIR: str = os.environ.get("OUTPUT_DIR", "/home/daniyar/moe_ws_outputs")

    # Model slice
    LAYER: int = int(os.environ.get("LAYER", "1"))
    MAX_EXPERTS: int = int(os.environ.get("MAX_EXPERTS", "16"))

    # Calibration: npz with key 'X' shape (N,H)
    CALIB_PATH: str = os.environ.get("CALIB_PATH", "").strip()
    CALIB_SAMPLES: int = int(os.environ.get("CALIB_SAMPLES", "2048"))

    # If STRICT_CALIB=1 and ridge needs calib but missing -> error.
    STRICT_CALIB: bool = os.environ.get("STRICT_CALIB", "0") == "1"

    # If ridge needs calib but missing: allow random (proxy only).
    ALLOW_RANDOM_CALIB: bool = os.environ.get("ALLOW_RANDOM_CALIB", "1") == "1"

    # Optional: auto-capture calib via transformers (best-effort)
    CAPTURE_CALIB: bool = os.environ.get("CAPTURE_CALIB", "0") == "1"
    CAPTURE_OUT_X: str = os.environ.get("CAPTURE_OUT_X", "").strip()  # if empty -> OUTPUT_DIR/calib_layer{L}.npz
    CAPTURE_OUT_P: str = os.environ.get("CAPTURE_OUT_P", "").strip()  # if empty -> OUTPUT_DIR/router_layer{L}.npz
    CAPTURE_TEXT: str = os.environ.get("CAPTURE_TEXT", "Hello world.").strip()
    CAPTURE_REPEAT: int = int(os.environ.get("CAPTURE_REPEAT", "128"))
    CAPTURE_BATCH_TOKENS: int = int(os.environ.get("CAPTURE_BATCH_TOKENS", "256"))

    # Router: npz with key 'P' shape (N, E_total) or (N, E_used)
    ROUTER_PATH: str = os.environ.get("ROUTER_PATH", "").strip()
    ROUTER_EIDS_ARE_GLOBAL: bool = os.environ.get("ROUTER_EIDS_ARE_GLOBAL", "1") == "1"

    # Build square mats
    LIN_MODE: str = os.environ.get("LIN_MODE", "ridge").strip().lower()  # ridge|weff_gate|weff
    RIDGE_DAMP: float = float(os.environ.get("RIDGE_DAMP", "1e-3"))      # relative to trace(XtX)/H
    RIDGE_WEIGHTED: bool = os.environ.get("RIDGE_WEIGHTED", "0") == "1"  # use router P as weights

    # If ridge/weff_gate needs X but X missing and STRICT_CALIB=0:
    #   - ridge falls back to weff unless ALLOW_RANDOM_CALIB=1
    AUTO_FALLBACK_LIN: bool = os.environ.get("AUTO_FALLBACK_LIN", "1") == "1"

    # For weff_gate when calib missing: approximate gate mean using weight-stat Monte Carlo
    WEFF_GATE_MC_SAMPLES: int = int(os.environ.get("WEFF_GATE_MC_SAMPLES", "64"))

    # Optional similarity rotation (applies after Ws build)
    USE_HADAMARD: bool = os.environ.get("USE_HADAMARD", "0") == "1"
    HAD_SEED: int = int(os.environ.get("HAD_SEED", "1234"))

    # Normalize each expert W to Frobenius norm 1 (keeps eval stable)
    NORMALIZE_W: bool = os.environ.get("NORMALIZE_W", "1") == "1"

    # Cache
    CACHE_MODE: str = os.environ.get("CACHE_MODE", "readwrite").strip().lower()  # off|read|write|readwrite
    CACHE_NAME: str = os.environ.get("CACHE_NAME", "").strip()
    FORCE_REBUILD_WS: bool = os.environ.get("FORCE_REBUILD_WS", "0") == "1"

    # Basis mode (compression-critical)
    BASIS_MODE: str = os.environ.get("BASIS_MODE", "dense_train").strip().lower()  # dense_train | hadamard_perm

    # Clustering
    M_CLUSTERS: int = int(os.environ.get("M_CLUSTERS", "0"))  # 0 => auto
    CLUSTER_ITERS: int = int(os.environ.get("CLUSTER_ITERS", "80"))
    CLUSTER_RESTARTS: int = int(os.environ.get("CLUSTER_RESTARTS", "4"))
    CLUSTER_FEAT_D: int = int(os.environ.get("CLUSTER_FEAT_D", "64"))
    CLUSTER_TARGET_SIZE: int = int(os.environ.get("CLUSTER_TARGET_SIZE", "4"))  # auto-M aims for this size
    CLUSTER_MIN_SIZE: int = int(os.environ.get("CLUSTER_MIN_SIZE", "2"))        # reassign tiny clusters

    # Training (U,V) only for BASIS_MODE=dense_train
    TRAIN_STEPS: int = int(os.environ.get("TRAIN_STEPS", "0"))
    SUBM: int = int(os.environ.get("SUBM", "256"))
    BATCH_E: int = int(os.environ.get("BATCH_E", "4"))
    LR_UV: float = float(os.environ.get("LR_UV", "1e-2"))
    REORTHO_EVERY: int = int(os.environ.get("REORTHO_EVERY", "1"))
    REPORT_EVERY: int = int(os.environ.get("REPORT_EVERY", "5"))
    TRAIN_MIN_CLUSTER: int = int(os.environ.get("TRAIN_MIN_CLUSTER", "3"))
    GRAD_CLIP: float = float(os.environ.get("GRAD_CLIP", "1.0"))

    # Training objective knobs
    TRAIN_OBJ: str = os.environ.get("TRAIN_OBJ", "energy_ratio").strip().lower()  # energy_ratio|logratio
    TRAIN_LAM_BLOCK: float = float(os.environ.get("TRAIN_LAM_BLOCK", "0.1"))
    TRAIN_BLOCK_USE_CORE_BLOCK: bool = os.environ.get("TRAIN_BLOCK_USE_CORE_BLOCK", "1") == "1"

    # Core model
    CORE_MODE: str = os.environ.get("CORE_MODE", "blocktopk_perexpert").strip().lower()
    CORE_AGG: str = os.environ.get("CORE_AGG", "mean").strip().lower()  # mean|max for shared blocktopk
    CORE_BLOCK: int = int(os.environ.get("CORE_BLOCK", "64"))
    CORE_TARGET: float = float(os.environ.get("CORE_TARGET", "0.90"))
    CORE_MAX_BLOCKS: int = int(os.environ.get("CORE_MAX_BLOCKS", "512"))

    # Residual
    RES_RANK: int = int(os.environ.get("RES_RANK", "64"))
    RES_COEF_MODE: str = os.environ.get("RES_COEF_MODE", "full").strip().lower()  # full|diag
    RES_BLOCKS: int = int(os.environ.get("RES_BLOCKS", "128"))
    RES_BSIZE: int = int(os.environ.get("RES_BSIZE", "64"))

    # Quantization (payload size)
    QMODE: str = os.environ.get("QMODE", "none").strip().lower()  # none | int8

    # Eval
    EVAL_TRIALS: int = int(os.environ.get("EVAL_TRIALS", "8"))
    ROUTED_K: int = int(os.environ.get("ROUTED_K", "8"))
    EVAL_BATCH: int = int(os.environ.get("EVAL_BATCH", "4"))
    EVAL_PER_EXPERT: bool = os.environ.get("EVAL_PER_EXPERT", "1") == "1"

cfg = Cfg()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

def cache_path():
    if cfg.CACHE_NAME:
        return os.path.join(cfg.OUTPUT_DIR, cfg.CACHE_NAME)
    tag = f"deepseek_layer{cfg.LAYER}_E{cfg.MAX_EXPERTS}_{cfg.LIN_MODE}_Ws_cache_v7.npz"
    return os.path.join(cfg.OUTPUT_DIR, tag)

def _meta_dict():
    return dict(
        model_dir=cfg.MODEL_DIR,
        layer=cfg.LAYER,
        max_experts=cfg.MAX_EXPERTS,
        lin_mode=cfg.LIN_MODE,
        ridge_damp=cfg.RIDGE_DAMP,
        ridge_weighted=cfg.RIDGE_WEIGHTED,
        calib_path=cfg.CALIB_PATH or "",
        calib_samples=cfg.CALIB_SAMPLES,
        router_path=cfg.ROUTER_PATH or "",
        router_global=cfg.ROUTER_EIDS_ARE_GLOBAL,
        hadamard=cfg.USE_HADAMARD,
        had_seed=cfg.HAD_SEED,
        normalize_w=cfg.NORMALIZE_W,
    )

def _encode_meta(meta: dict) -> np.ndarray:
    b = json.dumps(meta, sort_keys=True).encode("utf-8")
    return np.frombuffer(b, dtype=np.uint8)

def _decode_meta(arr: np.ndarray) -> dict:
    try:
        b = bytes(arr.tolist())
        return json.loads(b.decode("utf-8"))
    except Exception:
        return {}

def try_load_npz(path: str) -> Optional[Dict[str, np.ndarray]]:
    if not os.path.isfile(path):
        return None
    z = np.load(path, allow_pickle=False)
    out = {k: z[k] for k in z.files}
    z.close()
    return out

def save_npz(path: str, arrays: Dict[str, np.ndarray]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez(path, **arrays)

# ----------------------------
# Offline shard loading
# ----------------------------
def read_index(model_dir: str) -> Dict[str, str]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    wm = obj.get("weight_map", {})
    if not wm:
        raise RuntimeError("Index JSON has empty weight_map.")
    return wm

def find_layer_expert_ids(weight_map: Dict[str, str], layer: int) -> List[int]:
    pat = re.compile(rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.")
    ids = set()
    for k in weight_map.keys():
        m = pat.match(k)
        if m:
            ids.add(int(m.group(1)))
    return sorted(ids)

def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    prefix = f"model.layers.{layer}.mlp.experts.{eid}."
    def pick(cands: List[str]) -> Optional[str]:
        for suf in cands:
            k = prefix + suf
            if k in weight_map:
                return k
        return None
    up   = pick(["up_proj.weight", "w3.weight", "w1.weight"])
    gate = pick(["gate_proj.weight", "w1.weight", "w3.weight"])
    down = pick(["down_proj.weight", "w2.weight"])
    if up is None or gate is None or down is None:
        return {}
    if up == gate:
        # try to disambiguate if fallback collided
        gate2 = pick(["gate_proj.weight"])
        if gate2:
            gate = gate2
    return {"up": up, "gate": gate, "down": down}

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], keys: List[str]) -> Dict[str, torch.Tensor]:
    by_shard: Dict[str, List[str]] = {}
    for k in keys:
        shard = weight_map.get(k, None)
        if shard is None:
            raise KeyError(f"Key not in weight_map: {k}")
        by_shard.setdefault(shard, []).append(k)

    out: Dict[str, torch.Tensor] = {}
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(sp):
            raise FileNotFoundError(f"Missing shard: {sp}")
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks:
                out[k] = f.get_tensor(k)
    return out

# ----------------------------
# Router
# ----------------------------
def load_router_P() -> Optional[np.ndarray]:
    if not cfg.ROUTER_PATH:
        return None
    if not os.path.isfile(cfg.ROUTER_PATH):
        raise FileNotFoundError(f"ROUTER_PATH not found: {cfg.ROUTER_PATH}")
    z = np.load(cfg.ROUTER_PATH, allow_pickle=False)
    if "P" not in z.files:
        raise KeyError(f"ROUTER npz missing key 'P'. Has keys: {list(z.files)}")
    P = z["P"].astype(np.float32, copy=False)
    z.close()
    log(f"[router] Loaded P: {tuple(P.shape)}")
    return P

# ----------------------------
# Hadamard utilities
# ----------------------------
def is_power_of_two(n: int) -> bool:
    return (n > 0) and ((n & (n - 1)) == 0)

def fwht(x: torch.Tensor) -> torch.Tensor:
    """
    Fast Walsh–Hadamard Transform on last dimension (unscaled).
    Works for any prefix dims, requires last dim to be power-of-two.
    """
    n = x.shape[-1]
    if not is_power_of_two(n):
        raise ValueError(f"FWHT requires power-of-two, got {n}")
    orig_shape = x.shape
    y = x.reshape(-1, n).contiguous()
    h = 1
    while h < n:
        y = y.reshape(-1, n // (2 * h), 2, h)
        a = y[:, :, 0, :]
        b = y[:, :, 1, :]
        y[:, :, 0, :] = a + b
        y[:, :, 1, :] = a - b
        y = y.reshape(-1, n)
        h *= 2
    return y.reshape(orig_shape)

def fwht_ortho(x: torch.Tensor) -> torch.Tensor:
    return fwht(x) / math.sqrt(x.shape[-1])

# ----------------------------
# Quantization (optional)
# ----------------------------
class QTensor:
    __slots__ = ("q", "scale", "shape", "per_row")
    def __init__(self, q: torch.Tensor, scale: torch.Tensor, shape: Tuple[int, ...], per_row: bool):
        self.q = q
        self.scale = scale
        self.shape = shape
        self.per_row = per_row

def quantize_int8(t: torch.Tensor, per_row: bool = False) -> Any:
    if cfg.QMODE != "int8":
        return t
    tt = t.to(torch.float32)
    if per_row and tt.ndim == 2:
        mx = tt.abs().amax(dim=1).clamp_min(1e-8)  # (m,)
        scale = (mx / 127.0).to(torch.float32)
        q = torch.round(tt / scale.unsqueeze(1)).clamp(-127, 127).to(torch.int8)
        return QTensor(q=q, scale=scale, shape=tuple(tt.shape), per_row=True)
    else:
        mx = tt.abs().max().clamp_min(1e-8)
        scale = (mx / 127.0).to(torch.float32)
        q = torch.round(tt / scale).clamp(-127, 127).to(torch.int8)
        return QTensor(q=q, scale=scale, shape=tuple(tt.shape), per_row=False)

def dequantize(x: Any, device: torch.device, dtype: torch.dtype = DTYPE_ACC) -> torch.Tensor:
    if isinstance(x, QTensor):
        if x.per_row:
            t = x.q.to(torch.float32) * x.scale.to(torch.float32).unsqueeze(1)
        else:
            t = x.q.to(torch.float32) * x.scale.to(torch.float32)
        return t.to(device=device, dtype=dtype)
    return x.to(device=device, dtype=dtype)

# ----------------------------
# Optional: capture calib X + router P from transformers (best-effort)
# ----------------------------
def _default_capture_paths() -> Tuple[str, str]:
    xout = cfg.CAPTURE_OUT_X or os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
    pout = cfg.CAPTURE_OUT_P or os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
    return xout, pout

def try_capture_calib_and_router(H: int, E_total_guess: int) -> Tuple[Optional[str], Optional[str]]:
    """
    Best-effort, optional. Requires transformers installed + model loadable from MODEL_DIR.
    Produces:
      - X npz with key 'X' shape (N,H)
      - P npz with key 'P' shape (N,E_total_guess) if router module found
    """
    if not cfg.CAPTURE_CALIB:
        return None, None
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer
    except Exception as e:
        log(f"[capture] transformers not available: {e}")
        return None, None

    xout, pout = _default_capture_paths()
    log(f"[capture] trying transformers capture -> {xout} and {pout}")

    tok = AutoTokenizer.from_pretrained(cfg.MODEL_DIR, use_fast=True)
    model = AutoModelForCausalLM.from_pretrained(cfg.MODEL_DIR, torch_dtype=torch.float16, device_map=None)
    model.eval()
    model.to(DEVICE)

    # Hook layer input (mlp input). We try common paths; if not found, fallback to module name search.
    layer = None
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        if cfg.LAYER < len(model.model.layers):
            layer = model.model.layers[cfg.LAYER]
    if layer is None:
        log("[capture] could not locate model.model.layers[LAYER]; capture aborted.")
        return None, None

    # Find MLP module
    mlp = getattr(layer, "mlp", None)
    if mlp is None:
        log("[capture] layer has no .mlp; capture aborted.")
        return None, None

    X_chunks: List[torch.Tensor] = []
    P_chunks: List[torch.Tensor] = []

    # Try to locate a router/gate module inside mlp (best effort)
    gate_mod = None
    gate_name = None
    for name, mod in mlp.named_modules():
        # heuristic: linear with out_features >= experts
        if isinstance(mod, nn.Linear) and getattr(mod, "out_features", 0) >= E_total_guess:
            if ("gate" in name.lower()) or ("router" in name.lower()):
                gate_mod = mod
                gate_name = name
                break

    if gate_mod is not None:
        log(f"[capture] found gate-like module: mlp.{gate_name} (Linear out={gate_mod.out_features})")
    else:
        log("[capture] no gate-like Linear found; will capture X only.")

    # Hook mlp input
    def mlp_pre_hook(_module, inputs):
        hs = inputs[0]
        if hs is None:
            return
        hs = hs.detach().to(torch.float32)
        X_chunks.append(hs.reshape(-1, hs.shape[-1]).cpu())

    # Hook gate output if found
    def gate_hook(_module, inputs, output):
        out = output
        if isinstance(out, (tuple, list)):
            out = out[0]
        if torch.is_tensor(out):
            logits = out.detach().to(torch.float32)
            # softmax -> probs
            probs = torch.softmax(logits, dim=-1)
            P_chunks.append(probs.reshape(-1, probs.shape[-1]).cpu())

    h1 = mlp.register_forward_pre_hook(mlp_pre_hook)
    h2 = gate_mod.register_forward_hook(gate_hook) if gate_mod is not None else None

    # Build input batch
    text = (cfg.CAPTURE_TEXT + "\n") * cfg.CAPTURE_REPEAT
    enc = tok(text, return_tensors="pt", truncation=True, max_length=cfg.CAPTURE_BATCH_TOKENS)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    with torch.no_grad():
        _ = model(**enc)

    # remove hooks
    h1.remove()
    if h2 is not None:
        h2.remove()

    if not X_chunks:
        log("[capture] no X collected; capture failed.")
        return None, None

    X = torch.cat(X_chunks, dim=0)
    if X.shape[0] > cfg.CALIB_SAMPLES:
        X = X[:cfg.CALIB_SAMPLES]
    np.savez(xout, X=X.numpy().astype(np.float32))
    log(f"[capture] wrote X: {xout}  shape={tuple(X.shape)}")

    p_path_written = None
    if P_chunks:
        P = torch.cat(P_chunks, dim=0)
        # pad/truncate to E_total_guess
        if P.shape[1] >= E_total_guess:
            P = P[:, :E_total_guess]
        else:
            pad = torch.zeros(P.shape[0], E_total_guess - P.shape[1])
            P = torch.cat([P, pad], dim=1)
        if P.shape[0] > X.shape[0]:
            P = P[:X.shape[0]]
        elif P.shape[0] < X.shape[0]:
            X = X[:P.shape[0]]
        np.savez(pout, P=P.numpy().astype(np.float32))
        log(f"[capture] wrote P: {pout}  shape={tuple(P.shape)}")
        p_path_written = pout

    return xout, p_path_written

# ----------------------------
# Calibration X
# ----------------------------
def load_calib_X(H: int) -> Optional[torch.Tensor]:
    if cfg.CALIB_PATH and os.path.isfile(cfg.CALIB_PATH):
        z = np.load(cfg.CALIB_PATH, allow_pickle=False)
        if "X" not in z.files:
            raise KeyError(f"CALIB npz missing key 'X'. Has keys: {list(z.files)}")
        X = torch.from_numpy(z["X"]).to(DTYPE_ACC).to(DEVICE)
        z.close()
        if X.ndim != 2 or X.shape[1] != H:
            raise RuntimeError(f"Bad X shape: {tuple(X.shape)} expected (*,{H})")
        if X.shape[0] > cfg.CALIB_SAMPLES:
            X = X[:cfg.CALIB_SAMPLES]
        log(f"[calib] Loaded X: {tuple(X.shape)}")
        return X

    # Attempt capture if enabled
    if cfg.CAPTURE_CALIB:
        # E_total_guess doesn't matter much; use 256 as safe upper guess
        xout, pout = try_capture_calib_and_router(H, E_total_guess=256)
        if xout:
            cfg.CALIB_PATH = xout
        if pout and not cfg.ROUTER_PATH:
            cfg.ROUTER_PATH = pout
        if cfg.CALIB_PATH and os.path.isfile(cfg.CALIB_PATH):
            return load_calib_X(H)

    # No calib available
    return None

# ----------------------------
# MLP forward (for ridge)
# ----------------------------
@torch.no_grad()
def forward_mlp_no_grad(X: torch.Tensor, W_gate: torch.Tensor, W_up: torch.Tensor, W_down: torch.Tensor) -> torch.Tensor:
    Xf = X.to(DTYPE_ACC)
    up = Xf @ W_up.to(DTYPE_ACC).t()
    gate = Xf @ W_gate.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    y = hid @ W_down.to(DTYPE_ACC).t()
    return y

# ----------------------------
# Gate-mean approximation when X missing (for weff_gate)
# ----------------------------
@torch.no_grad()
def gate_mean_from_weight_stats(W_gate: torch.Tensor, nsamp: int) -> torch.Tensor:
    """
    Approximate m_j = E[silu(z_j)] where z_j ~ N(0, sigma_j^2).
    sigma_j estimated from row-norm of gate weight assuming x ~ N(0, I).
    Vectorized Monte Carlo over d_ff rows.
    """
    Wg = W_gate.to(DTYPE_ACC)
    sigma = torch.linalg.norm(Wg, dim=1).clamp_min(1e-8)  # (d_ff,)
    g = torch.Generator(device=Wg.device).manual_seed(SEED + 999)
    z = torch.randn(nsamp, sigma.shape[0], generator=g, device=Wg.device, dtype=DTYPE_ACC) * sigma.view(1, -1)
    m = F.silu(z).mean(dim=0)  # (d_ff,)
    return m

# ----------------------------
# Build expert square Ws
# ----------------------------
@torch.no_grad()
def build_Ws_square() -> Tuple[List[int], torch.Tensor]:
    wm = read_index(cfg.MODEL_DIR)
    all_eids = find_layer_expert_ids(wm, cfg.LAYER)
    if not all_eids:
        raise RuntimeError(f"No experts found for layer={cfg.LAYER}")
    eids = all_eids[:cfg.MAX_EXPERTS]
    log(f"[found] layer {cfg.LAYER} experts: {all_eids}  (using {len(eids)})")

    per_e: Dict[int, Dict[str, str]] = {}
    need_keys: List[str] = []
    for eid in eids:
        kk = pick_expert_tensor_keys(wm, cfg.LAYER, eid)
        if not kk or ("up" not in kk) or ("down" not in kk) or ("gate" not in kk):
            raise RuntimeError(f"Expert {eid} missing keys: {kk}")
        per_e[eid] = kk
        need_keys += [kk["up"], kk["down"], kk["gate"]]

    log("[load] reading tensors from shards ...")
    T = load_tensors_from_shards(cfg.MODEL_DIR, wm, sorted(set(need_keys)))

    W_up0 = T[per_e[eids[0]]["up"]].to(DTYPE_W)
    dff, H = W_up0.shape
    log(f"[shape] H={H} d_ff={dff}")

    # Load X only if needed
    X: Optional[torch.Tensor] = None
    if cfg.LIN_MODE in ("ridge", "weff_gate"):
        X = load_calib_X(H)

    # Handle missing calib
    lin_mode = cfg.LIN_MODE
    if X is None and cfg.LIN_MODE == "ridge":
        if cfg.STRICT_CALIB:
            raise RuntimeError("CALIB_PATH is missing and STRICT_CALIB=1. Provide real X or disable strict.")
        if cfg.ALLOW_RANDOM_CALIB:
            log("[calib] CALIB_PATH missing -> using RANDOM X (proxy only; not meaningful for real accuracy).")
            X = torch.randn(cfg.CALIB_SAMPLES, H, dtype=DTYPE_ACC, device=DEVICE)
        elif cfg.AUTO_FALLBACK_LIN:
            log("[calib] CALIB_PATH missing and ALLOW_RANDOM_CALIB=0 -> falling back LIN_MODE=weff (proxy).")
            lin_mode = "weff"
        else:
            raise RuntimeError("CALIB_PATH missing; set ALLOW_RANDOM_CALIB=1 or AUTO_FALLBACK_LIN=1 or provide CALIB_PATH.")

    if X is None and cfg.LIN_MODE == "weff_gate":
        if cfg.STRICT_CALIB:
            raise RuntimeError("CALIB_PATH missing and STRICT_CALIB=1, but LIN_MODE=weff_gate needs X (or stats).")
        log("[calib] CALIB_PATH missing -> weff_gate will use WEIGHT-STATS Monte Carlo for gate mean.")

    # Router P optional
    P = load_router_P() if cfg.ROUTER_PATH else None

    use_post_had = cfg.USE_HADAMARD
    if use_post_had and not is_power_of_two(H):
        log(f"[hadamard] H={H} not power-of-two; disabling USE_HADAMARD.")
        use_post_had = False

    Xf = X.to(DTYPE_ACC) if X is not None else None
    cholG = None
    lam = None
    I = torch.eye(H, dtype=DTYPE_ACC, device=DEVICE)

    if lin_mode == "ridge":
        assert Xf is not None
        XtX = (Xf.t() @ Xf)
        lam = cfg.RIDGE_DAMP * float(torch.trace(XtX).item()) / float(H)
        if not (cfg.RIDGE_WEIGHTED and (P is not None)):
            G = XtX + lam * I
            cholG = torch.linalg.cholesky(G)

    Ws: List[torch.Tensor] = []
    for i, eid in enumerate(tqdm(eids, desc="Build Ws", leave=False)):
        W_up   = T[per_e[eid]["up"]].to(DTYPE_W).to(DEVICE)
        W_down = T[per_e[eid]["down"]].to(DTYPE_W).to(DEVICE)
        W_gate = T[per_e[eid]["gate"]].to(DTYPE_W).to(DEVICE)

        if lin_mode == "weff":
            W = (W_down.to(DTYPE_ACC) @ W_up.to(DTYPE_ACC))

        elif lin_mode == "weff_gate":
            if Xf is not None:
                gate_act = (Xf @ W_gate.to(DTYPE_ACC).t())
                m = F.silu(gate_act).mean(dim=0)  # (dff,)
            else:
                m = gate_mean_from_weight_stats(W_gate, nsamp=cfg.WEFF_GATE_MC_SAMPLES)  # (dff,)
            W = (W_down.to(DTYPE_ACC) * m.view(1, -1)) @ W_up.to(DTYPE_ACC)

        elif lin_mode == "ridge":
            assert X is not None and Xf is not None
            Y = forward_mlp_no_grad(X, W_gate, W_up, W_down).to(DTYPE_ACC)  # (N,H)

            # weighted ridge if router exists
            if cfg.RIDGE_WEIGHTED and (P is not None):
                if cfg.ROUTER_EIDS_ARE_GLOBAL:
                    if eid >= P.shape[1]:
                        raise RuntimeError(f"ROUTER P has {P.shape[1]} experts but eid={eid} requested.")
                    w = torch.from_numpy(P[:X.shape[0], eid]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
                else:
                    if i >= P.shape[1]:
                        raise RuntimeError(f"ROUTER P has {P.shape[1]} columns but i={i} requested.")
                    w = torch.from_numpy(P[:X.shape[0], i]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)

                sw = torch.sqrt(w + 1e-12).view(-1, 1)
                Xw = Xf * sw
                Yw = Y * sw

                XtX_e = Xw.t() @ Xw
                lam_e = cfg.RIDGE_DAMP * float(torch.trace(XtX_e).item()) / float(H)
                G_e = XtX_e + lam_e * I
                chol = torch.linalg.cholesky(G_e)
                XtY = Xw.t() @ Yw
                Wt = torch.cholesky_solve(XtY, chol)  # (H,H)
                W = Wt.t().contiguous()
            else:
                assert cholG is not None
                XtY = Xf.t() @ Y
                Wt = torch.cholesky_solve(XtY, cholG)  # (H,H)
                W = Wt.t().contiguous()
        else:
            raise ValueError("LIN_MODE must be ridge|weff_gate|weff")

        if use_post_had:
            # W <- H W H (orthonormal H); apply right then left via transpose trick
            W = fwht_ortho(W)              # right-multiply by H^T == H (on rows) is not correct
            # Correct: multiply on RIGHT => transform last dim:
            # Already done above (fwht on last dim). Now left multiply by H:
            W = fwht_ortho(W.t().contiguous()).t().contiguous()

        if cfg.NORMALIZE_W:
            fn = torch.linalg.norm(W, ord="fro").clamp_min(1e-12)
            W = (W / fn).contiguous()

        Ws.append(W.contiguous())

    WsT = torch.stack(Ws, dim=0).to(DTYPE_ACC)  # (E,H,H)
    return eids, WsT

def load_or_build_Ws() -> Tuple[List[int], torch.Tensor]:
    p = cache_path()
    do_read = cfg.CACHE_MODE in ("read", "readwrite")
    do_write = cfg.CACHE_MODE in ("write", "readwrite")

    if do_read and (not cfg.FORCE_REBUILD_WS) and os.path.isfile(p):
        z = try_load_npz(p)
        if z and ("expert_ids" in z) and ("Ws" in z):
            ok = True
            if "meta" in z:
                meta = _decode_meta(z["meta"])
                ok = (meta == _meta_dict())
            if ok:
                eids = [int(x) for x in z["expert_ids"].tolist()]
                Ws = torch.from_numpy(z["Ws"]).to(DTYPE_ACC).to(DEVICE)
                log(f"[cache] loaded Ws: {p}  Ws={tuple(Ws.shape)}")
                return eids, Ws
            log("[cache] meta mismatch -> rebuilding Ws.")
        else:
            log("[cache] invalid cache -> rebuilding Ws.")

    eids, Ws = build_Ws_square()
    if do_write:
        save_npz(p, {
            "meta": _encode_meta(_meta_dict()),
            "expert_ids": np.array(eids, dtype=np.int32),
            "Ws": Ws.detach().cpu().numpy().astype(np.float32),
        })
        log(f"[cache] wrote: {p}  ({os.path.getsize(p)/1e6:.2f} MB)")
    return eids, Ws

# ----------------------------
# Basis (dense or implicit)
# ----------------------------
class ImplicitHadamardPerm:
    """
    Orthonormal U:
      x @ U = FWHT( (x[:, perm] * sign) ) / sqrt(n)
    """
    def __init__(self, n: int, seed: int):
        if not is_power_of_two(n):
            raise ValueError(f"hadamard_perm requires power-of-two n, got {n}")
        g = torch.Generator(device="cpu").manual_seed(seed)
        perm = torch.randperm(n, generator=g)
        inv = torch.empty_like(perm)
        inv[perm] = torch.arange(n)
        sign = (torch.randint(0, 2, (n,), generator=g, dtype=torch.int8) * 2 - 1).to(torch.int8)
        self.n = n
        self.perm = perm
        self.inv_perm = inv
        self.sign = sign

    def apply(self, x: torch.Tensor) -> torch.Tensor:
        xp = x.index_select(-1, self.perm.to(x.device))
        xs = xp * self.sign.to(x.device).to(x.dtype)
        return fwht_ortho(xs)

    def apply_T(self, x: torch.Tensor) -> torch.Tensor:
        y = fwht_ortho(x)
        y = y * self.sign.to(y.device).to(y.dtype)
        return y.index_select(-1, self.inv_perm.to(y.device))

    def project_matrix(self, W: torch.Tensor, V: "ImplicitHadamardPerm") -> torch.Tensor:
        # X = U^T W V
        WTU = self.apply(W.t().contiguous()).t().contiguous()
        return V.apply(WTU.contiguous())

    def reconstruct_matrix(self, X: torch.Tensor, V: "ImplicitHadamardPerm") -> torch.Tensor:
        # W = U X V^T
        XTUT = self.apply_T(X.t().contiguous()).t().contiguous()
        return V.apply_T(XTUT.contiguous())

class DenseBasis(nn.Module):
    def __init__(self, U_init: torch.Tensor):
        super().__init__()
        self.M = nn.Parameter(U_init.clone().to(DEVICE))

    def orthogonal(self) -> torch.Tensor:
        Q, _ = torch.linalg.qr(self.M)
        return Q

# ----------------------------
# Clustering
# ----------------------------
@torch.no_grad()
def expert_features(Ws: torch.Tensor, d: int) -> torch.Tensor:
    E, n, _ = Ws.shape
    d = int(d)
    g = torch.Generator(device="cpu").manual_seed(SEED)
    Rsign = (torch.randint(0, 2, (n, d), generator=g, dtype=torch.int8) * 2 - 1).to(DTYPE_ACC).to(DEVICE)

    feats = []
    for e in range(E):
        W = Ws[e]
        rowE = (W * W).sum(dim=1)  # (n,)
        colE = (W * W).sum(dim=0)  # (n,)
        f = torch.cat([(rowE @ Rsign), (colE @ Rsign)], dim=0)
        feats.append(f.unsqueeze(0))
    X = torch.cat(feats, dim=0)  # (E, 2d)
    X = X / (X.norm(dim=1, keepdim=True).clamp_min(1e-12))
    return X

@torch.no_grad()
def kmeans_pp_init(X: torch.Tensor, k: int, g: torch.Generator) -> torch.Tensor:
    n = X.shape[0]
    centers = []
    first = torch.randint(0, n, (1,), generator=g).item()
    centers.append(X[first].clone())
    for _ in range(1, k):
        C = torch.stack(centers, dim=0)
        dist2 = torch.cdist(X, C).pow(2).amin(dim=1)
        probs = dist2 / dist2.sum().clamp_min(1e-12)
        idx = torch.multinomial(probs, 1, generator=g).item()
        centers.append(X[idx].clone())
    return torch.stack(centers, dim=0)

@torch.no_grad()
def kmeans_torch(X: torch.Tensor, k: int, iters: int, restarts: int) -> Tuple[torch.Tensor, torch.Tensor]:
    best_inertia = None
    best_lab = None
    best_C = None
    for r in range(max(1, restarts)):
        g = torch.Generator(device="cpu").manual_seed(SEED + 1000 + r)
        C = kmeans_pp_init(X, k=k, g=g)

        for _ in range(iters):
            dist = torch.cdist(X, C)
            lab = dist.argmin(dim=1)
            for j in range(k):
                m = (lab == j)
                if m.any():
                    C[j] = X[m].mean(dim=0)
                else:
                    far = dist.max(dim=1).indices[0].item()
                    C[j] = X[far].clone()

        dist = torch.cdist(X, C)
        lab = dist.argmin(dim=1)
        inertia = float((dist.gather(1, lab.view(-1, 1)).pow(2)).sum().item())
        if (best_inertia is None) or (inertia < best_inertia):
            best_inertia = inertia
            best_lab = lab.clone()
            best_C = C.clone()
    return best_lab, best_C  # type: ignore

@torch.no_grad()
def enforce_min_cluster_size(labels: torch.Tensor, centers: torch.Tensor, X: torch.Tensor, min_size: int) -> torch.Tensor:
    if min_size <= 1:
        return labels
    k = centers.shape[0]
    counts = torch.bincount(labels, minlength=k)
    big = (counts >= min_size).nonzero(as_tuple=False).flatten()
    small = (counts < min_size).nonzero(as_tuple=False).flatten()
    if big.numel() == 0 or small.numel() == 0:
        return labels
    big_centers = centers[big]
    for c in small.tolist():
        idxs = (labels == c).nonzero(as_tuple=False).flatten()
        if idxs.numel() == 0:
            continue
        d = torch.cdist(X[idxs], big_centers)
        nn = d.argmin(dim=1)
        labels[idxs] = big[nn]
    return labels

# ----------------------------
# Core selection
# ----------------------------
@torch.no_grad()
def _block_energy_grid(X: torch.Tensor, b: int) -> torch.Tensor:
    n = X.shape[0]
    nb = (n + b - 1) // b
    if (n % b) != 0:
        pad = nb * b - n
        Xp = torch.zeros(nb*b, nb*b, dtype=X.dtype, device=X.device)
        Xp[:n, :n] = X
        X = Xp
        n = nb*b
    Xb = X.view(nb, b, nb, b).permute(0, 2, 1, 3).contiguous()  # (nb,nb,b,b)
    E = (Xb * Xb).sum(dim=(2,3))  # (nb,nb)
    return E

@torch.no_grad()
def _pick_top_blocks_from_energy(Eg: torch.Tensor, tot_energy: float, b: int, target: float, max_blocks: int) -> Tuple[List[Tuple[int,int,int,int]], float]:
    nb = Eg.shape[0]
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    csum = torch.cumsum(flat[order], dim=0)
    frac = csum / max(tot_energy, 1e-12)
    need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else flat.numel())
    K = min(need, max_blocks, flat.numel())
    pick = order[:K].tolist()
    kept = float(flat[order[:K]].sum().item())
    blocks = []
    for idx in pick:
        bi = idx // nb
        bj = idx % nb
        blocks.append((bi*b, bj*b, b, b))
    return blocks, kept / max(tot_energy, 1e-12)

@torch.no_grad()
def choose_core_blocks(X_list: List[torch.Tensor], mode: str, agg: str, block: int, target: float, max_blocks: int):
    if mode == "none":
        return {"mode": "none", "blocks_shared": [], "blocks_per_expert": None, "energy_fracs": []}
    n = X_list[0].shape[0]
    b = int(block)
    if b <= 0:
        return {"mode": "none", "blocks_shared": [], "blocks_per_expert": None, "energy_fracs": []}

    if mode == "blocktopk":
        Eg_all = []
        tots = []
        for X in X_list:
            tots.append(float((X*X).sum().item()))
            Eg_all.append(_block_energy_grid(X, b).to(DTYPE_ACC))
        tot = float(np.mean(tots))
        Eg = torch.stack(Eg_all, dim=0).amax(dim=0) if agg == "max" else torch.stack(Eg_all, dim=0).mean(dim=0)
        blocks, ef = _pick_top_blocks_from_energy(Eg, tot, b, target, max_blocks)
        return {"mode": "blocktopk", "blocks_shared": blocks, "blocks_per_expert": None, "energy_fracs": [ef]}

    if mode == "blocktopk_perexpert":
        blocks_per = []
        efracs = []
        for X in X_list:
            tot = float((X*X).sum().item())
            Eg = _block_energy_grid(X, b).to(DTYPE_ACC)
            blocks, ef = _pick_top_blocks_from_energy(Eg, tot, b, target, max_blocks)
            blocks_per.append(blocks)
            efracs.append(ef)
        return {"mode": "blocktopk_perexpert", "blocks_shared": [], "blocks_per_expert": blocks_per, "energy_fracs": efracs}

    if mode == "blockdiag":
        nb = (n + b - 1) // b
        diagE = torch.zeros(nb, dtype=DTYPE_ACC, device=X_list[0].device)
        tots = []
        for X in X_list:
            tots.append(float((X*X).sum().item()))
            Eg = _block_energy_grid(X, b).to(DTYPE_ACC)
            diagE += torch.diagonal(Eg, 0)
        tot = float(np.mean(tots))
        diagE /= max(1, len(X_list))
        order = torch.argsort(diagE, descending=True)
        csum = torch.cumsum(diagE[order], dim=0)
        frac = csum / max(tot, 1e-12)
        need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else nb)
        K = min(need, max_blocks, nb)
        blocks = []
        for bi in order[:K].tolist():
            i0 = bi * b
            blocks.append((i0, i0, b, b))
        ef = float(frac[K-1].item()) if K > 0 else 0.0
        return {"mode": "blockdiag", "blocks_shared": blocks, "blocks_per_expert": None, "energy_fracs": [ef]}

    raise ValueError("CORE_MODE must be blocktopk_perexpert|blocktopk|blockdiag|none")

# ----------------------------
# Residual blocks: vectorized top-k
# ----------------------------
@torch.no_grad()
def topk_blocks(R: torch.Tensor, t: int, b: int) -> List[Tuple[int,int,torch.Tensor]]:
    if t <= 0 or b <= 0:
        return []
    Eg = _block_energy_grid(R, b).to(DTYPE_ACC)
    nb = Eg.shape[0]
    flat = Eg.reshape(-1)
    K = min(int(t), flat.numel())
    idx = torch.argsort(flat, descending=True)[:K]
    blocks: List[Tuple[int,int,torch.Tensor]] = []
    for k in idx.tolist():
        bi = k // nb
        bj = k % nb
        i0 = bi*b
        j0 = bj*b
        i1 = min(R.shape[0], i0+b)
        j1 = min(R.shape[1], j0+b)
        blocks.append((i0, j0, R[i0:i1, j0:j1].clone()))
    return blocks

# ----------------------------
# Payload build
# ----------------------------
@torch.no_grad()
def svd_init(W_mean: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    U, _, Vh = torch.linalg.svd(W_mean, full_matrices=False)
    return U.contiguous(), Vh.t().contiguous()

@torch.no_grad()
def build_payload_from_X_list(X_list: List[torch.Tensor], idx: List[int], basis: dict, n: int) -> dict:
    core = choose_core_blocks(X_list, cfg.CORE_MODE, cfg.CORE_AGG, cfg.CORE_BLOCK, cfg.CORE_TARGET, cfg.CORE_MAX_BLOCKS)

    # core blocks per expert
    core_blocks: List[List[Tuple[int,int,Any]]] = []
    if core["blocks_per_expert"] is not None:
        for X, bl in zip(X_list, core["blocks_per_expert"]):
            lst = []
            for (i0, j0, h, w) in bl:
                h = min(h, X.shape[0]-i0); w = min(w, X.shape[1]-j0)
                if h > 0 and w > 0:
                    B = X[i0:i0+h, j0:j0+w].to(DTYPE_W).contiguous()
                    lst.append((i0, j0, quantize_int8(B)))
            core_blocks.append(lst)
    else:
        bl = core["blocks_shared"]
        for X in X_list:
            lst = []
            for (i0, j0, h, w) in bl:
                h = min(h, X.shape[0]-i0); w = min(w, X.shape[1]-j0)
                if h > 0 and w > 0:
                    B = X[i0:i0+h, j0:j0+w].to(DTYPE_W).contiguous()
                    lst.append((i0, j0, quantize_int8(B)))
            core_blocks.append(lst)

    # residual after core
    R_list = []
    for X, cb in zip(X_list, core_blocks):
        Xc = torch.zeros_like(X)
        for (i0, j0, Bq) in cb:
            B = dequantize(Bq, device=X.device, dtype=DTYPE_ACC)
            h, w = B.shape
            Xc[i0:i0+h, j0:j0+w] = B
        R_list.append((X - Xc).contiguous())

    # shared low-rank bases DL/DR from concatenated residuals
    r = min(int(cfg.RES_RANK), X_list[0].shape[0])
    DLq = DRq = None
    coef = None
    if r > 0:
        Rcat = torch.cat(R_list, dim=0)  # (E*n, n)
        RcatT = torch.cat([R.t() for R in R_list], dim=0)

        q = min(Rcat.shape[1], r + 16)
        try:
            _, _, V = torch.svd_lowrank(Rcat, q=q, niter=2)
            DR = V[:, :r].contiguous()
        except Exception:
            _, _, Vh = torch.linalg.svd(Rcat, full_matrices=False)
            DR = Vh.t()[:, :r].contiguous()

        q2 = min(RcatT.shape[1], r + 16)
        try:
            _, _, V2 = torch.svd_lowrank(RcatT, q=q2, niter=2)
            DL = V2[:, :r].contiguous()
        except Exception:
            _, _, Vh2 = torch.linalg.svd(RcatT, full_matrices=False)
            DL = Vh2.t()[:, :r].contiguous()

        # per-expert coefficients
        if cfg.RES_COEF_MODE == "diag":
            gam = []
            for Rm in R_list:
                D = torch.diagonal(DL.t() @ Rm @ DR, 0).contiguous()  # (r,)
                gam.append(D)
            coef = torch.stack(gam, dim=0).contiguous()  # (Ecl,r)
        else:
            As = []
            for Rm in R_list:
                A = (DL.t() @ Rm @ DR).contiguous()  # (r,r)
                As.append(A)
            coef = torch.stack(As, dim=0).contiguous()  # (Ecl,r,r)

        DLq = quantize_int8(DL.to(DTYPE_W).contiguous())
        DRq = quantize_int8(DR.to(DTYPE_W).contiguous())

    # sparse residual blocks after low-rank
    blocks_list: List[List[Tuple[int,int,Any]]] = []
    for j, Rm in enumerate(R_list):
        R2 = Rm
        if r > 0 and DLq is not None and DRq is not None and coef is not None:
            DL = dequantize(DLq, device=Rm.device, dtype=DTYPE_ACC)
            DR = dequantize(DRq, device=Rm.device, dtype=DTYPE_ACC)
            if cfg.RES_COEF_MODE == "diag":
                g = coef[j].to(DTYPE_ACC).to(Rm.device)
                R2 = R2 - (DL * g.view(1, -1)) @ DR.t()
            else:
                A = coef[j].to(DTYPE_ACC).to(Rm.device)
                R2 = R2 - (DL @ A) @ DR.t()
            R2 = R2.contiguous()

        blks = topk_blocks(R2, cfg.RES_BLOCKS, cfg.RES_BSIZE)
        out_blks = []
        for (i0, j0, Bb) in blks:
            out_blks.append((i0, j0, quantize_int8(Bb.to(DTYPE_W).contiguous())))
        blocks_list.append(out_blks)

    out = {
        "n": n,
        "idx": idx,
        "basis": basis,            # either {"U":..., "V":...} or {"Ub":..., "Vb":...}
        "core": core,
        "core_blocks": core_blocks,
        "DL": DLq, "DR": DRq,
        "coef": coef,
        "blocks": blocks_list,
    }
    return out

@torch.no_grad()
def build_payload_for_cluster_dense(Ws: torch.Tensor, idx: List[int], U: torch.Tensor, V: torch.Tensor) -> dict:
    X_list = [(U.t() @ Ws[e] @ V).contiguous() for e in idx]
    basis = {
        "type": "dense",
        "U": quantize_int8(U.to(DTYPE_W).contiguous()),
        "V": quantize_int8(V.to(DTYPE_W).contiguous()),
    }
    return build_payload_from_X_list(X_list, idx, basis=basis, n=U.shape[0])

@torch.no_grad()
def build_payload_for_cluster_hadamard(Ws: torch.Tensor, idx: List[int], Ub: ImplicitHadamardPerm, Vb: ImplicitHadamardPerm) -> dict:
    X_list = [Ub.project_matrix(Ws[e], Vb).contiguous() for e in idx]
    basis = {"type": "hadamard_perm", "Ub": Ub, "Vb": Vb}
    return build_payload_from_X_list(X_list, idx, basis=basis, n=Ub.n)

# ----------------------------
# Runtime
# ----------------------------
class KTXRuntime(nn.Module):
    def __init__(self, payloads: List[Optional[dict]], basis_of_e: Dict[int, Tuple[int,int]], n: int):
        super().__init__()
        self.payloads = payloads
        self.map = basis_of_e
        self.n = n

    @torch.no_grad()
    def forward(self, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
        y = torch.zeros_like(x)
        used: Dict[int, List[Tuple[float,int]]] = {}
        for a, e in zip(gates.tolist(), routed):
            if e in self.map:
                m, j = self.map[e]
                used.setdefault(m, []).append((a, j))

        for m, items in used.items():
            P = self.payloads[m]
            if P is None:
                continue

            basis = P["basis"]
            if basis["type"] == "hadamard_perm":
                Ub: ImplicitHadamardPerm = basis["Ub"]
                Vb: ImplicitHadamardPerm = basis["Vb"]
                z = Ub.apply(x)
            else:
                U = dequantize(basis["U"], device=x.device, dtype=DTYPE_ACC)
                V = dequantize(basis["V"], device=x.device, dtype=DTYPE_ACC)
                z = x @ U

            u_acc = torch.zeros_like(z)

            DLq = P.get("DL", None)
            DRq = P.get("DR", None)
            coef = P.get("coef", None)
            has_lr = (DLq is not None) and (DRq is not None) and (coef is not None)

            if has_lr:
                DL = dequantize(DLq, device=x.device, dtype=DTYPE_ACC)
                DR = dequantize(DRq, device=x.device, dtype=DTYPE_ACC)
            else:
                DL = DR = None

            for (a, j) in items:
                u = torch.zeros_like(z)

                # core blocks
                for (i0, j0, Bq) in P["core_blocks"][j]:
                    Bc = dequantize(Bq, device=x.device, dtype=DTYPE_ACC)
                    h, w = Bc.shape
                    u[:, j0:j0+w] += z[:, i0:i0+h] @ Bc

                # low-rank residual
                if has_lr and DL is not None and DR is not None:
                    t = (z @ DL)  # (B,r)
                    if cfg.RES_COEF_MODE == "diag":
                        g = coef[j].to(DTYPE_ACC).to(x.device)
                        t = t * g.view(1, -1)
                    else:
                        A = coef[j].to(DTYPE_ACC).to(x.device)
                        t = t @ A
                    u += t @ DR.t()

                # sparse residual blocks
                for (i0, j0, Bbq) in P["blocks"][j]:
                    Bb = dequantize(Bbq, device=x.device, dtype=DTYPE_ACC)
                    h, w = Bb.shape
                    u[:, j0:j0+w] += z[:, i0:i0+h] @ Bb

                u_acc += float(a) * u

            if basis["type"] == "hadamard_perm":
                y += Vb.apply_T(u_acc)
            else:
                V = dequantize(basis["V"], device=x.device, dtype=DTYPE_ACC)
                y += u_acc @ V.t()

        return y

# ----------------------------
# Reconstruction (per-expert error)
# ----------------------------
@torch.no_grad()
def reconstruct_W_for_expert(payload: dict, j: int) -> torch.Tensor:
    n = int(payload["n"])
    X = torch.zeros(n, n, dtype=DTYPE_ACC, device=DEVICE)

    for (i0, j0, Bq) in payload["core_blocks"][j]:
        B = dequantize(Bq, device=DEVICE, dtype=DTYPE_ACC)
        h, w = B.shape
        X[i0:i0+h, j0:j0+w] = B

    DLq = payload.get("DL", None)
    DRq = payload.get("DR", None)
    coef = payload.get("coef", None)
    if DLq is not None and DRq is not None and coef is not None:
        DL = dequantize(DLq, device=DEVICE, dtype=DTYPE_ACC)
        DR = dequantize(DRq, device=DEVICE, dtype=DTYPE_ACC)
        if cfg.RES_COEF_MODE == "diag":
            g = coef[j].to(DTYPE_ACC).to(DEVICE)
            X = X + (DL * g.view(1, -1)) @ DR.t()
        else:
            A = coef[j].to(DTYPE_ACC).to(DEVICE)
            X = X + (DL @ A) @ DR.t()

    for (i0, j0, Bbq) in payload["blocks"][j]:
        Bb = dequantize(Bbq, device=DEVICE, dtype=DTYPE_ACC)
        h, w = Bb.shape
        X[i0:i0+h, j0:j0+w] += Bb

    basis = payload["basis"]
    if basis["type"] == "hadamard_perm":
        Ub: ImplicitHadamardPerm = basis["Ub"]
        Vb: ImplicitHadamardPerm = basis["Vb"]
        W = Ub.reconstruct_matrix(X, Vb)
    else:
        U = dequantize(basis["U"], device=DEVICE, dtype=DTYPE_ACC)
        V = dequantize(basis["V"], device=DEVICE, dtype=DTYPE_ACC)
        W = (U @ X @ V.t()).contiguous()
    return W

# ----------------------------
# Eval
# ----------------------------
@torch.no_grad()
def frob(A: torch.Tensor) -> torch.Tensor:
    return torch.linalg.norm(A, ord="fro")

@torch.no_grad()
def dense_apply(Ws: torch.Tensor, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
    n = Ws.shape[-1]
    Wsum = torch.zeros(n, n, dtype=DTYPE_ACC, device=x.device)
    for a, e in zip(gates, routed):
        Wsum += float(a.item()) * Ws[e]
    return x @ Wsum

@torch.no_grad()
def pick_routes_from_router(P: np.ndarray, eids: List[int], trial: int, k: int) -> Tuple[List[int], torch.Tensor]:
    E = len(eids)
    nrows = P.shape[0]
    r = (SEED + 777 + trial) % max(1, nrows)
    p = P[r].astype(np.float32, copy=False)

    if cfg.ROUTER_EIDS_ARE_GLOBAL:
        probs = np.array([p[eid] if eid < p.shape[0] else 0.0 for eid in eids], dtype=np.float32)
    else:
        probs = p[:E].copy()

    probs = np.maximum(probs, 0.0)
    if probs.sum() <= 0:
        routed = random.sample(range(E), k=min(k, E))
        g = torch.rand(len(routed), dtype=DTYPE_ACC, device=DEVICE)
        g = g / g.sum().clamp_min(1e-12)
        return routed, g

    top = np.argsort(-probs)[:min(k, E)]
    w = probs[top]
    w = w / max(w.sum(), 1e-12)
    return top.tolist(), torch.from_numpy(w).to(DTYPE_ACC).to(DEVICE)

@torch.no_grad()
def eval_runtime(rt: KTXRuntime, Ws: torch.Tensor, eids: List[int], trials: int, routed_k: int, batch: int):
    P = load_router_P() if cfg.ROUTER_PATH else None
    errs = []
    for t in range(trials):
        x = torch.randn(batch, rt.n, dtype=DTYPE_ACC, device=DEVICE)
        if P is not None:
            routed, gates = pick_routes_from_router(P, eids, t, routed_k)
        else:
            routed = random.sample(range(Ws.shape[0]), k=min(routed_k, Ws.shape[0]))
            gates = torch.rand(len(routed), dtype=DTYPE_ACC, device=DEVICE)
            gates = gates / gates.sum().clamp_min(1e-12)

        y_hat = rt(x, routed, gates)
        y_ref = dense_apply(Ws, x, routed, gates)
        err = float((frob(y_hat - y_ref) / (frob(y_ref) + 1e-12)).item())
        errs.append(err)

    log(f"[eval] routed forward rel-error = {float(np.mean(errs)):.6f} ± {float(np.std(errs)):.6f}  (trials={trials})")

@torch.no_grad()
def eval_per_expert(payloads: List[Optional[dict]], basis_of_e: Dict[int, Tuple[int,int]], Ws: torch.Tensor):
    errs = []
    for e in range(Ws.shape[0]):
        m, j = basis_of_e[e]
        P = payloads[m]
        if P is None:
            continue
        What = reconstruct_W_for_expert(P, j)
        err = float((frob(What - Ws[e]) / (frob(Ws[e]) + 1e-12)).item())
        errs.append(err)

    if errs:
        mean_err = float(np.mean(errs))
        p95_err  = float(np.percentile(errs, 95))
        max_err  = float(np.max(errs))
        log(f"[eval] per-expert W rel-error  mean={mean_err:.6f}  p95={p95_err:.6f}  max={max_err:.6f}")
    else:
        log("[eval] per-expert W rel-error  (no experts?)")

# ----------------------------
# Rough payload size estimate
# ----------------------------
def sizeof_payload(payloads: List[Optional[dict]]) -> float:
    def tensor_bytes(t: Any) -> int:
        if t is None:
            return 0
        if isinstance(t, QTensor):
            bq = t.q.numel() * t.q.element_size()
            bs = t.scale.numel() * t.scale.element_size()
            return bq + bs
        if torch.is_tensor(t):
            return t.numel() * t.element_size()
        return 0

    total = 0
    for P in payloads:
        if P is None:
            continue
        basis = P["basis"]
        if basis["type"] == "dense":
            total += tensor_bytes(basis["U"]) + tensor_bytes(basis["V"])
        else:
            Ub: ImplicitHadamardPerm = basis["Ub"]
            Vb: ImplicitHadamardPerm = basis["Vb"]
            total += Ub.perm.numel() * 4 + Ub.inv_perm.numel() * 4 + Ub.sign.numel() * 1
            total += Vb.perm.numel() * 4 + Vb.inv_perm.numel() * 4 + Vb.sign.numel() * 1

        for perE in P["core_blocks"]:
            for (_, _, Bq) in perE:
                total += tensor_bytes(Bq)
        for perE in P["blocks"]:
            for (_, _, Bq) in perE:
                total += tensor_bytes(Bq)

        total += tensor_bytes(P.get("DL")) + tensor_bytes(P.get("DR"))
        coef = P.get("coef", None)
        if torch.is_tensor(coef):
            total += coef.numel() * coef.element_size()

    return total / 1e6

# ----------------------------
# Main
# ----------------------------
def banner():
    log("== DeepSeek KT++-X Optimized (offline) v7 ==")
    log(f"Time:        {now()}")
    log(f"MODEL_DIR:   {cfg.MODEL_DIR}")
    log(f"OUTPUT_DIR:  {cfg.OUTPUT_DIR}")
    log(f"LAYER:       {cfg.LAYER}")
    log(f"MAX_EXPERTS:  {cfg.MAX_EXPERTS}")
    log(f"CALIB_PATH:  {cfg.CALIB_PATH or '(none)'}  CALIB_SAMPLES={cfg.CALIB_SAMPLES}  STRICT_CALIB={cfg.STRICT_CALIB}  ALLOW_RANDOM_CALIB={cfg.ALLOW_RANDOM_CALIB}")
    log(f"CAPTURE_CALIB={cfg.CAPTURE_CALIB}  (optional transformers capture)")
    log(f"ROUTER_PATH: {cfg.ROUTER_PATH or '(none)'}  RIDGE_WEIGHTED={cfg.RIDGE_WEIGHTED}")
    log(f"LIN_MODE:    {cfg.LIN_MODE}  (RIDGE_DAMP={cfg.RIDGE_DAMP})  AUTO_FALLBACK_LIN={cfg.AUTO_FALLBACK_LIN}")
    log(f"BASIS_MODE:  {cfg.BASIS_MODE}")
    log(f"CORE_MODE:   {cfg.CORE_MODE}  agg={cfg.CORE_AGG} block={cfg.CORE_BLOCK} target={cfg.CORE_TARGET} max_blocks={cfg.CORE_MAX_BLOCKS}")
    log(f"RESIDUAL:    rank={cfg.RES_RANK}  coef={cfg.RES_COEF_MODE}  blocks={cfg.RES_BLOCKS}  bsize={cfg.RES_BSIZE}")
    log(f"QUANT:       QMODE={cfg.QMODE}")
    log(f"TRAIN:       steps={cfg.TRAIN_STEPS} (dense_train only) lr={cfg.LR_UV} subm={cfg.SUBM} batchE={cfg.BATCH_E}")
    log(f"CLUSTER:     M={cfg.M_CLUSTERS or '(auto)'} iters={cfg.CLUSTER_ITERS} restarts={cfg.CLUSTER_RESTARTS} feat_d={cfg.CLUSTER_FEAT_D} target_size={cfg.CLUSTER_TARGET_SIZE} min_size={cfg.CLUSTER_MIN_SIZE}")
    log(f"Hadamard(post-W): USE_HADAMARD={cfg.USE_HADAMARD} seed={cfg.HAD_SEED}")
    log(f"Threads:     {NTHREADS}  Torch={torch.__version__} device={DEVICE}")
    log(f"Cache:       {cfg.CACHE_MODE}  path={cache_path()}  force_rebuild={cfg.FORCE_REBUILD_WS}")
    log("")

def main():
    banner()

    eids, Ws = load_or_build_Ws()
    E, n, _ = Ws.shape
    log(f"[Ws] shape={tuple(Ws.shape)} experts={eids[:8]}{'...' if len(eids)>8 else ''}")

    # Choose cluster count
    if cfg.M_CLUSTERS > 0:
        M = min(cfg.M_CLUSTERS, E)
    else:
        M = max(1, min(E, int(round(E / max(1, cfg.CLUSTER_TARGET_SIZE)))))
    log(f"[cluster] M={M}")

    Xfeat = expert_features(Ws, d=cfg.CLUSTER_FEAT_D)
    labels, centers = kmeans_torch(Xfeat, k=M, iters=cfg.CLUSTER_ITERS, restarts=cfg.CLUSTER_RESTARTS)
    labels = enforce_min_cluster_size(labels, centers, Xfeat, min_size=cfg.CLUSTER_MIN_SIZE)

    clusters = [torch.nonzero(labels == m, as_tuple=False).flatten().tolist() for m in range(M)]
    log(f"[cluster] sizes: {[len(c) for c in clusters]}")

    payloads: List[Optional[dict]] = []
    basis_of_e: Dict[int, Tuple[int,int]] = {}

    log("[build] payloads ...")

    if cfg.BASIS_MODE == "dense_train":
        # Init U,V per cluster via SVD(mean W)
        U_par: List[Optional[DenseBasis]] = []
        V_par: List[Optional[DenseBasis]] = []
        for m, idx in enumerate(clusters):
            if len(idx) == 0:
                U_par.append(None); V_par.append(None)
                continue
            Wm = Ws[idx].mean(dim=0)
            U0, V0 = svd_init(Wm)
            U_par.append(DenseBasis(U0))
            V_par.append(DenseBasis(V0))

        # Optional training (off by default). Best used ONLY with real CALIB_PATH.
        if cfg.TRAIN_STEPS > 0:
            params = [uv.M for uv in (U_par + V_par) if uv is not None]
            opt = torch.optim.AdamW(params, lr=cfg.LR_UV, weight_decay=0.0)

            core_block_for_train = cfg.CORE_BLOCK if cfg.TRAIN_BLOCK_USE_CORE_BLOCK else max(16, cfg.SUBM // 4)

            def block_group_sparsity_penalty(Xs: torch.Tensor, block: int) -> torch.Tensor:
                Eb, s, _ = Xs.shape
                b = int(block)
                nb = s // b
                if b <= 0 or nb <= 0:
                    return torch.zeros((), dtype=DTYPE_ACC, device=Xs.device)
                s2 = nb * b
                X = Xs[:, :s2, :s2].contiguous()
                Xb = X.view(Eb, nb, b, nb, b).permute(0, 1, 3, 2, 4).contiguous()
                Eblk = (Xb * Xb).sum(dim=(3,4))
                Pm = Eblk.mean(dim=0)
                return torch.sqrt(Pm + 1e-12).sum() / (Pm.sum() + 1e-12)

            def slice_X_batch(Ws_batch: torch.Tensor, U: torch.Tensor, V: torch.Tensor, S: torch.Tensor) -> torch.Tensor:
                U_S = U[:, S]
                V_S = V[:, S]
                T = Ws_batch @ V_S
                return torch.matmul(U_S.t().unsqueeze(0), T)

            t0 = time.perf_counter()
            with torch.enable_grad():
                for step in range(1, cfg.TRAIN_STEPS + 1):
                    S = torch.randperm(n, device=DEVICE)[:min(cfg.SUBM, n)]
                    L_total = None
                    n_terms = 0

                    for m, idx in enumerate(clusters):
                        if len(idx) < cfg.TRAIN_MIN_CLUSTER:
                            continue
                        Uo = U_par[m].orthogonal()
                        Vo = V_par[m].orthogonal()

                        if 0 < cfg.BATCH_E < len(idx):
                            pick = torch.randperm(len(idx), device=DEVICE)[:cfg.BATCH_E].tolist()
                            idx_step = [idx[p] for p in pick]
                        else:
                            idx_step = idx

                        Ws_batch = Ws[idx_step]
                        Xs = slice_X_batch(Ws_batch, Uo, Vo, S)  # (Eb,s,s)

                        diag = torch.diagonal(Xs, dim1=1, dim2=2)
                        diagE = (diag * diag).mean().clamp_min(1e-12)
                        off = Xs - torch.diag_embed(diag)
                        offE = (off * off).mean()

                        if cfg.TRAIN_OBJ == "logratio":
                            loss = torch.log(offE + 1e-12) - torch.log(diagE + 1e-12)
                        else:
                            loss = offE / diagE

                        if cfg.TRAIN_LAM_BLOCK > 0 and cfg.CORE_MODE.startswith("block"):
                            loss = loss + cfg.TRAIN_LAM_BLOCK * block_group_sparsity_penalty(Xs, core_block_for_train)

                        L_total = loss if (L_total is None) else (L_total + loss)
                        n_terms += 1

                    if L_total is None:
                        log("[train] skipped (no clusters >= TRAIN_MIN_CLUSTER)")
                        break

                    L_total = L_total / max(1, n_terms)
                    opt.zero_grad(set_to_none=True)
                    L_total.backward()
                    if cfg.GRAD_CLIP > 0:
                        torch.nn.utils.clip_grad_norm_(params, max_norm=cfg.GRAD_CLIP)
                    opt.step()

                    if (step % cfg.REORTHO_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                        with torch.no_grad():
                            for uv in (U_par + V_par):
                                if uv is not None:
                                    uv.M.copy_(uv.orthogonal())

                    if step == 1 or (step % cfg.REPORT_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                        t1 = time.perf_counter()
                        log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} | loss={float(L_total.item()):.6f}  (+{t1-t0:.1f}s)")
                        t0 = t1

        for m, idx in enumerate(clusters):
            if len(idx) == 0:
                payloads.append(None)
                continue
            Uo = U_par[m].orthogonal().detach()
            Vo = V_par[m].orthogonal().detach()
            Pld = build_payload_for_cluster_dense(Ws, idx, Uo, Vo)
            payloads.append(Pld)
            for j, e in enumerate(idx):
                basis_of_e[e] = (m, j)

            efr = Pld["core"]["energy_fracs"]
            ef_mean = float(np.mean(efr)) if efr else 0.0
            ef_min  = float(np.min(efr)) if efr else 0.0
            nb_mean = float(np.mean([len(x) for x in Pld["core_blocks"]])) if Pld["core_blocks"] else 0.0
            log(f"  - basis {m}: E={len(idx)} core={Pld['core']['mode']} core_energy(mean/min)≈{ef_mean:.3f}/{ef_min:.3f} core_blocks(mean)≈{nb_mean:.1f}")

    elif cfg.BASIS_MODE == "hadamard_perm":
        if not is_power_of_two(n):
            raise RuntimeError(f"BASIS_MODE=hadamard_perm requires H power-of-two; got H={n}")
        for m, idx in enumerate(clusters):
            if len(idx) == 0:
                payloads.append(None)
                continue
            Ub = ImplicitHadamardPerm(n, seed=SEED + 10000 + 13*m)
            Vb = ImplicitHadamardPerm(n, seed=SEED + 20000 + 17*m)
            Pld = build_payload_for_cluster_hadamard(Ws, idx, Ub, Vb)
            payloads.append(Pld)
            for j, e in enumerate(idx):
                basis_of_e[e] = (m, j)

            efr = Pld["core"]["energy_fracs"]
            ef_mean = float(np.mean(efr)) if efr else 0.0
            ef_min  = float(np.min(efr)) if efr else 0.0
            nb_mean = float(np.mean([len(x) for x in Pld["core_blocks"]])) if Pld["core_blocks"] else 0.0
            log(f"  - basis {m}: E={len(idx)} core={Pld['core']['mode']} core_energy(mean/min)≈{ef_mean:.3f}/{ef_min:.3f} core_blocks(mean)≈{nb_mean:.1f}")

    else:
        raise ValueError("BASIS_MODE must be dense_train or hadamard_perm")

    rt = KTXRuntime(payloads, basis_of_e, n=n)

    mb = sizeof_payload(payloads)
    log(f"[size] approx payload bytes ≈ {mb:.2f} MB  (dense_train stores full U/V -> huge; hadamard_perm avoids that)")

    if cfg.EVAL_PER_EXPERT:
        eval_per_expert(payloads, basis_of_e, Ws)

    eval_runtime(rt, Ws, eids, trials=cfg.EVAL_TRIALS, routed_k=cfg.ROUTED_K, batch=cfg.EVAL_BATCH)

    log("\n✅ Done.")
    log("Practical next steps for real accuracy:")
    log("  1) Provide real CALIB_PATH (true layer inputs) and (ideally) ROUTER_PATH, then set RIDGE_WEIGHTED=1.")
    log("  2) For real compression (payload size), use BASIS_MODE=hadamard_perm (implicit basis).")
    log("  3) To push accuracy: CORE_TARGET↑, CORE_MAX_BLOCKS↑, RES_RANK↑, RES_BLOCKS↑.")

# Run
main()


== DeepSeek KT++-X Optimized (offline) v7 ==
Time:        2026-01-13 10:25:59
MODEL_DIR:   /home/daniyar/deepseek-model
OUTPUT_DIR:  /home/daniyar/moe_ws_outputs
LAYER:       1
MAX_EXPERTS:  16
CALIB_PATH:  (none)  CALIB_SAMPLES=2048  STRICT_CALIB=False  ALLOW_RANDOM_CALIB=True
CAPTURE_CALIB=False  (optional transformers capture)
ROUTER_PATH: (none)  RIDGE_WEIGHTED=False
LIN_MODE:    ridge  (RIDGE_DAMP=0.001)  AUTO_FALLBACK_LIN=True
BASIS_MODE:  dense_train
CORE_MODE:   blocktopk_perexpert  agg=mean block=64 target=0.9 max_blocks=512
RESIDUAL:    rank=64  coef=full  blocks=128  bsize=64
QUANT:       QMODE=none
TRAIN:       steps=0 (dense_train only) lr=0.01 subm=256 batchE=4
CLUSTER:     M=(auto) iters=80 restarts=4 feat_d=64 target_size=4 min_size=2
Hadamard(post-W): USE_HADAMARD=False seed=1234
Threads:     8  Torch=2.4.1+cpu device=cpu
Cache:       readwrite  path=/home/daniyar/moe_ws_outputs/deepseek_layer1_E16_ridge_Ws_cache_v7.npz  force_rebuild=False

[cache] loaded Ws: /home/da

In [7]:
#!/usr/bin/env python3
# ============================================================
# DeepSeek OFFLINE Bridge v2 — Capture real Calib X (and try Router P)
# Fixes your crash: DynamicCache has no get_usable_length
# by forcing use_cache=False + adding a small compat shim.
#
# Outputs:
#   OUTPUT_DIR/calib_layer{L}_X.npz   (key 'X': [N,H])
#   OUTPUT_DIR/router_layer{L}_P.npz  (key 'P': [N,E_total]) if captured
#   OUTPUT_DIR/kt_env_layer{L}.sh     (source this before running KT++)
#
# Env (optional):
#   MODEL_DIR=/home/daniyar/deepseek-model
#   OUTPUT_DIR=/home/daniyar/moe_ws_outputs
#   LAYER=1
#   MAX_EXPERTS=16
#   CALIB_SAMPLES=1024
#   CAPTURE_MAX_TOKENS=256
#   CAPTURE_ITERS=32
#   CAPTURE_TEXT="Hello world."
#   CAPTURE_TEXT_PATH=/path/to/prompts.txt
#   CAPTURE_ROUTER=1
#   TRUST_REMOTE_CODE=1
#   LOCAL_FILES_ONLY=1
#   AUTO_PIP=1
#   CAPTURE_DTYPE=float16
# ============================================================

import os, re, json, math, time, random, sys, subprocess
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import torch
import torch.nn as nn

try:
    from safetensors import safe_open
except Exception as e:
    raise RuntimeError("Missing safetensors. Install: pip install safetensors") from e

try:
    from tqdm import tqdm
except Exception:
    def tqdm(x, **kwargs):  # type: ignore
        return x

# ----------------------------
# Determinism / threads
# ----------------------------
NTHREADS = int(os.environ.get("KTXX_THREADS", "8"))
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(NTHREADS))
try:
    torch.set_num_threads(NTHREADS)
except Exception:
    pass

SEED = int(os.environ.get("SEED", "1234"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(os.environ.get("DEVICE", "cpu"))

def now():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def log(msg: str):
    print(msg, flush=True)

# ----------------------------
# Config
# ----------------------------
@dataclass
class Cfg:
    MODEL_DIR: str = os.environ.get("MODEL_DIR", "/home/daniyar/deepseek-model")
    OUTPUT_DIR: str = os.environ.get("OUTPUT_DIR", "/home/daniyar/moe_ws_outputs")
    LAYER: int = int(os.environ.get("LAYER", "1"))
    MAX_EXPERTS: int = int(os.environ.get("MAX_EXPERTS", "16"))

    CALIB_SAMPLES: int = int(os.environ.get("CALIB_SAMPLES", "1024"))
    CAPTURE_MAX_TOKENS: int = int(os.environ.get("CAPTURE_MAX_TOKENS", "256"))
    CAPTURE_ITERS: int = int(os.environ.get("CAPTURE_ITERS", "32"))

    CAPTURE_TEXT: str = os.environ.get("CAPTURE_TEXT", "Hello world.\n").strip() + "\n"
    CAPTURE_TEXT_PATH: str = os.environ.get("CAPTURE_TEXT_PATH", "").strip()

    CAPTURE_ROUTER: bool = os.environ.get("CAPTURE_ROUTER", "1") == "1"

    TRUST_REMOTE_CODE: bool = os.environ.get("TRUST_REMOTE_CODE", "1") == "1"
    LOCAL_FILES_ONLY: bool = os.environ.get("LOCAL_FILES_ONLY", "1") == "1"
    AUTO_PIP: bool = os.environ.get("AUTO_PIP", "1") == "1"
    CAPTURE_DTYPE: str = os.environ.get("CAPTURE_DTYPE", "float16").strip().lower()  # float16|bfloat16|float32

cfg = Cfg()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# ----------------------------
# Index / shard helpers (to infer expert count and hidden size)
# ----------------------------
def read_index(model_dir: str) -> Dict[str, str]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    wm = obj.get("weight_map", {})
    if not wm:
        raise RuntimeError("Index JSON has empty weight_map.")
    return wm

def find_layer_expert_ids(weight_map: Dict[str, str], layer: int) -> List[int]:
    pat = re.compile(rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.")
    ids = set()
    for k in weight_map.keys():
        m = pat.match(k)
        if m:
            ids.add(int(m.group(1)))
    return sorted(ids)

def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    prefix = f"model.layers.{layer}.mlp.experts.{eid}."
    def pick(cands: List[str]) -> Optional[str]:
        for suf in cands:
            k = prefix + suf
            if k in weight_map:
                return k
        return None
    up   = pick(["up_proj.weight", "w3.weight", "w1.weight"])
    gate = pick(["gate_proj.weight", "w1.weight", "w3.weight"])
    down = pick(["down_proj.weight", "w2.weight"])
    if up is None or gate is None or down is None:
        return {}
    if up == gate:
        g2 = pick(["gate_proj.weight"])
        if g2:
            gate = g2
    return {"up": up, "gate": gate, "down": down}

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], keys: List[str]) -> Dict[str, torch.Tensor]:
    by_shard: Dict[str, List[str]] = {}
    for k in keys:
        shard = weight_map.get(k, None)
        if shard is None:
            raise KeyError(f"Key not in weight_map: {k}")
        by_shard.setdefault(shard, []).append(k)

    out: Dict[str, torch.Tensor] = {}
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(sp):
            raise FileNotFoundError(f"Missing shard: {sp}")
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks:
                out[k] = f.get_tensor(k)
    return out

# ----------------------------
# NPZ save
# ----------------------------
def save_npz(path: str, **arrays: Any):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez(path, **arrays)

# ----------------------------
# Transformers import + optional pip
# ----------------------------
def _pip_install(pkgs: List[str]):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU"] + pkgs)

def _import_transformers():
    try:
        from transformers import AutoTokenizer, AutoModelForCausalLM
        return AutoTokenizer, AutoModelForCausalLM
    except Exception:
        if not cfg.AUTO_PIP:
            raise
        log("[pip] Installing transformers + sentencepiece (local env) ...")
        _pip_install(["transformers>=4.41.0", "sentencepiece>=0.2.0"])
        from transformers import AutoTokenizer, AutoModelForCausalLM
        return AutoTokenizer, AutoModelForCausalLM

def _dtype_from_str(s: str) -> torch.dtype:
    if s == "float16":
        return torch.float16
    if s == "bfloat16":
        return torch.bfloat16
    if s == "float32":
        return torch.float32
    return torch.float16

# ----------------------------
# Cache compat shim (fixes your exact crash)
# ----------------------------
def _patch_dynamic_cache():
    try:
        import transformers
        # Newer transformers has cache_utils.DynamicCache. Some versions lack get_usable_length.
        from transformers.cache_utils import DynamicCache  # type: ignore
        if not hasattr(DynamicCache, "get_usable_length"):
            def _get_usable_length(self, seq_length: int):
                # Best-effort: return current cached seq length if available, else 0.
                if hasattr(self, "get_seq_length"):
                    try:
                        return int(self.get_seq_length())  # some versions
                    except TypeError:
                        try:
                            return int(self.get_seq_length(0))  # other versions
                        except Exception:
                            return 0
                if hasattr(self, "seqlen"):
                    try:
                        return int(self.seqlen)
                    except Exception:
                        return 0
                return 0
            DynamicCache.get_usable_length = _get_usable_length  # type: ignore
            log("[patch] Added DynamicCache.get_usable_length compat shim.")
    except Exception:
        # If cache_utils not present, ignore.
        pass

# ----------------------------
# Model structure helpers
# ----------------------------
def _find_layers_list(model: Any) -> Optional[List[Any]]:
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        try: return list(model.model.layers)
        except Exception: pass
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        try: return list(model.transformer.h)
        except Exception: pass
    if hasattr(model, "layers"):
        try: return list(model.layers)
        except Exception: pass
    if hasattr(model, "decoder") and hasattr(model.decoder, "layers"):
        try: return list(model.decoder.layers)
        except Exception: pass
    return None

def _guess_mlp(layer: Any) -> Optional[nn.Module]:
    for name in ["mlp", "feed_forward", "ffn", "moe", "MoE", "Mlp"]:
        if hasattr(layer, name):
            m = getattr(layer, name)
            if isinstance(m, nn.Module):
                return m
    best = None
    best_score = -1e9
    for n, m in layer.named_modules():
        if not isinstance(m, nn.Module):
            continue
        ln = n.lower()
        score = 0.0
        if "moe" in ln: score += 4.0
        if "mlp" in ln: score += 3.0
        if "router" in ln or "gate" in ln: score += 2.0
        if "expert" in ln: score += 1.0
        if score > best_score:
            best_score = score
            best = m
    return best

def _router_candidates(layer: nn.Module, H: int, E_total: int) -> List[Tuple[str, nn.Linear]]:
    """
    Find likely router Linear modules:
      - in_features == H
      - out_features ~ E_total (prefer exact / close)
      - exclude experts.*.gate_proj etc by filtering names containing 'experts.<digit>'
    """
    cands: List[Tuple[str, nn.Linear]] = []
    bad_re = re.compile(r"experts\.\d+")
    for name, mod in layer.named_modules():
        if not isinstance(mod, nn.Linear):
            continue
        if getattr(mod, "in_features", None) != H:
            continue
        out_f = int(getattr(mod, "out_features", 0))
        # filter out expert gate projections (1408 etc) by limiting size
        if out_f < E_total:
            continue
        if out_f > max(E_total * 8, E_total + 64):  # for E_total=64, cap at 512
            continue
        if bad_re.search(name):
            continue
        cands.append((name, mod))

    def score(item: Tuple[str, nn.Linear]) -> float:
        name, mod = item
        out_f = int(mod.out_features)
        s = 0.0
        ln = name.lower()
        if "router" in ln: s += 10.0
        if "gate" in ln: s += 5.0
        # prefer exact E_total
        s += 20.0 - 0.2 * abs(out_f - E_total)
        # prefer smaller headroom
        s -= 0.05 * (out_f - E_total)
        return s

    cands.sort(key=score, reverse=True)
    return cands

# ----------------------------
# Capture engine
# ----------------------------
class Collector:
    def __init__(self, H: int, E_total: int, want_router: bool, N: int):
        self.H = H
        self.E_total = E_total
        self.want_router = want_router
        self.N = N
        self.X = torch.empty((N, H), dtype=torch.float32, device="cpu")
        self.P = torch.empty((N, E_total), dtype=torch.float32, device="cpu") if want_router else None
        self.nX = 0
        self.nP = 0

    def add_X(self, hs: torch.Tensor):
        if self.nX >= self.N:
            return
        hs = hs.detach()
        if hs.dtype != torch.float32:
            hs = hs.to(torch.float32)
        hs = hs.reshape(-1, hs.shape[-1]).cpu()
        if hs.shape[1] != self.H:
            return
        take = min(self.N - self.nX, hs.shape[0])
        if take > 0:
            self.X[self.nX:self.nX+take].copy_(hs[:take])
            self.nX += take

    def add_router_logits(self, logits: torch.Tensor):
        if (not self.want_router) or (self.P is None) or (self.nP >= self.N):
            return
        out = logits.detach()
        if out.dtype != torch.float32:
            out = out.to(torch.float32)
        out = out.reshape(-1, out.shape[-1]).cpu()
        if out.shape[1] < self.E_total:
            return
        probs = torch.softmax(out[:, :self.E_total], dim=-1)
        take = min(self.N - self.nP, probs.shape[0])
        if take > 0:
            self.P[self.nP:self.nP+take].copy_(probs[:take])
            self.nP += take

def capture_XP(model_dir: str, layer_idx: int, H: int, E_total: int,
               out_x: str, out_p: str) -> Tuple[str, Optional[str]]:

    AutoTokenizer, AutoModelForCausalLM = _import_transformers()
    _patch_dynamic_cache()

    log("[capture] Loading tokenizer/model from local files...")
    tok = AutoTokenizer.from_pretrained(
        model_dir,
        use_fast=True,
        local_files_only=cfg.LOCAL_FILES_ONLY,
        trust_remote_code=cfg.TRUST_REMOTE_CODE,
    )
    if tok.pad_token is None and tok.eos_token is not None:
        tok.pad_token = tok.eos_token

    cap_dtype = _dtype_from_str(cfg.CAPTURE_DTYPE)

    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        torch_dtype=cap_dtype,
        low_cpu_mem_usage=True,
        device_map=None,
        local_files_only=cfg.LOCAL_FILES_ONLY,
        trust_remote_code=cfg.TRUST_REMOTE_CODE,
    )
    model.eval()
    model.to(DEVICE)

    # FORCE CACHE OFF (this prevents your crash)
    try:
        model.config.use_cache = False
    except Exception:
        pass
    try:
        if hasattr(model, "generation_config") and model.generation_config is not None:
            model.generation_config.use_cache = False
    except Exception:
        pass

    layers = _find_layers_list(model)
    if layers is None:
        raise RuntimeError("Could not locate transformer layers list.")
    if layer_idx < 0 or layer_idx >= len(layers):
        raise RuntimeError(f"LAYER={layer_idx} out of range. Model has {len(layers)} layers.")

    layer = layers[layer_idx]
    mlp = _guess_mlp(layer)
    if mlp is None:
        raise RuntimeError("Could not locate MLP/MoE module for the target layer.")

    coll = Collector(H=H, E_total=E_total, want_router=cfg.CAPTURE_ROUTER, N=cfg.CALIB_SAMPLES)

    # Capture X: MLP input
    h_mlp = mlp.register_forward_pre_hook(
        lambda _m, inp: coll.add_X(inp[0]) if (inp and torch.is_tensor(inp[0])) else None
    )

    # Try router: hook best candidates on the LAYER (not inside experts)
    router_name = None
    router_hooks = []
    if cfg.CAPTURE_ROUTER:
        cands = _router_candidates(layer, H=H, E_total=E_total)
        if cands:
            # hook a few best
            for nm, mod in cands[:6]:
                def _mk(nm_):
                    def _hook(_m, _inp, out):
                        o = out[0] if isinstance(out, (tuple, list)) else out
                        if torch.is_tensor(o):
                            coll.add_router_logits(o)
                    return _hook
                router_hooks.append((nm, mod.register_forward_hook(_mk(nm))))
            router_name = cands[0][0]
            log(f"[capture] Hooked router candidates: {len(router_hooks)} (best='{router_name}')")
        else:
            log("[capture] No router candidates found (layer-level). Will capture X only.")

    # prompts
    prompts: List[str] = []
    if cfg.CAPTURE_TEXT_PATH and os.path.isfile(cfg.CAPTURE_TEXT_PATH):
        with open(cfg.CAPTURE_TEXT_PATH, "r", encoding="utf-8") as f:
            prompts = [ln.strip() for ln in f if ln.strip()]
        log(f"[capture] Loaded {len(prompts)} prompts from CAPTURE_TEXT_PATH.")
    if not prompts:
        prompts = [cfg.CAPTURE_TEXT]

    # forward loop
    with torch.inference_mode():
        for it in range(cfg.CAPTURE_ITERS):
            if coll.nX >= cfg.CALIB_SAMPLES and (not cfg.CAPTURE_ROUTER or coll.nP >= cfg.CALIB_SAMPLES):
                break
            text = prompts[it % len(prompts)]
            enc = tok(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=cfg.CAPTURE_MAX_TOKENS,
                padding=False,
            )
            enc = {k: v.to(DEVICE) for k, v in enc.items()}

            # >>> KEY FIX: use_cache=False <<<
            try:
                _ = model(**enc, use_cache=False, return_dict=True)
            except TypeError:
                # some custom forwards may not accept return_dict in signature; retry minimal
                _ = model(**enc, use_cache=False)
            except Exception as e:
                # If anything weird happens, bail but still save what we collected.
                log(f"[capture] forward error at iter {it+1}: {type(e).__name__}: {e}")
                break

            if (it + 1) % 4 == 0 or it == 0:
                log(f"[capture] iter {it+1}/{cfg.CAPTURE_ITERS}  nX={coll.nX} nP={coll.nP}")

    # cleanup hooks
    h_mlp.remove()
    for nm, hh in router_hooks:
        try: hh.remove()
        except Exception: pass

    if coll.nX == 0:
        raise RuntimeError("Capture failed: collected 0 X rows.")

    # Save X
    X = coll.X[:coll.nX].numpy().astype(np.float32)
    save_npz(out_x, X=X)
    log(f"[capture] wrote X -> {out_x}  shape={X.shape}")

    # Save P if present
    p_written = None
    if cfg.CAPTURE_ROUTER and coll.P is not None and coll.nP > 0:
        N = min(coll.nX, coll.nP)
        X2 = coll.X[:N].numpy().astype(np.float32)
        P2 = coll.P[:N].numpy().astype(np.float32)
        save_npz(out_x, X=X2)  # align
        save_npz(out_p, P=P2)
        log(f"[capture] wrote P -> {out_p}  shape={P2.shape}  (router='{router_name or 'unknown'}')")
        p_written = out_p
    else:
        log("[capture] Router P not captured (router module may be custom/non-Linear, or not found).")

    return out_x, p_written

# ----------------------------
# Main
# ----------------------------
def banner():
    log("== DeepSeek OFFLINE Bridge v2: Capture X/P ==")
    log(f"Time:        {now()}")
    log(f"MODEL_DIR:   {cfg.MODEL_DIR}")
    log(f"OUTPUT_DIR:  {cfg.OUTPUT_DIR}")
    log(f"LAYER:       {cfg.LAYER}")
    log(f"MAX_EXPERTS:  {cfg.MAX_EXPERTS}")
    log(f"DEVICE:      {DEVICE}  Torch={torch.__version__} threads={NTHREADS}")
    log(f"CAPTURE:     ROUTER={cfg.CAPTURE_ROUTER}  CALIB_SAMPLES={cfg.CALIB_SAMPLES}")
    log(f"CAPTURE_CFG: max_tokens={cfg.CAPTURE_MAX_TOKENS} iters={cfg.CAPTURE_ITERS} dtype={cfg.CAPTURE_DTYPE}")
    log(f"HF:          trust_remote_code={cfg.TRUST_REMOTE_CODE} local_files_only={cfg.LOCAL_FILES_ONLY} auto_pip={cfg.AUTO_PIP}")
    log("")

def main():
    banner()

    wm = read_index(cfg.MODEL_DIR)
    all_eids = find_layer_expert_ids(wm, cfg.LAYER)
    if not all_eids:
        raise RuntimeError(f"No experts found for layer={cfg.LAYER}. Check MODEL_DIR and LAYER.")
    eids = all_eids[:cfg.MAX_EXPERTS]
    E_total = len(all_eids)

    kk0 = pick_expert_tensor_keys(wm, cfg.LAYER, eids[0])
    if not kk0:
        raise RuntimeError("Could not find expert tensor keys to infer H.")
    t0 = load_tensors_from_shards(cfg.MODEL_DIR, wm, [kk0["up"]])[kk0["up"]]
    H = int(t0.shape[1])

    out_x = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
    out_p = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
    out_env = os.path.join(cfg.OUTPUT_DIR, f"kt_env_layer{cfg.LAYER}.sh")

    log(f"[found] layer={cfg.LAYER} total_experts={E_total} using={len(eids)} H={H} eids={eids}")
    log("[calib] Capturing X (and maybe P) via transformers with use_cache=False ...")

    x_path, p_path = capture_XP(cfg.MODEL_DIR, cfg.LAYER, H, E_total, out_x, out_p)

    # Write env helper for KT++
    lines = []
    lines.append(f"export CALIB_PATH='{x_path}'")
    if p_path and os.path.isfile(p_path):
        lines.append(f"export ROUTER_PATH='{p_path}'")
        lines.append("export RIDGE_WEIGHTED=1")
    else:
        lines.append("# export ROUTER_PATH='...'  # not captured")
        lines.append("export RIDGE_WEIGHTED=0")
    lines.append("export FORCE_REBUILD_WS=1")
    lines.append(f"export LAYER='{cfg.LAYER}'")
    lines.append(f"export MAX_EXPERTS='{cfg.MAX_EXPERTS}'")
    txt = "\n".join(lines) + "\n"
    with open(out_env, "w", encoding="utf-8") as f:
        f.write(txt)

    log(f"[save] wrote env helper -> {out_env}")
    log("\n✅ Bridge complete. Next:")
    log(f"   source '{out_env}'")
    log("   # then run your KT++/KT++-X compression code")
    if not (p_path and os.path.isfile(p_path)):
        log("\nNote: Router P was not captured. That's OK for weff_gate / ridge_unweighted.")
        log("If you need RIDGE_WEIGHTED=1, router capture must succeed (may require model-specific router hook).")

if __name__ == "__main__":
    main()


== DeepSeek OFFLINE Bridge v2: Capture X/P ==
Time:        2026-01-13 10:12:14
MODEL_DIR:   /home/daniyar/deepseek-model
OUTPUT_DIR:  /home/daniyar/moe_ws_outputs
LAYER:       1
MAX_EXPERTS:  16
DEVICE:      cpu  Torch=2.4.1+cpu threads=8
CAPTURE:     ROUTER=True  CALIB_SAMPLES=1024
CAPTURE_CFG: max_tokens=256 iters=32 dtype=float16
HF:          trust_remote_code=True local_files_only=True auto_pip=True

[found] layer=1 total_experts=64 using=16 H=2048 eids=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
[calib] Capturing X (and maybe P) via transformers with use_cache=False ...
[patch] Added DynamicCache.get_usable_length compat shim.
[capture] Loading tokenizer/model from local files...


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

[capture] No router candidates found (layer-level). Will capture X only.
[capture] iter 1/32  nX=5 nP=0
[capture] iter 4/32  nX=20 nP=0
[capture] iter 8/32  nX=40 nP=0
[capture] iter 12/32  nX=60 nP=0
[capture] iter 16/32  nX=80 nP=0
[capture] iter 20/32  nX=100 nP=0
[capture] iter 24/32  nX=120 nP=0
[capture] iter 28/32  nX=140 nP=0
[capture] iter 32/32  nX=160 nP=0
[capture] wrote X -> /home/daniyar/moe_ws_outputs/calib_layer1_X.npz  shape=(160, 2048)
[capture] Router P not captured (router module may be custom/non-Linear, or not found).
[save] wrote env helper -> /home/daniyar/moe_ws_outputs/kt_env_layer1.sh

✅ Bridge complete. Next:
   source '/home/daniyar/moe_ws_outputs/kt_env_layer1.sh'
   # then run your KT++/KT++-X compression code

Note: Router P was not captured. That's OK for weff_gate / ridge_unweighted.
If you need RIDGE_WEIGHTED=1, router capture must succeed (may require model-specific router hook).


In [10]:
#!/usr/bin/env python3
# ============================================================
# DeepSeek KT++-X OFFLINE (single-cell) v8
#
# What to run next after Bridge v2:
#   - Uses CALIB_PATH=/.../calib_layer{L}_X.npz (key 'X')
#   - Builds expert square Ws (E,H,H) via ridge / weff_gate / weff
#   - Clusters experts
#   - Builds KT++ payload:
#       core block-top-K (per expert or shared) + residual low-rank + residual sparse blocks
#   - Basis modes:
#       BASIS_MODE=hadamard_perm (DEFAULT, tiny basis, good compression)
#       BASIS_MODE=svd           (explicit U,V per cluster)
#       BASIS_MODE=dense_train   (explicit U,V + optional training; BIG payload)
#   - Evaluates per-expert & routed-mixture operator error using random x.
#
# Outputs (in OUTPUT_DIR):
#   - Ws cache npz: Ws_cache_layer{L}_E{E}_{LIN_MODE}.npz
#   - Payload npz:  ktx_payload_layer{L}_E{E}_{BASIS_MODE}.npz
#
# Key env vars (optional):
#   MODEL_DIR, OUTPUT_DIR, LAYER, MAX_EXPERTS
#   CALIB_PATH, ROUTER_PATH, RIDGE_WEIGHTED (router optional; you don't have it)
#   LIN_MODE=ridge|weff_gate|weff
#   BASIS_MODE=hadamard_perm|svd|dense_train
#   CORE_MODE=blocktopk_perexpert|blocktopk|blockdiag|none
#   CORE_BLOCK=64 CORE_TARGET=0.90 CORE_MAX_BLOCKS=512
#   RES_RANK=64 RES_COEF=full|diag RES_BLOCKS=128 RES_BSIZE=64
#   M_CLUSTERS=0 TARGET_CLUSTER_SIZE=4 CLUSTER_RESTARTS=4 CLUSTER_ITERS=80
#   TRAIN_STEPS (only for dense_train)
# ============================================================

import os, re, json, math, time, random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from safetensors import safe_open
except Exception as e:
    raise RuntimeError("This script requires safetensors (pip install safetensors).") from e

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):  # type: ignore
        return x

# ----------------------------
# Determinism / threads
# ----------------------------
NTHREADS = int(os.environ.get("KTXX_THREADS", "8"))
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(NTHREADS))
try:
    torch.set_num_threads(NTHREADS)
except Exception:
    pass

SEED = int(os.environ.get("SEED", "1234"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(os.environ.get("DEVICE", "cpu"))
DTYPE_ACC = torch.float32
DTYPE_STORE = torch.float16  # for storing blocks if QMODE=float16

def now():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def log(msg: str):
    print(msg, flush=True)

# ----------------------------
# Config
# ----------------------------
@dataclass
class Cfg:
    MODEL_DIR: str = os.environ.get("MODEL_DIR", "/home/daniyar/deepseek-model")
    OUTPUT_DIR: str = os.environ.get("OUTPUT_DIR", "/home/daniyar/moe_ws_outputs")
    LAYER: int = int(os.environ.get("LAYER", "1"))
    MAX_EXPERTS: int = int(os.environ.get("MAX_EXPERTS", "16"))

    # Calibration / router
    CALIB_PATH: str = os.environ.get("CALIB_PATH", "").strip()
    ROUTER_PATH: str = os.environ.get("ROUTER_PATH", "").strip()
    CALIB_SAMPLES: int = int(os.environ.get("CALIB_SAMPLES", "4096"))  # cap if X has more
    RIDGE_WEIGHTED: bool = os.environ.get("RIDGE_WEIGHTED", "0") == "1"
    ROUTER_EIDS_ARE_GLOBAL: bool = os.environ.get("ROUTER_EIDS_ARE_GLOBAL", "1") == "1"

    # Build Ws
    LIN_MODE: str = os.environ.get("LIN_MODE", "ridge").strip().lower()  # ridge|weff_gate|weff
    RIDGE_DAMP: float = float(os.environ.get("RIDGE_DAMP", "1e-3"))
    AUTO_FALLBACK_LIN: bool = os.environ.get("AUTO_FALLBACK_LIN", "1") == "1"
    STRICT_RIDGE_NEEDS_X: bool = os.environ.get("STRICT_RIDGE_NEEDS_X", "0") == "1"
    WEFF_GATE_MC_SAMPLES: int = int(os.environ.get("WEFF_GATE_MC_SAMPLES", "256"))
    NORMALIZE_W: bool = os.environ.get("NORMALIZE_W", "1") == "1"

    # Basis / clustering
    BASIS_MODE: str = os.environ.get("BASIS_MODE", "hadamard_perm").strip().lower()  # hadamard_perm|svd|dense_train
    HAD_SEED: int = int(os.environ.get("HAD_SEED", "1234"))

    M_CLUSTERS: int = int(os.environ.get("M_CLUSTERS", "0"))  # 0 => auto from TARGET_CLUSTER_SIZE
    TARGET_CLUSTER_SIZE: int = int(os.environ.get("TARGET_CLUSTER_SIZE", "4"))
    CLUSTER_FEAT_D: int = int(os.environ.get("CLUSTER_FEAT_D", "64"))
    CLUSTER_ITERS: int = int(os.environ.get("CLUSTER_ITERS", "80"))
    CLUSTER_RESTARTS: int = int(os.environ.get("CLUSTER_RESTARTS", "4"))
    CLUSTER_MIN_SIZE: int = int(os.environ.get("CLUSTER_MIN_SIZE", "1"))

    # Dense_train only
    TRAIN_STEPS: int = int(os.environ.get("TRAIN_STEPS", "0"))
    TRAIN_LR: float = float(os.environ.get("TRAIN_LR", "1e-2"))
    TRAIN_SUBM: int = int(os.environ.get("TRAIN_SUBM", "256"))
    TRAIN_BATCH_E: int = int(os.environ.get("TRAIN_BATCH_E", "4"))
    TRAIN_MIN_CLUSTER: int = int(os.environ.get("TRAIN_MIN_CLUSTER", "2"))
    TRAIN_REORTHO_EVERY: int = int(os.environ.get("TRAIN_REORTHO_EVERY", "4"))

    # Core
    CORE_MODE: str = os.environ.get("CORE_MODE", "blocktopk_perexpert").strip().lower()
    CORE_AGG: str = os.environ.get("CORE_AGG", "mean").strip().lower()
    CORE_BLOCK: int = int(os.environ.get("CORE_BLOCK", "64"))
    CORE_TARGET: float = float(os.environ.get("CORE_TARGET", "0.90"))
    CORE_MAX_BLOCKS: int = int(os.environ.get("CORE_MAX_BLOCKS", "512"))

    # Residual
    RES_RANK: int = int(os.environ.get("RES_RANK", "64"))
    RES_COEF: str = os.environ.get("RES_COEF", "full").strip().lower()  # full|diag
    RES_BLOCKS: int = int(os.environ.get("RES_BLOCKS", "128"))
    RES_BSIZE: int = int(os.environ.get("RES_BSIZE", "64"))

    # Quantization (optional)
    QMODE: str = os.environ.get("QMODE", "none").strip().lower()  # none|float16|int8
    Q_PERBLOCK: bool = os.environ.get("Q_PERBLOCK", "1") == "1"

    # Eval
    EVAL_TRIALS: int = int(os.environ.get("EVAL_TRIALS", "8"))
    EVAL_BATCH: int = int(os.environ.get("EVAL_BATCH", "2"))
    ROUTED_K: int = int(os.environ.get("ROUTED_K", "8"))

cfg = Cfg()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# ----------------------------
# IO helpers
# ----------------------------
def save_npz(path: str, arrays: Dict[str, np.ndarray]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez(path, **arrays)

def try_load_npz(path: str) -> Optional[Dict[str, np.ndarray]]:
    if not os.path.isfile(path):
        return None
    z = np.load(path, allow_pickle=False)
    out = {k: z[k] for k in z.files}
    z.close()
    return out

def _encode_meta(meta: dict) -> np.ndarray:
    b = json.dumps(meta, sort_keys=True).encode("utf-8")
    return np.frombuffer(b, dtype=np.uint8)

def _decode_meta(arr: np.ndarray) -> dict:
    try:
        b = bytes(arr.tolist())
        return json.loads(b.decode("utf-8"))
    except Exception:
        return {}

# ----------------------------
# Safetensors / model index
# ----------------------------
def read_index(model_dir: str) -> Dict[str, str]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    wm = obj.get("weight_map", {})
    if not wm:
        raise RuntimeError("Index JSON has empty weight_map.")
    return wm

def find_layer_expert_ids(weight_map: Dict[str, str], layer: int) -> List[int]:
    pat = re.compile(rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.")
    ids = set()
    for k in weight_map.keys():
        m = pat.match(k)
        if m:
            ids.add(int(m.group(1)))
    return sorted(ids)

def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    prefix = f"model.layers.{layer}.mlp.experts.{eid}."
    def pick(cands: List[str]) -> Optional[str]:
        for suf in cands:
            k = prefix + suf
            if k in weight_map:
                return k
        return None
    up   = pick(["up_proj.weight", "w3.weight", "w1.weight"])
    gate = pick(["gate_proj.weight", "w1.weight", "w3.weight"])
    down = pick(["down_proj.weight", "w2.weight"])
    if up is None or gate is None or down is None:
        return {}
    if up == gate:
        g2 = pick(["gate_proj.weight"])
        if g2:
            gate = g2
    return {"up": up, "gate": gate, "down": down}

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], keys: List[str]) -> Dict[str, torch.Tensor]:
    by_shard: Dict[str, List[str]] = {}
    for k in keys:
        shard = weight_map.get(k, None)
        if shard is None:
            raise KeyError(f"Key not in weight_map: {k}")
        by_shard.setdefault(shard, []).append(k)

    out: Dict[str, torch.Tensor] = {}
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(sp):
            raise FileNotFoundError(f"Missing shard: {sp}")
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks:
                out[k] = f.get_tensor(k)
    return out

# ----------------------------
# Calibration / router
# ----------------------------
def default_calib_path() -> str:
    p = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
    return p if os.path.isfile(p) else ""

def load_calib_X(H: int) -> Optional[torch.Tensor]:
    path = cfg.CALIB_PATH or default_calib_path()
    if not path:
        return None
    if not os.path.isfile(path):
        raise FileNotFoundError(f"CALIB_PATH not found: {path}")
    z = np.load(path, allow_pickle=False)
    if "X" not in z.files:
        raise KeyError(f"CALIB npz missing key 'X'. Keys={list(z.files)}")
    X = torch.from_numpy(z["X"]).to(DTYPE_ACC)
    z.close()
    if X.ndim != 2 or X.shape[1] != H:
        raise RuntimeError(f"Bad X shape: {tuple(X.shape)} expected (*,{H})")
    if X.shape[0] > cfg.CALIB_SAMPLES:
        X = X[:cfg.CALIB_SAMPLES]
    return X.to(DEVICE)

def load_router_P() -> Optional[np.ndarray]:
    if not cfg.ROUTER_PATH:
        return None
    if not os.path.isfile(cfg.ROUTER_PATH):
        raise FileNotFoundError(f"ROUTER_PATH not found: {cfg.ROUTER_PATH}")
    z = np.load(cfg.ROUTER_PATH, allow_pickle=False)
    if "P" not in z.files:
        raise KeyError(f"ROUTER npz missing key 'P'. Keys={list(z.files)}")
    P = z["P"].astype(np.float32, copy=False)
    z.close()
    return P

# ----------------------------
# Hadamard (FWHT)
# ----------------------------
def is_power_of_two(n: int) -> bool:
    return (n > 0) and ((n & (n - 1)) == 0)

def fwht(x: torch.Tensor) -> torch.Tensor:
    n = x.shape[-1]
    if not is_power_of_two(n):
        raise ValueError(f"FWHT requires power-of-two, got {n}")
    orig = x.shape
    y = x.reshape(-1, n).contiguous()
    h = 1
    while h < n:
        y = y.reshape(-1, n // (2*h), 2, h)
        a = y[:, :, 0, :]
        b = y[:, :, 1, :]
        y[:, :, 0, :] = a + b
        y[:, :, 1, :] = a - b
        y = y.reshape(-1, n)
        h *= 2
    return y.reshape(orig)

def fwht_ortho(x: torch.Tensor) -> torch.Tensor:
    return fwht(x) / math.sqrt(x.shape[-1])

# ----------------------------
# Basis objects (dense or implicit hadamard_perm)
# ----------------------------
class Basis:
    mode: str
    def right(self, x: torch.Tensor, transpose: bool = False) -> torch.Tensor:
        raise NotImplementedError()

class DenseBasis(Basis):
    def __init__(self, M: torch.Tensor):
        self.mode = "dense"
        self.M = M.to(DEVICE).to(DTYPE_ACC).contiguous()  # (n,n)
    def right(self, x: torch.Tensor, transpose: bool = False) -> torch.Tensor:
        if transpose:
            return x @ self.M.t()
        return x @ self.M

class HadamardPermBasis(Basis):
    """
    Represents B = P @ D @ H  (right-multiply form)
      x @ B = fwht_ortho( x[:, perm] * sign )
      x @ B^T = ( fwht_ortho(x) * sign )[:, inv_perm]
    """
    def __init__(self, n: int, seed: int):
        if not is_power_of_two(n):
            raise ValueError(f"hadamard_perm requires power-of-two n, got {n}")
        self.mode = "hadamard_perm"
        g = np.random.default_rng(seed)
        perm = np.array(g.permutation(n), dtype=np.int32)
        inv = np.empty_like(perm)
        inv[perm] = np.arange(n, dtype=np.int32)
        sign = g.choice([-1.0, 1.0], size=(n,), replace=True).astype(np.float32)
        self.perm = torch.from_numpy(perm).to(torch.long)
        self.inv_perm = torch.from_numpy(inv).to(torch.long)
        self.sign = torch.from_numpy(sign).to(DTYPE_ACC)
    def right(self, x: torch.Tensor, transpose: bool = False) -> torch.Tensor:
        if not transpose:
            y = x.index_select(dim=1, index=self.perm.to(x.device))
            y = y * self.sign.to(x.device).view(1, -1)
            return fwht_ortho(y)
        else:
            y = fwht_ortho(x)
            y = y * self.sign.to(x.device).view(1, -1)
            return y.index_select(dim=1, index=self.inv_perm.to(x.device))

def apply_Ut_W_V(W: torch.Tensor, U: Basis, V: Basis) -> torch.Tensor:
    # Compute X = U^T W V without materializing matrices when hadamard_perm.
    # Step1: A = W V
    A = V.right(W, transpose=False)  # treat W as (n,n) row-batch
    # Step2: X = U^T A = (A^T U)^T
    B = A.t().contiguous()
    C = U.right(B, transpose=False)  # B @ U
    return C.t().contiguous()

# ----------------------------
# Linearization / Ws build
# ----------------------------
@torch.no_grad()
def forward_mlp(X: torch.Tensor, W_gate: torch.Tensor, W_up: torch.Tensor, W_down: torch.Tensor) -> torch.Tensor:
    Xf = X.to(DTYPE_ACC)
    up = Xf @ W_up.to(DTYPE_ACC).t()
    gate = Xf @ W_gate.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    y = hid @ W_down.to(DTYPE_ACC).t()
    return y

@torch.no_grad()
def gate_mean_from_weight_stats(W_gate: torch.Tensor, nsamp: int) -> torch.Tensor:
    Wg = W_gate.to(DTYPE_ACC)
    sigma = torch.linalg.norm(Wg, dim=1).clamp_min(1e-8)  # (d_ff,)
    g = torch.Generator(device=Wg.device).manual_seed(SEED + 777)
    z = torch.randn(nsamp, sigma.numel(), generator=g, device=Wg.device, dtype=DTYPE_ACC) * sigma.view(1, -1)
    return F.silu(z).mean(dim=0)

def ws_cache_path(e: int) -> str:
    return os.path.join(cfg.OUTPUT_DIR, f"Ws_cache_layer{cfg.LAYER}_E{e}_{cfg.LIN_MODE}.npz")

def ws_cache_meta(eids: List[int]) -> dict:
    p = cfg.CALIB_PATH or default_calib_path()
    return dict(
        model_dir=cfg.MODEL_DIR,
        layer=cfg.LAYER,
        eids=eids,
        lin_mode=cfg.LIN_MODE,
        ridge_damp=cfg.RIDGE_DAMP,
        ridge_weighted=cfg.RIDGE_WEIGHTED,
        calib_path=p or "",
        router_path=cfg.ROUTER_PATH or "",
        normalize_w=cfg.NORMALIZE_W,
        seed=SEED,
    )

@torch.no_grad()
def load_or_build_Ws() -> Tuple[List[int], torch.Tensor]:
    wm = read_index(cfg.MODEL_DIR)
    all_eids = find_layer_expert_ids(wm, cfg.LAYER)
    if not all_eids:
        raise RuntimeError(f"No experts found for layer={cfg.LAYER}")
    eids = all_eids[:cfg.MAX_EXPERTS]

    cpath = ws_cache_path(len(eids))
    z = try_load_npz(cpath)
    if z and ("Ws" in z) and ("expert_ids" in z) and ("meta" in z):
        meta_ok = (_decode_meta(z["meta"]) == ws_cache_meta(eids))
        if meta_ok:
            Ws = torch.from_numpy(z["Ws"]).to(DTYPE_ACC).to(DEVICE)
            log(f"[cache] loaded Ws: {cpath}  Ws={tuple(Ws.shape)}")
            return [int(x) for x in z["expert_ids"].tolist()], Ws
        else:
            log("[cache] Ws meta mismatch -> rebuilding.")

    # Load tensors for selected experts
    per_e = {}
    need_keys = []
    for eid in eids:
        kk = pick_expert_tensor_keys(wm, cfg.LAYER, eid)
        if not kk:
            raise RuntimeError(f"Expert {eid} missing keys in index.")
        per_e[eid] = kk
        need_keys += [kk["up"], kk["down"], kk["gate"]]

    log("[load] reading tensors from shards ...")
    T = load_tensors_from_shards(cfg.MODEL_DIR, wm, sorted(set(need_keys)))

    W_up0 = T[per_e[eids[0]]["up"]]
    dff, H = int(W_up0.shape[0]), int(W_up0.shape[1])
    log(f"[shape] H={H} d_ff={dff}")

    X = load_calib_X(H)
    if X is not None:
        log(f"[calib] X loaded: {tuple(X.shape)}  (path={cfg.CALIB_PATH or default_calib_path()})")
    else:
        log("[calib] X missing.")

    P = load_router_P()
    if cfg.RIDGE_WEIGHTED and (P is None):
        log("[router] ROUTER_PATH missing -> forcing RIDGE_WEIGHTED=0")
        cfg.RIDGE_WEIGHTED = False

    lin_mode = cfg.LIN_MODE
    if lin_mode == "ridge" and X is None:
        if cfg.STRICT_RIDGE_NEEDS_X:
            raise RuntimeError("LIN_MODE=ridge requires CALIB_PATH but X is missing.")
        if cfg.AUTO_FALLBACK_LIN:
            log("[calib] LIN_MODE=ridge but X missing -> AUTO_FALLBACK_LIN -> switching to LIN_MODE=weff")
            lin_mode = "weff"
        else:
            raise RuntimeError("LIN_MODE=ridge but X missing. Provide CALIB_PATH or set AUTO_FALLBACK_LIN=1.")

    # Precompute ridge cholesky if unweighted
    cholG = None
    I = torch.eye(H, dtype=DTYPE_ACC, device=DEVICE)
    lam = None
    if lin_mode == "ridge":
        assert X is not None
        Xf = X.to(DTYPE_ACC)
        XtX = Xf.t() @ Xf
        lam = cfg.RIDGE_DAMP * float(torch.trace(XtX).item()) / float(H)
        if not cfg.RIDGE_WEIGHTED:
            cholG = torch.linalg.cholesky(XtX + lam * I)

    Ws_list: List[torch.Tensor] = []
    for i, eid in enumerate(tqdm(eids, desc=f"Build Ws ({lin_mode})")):
        W_up   = T[per_e[eid]["up"]].to(DEVICE)
        W_down = T[per_e[eid]["down"]].to(DEVICE)
        W_gate = T[per_e[eid]["gate"]].to(DEVICE)

        if lin_mode == "weff":
            W = (W_down.to(DTYPE_ACC) @ W_up.to(DTYPE_ACC)).contiguous()

        elif lin_mode == "weff_gate":
            if X is not None:
                gate_act = (X.to(DTYPE_ACC) @ W_gate.to(DTYPE_ACC).t())
                m = F.silu(gate_act).mean(dim=0)
            else:
                m = gate_mean_from_weight_stats(W_gate, nsamp=cfg.WEFF_GATE_MC_SAMPLES)
            W = ((W_down.to(DTYPE_ACC) * m.view(1, -1)) @ W_up.to(DTYPE_ACC)).contiguous()

        elif lin_mode == "ridge":
            assert X is not None
            Xf = X.to(DTYPE_ACC)
            Y = forward_mlp(X, W_gate, W_up, W_down).to(DTYPE_ACC)
            if cfg.RIDGE_WEIGHTED and (P is not None):
                if cfg.ROUTER_EIDS_ARE_GLOBAL:
                    w = torch.from_numpy(P[:X.shape[0], eid]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
                else:
                    w = torch.from_numpy(P[:X.shape[0], i]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
                sw = torch.sqrt(w + 1e-12).view(-1, 1)
                Xw = Xf * sw
                Yw = Y * sw
                XtX_e = Xw.t() @ Xw
                lam_e = cfg.RIDGE_DAMP * float(torch.trace(XtX_e).item()) / float(H)
                chol = torch.linalg.cholesky(XtX_e + lam_e * I)
                XtY = Xw.t() @ Yw
                Wt = torch.cholesky_solve(XtY, chol)
                W = Wt.t().contiguous()
            else:
                assert cholG is not None
                XtY = Xf.t() @ Y
                Wt = torch.cholesky_solve(XtY, cholG)
                W = Wt.t().contiguous()
        else:
            raise ValueError("LIN_MODE must be ridge|weff_gate|weff")

        if cfg.NORMALIZE_W:
            fn = torch.linalg.norm(W, ord="fro").clamp_min(1e-12)
            W = (W / fn).contiguous()

        Ws_list.append(W)

    Ws = torch.stack(Ws_list, dim=0).to(DTYPE_ACC).to(DEVICE)

    # write cache
    meta = ws_cache_meta(eids)
    save_npz(cpath, {
        "meta": _encode_meta(meta),
        "expert_ids": np.array(eids, dtype=np.int32),
        "Ws": Ws.detach().cpu().numpy().astype(np.float32),
    })
    log(f"[cache] wrote Ws -> {cpath}  size={os.path.getsize(cpath)/1e6:.2f} MB")
    return eids, Ws

# ----------------------------
# Clustering (kmeans++ + restarts)
# ----------------------------
@torch.no_grad()
def random_proj_features(Ws: torch.Tensor, d: int) -> torch.Tensor:
    # cheap features: projected row/col energy
    E, n, _ = Ws.shape
    g = torch.Generator(device="cpu").manual_seed(SEED)
    R = (torch.randint(0, 2, (n, d), generator=g, dtype=torch.int8) * 2 - 1).to(DTYPE_ACC).to(DEVICE)
    feats = []
    for e in range(E):
        W = Ws[e]
        row = torch.diag(W @ W.t())
        col = torch.diag(W.t() @ W)
        feats.append(torch.cat([(row @ R), (col @ R)], dim=0).unsqueeze(0))  # (1,2d)
    X = torch.cat(feats, dim=0)
    X = (X - X.mean(dim=0, keepdim=True)) / (X.std(dim=0, keepdim=True) + 1e-6)
    return X

@torch.no_grad()
def kmeanspp_init(X: torch.Tensor, k: int) -> torch.Tensor:
    g = torch.Generator(device=X.device).manual_seed(SEED)
    n = X.shape[0]
    centers = []
    idx0 = torch.randint(0, n, (1,), generator=g, device=X.device).item()
    centers.append(X[idx0].clone())
    for _ in range(1, k):
        C = torch.stack(centers, dim=0)
        dist2 = torch.cdist(X, C).pow(2).min(dim=1).values
        prob = dist2 / dist2.sum().clamp_min(1e-12)
        idx = torch.multinomial(prob, num_samples=1, generator=g).item()
        centers.append(X[idx].clone())
    return torch.stack(centers, dim=0)

@torch.no_grad()
def kmeans_torch(X: torch.Tensor, k: int, iters: int, restarts: int) -> torch.Tensor:
    best_lab = None
    best_inertia = float("inf")
    for r in range(max(1, restarts)):
        C = kmeanspp_init(X, k)
        for _ in range(iters):
            dist = torch.cdist(X, C)
            lab = dist.argmin(dim=1)
            # fix empties
            for j in range(k):
                m = (lab == j)
                if m.any():
                    C[j] = X[m].mean(dim=0)
                else:
                    # re-seed empty to farthest point
                    far = dist.min(dim=1).values.argmax().item()
                    C[j] = X[far].clone()
        inertia = float((torch.cdist(X, C).min(dim=1).values ** 2).sum().item())
        if inertia < best_inertia:
            best_inertia = inertia
            best_lab = lab.clone()
    assert best_lab is not None
    return best_lab

@torch.no_grad()
def enforce_min_cluster_size(labels: torch.Tensor, X: torch.Tensor, min_size: int) -> torch.Tensor:
    if min_size <= 1:
        return labels
    k = int(labels.max().item()) + 1
    counts = torch.bincount(labels, minlength=k)
    big = (counts >= min_size).nonzero(as_tuple=False).flatten()
    small = (counts < min_size).nonzero(as_tuple=False).flatten()
    if big.numel() == 0 or small.numel() == 0:
        return labels
    big_centers = torch.stack([X[labels == j].mean(dim=0) for j in big.tolist()], dim=0)
    for c in small.tolist():
        idxs = (labels == c).nonzero(as_tuple=False).flatten()
        if idxs.numel() == 0:
            continue
        d = torch.cdist(X[idxs], big_centers)
        nn = d.argmin(dim=1)
        labels[idxs] = big[nn]
    return labels

# ----------------------------
# Core block selection
# ----------------------------
@torch.no_grad()
def _block_energy_grid(X: torch.Tensor, b: int) -> torch.Tensor:
    n = X.shape[0]
    nb = (n + b - 1) // b
    if (n % b) != 0:
        pad = nb * b - n
        Xp = torch.zeros(nb*b, nb*b, dtype=X.dtype, device=X.device)
        Xp[:n, :n] = X
        X = Xp
        n = nb*b
    Xb = X.view(nb, b, nb, b).permute(0, 2, 1, 3).contiguous()  # (nb,nb,b,b)
    E = (Xb * Xb).sum(dim=(2,3))  # (nb,nb)
    return E

@torch.no_grad()
def _pick_top_blocks(Eg: torch.Tensor, target_frac: float, max_blocks: int) -> List[Tuple[int,int]]:
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    csum = torch.cumsum(flat[order], dim=0)
    tot = float(flat.sum().item())
    if tot <= 1e-12:
        return []
    frac = csum / tot
    need = int((torch.nonzero(frac >= target_frac, as_tuple=False)[0].item() + 1) if (frac >= target_frac).any() else flat.numel())
    K = min(need, max_blocks, flat.numel())
    pick = order[:K].tolist()
    nb = Eg.shape[0]
    out = [(p // nb, p % nb) for p in pick]
    return out

@torch.no_grad()
def choose_core_blocks(X_list: List[torch.Tensor]) -> Dict[str, Any]:
    mode = cfg.CORE_MODE
    b = int(cfg.CORE_BLOCK)
    if mode == "none":
        return {"mode": "none", "shared": None, "per": None}

    if mode == "blockdiag":
        n = X_list[0].shape[0]
        nb = (n + b - 1) // b
        diagE = torch.zeros(nb, dtype=DTYPE_ACC, device=X_list[0].device)
        for X in X_list:
            Eg = _block_energy_grid(X, b).to(DTYPE_ACC)
            diagE += torch.diagonal(Eg, 0)
        diagE /= max(1, len(X_list))
        # pick diag blocks
        order = torch.argsort(diagE, descending=True)
        csum = torch.cumsum(diagE[order], dim=0)
        tot = float(diagE.sum().item())
        K = int((torch.nonzero((csum / max(tot,1e-12)) >= cfg.CORE_TARGET, as_tuple=False)[0].item() + 1) if tot > 0 and ((csum/tot) >= cfg.CORE_TARGET).any() else nb)
        K = min(K, cfg.CORE_MAX_BLOCKS, nb)
        blocks = [(int(i.item()), int(i.item())) for i in order[:K]]
        return {"mode": "blockdiag", "shared": blocks, "per": None}

    if mode == "blocktopk":
        Eg_all = torch.stack([_block_energy_grid(X, b).to(DTYPE_ACC) for X in X_list], dim=0)
        Eg = Eg_all.amax(dim=0) if cfg.CORE_AGG == "max" else Eg_all.mean(dim=0)
        shared = _pick_top_blocks(Eg, cfg.CORE_TARGET, cfg.CORE_MAX_BLOCKS)
        return {"mode": "blocktopk", "shared": shared, "per": None}

    if mode == "blocktopk_perexpert":
        per = []
        for X in X_list:
            Eg = _block_energy_grid(X, b).to(DTYPE_ACC)
            per.append(_pick_top_blocks(Eg, cfg.CORE_TARGET, cfg.CORE_MAX_BLOCKS))
        return {"mode": "blocktopk_perexpert", "shared": None, "per": per}

    raise ValueError("CORE_MODE must be blocktopk_perexpert|blocktopk|blockdiag|none")

# ----------------------------
# Faster residual sparse block pick (grid-level, no overlap)
# ----------------------------
@torch.no_grad()
def pick_residual_blocks(R: torch.Tensor, b: int, k: int) -> List[Tuple[int,int,torch.Tensor]]:
    n = R.shape[0]
    nb = (n + b - 1) // b
    Eg = _block_energy_grid(R, b).to(DTYPE_ACC)  # (nb,nb)
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    used = torch.zeros((nb, nb), dtype=torch.bool, device=R.device)
    out: List[Tuple[int,int,torch.Tensor]] = []
    for idx in order.tolist():
        if len(out) >= k:
            break
        bi = idx // nb
        bj = idx % nb
        if used[bi, bj]:
            continue
        used[bi, bj] = True
        i0, j0 = bi * b, bj * b
        i1, j1 = min(n, i0 + b), min(n, j0 + b)
        blk = R[i0:i1, j0:j1].contiguous()
        out.append((i0, j0, blk))
    return out

# ----------------------------
# Quant (optional)
# ----------------------------
@torch.no_grad()
def qpack(t: torch.Tensor) -> Dict[str, Any]:
    # int8 symmetric
    x = t.detach().to(torch.float32)
    maxabs = float(x.abs().max().item())
    if maxabs < 1e-12:
        q = torch.zeros_like(x, dtype=torch.int8)
        s = torch.tensor(1.0, dtype=torch.float32)
    else:
        s = torch.tensor(maxabs / 127.0, dtype=torch.float32)
        q = torch.clamp(torch.round(x / s), -127, 127).to(torch.int8)
    return {"q": q.cpu().numpy(), "s": np.array([float(s.item())], dtype=np.float32), "shape": np.array(x.shape, dtype=np.int32)}

@torch.no_grad()
def qunpack(obj: Dict[str, Any], device: torch.device) -> torch.Tensor:
    q = torch.from_numpy(obj["q"]).to(torch.int8).to(device)
    s = float(obj["s"][0])
    return (q.to(torch.float32) * s).to(DTYPE_ACC)

# ----------------------------
# Payload build per cluster
# ----------------------------
@torch.no_grad()
def svd_init(Wm: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    U, _, Vh = torch.linalg.svd(Wm, full_matrices=False)
    return U.to(DTYPE_ACC).contiguous(), Vh.t().to(DTYPE_ACC).contiguous()

class OrthoParam(nn.Module):
    def __init__(self, M_init: torch.Tensor):
        super().__init__()
        self.M = nn.Parameter(M_init.clone().to(DEVICE))
    def orthogonal(self) -> torch.Tensor:
        Q, _ = torch.linalg.qr(self.M)
        return Q

@torch.no_grad()
def build_payload_for_cluster(Ws: torch.Tensor, idx: List[int], U: Basis, V: Basis) -> Dict[str, Any]:
    # X_e = U^T W_e V
    X_list = [apply_Ut_W_V(Ws[e], U, V) for e in idx]

    core = choose_core_blocks(X_list)
    b = int(cfg.CORE_BLOCK)

    # Materialize core blocks per expert
    core_blocks: List[List[Tuple[int,int,Any]]] = []
    if core["per"] is not None:
        for X, blocks in zip(X_list, core["per"]):
            lst = []
            for (bi, bj) in blocks:
                i0, j0 = bi*b, bj*b
                i1, j1 = min(X.shape[0], i0+b), min(X.shape[1], j0+b)
                B = X[i0:i1, j0:j1].contiguous()
                if cfg.QMODE == "int8":
                    lst.append((i0, j0, qpack(B)))
                elif cfg.QMODE == "float16":
                    lst.append((i0, j0, B.to(DTYPE_STORE).cpu().numpy()))
                else:
                    lst.append((i0, j0, B))
            core_blocks.append(lst)
    else:
        shared = core["shared"] or []
        for X in X_list:
            lst = []
            for (bi, bj) in shared:
                i0, j0 = bi*b, bj*b
                i1, j1 = min(X.shape[0], i0+b), min(X.shape[1], j0+b)
                B = X[i0:i1, j0:j1].contiguous()
                if cfg.QMODE == "int8":
                    lst.append((i0, j0, qpack(B)))
                elif cfg.QMODE == "float16":
                    lst.append((i0, j0, B.to(DTYPE_STORE).cpu().numpy()))
                else:
                    lst.append((i0, j0, B))
            core_blocks.append(lst)

    # Residuals
    R_list = []
    for X, cb in zip(X_list, core_blocks):
        Xc = torch.zeros_like(X)
        for (i0, j0, obj) in cb:
            if cfg.QMODE == "int8":
                B = qunpack(obj, X.device)
            elif cfg.QMODE == "float16":
                B = torch.from_numpy(obj).to(DTYPE_ACC).to(X.device)
            else:
                B = obj
            h, w = B.shape
            Xc[i0:i0+h, j0:j0+w] = B
        R_list.append((X - Xc).contiguous())

    # Low-rank basis from mean residual
    Rmean = torch.stack(R_list, dim=0).mean(dim=0)
    U_s, _, Vh_s = torch.linalg.svd(Rmean, full_matrices=False)
    r = int(min(cfg.RES_RANK, U_s.shape[1]))
    DL = U_s[:, :r].to(DTYPE_ACC).contiguous()
    DR = Vh_s.t()[:, :r].to(DTYPE_ACC).contiguous()

    # Per-expert coefficients
    coef_list = []
    res_sparse_blocks: List[List[Tuple[int,int,Any]]] = []

    for Rm in R_list:
        if cfg.RES_COEF == "diag":
            # diagonal coefficients: g[i] = dl_i^T R dr_i
            tmp = (DL * (Rm @ DR)).sum(dim=0)  # (r,)
            if cfg.QMODE == "int8":
                coef_list.append(qpack(tmp))
            elif cfg.QMODE == "float16":
                coef_list.append(tmp.to(DTYPE_STORE).cpu().numpy())
            else:
                coef_list.append(tmp)
            R2 = (Rm - (DL * tmp.view(1, -1)) @ DR.t()).contiguous()
        else:
            # full C (r,r): best accuracy for small r
            C = (DL.t() @ Rm @ DR).contiguous()  # (r,r)
            if cfg.QMODE == "int8":
                coef_list.append(qpack(C))
            elif cfg.QMODE == "float16":
                coef_list.append(C.to(DTYPE_STORE).cpu().numpy())
            else:
                coef_list.append(C)
            R2 = (Rm - (DL @ C @ DR.t())).contiguous()

        # sparse residual blocks
        blks = pick_residual_blocks(R2, b=int(cfg.RES_BSIZE), k=int(cfg.RES_BLOCKS))
        bl_out = []
        for (i0, j0, B) in blks:
            if cfg.QMODE == "int8":
                bl_out.append((i0, j0, qpack(B)))
            elif cfg.QMODE == "float16":
                bl_out.append((i0, j0, B.to(DTYPE_STORE).cpu().numpy()))
            else:
                bl_out.append((i0, j0, B))
        res_sparse_blocks.append(bl_out)

    # Pack DL/DR
    if cfg.QMODE == "int8":
        DL_pack, DR_pack = qpack(DL), qpack(DR)
    elif cfg.QMODE == "float16":
        DL_pack, DR_pack = DL.to(DTYPE_STORE).cpu().numpy(), DR.to(DTYPE_STORE).cpu().numpy()
    else:
        DL_pack, DR_pack = DL, DR

    return {
        "idx": idx,
        "U": U, "V": V,
        "core": core,
        "core_blocks": core_blocks,
        "DL": DL_pack, "DR": DR_pack,
        "coef": coef_list,
        "res_blocks": res_sparse_blocks,
        "r": r,
    }

# ----------------------------
# Runtime
# ----------------------------
class KTXRuntime:
    def __init__(self, payloads: List[Optional[Dict[str, Any]]], map_e: Dict[int, Tuple[int,int]], n: int):
        self.payloads = payloads
        self.map_e = map_e
        self.n = n

    @torch.no_grad()
    def apply_one_expert(self, x: torch.Tensor, e: int) -> torch.Tensor:
        # y = x @ W_hat(e)
        m, j = self.map_e[e]
        P = self.payloads[m]
        assert P is not None

        U: Basis = P["U"]
        V: Basis = P["V"]

        z = U.right(x, transpose=False)  # x @ U
        u = torch.zeros_like(z)

        # core blocks
        for (i0, j0, obj) in P["core_blocks"][j]:
            if cfg.QMODE == "int8":
                B = qunpack(obj, z.device)
            elif cfg.QMODE == "float16":
                B = torch.from_numpy(obj).to(DTYPE_ACC).to(z.device)
            else:
                B = obj
            h, w = B.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ B

        # low-rank
        if cfg.QMODE == "int8":
            DL = qunpack(P["DL"], z.device)
            DR = qunpack(P["DR"], z.device)
        elif cfg.QMODE == "float16":
            DL = torch.from_numpy(P["DL"]).to(DTYPE_ACC).to(z.device)
            DR = torch.from_numpy(P["DR"]).to(DTYPE_ACC).to(z.device)
        else:
            DL = P["DL"]
            DR = P["DR"]

        coef = P["coef"][j]
        if cfg.QMODE == "int8":
            C = qunpack(coef, z.device)
        elif cfg.QMODE == "float16":
            C = torch.from_numpy(coef).to(DTYPE_ACC).to(z.device)
        else:
            C = coef

        if cfg.RES_COEF == "diag":
            # diag g
            g = C  # (r,)
            u += ((z @ DL) * g.view(1, -1)) @ DR.t()
        else:
            # full C (r,r)
            u += (z @ DL) @ C @ DR.t()

        # sparse residual blocks
        for (i0, j0, obj) in P["res_blocks"][j]:
            if cfg.QMODE == "int8":
                B = qunpack(obj, z.device)
            elif cfg.QMODE == "float16":
                B = torch.from_numpy(obj).to(DTYPE_ACC).to(z.device)
            else:
                B = obj
            h, w = B.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ B

        # y = u @ V^T
        y = V.right(u, transpose=True)
        return y

    @torch.no_grad()
    def apply_mixture(self, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
        y = torch.zeros_like(x)
        for a, e in zip(gates.tolist(), routed):
            y += float(a) * self.apply_one_expert(x, e)
        return y

# ----------------------------
# Eval
# ----------------------------
@torch.no_grad()
def eval_per_expert(rt: KTXRuntime, Ws: torch.Tensor, trials: int, batch: int):
    E, n, _ = Ws.shape
    errs = []
    for e in range(E):
        ee = []
        for _ in range(trials):
            x = torch.randn(batch, n, dtype=DTYPE_ACC, device=DEVICE)
            y_ref = x @ Ws[e]
            y_hat = rt.apply_one_expert(x, e)
            num = torch.linalg.norm(y_hat - y_ref)
            den = torch.linalg.norm(y_ref).clamp_min(1e-12)
            ee.append(float((num / den).item()))
        errs.append(float(np.mean(ee)))
    p95 = float(np.percentile(errs, 95))
    log(f"[eval] per-expert operator rel-error  mean={float(np.mean(errs)):.6f}  p95={p95:.6f}  max={float(np.max(errs)):.6f}")

@torch.no_grad()
def eval_routed(rt: KTXRuntime, Ws: torch.Tensor, trials: int, routed_k: int, batch: int):
    E, n, _ = Ws.shape
    errs = []
    for _ in range(trials):
        x = torch.randn(batch, n, dtype=DTYPE_ACC, device=DEVICE)
        routed = random.sample(range(E), k=min(routed_k, E))
        gates = torch.rand(len(routed), dtype=DTYPE_ACC, device=DEVICE)
        gates = gates / gates.sum().clamp_min(1e-12)

        # ref
        Wsum = torch.zeros(n, n, dtype=DTYPE_ACC, device=DEVICE)
        for a, e in zip(gates, routed):
            Wsum += float(a.item()) * Ws[e]
        y_ref = x @ Wsum

        y_hat = rt.apply_mixture(x, routed, gates)
        num = torch.linalg.norm(y_hat - y_ref)
        den = torch.linalg.norm(y_ref).clamp_min(1e-12)
        errs.append(float((num / den).item()))
    log(f"[eval] routed forward rel-error = {float(np.mean(errs)):.6f} ± {float(np.std(errs)):.6f}  (trials={trials})")

# ----------------------------
# Payload size estimation + save
# ----------------------------
def approx_payload_bytes(payloads: List[Optional[Dict[str, Any]]]) -> int:
    tot = 0
    def add_arr(a: Any):
        nonlocal tot
        if a is None:
            return
        if isinstance(a, np.ndarray):
            tot += a.nbytes
        elif torch.is_tensor(a):
            tot += a.numel() * a.element_size()
        elif isinstance(a, dict) and ("q" in a) and ("s" in a):
            tot += a["q"].nbytes + a["s"].nbytes + a["shape"].nbytes
        elif isinstance(a, list):
            for x in a:
                add_arr(x)
    for P in payloads:
        if P is None:
            continue
        # basis
        U, V = P["U"], P["V"]
        if isinstance(U, DenseBasis): add_arr(U.M)
        if isinstance(V, DenseBasis): add_arr(V.M)
        if isinstance(U, HadamardPermBasis):
            add_arr(U.perm.cpu().numpy()); add_arr(U.inv_perm.cpu().numpy()); add_arr(U.sign.cpu().numpy())
        if isinstance(V, HadamardPermBasis):
            add_arr(V.perm.cpu().numpy()); add_arr(V.inv_perm.cpu().numpy()); add_arr(V.sign.cpu().numpy())

        # core blocks
        for ex in P["core_blocks"]:
            for (_, _, obj) in ex:
                add_arr(obj)
        # lowrank
        add_arr(P["DL"]); add_arr(P["DR"])
        for c in P["coef"]:
            add_arr(c)
        # residual blocks
        for ex in P["res_blocks"]:
            for (_, _, obj) in ex:
                add_arr(obj)
    return tot

def save_payload_npz(path: str, payloads: List[Optional[Dict[str, Any]]], map_e: Dict[int, Tuple[int,int]], meta: Dict[str, Any]):
    # Save minimal portable representation (numpy only)
    out: Dict[str, np.ndarray] = {}
    out["meta"] = _encode_meta(meta)
    out["map_e"] = np.array([[k, v[0], v[1]] for k, v in sorted(map_e.items())], dtype=np.int32)

    # Payloads: store per cluster in flattened lists.
    # NOTE: For hadamard_perm, store perm/inv/sign. For dense, store matrices.
    for mi, P in enumerate(payloads):
        if P is None:
            out[f"cluster{mi}.empty"] = np.array([1], dtype=np.int8)
            continue
        out[f"cluster{mi}.idx"] = np.array(P["idx"], dtype=np.int32)
        out[f"cluster{mi}.r"] = np.array([P["r"]], dtype=np.int32)

        # basis
        U, V = P["U"], P["V"]
        out[f"cluster{mi}.basis_mode"] = np.array([0 if isinstance(U, HadamardPermBasis) else 1], dtype=np.int8)
        if isinstance(U, HadamardPermBasis):
            out[f"cluster{mi}.U.perm"] = U.perm.cpu().numpy().astype(np.int32)
            out[f"cluster{mi}.U.inv"]  = U.inv_perm.cpu().numpy().astype(np.int32)
            out[f"cluster{mi}.U.sign"] = U.sign.cpu().numpy().astype(np.float32)
        else:
            out[f"cluster{mi}.U"] = U.M.detach().cpu().numpy().astype(np.float32)

        if isinstance(V, HadamardPermBasis):
            out[f"cluster{mi}.V.perm"] = V.perm.cpu().numpy().astype(np.int32)
            out[f"cluster{mi}.V.inv"]  = V.inv_perm.cpu().numpy().astype(np.int32)
            out[f"cluster{mi}.V.sign"] = V.sign.cpu().numpy().astype(np.float32)
        else:
            out[f"cluster{mi}.V"] = V.M.detach().cpu().numpy().astype(np.float32)

        # core blocks
        # store as ragged: (count, 2) indices and concatenated data blobs are out of scope here;
        # we save straightforwardly by numbering blocks.
        for ej, ex in enumerate(P["core_blocks"]):
            out[f"cluster{mi}.core.count.e{ej}"] = np.array([len(ex)], dtype=np.int32)
            for bi, (i0, j0, obj) in enumerate(ex):
                out[f"cluster{mi}.core.pos.e{ej}.b{bi}"] = np.array([i0, j0], dtype=np.int32)
                if cfg.QMODE == "int8":
                    out[f"cluster{mi}.core.q.e{ej}.b{bi}"] = obj["q"]
                    out[f"cluster{mi}.core.s.e{ej}.b{bi}"] = obj["s"]
                    out[f"cluster{mi}.core.sh.e{ej}.b{bi}"] = obj["shape"]
                elif cfg.QMODE == "float16":
                    out[f"cluster{mi}.core.f16.e{ej}.b{bi}"] = obj
                else:
                    out[f"cluster{mi}.core.f32.e{ej}.b{bi}"] = obj.detach().cpu().numpy().astype(np.float32)

        # DL/DR
        def store_mat(prefix: str, obj: Any):
            if cfg.QMODE == "int8":
                out[prefix + ".q"] = obj["q"]; out[prefix + ".s"] = obj["s"]; out[prefix + ".sh"] = obj["shape"]
            elif cfg.QMODE == "float16":
                out[prefix + ".f16"] = obj
            else:
                out[prefix + ".f32"] = obj.detach().cpu().numpy().astype(np.float32)

        store_mat(f"cluster{mi}.DL", P["DL"])
        store_mat(f"cluster{mi}.DR", P["DR"])

        # coef
        for ej, c in enumerate(P["coef"]):
            store_mat(f"cluster{mi}.coef.e{ej}", c)

        # residual blocks
        for ej, ex in enumerate(P["res_blocks"]):
            out[f"cluster{mi}.res.count.e{ej}"] = np.array([len(ex)], dtype=np.int32)
            for bi, (i0, j0, obj) in enumerate(ex):
                out[f"cluster{mi}.res.pos.e{ej}.b{bi}"] = np.array([i0, j0], dtype=np.int32)
                if cfg.QMODE == "int8":
                    out[f"cluster{mi}.res.q.e{ej}.b{bi}"] = obj["q"]
                    out[f"cluster{mi}.res.s.e{ej}.b{bi}"] = obj["s"]
                    out[f"cluster{mi}.res.sh.e{ej}.b{bi}"] = obj["shape"]
                elif cfg.QMODE == "float16":
                    out[f"cluster{mi}.res.f16.e{ej}.b{bi}"] = obj
                else:
                    out[f"cluster{mi}.res.f32.e{ej}.b{bi}"] = obj.detach().cpu().numpy().astype(np.float32)

    save_npz(path, out)
    log(f"[save] payload -> {path}  size={os.path.getsize(path)/1e6:.2f} MB")

# ----------------------------
# Main
# ----------------------------
def banner():
    log("== DeepSeek KT++-X OFFLINE v8 ==")
    log(f"Time:        {now()}")
    log(f"MODEL_DIR:   {cfg.MODEL_DIR}")
    log(f"OUTPUT_DIR:  {cfg.OUTPUT_DIR}")
    log(f"LAYER:       {cfg.LAYER}")
    log(f"MAX_EXPERTS:  {cfg.MAX_EXPERTS}")
    log(f"DEVICE:      {DEVICE}  Torch={torch.__version__} threads={NTHREADS}")
    log(f"CALIB_PATH:  {cfg.CALIB_PATH or default_calib_path() or '(none)'}  CALIB_SAMPLES(cap)={cfg.CALIB_SAMPLES}")
    log(f"ROUTER_PATH: {cfg.ROUTER_PATH or '(none)'}  RIDGE_WEIGHTED={cfg.RIDGE_WEIGHTED}")
    log(f"LIN_MODE:    {cfg.LIN_MODE}  RIDGE_DAMP={cfg.RIDGE_DAMP}  fallback={cfg.AUTO_FALLBACK_LIN}")
    log(f"BASIS_MODE:  {cfg.BASIS_MODE}  HAD_SEED={cfg.HAD_SEED}")
    log(f"CORE:        {cfg.CORE_MODE} block={cfg.CORE_BLOCK} target={cfg.CORE_TARGET} max_blocks={cfg.CORE_MAX_BLOCKS}")
    log(f"RESIDUAL:    rank={cfg.RES_RANK} coef={cfg.RES_COEF} blocks={cfg.RES_BLOCKS} bsize={cfg.RES_BSIZE}")
    log(f"QUANT:       QMODE={cfg.QMODE} (per-block={cfg.Q_PERBLOCK})")
    log("")

def main():
    banner()

    eids, Ws = load_or_build_Ws()
    E, n, _ = Ws.shape
    log(f"[Ws] shape={tuple(Ws.shape)} experts={eids[:8]}{'...' if len(eids)>8 else ''}")

    # Choose cluster count
    if cfg.M_CLUSTERS > 0:
        M = min(cfg.M_CLUSTERS, E)
    else:
        ts = max(1, cfg.TARGET_CLUSTER_SIZE)
        M = max(1, min(E, int(round(E / ts))))
    log(f"[cluster] target_size={cfg.TARGET_CLUSTER_SIZE} => M={M}")

    # Cluster
    Xfeat = random_proj_features(Ws, d=cfg.CLUSTER_FEAT_D)
    labels = kmeans_torch(Xfeat, k=M, iters=cfg.CLUSTER_ITERS, restarts=cfg.CLUSTER_RESTARTS)
    labels = enforce_min_cluster_size(labels, Xfeat, min_size=cfg.CLUSTER_MIN_SIZE)
    clusters = [torch.nonzero(labels == m, as_tuple=False).flatten().tolist() for m in range(M)]
    sizes = [len(c) for c in clusters]
    log(f"[cluster] sizes: {sizes}")

    # Drop empty clusters
    keep = [i for i, s in enumerate(sizes) if s > 0]
    clusters = [clusters[i] for i in keep]
    M2 = len(clusters)
    log(f"[cluster] non-empty clusters: {M2}")

    # Build bases per cluster
    payloads: List[Optional[Dict[str, Any]]] = []
    map_e: Dict[int, Tuple[int,int]] = {}

    # Optional dense_train (explicit U,V)
    U_par: List[Optional[OrthoParam]] = []
    V_par: List[Optional[OrthoParam]] = []

    if cfg.BASIS_MODE in ("svd", "dense_train"):
        # init dense U,V for each cluster
        for idx in clusters:
            Wm = Ws[idx].mean(dim=0)
            U0, V0 = svd_init(Wm)
            U_par.append(OrthoParam(U0))
            V_par.append(OrthoParam(V0))

        if cfg.BASIS_MODE == "dense_train" and cfg.TRAIN_STEPS > 0:
            params = [uv.M for uv in (U_par + V_par) if uv is not None]
            opt = torch.optim.Adam(params, lr=cfg.TRAIN_LR)
            log(f"[train] dense_train steps={cfg.TRAIN_STEPS} lr={cfg.TRAIN_LR}")

            with torch.enable_grad():
                for step in range(1, cfg.TRAIN_STEPS + 1):
                    S = torch.randperm(n, device=DEVICE)[:min(cfg.TRAIN_SUBM, n)]
                    L = None
                    terms = 0
                    for m, idx in enumerate(clusters):
                        if len(idx) < cfg.TRAIN_MIN_CLUSTER:
                            continue
                        Uo = U_par[m].orthogonal()
                        Vo = V_par[m].orthogonal()
                        pick = idx
                        if 0 < cfg.TRAIN_BATCH_E < len(idx):
                            pidx = torch.randperm(len(idx), device=DEVICE)[:cfg.TRAIN_BATCH_E].tolist()
                            pick = [idx[i] for i in pidx]
                        # measure offdiag/diag in a random subspace
                        # build Xs = U_S^T W V_S
                        U_S = Uo[:, S]
                        V_S = Vo[:, S]
                        Ws_b = Ws[pick]
                        Tm = Ws_b @ V_S
                        Xs = torch.matmul(U_S.t().unsqueeze(0), Tm)  # (Eb,s,s)
                        D = torch.diagonal(Xs, dim1=1, dim2=2)
                        off = (Xs - torch.diag_embed(D)).abs().mean()
                        diag = D.abs().mean().clamp_min(1e-6)
                        loss = torch.log(off + 1e-6) - torch.log(diag)
                        L = loss if L is None else (L + loss)
                        terms += 1
                    if L is None:
                        break
                    L = L / max(1, terms)
                    opt.zero_grad(set_to_none=True)
                    L.backward()
                    opt.step()
                    if (step % cfg.TRAIN_REORTHO_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                        with torch.no_grad():
                            for uv in (U_par + V_par):
                                if uv is not None:
                                    uv.M.copy_(uv.orthogonal())
                    if step == 1 or (step % 5) == 0 or step == cfg.TRAIN_STEPS:
                        log(f"[train] step {step}/{cfg.TRAIN_STEPS} loss={float(L.item()):.4f}")

    log("[build] payloads ...")
    for m, idx in enumerate(clusters):
        # basis
        if cfg.BASIS_MODE == "hadamard_perm":
            U = HadamardPermBasis(n=n, seed=cfg.HAD_SEED + 1000 + m)
            V = HadamardPermBasis(n=n, seed=cfg.HAD_SEED + 2000 + m)
        elif cfg.BASIS_MODE in ("svd", "dense_train"):
            U = DenseBasis(U_par[m].orthogonal().detach())
            V = DenseBasis(V_par[m].orthogonal().detach())
        else:
            raise ValueError("BASIS_MODE must be hadamard_perm|svd|dense_train")

        P = build_payload_for_cluster(Ws, idx, U, V)
        payloads.append(P)

        for j, e in enumerate(idx):
            map_e[e] = (m, j)

        # quick core stats
        core_counts = [len(ex) for ex in P["core_blocks"]]
        log(f"  - basis {m}: E={len(idx)} core={cfg.CORE_MODE} core_blocks(mean)≈{float(np.mean(core_counts)):.1f} r={P['r']}")

    # Runtime + eval
    rt = KTXRuntime(payloads, map_e, n=n)

    # Size estimate
    sz = approx_payload_bytes(payloads)
    log(f"[size] approx payload bytes ≈ {sz/1024/1024:.2f} MB  (BASIS_MODE={cfg.BASIS_MODE}, QMODE={cfg.QMODE})")

    eval_per_expert(rt, Ws, trials=max(2, cfg.EVAL_TRIALS//2), batch=cfg.EVAL_BATCH)
    eval_routed(rt, Ws, trials=cfg.EVAL_TRIALS, routed_k=cfg.ROUTED_K, batch=cfg.EVAL_BATCH)

    # Save payload
    payload_path = os.path.join(cfg.OUTPUT_DIR, f"ktx_payload_layer{cfg.LAYER}_E{E}_{cfg.BASIS_MODE}_q{cfg.QMODE}.npz")
    meta = dict(
        time=now(),
        model_dir=cfg.MODEL_DIR,
        output_dir=cfg.OUTPUT_DIR,
        layer=cfg.LAYER,
        expert_ids=eids,
        lin_mode=cfg.LIN_MODE,
        ridge_damp=cfg.RIDGE_DAMP,
        ridge_weighted=cfg.RIDGE_WEIGHTED,
        calib_path=(cfg.CALIB_PATH or default_calib_path() or ""),
        router_path=(cfg.ROUTER_PATH or ""),
        basis_mode=cfg.BASIS_MODE,
        core_mode=cfg.CORE_MODE,
        core_block=cfg.CORE_BLOCK,
        core_target=cfg.CORE_TARGET,
        core_max_blocks=cfg.CORE_MAX_BLOCKS,
        res_rank=cfg.RES_RANK,
        res_coef=cfg.RES_COEF,
        res_blocks=cfg.RES_BLOCKS,
        res_bsize=cfg.RES_BSIZE,
        qmode=cfg.QMODE,
        seed=SEED,
        device=str(DEVICE),
        H=n,
    )
    save_payload_npz(payload_path, payloads, map_e, meta)

    log("\n✅ Done.")
    log("If you want better ridge accuracy:")
    log("  - capture more X rows (you only got 160). Increase CAPTURE_MAX_TOKENS and CAPTURE_ITERS / use longer prompt text.")
    log("If you want smaller payload:")
    log("  - keep BASIS_MODE=hadamard_perm, and try QMODE=float16 or QMODE=int8 (may reduce accuracy).")

if __name__ == "__main__":
    main()


== DeepSeek KT++-X OFFLINE v8 ==
Time:        2026-01-13 10:41:46
MODEL_DIR:   /home/daniyar/deepseek-model
OUTPUT_DIR:  /home/daniyar/moe_ws_outputs
LAYER:       1
MAX_EXPERTS:  16
DEVICE:      cpu  Torch=2.4.1+cpu threads=8
CALIB_PATH:  /home/daniyar/moe_ws_outputs/calib_layer1_X.npz  CALIB_SAMPLES(cap)=4096
ROUTER_PATH: (none)  RIDGE_WEIGHTED=False
LIN_MODE:    ridge  RIDGE_DAMP=0.001  fallback=True
BASIS_MODE:  hadamard_perm  HAD_SEED=1234
CORE:        blocktopk_perexpert block=64 target=0.9 max_blocks=512
RESIDUAL:    rank=64 coef=full blocks=128 bsize=64
QUANT:       QMODE=none (per-block=True)

[cache] Ws meta mismatch -> rebuilding.
[load] reading tensors from shards ...
[shape] H=2048 d_ff=1408
[calib] X loaded: (160, 2048)  (path=/home/daniyar/moe_ws_outputs/calib_layer1_X.npz)


Build Ws (ridge):   0%|          | 0/16 [00:00<?, ?it/s]

[cache] wrote Ws -> /home/daniyar/moe_ws_outputs/Ws_cache_layer1_E16_ridge.npz  size=268.44 MB
[Ws] shape=(16, 2048, 2048) experts=[0, 1, 2, 3, 4, 5, 6, 7]...
[cluster] target_size=4 => M=4
[cluster] sizes: [11, 2, 1, 2]
[cluster] non-empty clusters: 4
[build] payloads ...
  - basis 0: E=11 core=blocktopk_perexpert core_blocks(mean)≈512.0 r=64
  - basis 1: E=2 core=blocktopk_perexpert core_blocks(mean)≈483.0 r=64
  - basis 2: E=1 core=blocktopk_perexpert core_blocks(mean)≈139.0 r=64
  - basis 3: E=2 core=blocktopk_perexpert core_blocks(mean)≈512.0 r=64
[size] approx payload bytes ≈ 157.83 MB  (BASIS_MODE=hadamard_perm, QMODE=none)
[eval] per-expert operator rel-error  mean=1.035362  p95=1.102530  max=1.170229
[eval] routed forward rel-error = 1.026430 ± 0.021613  (trials=8)
[save] payload -> /home/daniyar/moe_ws_outputs/ktx_payload_layer1_E16_hadamard_perm_qnone.npz  size=170.98 MB

✅ Done.
If you want better ridge accuracy:
  - capture more X rows (you only got 160). Increase CAPTURE_

In [11]:
#!/usr/bin/env python3
# ============================================================
# DeepSeek KT++-X OFFLINE v9 (fixed FWHT + v6.1 training + split)
#
# Fixes:
#   - Correct FWHT (no aliased in-place bug) -> hadamard_perm works.
#
# Adds:
#   - Hierarchical cluster splitting (avoid mega-clusters)
#   - Basis modes:
#       hadamard_perm  : tiny basis storage, fast, but often dense X
#       svd            : good baseline, moderate basis storage
#       dense_train    : best sparsity/accuracy; store U,V (float16 recommended)
#   - Training objective (like v6.1):
#       logratio(offdiag/diag) + block-group + guidance mask penalty
#
# Outputs:
#   - Ws cache:  OUTPUT_DIR/Ws_cache_layer{L}_E{E}_{LIN_MODE}_v9.npz
#   - Payload:   OUTPUT_DIR/ktx_payload_layer{L}_E{E}_{BASIS_MODE}_v9_q{QMODE}.npz
#
# Env knobs:
#   PRESET=ultra|balanced|compact  (sets sensible defaults)
#
#   MODEL_DIR, OUTPUT_DIR, LAYER, MAX_EXPERTS
#   CALIB_PATH (npz key 'X' shape (N,H)), CALIB_SAMPLES
#   ROUTER_PATH (npz key 'P' optional), RIDGE_WEIGHTED=0/1
#
#   LIN_MODE=ridge|weff_gate|weff
#   RIDGE_DAMP=1e-3 (try 1e-2 if X small)
#
#   BASIS_MODE=hadamard_perm|svd|dense_train
#   TRAIN_STEPS, TRAIN_WARMUP, TRAIN_LR, SUBM, BATCH_E
#   TRAIN_LAM_BLOCK, TRAIN_LAM_GUIDE, TRAIN_GUIDE_EVERY, TRAIN_GUIDE_TARGET
#
#   CLUSTER params: M0 auto, M_MAX, CLUSTER_MAX_SIZE, etc.
#
#   CORE_MODE=blocktopk_perexpert|blocktopk|blockdiag|none
#   CORE_BLOCK, CORE_TARGET, CORE_MAX_BLOCKS
#
#   RES_RANK, RES_COEF=diag|full, RES_BLOCKS, RES_BSIZE
#
#   QMODE=none|float16|int8  (blocks+lowrank+coef); dense basis stored as float16 by default
# ============================================================

import os, re, json, math, time, random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from safetensors import safe_open

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):  # type: ignore
        return x

# ----------------------------
# Threads / determinism
# ----------------------------
NTHREADS = int(os.environ.get("KTXX_THREADS", "8"))
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(NTHREADS))
try:
    torch.set_num_threads(NTHREADS)
except Exception:
    pass

SEED = int(os.environ.get("SEED", "1234"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(os.environ.get("DEVICE", "cpu"))
DTYPE_ACC = torch.float32
DTYPE_STORE = torch.float16

def now():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def log(msg: str):
    print(msg, flush=True)

# ----------------------------
# Presets
# ----------------------------
PRESET = os.environ.get("PRESET", "").strip().lower()

def preset_default(key: str, default: str) -> str:
    return os.environ.get(key, default)

# ----------------------------
# Config
# ----------------------------
@dataclass
class Cfg:
    MODEL_DIR: str = preset_default("MODEL_DIR", "/home/daniyar/deepseek-model")
    OUTPUT_DIR: str = preset_default("OUTPUT_DIR", "/home/daniyar/moe_ws_outputs")
    LAYER: int = int(preset_default("LAYER", "1"))
    MAX_EXPERTS: int = int(preset_default("MAX_EXPERTS", "16"))

    CALIB_PATH: str = preset_default("CALIB_PATH", "").strip()
    CALIB_SAMPLES: int = int(preset_default("CALIB_SAMPLES", "4096"))

    ROUTER_PATH: str = preset_default("ROUTER_PATH", "").strip()
    RIDGE_WEIGHTED: bool = preset_default("RIDGE_WEIGHTED", "0") == "1"
    ROUTER_EIDS_ARE_GLOBAL: bool = preset_default("ROUTER_EIDS_ARE_GLOBAL", "1") == "1"

    LIN_MODE: str = preset_default("LIN_MODE", "ridge").strip().lower()
    RIDGE_DAMP: float = float(preset_default("RIDGE_DAMP", "1e-3"))
    NORMALIZE_W: bool = preset_default("NORMALIZE_W", "1") == "1"

    BASIS_MODE: str = preset_default("BASIS_MODE", "dense_train").strip().lower()
    HAD_SEED: int = int(preset_default("HAD_SEED", "1234"))

    # Clustering
    M0: int = int(preset_default("M0", "0"))  # 0 => auto
    M_MAX: int = int(preset_default("M_MAX", "16"))
    CLUSTER_FEAT_D: int = int(preset_default("CLUSTER_FEAT_D", "64"))
    CLUSTER_ITERS: int = int(preset_default("CLUSTER_ITERS", "60"))
    CLUSTER_RESTARTS: int = int(preset_default("CLUSTER_RESTARTS", "4"))
    CLUSTER_MIN_SIZE: int = int(preset_default("CLUSTER_MIN_SIZE", "1"))
    CLUSTER_MAX_SIZE: int = int(preset_default("CLUSTER_MAX_SIZE", "3"))
    SPLIT_ITERS: int = int(preset_default("SPLIT_ITERS", "50"))

    # Training
    TRAIN_STEPS: int = int(preset_default("TRAIN_STEPS", "24"))
    TRAIN_WARMUP: int = int(preset_default("TRAIN_WARMUP", "6"))
    TRAIN_LR: float = float(preset_default("TRAIN_LR", "5e-2"))
    SUBM: int = int(preset_default("SUBM", "256"))
    BATCH_E: int = int(preset_default("BATCH_E", "4"))
    TRAIN_MIN_CLUSTER: int = int(preset_default("TRAIN_MIN_CLUSTER", "2"))
    REORTHO_EVERY: int = int(preset_default("REORTHO_EVERY", "4"))
    REPORT_EVERY: int = int(preset_default("REPORT_EVERY", "4"))
    GRAD_CLIP: float = float(preset_default("GRAD_CLIP", "1.0"))

    TRAIN_OBJ: str = preset_default("TRAIN_OBJ", "logratio").strip().lower()
    TRAIN_LAM_BLOCK: float = float(preset_default("TRAIN_LAM_BLOCK", "0.10"))
    TRAIN_LAM_GUIDE: float = float(preset_default("TRAIN_LAM_GUIDE", "1.0"))
    TRAIN_GUIDE_EVERY: int = int(preset_default("TRAIN_GUIDE_EVERY", "2"))
    TRAIN_GUIDE_TARGET: float = float(preset_default("TRAIN_GUIDE_TARGET", "0.75"))
    TRAIN_GUIDE_MAX_BLOCKS: int = int(preset_default("TRAIN_GUIDE_MAX_BLOCKS", "256"))

    # Core / residual
    CORE_MODE: str = preset_default("CORE_MODE", "blocktopk_perexpert").strip().lower()
    CORE_AGG: str = preset_default("CORE_AGG", "mean").strip().lower()
    CORE_BLOCK: int = int(preset_default("CORE_BLOCK", "64"))
    CORE_TARGET: float = float(preset_default("CORE_TARGET", "0.85"))
    CORE_MAX_BLOCKS: int = int(preset_default("CORE_MAX_BLOCKS", "256"))

    RES_RANK: int = int(preset_default("RES_RANK", "512"))
    RES_COEF: str = preset_default("RES_COEF", "diag").strip().lower()  # diag works well
    RES_BLOCKS: int = int(preset_default("RES_BLOCKS", "192"))
    RES_BSIZE: int = int(preset_default("RES_BSIZE", "64"))

    # Quant
    QMODE: str = preset_default("QMODE", "none").strip().lower()  # none|float16|int8
    BASIS_STORE_DTYPE: str = preset_default("BASIS_STORE_DTYPE", "float16").strip().lower()  # float16|float32

    # Eval
    EVAL_TRIALS: int = int(preset_default("EVAL_TRIALS", "8"))
    EVAL_BATCH: int = int(preset_default("EVAL_BATCH", "2"))
    ROUTED_K: int = int(preset_default("ROUTED_K", "8"))

cfg = Cfg()

# Apply presets (override cfg defaults via env only if not set)
if PRESET == "ultra":
    os.environ.setdefault("BASIS_MODE", cfg.BASIS_MODE if "BASIS_MODE" in os.environ else "dense_train")
    os.environ.setdefault("TRAIN_STEPS", "48")
    os.environ.setdefault("TRAIN_WARMUP", "10")
    os.environ.setdefault("TRAIN_LAM_GUIDE", "2.0")
    os.environ.setdefault("TRAIN_GUIDE_EVERY", "1")
    os.environ.setdefault("CORE_TARGET", "0.90")
    os.environ.setdefault("CORE_MAX_BLOCKS", "384")
    os.environ.setdefault("RES_RANK", "768")
    os.environ.setdefault("RES_BLOCKS", "256")
    os.environ.setdefault("RIDGE_DAMP", "1e-2")  # helps when X is tiny
    os.environ.setdefault("BASIS_STORE_DTYPE", "float16")
    # rebuild cfg with updated envs
    cfg = Cfg()

elif PRESET == "compact":
    os.environ.setdefault("BASIS_MODE", cfg.BASIS_MODE if "BASIS_MODE" in os.environ else "hadamard_perm")
    os.environ.setdefault("QMODE", "float16")
    os.environ.setdefault("CORE_TARGET", "0.80")
    os.environ.setdefault("CORE_MAX_BLOCKS", "192")
    os.environ.setdefault("RES_RANK", "256")
    os.environ.setdefault("RES_BLOCKS", "128")
    os.environ.setdefault("RIDGE_DAMP", "1e-2")
    cfg = Cfg()

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# ----------------------------
# NPZ meta helpers
# ----------------------------
def save_npz(path: str, arrays: Dict[str, np.ndarray]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez(path, **arrays)

def try_load_npz(path: str) -> Optional[Dict[str, np.ndarray]]:
    if not os.path.isfile(path):
        return None
    z = np.load(path, allow_pickle=False)
    out = {k: z[k] for k in z.files}
    z.close()
    return out

def _encode_meta(meta: dict) -> np.ndarray:
    b = json.dumps(meta, sort_keys=True).encode("utf-8")
    return np.frombuffer(b, dtype=np.uint8)

def _decode_meta(arr: np.ndarray) -> dict:
    try:
        b = bytes(arr.tolist())
        return json.loads(b.decode("utf-8"))
    except Exception:
        return {}

# ----------------------------
# Offline shard loading
# ----------------------------
def read_index(model_dir: str) -> Dict[str, str]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    wm = obj.get("weight_map", {})
    if not wm:
        raise RuntimeError("Index JSON has empty weight_map.")
    return wm

def find_layer_expert_ids(weight_map: Dict[str, str], layer: int) -> List[int]:
    pat = re.compile(rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.")
    ids = set()
    for k in weight_map.keys():
        m = pat.match(k)
        if m:
            ids.add(int(m.group(1)))
    return sorted(ids)

def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    prefix = f"model.layers.{layer}.mlp.experts.{eid}."
    def pick(cands: List[str]) -> Optional[str]:
        for suf in cands:
            k = prefix + suf
            if k in weight_map:
                return k
        return None
    up   = pick(["up_proj.weight", "w3.weight", "w1.weight"])
    gate = pick(["gate_proj.weight", "w1.weight", "w3.weight"])
    down = pick(["down_proj.weight", "w2.weight"])
    if up is None or gate is None or down is None:
        return {}
    if up == gate:
        g2 = pick(["gate_proj.weight"])
        if g2:
            gate = g2
    return {"up": up, "gate": gate, "down": down}

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], keys: List[str]) -> Dict[str, torch.Tensor]:
    by_shard: Dict[str, List[str]] = {}
    for k in keys:
        shard = weight_map.get(k, None)
        if shard is None:
            raise KeyError(f"Key not in weight_map: {k}")
        by_shard.setdefault(shard, []).append(k)

    out: Dict[str, torch.Tensor] = {}
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(sp):
            raise FileNotFoundError(f"Missing shard: {sp}")
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks:
                out[k] = f.get_tensor(k)
    return out

# ----------------------------
# Calibration / router
# ----------------------------
def default_calib_path() -> str:
    p = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
    return p if os.path.isfile(p) else ""

def load_calib_X(H: int) -> Optional[torch.Tensor]:
    path = cfg.CALIB_PATH or default_calib_path()
    if not path:
        return None
    z = np.load(path, allow_pickle=False)
    if "X" not in z.files:
        raise KeyError(f"CALIB npz missing key 'X'. Keys={list(z.files)}")
    X = torch.from_numpy(z["X"]).to(DTYPE_ACC)
    z.close()
    if X.ndim != 2 or X.shape[1] != H:
        raise RuntimeError(f"Bad X shape: {tuple(X.shape)} expected (*,{H})")
    if X.shape[0] > cfg.CALIB_SAMPLES:
        X = X[:cfg.CALIB_SAMPLES]
    return X.to(DEVICE)

def load_router_P() -> Optional[np.ndarray]:
    if not cfg.ROUTER_PATH:
        return None
    z = np.load(cfg.ROUTER_PATH, allow_pickle=False)
    if "P" not in z.files:
        raise KeyError(f"ROUTER npz missing key 'P'. Keys={list(z.files)}")
    P = z["P"].astype(np.float32, copy=False)
    z.close()
    return P

# ----------------------------
# Correct FWHT (NO aliasing bug)
# ----------------------------
def is_power_of_two(n: int) -> bool:
    return (n > 0) and ((n & (n - 1)) == 0)

def fwht(x: torch.Tensor) -> torch.Tensor:
    """
    Walsh-Hadamard transform on last dimension.
    Correct implementation: no in-place aliasing between a/b views.
    """
    n = x.shape[-1]
    if not is_power_of_two(n):
        raise ValueError(f"FWHT requires power-of-two, got {n}")
    orig = x.shape
    y = x.reshape(-1, n).contiguous()
    h = 1
    while h < n:
        y = y.view(-1, n // (2*h), 2, h)
        a = y[:, :, 0, :].clone()
        b = y[:, :, 1, :].clone()
        y[:, :, 0, :] = a + b
        y[:, :, 1, :] = a - b
        y = y.view(-1, n)
        h *= 2
    return y.view(orig)

def fwht_ortho(x: torch.Tensor) -> torch.Tensor:
    return fwht(x) / math.sqrt(x.shape[-1])

# ----------------------------
# Basis abstractions
# ----------------------------
class Basis:
    def right(self, x: torch.Tensor, transpose: bool = False) -> torch.Tensor:
        raise NotImplementedError()

class DenseBasis(Basis):
    def __init__(self, M: torch.Tensor):
        self.M = M.to(DEVICE).to(DTYPE_ACC).contiguous()
    def right(self, x: torch.Tensor, transpose: bool = False) -> torch.Tensor:
        return x @ (self.M.t() if transpose else self.M)

class HadamardPermBasis(Basis):
    """
    Represents B = P @ D @ H, applied on the RIGHT: x @ B.
      right(x)      = fwht_ortho( x[:,perm] * sign )
      right(x, T)   = ( fwht_ortho(x) * sign )[:, inv_perm]
    """
    def __init__(self, n: int, seed: int):
        if not is_power_of_two(n):
            raise ValueError(f"hadamard_perm requires power-of-two n, got {n}")
        g = np.random.default_rng(seed)
        perm = np.array(g.permutation(n), dtype=np.int32)
        inv = np.empty_like(perm)
        inv[perm] = np.arange(n, dtype=np.int32)
        sign = g.choice([-1.0, 1.0], size=(n,), replace=True).astype(np.float32)
        self.perm = torch.from_numpy(perm).to(torch.long)
        self.inv_perm = torch.from_numpy(inv).to(torch.long)
        self.sign = torch.from_numpy(sign).to(DTYPE_ACC)
    def right(self, x: torch.Tensor, transpose: bool = False) -> torch.Tensor:
        if not transpose:
            y = x.index_select(dim=1, index=self.perm.to(x.device))
            y = y * self.sign.to(x.device).view(1, -1)
            return fwht_ortho(y)
        else:
            y = fwht_ortho(x)
            y = y * self.sign.to(x.device).view(1, -1)
            return y.index_select(dim=1, index=self.inv_perm.to(x.device))

@torch.no_grad()
def apply_Ut_W_V(W: torch.Tensor, U: Basis, V: Basis) -> torch.Tensor:
    # X = U^T W V  using right-multiply primitives
    A = V.right(W, transpose=False)      # W V
    B = A.t().contiguous()               # (W V)^T
    C = U.right(B, transpose=False)      # (W V)^T U
    return C.t().contiguous()            # U^T W V

# ----------------------------
# Build Ws (ridge/weff)
# ----------------------------
@torch.no_grad()
def forward_mlp(X: torch.Tensor, W_gate: torch.Tensor, W_up: torch.Tensor, W_down: torch.Tensor) -> torch.Tensor:
    Xf = X.to(DTYPE_ACC)
    up = Xf @ W_up.to(DTYPE_ACC).t()
    gate = Xf @ W_gate.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    y = hid @ W_down.to(DTYPE_ACC).t()
    return y

@torch.no_grad()
def gate_mean_from_weight_stats(W_gate: torch.Tensor, nsamp: int) -> torch.Tensor:
    Wg = W_gate.to(DTYPE_ACC)
    sigma = torch.linalg.norm(Wg, dim=1).clamp_min(1e-8)
    g = torch.Generator(device=Wg.device).manual_seed(SEED + 777)
    z = torch.randn(nsamp, sigma.numel(), generator=g, device=Wg.device, dtype=DTYPE_ACC) * sigma.view(1, -1)
    return F.silu(z).mean(dim=0)

def ws_cache_path(e: int) -> str:
    return os.path.join(cfg.OUTPUT_DIR, f"Ws_cache_layer{cfg.LAYER}_E{e}_{cfg.LIN_MODE}_v9.npz")

def ws_meta(eids: List[int]) -> dict:
    return dict(
        model_dir=cfg.MODEL_DIR,
        layer=cfg.LAYER,
        eids=eids,
        lin_mode=cfg.LIN_MODE,
        ridge_damp=cfg.RIDGE_DAMP,
        ridge_weighted=cfg.RIDGE_WEIGHTED,
        calib_path=(cfg.CALIB_PATH or default_calib_path() or ""),
        router_path=(cfg.ROUTER_PATH or ""),
        normalize_w=cfg.NORMALIZE_W,
        seed=SEED,
    )

@torch.no_grad()
def load_or_build_Ws() -> Tuple[List[int], torch.Tensor]:
    wm = read_index(cfg.MODEL_DIR)
    all_eids = find_layer_expert_ids(wm, cfg.LAYER)
    if not all_eids:
        raise RuntimeError(f"No experts found for layer={cfg.LAYER}")
    eids = all_eids[:cfg.MAX_EXPERTS]

    cpath = ws_cache_path(len(eids))
    z = try_load_npz(cpath)
    if z and ("Ws" in z) and ("expert_ids" in z) and ("meta" in z):
        if _decode_meta(z["meta"]) == ws_meta(eids):
            Ws = torch.from_numpy(z["Ws"]).to(DTYPE_ACC).to(DEVICE)
            log(f"[cache] loaded Ws: {cpath}  Ws={tuple(Ws.shape)}")
            return [int(x) for x in z["expert_ids"].tolist()], Ws
        log("[cache] Ws meta mismatch -> rebuilding.")

    per_e = {}
    need_keys = []
    for eid in eids:
        kk = pick_expert_tensor_keys(wm, cfg.LAYER, eid)
        if not kk:
            raise RuntimeError(f"Expert {eid} missing tensors in index.")
        per_e[eid] = kk
        need_keys += [kk["up"], kk["down"], kk["gate"]]

    log("[load] reading tensors from shards ...")
    T = load_tensors_from_shards(cfg.MODEL_DIR, wm, sorted(set(need_keys)))

    W_up0 = T[per_e[eids[0]]["up"]]
    dff, H = int(W_up0.shape[0]), int(W_up0.shape[1])
    log(f"[shape] H={H} d_ff={dff}")

    X = load_calib_X(H)
    if X is None:
        raise RuntimeError("Need CALIB_PATH (X) for ridge. You already have it; set CALIB_PATH.")
    log(f"[calib] X loaded: {tuple(X.shape)}")

    P = load_router_P()
    if cfg.RIDGE_WEIGHTED and (P is None):
        log("[router] missing ROUTER_PATH -> forcing RIDGE_WEIGHTED=0")
        cfg.RIDGE_WEIGHTED = False

    I = torch.eye(H, dtype=DTYPE_ACC, device=DEVICE)
    Xf = X.to(DTYPE_ACC)
    XtX = Xf.t() @ Xf
    lam = cfg.RIDGE_DAMP * float(torch.trace(XtX).item()) / float(H)
    cholG = torch.linalg.cholesky(XtX + lam * I)

    Ws_list: List[torch.Tensor] = []
    for i, eid in enumerate(tqdm(eids, desc=f"Build Ws ({cfg.LIN_MODE})")):
        W_up   = T[per_e[eid]["up"]].to(DEVICE)
        W_down = T[per_e[eid]["down"]].to(DEVICE)
        W_gate = T[per_e[eid]["gate"]].to(DEVICE)

        if cfg.LIN_MODE == "weff":
            W = (W_down.to(DTYPE_ACC) @ W_up.to(DTYPE_ACC)).contiguous()

        elif cfg.LIN_MODE == "weff_gate":
            gate_act = (Xf @ W_gate.to(DTYPE_ACC).t())
            m = F.silu(gate_act).mean(dim=0)
            W = ((W_down.to(DTYPE_ACC) * m.view(1, -1)) @ W_up.to(DTYPE_ACC)).contiguous()

        elif cfg.LIN_MODE == "ridge":
            Y = forward_mlp(X, W_gate, W_up, W_down).to(DTYPE_ACC)
            if cfg.RIDGE_WEIGHTED and (P is not None):
                if cfg.ROUTER_EIDS_ARE_GLOBAL:
                    w = torch.from_numpy(P[:X.shape[0], eid]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
                else:
                    w = torch.from_numpy(P[:X.shape[0], i]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
                sw = torch.sqrt(w + 1e-12).view(-1, 1)
                Xw = Xf * sw
                Yw = Y * sw
                XtX_e = Xw.t() @ Xw
                lam_e = cfg.RIDGE_DAMP * float(torch.trace(XtX_e).item()) / float(H)
                chol = torch.linalg.cholesky(XtX_e + lam_e * I)
                XtY = Xw.t() @ Yw
                Wt = torch.cholesky_solve(XtY, chol)
                W = Wt.t().contiguous()
            else:
                XtY = Xf.t() @ Y
                Wt = torch.cholesky_solve(XtY, cholG)
                W = Wt.t().contiguous()
        else:
            raise ValueError("LIN_MODE must be ridge|weff_gate|weff")

        if cfg.NORMALIZE_W:
            fn = torch.linalg.norm(W, ord="fro").clamp_min(1e-12)
            W = (W / fn).contiguous()

        Ws_list.append(W)

    Ws = torch.stack(Ws_list, dim=0).to(DTYPE_ACC).to(DEVICE)

    save_npz(cpath, {
        "meta": _encode_meta(ws_meta(eids)),
        "expert_ids": np.array(eids, dtype=np.int32),
        "Ws": Ws.detach().cpu().numpy().astype(np.float32),
    })
    log(f"[cache] wrote Ws -> {cpath}  size={os.path.getsize(cpath)/1e6:.2f} MB")
    return eids, Ws

# ----------------------------
# Clustering (kmeans++ + restarts + hierarchical split)
# ----------------------------
@torch.no_grad()
def random_proj_features(Ws: torch.Tensor, d: int) -> torch.Tensor:
    E, n, _ = Ws.shape
    g = torch.Generator(device="cpu").manual_seed(SEED)
    R = (torch.randint(0, 2, (n, d), generator=g, dtype=torch.int8) * 2 - 1).to(DTYPE_ACC).to(DEVICE)
    feats = []
    for e in range(E):
        W = Ws[e]
        row = torch.diag(W @ W.t())
        col = torch.diag(W.t() @ W)
        feats.append(torch.cat([(row @ R), (col @ R)], dim=0).unsqueeze(0))
    X = torch.cat(feats, dim=0)
    X = (X - X.mean(dim=0, keepdim=True)) / (X.std(dim=0, keepdim=True) + 1e-6)
    return X

@torch.no_grad()
def kmeanspp_init(X: torch.Tensor, k: int) -> torch.Tensor:
    g = torch.Generator(device=X.device).manual_seed(SEED)
    n = X.shape[0]
    centers = []
    idx0 = torch.randint(0, n, (1,), generator=g, device=X.device).item()
    centers.append(X[idx0].clone())
    for _ in range(1, k):
        C = torch.stack(centers, dim=0)
        dist2 = torch.cdist(X, C).pow(2).min(dim=1).values
        prob = dist2 / dist2.sum().clamp_min(1e-12)
        idx = torch.multinomial(prob, num_samples=1, generator=g).item()
        centers.append(X[idx].clone())
    return torch.stack(centers, dim=0)

@torch.no_grad()
def kmeans_torch(X: torch.Tensor, k: int, iters: int, restarts: int) -> torch.Tensor:
    best_lab = None
    best_inertia = float("inf")
    for _ in range(max(1, restarts)):
        C = kmeanspp_init(X, k)
        for _ in range(iters):
            dist = torch.cdist(X, C)
            lab = dist.argmin(dim=1)
            # handle empties
            for j in range(k):
                m = (lab == j)
                if m.any():
                    C[j] = X[m].mean(dim=0)
                else:
                    far = dist.min(dim=1).values.argmax().item()
                    C[j] = X[far].clone()
        inertia = float((torch.cdist(X, C).min(dim=1).values ** 2).sum().item())
        if inertia < best_inertia:
            best_inertia = inertia
            best_lab = lab.clone()
    assert best_lab is not None
    return best_lab.to(torch.int64)

@torch.no_grad()
def relabel_contiguous(labels: torch.Tensor) -> torch.Tensor:
    labels = labels.to(torch.int64)
    uniq = torch.unique(labels)
    out = labels.clone()
    for new, old in enumerate(uniq.tolist()):
        out[labels == int(old)] = int(new)
    return out

@torch.no_grad()
def enforce_min_cluster_size(labels: torch.Tensor, X: torch.Tensor, min_size: int) -> torch.Tensor:
    if min_size <= 1:
        return relabel_contiguous(labels)
    labels = relabel_contiguous(labels)
    k = int(labels.max().item()) + 1
    counts = torch.bincount(labels, minlength=k)
    big = (counts >= min_size).nonzero(as_tuple=False).flatten()
    small = (counts < min_size).nonzero(as_tuple=False).flatten()
    if big.numel() == 0 or small.numel() == 0:
        return labels
    big_centers = torch.stack([X[labels == j].mean(dim=0) for j in big.tolist()], dim=0)
    for c in small.tolist():
        idxs = (labels == c).nonzero(as_tuple=False).flatten()
        if idxs.numel() == 0:
            continue
        d = torch.cdist(X[idxs], big_centers)
        nn = d.argmin(dim=1)
        labels[idxs] = big[nn]
    return relabel_contiguous(labels)

@torch.no_grad()
def hierarchical_split(X: torch.Tensor, labels: torch.Tensor, max_size: int, max_k: int, split_iters: int) -> torch.Tensor:
    labels = relabel_contiguous(labels)
    if max_size <= 0:
        return labels
    while True:
        K = int(labels.max().item()) + 1
        if K >= max_k:
            break
        counts = torch.bincount(labels, minlength=K)
        biggest = int(torch.argmax(counts).item())
        bigsz = int(counts[biggest].item())
        if bigsz <= max_size:
            break
        idxs = (labels == biggest).nonzero(as_tuple=False).flatten()
        if idxs.numel() < 2:
            break
        sub = X[idxs]
        sub_lab = kmeans_torch(sub, k=2, iters=split_iters, restarts=1)
        a = idxs[sub_lab == 0]
        b = idxs[sub_lab == 1]
        if a.numel() == 0 or b.numel() == 0:
            break
        labels[b] = K
        labels = relabel_contiguous(labels)
    return labels

# ----------------------------
# Training objective (dense_train)
# ----------------------------
class OrthoParam(nn.Module):
    def __init__(self, M_init: torch.Tensor):
        super().__init__()
        self.M = nn.Parameter(M_init.clone().to(DEVICE))
    def orthogonal(self) -> torch.Tensor:
        Q, _ = torch.linalg.qr(self.M)
        return Q

@torch.no_grad()
def svd_init(Wm: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    U, _, Vh = torch.linalg.svd(Wm, full_matrices=False)
    return U.to(DTYPE_ACC).contiguous(), Vh.t().to(DTYPE_ACC).contiguous()

def schedule(step: int, warmup: int, total: int) -> float:
    if total <= 0:
        return 1.0
    if step <= warmup:
        return 0.0
    return float(min(1.0, max(0.0, (step - warmup) / max(1, (total - warmup)))))

def slice_X_batch(Ws_batch: torch.Tensor, U: torch.Tensor, V: torch.Tensor, S: torch.Tensor) -> torch.Tensor:
    U_S = U[:, S]              # (n,s)
    V_S = V[:, S]              # (n,s)
    T = Ws_batch @ V_S         # (Eb,n,s)
    Xs = torch.matmul(U_S.t().unsqueeze(0), T)  # (Eb,s,s)
    return Xs

def offdiag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    Xoff = Xs - torch.diag_embed(D)
    return Xoff.abs().mean()

def diag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    return D.abs().mean()

def block_group_sparsity_penalty(Xs: torch.Tensor, block: int) -> torch.Tensor:
    Eb, s, _ = Xs.shape
    b = int(block)
    nb = s // b
    if b <= 0 or nb <= 0:
        return torch.zeros((), dtype=DTYPE_ACC, device=Xs.device)
    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()
    Xb = X.view(Eb, nb, b, nb, b).permute(0, 1, 3, 2, 4).contiguous()
    Eblk = (Xb * Xb).sum(dim=(3,4))
    P = Eblk.mean(dim=0)
    return torch.sqrt(P + 1e-12).sum() / (P.sum() + 1e-12)

@torch.no_grad()
def make_guidance_mask_from_Xs(Xs: torch.Tensor, block: int, target: float, max_blocks: int) -> Tuple[torch.Tensor, float, int]:
    Eb, s, _ = Xs.shape
    b = int(block)
    nb = s // b
    if b <= 0 or nb <= 0:
        return torch.ones(s, s, dtype=DTYPE_ACC, device=Xs.device), 1.0, 0
    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()
    Xb = X.view(Eb, nb, b, nb, b).permute(0, 1, 3, 2, 4).contiguous()
    Eg = (Xb * Xb).sum(dim=(3,4)).mean(dim=0)  # (nb,nb)
    tot = float((X * X).sum().item()) / max(1, Eb)
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    csum = torch.cumsum(flat[order], dim=0)
    frac = csum / max(tot, 1e-12)
    need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else flat.numel())
    K = min(need, max_blocks, flat.numel())
    pick = order[:K]
    mask = torch.zeros(s2, s2, dtype=DTYPE_ACC, device=Xs.device)
    for idx in pick.tolist():
        bi = idx // nb
        bj = idx % nb
        i0 = bi * b
        j0 = bj * b
        mask[i0:i0+b, j0:j0+b] = 1.0
    if s2 < s:
        full = torch.zeros(s, s, dtype=DTYPE_ACC, device=Xs.device)
        full[:s2, :s2] = mask
        mask = full
    ef = float(frac[K-1].item()) if K > 0 else 0.0
    return mask, ef, int(K)

# ----------------------------
# Core selection (energy-aware)
# ----------------------------
@torch.no_grad()
def _block_energy_grid(X: torch.Tensor, b: int) -> Tuple[torch.Tensor, float, int]:
    n = X.shape[0]
    nb = (n + b - 1) // b
    Xp = X
    if (n % b) != 0:
        Xp = torch.zeros(nb*b, nb*b, dtype=X.dtype, device=X.device)
        Xp[:n, :n] = X
    Xb = Xp.view(nb, b, nb, b).permute(0, 2, 1, 3).contiguous()
    Eg = (Xb * Xb).sum(dim=(2,3))
    tot = float((Xp * Xp).sum().item())
    return Eg, tot, nb

@torch.no_grad()
def _pick_top_blocks(Eg: torch.Tensor, tot_energy: float, b: int, target: float, max_blocks: int) -> Tuple[List[Tuple[int,int]], float]:
    nb = Eg.shape[0]
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    csum = torch.cumsum(flat[order], dim=0)
    frac = csum / max(tot_energy, 1e-12)
    need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else flat.numel())
    K = min(need, max_blocks, flat.numel())
    pick = order[:K].tolist()
    blocks = [(p // nb, p % nb) for p in pick]
    eff = float(frac[K-1].item()) if K > 0 else 0.0
    return blocks, eff

@torch.no_grad()
def choose_core_blocks(X_list: List[torch.Tensor]) -> Dict[str, Any]:
    mode = cfg.CORE_MODE
    b = int(cfg.CORE_BLOCK)
    if mode == "none":
        return {"mode": "none", "shared": None, "per": None, "energy_fracs": []}

    if mode == "blockdiag":
        Eg_sum = None
        tot = 0.0
        for X in X_list:
            Eg, te, nb = _block_energy_grid(X, b)
            tot += te
            diag = torch.diagonal(Eg, 0)
            if Eg_sum is None:
                Eg_sum = torch.zeros_like(Eg)
            Eg_sum += torch.diag(diag)
        Eg_mean = Eg_sum / max(1, len(X_list))
        blocks, eff = _pick_top_blocks(Eg_mean, tot / max(1, len(X_list)), b, cfg.CORE_TARGET, cfg.CORE_MAX_BLOCKS)
        return {"mode": "blockdiag", "shared": blocks, "per": None, "energy_fracs": [eff]}

    if mode == "blocktopk":
        Eg_all = []
        tots = []
        for X in X_list:
            Eg, te, nb = _block_energy_grid(X, b)
            Eg_all.append(Eg)
            tots.append(te)
        Eg_stack = torch.stack(Eg_all, dim=0)
        Eg = Eg_stack.amax(dim=0) if cfg.CORE_AGG == "max" else Eg_stack.mean(dim=0)
        blocks, eff = _pick_top_blocks(Eg, float(np.mean(tots)), b, cfg.CORE_TARGET, cfg.CORE_MAX_BLOCKS)
        return {"mode": "blocktopk", "shared": blocks, "per": None, "energy_fracs": [eff]}

    if mode == "blocktopk_perexpert":
        per = []
        efs = []
        for X in X_list:
            Eg, te, nb = _block_energy_grid(X, b)
            blocks, eff = _pick_top_blocks(Eg, te, b, cfg.CORE_TARGET, cfg.CORE_MAX_BLOCKS)
            per.append(blocks)
            efs.append(eff)
        return {"mode": "blocktopk_perexpert", "shared": None, "per": per, "energy_fracs": efs}

    raise ValueError("CORE_MODE must be blocktopk_perexpert|blocktopk|blockdiag|none")

# ----------------------------
# Residual block pick (fast)
# ----------------------------
@torch.no_grad()
def pick_residual_blocks(R: torch.Tensor, b: int, k: int) -> List[Tuple[int,int,torch.Tensor]]:
    n = R.shape[0]
    nb = (n + b - 1) // b
    Eg, te, _ = _block_energy_grid(R, b)
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    used = torch.zeros((nb, nb), dtype=torch.bool, device=R.device)
    out: List[Tuple[int,int,torch.Tensor]] = []
    for idx in order.tolist():
        if len(out) >= k:
            break
        bi = idx // nb
        bj = idx % nb
        if used[bi, bj]:
            continue
        used[bi, bj] = True
        i0, j0 = bi * b, bj * b
        i1, j1 = min(n, i0 + b), min(n, j0 + b)
        out.append((i0, j0, R[i0:i1, j0:j1].contiguous()))
    return out

# ----------------------------
# Quant pack (optional)
# ----------------------------
@torch.no_grad()
def qpack_int8(t: torch.Tensor) -> Dict[str, Any]:
    x = t.detach().to(torch.float32)
    maxabs = float(x.abs().max().item())
    if maxabs < 1e-12:
        q = torch.zeros_like(x, dtype=torch.int8)
        s = 1.0
    else:
        s = maxabs / 127.0
        q = torch.clamp(torch.round(x / s), -127, 127).to(torch.int8)
    return {"q": q.cpu().numpy(), "s": np.array([s], dtype=np.float32), "shape": np.array(x.shape, dtype=np.int32)}

@torch.no_grad()
def qunpack_int8(obj: Dict[str, Any], device: torch.device) -> torch.Tensor:
    q = torch.from_numpy(obj["q"]).to(torch.int8).to(device)
    s = float(obj["s"][0])
    return (q.to(torch.float32) * s).to(DTYPE_ACC)

def pack_tensor(t: torch.Tensor) -> Any:
    if cfg.QMODE == "int8":
        return qpack_int8(t)
    if cfg.QMODE == "float16":
        return t.to(DTYPE_STORE).cpu().numpy()
    return t  # torch tensor

def unpack_tensor(obj: Any, device: torch.device) -> torch.Tensor:
    if cfg.QMODE == "int8":
        return qunpack_int8(obj, device)
    if cfg.QMODE == "float16":
        return torch.from_numpy(obj).to(DTYPE_ACC).to(device)
    return obj.to(device)

# ----------------------------
# Build payload per cluster
# ----------------------------
@torch.no_grad()
def build_payload_for_cluster(Ws: torch.Tensor, idx: List[int], U: Basis, V: Basis) -> Dict[str, Any]:
    # X_e = U^T W_e V
    X_list = []
    for e in idx:
        W = Ws[e]
        if isinstance(U, DenseBasis) and isinstance(V, DenseBasis):
            X = (U.M.t() @ W @ V.M).contiguous()
        else:
            X = apply_Ut_W_V(W, U, V)
        X_list.append(X)

    core = choose_core_blocks(X_list)
    b = int(cfg.CORE_BLOCK)

    core_blocks: List[List[Tuple[int,int,Any]]] = []
    if core["per"] is not None:
        for X, blocks in zip(X_list, core["per"]):
            lst = []
            for (bi, bj) in blocks:
                i0, j0 = bi*b, bj*b
                i1, j1 = min(X.shape[0], i0+b), min(X.shape[1], j0+b)
                lst.append((i0, j0, pack_tensor(X[i0:i1, j0:j1].contiguous())))
            core_blocks.append(lst)
    else:
        shared = core["shared"] or []
        for X in X_list:
            lst = []
            for (bi, bj) in shared:
                i0, j0 = bi*b, bj*b
                i1, j1 = min(X.shape[0], i0+b), min(X.shape[1], j0+b)
                lst.append((i0, j0, pack_tensor(X[i0:i1, j0:j1].contiguous())))
            core_blocks.append(lst)

    # Residual after core
    R_list = []
    for X, cb in zip(X_list, core_blocks):
        Xc = torch.zeros_like(X)
        for (i0, j0, obj) in cb:
            B = unpack_tensor(obj, X.device)
            h, w = B.shape
            Xc[i0:i0+h, j0:j0+w] = B
        R_list.append((X - Xc).contiguous())

    # Low-rank basis from mean residual
    Rmean = torch.stack(R_list, dim=0).mean(dim=0)
    U_s, _, Vh_s = torch.linalg.svd(Rmean, full_matrices=False)
    r = int(min(cfg.RES_RANK, U_s.shape[1]))
    DL = U_s[:, :r].to(DTYPE_ACC).contiguous()
    DR = Vh_s.t()[:, :r].to(DTYPE_ACC).contiguous()

    coef_list = []
    res_blocks: List[List[Tuple[int,int,Any]]] = []

    for Rm in R_list:
        if cfg.RES_COEF == "diag":
            g = torch.sum(DL * (Rm @ DR), dim=0).contiguous()  # (r,)
            coef_list.append(pack_tensor(g))
            R2 = (Rm - (DL * g.view(1, -1)) @ DR.t()).contiguous()
        else:
            C = (DL.t() @ Rm @ DR).contiguous()  # (r,r)
            coef_list.append(pack_tensor(C))
            R2 = (Rm - (DL @ C @ DR.t())).contiguous()

        blks = pick_residual_blocks(R2, b=int(cfg.RES_BSIZE), k=int(cfg.RES_BLOCKS))
        out = []
        for (i0, j0, B) in blks:
            out.append((i0, j0, pack_tensor(B)))
        res_blocks.append(out)

    return {
        "idx": idx,
        "U": U, "V": V,
        "core": core,
        "core_blocks": core_blocks,
        "DL": pack_tensor(DL),
        "DR": pack_tensor(DR),
        "coef": coef_list,
        "res_blocks": res_blocks,
        "r": r,
    }

# ----------------------------
# Runtime
# ----------------------------
class KTXRuntime:
    def __init__(self, payloads: List[Dict[str, Any]], map_e: Dict[int, Tuple[int,int]], n: int):
        self.payloads = payloads
        self.map_e = map_e
        self.n = n

    @torch.no_grad()
    def apply_one_expert(self, x: torch.Tensor, e: int) -> torch.Tensor:
        m, j = self.map_e[e]
        P = self.payloads[m]
        U: Basis = P["U"]
        V: Basis = P["V"]

        z = U.right(x, transpose=False)           # x @ U
        u = torch.zeros_like(z)

        # core
        for (i0, j0, obj) in P["core_blocks"][j]:
            B = unpack_tensor(obj, z.device)
            h, w = B.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ B

        # low-rank
        DL = unpack_tensor(P["DL"], z.device)
        DR = unpack_tensor(P["DR"], z.device)
        coef = P["coef"][j]
        C = unpack_tensor(coef, z.device)

        if cfg.RES_COEF == "diag":
            g = C
            u += ((z @ DL) * g.view(1, -1)) @ DR.t()
        else:
            u += (z @ DL) @ C @ DR.t()

        # sparse residual blocks
        for (i0, j0, obj) in P["res_blocks"][j]:
            B = unpack_tensor(obj, z.device)
            h, w = B.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ B

        y = V.right(u, transpose=True)            # u @ V^T
        return y

    @torch.no_grad()
    def apply_mixture(self, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
        y = torch.zeros_like(x)
        for a, e in zip(gates.tolist(), routed):
            y += float(a) * self.apply_one_expert(x, e)
        return y

# ----------------------------
# Eval
# ----------------------------
@torch.no_grad()
def eval_per_expert(rt: KTXRuntime, Ws: torch.Tensor, trials: int, batch: int):
    E, n, _ = Ws.shape
    errs = []
    for e in range(E):
        ee = []
        for _ in range(trials):
            x = torch.randn(batch, n, dtype=DTYPE_ACC, device=DEVICE)
            y_ref = x @ Ws[e]
            y_hat = rt.apply_one_expert(x, e)
            num = torch.linalg.norm(y_hat - y_ref)
            den = torch.linalg.norm(y_ref).clamp_min(1e-12)
            ee.append(float((num / den).item()))
        errs.append(float(np.mean(ee)))
    log(f"[eval] per-expert rel-error  mean={float(np.mean(errs)):.6f}  p95={float(np.percentile(errs,95)):.6f}  max={float(np.max(errs)):.6f}")

@torch.no_grad()
def eval_routed(rt: KTXRuntime, Ws: torch.Tensor, trials: int, routed_k: int, batch: int):
    E, n, _ = Ws.shape
    errs = []
    for _ in range(trials):
        x = torch.randn(batch, n, dtype=DTYPE_ACC, device=DEVICE)
        routed = random.sample(range(E), k=min(routed_k, E))
        gates = torch.rand(len(routed), dtype=DTYPE_ACC, device=DEVICE)
        gates = gates / gates.sum().clamp_min(1e-12)

        Wsum = torch.zeros(n, n, dtype=DTYPE_ACC, device=DEVICE)
        for a, e in zip(gates, routed):
            Wsum += float(a.item()) * Ws[e]
        y_ref = x @ Wsum

        y_hat = rt.apply_mixture(x, routed, gates)
        num = torch.linalg.norm(y_hat - y_ref)
        den = torch.linalg.norm(y_ref).clamp_min(1e-12)
        errs.append(float((num / den).item()))
    log(f"[eval] routed rel-error = {float(np.mean(errs)):.6f} ± {float(np.std(errs)):.6f}  (trials={trials})")

# ----------------------------
# Save payload (portable-ish npz)
# ----------------------------
def store_basis(b: Basis) -> Dict[str, Any]:
    if isinstance(b, HadamardPermBasis):
        return {
            "type": "hadamard_perm",
            "perm": b.perm.cpu().numpy().astype(np.int32),
            "inv": b.inv_perm.cpu().numpy().astype(np.int32),
            "sign": b.sign.cpu().numpy().astype(np.float32),
        }
    assert isinstance(b, DenseBasis)
    M = b.M.detach().cpu()
    if cfg.BASIS_STORE_DTYPE == "float16":
        return {"type": "dense", "M": M.to(torch.float16).numpy()}
    return {"type": "dense", "M": M.to(torch.float32).numpy()}

def save_payload(path: str, payloads: List[Dict[str, Any]], map_e: Dict[int, Tuple[int,int]], meta: Dict[str, Any]):
    out: Dict[str, np.ndarray] = {}
    out["meta"] = _encode_meta(meta)
    out["map_e"] = np.array([[k, v[0], v[1]] for k, v in sorted(map_e.items())], dtype=np.int32)

    for mi, P in enumerate(payloads):
        out[f"c{mi}.idx"] = np.array(P["idx"], dtype=np.int32)
        out[f"c{mi}.r"] = np.array([P["r"]], dtype=np.int32)

        Ub = store_basis(P["U"])
        Vb = store_basis(P["V"])
        out[f"c{mi}.U.type"] = np.array([0 if Ub["type"]=="hadamard_perm" else 1], dtype=np.int8)
        out[f"c{mi}.V.type"] = np.array([0 if Vb["type"]=="hadamard_perm" else 1], dtype=np.int8)

        if Ub["type"]=="hadamard_perm":
            out[f"c{mi}.U.perm"] = Ub["perm"]; out[f"c{mi}.U.inv"] = Ub["inv"]; out[f"c{mi}.U.sign"] = Ub["sign"]
        else:
            out[f"c{mi}.U.M"] = Ub["M"]
        if Vb["type"]=="hadamard_perm":
            out[f"c{mi}.V.perm"] = Vb["perm"]; out[f"c{mi}.V.inv"] = Vb["inv"]; out[f"c{mi}.V.sign"] = Vb["sign"]
        else:
            out[f"c{mi}.V.M"] = Vb["M"]

        # store packed tensors as raw arrays (int8 payload stored as q/s/shape triplets)
        def store_packed(prefix: str, obj: Any):
            if cfg.QMODE == "int8":
                out[prefix+".q"] = obj["q"]
                out[prefix+".s"] = obj["s"]
                out[prefix+".sh"] = obj["shape"]
            elif cfg.QMODE == "float16":
                out[prefix+".f16"] = obj
            else:
                out[prefix+".f32"] = obj.detach().cpu().numpy().astype(np.float32)

        # core blocks
        for ej, ex in enumerate(P["core_blocks"]):
            out[f"c{mi}.core.count.e{ej}"] = np.array([len(ex)], dtype=np.int32)
            for bi, (i0, j0, obj) in enumerate(ex):
                out[f"c{mi}.core.pos.e{ej}.b{bi}"] = np.array([i0, j0], dtype=np.int32)
                store_packed(f"c{mi}.core.val.e{ej}.b{bi}", obj)

        store_packed(f"c{mi}.DL", P["DL"])
        store_packed(f"c{mi}.DR", P["DR"])
        for ej, c in enumerate(P["coef"]):
            store_packed(f"c{mi}.coef.e{ej}", c)

        for ej, ex in enumerate(P["res_blocks"]):
            out[f"c{mi}.res.count.e{ej}"] = np.array([len(ex)], dtype=np.int32)
            for bi, (i0, j0, obj) in enumerate(ex):
                out[f"c{mi}.res.pos.e{ej}.b{bi}"] = np.array([i0, j0], dtype=np.int32)
                store_packed(f"c{mi}.res.val.e{ej}.b{bi}", obj)

    save_npz(path, out)
    log(f"[save] payload -> {path}  size={os.path.getsize(path)/1e6:.2f} MB")

# ----------------------------
# Main
# ----------------------------
def banner():
    log("== DeepSeek KT++-X OFFLINE v9 ==")
    log(f"Time:        {now()}")
    log(f"MODEL_DIR:   {cfg.MODEL_DIR}")
    log(f"OUTPUT_DIR:  {cfg.OUTPUT_DIR}")
    log(f"LAYER:       {cfg.LAYER}")
    log(f"MAX_EXPERTS:  {cfg.MAX_EXPERTS}")
    log(f"CALIB_PATH:  {cfg.CALIB_PATH or default_calib_path() or '(none)'}  CALIB_SAMPLES(cap)={cfg.CALIB_SAMPLES}")
    log(f"ROUTER_PATH: {cfg.ROUTER_PATH or '(none)'}  RIDGE_WEIGHTED={cfg.RIDGE_WEIGHTED}")
    log(f"LIN_MODE:    {cfg.LIN_MODE}  RIDGE_DAMP={cfg.RIDGE_DAMP}")
    log(f"BASIS_MODE:  {cfg.BASIS_MODE}")
    log(f"CLUSTER:     M0={cfg.M0 or '(auto)'} M_MAX={cfg.M_MAX} max_size={cfg.CLUSTER_MAX_SIZE} iters={cfg.CLUSTER_ITERS}")
    log(f"TRAIN:       steps={cfg.TRAIN_STEPS} warmup={cfg.TRAIN_WARMUP} lr={cfg.TRAIN_LR} subm={cfg.SUBM} batchE={cfg.BATCH_E}")
    log(f"CORE:        {cfg.CORE_MODE} block={cfg.CORE_BLOCK} target={cfg.CORE_TARGET} max_blocks={cfg.CORE_MAX_BLOCKS}")
    log(f"RESIDUAL:    rank={cfg.RES_RANK} coef={cfg.RES_COEF} blocks={cfg.RES_BLOCKS} bsize={cfg.RES_BSIZE}")
    log(f"QUANT:       QMODE={cfg.QMODE}  BASIS_STORE_DTYPE={cfg.BASIS_STORE_DTYPE}")
    log("")

def main():
    banner()
    eids, Ws = load_or_build_Ws()
    E, n, _ = Ws.shape
    log(f"[Ws] shape={tuple(Ws.shape)}")

    # Clustering
    Xfeat = random_proj_features(Ws, d=cfg.CLUSTER_FEAT_D)

    if cfg.M0 > 0:
        M0 = min(cfg.M0, E)
    else:
        M0 = int(round(2.0 * math.sqrt(E)))
        M0 = max(6, min(M0, E))
    M0 = max(2, min(M0, E))
    labels = kmeans_torch(Xfeat, k=M0, iters=cfg.CLUSTER_ITERS, restarts=cfg.CLUSTER_RESTARTS)
    labels = enforce_min_cluster_size(labels, Xfeat, min_size=cfg.CLUSTER_MIN_SIZE)
    labels = hierarchical_split(Xfeat, labels, max_size=cfg.CLUSTER_MAX_SIZE, max_k=min(cfg.M_MAX, E), split_iters=cfg.SPLIT_ITERS)
    M = int(labels.max().item()) + 1
    clusters = [torch.nonzero(labels == m, as_tuple=False).flatten().tolist() for m in range(M)]
    log(f"[cluster] M={M} sizes={[len(c) for c in clusters]}")

    # Build bases
    payloads: List[Dict[str, Any]] = []
    map_e: Dict[int, Tuple[int,int]] = {}

    if cfg.BASIS_MODE in ("svd", "dense_train"):
        U_par: List[Optional[OrthoParam]] = []
        V_par: List[Optional[OrthoParam]] = []
        for idx in clusters:
            if len(idx) == 0:
                U_par.append(None); V_par.append(None)
                continue
            Wm = Ws[idx].mean(dim=0)
            U0, V0 = svd_init(Wm)
            U_par.append(OrthoParam(U0))
            V_par.append(OrthoParam(V0))

        # Train dense bases
        if cfg.BASIS_MODE == "dense_train" and cfg.TRAIN_STEPS > 0:
            params = [uv.M for uv in (U_par + V_par) if uv is not None]
            opt = torch.optim.Adam(params, lr=cfg.TRAIN_LR)

            guidance_masks: Dict[int, torch.Tensor] = {}
            guidance_stats: Dict[int, Tuple[float,int]] = {}
            t0 = time.perf_counter()

            with torch.enable_grad():
                for step in range(1, cfg.TRAIN_STEPS + 1):
                    S = torch.randperm(n, device=DEVICE)[:min(cfg.SUBM, n)]

                    # refresh guidance
                    if cfg.TRAIN_LAM_GUIDE > 0 and (step % max(1, cfg.TRAIN_GUIDE_EVERY) == 0 or step == 1):
                        with torch.no_grad():
                            guidance_masks.clear()
                            guidance_stats.clear()
                            for m, idx in enumerate(clusters):
                                if len(idx) < cfg.TRAIN_MIN_CLUSTER:
                                    continue
                                Uo = U_par[m].orthogonal()
                                Vo = V_par[m].orthogonal()
                                pick = idx
                                if 0 < cfg.BATCH_E < len(idx):
                                    pidx = torch.randperm(len(idx), device=DEVICE)[:cfg.BATCH_E].tolist()
                                    pick = [idx[i] for i in pidx]
                                Xs_ng = slice_X_batch(Ws[pick], Uo, Vo, S).detach()
                                mask, ef, kblk = make_guidance_mask_from_Xs(
                                    Xs_ng.to(DTYPE_ACC),
                                    block=cfg.CORE_BLOCK,
                                    target=cfg.TRAIN_GUIDE_TARGET,
                                    max_blocks=cfg.TRAIN_GUIDE_MAX_BLOCKS
                                )
                                guidance_masks[m] = mask
                                guidance_stats[m] = (ef, kblk)

                    lam_ramp = schedule(step, cfg.TRAIN_WARMUP, cfg.TRAIN_STEPS)
                    lam_block = cfg.TRAIN_LAM_BLOCK * lam_ramp
                    lam_guide = cfg.TRAIN_LAM_GUIDE * lam_ramp

                    L_total = None
                    terms = 0
                    for m, idx in enumerate(clusters):
                        if len(idx) < cfg.TRAIN_MIN_CLUSTER:
                            continue
                        Uo = U_par[m].orthogonal()
                        Vo = V_par[m].orthogonal()

                        pick = idx
                        if 0 < cfg.BATCH_E < len(idx):
                            pidx = torch.randperm(len(idx), device=DEVICE)[:cfg.BATCH_E].tolist()
                            pick = [idx[i] for i in pidx]

                        Xs = slice_X_batch(Ws[pick], Uo, Vo, S).to(DTYPE_ACC)
                        off = offdiag_abs_mean(Xs)
                        diag = diag_abs_mean(Xs).clamp_min(1e-6)

                        if cfg.TRAIN_OBJ == "ratio":
                            base = off / diag
                        else:
                            base = torch.log(off + 1e-6) - torch.log(diag)

                        if lam_block > 0 and cfg.CORE_MODE.startswith("block"):
                            base = base + lam_block * block_group_sparsity_penalty(Xs, cfg.CORE_BLOCK)

                        if lam_guide > 0 and (m in guidance_masks):
                            Mmask = guidance_masks[m]
                            Etot = (Xs * Xs).mean().clamp_min(1e-12)
                            Eout = ((Xs * (1.0 - Mmask)) ** 2).mean()
                            base = base + lam_guide * (Eout / Etot)

                        L_total = base if (L_total is None) else (L_total + base)
                        terms += 1

                    if L_total is None:
                        break
                    L_total = L_total / max(1, terms)

                    opt.zero_grad(set_to_none=True)
                    L_total.backward()
                    if cfg.GRAD_CLIP > 0:
                        torch.nn.utils.clip_grad_norm_(params, max_norm=cfg.GRAD_CLIP)
                    opt.step()

                    if (step % cfg.REORTHO_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                        with torch.no_grad():
                            for uv in (U_par + V_par):
                                if uv is not None:
                                    uv.M.copy_(uv.orthogonal())

                    if step == 1 or (step % cfg.REPORT_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                        t1 = time.perf_counter()
                        if len(guidance_stats) > 0:
                            ef_mean = float(np.mean([v[0] for v in guidance_stats.values()]))
                            kb_mean = float(np.mean([v[1] for v in guidance_stats.values()]))
                            log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} loss={float(L_total.item()):.4f} "
                                f"lam_block={lam_block:.3f} lam_guide={lam_guide:.3f} "
                                f"guide_energy≈{ef_mean:.3f} guide_blocks≈{kb_mean:.1f} (+{t1-t0:.1f}s)")
                        else:
                            log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} loss={float(L_total.item()):.4f} "
                                f"lam_block={lam_block:.3f} lam_guide={lam_guide:.3f} (+{t1-t0:.1f}s)")
                        t0 = t1

        # Build payloads
        log("[build] payloads ...")
        for m, idx in enumerate(clusters):
            if len(idx) == 0:
                continue
            Uo = U_par[m].orthogonal().detach()
            Vo = V_par[m].orthogonal().detach()
            U = DenseBasis(Uo)
            V = DenseBasis(Vo)

            P = build_payload_for_cluster(Ws, idx, U, V)
            payloads.append(P)
            mi = len(payloads) - 1
            for j, e in enumerate(idx):
                map_e[e] = (mi, j)

            efr = P["core"]["energy_fracs"]
            log(f"  - cluster{mi}: E={len(idx)} core_blocks(mean)≈{float(np.mean([len(x) for x in P['core_blocks']])):.1f} "
                f"core_energy(mean/min)≈{float(np.mean(efr)):.3f}/{float(np.min(efr)):.3f} r={P['r']}")

    elif cfg.BASIS_MODE == "hadamard_perm":
        log("[build] payloads ...")
        for m, idx in enumerate(clusters):
            if len(idx) == 0:
                continue
            U = HadamardPermBasis(n=n, seed=cfg.HAD_SEED + 1000 + m)
            V = HadamardPermBasis(n=n, seed=cfg.HAD_SEED + 2000 + m)
            P = build_payload_for_cluster(Ws, idx, U, V)
            payloads.append(P)
            mi = len(payloads) - 1
            for j, e in enumerate(idx):
                map_e[e] = (mi, j)
            efr = P["core"]["energy_fracs"]
            log(f"  - cluster{mi}: E={len(idx)} core_blocks(mean)≈{float(np.mean([len(x) for x in P['core_blocks']])):.1f} "
                f"core_energy(mean/min)≈{float(np.mean(efr)):.3f}/{float(np.min(efr)):.3f} r={P['r']}")
    else:
        raise ValueError("BASIS_MODE must be hadamard_perm|svd|dense_train")

    rt = KTXRuntime(payloads, map_e, n=n)

    eval_per_expert(rt, Ws, trials=max(2, cfg.EVAL_TRIALS//2), batch=cfg.EVAL_BATCH)
    eval_routed(rt, Ws, trials=cfg.EVAL_TRIALS, routed_k=cfg.ROUTED_K, batch=cfg.EVAL_BATCH)

    # Save
    payload_path = os.path.join(cfg.OUTPUT_DIR, f"ktx_payload_layer{cfg.LAYER}_E{E}_{cfg.BASIS_MODE}_v9_q{cfg.QMODE}.npz")
    meta = dict(
        time=now(), preset=PRESET,
        model_dir=cfg.MODEL_DIR, output_dir=cfg.OUTPUT_DIR,
        layer=cfg.LAYER, expert_ids=eids,
        lin_mode=cfg.LIN_MODE, ridge_damp=cfg.RIDGE_DAMP, ridge_weighted=cfg.RIDGE_WEIGHTED,
        calib_path=(cfg.CALIB_PATH or default_calib_path() or ""),
        router_path=(cfg.ROUTER_PATH or ""),
        basis_mode=cfg.BASIS_MODE,
        cluster=dict(M0=cfg.M0, M_MAX=cfg.M_MAX, max_size=cfg.CLUSTER_MAX_SIZE),
        train=dict(steps=cfg.TRAIN_STEPS, warmup=cfg.TRAIN_WARMUP, lr=cfg.TRAIN_LR,
                   lam_block=cfg.TRAIN_LAM_BLOCK, lam_guide=cfg.TRAIN_LAM_GUIDE,
                   guide_every=cfg.TRAIN_GUIDE_EVERY, guide_target=cfg.TRAIN_GUIDE_TARGET),
        core=dict(mode=cfg.CORE_MODE, block=cfg.CORE_BLOCK, target=cfg.CORE_TARGET, max_blocks=cfg.CORE_MAX_BLOCKS),
        residual=dict(rank=cfg.RES_RANK, coef=cfg.RES_COEF, blocks=cfg.RES_BLOCKS, bsize=cfg.RES_BSIZE),
        qmode=cfg.QMODE, basis_store_dtype=cfg.BASIS_STORE_DTYPE,
        seed=SEED, device=str(DEVICE), H=n,
    )
    save_payload(payload_path, payloads, map_e, meta)

    log("\n✅ Done.")
    log("If you still see core_blocks pegged at CORE_MAX_BLOCKS:")
    log("  - switch to BASIS_MODE=dense_train (not hadamard_perm)")
    log("  - set CLUSTER_MAX_SIZE=2 and M_MAX=16")
    log("  - increase TRAIN_LAM_GUIDE and TRAIN_GUIDE_EVERY=1")
    log("For true 99–100% accuracy you also need more X rows than 160 (capture more tokens).")

if __name__ == "__main__":
    main()


== DeepSeek KT++-X OFFLINE v9 ==
Time:        2026-01-13 10:56:01
MODEL_DIR:   /home/daniyar/deepseek-model
OUTPUT_DIR:  /home/daniyar/moe_ws_outputs
LAYER:       1
MAX_EXPERTS:  16
CALIB_PATH:  /home/daniyar/moe_ws_outputs/calib_layer1_X.npz  CALIB_SAMPLES(cap)=4096
ROUTER_PATH: (none)  RIDGE_WEIGHTED=False
LIN_MODE:    ridge  RIDGE_DAMP=0.001
BASIS_MODE:  dense_train
CLUSTER:     M0=(auto) M_MAX=16 max_size=3 iters=60
TRAIN:       steps=24 warmup=6 lr=0.05 subm=256 batchE=4
CORE:        blocktopk_perexpert block=64 target=0.85 max_blocks=256
RESIDUAL:    rank=512 coef=diag blocks=192 bsize=64
QUANT:       QMODE=none  BASIS_STORE_DTYPE=float16

[load] reading tensors from shards ...
[shape] H=2048 d_ff=1408
[calib] X loaded: (160, 2048)


Build Ws (ridge):   0%|          | 0/16 [00:00<?, ?it/s]

[cache] wrote Ws -> /home/daniyar/moe_ws_outputs/Ws_cache_layer1_E16_ridge_v9.npz  size=268.44 MB
[Ws] shape=(16, 2048, 2048)
[cluster] M=11 sizes=[1, 1, 1, 1, 1, 2, 1, 1, 1, 3, 3]
[train] step   1/24 loss=-3.6046 lam_block=0.000 lam_guide=0.000 guide_energy≈0.871 guide_blocks≈4.0 (+3.7s)
[train] step   4/24 loss=-1.2530 lam_block=0.000 lam_guide=0.000 guide_energy≈0.755 guide_blocks≈9.7 (+12.9s)
[train] step   8/24 loss=-1.5404 lam_block=0.011 lam_guide=0.111 guide_energy≈0.775 guide_blocks≈7.0 (+15.3s)
[train] step  12/24 loss=-1.2840 lam_block=0.033 lam_guide=0.333 guide_energy≈0.824 guide_blocks≈4.0 (+15.2s)
[train] step  16/24 loss=-0.0844 lam_block=0.056 lam_guide=0.556 guide_energy≈0.771 guide_blocks≈8.7 (+15.0s)
[train] step  20/24 loss=2.2563 lam_block=0.078 lam_guide=0.778 guide_energy≈0.780 guide_blocks≈10.7 (+14.9s)
[train] step  24/24 loss=1.4931 lam_block=0.100 lam_guide=1.000 guide_energy≈0.769 guide_blocks≈8.0 (+14.9s)
[build] payloads ...
  - cluster0: E=1 core_blocks(

In [ ]:
#BEST RESULTS

In [14]:
#!/usr/bin/env python3
# ============================================================
# DeepSeek KT++-X OFFLINE v10
# - dense_train basis (best accuracy)
# - improved clustering (kmeans++ restarts + merge tiny clusters)
# - adaptive refinement: add extra residual blocks to hard experts
#
# NOTE: Your main limiter is still CALIB X rows (you have 160).
# ============================================================

import os, re, json, math, time, random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any, Set

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors import safe_open

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):  # type: ignore
        return x

# ----------------------------
# Threads / determinism
# ----------------------------
NTHREADS = int(os.environ.get("KTXX_THREADS", "8"))
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(NTHREADS))
try:
    torch.set_num_threads(NTHREADS)
except Exception:
    pass

SEED = int(os.environ.get("SEED", "1234"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(os.environ.get("DEVICE", "cpu"))
DTYPE_ACC = torch.float32
DTYPE_STORE = torch.float16

def now():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def log(msg: str):
    print(msg, flush=True)

# ----------------------------
# Presets
# ----------------------------
PRESET = os.environ.get("PRESET", "").strip().lower()

def env(key: str, default: str) -> str:
    return os.environ.get(key, default)

# ----------------------------
# Config
# ----------------------------
@dataclass
class Cfg:
    MODEL_DIR: str = env("MODEL_DIR", "/home/daniyar/deepseek-model")
    OUTPUT_DIR: str = env("OUTPUT_DIR", "/home/daniyar/moe_ws_outputs")
    LAYER: int = int(env("LAYER", "1"))
    MAX_EXPERTS: int = int(env("MAX_EXPERTS", "16"))

    CALIB_PATH: str = env("CALIB_PATH", "").strip()
    CALIB_SAMPLES: int = int(env("CALIB_SAMPLES", "4096"))

    ROUTER_PATH: str = env("ROUTER_PATH", "").strip()
    RIDGE_WEIGHTED: bool = env("RIDGE_WEIGHTED", "0") == "1"
    ROUTER_EIDS_ARE_GLOBAL: bool = env("ROUTER_EIDS_ARE_GLOBAL", "1") == "1"

    LIN_MODE: str = env("LIN_MODE", "ridge").strip().lower()
    RIDGE_DAMP: float = float(env("RIDGE_DAMP", "1e-3"))
    NORMALIZE_W: bool = env("NORMALIZE_W", "1") == "1"

    BASIS_MODE: str = env("BASIS_MODE", "dense_train").strip().lower()
    BASIS_STORE_DTYPE: str = env("BASIS_STORE_DTYPE", "float16").strip().lower()  # float16|float32

    # Clustering
    M0: int = int(env("M0", "0"))          # 0 => auto
    M_MAX: int = int(env("M_MAX", "16"))
    CLUSTER_FEAT_D: int = int(env("CLUSTER_FEAT_D", "64"))
    CLUSTER_ITERS: int = int(env("CLUSTER_ITERS", "60"))
    CLUSTER_RESTARTS: int = int(env("CLUSTER_RESTARTS", "4"))
    CLUSTER_MIN_SIZE: int = int(env("CLUSTER_MIN_SIZE", "2"))     # v10 default: avoid singletons
    CLUSTER_MAX_SIZE: int = int(env("CLUSTER_MAX_SIZE", "4"))     # allow slightly bigger groups
    SPLIT_ITERS: int = int(env("SPLIT_ITERS", "50"))
    MERGE_TINY: bool = env("MERGE_TINY", "1") == "1"
    MERGE_TINY_MAX: int = int(env("MERGE_TINY_MAX", "2"))         # merge clusters of size <= this

    # Training
    TRAIN_STEPS: int = int(env("TRAIN_STEPS", "24"))
    TRAIN_WARMUP: int = int(env("TRAIN_WARMUP", "6"))
    TRAIN_LR: float = float(env("TRAIN_LR", "5e-2"))
    SUBM: int = int(env("SUBM", "256"))
    BATCH_E: int = int(env("BATCH_E", "4"))
    TRAIN_MIN_CLUSTER: int = int(env("TRAIN_MIN_CLUSTER", "2"))
    REORTHO_EVERY: int = int(env("REORTHO_EVERY", "4"))
    REPORT_EVERY: int = int(env("REPORT_EVERY", "4"))
    GRAD_CLIP: float = float(env("GRAD_CLIP", "1.0"))

    TRAIN_OBJ: str = env("TRAIN_OBJ", "logratio").strip().lower()
    TRAIN_LAM_BLOCK: float = float(env("TRAIN_LAM_BLOCK", "0.10"))
    TRAIN_LAM_GUIDE: float = float(env("TRAIN_LAM_GUIDE", "1.0"))
    TRAIN_GUIDE_EVERY: int = int(env("TRAIN_GUIDE_EVERY", "2"))
    TRAIN_GUIDE_TARGET: float = float(env("TRAIN_GUIDE_TARGET", "0.75"))
    TRAIN_GUIDE_MAX_BLOCKS: int = int(env("TRAIN_GUIDE_MAX_BLOCKS", "256"))

    # Core / residual
    CORE_MODE: str = env("CORE_MODE", "blocktopk_perexpert").strip().lower()
    CORE_AGG: str = env("CORE_AGG", "mean").strip().lower()
    CORE_BLOCK: int = int(env("CORE_BLOCK", "64"))
    CORE_TARGET: float = float(env("CORE_TARGET", "0.85"))
    CORE_MAX_BLOCKS: int = int(env("CORE_MAX_BLOCKS", "256"))

    RES_RANK: int = int(env("RES_RANK", "512"))
    RES_COEF: str = env("RES_COEF", "diag").strip().lower()    # diag|full
    RES_BLOCKS: int = int(env("RES_BLOCKS", "192"))
    RES_BSIZE: int = int(env("RES_BSIZE", "64"))

    # Adaptive refinement
    REFINE_ENABLE: bool = env("REFINE_ENABLE", "1") == "1"
    REFINE_ERR_TARGET: float = float(env("REFINE_ERR_TARGET", "0.05"))    # try push per-expert <5%
    REFINE_MAX_EXTRA_BLOCKS: int = int(env("REFINE_MAX_EXTRA_BLOCKS", "256"))
    REFINE_BSIZE: int = int(env("REFINE_BSIZE", "64"))

    # Quant
    QMODE: str = env("QMODE", "none").strip().lower()  # none|float16|int8

    # Eval
    EVAL_TRIALS: int = int(env("EVAL_TRIALS", "8"))
    EVAL_BATCH: int = int(env("EVAL_BATCH", "2"))
    ROUTED_K: int = int(env("ROUTED_K", "8"))

cfg = Cfg()

if PRESET == "ultra":
    os.environ.setdefault("TRAIN_STEPS", "64")
    os.environ.setdefault("TRAIN_WARMUP", "12")
    os.environ.setdefault("TRAIN_LAM_GUIDE", "2.0")
    os.environ.setdefault("TRAIN_GUIDE_EVERY", "1")
    os.environ.setdefault("CORE_BLOCK", "32")
    os.environ.setdefault("CORE_TARGET", "0.92")
    os.environ.setdefault("CORE_MAX_BLOCKS", "512")
    os.environ.setdefault("RES_RANK", "1024")
    os.environ.setdefault("RES_BLOCKS", "512")
    os.environ.setdefault("RIDGE_DAMP", "1e-2")
    cfg = Cfg()

elif PRESET == "compact":
    os.environ.setdefault("QMODE", "float16")
    os.environ.setdefault("CORE_BLOCK", "64")
    os.environ.setdefault("CORE_TARGET", "0.85")
    os.environ.setdefault("RES_RANK", "512")
    os.environ.setdefault("RES_BLOCKS", "192")
    cfg = Cfg()

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# ----------------------------
# NPZ helpers
# ----------------------------
def save_npz(path: str, arrays: Dict[str, np.ndarray]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez(path, **arrays)

def try_load_npz(path: str) -> Optional[Dict[str, np.ndarray]]:
    if not os.path.isfile(path):
        return None
    z = np.load(path, allow_pickle=False)
    out = {k: z[k] for k in z.files}
    z.close()
    return out

def _encode_meta(meta: dict) -> np.ndarray:
    b = json.dumps(meta, sort_keys=True).encode("utf-8")
    return np.frombuffer(b, dtype=np.uint8)

def _decode_meta(arr: np.ndarray) -> dict:
    try:
        b = bytes(arr.tolist())
        return json.loads(b.decode("utf-8"))
    except Exception:
        return {}

# ----------------------------
# Offline shard loading
# ----------------------------
def read_index(model_dir: str) -> Dict[str, str]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    wm = obj.get("weight_map", {})
    if not wm:
        raise RuntimeError("Index JSON has empty weight_map.")
    return wm

def find_layer_expert_ids(weight_map: Dict[str, str], layer: int) -> List[int]:
    pat = re.compile(rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.")
    ids = set()
    for k in weight_map.keys():
        m = pat.match(k)
        if m:
            ids.add(int(m.group(1)))
    return sorted(ids)

def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    prefix = f"model.layers.{layer}.mlp.experts.{eid}."
    def pick(cands: List[str]) -> Optional[str]:
        for suf in cands:
            k = prefix + suf
            if k in weight_map:
                return k
        return None
    up   = pick(["up_proj.weight", "w3.weight", "w1.weight"])
    gate = pick(["gate_proj.weight", "w1.weight", "w3.weight"])
    down = pick(["down_proj.weight", "w2.weight"])
    if up is None or gate is None or down is None:
        return {}
    if up == gate:
        g2 = pick(["gate_proj.weight"])
        if g2:
            gate = g2
    return {"up": up, "gate": gate, "down": down}

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], keys: List[str]) -> Dict[str, torch.Tensor]:
    by_shard: Dict[str, List[str]] = {}
    for k in keys:
        shard = weight_map.get(k, None)
        if shard is None:
            raise KeyError(f"Key not in weight_map: {k}")
        by_shard.setdefault(shard, []).append(k)

    out: Dict[str, torch.Tensor] = {}
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(sp):
            raise FileNotFoundError(f"Missing shard: {sp}")
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks:
                out[k] = f.get_tensor(k)
    return out

# ----------------------------
# Calibration / router
# ----------------------------
def load_calib_X(H: int) -> torch.Tensor:
    if not cfg.CALIB_PATH:
        raise RuntimeError("CALIB_PATH is required for ridge mode.")
    z = np.load(cfg.CALIB_PATH, allow_pickle=False)
    if "X" not in z.files:
        raise KeyError(f"CALIB npz missing key 'X'. Keys={list(z.files)}")
    X = torch.from_numpy(z["X"]).to(DTYPE_ACC)
    z.close()
    if X.ndim != 2 or X.shape[1] != H:
        raise RuntimeError(f"Bad X shape: {tuple(X.shape)} expected (*,{H})")
    if X.shape[0] > cfg.CALIB_SAMPLES:
        X = X[:cfg.CALIB_SAMPLES]
    return X.to(DEVICE)

def load_router_P() -> Optional[np.ndarray]:
    if not cfg.ROUTER_PATH:
        return None
    z = np.load(cfg.ROUTER_PATH, allow_pickle=False)
    if "P" not in z.files:
        raise KeyError(f"ROUTER npz missing key 'P'. Keys={list(z.files)}")
    P = z["P"].astype(np.float32, copy=False)
    z.close()
    return P

# ----------------------------
# Build Ws (ridge/weff)
# ----------------------------
@torch.no_grad()
def forward_mlp(X: torch.Tensor, W_gate: torch.Tensor, W_up: torch.Tensor, W_down: torch.Tensor) -> torch.Tensor:
    Xf = X.to(DTYPE_ACC)
    up = Xf @ W_up.to(DTYPE_ACC).t()
    gate = Xf @ W_gate.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    y = hid @ W_down.to(DTYPE_ACC).t()
    return y

def ws_cache_path(e: int) -> str:
    return os.path.join(cfg.OUTPUT_DIR, f"Ws_cache_layer{cfg.LAYER}_E{e}_{cfg.LIN_MODE}_v10.npz")

def ws_meta(eids: List[int]) -> dict:
    return dict(
        model_dir=cfg.MODEL_DIR,
        layer=cfg.LAYER,
        eids=eids,
        lin_mode=cfg.LIN_MODE,
        ridge_damp=cfg.RIDGE_DAMP,
        ridge_weighted=cfg.RIDGE_WEIGHTED,
        calib_path=cfg.CALIB_PATH,
        router_path=(cfg.ROUTER_PATH or ""),
        normalize_w=cfg.NORMALIZE_W,
        seed=SEED,
    )

@torch.no_grad()
def load_or_build_Ws() -> Tuple[List[int], torch.Tensor]:
    wm = read_index(cfg.MODEL_DIR)
    all_eids = find_layer_expert_ids(wm, cfg.LAYER)
    if not all_eids:
        raise RuntimeError(f"No experts found for layer={cfg.LAYER}")
    eids = all_eids[:cfg.MAX_EXPERTS]

    cpath = ws_cache_path(len(eids))
    z = try_load_npz(cpath)
    if z and ("Ws" in z) and ("expert_ids" in z) and ("meta" in z):
        if _decode_meta(z["meta"]) == ws_meta(eids):
            Ws = torch.from_numpy(z["Ws"]).to(DTYPE_ACC).to(DEVICE)
            log(f"[cache] loaded Ws: {cpath}  Ws={tuple(Ws.shape)}")
            return [int(x) for x in z["expert_ids"].tolist()], Ws
        log("[cache] Ws meta mismatch -> rebuilding.")

    per_e = {}
    need_keys = []
    for eid in eids:
        kk = pick_expert_tensor_keys(wm, cfg.LAYER, eid)
        if not kk:
            raise RuntimeError(f"Expert {eid} missing tensors in index.")
        per_e[eid] = kk
        need_keys += [kk["up"], kk["down"], kk["gate"]]

    log("[load] reading tensors from shards ...")
    T = load_tensors_from_shards(cfg.MODEL_DIR, wm, sorted(set(need_keys)))

    W_up0 = T[per_e[eids[0]]["up"]]
    dff, H = int(W_up0.shape[0]), int(W_up0.shape[1])
    log(f"[shape] H={H} d_ff={dff}")

    X = load_calib_X(H)
    log(f"[calib] X loaded: {tuple(X.shape)}")
    P = load_router_P()
    if cfg.RIDGE_WEIGHTED and (P is None):
        log("[router] missing ROUTER_PATH -> forcing RIDGE_WEIGHTED=0")
        cfg.RIDGE_WEIGHTED = False

    I = torch.eye(H, dtype=DTYPE_ACC, device=DEVICE)
    Xf = X.to(DTYPE_ACC)
    XtX = Xf.t() @ Xf
    lam = cfg.RIDGE_DAMP * float(torch.trace(XtX).item()) / float(H)
    cholG = torch.linalg.cholesky(XtX + lam * I)

    Ws_list: List[torch.Tensor] = []
    for i, eid in enumerate(tqdm(eids, desc=f"Build Ws ({cfg.LIN_MODE})")):
        W_up   = T[per_e[eid]["up"]].to(DEVICE)
        W_down = T[per_e[eid]["down"]].to(DEVICE)
        W_gate = T[per_e[eid]["gate"]].to(DEVICE)

        if cfg.LIN_MODE == "ridge":
            Y = forward_mlp(X, W_gate, W_up, W_down).to(DTYPE_ACC)
            if cfg.RIDGE_WEIGHTED and (P is not None):
                if cfg.ROUTER_EIDS_ARE_GLOBAL:
                    w = torch.from_numpy(P[:X.shape[0], eid]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
                else:
                    w = torch.from_numpy(P[:X.shape[0], i]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
                sw = torch.sqrt(w + 1e-12).view(-1, 1)
                Xw = Xf * sw
                Yw = Y * sw
                XtX_e = Xw.t() @ Xw
                lam_e = cfg.RIDGE_DAMP * float(torch.trace(XtX_e).item()) / float(H)
                chol = torch.linalg.cholesky(XtX_e + lam_e * I)
                XtY = Xw.t() @ Yw
                Wt = torch.cholesky_solve(XtY, chol)
                W = Wt.t().contiguous()
            else:
                XtY = Xf.t() @ Y
                Wt = torch.cholesky_solve(XtY, cholG)
                W = Wt.t().contiguous()
        else:
            raise ValueError("v10 code focuses on ridge. Use v9 if you need weff/weff_gate.")

        if cfg.NORMALIZE_W:
            fn = torch.linalg.norm(W, ord="fro").clamp_min(1e-12)
            W = (W / fn).contiguous()
        Ws_list.append(W)

    Ws = torch.stack(Ws_list, dim=0).to(DTYPE_ACC).to(DEVICE)

    save_npz(cpath, {
        "meta": _encode_meta(ws_meta(eids)),
        "expert_ids": np.array(eids, dtype=np.int32),
        "Ws": Ws.detach().cpu().numpy().astype(np.float32),
    })
    log(f"[cache] wrote Ws -> {cpath}  size={os.path.getsize(cpath)/1e6:.2f} MB")
    return eids, Ws

# ----------------------------
# Clustering
# ----------------------------
@torch.no_grad()
def random_proj_features(Ws: torch.Tensor, d: int) -> torch.Tensor:
    E, n, _ = Ws.shape
    g = torch.Generator(device="cpu").manual_seed(SEED)
    R = (torch.randint(0, 2, (n, d), generator=g, dtype=torch.int8) * 2 - 1).to(DTYPE_ACC).to(DEVICE)
    feats = []
    for e in range(E):
        W = Ws[e]
        row = torch.diag(W @ W.t())
        col = torch.diag(W.t() @ W)
        feats.append(torch.cat([(row @ R), (col @ R)], dim=0).unsqueeze(0))
    X = torch.cat(feats, dim=0)
    X = (X - X.mean(dim=0, keepdim=True)) / (X.std(dim=0, keepdim=True) + 1e-6)
    return X

@torch.no_grad()
def kmeanspp_init(X: torch.Tensor, k: int) -> torch.Tensor:
    g = torch.Generator(device=X.device).manual_seed(SEED)
    n = X.shape[0]
    centers = []
    idx0 = torch.randint(0, n, (1,), generator=g, device=X.device).item()
    centers.append(X[idx0].clone())
    for _ in range(1, k):
        C = torch.stack(centers, dim=0)
        dist2 = torch.cdist(X, C).pow(2).min(dim=1).values
        prob = dist2 / dist2.sum().clamp_min(1e-12)
        idx = torch.multinomial(prob, num_samples=1, generator=g).item()
        centers.append(X[idx].clone())
    return torch.stack(centers, dim=0)

@torch.no_grad()
def kmeans_torch(X: torch.Tensor, k: int, iters: int, restarts: int) -> torch.Tensor:
    best_lab = None
    best_inertia = float("inf")
    for _ in range(max(1, restarts)):
        C = kmeanspp_init(X, k)
        for _ in range(iters):
            dist = torch.cdist(X, C)
            lab = dist.argmin(dim=1)
            for j in range(k):
                m = (lab == j)
                if m.any():
                    C[j] = X[m].mean(dim=0)
                else:
                    far = dist.min(dim=1).values.argmax().item()
                    C[j] = X[far].clone()
        inertia = float((torch.cdist(X, C).min(dim=1).values ** 2).sum().item())
        if inertia < best_inertia:
            best_inertia = inertia
            best_lab = lab.clone()
    assert best_lab is not None
    return best_lab.to(torch.int64)

@torch.no_grad()
def relabel_contiguous(labels: torch.Tensor) -> torch.Tensor:
    labels = labels.to(torch.int64)
    uniq = torch.unique(labels)
    out = labels.clone()
    for new, old in enumerate(uniq.tolist()):
        out[labels == int(old)] = int(new)
    return out

@torch.no_grad()
def enforce_min_cluster_size(labels: torch.Tensor, X: torch.Tensor, min_size: int) -> torch.Tensor:
    if min_size <= 1:
        return relabel_contiguous(labels)
    labels = relabel_contiguous(labels)
    k = int(labels.max().item()) + 1
    counts = torch.bincount(labels, minlength=k)
    big = (counts >= min_size).nonzero(as_tuple=False).flatten()
    small = (counts < min_size).nonzero(as_tuple=False).flatten()
    if big.numel() == 0 or small.numel() == 0:
        return labels
    big_centers = torch.stack([X[labels == j].mean(dim=0) for j in big.tolist()], dim=0)
    for c in small.tolist():
        idxs = (labels == c).nonzero(as_tuple=False).flatten()
        if idxs.numel() == 0:
            continue
        d = torch.cdist(X[idxs], big_centers)
        nn = d.argmin(dim=1)
        labels[idxs] = big[nn]
    return relabel_contiguous(labels)

@torch.no_grad()
def hierarchical_split(X: torch.Tensor, labels: torch.Tensor, max_size: int, max_k: int, split_iters: int) -> torch.Tensor:
    labels = relabel_contiguous(labels)
    if max_size <= 0:
        return labels
    while True:
        K = int(labels.max().item()) + 1
        if K >= max_k:
            break
        counts = torch.bincount(labels, minlength=K)
        biggest = int(torch.argmax(counts).item())
        bigsz = int(counts[biggest].item())
        if bigsz <= max_size:
            break
        idxs = (labels == biggest).nonzero(as_tuple=False).flatten()
        if idxs.numel() < 2:
            break
        sub = X[idxs]
        sub_lab = kmeans_torch(sub, k=2, iters=split_iters, restarts=1)
        a = idxs[sub_lab == 0]
        b = idxs[sub_lab == 1]
        if a.numel() == 0 or b.numel() == 0:
            break
        labels[b] = K
        labels = relabel_contiguous(labels)
    return labels

@torch.no_grad()
def merge_tiny_clusters(X: torch.Tensor, labels: torch.Tensor, max_tiny: int) -> torch.Tensor:
    """
    Merge clusters with size <= max_tiny into nearest non-tiny cluster by center distance.
    """
    labels = relabel_contiguous(labels)
    K = int(labels.max().item()) + 1
    counts = torch.bincount(labels, minlength=K)

    # compute centers
    centers = []
    for k in range(K):
        idx = (labels == k).nonzero(as_tuple=False).flatten()
        centers.append(X[idx].mean(dim=0))
    C = torch.stack(centers, dim=0)

    tiny = (counts <= max_tiny).nonzero(as_tuple=False).flatten().tolist()
    big  = (counts >  max_tiny).nonzero(as_tuple=False).flatten().tolist()
    if len(tiny) == 0 or len(big) == 0:
        return labels

    C_big = C[big]
    for t in tiny:
        idx = (labels == t).nonzero(as_tuple=False).flatten()
        if idx.numel() == 0:
            continue
        d = torch.cdist(X[idx], C_big)
        nn = d.argmin(dim=1)
        labels[idx] = torch.tensor([big[i] for i in nn.tolist()], device=labels.device, dtype=labels.dtype)

    return relabel_contiguous(labels)

# ----------------------------
# Training (dense bases)
# ----------------------------
class OrthoParam(nn.Module):
    def __init__(self, M_init: torch.Tensor):
        super().__init__()
        self.M = nn.Parameter(M_init.clone().to(DEVICE))
    def orthogonal(self) -> torch.Tensor:
        Q, _ = torch.linalg.qr(self.M)
        return Q

@torch.no_grad()
def svd_init(Wm: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    U, _, Vh = torch.linalg.svd(Wm, full_matrices=False)
    return U.to(DTYPE_ACC).contiguous(), Vh.t().to(DTYPE_ACC).contiguous()

def schedule(step: int, warmup: int, total: int) -> float:
    if total <= 0:
        return 1.0
    if step <= warmup:
        return 0.0
    return float(min(1.0, max(0.0, (step - warmup) / max(1, (total - warmup)))))

def slice_X_batch(Ws_batch: torch.Tensor, U: torch.Tensor, V: torch.Tensor, S: torch.Tensor) -> torch.Tensor:
    U_S = U[:, S]
    V_S = V[:, S]
    T = Ws_batch @ V_S
    Xs = torch.matmul(U_S.t().unsqueeze(0), T)
    return Xs

def offdiag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    Xoff = Xs - torch.diag_embed(D)
    return Xoff.abs().mean()

def diag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    return D.abs().mean()

def block_group_sparsity_penalty(Xs: torch.Tensor, block: int) -> torch.Tensor:
    Eb, s, _ = Xs.shape
    b = int(block)
    nb = s // b
    if b <= 0 or nb <= 0:
        return torch.zeros((), dtype=DTYPE_ACC, device=Xs.device)
    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()
    Xb = X.view(Eb, nb, b, nb, b).permute(0, 1, 3, 2, 4).contiguous()
    Eblk = (Xb * Xb).sum(dim=(3,4))
    P = Eblk.mean(dim=0)
    return torch.sqrt(P + 1e-12).sum() / (P.sum() + 1e-12)

@torch.no_grad()
def make_guidance_mask_from_Xs(Xs: torch.Tensor, block: int, target: float, max_blocks: int) -> Tuple[torch.Tensor, float, int]:
    Eb, s, _ = Xs.shape
    b = int(block)
    nb = s // b
    if b <= 0 or nb <= 0:
        return torch.ones(s, s, dtype=DTYPE_ACC, device=Xs.device), 1.0, 0
    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()
    Xb = X.view(Eb, nb, b, nb, b).permute(0, 1, 3, 2, 4).contiguous()
    Eg = (Xb * Xb).sum(dim=(3,4)).mean(dim=0)
    tot = float((X * X).sum().item()) / max(1, Eb)

    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    csum = torch.cumsum(flat[order], dim=0)
    frac = csum / max(tot, 1e-12)
    need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else flat.numel())
    K = min(need, max_blocks, flat.numel())
    pick = order[:K]

    mask = torch.zeros(s2, s2, dtype=DTYPE_ACC, device=Xs.device)
    for idx in pick.tolist():
        bi = idx // nb
        bj = idx % nb
        i0 = bi * b
        j0 = bj * b
        mask[i0:i0+b, j0:j0+b] = 1.0

    if s2 < s:
        full = torch.zeros(s, s, dtype=DTYPE_ACC, device=Xs.device)
        full[:s2, :s2] = mask
        mask = full

    ef = float(frac[K-1].item()) if K > 0 else 0.0
    return mask, ef, int(K)

# ----------------------------
# Core + residual building
# ----------------------------
@torch.no_grad()
def _block_energy_grid(X: torch.Tensor, b: int) -> Tuple[torch.Tensor, float, int]:
    n = X.shape[0]
    nb = (n + b - 1) // b
    Xp = X
    if (n % b) != 0:
        Xp = torch.zeros(nb*b, nb*b, dtype=X.dtype, device=X.device)
        Xp[:n, :n] = X
    Xb = Xp.view(nb, b, nb, b).permute(0, 2, 1, 3).contiguous()
    Eg = (Xb * Xb).sum(dim=(2,3))
    tot = float((Xp * Xp).sum().item())
    return Eg, tot, nb

@torch.no_grad()
def _pick_top_blocks(Eg: torch.Tensor, tot_energy: float, target: float, max_blocks: int) -> Tuple[List[int], float]:
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    csum = torch.cumsum(flat[order], dim=0)
    frac = csum / max(tot_energy, 1e-12)
    need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else flat.numel())
    K = min(need, max_blocks, flat.numel())
    pick = order[:K].tolist()
    eff = float(frac[K-1].item()) if K > 0 else 0.0
    return pick, eff

@torch.no_grad()
def choose_core_blocks(X_list: List[torch.Tensor]) -> Dict[str, Any]:
    mode = cfg.CORE_MODE
    b = int(cfg.CORE_BLOCK)
    if mode == "none":
        return {"mode": "none", "per": None, "energy_fracs": []}

    if mode != "blocktopk_perexpert":
        raise ValueError("v10 focuses on blocktopk_perexpert for best accuracy.")

    per = []
    efs = []
    for X in X_list:
        Eg, te, nb = _block_energy_grid(X, b)
        pick, eff = _pick_top_blocks(Eg, te, cfg.CORE_TARGET, cfg.CORE_MAX_BLOCKS)
        # store flattened indices; decode later
        per.append((pick, nb))
        efs.append(eff)
    return {"mode": "blocktopk_perexpert", "per": per, "energy_fracs": efs}

@torch.no_grad()
def pick_residual_blocks(R: torch.Tensor, b: int, k: int, avoid: Set[Tuple[int,int]]) -> List[Tuple[int,int,torch.Tensor]]:
    n = R.shape[0]
    Eg, te, nb = _block_energy_grid(R, b)
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    out: List[Tuple[int,int,torch.Tensor]] = []
    for idx in order.tolist():
        if len(out) >= k:
            break
        bi = idx // nb
        bj = idx % nb
        i0, j0 = bi*b, bj*b
        if (i0, j0) in avoid:
            continue
        i1, j1 = min(n, i0+b), min(n, j0+b)
        out.append((i0, j0, R[i0:i1, j0:j1].contiguous()))
        avoid.add((i0, j0))
    return out

# ----------------------------
# Quant pack
# ----------------------------
@torch.no_grad()
def qpack_int8(t: torch.Tensor) -> Dict[str, Any]:
    x = t.detach().to(torch.float32)
    maxabs = float(x.abs().max().item())
    if maxabs < 1e-12:
        q = torch.zeros_like(x, dtype=torch.int8)
        s = 1.0
    else:
        s = maxabs / 127.0
        q = torch.clamp(torch.round(x / s), -127, 127).to(torch.int8)
    return {"q": q.cpu().numpy(), "s": np.array([s], dtype=np.float32), "shape": np.array(x.shape, dtype=np.int32)}

@torch.no_grad()
def qunpack_int8(obj: Dict[str, Any], device: torch.device) -> torch.Tensor:
    q = torch.from_numpy(obj["q"]).to(torch.int8).to(device)
    s = float(obj["s"][0])
    return (q.to(torch.float32) * s).to(DTYPE_ACC)

def pack_tensor(t: torch.Tensor) -> Any:
    if cfg.QMODE == "int8":
        return qpack_int8(t)
    if cfg.QMODE == "float16":
        return t.to(DTYPE_STORE).cpu().numpy()
    return t.detach().cpu().numpy().astype(np.float32)

def unpack_tensor(obj: Any, device: torch.device) -> torch.Tensor:
    if cfg.QMODE == "int8":
        return qunpack_int8(obj, device)
    if cfg.QMODE == "float16":
        return torch.from_numpy(obj).to(DTYPE_ACC).to(device)
    return torch.from_numpy(obj).to(DTYPE_ACC).to(device)

# ----------------------------
# Basis (dense)
# ----------------------------
class DenseBasis:
    def __init__(self, M: torch.Tensor):
        self.M = M.to(DEVICE).to(DTYPE_ACC).contiguous()

    def right(self, x: torch.Tensor, transpose: bool = False) -> torch.Tensor:
        return x @ (self.M.t() if transpose else self.M)

# ----------------------------
# Build payload per cluster + refine
# ----------------------------
@torch.no_grad()
def build_payload_for_cluster(Ws: torch.Tensor, idx: List[int], U: DenseBasis, V: DenseBasis) -> Dict[str, Any]:
    n = Ws.shape[-1]
    b = int(cfg.CORE_BLOCK)

    # X_e = U^T W_e V
    X_list = [(U.M.t() @ Ws[e] @ V.M).contiguous() for e in idx]

    core = choose_core_blocks(X_list)

    core_blocks: List[List[Tuple[int,int,Any]]] = []
    energy_fracs = []
    for X, (pick, nb), eff in zip(X_list, core["per"], core["energy_fracs"]):
        lst = []
        for flat_idx in pick:
            bi = flat_idx // nb
            bj = flat_idx % nb
            i0, j0 = bi*b, bj*b
            i1, j1 = min(n, i0+b), min(n, j0+b)
            lst.append((i0, j0, pack_tensor(X[i0:i1, j0:j1].contiguous())))
        core_blocks.append(lst)
        energy_fracs.append(eff)

    # Residual after core
    R_list = []
    for X, cb in zip(X_list, core_blocks):
        Xc = torch.zeros_like(X)
        for (i0, j0, obj) in cb:
            B = unpack_tensor(obj, X.device)
            h, w = B.shape
            Xc[i0:i0+h, j0:j0+w] = B
        R_list.append((X - Xc).contiguous())

    # Low-rank from mean residual
    Rmean = torch.stack(R_list, dim=0).mean(dim=0)
    U_s, _, Vh_s = torch.linalg.svd(Rmean, full_matrices=False)
    r = int(min(cfg.RES_RANK, U_s.shape[1]))
    DL = U_s[:, :r].to(DTYPE_ACC).contiguous()
    DR = Vh_s.t()[:, :r].to(DTYPE_ACC).contiguous()

    coef_list = []
    res_blocks: List[List[Tuple[int,int,Any]]] = []
    avoid_sets: List[Set[Tuple[int,int]]] = []

    for Rm, cb in zip(R_list, core_blocks):
        avoid: Set[Tuple[int,int]] = set((i0, j0) for (i0, j0, _) in cb)
        avoid_sets.append(avoid)

        if cfg.RES_COEF == "diag":
            g = torch.sum(DL * (Rm @ DR), dim=0).contiguous()
            coef_list.append(pack_tensor(g))
            R2 = (Rm - (DL * g.view(1, -1)) @ DR.t()).contiguous()
        else:
            C = (DL.t() @ Rm @ DR).contiguous()
            coef_list.append(pack_tensor(C))
            R2 = (Rm - (DL @ C @ DR.t())).contiguous()

        blks = pick_residual_blocks(R2, b=int(cfg.RES_BSIZE), k=int(cfg.RES_BLOCKS), avoid=avoid)
        out = [(i0, j0, pack_tensor(B)) for (i0, j0, B) in blks]
        res_blocks.append(out)

    # Adaptive refinement: add extra residual blocks for hard experts
    if cfg.REFINE_ENABLE and cfg.REFINE_MAX_EXTRA_BLOCKS > 0:
        for j, e in enumerate(idx):
            # reconstruct Xhat and compute residual, then add blocks until target or budget
            X = X_list[j]
            Xhat = torch.zeros_like(X)

            # core
            for (i0, j0, obj) in core_blocks[j]:
                B = unpack_tensor(obj, X.device)
                h, w = B.shape
                Xhat[i0:i0+h, j0:j0+w] = B

            # low-rank
            coef = unpack_tensor(coef_list[j], X.device)
            if cfg.RES_COEF == "diag":
                Xhat += (DL * coef.view(1, -1)) @ DR.t()
            else:
                Xhat += (DL @ coef @ DR.t())

            # res blocks
            for (i0, j0, obj) in res_blocks[j]:
                B = unpack_tensor(obj, X.device)
                h, w = B.shape
                Xhat[i0:i0+h, j0:j0+w] += B

            def rel_err(A: torch.Tensor, Bm: torch.Tensor) -> float:
                num = torch.linalg.norm(A - Bm)
                den = torch.linalg.norm(Bm).clamp_min(1e-12)
                return float((num / den).item())

            err = rel_err(Xhat, X)
            extra_used = 0

            while err > cfg.REFINE_ERR_TARGET and extra_used < cfg.REFINE_MAX_EXTRA_BLOCKS:
                Rcur = (X - Xhat).contiguous()
                new_blocks = pick_residual_blocks(Rcur, b=cfg.REFINE_BSIZE, k=1, avoid=avoid_sets[j])
                if not new_blocks:
                    break
                (i0, j0, B) = new_blocks[0]
                res_blocks[j].append((i0, j0, pack_tensor(B)))
                # update Xhat incrementally
                Xhat[i0:i0+B.shape[0], j0:j0+B.shape[1]] += B
                extra_used += 1
                if extra_used % 8 == 0:
                    err = rel_err(Xhat, X)
                # (don’t recompute every single time)

    return {
        "idx": idx,
        "U": U.M.detach().cpu(),   # store basis matrices
        "V": V.M.detach().cpu(),
        "core_energy_fracs": np.array(energy_fracs, dtype=np.float32),
        "core_blocks": core_blocks,
        "DL": pack_tensor(DL),
        "DR": pack_tensor(DR),
        "coef": coef_list,
        "res_blocks": res_blocks,
        "r": r,
    }

# ----------------------------
# Runtime for eval
# ----------------------------
class KTXRuntime:
    def __init__(self, payloads: List[Dict[str, Any]], map_e: Dict[int, Tuple[int,int]], n: int):
        self.payloads = payloads
        self.map_e = map_e
        self.n = n

    @torch.no_grad()
    def apply_one_expert(self, x: torch.Tensor, e: int) -> torch.Tensor:
        m, j = self.map_e[e]
        P = self.payloads[m]
        U = DenseBasis(P["U"].to(DEVICE).to(DTYPE_ACC))
        V = DenseBasis(P["V"].to(DEVICE).to(DTYPE_ACC))

        z = U.right(x, transpose=False)   # x @ U
        u = torch.zeros_like(z)

        for (i0, j0, obj) in P["core_blocks"][j]:
            B = unpack_tensor(obj, z.device)
            h, w = B.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ B

        DL = unpack_tensor(P["DL"], z.device)
        DR = unpack_tensor(P["DR"], z.device)
        coef = unpack_tensor(P["coef"][j], z.device)

        if cfg.RES_COEF == "diag":
            u += ((z @ DL) * coef.view(1, -1)) @ DR.t()
        else:
            u += (z @ DL) @ coef @ DR.t()

        for (i0, j0, obj) in P["res_blocks"][j]:
            B = unpack_tensor(obj, z.device)
            h, w = B.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ B

        y = V.right(u, transpose=True)    # u @ V^T
        return y

    @torch.no_grad()
    def apply_mixture(self, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
        y = torch.zeros_like(x)
        for a, e in zip(gates.tolist(), routed):
            y += float(a) * self.apply_one_expert(x, e)
        return y

# ----------------------------
# Eval
# ----------------------------
@torch.no_grad()
def eval_per_expert(rt: KTXRuntime, Ws: torch.Tensor, trials: int, batch: int):
    E, n, _ = Ws.shape
    errs = []
    for e in range(E):
        ee = []
        for _ in range(trials):
            x = torch.randn(batch, n, dtype=DTYPE_ACC, device=DEVICE)
            y_ref = x @ Ws[e]
            y_hat = rt.apply_one_expert(x, e)
            num = torch.linalg.norm(y_hat - y_ref)
            den = torch.linalg.norm(y_ref).clamp_min(1e-12)
            ee.append(float((num / den).item()))
        errs.append(float(np.mean(ee)))
    log(f"[eval] per-expert rel-error  mean={float(np.mean(errs)):.6f}  p95={float(np.percentile(errs,95)):.6f}  max={float(np.max(errs)):.6f}")

@torch.no_grad()
def eval_routed(rt: KTXRuntime, Ws: torch.Tensor, trials: int, routed_k: int, batch: int):
    E, n, _ = Ws.shape
    errs = []
    for _ in range(trials):
        x = torch.randn(batch, n, dtype=DTYPE_ACC, device=DEVICE)
        routed = random.sample(range(E), k=min(routed_k, E))
        gates = torch.rand(len(routed), dtype=DTYPE_ACC, device=DEVICE)
        gates = gates / gates.sum().clamp_min(1e-12)

        Wsum = torch.zeros(n, n, dtype=DTYPE_ACC, device=DEVICE)
        for a, e in zip(gates, routed):
            Wsum += float(a.item()) * Ws[e]
        y_ref = x @ Wsum

        y_hat = rt.apply_mixture(x, routed, gates)
        num = torch.linalg.norm(y_hat - y_ref)
        den = torch.linalg.norm(y_ref).clamp_min(1e-12)
        errs.append(float((num / den).item()))
    log(f"[eval] routed rel-error = {float(np.mean(errs)):.6f} ± {float(np.std(errs)):.6f}  (trials={trials})")

# ----------------------------
# Main
# ----------------------------
def banner():
    log("== DeepSeek KT++-X OFFLINE v10 ==")
    log(f"Time:        {now()}")
    log(f"MODEL_DIR:   {cfg.MODEL_DIR}")
    log(f"OUTPUT_DIR:  {cfg.OUTPUT_DIR}")
    log(f"LAYER:       {cfg.LAYER}")
    log(f"MAX_EXPERTS:  {cfg.MAX_EXPERTS}")
    log(f"CALIB_PATH:  {cfg.CALIB_PATH}  CALIB_SAMPLES(cap)={cfg.CALIB_SAMPLES}")
    log(f"LIN_MODE:    {cfg.LIN_MODE}  RIDGE_DAMP={cfg.RIDGE_DAMP}")
    log(f"BASIS_MODE:  {cfg.BASIS_MODE}")
    log(f"CLUSTER:     M0={cfg.M0 or '(auto)'} M_MAX={cfg.M_MAX} min_size={cfg.CLUSTER_MIN_SIZE} max_size={cfg.CLUSTER_MAX_SIZE}")
    log(f"TRAIN:       steps={cfg.TRAIN_STEPS} warmup={cfg.TRAIN_WARMUP} lr={cfg.TRAIN_LR} subm={cfg.SUBM} batchE={cfg.BATCH_E}")
    log(f"CORE:        {cfg.CORE_MODE} block={cfg.CORE_BLOCK} target={cfg.CORE_TARGET} max_blocks={cfg.CORE_MAX_BLOCKS}")
    log(f"RESIDUAL:    rank={cfg.RES_RANK} coef={cfg.RES_COEF} blocks={cfg.RES_BLOCKS} bsize={cfg.RES_BSIZE}")
    log(f"REFINE:      enable={cfg.REFINE_ENABLE} err_target={cfg.REFINE_ERR_TARGET} max_extra={cfg.REFINE_MAX_EXTRA_BLOCKS}")
    log(f"QUANT:       QMODE={cfg.QMODE}")
    log("")

def main():
    banner()
    if cfg.BASIS_MODE != "dense_train":
        raise RuntimeError("v10 script is specialized for dense_train.")

    eids, Ws = load_or_build_Ws()
    E, n, _ = Ws.shape
    log(f"[Ws] shape={tuple(Ws.shape)} experts={eids}")

    # Clustering
    Xfeat = random_proj_features(Ws, d=cfg.CLUSTER_FEAT_D)
    if cfg.M0 > 0:
        M0 = min(cfg.M0, E)
    else:
        # fewer clusters than v9 to reduce singletons
        M0 = max(4, min(int(round(1.5 * math.sqrt(E))), E))
    labels = kmeans_torch(Xfeat, k=M0, iters=cfg.CLUSTER_ITERS, restarts=cfg.CLUSTER_RESTARTS)
    labels = enforce_min_cluster_size(labels, Xfeat, min_size=cfg.CLUSTER_MIN_SIZE)
    labels = hierarchical_split(Xfeat, labels, max_size=cfg.CLUSTER_MAX_SIZE, max_k=min(cfg.M_MAX, E), split_iters=cfg.SPLIT_ITERS)

    if cfg.MERGE_TINY:
        labels = merge_tiny_clusters(Xfeat, labels, max_tiny=cfg.MERGE_TINY_MAX)

    M = int(labels.max().item()) + 1
    clusters = [torch.nonzero(labels == m, as_tuple=False).flatten().tolist() for m in range(M)]
    log(f"[cluster] M={M} sizes={[len(c) for c in clusters]}")

    # Init U,V per cluster
    U_par: List[Optional[OrthoParam]] = []
    V_par: List[Optional[OrthoParam]] = []
    for idx in clusters:
        if len(idx) == 0:
            U_par.append(None); V_par.append(None)
            continue
        Wm = Ws[idx].mean(dim=0)
        U0, V0 = svd_init(Wm)
        U_par.append(OrthoParam(U0))
        V_par.append(OrthoParam(V0))

    # Training
    if cfg.TRAIN_STEPS > 0:
        params = [uv.M for uv in (U_par + V_par) if uv is not None]
        opt = torch.optim.Adam(params, lr=cfg.TRAIN_LR)
        guidance_masks: Dict[int, torch.Tensor] = {}
        guidance_stats: Dict[int, Tuple[float,int]] = {}
        t0 = time.perf_counter()

        with torch.enable_grad():
            for step in range(1, cfg.TRAIN_STEPS + 1):
                S = torch.randperm(n, device=DEVICE)[:min(cfg.SUBM, n)]

                if cfg.TRAIN_LAM_GUIDE > 0 and (step % max(1, cfg.TRAIN_GUIDE_EVERY) == 0 or step == 1):
                    with torch.no_grad():
                        guidance_masks.clear()
                        guidance_stats.clear()
                        for m, idx in enumerate(clusters):
                            if len(idx) < cfg.TRAIN_MIN_CLUSTER:
                                continue
                            Uo = U_par[m].orthogonal()
                            Vo = V_par[m].orthogonal()
                            pick = idx
                            if 0 < cfg.BATCH_E < len(idx):
                                pidx = torch.randperm(len(idx), device=DEVICE)[:cfg.BATCH_E].tolist()
                                pick = [idx[i] for i in pidx]
                            Xs_ng = slice_X_batch(Ws[pick], Uo, Vo, S).detach()
                            mask, ef, kblk = make_guidance_mask_from_Xs(
                                Xs_ng.to(DTYPE_ACC),
                                block=cfg.CORE_BLOCK,
                                target=cfg.TRAIN_GUIDE_TARGET,
                                max_blocks=cfg.TRAIN_GUIDE_MAX_BLOCKS
                            )
                            guidance_masks[m] = mask
                            guidance_stats[m] = (ef, kblk)

                lam_ramp = schedule(step, cfg.TRAIN_WARMUP, cfg.TRAIN_STEPS)
                lam_block = cfg.TRAIN_LAM_BLOCK * lam_ramp
                lam_guide = cfg.TRAIN_LAM_GUIDE * lam_ramp

                L_total = None
                terms = 0
                for m, idx in enumerate(clusters):
                    if len(idx) < cfg.TRAIN_MIN_CLUSTER:
                        continue
                    Uo = U_par[m].orthogonal()
                    Vo = V_par[m].orthogonal()

                    pick = idx
                    if 0 < cfg.BATCH_E < len(idx):
                        pidx = torch.randperm(len(idx), device=DEVICE)[:cfg.BATCH_E].tolist()
                        pick = [idx[i] for i in pidx]

                    Xs = slice_X_batch(Ws[pick], Uo, Vo, S).to(DTYPE_ACC)
                    off = offdiag_abs_mean(Xs)
                    diag = diag_abs_mean(Xs).clamp_min(1e-6)

                    if cfg.TRAIN_OBJ == "ratio":
                        base = off / diag
                    else:
                        base = torch.log(off + 1e-6) - torch.log(diag)

                    if lam_block > 0 and cfg.CORE_MODE.startswith("block"):
                        base = base + lam_block * block_group_sparsity_penalty(Xs, cfg.CORE_BLOCK)

                    if lam_guide > 0 and (m in guidance_masks):
                        Mmask = guidance_masks[m]
                        Etot = (Xs * Xs).mean().clamp_min(1e-12)
                        Eout = ((Xs * (1.0 - Mmask)) ** 2).mean()
                        base = base + lam_guide * (Eout / Etot)

                    L_total = base if (L_total is None) else (L_total + base)
                    terms += 1

                if L_total is None:
                    break
                L_total = L_total / max(1, terms)

                opt.zero_grad(set_to_none=True)
                L_total.backward()
                if cfg.GRAD_CLIP > 0:
                    torch.nn.utils.clip_grad_norm_(params, max_norm=cfg.GRAD_CLIP)
                opt.step()

                if (step % cfg.REORTHO_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                    with torch.no_grad():
                        for uv in (U_par + V_par):
                            if uv is not None:
                                uv.M.copy_(uv.orthogonal())

                if step == 1 or (step % cfg.REPORT_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                    t1 = time.perf_counter()
                    if len(guidance_stats) > 0:
                        ef_mean = float(np.mean([v[0] for v in guidance_stats.values()]))
                        kb_mean = float(np.mean([v[1] for v in guidance_stats.values()]))
                        log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} loss={float(L_total.item()):.4f} "
                            f"lam_block={lam_block:.3f} lam_guide={lam_guide:.3f} "
                            f"guide_energy≈{ef_mean:.3f} guide_blocks≈{kb_mean:.1f} (+{t1-t0:.1f}s)")
                    else:
                        log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} loss={float(L_total.item()):.4f} "
                            f"lam_block={lam_block:.3f} lam_guide={lam_guide:.3f} (+{t1-t0:.1f}s)")
                    t0 = t1

    # Build payloads
    payloads: List[Dict[str, Any]] = []
    map_e: Dict[int, Tuple[int,int]] = {}
    log("[build] payloads ...")
    for m, idx in enumerate(clusters):
        if len(idx) == 0:
            continue
        Uo = U_par[m].orthogonal().detach()
        Vo = V_par[m].orthogonal().detach()
        U = DenseBasis(Uo)
        V = DenseBasis(Vo)

        P = build_payload_for_cluster(Ws, idx, U, V)
        payloads.append(P)
        mi = len(payloads) - 1
        for j, e in enumerate(idx):
            map_e[e] = (mi, j)

        core_ef = P["core_energy_fracs"]
        nb_mean = float(np.mean([len(x) for x in P["core_blocks"]])) if len(P["core_blocks"]) else 0.0
        log(f"  - cluster{mi}: E={len(idx)} core_blocks(mean)≈{nb_mean:.1f} core_energy(mean/min)≈{float(core_ef.mean()):.3f}/{float(core_ef.min()):.3f} r={P['r']}")

    rt = KTXRuntime(payloads, map_e, n=n)
    eval_per_expert(rt, Ws, trials=max(2, cfg.EVAL_TRIALS//2), batch=cfg.EVAL_BATCH)
    eval_routed(rt, Ws, trials=cfg.EVAL_TRIALS, routed_k=cfg.ROUTED_K, batch=cfg.EVAL_BATCH)

    log("\n✅ Done.")
    log("If you want <1–2% rel-error, you MUST capture more than 160 X rows.")
    log("Try PRESET=ultra after recapturing X (4k–32k rows).")

if __name__ == "__main__":
    banner()
    main()


== DeepSeek KT++-X OFFLINE v10 ==
Time:        2026-01-13 11:12:58
MODEL_DIR:   /home/daniyar/deepseek-model
OUTPUT_DIR:  /home/daniyar/moe_ws_outputs
LAYER:       1
MAX_EXPERTS:  16
CALIB_PATH:    CALIB_SAMPLES(cap)=4096
LIN_MODE:    ridge  RIDGE_DAMP=0.001
BASIS_MODE:  dense_train
CLUSTER:     M0=(auto) M_MAX=16 min_size=2 max_size=4
TRAIN:       steps=24 warmup=6 lr=0.05 subm=256 batchE=4
CORE:        blocktopk_perexpert block=64 target=0.85 max_blocks=256
RESIDUAL:    rank=512 coef=diag blocks=192 bsize=64
REFINE:      enable=True err_target=0.05 max_extra=256
QUANT:       QMODE=none

== DeepSeek KT++-X OFFLINE v10 ==
Time:        2026-01-13 11:12:58
MODEL_DIR:   /home/daniyar/deepseek-model
OUTPUT_DIR:  /home/daniyar/moe_ws_outputs
LAYER:       1
MAX_EXPERTS:  16
CALIB_PATH:    CALIB_SAMPLES(cap)=4096
LIN_MODE:    ridge  RIDGE_DAMP=0.001
BASIS_MODE:  dense_train
CLUSTER:     M0=(auto) M_MAX=16 min_size=2 max_size=4
TRAIN:       steps=24 warmup=6 lr=0.05 subm=256 batchE=4
CORE:    

RuntimeError: CALIB_PATH is required for ridge mode.

In [13]:
#!/usr/bin/env python3
# ============================================================
# DeepSeek KT++-X OFFLINE v10.1
# Fixes:
#  - If CALIB_PATH is empty: auto-detect OUTPUT_DIR/calib_layer{LAYER}_X.npz
#  - If still missing in ridge: fall back to random X with a warning (runs but low quality)
#  - Better banner printing
#  - No duplicate banner printing
# ============================================================

import os, re, json, math, time, random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any, Set

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors import safe_open

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):  # type: ignore
        return x

# ----------------------------
# Threads / determinism
# ----------------------------
NTHREADS = int(os.environ.get("KTXX_THREADS", "8"))
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(NTHREADS))
try:
    torch.set_num_threads(NTHREADS)
except Exception:
    pass

SEED = int(os.environ.get("SEED", "1234"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(os.environ.get("DEVICE", "cpu"))
DTYPE_ACC = torch.float32
DTYPE_STORE = torch.float16

def now():
    return time.strftime("%Y-%m-%d %H:%M:%S")

def log(msg: str):
    print(msg, flush=True)

# ----------------------------
# Presets (optional)
# ----------------------------
PRESET = os.environ.get("PRESET", "").strip().lower()

def env(key: str, default: str) -> str:
    return os.environ.get(key, default)

# ----------------------------
# Config
# ----------------------------
@dataclass
class Cfg:
    MODEL_DIR: str = env("MODEL_DIR", "/home/daniyar/deepseek-model")
    OUTPUT_DIR: str = env("OUTPUT_DIR", "/home/daniyar/moe_ws_outputs")
    LAYER: int = int(env("LAYER", "1"))
    MAX_EXPERTS: int = int(env("MAX_EXPERTS", "16"))

    CALIB_PATH: str = env("CALIB_PATH", "").strip()
    CALIB_SAMPLES: int = int(env("CALIB_SAMPLES", "4096"))

    ROUTER_PATH: str = env("ROUTER_PATH", "").strip()
    RIDGE_WEIGHTED: bool = env("RIDGE_WEIGHTED", "0") == "1"
    ROUTER_EIDS_ARE_GLOBAL: bool = env("ROUTER_EIDS_ARE_GLOBAL", "1") == "1"

    LIN_MODE: str = env("LIN_MODE", "ridge").strip().lower()
    RIDGE_DAMP: float = float(env("RIDGE_DAMP", "1e-3"))
    NORMALIZE_W: bool = env("NORMALIZE_W", "1") == "1"

    BASIS_MODE: str = env("BASIS_MODE", "dense_train").strip().lower()

    # Clustering
    M0: int = int(env("M0", "0"))          # 0 => auto
    M_MAX: int = int(env("M_MAX", "16"))
    CLUSTER_FEAT_D: int = int(env("CLUSTER_FEAT_D", "64"))
    CLUSTER_ITERS: int = int(env("CLUSTER_ITERS", "60"))
    CLUSTER_RESTARTS: int = int(env("CLUSTER_RESTARTS", "4"))
    CLUSTER_MIN_SIZE: int = int(env("CLUSTER_MIN_SIZE", "2"))
    CLUSTER_MAX_SIZE: int = int(env("CLUSTER_MAX_SIZE", "4"))
    SPLIT_ITERS: int = int(env("SPLIT_ITERS", "50"))
    MERGE_TINY: bool = env("MERGE_TINY", "1") == "1"
    MERGE_TINY_MAX: int = int(env("MERGE_TINY_MAX", "2"))

    # Training
    TRAIN_STEPS: int = int(env("TRAIN_STEPS", "24"))
    TRAIN_WARMUP: int = int(env("TRAIN_WARMUP", "6"))
    TRAIN_LR: float = float(env("TRAIN_LR", "5e-2"))
    SUBM: int = int(env("SUBM", "256"))
    BATCH_E: int = int(env("BATCH_E", "4"))
    TRAIN_MIN_CLUSTER: int = int(env("TRAIN_MIN_CLUSTER", "2"))
    REORTHO_EVERY: int = int(env("REORTHO_EVERY", "4"))
    REPORT_EVERY: int = int(env("REPORT_EVERY", "4"))
    GRAD_CLIP: float = float(env("GRAD_CLIP", "1.0"))

    TRAIN_OBJ: str = env("TRAIN_OBJ", "logratio").strip().lower()
    TRAIN_LAM_BLOCK: float = float(env("TRAIN_LAM_BLOCK", "0.10"))
    TRAIN_LAM_GUIDE: float = float(env("TRAIN_LAM_GUIDE", "1.0"))
    TRAIN_GUIDE_EVERY: int = int(env("TRAIN_GUIDE_EVERY", "2"))
    TRAIN_GUIDE_TARGET: float = float(env("TRAIN_GUIDE_TARGET", "0.75"))
    TRAIN_GUIDE_MAX_BLOCKS: int = int(env("TRAIN_GUIDE_MAX_BLOCKS", "256"))

    # Core / residual
    CORE_MODE: str = env("CORE_MODE", "blocktopk_perexpert").strip().lower()
    CORE_BLOCK: int = int(env("CORE_BLOCK", "64"))
    CORE_TARGET: float = float(env("CORE_TARGET", "0.85"))
    CORE_MAX_BLOCKS: int = int(env("CORE_MAX_BLOCKS", "256"))

    RES_RANK: int = int(env("RES_RANK", "512"))
    RES_COEF: str = env("RES_COEF", "diag").strip().lower()
    RES_BLOCKS: int = int(env("RES_BLOCKS", "192"))
    RES_BSIZE: int = int(env("RES_BSIZE", "64"))

    # Adaptive refinement
    REFINE_ENABLE: bool = env("REFINE_ENABLE", "1") == "1"
    REFINE_ERR_TARGET: float = float(env("REFINE_ERR_TARGET", "0.05"))
    REFINE_MAX_EXTRA_BLOCKS: int = int(env("REFINE_MAX_EXTRA_BLOCKS", "256"))
    REFINE_BSIZE: int = int(env("REFINE_BSIZE", "64"))

    # Quant
    QMODE: str = env("QMODE", "none").strip().lower()  # none|float16|int8

    # Eval
    EVAL_TRIALS: int = int(env("EVAL_TRIALS", "8"))
    EVAL_BATCH: int = int(env("EVAL_BATCH", "2"))
    ROUTED_K: int = int(env("ROUTED_K", "8"))

cfg = Cfg()

if PRESET == "ultra":
    os.environ.setdefault("TRAIN_STEPS", "64")
    os.environ.setdefault("TRAIN_WARMUP", "12")
    os.environ.setdefault("TRAIN_LAM_GUIDE", "2.0")
    os.environ.setdefault("TRAIN_GUIDE_EVERY", "1")
    os.environ.setdefault("CORE_BLOCK", "32")
    os.environ.setdefault("CORE_TARGET", "0.92")
    os.environ.setdefault("CORE_MAX_BLOCKS", "512")
    os.environ.setdefault("RES_RANK", "1024")
    os.environ.setdefault("RES_BLOCKS", "512")
    os.environ.setdefault("RIDGE_DAMP", "1e-2")
    cfg = Cfg()

elif PRESET == "compact":
    os.environ.setdefault("QMODE", "float16")
    os.environ.setdefault("CORE_BLOCK", "64")
    os.environ.setdefault("CORE_TARGET", "0.85")
    os.environ.setdefault("RES_RANK", "512")
    os.environ.setdefault("RES_BLOCKS", "192")
    cfg = Cfg()

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# ----------------------------
# NPZ helpers
# ----------------------------
def save_npz(path: str, arrays: Dict[str, np.ndarray]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez(path, **arrays)

def try_load_npz(path: str) -> Optional[Dict[str, np.ndarray]]:
    if not os.path.isfile(path):
        return None
    z = np.load(path, allow_pickle=False)
    out = {k: z[k] for k in z.files}
    z.close()
    return out

def _encode_meta(meta: dict) -> np.ndarray:
    b = json.dumps(meta, sort_keys=True).encode("utf-8")
    return np.frombuffer(b, dtype=np.uint8)

def _decode_meta(arr: np.ndarray) -> dict:
    try:
        b = bytes(arr.tolist())
        return json.loads(b.decode("utf-8"))
    except Exception:
        return {}

# ----------------------------
# Offline shard loading
# ----------------------------
def read_index(model_dir: str) -> Dict[str, str]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    wm = obj.get("weight_map", {})
    if not wm:
        raise RuntimeError("Index JSON has empty weight_map.")
    return wm

def find_layer_expert_ids(weight_map: Dict[str, str], layer: int) -> List[int]:
    pat = re.compile(rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.")
    ids = set()
    for k in weight_map.keys():
        m = pat.match(k)
        if m:
            ids.add(int(m.group(1)))
    return sorted(ids)

def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    prefix = f"model.layers.{layer}.mlp.experts.{eid}."
    def pick(cands: List[str]) -> Optional[str]:
        for suf in cands:
            k = prefix + suf
            if k in weight_map:
                return k
        return None
    up   = pick(["up_proj.weight", "w3.weight", "w1.weight"])
    gate = pick(["gate_proj.weight", "w1.weight", "w3.weight"])
    down = pick(["down_proj.weight", "w2.weight"])
    if up is None or gate is None or down is None:
        return {}
    if up == gate:
        g2 = pick(["gate_proj.weight"])
        if g2:
            gate = g2
    return {"up": up, "gate": gate, "down": down}

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], keys: List[str]) -> Dict[str, torch.Tensor]:
    by_shard: Dict[str, List[str]] = {}
    for k in keys:
        shard = weight_map.get(k, None)
        if shard is None:
            raise KeyError(f"Key not in weight_map: {k}")
        by_shard.setdefault(shard, []).append(k)

    out: Dict[str, torch.Tensor] = {}
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(sp):
            raise FileNotFoundError(f"Missing shard: {sp}")
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks:
                out[k] = f.get_tensor(k)
    return out

# ----------------------------
# Calibration / router
# ----------------------------
def autodetect_calib_path() -> Optional[str]:
    # your bridge writes: OUTPUT_DIR/calib_layer{LAYER}_X.npz
    cand = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
    if os.path.isfile(cand):
        return cand
    return None

def load_calib_X(H: int) -> torch.Tensor:
    # If empty, try auto-detect
    if not cfg.CALIB_PATH:
        cand = autodetect_calib_path()
        if cand:
            cfg.CALIB_PATH = cand
            log(f"[calib] CALIB_PATH not set -> auto-found {cfg.CALIB_PATH}")

    if not cfg.CALIB_PATH:
        # Allow fallback so script can still run, but warn loudly.
        log("⚠️  CALIB_PATH is missing. Falling back to RANDOM X (ridge will be low-quality).")
        X = torch.randn(cfg.CALIB_SAMPLES, H, dtype=DTYPE_ACC, device=DEVICE)
        return X

    if not os.path.isfile(cfg.CALIB_PATH):
        raise FileNotFoundError(f"CALIB_PATH not found: {cfg.CALIB_PATH}")

    z = np.load(cfg.CALIB_PATH, allow_pickle=False)
    if "X" not in z.files:
        raise KeyError(f"CALIB npz missing key 'X'. Keys={list(z.files)}")
    Xn = z["X"]
    z.close()
    X = torch.from_numpy(Xn).to(DTYPE_ACC)
    if X.ndim != 2 or X.shape[1] != H:
        raise RuntimeError(f"Bad X shape: {tuple(X.shape)} expected (*,{H})")
    if X.shape[0] > cfg.CALIB_SAMPLES:
        X = X[:cfg.CALIB_SAMPLES]
    return X.to(DEVICE)

def load_router_P() -> Optional[np.ndarray]:
    if not cfg.ROUTER_PATH:
        return None
    if not os.path.isfile(cfg.ROUTER_PATH):
        raise FileNotFoundError(f"ROUTER_PATH not found: {cfg.ROUTER_PATH}")
    z = np.load(cfg.ROUTER_PATH, allow_pickle=False)
    if "P" not in z.files:
        raise KeyError(f"ROUTER npz missing key 'P'. Keys={list(z.files)}")
    P = z["P"].astype(np.float32, copy=False)
    z.close()
    return P

# ----------------------------
# Build Ws (ridge)
# ----------------------------
@torch.no_grad()
def forward_mlp(X: torch.Tensor, W_gate: torch.Tensor, W_up: torch.Tensor, W_down: torch.Tensor) -> torch.Tensor:
    Xf = X.to(DTYPE_ACC)
    up = Xf @ W_up.to(DTYPE_ACC).t()
    gate = Xf @ W_gate.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    y = hid @ W_down.to(DTYPE_ACC).t()
    return y

def ws_cache_path(e: int) -> str:
    return os.path.join(cfg.OUTPUT_DIR, f"Ws_cache_layer{cfg.LAYER}_E{e}_{cfg.LIN_MODE}_v10_1.npz")

def ws_meta(eids: List[int]) -> dict:
    return dict(
        model_dir=cfg.MODEL_DIR,
        layer=cfg.LAYER,
        eids=eids,
        lin_mode=cfg.LIN_MODE,
        ridge_damp=cfg.RIDGE_DAMP,
        ridge_weighted=cfg.RIDGE_WEIGHTED,
        calib_path=(cfg.CALIB_PATH or ""),
        router_path=(cfg.ROUTER_PATH or ""),
        normalize_w=cfg.NORMALIZE_W,
        seed=SEED,
    )

@torch.no_grad()
def load_or_build_Ws() -> Tuple[List[int], torch.Tensor]:
    wm = read_index(cfg.MODEL_DIR)
    all_eids = find_layer_expert_ids(wm, cfg.LAYER)
    if not all_eids:
        raise RuntimeError(f"No experts found for layer={cfg.LAYER}")
    eids = all_eids[:cfg.MAX_EXPERTS]

    cpath = ws_cache_path(len(eids))
    z = try_load_npz(cpath)
    if z and ("Ws" in z) and ("expert_ids" in z) and ("meta" in z):
        if _decode_meta(z["meta"]) == ws_meta(eids):
            Ws = torch.from_numpy(z["Ws"]).to(DTYPE_ACC).to(DEVICE)
            log(f"[cache] loaded Ws: {cpath}  Ws={tuple(Ws.shape)}")
            return [int(x) for x in z["expert_ids"].tolist()], Ws
        log("[cache] Ws meta mismatch -> rebuilding.")

    per_e = {}
    need_keys = []
    for eid in eids:
        kk = pick_expert_tensor_keys(wm, cfg.LAYER, eid)
        if not kk:
            raise RuntimeError(f"Expert {eid} missing tensors in index.")
        per_e[eid] = kk
        need_keys += [kk["up"], kk["down"], kk["gate"]]

    log("[load] reading tensors from shards ...")
    T = load_tensors_from_shards(cfg.MODEL_DIR, wm, sorted(set(need_keys)))

    W_up0 = T[per_e[eids[0]]["up"]]
    dff, H = int(W_up0.shape[0]), int(W_up0.shape[1])
    log(f"[shape] H={H} d_ff={dff}")

    if cfg.LIN_MODE != "ridge":
        raise ValueError("v10.1 focuses on LIN_MODE=ridge.")

    X = load_calib_X(H)
    log(f"[calib] X loaded: {tuple(X.shape)}  (path={cfg.CALIB_PATH or '(random)'})")
    P = load_router_P()
    if cfg.RIDGE_WEIGHTED and (P is None):
        log("[router] missing ROUTER_PATH -> forcing RIDGE_WEIGHTED=0")
        cfg.RIDGE_WEIGHTED = False

    I = torch.eye(H, dtype=DTYPE_ACC, device=DEVICE)
    Xf = X.to(DTYPE_ACC)
    XtX = Xf.t() @ Xf
    lam = cfg.RIDGE_DAMP * float(torch.trace(XtX).item()) / float(H)
    cholG = torch.linalg.cholesky(XtX + lam * I)

    Ws_list: List[torch.Tensor] = []
    for i, eid in enumerate(tqdm(eids, desc=f"Build Ws ({cfg.LIN_MODE})")):
        W_up   = T[per_e[eid]["up"]].to(DEVICE)
        W_down = T[per_e[eid]["down"]].to(DEVICE)
        W_gate = T[per_e[eid]["gate"]].to(DEVICE)

        Y = forward_mlp(X, W_gate, W_up, W_down).to(DTYPE_ACC)
        if cfg.RIDGE_WEIGHTED and (P is not None):
            if cfg.ROUTER_EIDS_ARE_GLOBAL:
                w = torch.from_numpy(P[:X.shape[0], eid]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
            else:
                w = torch.from_numpy(P[:X.shape[0], i]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
            sw = torch.sqrt(w + 1e-12).view(-1, 1)
            Xw = Xf * sw
            Yw = Y * sw
            XtX_e = Xw.t() @ Xw
            lam_e = cfg.RIDGE_DAMP * float(torch.trace(XtX_e).item()) / float(H)
            chol = torch.linalg.cholesky(XtX_e + lam_e * I)
            XtY = Xw.t() @ Yw
            Wt = torch.cholesky_solve(XtY, chol)
            W = Wt.t().contiguous()
        else:
            XtY = Xf.t() @ Y
            Wt = torch.cholesky_solve(XtY, cholG)
            W = Wt.t().contiguous()

        if cfg.NORMALIZE_W:
            fn = torch.linalg.norm(W, ord="fro").clamp_min(1e-12)
            W = (W / fn).contiguous()
        Ws_list.append(W)

    Ws = torch.stack(Ws_list, dim=0).to(DTYPE_ACC).to(DEVICE)
    save_npz(cpath, {
        "meta": _encode_meta(ws_meta(eids)),
        "expert_ids": np.array(eids, dtype=np.int32),
        "Ws": Ws.detach().cpu().numpy().astype(np.float32),
    })
    log(f"[cache] wrote Ws -> {cpath}  size={os.path.getsize(cpath)/1e6:.2f} MB")
    return eids, Ws

# ----------------------------
# (The rest: clustering/training/payload/eval)
# Reuse your v10 payload/training logic unchanged to keep this message shorter.
# If you want, paste your v10 tail and I’ll splice it cleanly into v10.1.
# ----------------------------

def banner():
    log("== DeepSeek KT++-X OFFLINE v10.1 ==")
    log(f"Time:        {now()}")
    log(f"MODEL_DIR:   {cfg.MODEL_DIR}")
    log(f"OUTPUT_DIR:  {cfg.OUTPUT_DIR}")
    log(f"LAYER:       {cfg.LAYER}")
    log(f"MAX_EXPERTS:  {cfg.MAX_EXPERTS}")
    log(f"CALIB_PATH:  {cfg.CALIB_PATH or '(none)'}  CALIB_SAMPLES(cap)={cfg.CALIB_SAMPLES}")
    log(f"LIN_MODE:    {cfg.LIN_MODE}  RIDGE_DAMP={cfg.RIDGE_DAMP}")
    log("")

def main():
    banner()
    _ = load_or_build_Ws()
    log("✅ Ws build succeeded. Now paste the rest of v10 (cluster/train/build/eval) below, or keep using your v10 tail.")

if __name__ == "__main__":
    main()


== DeepSeek KT++-X OFFLINE v10.1 ==
Time:        2026-01-13 11:12:38
MODEL_DIR:   /home/daniyar/deepseek-model
OUTPUT_DIR:  /home/daniyar/moe_ws_outputs
LAYER:       1
MAX_EXPERTS:  16
CALIB_PATH:  (none)  CALIB_SAMPLES(cap)=4096
LIN_MODE:    ridge  RIDGE_DAMP=0.001

[load] reading tensors from shards ...
[shape] H=2048 d_ff=1408
[calib] CALIB_PATH not set -> auto-found /home/daniyar/moe_ws_outputs/calib_layer1_X.npz
[calib] X loaded: (160, 2048)  (path=/home/daniyar/moe_ws_outputs/calib_layer1_X.npz)


Build Ws (ridge):   0%|          | 0/16 [00:00<?, ?it/s]

[cache] wrote Ws -> /home/daniyar/moe_ws_outputs/Ws_cache_layer1_E16_ridge_v10_1.npz  size=268.44 MB
✅ Ws build succeeded. Now paste the rest of v10 (cluster/train/build/eval) below, or keep using your v10 tail.


In [16]:
#!/usr/bin/env python3
# ============================================================
# DeepSeek KT++-X OFFLINE v10.2 (single-file "giant" runner)
#
# Offline, local shards only. End-to-end:
#   1) (Optional) Capture CALIB X and ROUTER P via transformers (local_files_only)
#   2) Build Ws (square [E,H,H]) via LIN_MODE=ridge (unweighted or weighted if P exists)
#   3) Cluster (dense_train only), train orthogonal bases U/V for block sparsity
#   4) Build payload: core blocks + low-rank residual + residual blocks (+ optional refine)
#   5) Save payload NPZ + eval (per-expert and routed mixture)
#
# Key fixes vs your v10.1 stub:
#   - Includes full cluster/train/build/eval + payload save
#   - Auto-detects CALIB_PATH = OUTPUT_DIR/calib_layer{LAYER}_X.npz
#   - Optional capture to increase X rows dramatically (padding=max_length, drop pads)
#   - Optional router capture: tries to locate router weight/module; if fails, runs unweighted
#   - Faster low-rank via randomized SVD (no full 2048 SVD per cluster)
#   - Optional refine loop to push per-expert error down (size↑)
#
# IMPORTANT REALITY CHECK:
#   - True “99–100%” relative error vs *real routed MoE* generally needs:
#       (a) lots of real X rows (thousands+ tokens, not 160)
#       (b) router probabilities P for RIDGE_WEIGHTED (or equivalent)
#       (c) enough payload capacity (CORE_TARGET↑, CORE_MAX_BLOCKS↑, RES_RANK↑, RES_BLOCKS↑, refine)
#   - This script gives you the plumbing + knobs to get there; you must capture enough X (and ideally P).
# ============================================================

import os, re, json, math, time, random, sys
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors import safe_open

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):  # type: ignore
        return x

# ----------------------------
# Threads / determinism
# ----------------------------
def _int_env(k: str, d: int) -> int:
    try:
        return int(os.environ.get(k, str(d)))
    except Exception:
        return d

def _float_env(k: str, d: float) -> float:
    try:
        return float(os.environ.get(k, str(d)))
    except Exception:
        return d

def _str_env(k: str, d: str) -> str:
    return os.environ.get(k, d)

def _bool_env(k: str, d: bool) -> bool:
    v = os.environ.get(k, None)
    if v is None:
        return d
    return v.strip().lower() in ("1", "true", "yes", "y", "on")

NTHREADS = _int_env("KTXX_THREADS", 8)
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(NTHREADS))
try:
    torch.set_num_threads(NTHREADS)
except Exception:
    pass

SEED = _int_env("SEED", 1234)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(_str_env("DEVICE", "cpu"))
DTYPE_ACC = torch.float32
DTYPE_BASIS_STORE = torch.float16

def now() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")

def log(msg: str):
    print(msg, flush=True)

# ----------------------------
# Config
# ----------------------------
@dataclass
class Cfg:
    # Paths
    MODEL_DIR: str = _str_env("MODEL_DIR", "/home/daniyar/deepseek-model")
    OUTPUT_DIR: str = _str_env("OUTPUT_DIR", "/home/daniyar/moe_ws_outputs")

    # Slice
    LAYER: int = _int_env("LAYER", 1)
    MAX_EXPERTS: int = _int_env("MAX_EXPERTS", 16)

    # Calib / router npz
    CALIB_PATH: str = _str_env("CALIB_PATH", "").strip()
    CALIB_SAMPLES: int = _int_env("CALIB_SAMPLES", 4096)

    ROUTER_PATH: str = _str_env("ROUTER_PATH", "").strip()
    RIDGE_WEIGHTED: bool = _bool_env("RIDGE_WEIGHTED", False)
    ROUTER_EIDS_ARE_GLOBAL: bool = _bool_env("ROUTER_EIDS_ARE_GLOBAL", True)

    # Capture (optional)
    CAPTURE_ENABLE: bool = _bool_env("CAPTURE_ENABLE", False)   # capture if CALIB missing, or force
    CAPTURE_FORCE: bool = _bool_env("CAPTURE_FORCE", False)     # always capture/overwrite
    CAPTURE_MAX_TOKENS: int = _int_env("CAPTURE_MAX_TOKENS", 512)
    CAPTURE_ITERS: int = _int_env("CAPTURE_ITERS", 32)
    CAPTURE_BATCH: int = _int_env("CAPTURE_BATCH", 1)
    CAPTURE_TEXT: str = _str_env(
        "CAPTURE_TEXT",
        ("DeepSeek models use mixture-of-experts layers. "
         "We capture intermediate activations for calibration. "
         "This paragraph is repeated to create long contexts. " * 64)
    )
    CAPTURE_KEEP_PAD: bool = _bool_env("CAPTURE_KEEP_PAD", False)
    HF_TRUST_REMOTE_CODE: bool = _bool_env("HF_TRUST_REMOTE_CODE", True)
    HF_LOCAL_FILES_ONLY: bool = _bool_env("HF_LOCAL_FILES_ONLY", True)
    HF_AUTO_PIP: bool = _bool_env("HF_AUTO_PIP", False)  # set 1 if transformers missing and you allow pip

    # Ws build
    LIN_MODE: str = _str_env("LIN_MODE", "ridge").strip().lower()  # ridge only in this runner
    RIDGE_DAMP: float = _float_env("RIDGE_DAMP", 1e-3)
    NORMALIZE_W: bool = _bool_env("NORMALIZE_W", True)

    # Basis mode: dense_train (learn U/V) or hadamard_perm (fixed implicit, smaller)
    BASIS_MODE: str = _str_env("BASIS_MODE", "dense_train").strip().lower()  # dense_train|hadamard_perm
    BASIS_STORE_DTYPE: str = _str_env("BASIS_STORE_DTYPE", "float16").strip().lower()  # float16|float32

    # Dense_train init
    INIT_BASIS: str = _str_env("INIT_BASIS", "identity").strip().lower()  # identity|random

    # Clustering (dense_train)
    M0: int = _int_env("M0", 0)  # 0 => auto
    M_MAX: int = _int_env("M_MAX", 16)
    CLUSTER_FEAT_D: int = _int_env("CLUSTER_FEAT_D", 64)
    CLUSTER_ITERS: int = _int_env("CLUSTER_ITERS", 60)
    CLUSTER_RESTARTS: int = _int_env("CLUSTER_RESTARTS", 4)
    CLUSTER_MIN_SIZE: int = _int_env("CLUSTER_MIN_SIZE", 2)
    CLUSTER_MAX_SIZE: int = _int_env("CLUSTER_MAX_SIZE", 4)
    SPLIT_ITERS: int = _int_env("SPLIT_ITERS", 50)

    # Training (dense_train)
    TRAIN_STEPS: int = _int_env("TRAIN_STEPS", 24)
    TRAIN_WARMUP: int = _int_env("TRAIN_WARMUP", 6)
    TRAIN_LR: float = _float_env("TRAIN_LR", 5e-2)
    SUBM: int = _int_env("SUBM", 256)
    BATCH_E: int = _int_env("BATCH_E", 4)
    TRAIN_MIN_CLUSTER: int = _int_env("TRAIN_MIN_CLUSTER", 2)
    REORTHO_EVERY: int = _int_env("REORTHO_EVERY", 4)
    REPORT_EVERY: int = _int_env("REPORT_EVERY", 4)
    GRAD_CLIP: float = _float_env("GRAD_CLIP", 1.0)

    TRAIN_OBJ: str = _str_env("TRAIN_OBJ", "logratio").strip().lower()  # logratio|ratio
    TRAIN_LAM_BLOCK: float = _float_env("TRAIN_LAM_BLOCK", 0.10)
    TRAIN_LAM_GUIDE: float = _float_env("TRAIN_LAM_GUIDE", 1.0)
    TRAIN_GUIDE_EVERY: int = _int_env("TRAIN_GUIDE_EVERY", 2)
    TRAIN_GUIDE_TARGET: float = _float_env("TRAIN_GUIDE_TARGET", 0.75)
    TRAIN_GUIDE_MAX_BLOCKS: int = _int_env("TRAIN_GUIDE_MAX_BLOCKS", 256)

    # Core selection
    CORE_MODE: str = _str_env("CORE_MODE", "blocktopk_perexpert").strip().lower()  # blocktopk_perexpert|blocktopk|blockdiag|none
    CORE_AGG: str = _str_env("CORE_AGG", "mean").strip().lower()  # mean|max
    CORE_BLOCK: int = _int_env("CORE_BLOCK", 64)
    CORE_TARGET: float = _float_env("CORE_TARGET", 0.85)
    CORE_MAX_BLOCKS: int = _int_env("CORE_MAX_BLOCKS", 256)

    # Residual
    RES_RANK: int = _int_env("RES_RANK", 512)
    RES_BLOCKS: int = _int_env("RES_BLOCKS", 192)
    RES_BSIZE: int = _int_env("RES_BSIZE", 64)

    # Refine (optional)
    REFINE_ENABLE: bool = _bool_env("REFINE_ENABLE", True)
    REFINE_ERR_TARGET: float = _float_env("REFINE_ERR_TARGET", 0.05)  # per-expert target in basis space
    REFINE_MAX_EXTRA: int = _int_env("REFINE_MAX_EXTRA", 256)
    REFINE_BSIZE: int = _int_env("REFINE_BSIZE", 64)

    # Quant for block payloads
    QMODE: str = _str_env("QMODE", "none").strip().lower()  # none|float16|int8

    # Eval
    EVAL_TRIALS: int = _int_env("EVAL_TRIALS", 8)
    EVAL_BATCH: int = _int_env("EVAL_BATCH", 2)
    ROUTED_K: int = _int_env("ROUTED_K", 8)

cfg = Cfg()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# ----------------------------
# NPZ helpers
# ----------------------------
def save_npz(path: str, **arrays: Any):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez_compressed(path, **arrays)

def load_npz(path: str) -> Dict[str, np.ndarray]:
    z = np.load(path, allow_pickle=False)
    out = {k: z[k] for k in z.files}
    z.close()
    return out

def _encode_meta(meta: dict) -> np.ndarray:
    b = json.dumps(meta, sort_keys=True).encode("utf-8")
    return np.frombuffer(b, dtype=np.uint8)

def _decode_meta(arr: np.ndarray) -> dict:
    try:
        b = bytes(arr.tolist())
        return json.loads(b.decode("utf-8"))
    except Exception:
        return {}

# ----------------------------
# Offline shard loading
# ----------------------------
def read_index(model_dir: str) -> Dict[str, str]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    wm = obj.get("weight_map", {})
    if not wm:
        raise RuntimeError("Index JSON has empty weight_map.")
    return wm

def find_layer_expert_ids(weight_map: Dict[str, str], layer: int) -> List[int]:
    pat = re.compile(rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.")
    ids = set()
    for k in weight_map.keys():
        m = pat.match(k)
        if m:
            ids.add(int(m.group(1)))
    return sorted(ids)

def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    prefix = f"model.layers.{layer}.mlp.experts.{eid}."
    def pick(cands: List[str]) -> Optional[str]:
        for suf in cands:
            k = prefix + suf
            if k in weight_map:
                return k
        return None
    up   = pick(["up_proj.weight", "w3.weight", "w1.weight"])
    gate = pick(["gate_proj.weight", "w1.weight", "w3.weight"])
    down = pick(["down_proj.weight", "w2.weight"])
    if up is None or gate is None or down is None:
        return {}
    if up == gate:
        g2 = pick(["gate_proj.weight"])
        if g2:
            gate = g2
    return {"up": up, "gate": gate, "down": down}

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], keys: List[str]) -> Dict[str, torch.Tensor]:
    by_shard: Dict[str, List[str]] = {}
    for k in keys:
        shard = weight_map.get(k, None)
        if shard is None:
            raise KeyError(f"Key not in weight_map: {k}")
        by_shard.setdefault(shard, []).append(k)

    out: Dict[str, torch.Tensor] = {}
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(sp):
            raise FileNotFoundError(f"Missing shard: {sp}")
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks:
                out[k] = f.get_tensor(k)
    return out

# ----------------------------
# Hadamard / permutations (for BASIS_MODE=hadamard_perm)
# ----------------------------
def is_power_of_two(n: int) -> bool:
    return (n > 0) and ((n & (n - 1)) == 0)

def fwht_inplace(x: torch.Tensor) -> torch.Tensor:
    # x: (..., n) contiguous
    n = x.shape[-1]
    h = 1
    y = x
    while h < n:
        y = y.view(-1, n // (2*h), 2, h)
        a = y[:, :, 0, :]
        b = y[:, :, 1, :]
        y[:, :, 0, :] = a + b
        y[:, :, 1, :] = a - b
        y = y.view(-1, n)
        h *= 2
    return y.view(x.shape)

def fwht_ortho(x: torch.Tensor) -> torch.Tensor:
    n = x.shape[-1]
    y = fwht_inplace(x.contiguous())
    return y / math.sqrt(n)

def make_perm_sign(n: int, seed: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    g = torch.Generator(device="cpu").manual_seed(seed)
    perm = torch.randperm(n, generator=g)
    inv = torch.empty_like(perm)
    inv[perm] = torch.arange(n)
    signs = (torch.randint(0, 2, (n,), generator=g, dtype=torch.int8) * 2 - 1).to(torch.float32)
    return perm, inv, signs

@torch.no_grad()
def basis_apply_right(x: torch.Tensor, perm: torch.Tensor, signs: torch.Tensor) -> torch.Tensor:
    # x @ (H P D): fwht then perm then sign
    y = fwht_ortho(x)
    y = y[:, perm]
    y = y * signs.to(y.device).view(1, -1)
    return y

@torch.no_grad()
def basis_apply_right_T(x: torch.Tensor, invperm: torch.Tensor, signs: torch.Tensor) -> torch.Tensor:
    # x @ (H P D)^T = x @ (D P^T H): sign then invperm then fwht
    y = x * signs.to(x.device).view(1, -1)
    y = y[:, invperm]
    y = fwht_ortho(y)
    return y

@torch.no_grad()
def mat_to_basis_hadamard(W: torch.Tensor, pu: torch.Tensor, ipu: torch.Tensor, su: torch.Tensor,
                          pv: torch.Tensor, ipv: torch.Tensor, sv: torch.Tensor) -> torch.Tensor:
    # X = U^T W V, U=H P_u D_u, V=H P_v D_v
    # left: U^T = D_u P_u^T H
    # right: V = H P_v D_v
    T = W
    # H on left -> hadamard on rows: H W
    T = fwht_ortho(T.t().contiguous()).t().contiguous()
    # P_u^T on left -> permute rows by invperm_u
    T = T[ipu.to(T.device), :]
    # D_u on left -> sign rows
    T = T * su.to(T.device).view(-1, 1)
    # right multiply by H -> hadamard on columns (fwht over last dim for each row)
    T = fwht_ortho(T)
    # right multiply by P_v -> permute cols by perm_v
    T = T[:, pv.to(T.device)]
    # right multiply by D_v -> sign cols
    T = T * sv.to(T.device).view(1, -1)
    return T.contiguous()

@torch.no_grad()
def mat_from_basis_hadamard(X: torch.Tensor, pu: torch.Tensor, ipu: torch.Tensor, su: torch.Tensor,
                            pv: torch.Tensor, ipv: torch.Tensor, sv: torch.Tensor) -> torch.Tensor:
    # W = U X V^T, U=H P_u D_u, V^T = D_v P_v^T H
    T = X
    # D_u on left
    T = T * su.to(T.device).view(-1, 1)
    # P_u on left
    T = T[pu.to(T.device), :]
    # H on left
    T = fwht_ortho(T.t().contiguous()).t().contiguous()
    # right: D_v
    T = T * sv.to(T.device).view(1, -1)
    # right: P_v^T -> permute cols by invperm_v
    T = T[:, ipv.to(T.device)]
    # right: H
    T = fwht_ortho(T)
    return T.contiguous()

# ----------------------------
# Calib/router: auto-detect, load, optional capture
# ----------------------------
def autodetect_calib_path() -> Optional[str]:
    cand = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
    return cand if os.path.isfile(cand) else None

def autodetect_router_path() -> Optional[str]:
    cand = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
    return cand if os.path.isfile(cand) else None

def load_calib_X(path: str, H: int) -> torch.Tensor:
    z = np.load(path, allow_pickle=False)
    if "X" not in z.files:
        raise KeyError(f"CALIB npz missing key 'X'. Keys={list(z.files)}")
    Xn = z["X"].astype(np.float32, copy=False)
    z.close()
    X = torch.from_numpy(Xn)
    if X.ndim != 2 or X.shape[1] != H:
        raise RuntimeError(f"Bad X shape {tuple(X.shape)}, expected (*,{H})")
    if X.shape[0] > cfg.CALIB_SAMPLES:
        X = X[:cfg.CALIB_SAMPLES]
    return X.to(DEVICE, dtype=DTYPE_ACC)

def load_router_P(path: str) -> np.ndarray:
    z = np.load(path, allow_pickle=False)
    if "P" not in z.files:
        raise KeyError(f"ROUTER npz missing key 'P'. Keys={list(z.files)}")
    P = z["P"].astype(np.float32, copy=False)
    z.close()
    return P

def _maybe_autopip():
    if not cfg.HF_AUTO_PIP:
        return
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", "transformers", "sentencepiece", "tokenizers"])

def _patch_dynamic_cache_compat():
    # Some remote-code models expect DynamicCache.get_usable_length
    try:
        from transformers.cache_utils import DynamicCache  # type: ignore
        if not hasattr(DynamicCache, "get_usable_length"):
            def _get_usable_length(self, seq_length: int):
                # Conservative shim
                try:
                    return int(seq_length)
                except Exception:
                    return 0
            DynamicCache.get_usable_length = _get_usable_length  # type: ignore
            log("[patch] Added DynamicCache.get_usable_length compat shim.")
    except Exception:
        pass

class _Collector:
    def __init__(self, H: int, E_total: int, max_rows: int):
        self.H = H
        self.E_total = E_total
        self.max_rows = max_rows
        self.X_chunks: List[torch.Tensor] = []
        self.P_chunks: List[torch.Tensor] = []
        self.nX = 0
        self.nP = 0

    def add_X(self, hs: torch.Tensor, attn_mask: Optional[torch.Tensor]):
        # hs: (B,T,H)
        if hs is None:
            return
        if hs.ndim == 2:
            hs = hs.unsqueeze(0)
        if hs.ndim != 3 or hs.shape[-1] != self.H:
            return
        hs = hs.detach().to(torch.float32).cpu()
        if attn_mask is not None and (not cfg.CAPTURE_KEEP_PAD):
            m = attn_mask.detach().cpu().to(torch.bool)
            # flatten tokens
            flat = hs.reshape(-1, self.H)
            mflat = m.reshape(-1)
            flat = flat[mflat]
        else:
            flat = hs.reshape(-1, self.H)
        if flat.numel() == 0:
            return
        need = self.max_rows - self.nX
        if need <= 0:
            return
        if flat.shape[0] > need:
            flat = flat[:need]
        self.X_chunks.append(flat)
        self.nX += int(flat.shape[0])

    def add_P(self, logits: torch.Tensor, attn_mask: Optional[torch.Tensor]):
        # logits: (B,T,E')
        if logits is None:
            return
        if logits.ndim == 2:
            logits = logits.unsqueeze(0)
        if logits.ndim != 3:
            return
        L = logits.detach().to(torch.float32)
        P = torch.softmax(L, dim=-1)
        P = P[..., :self.E_total].cpu()
        if attn_mask is not None and (not cfg.CAPTURE_KEEP_PAD):
            m = attn_mask.detach().cpu().to(torch.bool)
            flat = P.reshape(-1, P.shape[-1])
            mflat = m.reshape(-1)
            flat = flat[mflat]
        else:
            flat = P.reshape(-1, P.shape[-1])
        if flat.numel() == 0:
            return
        need = self.max_rows - self.nP
        if need <= 0:
            return
        if flat.shape[0] > need:
            flat = flat[:need]
        self.P_chunks.append(flat)
        self.nP += int(flat.shape[0])

def capture_XP_transformers(model_dir: str, layer_idx: int, H: int, E_total: int,
                            out_x: str, out_p: str) -> Tuple[str, Optional[str]]:
    _maybe_autopip()
    _patch_dynamic_cache_compat()
    try:
        from transformers import AutoTokenizer, AutoModelForCausalLM  # type: ignore
    except Exception as e:
        raise RuntimeError("transformers not available; set HF_AUTO_PIP=1 or install transformers.") from e

    log("[capture] Loading tokenizer/model from local files...")
    tok = AutoTokenizer.from_pretrained(
        model_dir,
        trust_remote_code=cfg.HF_TRUST_REMOTE_CODE,
        local_files_only=cfg.HF_LOCAL_FILES_ONLY,
        use_fast=True,
    )
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token if tok.eos_token is not None else tok.unk_token

    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        trust_remote_code=cfg.HF_TRUST_REMOTE_CODE,
        local_files_only=cfg.HF_LOCAL_FILES_ONLY,
        torch_dtype=torch.float16,
        device_map=None,
        low_cpu_mem_usage=True,
    )
    model.eval()
    model.to(DEVICE)

    # Locate layers list
    layers = None
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        layers = list(model.model.layers)
    elif hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        layers = list(model.transformer.h)
    elif hasattr(model, "layers"):
        layers = list(model.layers)

    if layers is None:
        raise RuntimeError("Cannot locate transformer layers list (model.model.layers / transformer.h / layers).")
    if layer_idx < 0 or layer_idx >= len(layers):
        raise RuntimeError(f"LAYER={layer_idx} out of range. Model has {len(layers)} layers.")

    layer = layers[layer_idx]

    # Best-effort locate MLP module to hook its input (X)
    mlp = None
    if hasattr(layer, "mlp"):
        mlp = layer.mlp
    else:
        # search named_modules
        for n, m in layer.named_modules():
            if n.lower().endswith("mlp"):
                mlp = m
                break
    if mlp is None:
        raise RuntimeError("Could not find layer.mlp to hook for X capture.")

    # Router discovery:
    # Prefer a module/parameter with shape (E_total, H) and name contains router/gate
    router_linear: Optional[nn.Linear] = None
    router_weight: Optional[torch.Tensor] = None

    for name, mod in layer.named_modules():
        if isinstance(mod, nn.Linear) and getattr(mod, "in_features", None) == H and getattr(mod, "out_features", 0) >= E_total:
            nm = name.lower()
            score = 0
            if "router" in nm: score += 10
            if "gate" in nm: score += 6
            if "moe" in nm: score += 3
            if mod.out_features == E_total: score += 6
            # pick highest score
            if router_linear is None or score > 0:
                router_linear = mod

    if router_linear is None:
        # Try parameters
        best = None
        best_score = -1e9
        for pname, p in layer.named_parameters(recurse=True):
            if p.ndim == 2 and p.shape[0] >= E_total and p.shape[1] == H:
                nm = pname.lower()
                score = 0
                if "router" in nm: score += 10
                if "gate" in nm: score += 6
                if p.shape[0] == E_total: score += 6
                score -= 0.01 * float(p.shape[0] - E_total)
                if score > best_score:
                    best_score = score
                    best = p
        if best is not None:
            router_weight = best.detach()

    if router_linear is not None:
        log(f"[capture] Router candidate Linear found: in={router_linear.in_features} out={router_linear.out_features}")
    elif router_weight is not None:
        log(f"[capture] Router candidate weight found: shape={tuple(router_weight.shape)}")
    else:
        log("[capture] Router not found; will capture X only.")

    coll = _Collector(H=H, E_total=E_total, max_rows=cfg.CALIB_SAMPLES)

    # Hooks
    attn_mask_holder = {"mask": None}

    def mlp_pre_hook(_m, inputs):
        hs = inputs[0]
        am = attn_mask_holder["mask"]
        coll.add_X(hs, am)

        # If no router module, compute P from weight if available
        if router_linear is None and router_weight is not None and hs is not None:
            # hs: (B,T,H)
            if hs.ndim == 2:
                hs2 = hs.unsqueeze(0)
            else:
                hs2 = hs
            # logits = hs @ W^T
            W = router_weight.to(hs2.device, dtype=torch.float32)
            logits = torch.matmul(hs2.to(torch.float32), W.t())
            coll.add_P(logits, am)

    h_mlp = mlp.register_forward_pre_hook(mlp_pre_hook)

    h_router = None
    if router_linear is not None:
        def router_hook(_m, inputs, output):
            am = attn_mask_holder["mask"]
            out = output[0] if isinstance(output, (tuple, list)) else output
            if torch.is_tensor(out):
                # try to shape it to (B,T,E)
                if out.ndim == 2:
                    out2 = out.unsqueeze(0)
                else:
                    out2 = out
                coll.add_P(out2, am)
        h_router = router_linear.register_forward_hook(router_hook)

    # Drive forward passes
    # Use padding=max_length so you actually get many tokens; drop pads by attention_mask unless CAPTURE_KEEP_PAD=1.
    for it in range(cfg.CAPTURE_ITERS):
        text = cfg.CAPTURE_TEXT
        enc = tok(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=cfg.CAPTURE_MAX_TOKENS,
            padding="max_length",
        )
        # batch replicate
        for k in enc:
            if enc[k].ndim == 2 and cfg.CAPTURE_BATCH > 1:
                enc[k] = enc[k].repeat(cfg.CAPTURE_BATCH, 1)
        attn_mask_holder["mask"] = enc.get("attention_mask", None)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        with torch.inference_mode():
            _ = model(**enc, use_cache=False)

        if (it + 1) % 4 == 0 or it == 0 or (it + 1) == cfg.CAPTURE_ITERS:
            log(f"[capture] iter {it+1}/{cfg.CAPTURE_ITERS}  nX={coll.nX} nP={coll.nP}")
        if coll.nX >= cfg.CALIB_SAMPLES and (not cfg.RIDGE_WEIGHTED or coll.nP >= cfg.CALIB_SAMPLES):
            break

    h_mlp.remove()
    if h_router is not None:
        h_router.remove()

    if coll.nX == 0:
        raise RuntimeError("Capture failed: collected 0 X rows.")

    X = torch.cat(coll.X_chunks, dim=0)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
    save_npz(out_x, X=X)
    log(f"[capture] wrote X -> {out_x}  shape={tuple(X.shape)}")

    p_written = None
    if coll.nP > 0:
        P = torch.cat(coll.P_chunks, dim=0)[:min(cfg.CALIB_SAMPLES, coll.nP)].numpy().astype(np.float32)
        # Align length with X
        N = min(P.shape[0], X.shape[0])
        if N < X.shape[0]:
            X = X[:N]
            save_npz(out_x, X=X)
        P = P[:N]
        save_npz(out_p, P=P)
        log(f"[capture] wrote P -> {out_p}  shape={tuple(P.shape)}")
        p_written = out_p
    else:
        log("[capture] Router P not captured.")

    return out_x, p_written

# ----------------------------
# Build Ws (ridge) + cache
# ----------------------------
def ws_cache_path() -> str:
    return os.path.join(cfg.OUTPUT_DIR, f"Ws_cache_layer{cfg.LAYER}_E{cfg.MAX_EXPERTS}_{cfg.LIN_MODE}_v10_2.npz")

def ws_meta(eids: List[int]) -> dict:
    return dict(
        time=now(),
        model_dir=cfg.MODEL_DIR,
        output_dir=cfg.OUTPUT_DIR,
        layer=cfg.LAYER,
        max_experts=cfg.MAX_EXPERTS,
        expert_ids=eids,
        lin_mode=cfg.LIN_MODE,
        ridge_damp=cfg.RIDGE_DAMP,
        ridge_weighted=cfg.RIDGE_WEIGHTED,
        calib_path=cfg.CALIB_PATH or "",
        calib_samples=cfg.CALIB_SAMPLES,
        router_path=cfg.ROUTER_PATH or "",
        router_global=cfg.ROUTER_EIDS_ARE_GLOBAL,
        normalize_w=cfg.NORMALIZE_W,
        seed=SEED,
        device=str(DEVICE),
    )

@torch.no_grad()
def forward_mlp(X: torch.Tensor, W_gate: torch.Tensor, W_up: torch.Tensor, W_down: torch.Tensor) -> torch.Tensor:
    Xf = X.to(DTYPE_ACC)
    up = Xf @ W_up.to(DTYPE_ACC).t()
    gate = Xf @ W_gate.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    y = hid @ W_down.to(DTYPE_ACC).t()
    return y

def ensure_calib_router(H: int, E_total: int):
    # Auto-detect paths
    if not cfg.CALIB_PATH:
        cand = autodetect_calib_path()
        if cand:
            cfg.CALIB_PATH = cand
            log(f"[calib] CALIB_PATH not set -> auto-found {cfg.CALIB_PATH}")

    if not cfg.ROUTER_PATH:
        candp = autodetect_router_path()
        if candp:
            cfg.ROUTER_PATH = candp
            log(f"[router] ROUTER_PATH not set -> auto-found {cfg.ROUTER_PATH}")

    # Capture if needed/forced
    need_capture = cfg.CAPTURE_FORCE or (cfg.CAPTURE_ENABLE and (not cfg.CALIB_PATH))
    if need_capture:
        out_x = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
        out_p = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
        log("[calib] capturing X (and maybe P) via transformers...")
        x_path, p_path = capture_XP_transformers(cfg.MODEL_DIR, cfg.LAYER, H, E_total, out_x, out_p)
        cfg.CALIB_PATH = x_path
        if p_path:
            cfg.ROUTER_PATH = p_path

def load_or_build_Ws() -> Tuple[List[int], torch.Tensor]:
    # Read model tensors
    wm = read_index(cfg.MODEL_DIR)
    all_eids = find_layer_expert_ids(wm, cfg.LAYER)
    if not all_eids:
        raise RuntimeError(f"No experts found at layer={cfg.LAYER}")
    eids = all_eids[:cfg.MAX_EXPERTS]
    log(f"[found] layer={cfg.LAYER} total_experts={len(all_eids)} using={len(eids)} eids={eids}")

    # Try cache
    cpath = ws_cache_path()
    if os.path.isfile(cpath):
        z = load_npz(cpath)
        if "meta" in z and "Ws" in z and "expert_ids" in z:
            if _decode_meta(z["meta"]) == ws_meta(eids):
                Ws = torch.from_numpy(z["Ws"]).to(DTYPE_ACC).to(DEVICE)
                log(f"[cache] loaded Ws -> {cpath}  Ws={tuple(Ws.shape)}")
                return [int(x) for x in z["expert_ids"].tolist()], Ws
        log("[cache] Ws meta mismatch -> rebuilding.")

    # Load expert tensors
    per_e = {}
    need_keys = []
    for eid in eids:
        kk = pick_expert_tensor_keys(wm, cfg.LAYER, eid)
        if not kk:
            raise RuntimeError(f"Expert {eid} missing required tensors in index.")
        per_e[eid] = kk
        need_keys += [kk["up"], kk["down"], kk["gate"]]

    log("[load] reading tensors from shards ...")
    T = load_tensors_from_shards(cfg.MODEL_DIR, wm, sorted(set(need_keys)))

    W_up0 = T[per_e[eids[0]]["up"]]
    dff, H = int(W_up0.shape[0]), int(W_up0.shape[1])
    log(f"[shape] H={H} d_ff={dff}")

    if cfg.LIN_MODE != "ridge":
        raise RuntimeError("This runner supports LIN_MODE=ridge only.")

    # Ensure calib/router
    ensure_calib_router(H=H, E_total=len(all_eids))

    if not cfg.CALIB_PATH or (not os.path.isfile(cfg.CALIB_PATH)):
        raise RuntimeError("CALIB_PATH is required for ridge. Set CALIB_PATH or CAPTURE_ENABLE=1.")

    X = load_calib_X(cfg.CALIB_PATH, H)
    log(f"[calib] X loaded: {tuple(X.shape)}  (path={cfg.CALIB_PATH})")

    P = None
    if cfg.RIDGE_WEIGHTED:
        if cfg.ROUTER_PATH and os.path.isfile(cfg.ROUTER_PATH):
            P = load_router_P(cfg.ROUTER_PATH)
            log(f"[router] P loaded: {tuple(P.shape)}  (path={cfg.ROUTER_PATH})")
        else:
            log("[router] RIDGE_WEIGHTED=1 but ROUTER_PATH missing -> forcing RIDGE_WEIGHTED=0")
            cfg.RIDGE_WEIGHTED = False

    # Ridge precompute
    Xf = X.to(DTYPE_ACC)
    Hn = H
    I = torch.eye(Hn, dtype=DTYPE_ACC, device=DEVICE)
    XtX = Xf.t() @ Xf
    lam = cfg.RIDGE_DAMP * float(torch.trace(XtX).item()) / float(Hn)
    cholG = torch.linalg.cholesky(XtX + lam * I)

    Ws_list: List[torch.Tensor] = []
    for i, eid in enumerate(tqdm(eids, desc="Build Ws (ridge)")):
        W_up = T[per_e[eid]["up"]].to(DEVICE)
        W_dn = T[per_e[eid]["down"]].to(DEVICE)
        W_gt = T[per_e[eid]["gate"]].to(DEVICE)
        Y = forward_mlp(X, W_gt, W_up, W_dn).to(DTYPE_ACC)

        if cfg.RIDGE_WEIGHTED and (P is not None):
            if cfg.ROUTER_EIDS_ARE_GLOBAL:
                if eid >= P.shape[1]:
                    raise RuntimeError(f"P shape {P.shape} cannot index eid={eid}")
                w = torch.from_numpy(P[:X.shape[0], eid]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
            else:
                if i >= P.shape[1]:
                    raise RuntimeError(f"P shape {P.shape} cannot index i={i}")
                w = torch.from_numpy(P[:X.shape[0], i]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)

            sw = torch.sqrt(w + 1e-12).view(-1, 1)
            Xw = Xf * sw
            Yw = Y * sw
            XtX_e = Xw.t() @ Xw
            lam_e = cfg.RIDGE_DAMP * float(torch.trace(XtX_e).item()) / float(Hn)
            chol = torch.linalg.cholesky(XtX_e + lam_e * I)
            XtY = Xw.t() @ Yw
            Wt = torch.cholesky_solve(XtY, chol)
            W = Wt.t().contiguous()
        else:
            XtY = Xf.t() @ Y
            Wt = torch.cholesky_solve(XtY, cholG)
            W = Wt.t().contiguous()

        if cfg.NORMALIZE_W:
            fn = torch.linalg.norm(W, ord="fro").clamp_min(1e-12)
            W = (W / fn).contiguous()

        Ws_list.append(W)

    Ws = torch.stack(Ws_list, dim=0).to(DTYPE_ACC).to(DEVICE)  # (E,H,H)

    # Save cache
    save_npz(
        cpath,
        meta=_encode_meta(ws_meta(eids)),
        expert_ids=np.array(eids, dtype=np.int32),
        Ws=Ws.detach().cpu().numpy().astype(np.float32),
    )
    log(f"[cache] wrote Ws -> {cpath}  size={os.path.getsize(cpath)/1e6:.2f} MB")
    return eids, Ws

# ----------------------------
# Clustering helpers (dense_train)
# ----------------------------
@torch.no_grad()
def random_proj_features(Ws: torch.Tensor, d: int) -> torch.Tensor:
    # features from row/col energy projected to 2d
    E, n, _ = Ws.shape
    g = torch.Generator(device="cpu").manual_seed(SEED + 17)
    R = (torch.randint(0, 2, (n, d), generator=g, dtype=torch.int8) * 2 - 1).to(DTYPE_ACC).to(DEVICE)
    feats = []
    for e in range(E):
        W = Ws[e]
        row = torch.diag(W @ W.t())
        col = torch.diag(W.t() @ W)
        feats.append(torch.cat([(row @ R), (col @ R)], dim=0).unsqueeze(0))
    return torch.cat(feats, dim=0)  # (E, 2d)

@torch.no_grad()
def kmeans_torch(X: torch.Tensor, k: int, iters: int, restarts: int) -> Tuple[torch.Tensor, torch.Tensor]:
    best_lab = None
    best_C = None
    best_inertia = float("inf")
    g = torch.Generator(device="cpu").manual_seed(SEED + 999)

    for r in range(max(1, restarts)):
        idx = torch.randperm(X.shape[0], generator=g)[:k]
        C = X[idx].clone()
        for _ in range(iters):
            dist = torch.cdist(X, C)
            lab = dist.argmin(dim=1)
            for j in range(k):
                m = (lab == j)
                if m.any():
                    C[j] = X[m].mean(dim=0)
        dist = torch.cdist(X, C)
        inertia = float((dist.min(dim=1).values ** 2).sum().item())
        if inertia < best_inertia:
            best_inertia = inertia
            best_lab = lab.clone()
            best_C = C.clone()
    return best_lab.to(torch.int64), best_C

@torch.no_grad()
def relabel_contiguous(labels: torch.Tensor) -> torch.Tensor:
    labels = labels.to(torch.int64)
    uniq = torch.unique(labels)
    out = labels.clone()
    for new, old in enumerate(uniq.tolist()):
        out[labels == int(old)] = int(new)
    return out.to(torch.int64)

@torch.no_grad()
def merge_small_clusters(X: torch.Tensor, labels: torch.Tensor, min_size: int) -> torch.Tensor:
    # merge clusters with size < min_size into nearest (by centroid distance)
    labels = relabel_contiguous(labels)
    if min_size <= 1:
        return labels
    while True:
        K = int(labels.max().item()) + 1
        counts = torch.bincount(labels, minlength=K)
        small = (counts < min_size).nonzero(as_tuple=False).flatten()
        if small.numel() == 0:
            break
        # centroids
        C = torch.stack([X[labels == k].mean(dim=0) for k in range(K)], dim=0)
        for c in small.tolist():
            idxs = (labels == int(c)).nonzero(as_tuple=False).flatten()
            if idxs.numel() == 0:
                continue
            # nearest other centroid
            dist = torch.cdist(C[int(c)].unsqueeze(0), C).squeeze(0)
            dist[int(c)] = 1e9
            tgt = int(dist.argmin().item())
            labels[idxs] = tgt
        labels = relabel_contiguous(labels)
    return labels

@torch.no_grad()
def hierarchical_split(X: torch.Tensor, labels: torch.Tensor, max_size: int, max_k: int, split_iters: int) -> torch.Tensor:
    # split the largest cluster with k=2 until <=max_size or K>=max_k
    labels = relabel_contiguous(labels)
    if max_size <= 0:
        return labels
    while True:
        K = int(labels.max().item()) + 1
        if K >= max_k:
            break
        counts = torch.bincount(labels, minlength=K)
        biggest = int(torch.argmax(counts).item())
        bigsz = int(counts[biggest].item())
        if bigsz <= max_size:
            break
        idxs = (labels == biggest).nonzero(as_tuple=False).flatten()
        if idxs.numel() < 2:
            break
        Xsub = X[idxs]
        sub_lab, _ = kmeans_torch(Xsub, k=2, iters=split_iters, restarts=1)
        a = idxs[sub_lab == 0]
        b = idxs[sub_lab == 1]
        if a.numel() == 0 or b.numel() == 0:
            break
        labels[b] = K
        labels = relabel_contiguous(labels)
    return labels

# ----------------------------
# Dense_train basis training
# ----------------------------
class OrthoParam(nn.Module):
    def __init__(self, n: int, init: str):
        super().__init__()
        if init == "random":
            g = torch.Generator(device="cpu").manual_seed(SEED + 333)
            M = torch.randn(n, n, generator=g, dtype=DTYPE_ACC)
        else:
            M = torch.eye(n, dtype=DTYPE_ACC)
        self.M = nn.Parameter(M.to(DEVICE))

    def orthogonal(self) -> torch.Tensor:
        Q, _ = torch.linalg.qr(self.M)
        return Q

def schedule(step: int, warmup: int, total: int) -> float:
    if total <= 0:
        return 1.0
    if step <= warmup:
        return 0.0
    return float(min(1.0, max(0.0, (step - warmup) / max(1, (total - warmup)))))

@torch.no_grad()
def slice_X_batch(Ws_batch: torch.Tensor, U: torch.Tensor, V: torch.Tensor, S: torch.Tensor) -> torch.Tensor:
    U_S = U[:, S]                 # (n,s)
    V_S = V[:, S]                 # (n,s)
    T = Ws_batch @ V_S            # (Eb,n,s)
    Xs = torch.matmul(U_S.t().unsqueeze(0), T)  # (Eb,s,s)
    return Xs

def offdiag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    Xoff = Xs - torch.diag_embed(D)
    return Xoff.abs().mean()

def diag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    return D.abs().mean()

def block_group_sparsity_penalty(Xs: torch.Tensor, block: int) -> torch.Tensor:
    Eb, s, _ = Xs.shape
    b = int(block)
    if b <= 0:
        return torch.zeros((), dtype=DTYPE_ACC, device=Xs.device)
    nb = s // b
    if nb <= 0:
        return torch.zeros((), dtype=DTYPE_ACC, device=Xs.device)
    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()
    Xb = X.view(Eb, nb, b, nb, b).permute(0, 1, 3, 2, 4).contiguous()
    Eblk = (Xb * Xb).sum(dim=(3, 4))  # (Eb,nb,nb)
    P = Eblk.mean(dim=0)
    return torch.sqrt(P + 1e-12).sum() / (P.sum() + 1e-12)

@torch.no_grad()
def make_guidance_mask_from_Xs(Xs: torch.Tensor, block: int, target: float, max_blocks: int) -> Tuple[torch.Tensor, float, int]:
    Eb, s, _ = Xs.shape
    b = int(block)
    if b <= 0:
        return torch.ones(s, s, dtype=DTYPE_ACC, device=Xs.device), 1.0, 0
    nb = s // b
    if nb <= 0:
        return torch.ones(s, s, dtype=DTYPE_ACC, device=Xs.device), 1.0, 0
    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()
    Xb = X.view(Eb, nb, b, nb, b).permute(0, 1, 3, 2, 4).contiguous()
    Eg = (Xb * Xb).sum(dim=(3, 4)).mean(dim=0)  # (nb,nb)
    tot = float((X * X).sum().item()) / max(1, Eb)
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    csum = torch.cumsum(flat[order], dim=0)
    frac = csum / max(tot, 1e-12)
    need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else flat.numel())
    K = min(need, max_blocks, flat.numel())
    pick = order[:K]
    mask = torch.zeros(s2, s2, dtype=DTYPE_ACC, device=Xs.device)
    for idx in pick.tolist():
        bi = idx // nb
        bj = idx % nb
        i0 = bi * b
        j0 = bj * b
        mask[i0:i0 + b, j0:j0 + b] = 1.0
    if s2 < s:
        full = torch.zeros(s, s, dtype=DTYPE_ACC, device=Xs.device)
        full[:s2, :s2] = mask
        mask = full
    ef = float(frac[K - 1].item()) if K > 0 else 0.0
    return mask, ef, int(K)

# ----------------------------
# Block selection + residual building
# ----------------------------
@torch.no_grad()
def block_energy_grid(X: torch.Tensor, b: int) -> torch.Tensor:
    n = X.shape[0]
    nb = (n + b - 1) // b
    if (n % b) != 0:
        pad = nb * b - n
        Xp = torch.zeros(nb * b, nb * b, dtype=X.dtype, device=X.device)
        Xp[:n, :n] = X
        X = Xp
    Xb = X.view(nb, b, nb, b).permute(0, 2, 1, 3).contiguous()  # (nb,nb,b,b)
    E = (Xb * Xb).sum(dim=(2, 3))  # (nb,nb)
    return E

@torch.no_grad()
def pick_top_blocks_from_energy(Eg: torch.Tensor, tot_energy: float, b: int, target: float, max_blocks: int) -> Tuple[List[Tuple[int, int, int, int]], float]:
    nb = Eg.shape[0]
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    csum = torch.cumsum(flat[order], dim=0)
    frac = csum / max(tot_energy, 1e-12)
    need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else flat.numel())
    K = min(need, max_blocks, flat.numel())
    pick = order[:K].tolist()
    kept = float(flat[order[:K]].sum().item())
    blocks = []
    for idx in pick:
        bi = idx // nb
        bj = idx % nb
        blocks.append((bi * b, bj * b, b, b))
    return blocks, kept / max(tot_energy, 1e-12)

@torch.no_grad()
def choose_core_blocks(X_list: List[torch.Tensor]) -> dict:
    mode = cfg.CORE_MODE
    agg = cfg.CORE_AGG
    b = int(cfg.CORE_BLOCK)
    target = float(cfg.CORE_TARGET)
    max_blocks = int(cfg.CORE_MAX_BLOCKS)

    if mode == "none":
        return {"mode": "none", "blocks_shared": [], "blocks_per_expert": None, "energy_fracs": []}

    n = X_list[0].shape[0]
    if b <= 0:
        return {"mode": "none", "blocks_shared": [], "blocks_per_expert": None, "energy_fracs": []}

    if mode == "blockdiag":
        nb = (n + b - 1) // b
        diagE = torch.zeros(nb, dtype=DTYPE_ACC, device=X_list[0].device)
        tots = []
        for X in X_list:
            tots.append(float((X * X).sum().item()))
            Eg = block_energy_grid(X, b).to(DTYPE_ACC)
            diagE += torch.diagonal(Eg, 0)
        tot = float(np.mean(tots))
        diagE /= max(1, len(X_list))
        order = torch.argsort(diagE, descending=True)
        csum = torch.cumsum(diagE[order], dim=0)
        frac = csum / max(tot, 1e-12)
        need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else nb)
        K = min(need, max_blocks, nb)
        blocks = []
        for bi in order[:K].tolist():
            i0 = bi * b
            blocks.append((i0, i0, b, b))
        ef = float(frac[K - 1].item()) if K > 0 else 0.0
        return {"mode": "blockdiag", "blocks_shared": blocks, "blocks_per_expert": None, "energy_fracs": [ef]}

    if mode == "blocktopk":
        Eg_all = []
        tots = []
        for X in X_list:
            tots.append(float((X * X).sum().item()))
            Eg_all.append(block_energy_grid(X, b).to(DTYPE_ACC))
        tot = float(np.mean(tots))
        if agg == "max":
            Eg = torch.stack(Eg_all, dim=0).amax(dim=0)
        else:
            Eg = torch.stack(Eg_all, dim=0).mean(dim=0)
        blocks, ef = pick_top_blocks_from_energy(Eg, tot, b, target, max_blocks)
        return {"mode": "blocktopk", "blocks_shared": blocks, "blocks_per_expert": None, "energy_fracs": [ef]}

    if mode == "blocktopk_perexpert":
        blocks_per = []
        efracs = []
        for X in X_list:
            tot = float((X * X).sum().item())
            Eg = block_energy_grid(X, b).to(DTYPE_ACC)
            blocks, ef = pick_top_blocks_from_energy(Eg, tot, b, target, max_blocks)
            blocks_per.append(blocks)
            efracs.append(ef)
        return {"mode": "blocktopk_perexpert", "blocks_shared": [], "blocks_per_expert": blocks_per, "energy_fracs": efracs}

    raise ValueError("CORE_MODE must be blocktopk_perexpert|blocktopk|blockdiag|none")

@torch.no_grad()
def pick_top_blocks_values(X: torch.Tensor, blocks: List[Tuple[int, int, int, int]]) -> List[Tuple[int, int, torch.Tensor]]:
    out = []
    n = X.shape[0]
    for (i0, j0, h, w) in blocks:
        h = min(h, n - i0)
        w = min(w, n - j0)
        if h <= 0 or w <= 0:
            continue
        out.append((i0, j0, X[i0:i0 + h, j0:j0 + w].clone()))
    return out

# ----------------------------
# Randomized SVD for residual mean
# ----------------------------
@torch.no_grad()
def rand_svd(A: torch.Tensor, r: int, n_iter: int = 2) -> Tuple[torch.Tensor, torch.Tensor]:
    # returns (U, V) with shapes (n,r), (n,r) approximating A ≈ U diag(s) V^T
    n = A.shape[0]
    r = min(r, n)
    g = torch.Generator(device="cpu").manual_seed(SEED + 777)
    Omega = torch.randn(n, r, generator=g, dtype=DTYPE_ACC, device=A.device)
    Y = A @ Omega
    for _ in range(n_iter):
        Y = A @ (A.t() @ Y)
    Q, _ = torch.linalg.qr(Y)
    B = Q.t() @ A
    # small SVD
    Uhat, _, Vh = torch.linalg.svd(B, full_matrices=False)
    U = (Q @ Uhat[:, :r]).contiguous()
    V = (Vh.t()[:, :r]).contiguous()
    return U, V

# ----------------------------
# Quantization helpers for block payload
# ----------------------------
def q_block_int8(B: torch.Tensor) -> Tuple[np.ndarray, np.float16]:
    x = B.detach().cpu().to(torch.float32)
    maxabs = float(x.abs().max().item())
    if maxabs < 1e-12:
        scale = np.float16(1.0)
        q = np.zeros_like(x.numpy(), dtype=np.int8)
        return q, scale
    scale_f = maxabs / 127.0
    q = torch.clamp(torch.round(x / scale_f), -127, 127).to(torch.int8).cpu().numpy()
    return q, np.float16(scale_f)

# ----------------------------
# Payload build + runtime apply (for eval)
# ----------------------------
class Payload:
    def __init__(self):
        self.meta: dict = {}
        self.expert_ids: List[int] = []
        self.cluster_of_pos: List[int] = []
        self.clusters: List[List[int]] = []  # expert positions per cluster

        # Basis per cluster:
        # dense_train: U,V tensors
        # hadamard_perm: perms/signs (global; we store in cluster 0)
        self.U: List[Optional[torch.Tensor]] = []
        self.V: List[Optional[torch.Tensor]] = []
        self.pu: Optional[torch.Tensor] = None
        self.ipu: Optional[torch.Tensor] = None
        self.su: Optional[torch.Tensor] = None
        self.pv: Optional[torch.Tensor] = None
        self.ipv: Optional[torch.Tensor] = None
        self.sv: Optional[torch.Tensor] = None

        # Core blocks per expert position (ragged)
        self.core_blocks: List[List[Tuple[int, int, torch.Tensor]]] = []

        # Residual low-rank per cluster
        self.DL: List[Optional[torch.Tensor]] = []
        self.DR: List[Optional[torch.Tensor]] = []

        # gamma per expert position
        self.gam: List[Optional[torch.Tensor]] = []

        # Residual blocks per expert position
        self.res_blocks: List[List[Tuple[int, int, torch.Tensor]]] = []

    @torch.no_grad()
    def apply_basis(self, x: torch.Tensor, c: int, right: bool) -> torch.Tensor:
        # For vectors: apply x @ U (right=True uses U), or x @ V^T (right=False uses V^T in dense case)
        if cfg.BASIS_MODE == "hadamard_perm":
            assert self.pu is not None and self.ipu is not None and self.su is not None
            assert self.pv is not None and self.ipv is not None and self.sv is not None
            if right:
                # x @ U  (use u params)
                return basis_apply_right(x, self.pu, self.su)
            else:
                # x @ V^T
                return basis_apply_right_T(x, self.ipv, self.sv)
        else:
            U = self.U[c]
            V = self.V[c]
            assert U is not None and V is not None
            if right:
                return x @ U
            else:
                return x @ V.t()

    @torch.no_grad()
    def apply_expert(self, x: torch.Tensor, pos: int) -> torch.Tensor:
        # Apply approximate W for expert position pos to batch x: y = x @ W_hat
        c = self.cluster_of_pos[pos]
        if cfg.BASIS_MODE == "hadamard_perm":
            # z = x @ U
            z = basis_apply_right(x, self.pu, self.su)  # type: ignore[arg-type]
        else:
            U = self.U[c]; V = self.V[c]
            assert U is not None and V is not None
            z = x @ U

        # u = z @ X_hat  (block ops in basis)
        n = z.shape[1]
        u = torch.zeros_like(z)

        # Core
        for (i0, j0, Bc) in self.core_blocks[pos]:
            h, w = Bc.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ Bc

        # Low-rank
        DL = self.DL[c]; DR = self.DR[c]
        assert DL is not None and DR is not None
        g = self.gam[pos]
        assert g is not None
        u += ((z @ DL) * g.view(1, -1)) @ DR.t()

        # Residual blocks
        for (i0, j0, Bb) in self.res_blocks[pos]:
            h, w = Bb.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ Bb

        # Back to original basis: y = u @ V^T
        if cfg.BASIS_MODE == "hadamard_perm":
            y = basis_apply_right_T(u, self.ipv, self.sv)  # type: ignore[arg-type]
        else:
            V = self.V[c]
            assert V is not None
            y = u @ V.t()
        assert y.shape[1] == n
        return y

    @torch.no_grad()
    def apply_mixture(self, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
        # routed are expert positions [0..E-1] in this script
        y = torch.zeros_like(x)
        for a, epos in zip(gates.tolist(), routed):
            y += float(a) * self.apply_expert(x, int(epos))
        return y

# ----------------------------
# Build payload for cluster
# ----------------------------
@torch.no_grad()
def build_payload_for_cluster_dense(Ws: torch.Tensor, idx_pos: List[int], U: torch.Tensor, V: torch.Tensor) -> Tuple[List[List[Tuple[int,int,torch.Tensor]]], torch.Tensor, torch.Tensor, List[torch.Tensor], List[List[Tuple[int,int,torch.Tensor]]]]:
    # X_e = U^T W_e V
    X_list = [(U.t() @ Ws[p] @ V).contiguous() for p in idx_pos]

    core = choose_core_blocks(X_list)
    core_blocks_per_expert: List[List[Tuple[int,int,torch.Tensor]]] = []

    if core["blocks_per_expert"] is not None:
        for X, blocks in zip(X_list, core["blocks_per_expert"]):
            core_blocks_per_expert.append(pick_top_blocks_values(X, blocks))
    else:
        shared = core["blocks_shared"]
        for X in X_list:
            core_blocks_per_expert.append(pick_top_blocks_values(X, shared))

    # Residuals after core
    R_list = []
    for X, cb in zip(X_list, core_blocks_per_expert):
        Xc = torch.zeros_like(X)
        for (i0, j0, Bc) in cb:
            h, w = Bc.shape
            Xc[i0:i0+h, j0:j0+w] = Bc
        R_list.append((X - Xc).contiguous())

    # Low-rank shared per cluster using randomized SVD on mean residual
    Rmean = torch.stack(R_list, dim=0).mean(dim=0)
    r = min(cfg.RES_RANK, Rmean.shape[0])
    DL, DR = rand_svd(Rmean, r=r, n_iter=2)

    # gamma + residual blocks
    gam_list: List[torch.Tensor] = []
    res_blocks_per_expert: List[List[Tuple[int,int,torch.Tensor]]] = []
    for Rm in R_list:
        # diag coefficients: g = diag(DL^T R DR)
        g = torch.sum(DL * (Rm @ DR), dim=0)  # (r,)
        gam_list.append(g.contiguous())
        R2 = (Rm - (DL * g.view(1, -1)) @ DR.t()).contiguous()

        # pick top RES_BLOCKS blocks by energy grid
        b = int(cfg.RES_BSIZE)
        Eg = block_energy_grid(R2, b).to(DTYPE_ACC)
        flat = Eg.reshape(-1)
        order = torch.argsort(flat, descending=True)[:min(cfg.RES_BLOCKS, flat.numel())].tolist()
        nb = Eg.shape[0]
        blocks = []
        for idv in order:
            bi = idv // nb
            bj = idv % nb
            i0 = bi * b
            j0 = bj * b
            hh = min(b, R2.shape[0] - i0)
            ww = min(b, R2.shape[1] - j0)
            if hh > 0 and ww > 0:
                blocks.append((i0, j0, R2[i0:i0+hh, j0:j0+ww].clone()))
        res_blocks_per_expert.append(blocks)

    return core_blocks_per_expert, DL.contiguous(), DR.contiguous(), gam_list, res_blocks_per_expert

@torch.no_grad()
def refine_expert_blocks_dense(Ws: torch.Tensor, pos: int, U: torch.Tensor, V: torch.Tensor,
                              core_blocks: List[Tuple[int,int,torch.Tensor]],
                              DL: torch.Tensor, DR: torch.Tensor, g: torch.Tensor,
                              res_blocks: List[Tuple[int,int,torch.Tensor]]) -> List[Tuple[int,int,torch.Tensor]]:
    # Add extra residual blocks until error <= target or limit reached
    X = (U.t() @ Ws[pos] @ V).contiguous()

    def reconstruct() -> torch.Tensor:
        Xc = torch.zeros_like(X)
        for (i0, j0, Bc) in core_blocks:
            h, w = Bc.shape
            Xc[i0:i0+h, j0:j0+w] = Bc
        Xlr = (DL * g.view(1, -1)) @ DR.t()
        Xr = torch.zeros_like(X)
        for (i0, j0, Bb) in res_blocks:
            h, w = Bb.shape
            Xr[i0:i0+h, j0:j0+w] += Bb
        return (Xc + Xlr + Xr).contiguous()

    Xhat = reconstruct()
    denom = torch.linalg.norm(X, ord="fro").clamp_min(1e-12)
    err = float((torch.linalg.norm(Xhat - X, ord="fro") / denom).item())
    if err <= cfg.REFINE_ERR_TARGET:
        return res_blocks

    # compute current residual
    R = (X - Xhat).contiguous()
    added = 0
    b = int(cfg.REFINE_BSIZE)
    while err > cfg.REFINE_ERR_TARGET and added < cfg.REFINE_MAX_EXTRA:
        Eg = block_energy_grid(R, b).to(DTYPE_ACC)
        flat = Eg.reshape(-1)
        if float(flat.max().item()) <= 1e-18:
            break
        nb = Eg.shape[0]
        idx = int(torch.argmax(flat).item())
        bi = idx // nb
        bj = idx % nb
        i0 = bi * b
        j0 = bj * b
        hh = min(b, R.shape[0] - i0)
        ww = min(b, R.shape[1] - j0)
        if hh <= 0 or ww <= 0:
            break
        Bb = R[i0:i0+hh, j0:j0+ww].clone()
        res_blocks.append((i0, j0, Bb))
        # update residual by removing the added block
        R[i0:i0+hh, j0:j0+ww] -= Bb
        added += 1

        if added % 16 == 0 or added == cfg.REFINE_MAX_EXTRA:
            Xhat = reconstruct()
            err = float((torch.linalg.norm(Xhat - X, ord="fro") / denom).item())

    return res_blocks

# ----------------------------
# Serialize payload (ragged blocks without pickle)
# ----------------------------
def _pack_blocks_ragged(blocks_per_expert: List[List[Tuple[int,int,torch.Tensor]]], qmode: str) -> Dict[str, np.ndarray]:
    # Returns dict of arrays:
    #   exp_ptr: (E+1,) block index offsets
    #   blk_i0, blk_j0, blk_h, blk_w: (B,)
    #   blk_ptr: (B+1,) value offsets
    #   blk_val OR blk_q + blk_scale
    E = len(blocks_per_expert)
    exp_ptr = [0]
    blk_i0 = []
    blk_j0 = []
    blk_h = []
    blk_w = []
    blk_ptr = [0]
    vals_f16 = []
    vals_i8 = []
    scales = []

    for e in range(E):
        lst = blocks_per_expert[e]
        for (i0, j0, B) in lst:
            h, w = B.shape
            blk_i0.append(int(i0))
            blk_j0.append(int(j0))
            blk_h.append(int(h))
            blk_w.append(int(w))
            if qmode == "int8":
                q, sc = q_block_int8(B)
                vals_i8.append(q.reshape(-1))
                scales.append(sc)
                blk_ptr.append(blk_ptr[-1] + q.size)
            else:
                v = B.detach().cpu().to(torch.float16).numpy().reshape(-1)
                vals_f16.append(v)
                blk_ptr.append(blk_ptr[-1] + v.size)
        exp_ptr.append(len(blk_i0))

    out: Dict[str, np.ndarray] = {}
    out["exp_ptr"] = np.array(exp_ptr, dtype=np.int32)
    out["blk_i0"] = np.array(blk_i0, dtype=np.int16)
    out["blk_j0"] = np.array(blk_j0, dtype=np.int16)
    out["blk_h"] = np.array(blk_h, dtype=np.int16)
    out["blk_w"] = np.array(blk_w, dtype=np.int16)
    out["blk_ptr"] = np.array(blk_ptr, dtype=np.int64)

    if qmode == "int8":
        if vals_i8:
            out["blk_q"] = np.concatenate(vals_i8, axis=0).astype(np.int8, copy=False)
        else:
            out["blk_q"] = np.zeros((0,), dtype=np.int8)
        out["blk_scale"] = np.array(scales, dtype=np.float16)
    else:
        if vals_f16:
            out["blk_val"] = np.concatenate(vals_f16, axis=0).astype(np.float16, copy=False)
        else:
            out["blk_val"] = np.zeros((0,), dtype=np.float16)
    return out

def _unpack_blocks_for_runtime(pack: Dict[str, np.ndarray], qmode: str) -> List[List[Tuple[int,int,torch.Tensor]]]:
    exp_ptr = pack["exp_ptr"].astype(np.int32)
    blk_i0 = pack["blk_i0"].astype(np.int32)
    blk_j0 = pack["blk_j0"].astype(np.int32)
    blk_h = pack["blk_h"].astype(np.int32)
    blk_w = pack["blk_w"].astype(np.int32)
    blk_ptr = pack["blk_ptr"].astype(np.int64)

    if qmode == "int8":
        blk_q = pack["blk_q"].astype(np.int8)
        blk_scale = pack["blk_scale"].astype(np.float16)
    else:
        blk_val = pack["blk_val"].astype(np.float16)

    E = exp_ptr.shape[0] - 1
    out: List[List[Tuple[int,int,torch.Tensor]]] = []
    for e in range(E):
        b0 = int(exp_ptr[e])
        b1 = int(exp_ptr[e+1])
        lst = []
        for bi in range(b0, b1):
            i0 = int(blk_i0[bi]); j0 = int(blk_j0[bi])
            h = int(blk_h[bi]); w = int(blk_w[bi])
            v0 = int(blk_ptr[bi]); v1 = int(blk_ptr[bi+1])
            if qmode == "int8":
                q = blk_q[v0:v1].astype(np.int8, copy=False).astype(np.float32)
                sc = float(blk_scale[bi])
                B = torch.from_numpy((q * sc).reshape(h, w)).to(DEVICE, dtype=DTYPE_ACC)
            else:
                v = blk_val[v0:v1].astype(np.float16, copy=False).astype(np.float32)
                B = torch.from_numpy(v.reshape(h, w)).to(DEVICE, dtype=DTYPE_ACC)
            lst.append((i0, j0, B))
        out.append(lst)
    return out

# ----------------------------
# Evaluation
# ----------------------------
@torch.no_grad()
def frob(A: torch.Tensor) -> torch.Tensor:
    return torch.linalg.norm(A, ord="fro")

@torch.no_grad()
def dense_apply(Ws: torch.Tensor, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
    n = Ws.shape[-1]
    Wsum = torch.zeros(n, n, dtype=DTYPE_ACC, device=x.device)
    for a, pos in zip(gates, routed):
        Wsum += float(a.item()) * Ws[int(pos)]
    return x @ Wsum

@torch.no_grad()
def eval_payload(payload: Payload, Ws: torch.Tensor):
    E, n, _ = Ws.shape

    # per-expert matrix rel-error
    errs = []
    for pos in range(E):
        # reconstruct W_hat by applying to basis (use identity batch)
        # More efficient: test on random x and infer operator error; keep consistent with your logs:
        x = torch.randn(8, n, dtype=DTYPE_ACC, device=DEVICE)
        y_hat = payload.apply_expert(x, pos)
        y_ref = x @ Ws[pos]
        err = float((frob(y_hat - y_ref) / (frob(y_ref) + 1e-12)).item())
        errs.append(err)
    m = float(np.mean(errs)); p95 = float(np.percentile(errs, 95)); mx = float(np.max(errs))
    log(f"[eval] per-expert rel-error  mean={m:.6f}  p95={p95:.6f}  max={mx:.6f}")

    # routed mixture rel-error
    mix_errs = []
    for _ in range(cfg.EVAL_TRIALS):
        x = torch.randn(cfg.EVAL_BATCH, n, dtype=DTYPE_ACC, device=DEVICE)
        routed = random.sample(range(E), k=min(cfg.ROUTED_K, E))
        gates = torch.rand(len(routed), dtype=DTYPE_ACC, device=DEVICE)
        gates = gates / gates.sum().clamp_min(1e-12)
        y_hat = payload.apply_mixture(x, routed, gates)
        y_ref = dense_apply(Ws, x, routed, gates)
        mix_errs.append(float((frob(y_hat - y_ref) / (frob(y_ref) + 1e-12)).item()))
    log(f"[eval] routed rel-error = {float(np.mean(mix_errs)):.6f} ± {float(np.std(mix_errs)):.6f}  (trials={cfg.EVAL_TRIALS})")

# ----------------------------
# Main pipeline
# ----------------------------
def banner():
    log(f"== DeepSeek KT++-X OFFLINE v10.2 ==")
    log(f"Time:        {now()}")
    log(f"MODEL_DIR:   {cfg.MODEL_DIR}")
    log(f"OUTPUT_DIR:  {cfg.OUTPUT_DIR}")
    log(f"LAYER:       {cfg.LAYER}")
    log(f"MAX_EXPERTS:  {cfg.MAX_EXPERTS}")
    log(f"CALIB_PATH:  {cfg.CALIB_PATH or '(none)'}  CALIB_SAMPLES(cap)={cfg.CALIB_SAMPLES}")
    log(f"ROUTER_PATH: {cfg.ROUTER_PATH or '(none)'}  RIDGE_WEIGHTED={cfg.RIDGE_WEIGHTED}")
    log(f"LIN_MODE:    {cfg.LIN_MODE}  RIDGE_DAMP={cfg.RIDGE_DAMP}")
    log(f"BASIS_MODE:  {cfg.BASIS_MODE}")
    log(f"CLUSTER:     M0={(cfg.M0 if cfg.M0>0 else '(auto)')} M_MAX={cfg.M_MAX} min_size={cfg.CLUSTER_MIN_SIZE} max_size={cfg.CLUSTER_MAX_SIZE} iters={cfg.CLUSTER_ITERS} restarts={cfg.CLUSTER_RESTARTS}")
    log(f"TRAIN:       steps={cfg.TRAIN_STEPS} warmup={cfg.TRAIN_WARMUP} lr={cfg.TRAIN_LR} subm={cfg.SUBM} batchE={cfg.BATCH_E}")
    log(f"CORE:        {cfg.CORE_MODE} block={cfg.CORE_BLOCK} target={cfg.CORE_TARGET} max_blocks={cfg.CORE_MAX_BLOCKS}")
    log(f"RESIDUAL:    rank={cfg.RES_RANK} blocks={cfg.RES_BLOCKS} bsize={cfg.RES_BSIZE}")
    log(f"REFINE:      enable={cfg.REFINE_ENABLE} err_target={cfg.REFINE_ERR_TARGET} max_extra={cfg.REFINE_MAX_EXTRA}")
    log(f"QUANT:       QMODE={cfg.QMODE}  BASIS_STORE_DTYPE={cfg.BASIS_STORE_DTYPE}")
    log(f"DEVICE:      {DEVICE}  Torch={torch.__version__} threads={NTHREADS}")
    log("")

def main():
    banner()

    # Build Ws (ridge)
    expert_ids, Ws = load_or_build_Ws()
    E, n, _ = Ws.shape
    log(f"[Ws] shape={tuple(Ws.shape)}")

    payload = Payload()
    payload.expert_ids = expert_ids

    # Determine clusters + basis
    if cfg.BASIS_MODE == "hadamard_perm":
        if not is_power_of_two(n):
            raise RuntimeError(f"hadamard_perm requires H power-of-two, got H={n}")
        # One cluster
        payload.clusters = [list(range(E))]
        payload.cluster_of_pos = [0 for _ in range(E)]
        payload.U = [None]
        payload.V = [None]
        # Create perms/signs
        pu, ipu, su = make_perm_sign(n, SEED + 111)
        pv, ipv, sv = make_perm_sign(n, SEED + 222)
        payload.pu, payload.ipu, payload.su = pu, ipu, su
        payload.pv, payload.ipv, payload.sv = pv, ipv, sv
        log("[basis] hadamard_perm: using 1 cluster with implicit basis.")
    else:
        # Dense train: cluster experts by Ws features
        Xfeat = random_proj_features(Ws, d=cfg.CLUSTER_FEAT_D)
        if cfg.M0 > 0:
            M0 = min(cfg.M0, E)
        else:
            M0 = int(round(2.0 * math.sqrt(E)))
            M0 = max(4, min(M0, E))
        M0 = max(2, min(M0, E))
        labels, _ = kmeans_torch(Xfeat, k=M0, iters=cfg.CLUSTER_ITERS, restarts=cfg.CLUSTER_RESTARTS)
        labels = merge_small_clusters(Xfeat, labels, min_size=cfg.CLUSTER_MIN_SIZE)
        labels = hierarchical_split(Xfeat, labels, max_size=cfg.CLUSTER_MAX_SIZE, max_k=min(cfg.M_MAX, E), split_iters=cfg.SPLIT_ITERS)
        labels = merge_small_clusters(Xfeat, labels, min_size=cfg.CLUSTER_MIN_SIZE)
        M = int(labels.max().item()) + 1
        clusters = [torch.nonzero(labels == m, as_tuple=False).flatten().tolist() for m in range(M)]
        clusters = [c for c in clusters if len(c) > 0]
        M = len(clusters)
        log(f"[cluster] M={M} sizes={[len(c) for c in clusters]}")
        payload.clusters = clusters
        payload.cluster_of_pos = [0] * E
        for m, idx in enumerate(clusters):
            for pos in idx:
                payload.cluster_of_pos[int(pos)] = m

        # Init U/V per cluster
        U_par: List[OrthoParam] = []
        V_par: List[OrthoParam] = []
        for _ in range(M):
            U_par.append(OrthoParam(n, init=cfg.INIT_BASIS))
            V_par.append(OrthoParam(n, init=cfg.INIT_BASIS))

        # Train orthogonal bases
        guidance_masks: Dict[int, torch.Tensor] = {}
        guidance_stats: Dict[int, Tuple[float, int]] = {}

        if cfg.TRAIN_STEPS > 0:
            params = [p.M for p in U_par] + [p.M for p in V_par]
            opt = torch.optim.Adam(params, lr=cfg.TRAIN_LR)
            t0 = time.perf_counter()

            with torch.enable_grad():
                for step in range(1, cfg.TRAIN_STEPS + 1):
                    S = torch.randperm(n, device=DEVICE)[:min(cfg.SUBM, n)]

                    # Guidance update
                    if cfg.TRAIN_LAM_GUIDE > 0 and (step % max(1, cfg.TRAIN_GUIDE_EVERY) == 0 or step == 1):
                        with torch.no_grad():
                            guidance_masks.clear()
                            guidance_stats.clear()
                            for m, idx in enumerate(clusters):
                                if len(idx) < cfg.TRAIN_MIN_CLUSTER:
                                    continue
                                Uo = U_par[m].orthogonal()
                                Vo = V_par[m].orthogonal()
                                if 0 < cfg.BATCH_E < len(idx):
                                    pick = torch.randperm(len(idx), device=DEVICE)[:cfg.BATCH_E].tolist()
                                    idx_step = [idx[p] for p in pick]
                                else:
                                    idx_step = idx
                                Xs_ng = slice_X_batch(Ws[idx_step], Uo, Vo, S).detach()
                                mask, ef, kblk = make_guidance_mask_from_Xs(
                                    Xs_ng.to(DTYPE_ACC),
                                    block=cfg.CORE_BLOCK,
                                    target=cfg.TRAIN_GUIDE_TARGET,
                                    max_blocks=cfg.TRAIN_GUIDE_MAX_BLOCKS,
                                )
                                guidance_masks[m] = mask
                                guidance_stats[m] = (ef, kblk)

                    lam_ramp = schedule(step, cfg.TRAIN_WARMUP, cfg.TRAIN_STEPS)
                    lam_block = cfg.TRAIN_LAM_BLOCK * lam_ramp
                    lam_guide = cfg.TRAIN_LAM_GUIDE * lam_ramp

                    L_total = None
                    n_terms = 0

                    for m, idx in enumerate(clusters):
                        if len(idx) < cfg.TRAIN_MIN_CLUSTER:
                            continue

                        Uo = U_par[m].orthogonal()
                        Vo = V_par[m].orthogonal()

                        if 0 < cfg.BATCH_E < len(idx):
                            pick = torch.randperm(len(idx), device=DEVICE)[:cfg.BATCH_E].tolist()
                            idx_step = [idx[p] for p in pick]
                        else:
                            idx_step = idx

                        Xs = slice_X_batch(Ws[idx_step], Uo, Vo, S).to(DTYPE_ACC)

                        off = offdiag_abs_mean(Xs)
                        diag = diag_abs_mean(Xs).clamp_min(1e-6)

                        if cfg.TRAIN_OBJ == "ratio":
                            base = off / diag
                        else:
                            base = torch.log(off + 1e-6) - torch.log(diag)

                        if lam_block > 0 and cfg.CORE_MODE.startswith("block"):
                            base = base + lam_block * block_group_sparsity_penalty(Xs, cfg.CORE_BLOCK)

                        if lam_guide > 0 and (m in guidance_masks):
                            Mmask = guidance_masks[m]
                            Etot = (Xs * Xs).mean().clamp_min(1e-12)
                            Eout = ((Xs * (1.0 - Mmask)) ** 2).mean()
                            base = base + lam_guide * (Eout / Etot)

                        L_total = base if (L_total is None) else (L_total + base)
                        n_terms += 1

                    if L_total is None:
                        log("[train] skipped (no clusters >= TRAIN_MIN_CLUSTER)")
                        break

                    L_total = L_total / max(1, n_terms)
                    opt.zero_grad(set_to_none=True)
                    L_total.backward()
                    if cfg.GRAD_CLIP > 0:
                        torch.nn.utils.clip_grad_norm_(params, max_norm=cfg.GRAD_CLIP)
                    opt.step()

                    if (step % cfg.REORTHO_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                        with torch.no_grad():
                            for p in U_par:
                                p.M.copy_(p.orthogonal())
                            for p in V_par:
                                p.M.copy_(p.orthogonal())

                    if step == 1 or (step % cfg.REPORT_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                        t1 = time.perf_counter()
                        if len(guidance_stats) > 0:
                            ef_mean = float(np.mean([v[0] for v in guidance_stats.values()]))
                            kb_mean = float(np.mean([v[1] for v in guidance_stats.values()]))
                            log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} loss={float(L_total.item()):.4f} "
                                f"lam_block={lam_block:.3f} lam_guide={lam_guide:.3f} "
                                f"guide_energy≈{ef_mean:.3f} guide_blocks≈{kb_mean:.1f} (+{t1-t0:.1f}s)")
                        else:
                            log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} loss={float(L_total.item()):.4f} "
                                f"lam_block={lam_block:.3f} lam_guide={lam_guide:.3f} (+{t1-t0:.1f}s)")
                        t0 = t1

        # Freeze basis
        payload.U = []
        payload.V = []
        for m in range(len(clusters)):
            payload.U.append(U_par[m].orthogonal().detach().contiguous())
            payload.V.append(V_par[m].orthogonal().detach().contiguous())
        payload.DL = [None] * len(clusters)
        payload.DR = [None] * len(clusters)

    # Build payload blocks / residuals
    payload.core_blocks = [[] for _ in range(E)]
    payload.res_blocks = [[] for _ in range(E)]
    payload.gam = [None for _ in range(E)]
    if cfg.BASIS_MODE == "hadamard_perm":
        # Build X in basis using implicit hadamard, one cluster only
        pu, ipu, su = payload.pu, payload.ipu, payload.su
        pv, ipv, sv = payload.pv, payload.ipv, payload.sv
        assert pu is not None and ipu is not None and su is not None and pv is not None and ipv is not None and sv is not None

        X_list = [mat_to_basis_hadamard(Ws[pos], pu, ipu, su, pv, ipv, sv) for pos in range(E)]
        core = choose_core_blocks(X_list)

        # Core per expert
        if core["blocks_per_expert"] is not None:
            for pos, blocks in enumerate(core["blocks_per_expert"]):
                payload.core_blocks[pos] = pick_top_blocks_values(X_list[pos], blocks)
        else:
            shared = core["blocks_shared"]
            for pos in range(E):
                payload.core_blocks[pos] = pick_top_blocks_values(X_list[pos], shared)

        # Residuals and shared low-rank over all experts
        R_list = []
        for pos in range(E):
            X = X_list[pos]
            Xc = torch.zeros_like(X)
            for (i0, j0, Bc) in payload.core_blocks[pos]:
                h, w = Bc.shape
                Xc[i0:i0+h, j0:j0+w] = Bc
            R_list.append((X - Xc).contiguous())

        Rmean = torch.stack(R_list, dim=0).mean(dim=0)
        r = min(cfg.RES_RANK, n)
        DL, DR = rand_svd(Rmean, r=r, n_iter=2)
        payload.DL = [DL.contiguous()]
        payload.DR = [DR.contiguous()]

        # gam + residual blocks (+ refine)
        for pos in range(E):
            Rm = R_list[pos]
            g = torch.sum(DL * (Rm @ DR), dim=0).contiguous()
            payload.gam[pos] = g
            R2 = (Rm - (DL * g.view(1, -1)) @ DR.t()).contiguous()

            b = int(cfg.RES_BSIZE)
            Eg = block_energy_grid(R2, b).to(DTYPE_ACC)
            flat = Eg.reshape(-1)
            order = torch.argsort(flat, descending=True)[:min(cfg.RES_BLOCKS, flat.numel())].tolist()
            nb = Eg.shape[0]
            blocks = []
            for idv in order:
                bi = idv // nb
                bj = idv % nb
                i0 = bi * b
                j0 = bj * b
                hh = min(b, n - i0)
                ww = min(b, n - j0)
                if hh > 0 and ww > 0:
                    blocks.append((i0, j0, R2[i0:i0+hh, j0:j0+ww].clone()))
            payload.res_blocks[pos] = blocks

        log("[build] payloads ...")
        efr = core["energy_fracs"]
        efm = float(np.mean(efr)) if len(efr) else 0.0
        efmin = float(np.min(efr)) if len(efr) else 0.0
        nbm = float(np.mean([len(payload.core_blocks[p]) for p in range(E)]))
        log(f"  - cluster0: E={E} core_blocks(mean)≈{nbm:.1f} core_energy(mean/min)≈{efm:.3f}/{efmin:.3f} r={r}")

    else:
        log("[build] payloads ...")
        M = len(payload.clusters)
        payload.DL = [None] * M
        payload.DR = [None] * M
        # Build per cluster, fill per expert
        for m, idx in enumerate(payload.clusters):
            U = payload.U[m]; V = payload.V[m]
            assert U is not None and V is not None

            core_blocks_per, DL, DR, gam_list, res_blocks_per = build_payload_for_cluster_dense(Ws, idx, U, V)
            payload.DL[m] = DL
            payload.DR[m] = DR

            # optional refine per expert (can cost time)
            if cfg.REFINE_ENABLE:
                for j, pos in enumerate(idx):
                    res_blocks_per[j] = refine_expert_blocks_dense(
                        Ws, pos, U, V,
                        core_blocks_per[j],
                        DL, DR, gam_list[j],
                        res_blocks_per[j],
                    )

            # write to payload
            for j, pos in enumerate(idx):
                payload.core_blocks[pos] = core_blocks_per[j]
                payload.res_blocks[pos] = res_blocks_per[j]
                payload.gam[pos] = gam_list[j].contiguous()

            # log cluster
            efr = choose_core_blocks([(U.t() @ Ws[p] @ V).contiguous() for p in idx])["energy_fracs"]
            nbm = float(np.mean([len(payload.core_blocks[p]) for p in idx])) if idx else 0.0
            efm = float(np.mean(efr)) if len(efr) else 0.0
            efmin = float(np.min(efr)) if len(efr) else 0.0
            log(f"  - cluster{m}: E={len(idx)} core_blocks(mean)≈{nbm:.1f} core_energy(mean/min)≈{efm:.3f}/{efmin:.3f} r={DL.shape[1]}")

    # Build runtime (Payload already has tensors). Evaluate
    eval_payload(payload, Ws)

    # Serialize payload
    out_payload = os.path.join(
        cfg.OUTPUT_DIR,
        f"ktx_payload_layer{cfg.LAYER}_E{E}_{cfg.BASIS_MODE}_q{cfg.QMODE}_v10_2.npz"
    )

    # Basis store dtype
    store_dtype = np.float16 if cfg.BASIS_STORE_DTYPE == "float16" else np.float32

    meta = ws_meta(expert_ids)
    meta.update({
        "script": "ktxx_offline_v10_2",
        "basis_mode": cfg.BASIS_MODE,
        "qmode": cfg.QMODE,
        "core_mode": cfg.CORE_MODE,
        "core_block": cfg.CORE_BLOCK,
        "core_target": cfg.CORE_TARGET,
        "core_max_blocks": cfg.CORE_MAX_BLOCKS,
        "res_rank": cfg.RES_RANK,
        "res_blocks": cfg.RES_BLOCKS,
        "res_bsize": cfg.RES_BSIZE,
        "refine_enable": cfg.REFINE_ENABLE,
        "refine_err_target": cfg.REFINE_ERR_TARGET,
        "refine_max_extra": cfg.REFINE_MAX_EXTRA,
    })

    # Pack ragged blocks
    core_pack = _pack_blocks_ragged(payload.core_blocks, qmode=cfg.QMODE)
    res_pack = _pack_blocks_ragged(payload.res_blocks, qmode=cfg.QMODE)

    # Pack cluster structure
    cluster_of = np.array(payload.cluster_of_pos, dtype=np.int16)
    # cluster list as offsets
    cl_off = [0]
    cl_cat = []
    for c in payload.clusters:
        cl_cat.extend(c)
        cl_off.append(len(cl_cat))
    cl_off = np.array(cl_off, dtype=np.int32)
    cl_cat = np.array(cl_cat, dtype=np.int16)

    arrays: Dict[str, Any] = {}
    arrays["meta"] = _encode_meta(meta)
    arrays["expert_ids"] = np.array(payload.expert_ids, dtype=np.int32)
    arrays["cluster_of_pos"] = cluster_of
    arrays["cluster_offsets"] = cl_off
    arrays["cluster_concat"] = cl_cat

    # Basis
    if cfg.BASIS_MODE == "hadamard_perm":
        arrays["pu"] = payload.pu.numpy().astype(np.int32)  # type: ignore[union-attr]
        arrays["ipu"] = payload.ipu.numpy().astype(np.int32)  # type: ignore[union-attr]
        arrays["su"] = payload.su.numpy().astype(store_dtype)  # type: ignore[union-attr]
        arrays["pv"] = payload.pv.numpy().astype(np.int32)  # type: ignore[union-attr]
        arrays["ipv"] = payload.ipv.numpy().astype(np.int32)  # type: ignore[union-attr]
        arrays["sv"] = payload.sv.numpy().astype(store_dtype)  # type: ignore[union-attr]
    else:
        for m in range(len(payload.clusters)):
            U = payload.U[m]; V = payload.V[m]
            DL = payload.DL[m]; DR = payload.DR[m]
            assert U is not None and V is not None and DL is not None and DR is not None
            arrays[f"U_{m}"] = U.detach().cpu().numpy().astype(store_dtype)
            arrays[f"V_{m}"] = V.detach().cpu().numpy().astype(store_dtype)
            arrays[f"DL_{m}"] = DL.detach().cpu().numpy().astype(store_dtype)
            arrays[f"DR_{m}"] = DR.detach().cpu().numpy().astype(store_dtype)

    # gamma per expert (pad to RES_RANK with zeros for uniform shape)
    r_used = min(cfg.RES_RANK, n)
    G = np.zeros((E, r_used), dtype=store_dtype)
    for pos in range(E):
        g = payload.gam[pos]
        if g is None:
            continue
        gg = g.detach().cpu().numpy().astype(np.float32, copy=False)
        G[pos, :min(r_used, gg.shape[0])] = gg[:min(r_used, gg.shape[0])].astype(store_dtype)
    arrays["gam"] = G

    # blocks packs
    for k, v in core_pack.items():
        arrays["core_" + k] = v
    for k, v in res_pack.items():
        arrays["res_" + k] = v

    save_npz(out_payload, **arrays)

    log(f"[save] payload -> {out_payload}  size={os.path.getsize(out_payload)/1e6:.2f} MB")
    log("✅ Done.")

if __name__ == "__main__":
    main()


== DeepSeek KT++-X OFFLINE v10.2 ==
Time:        2026-01-13 11:34:12
MODEL_DIR:   /home/daniyar/deepseek-model
OUTPUT_DIR:  /home/daniyar/moe_ws_outputs
LAYER:       1
MAX_EXPERTS:  16
CALIB_PATH:  (none)  CALIB_SAMPLES(cap)=4096
ROUTER_PATH: (none)  RIDGE_WEIGHTED=False
LIN_MODE:    ridge  RIDGE_DAMP=0.001
BASIS_MODE:  dense_train
CLUSTER:     M0=(auto) M_MAX=16 min_size=2 max_size=4 iters=60 restarts=4
TRAIN:       steps=24 warmup=6 lr=0.05 subm=256 batchE=4
CORE:        blocktopk_perexpert block=64 target=0.85 max_blocks=256
RESIDUAL:    rank=512 blocks=192 bsize=64
REFINE:      enable=True err_target=0.05 max_extra=256
QUANT:       QMODE=none  BASIS_STORE_DTYPE=float16
DEVICE:      cpu  Torch=2.4.1+cpu threads=8

[found] layer=1 total_experts=64 using=16 eids=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
[load] reading tensors from shards ...
[shape] H=2048 d_ff=1408
[calib] CALIB_PATH not set -> auto-found /home/daniyar/moe_ws_outputs/calib_layer1_X.npz
[calib] X loaded: 

Build Ws (ridge):   0%|          | 0/16 [00:00<?, ?it/s]

[cache] wrote Ws -> /home/daniyar/moe_ws_outputs/Ws_cache_layer1_E16_ridge_v10_2.npz  size=250.24 MB
[Ws] shape=(16, 2048, 2048)
[cluster] M=6 sizes=[4, 2, 2, 2, 2, 4]


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [17]:
#!/usr/bin/env python3
# ============================================================
# DeepSeek KT++-X OFFLINE v10.3 (single-file runner)
#
# Fixes vs v10.2:
#  - FIXED: slice_X_batch MUST be differentiable (removed @torch.no_grad)
#  - FIXED: FWHT aliasing bug (clone a/b)
#  - FIXED: hadamard basis ordering consistent with v9 (perm->sign->H)
#  - FIXED: if NORMALIZE_W=1 we store per-expert scale and apply it in runtime/eval
#
# Pipeline:
#   1) (Optional) capture calib X and router P (local transformers, local_files_only)
#   2) build Ws via ridge
#   3) cluster (dense_train), train orthogonal bases U/V
#   4) build payload: core blocks + low-rank + residual blocks (+ optional refine)
#   5) eval, save payload npz (ragged storage, no pickle)
# ============================================================

import os, re, json, math, time, random, sys
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors import safe_open

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):  # type: ignore
        return x

# ----------------------------
# Env helpers / determinism
# ----------------------------
def _int_env(k: str, d: int) -> int:
    try:
        return int(os.environ.get(k, str(d)))
    except Exception:
        return d

def _float_env(k: str, d: float) -> float:
    try:
        return float(os.environ.get(k, str(d)))
    except Exception:
        return d

def _str_env(k: str, d: str) -> str:
    return os.environ.get(k, d)

def _bool_env(k: str, d: bool) -> bool:
    v = os.environ.get(k, None)
    if v is None:
        return d
    return v.strip().lower() in ("1", "true", "yes", "y", "on")

NTHREADS = _int_env("KTXX_THREADS", 8)
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(NTHREADS))
try:
    torch.set_num_threads(NTHREADS)
except Exception:
    pass

SEED = _int_env("SEED", 1234)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(_str_env("DEVICE", "cpu"))
DTYPE_ACC = torch.float32

def now() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")

def log(msg: str):
    print(msg, flush=True)

# ----------------------------
# Config
# ----------------------------
@dataclass
class Cfg:
    MODEL_DIR: str = _str_env("MODEL_DIR", "/home/daniyar/deepseek-model")
    OUTPUT_DIR: str = _str_env("OUTPUT_DIR", "/home/daniyar/moe_ws_outputs")

    LAYER: int = _int_env("LAYER", 1)
    MAX_EXPERTS: int = _int_env("MAX_EXPERTS", 16)

    CALIB_PATH: str = _str_env("CALIB_PATH", "").strip()
    CALIB_SAMPLES: int = _int_env("CALIB_SAMPLES", 4096)

    ROUTER_PATH: str = _str_env("ROUTER_PATH", "").strip()
    RIDGE_WEIGHTED: bool = _bool_env("RIDGE_WEIGHTED", False)
    ROUTER_EIDS_ARE_GLOBAL: bool = _bool_env("ROUTER_EIDS_ARE_GLOBAL", True)

    # Optional capture
    CAPTURE_ENABLE: bool = _bool_env("CAPTURE_ENABLE", False)
    CAPTURE_FORCE: bool = _bool_env("CAPTURE_FORCE", False)
    CAPTURE_MAX_TOKENS: int = _int_env("CAPTURE_MAX_TOKENS", 512)
    CAPTURE_ITERS: int = _int_env("CAPTURE_ITERS", 32)
    CAPTURE_BATCH: int = _int_env("CAPTURE_BATCH", 1)
    CAPTURE_KEEP_PAD: bool = _bool_env("CAPTURE_KEEP_PAD", False)
    HF_TRUST_REMOTE_CODE: bool = _bool_env("HF_TRUST_REMOTE_CODE", True)
    HF_LOCAL_FILES_ONLY: bool = _bool_env("HF_LOCAL_FILES_ONLY", True)
    HF_AUTO_PIP: bool = _bool_env("HF_AUTO_PIP", False)
    CAPTURE_TEXT: str = _str_env(
        "CAPTURE_TEXT",
        ("DeepSeek models use mixture-of-experts layers. "
         "We capture intermediate activations for calibration. "
         "Use multiple diverse prompts in real runs.\n" * 128)
    )

    # Ws build
    LIN_MODE: str = _str_env("LIN_MODE", "ridge").strip().lower()  # ridge only here
    RIDGE_DAMP: float = _float_env("RIDGE_DAMP", 1e-3)
    NORMALIZE_W: bool = _bool_env("NORMALIZE_W", True)

    # Basis mode
    BASIS_MODE: str = _str_env("BASIS_MODE", "dense_train").strip().lower()  # dense_train|hadamard_perm
    BASIS_STORE_DTYPE: str = _str_env("BASIS_STORE_DTYPE", "float16").strip().lower()  # float16|float32
    INIT_BASIS: str = _str_env("INIT_BASIS", "identity").strip().lower()  # identity|random
    HAD_SEED: int = _int_env("HAD_SEED", 1234)

    # Clustering (dense_train)
    M0: int = _int_env("M0", 0)
    M_MAX: int = _int_env("M_MAX", 16)
    CLUSTER_FEAT_D: int = _int_env("CLUSTER_FEAT_D", 64)
    CLUSTER_ITERS: int = _int_env("CLUSTER_ITERS", 60)
    CLUSTER_RESTARTS: int = _int_env("CLUSTER_RESTARTS", 4)
    CLUSTER_MIN_SIZE: int = _int_env("CLUSTER_MIN_SIZE", 2)
    CLUSTER_MAX_SIZE: int = _int_env("CLUSTER_MAX_SIZE", 4)
    SPLIT_ITERS: int = _int_env("SPLIT_ITERS", 50)

    # Training (dense_train)
    TRAIN_STEPS: int = _int_env("TRAIN_STEPS", 24)
    TRAIN_WARMUP: int = _int_env("TRAIN_WARMUP", 6)
    TRAIN_LR: float = _float_env("TRAIN_LR", 5e-2)
    SUBM: int = _int_env("SUBM", 256)
    BATCH_E: int = _int_env("BATCH_E", 4)
    TRAIN_MIN_CLUSTER: int = _int_env("TRAIN_MIN_CLUSTER", 2)
    REORTHO_EVERY: int = _int_env("REORTHO_EVERY", 4)
    REPORT_EVERY: int = _int_env("REPORT_EVERY", 4)
    GRAD_CLIP: float = _float_env("GRAD_CLIP", 1.0)

    TRAIN_OBJ: str = _str_env("TRAIN_OBJ", "logratio").strip().lower()  # logratio|ratio
    TRAIN_LAM_BLOCK: float = _float_env("TRAIN_LAM_BLOCK", 0.10)
    TRAIN_LAM_GUIDE: float = _float_env("TRAIN_LAM_GUIDE", 1.0)
    TRAIN_GUIDE_EVERY: int = _int_env("TRAIN_GUIDE_EVERY", 2)
    TRAIN_GUIDE_TARGET: float = _float_env("TRAIN_GUIDE_TARGET", 0.75)
    TRAIN_GUIDE_MAX_BLOCKS: int = _int_env("TRAIN_GUIDE_MAX_BLOCKS", 256)

    # Core selection
    CORE_MODE: str = _str_env("CORE_MODE", "blocktopk_perexpert").strip().lower()
    CORE_AGG: str = _str_env("CORE_AGG", "mean").strip().lower()
    CORE_BLOCK: int = _int_env("CORE_BLOCK", 64)
    CORE_TARGET: float = _float_env("CORE_TARGET", 0.85)
    CORE_MAX_BLOCKS: int = _int_env("CORE_MAX_BLOCKS", 256)

    # Residual
    RES_RANK: int = _int_env("RES_RANK", 512)
    RES_BLOCKS: int = _int_env("RES_BLOCKS", 192)
    RES_BSIZE: int = _int_env("RES_BSIZE", 64)

    # Refine
    REFINE_ENABLE: bool = _bool_env("REFINE_ENABLE", True)
    REFINE_ERR_TARGET: float = _float_env("REFINE_ERR_TARGET", 0.05)
    REFINE_MAX_EXTRA: int = _int_env("REFINE_MAX_EXTRA", 256)
    REFINE_BSIZE: int = _int_env("REFINE_BSIZE", 64)

    # Quant for blocks
    QMODE: str = _str_env("QMODE", "none").strip().lower()  # none|float16|int8

    # Eval
    EVAL_TRIALS: int = _int_env("EVAL_TRIALS", 8)
    EVAL_BATCH: int = _int_env("EVAL_BATCH", 2)
    ROUTED_K: int = _int_env("ROUTED_K", 8)

cfg = Cfg()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# ----------------------------
# NPZ helpers
# ----------------------------
def save_npz(path: str, **arrays: Any):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez_compressed(path, **arrays)

def load_npz(path: str) -> Dict[str, np.ndarray]:
    z = np.load(path, allow_pickle=False)
    out = {k: z[k] for k in z.files}
    z.close()
    return out

def _encode_meta(meta: dict) -> np.ndarray:
    b = json.dumps(meta, sort_keys=True).encode("utf-8")
    return np.frombuffer(b, dtype=np.uint8)

def _decode_meta(arr: np.ndarray) -> dict:
    try:
        b = bytes(arr.tolist())
        return json.loads(b.decode("utf-8"))
    except Exception:
        return {}

# ----------------------------
# Offline shard loading
# ----------------------------
def read_index(model_dir: str) -> Dict[str, str]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    wm = obj.get("weight_map", {})
    if not wm:
        raise RuntimeError("Index JSON has empty weight_map.")
    return wm

def find_layer_expert_ids(weight_map: Dict[str, str], layer: int) -> List[int]:
    pat = re.compile(rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.")
    ids = set()
    for k in weight_map.keys():
        m = pat.match(k)
        if m:
            ids.add(int(m.group(1)))
    return sorted(ids)

def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    prefix = f"model.layers.{layer}.mlp.experts.{eid}."
    def pick(cands: List[str]) -> Optional[str]:
        for suf in cands:
            k = prefix + suf
            if k in weight_map:
                return k
        return None
    up   = pick(["up_proj.weight", "w3.weight", "w1.weight"])
    gate = pick(["gate_proj.weight", "w1.weight", "w3.weight"])
    down = pick(["down_proj.weight", "w2.weight"])
    if up is None or gate is None or down is None:
        return {}
    if up == gate:
        g2 = pick(["gate_proj.weight"])
        if g2:
            gate = g2
    return {"up": up, "gate": gate, "down": down}

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], keys: List[str]) -> Dict[str, torch.Tensor]:
    by_shard: Dict[str, List[str]] = {}
    for k in keys:
        shard = weight_map.get(k, None)
        if shard is None:
            raise KeyError(f"Key not in weight_map: {k}")
        by_shard.setdefault(shard, []).append(k)

    out: Dict[str, torch.Tensor] = {}
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(sp):
            raise FileNotFoundError(f"Missing shard: {sp}")
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks:
                out[k] = f.get_tensor(k)
    return out

# ----------------------------
# Correct FWHT (no aliasing)
# ----------------------------
def is_power_of_two(n: int) -> bool:
    return (n > 0) and ((n & (n - 1)) == 0)

def fwht(x: torch.Tensor) -> torch.Tensor:
    """
    Walsh-Hadamard transform on last dimension.
    Correct implementation: avoid in-place aliasing by cloning sub-blocks.
    """
    n = x.shape[-1]
    if not is_power_of_two(n):
        raise ValueError(f"FWHT requires power-of-two, got {n}")
    orig = x.shape
    y = x.reshape(-1, n).contiguous()
    h = 1
    while h < n:
        y = y.view(-1, n // (2*h), 2, h)
        a = y[:, :, 0, :].clone()
        b = y[:, :, 1, :].clone()
        y[:, :, 0, :] = a + b
        y[:, :, 1, :] = a - b
        y = y.view(-1, n)
        h *= 2
    return y.view(orig)

def fwht_ortho(x: torch.Tensor) -> torch.Tensor:
    return fwht(x) / math.sqrt(x.shape[-1])

# ----------------------------
# Hadamard-perm basis (CONSISTENT with v9)
# U = P D H, applied on RIGHT:
#   x @ U = fwht_ortho( (x[:,perm] * sign) )
# U^T:
#   x @ U^T = (fwht_ortho(x) * sign)[:, inv_perm]
# ----------------------------
@torch.no_grad()
def make_perm_sign(n: int, seed: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    g = np.random.default_rng(seed)
    perm = np.array(g.permutation(n), dtype=np.int64)
    inv = np.empty_like(perm)
    inv[perm] = np.arange(n, dtype=np.int64)
    sign = g.choice([-1.0, 1.0], size=(n,), replace=True).astype(np.float32)
    return (
        torch.from_numpy(perm),
        torch.from_numpy(inv),
        torch.from_numpy(sign),
    )

@torch.no_grad()
def had_right(x: torch.Tensor, perm: torch.Tensor, sign: torch.Tensor) -> torch.Tensor:
    y = x.index_select(1, perm.to(x.device))
    y = y * sign.to(x.device).view(1, -1)
    return fwht_ortho(y)

@torch.no_grad()
def had_right_T(x: torch.Tensor, inv_perm: torch.Tensor, sign: torch.Tensor) -> torch.Tensor:
    y = fwht_ortho(x)
    y = y * sign.to(x.device).view(1, -1)
    return y.index_select(1, inv_perm.to(x.device))

@torch.no_grad()
def mat_to_basis_hadamard(W: torch.Tensor,
                          pu: torch.Tensor, ipu: torch.Tensor, su: torch.Tensor,
                          pv: torch.Tensor, ipv: torch.Tensor, sv: torch.Tensor) -> torch.Tensor:
    # X = U^T W V, U=P_u D_u H, V=P_v D_v H
    # U^T = H D_u P_u^T
    # V   = P_v D_v H
    T = W

    # Right multiply by V = P_v D_v H:
    T = T.index_select(1, pv.to(T.device))                    # columns perm
    T = T * sv.to(T.device).view(1, -1)                       # columns sign
    T = fwht_ortho(T)                                         # H on columns

    # Left multiply by U^T = H D_u P_u^T:
    T = T.index_select(0, ipu.to(T.device))                   # rows perm by P^T
    T = T * su.to(T.device).view(-1, 1)                       # rows sign
    T = fwht_ortho(T.t().contiguous()).t().contiguous()       # H on rows
    return T.contiguous()

@torch.no_grad()
def mat_from_basis_hadamard(X: torch.Tensor,
                            pu: torch.Tensor, ipu: torch.Tensor, su: torch.Tensor,
                            pv: torch.Tensor, ipv: torch.Tensor, sv: torch.Tensor) -> torch.Tensor:
    # W = U X V^T, U=P_u D_u H, V^T = H D_v P_v^T
    T = X

    # Left multiply by U = P_u D_u H: first H on rows, then sign, then perm
    T = fwht_ortho(T.t().contiguous()).t().contiguous()
    T = T * su.to(T.device).view(-1, 1)
    T = T.index_select(0, pu.to(T.device))

    # Right multiply by V^T = H D_v P_v^T: first H on columns, then sign, then perm by P^T
    T = fwht_ortho(T)
    T = T * sv.to(T.device).view(1, -1)
    T = T.index_select(1, ipv.to(T.device))
    return T.contiguous()

# ----------------------------
# Calib/router IO + optional capture
# ----------------------------
def autodetect_calib_path() -> Optional[str]:
    cand = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
    return cand if os.path.isfile(cand) else None

def autodetect_router_path() -> Optional[str]:
    cand = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
    return cand if os.path.isfile(cand) else None

def load_calib_X(path: str, H: int) -> torch.Tensor:
    z = np.load(path, allow_pickle=False)
    if "X" not in z.files:
        raise KeyError(f"CALIB npz missing key 'X'. Keys={list(z.files)}")
    Xn = z["X"].astype(np.float32, copy=False)
    z.close()
    X = torch.from_numpy(Xn)
    if X.ndim != 2 or X.shape[1] != H:
        raise RuntimeError(f"Bad X shape {tuple(X.shape)} expected (*,{H})")
    if X.shape[0] > cfg.CALIB_SAMPLES:
        X = X[:cfg.CALIB_SAMPLES]
    return X.to(DEVICE, dtype=DTYPE_ACC)

def load_router_P(path: str) -> np.ndarray:
    z = np.load(path, allow_pickle=False)
    if "P" not in z.files:
        raise KeyError(f"ROUTER npz missing key 'P'. Keys={list(z.files)}")
    P = z["P"].astype(np.float32, copy=False)
    z.close()
    return P

def _maybe_autopip():
    if not cfg.HF_AUTO_PIP:
        return
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU",
                           "transformers", "tokenizers", "sentencepiece"])

def _patch_dynamic_cache_compat():
    try:
        from transformers.cache_utils import DynamicCache  # type: ignore
        if not hasattr(DynamicCache, "get_usable_length"):
            def _get_usable_length(self, seq_length: int):
                return int(seq_length)
            DynamicCache.get_usable_length = _get_usable_length  # type: ignore
            log("[patch] Added DynamicCache.get_usable_length shim.")
    except Exception:
        pass

class _Collector:
    def __init__(self, H: int, E_total: int, max_rows: int):
        self.H = H
        self.E_total = E_total
        self.max_rows = max_rows
        self.X_chunks: List[torch.Tensor] = []
        self.P_chunks: List[torch.Tensor] = []
        self.nX = 0
        self.nP = 0

    def add_X(self, hs: torch.Tensor, attn_mask: Optional[torch.Tensor]):
        if hs is None:
            return
        if hs.ndim == 2:
            hs = hs.unsqueeze(0)
        if hs.ndim != 3 or hs.shape[-1] != self.H:
            return
        hs = hs.detach().to(torch.float32).cpu()
        if attn_mask is not None and (not cfg.CAPTURE_KEEP_PAD):
            m = attn_mask.detach().cpu().to(torch.bool)
            flat = hs.reshape(-1, self.H)
            flat = flat[m.reshape(-1)]
        else:
            flat = hs.reshape(-1, self.H)
        if flat.numel() == 0:
            return
        need = self.max_rows - self.nX
        if need <= 0:
            return
        if flat.shape[0] > need:
            flat = flat[:need]
        self.X_chunks.append(flat)
        self.nX += int(flat.shape[0])

    def add_P_from_logits(self, logits: torch.Tensor, attn_mask: Optional[torch.Tensor]):
        if logits is None:
            return
        if logits.ndim == 2:
            logits = logits.unsqueeze(0)
        if logits.ndim != 3:
            return
        P = torch.softmax(logits.detach().to(torch.float32), dim=-1)[..., :self.E_total].cpu()
        if attn_mask is not None and (not cfg.CAPTURE_KEEP_PAD):
            m = attn_mask.detach().cpu().to(torch.bool)
            flat = P.reshape(-1, P.shape[-1])
            flat = flat[m.reshape(-1)]
        else:
            flat = P.reshape(-1, P.shape[-1])
        if flat.numel() == 0:
            return
        need = self.max_rows - self.nP
        if need <= 0:
            return
        if flat.shape[0] > need:
            flat = flat[:need]
        self.P_chunks.append(flat)
        self.nP += int(flat.shape[0])

def capture_XP_transformers(model_dir: str, layer_idx: int, H: int, E_total: int,
                            out_x: str, out_p: str) -> Tuple[str, Optional[str]]:
    _maybe_autopip()
    _patch_dynamic_cache_compat()
    try:
        from transformers import AutoTokenizer, AutoModelForCausalLM  # type: ignore
    except Exception as e:
        raise RuntimeError("transformers missing. Install or set HF_AUTO_PIP=1.") from e

    log("[capture] Loading tokenizer/model (local_files_only recommended)...")
    tok = AutoTokenizer.from_pretrained(
        model_dir,
        trust_remote_code=cfg.HF_TRUST_REMOTE_CODE,
        local_files_only=cfg.HF_LOCAL_FILES_ONLY,
        use_fast=True,
    )
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token if tok.eos_token is not None else tok.unk_token

    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        trust_remote_code=cfg.HF_TRUST_REMOTE_CODE,
        local_files_only=cfg.HF_LOCAL_FILES_ONLY,
        torch_dtype=torch.float16,
        device_map=None,
        low_cpu_mem_usage=True,
    )
    model.eval().to(DEVICE)

    # Find layer list
    layers = None
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        layers = list(model.model.layers)
    elif hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        layers = list(model.transformer.h)
    elif hasattr(model, "layers"):
        layers = list(model.layers)
    if layers is None:
        raise RuntimeError("Can't locate model layers list.")
    if not (0 <= layer_idx < len(layers)):
        raise RuntimeError(f"LAYER={layer_idx} out of range (0..{len(layers)-1})")

    layer = layers[layer_idx]
    mlp = getattr(layer, "mlp", None)
    if mlp is None:
        # best-effort search
        for n, m in layer.named_modules():
            if n.lower().endswith("mlp"):
                mlp = m
                break
    if mlp is None:
        raise RuntimeError("Could not find layer.mlp for X capture.")

    # Try to find router logits module
    router_linear: Optional[nn.Linear] = None
    for name, mod in layer.named_modules():
        if isinstance(mod, nn.Linear) and getattr(mod, "in_features", None) == H and getattr(mod, "out_features", 0) >= E_total:
            nm = name.lower()
            if ("router" in nm) or ("gate" in nm) or ("moe" in nm):
                router_linear = mod
                break

    coll = _Collector(H=H, E_total=E_total, max_rows=cfg.CALIB_SAMPLES)
    attn_mask_holder = {"mask": None}

    def mlp_pre_hook(_m, inputs):
        hs = inputs[0]
        coll.add_X(hs, attn_mask_holder["mask"])

    h_mlp = mlp.register_forward_pre_hook(mlp_pre_hook)

    h_router = None
    if router_linear is not None:
        def router_hook(_m, _in, out):
            logits = out[0] if isinstance(out, (tuple, list)) else out
            if torch.is_tensor(logits):
                coll.add_P_from_logits(logits, attn_mask_holder["mask"])
        h_router = router_linear.register_forward_hook(router_hook)
        log(f"[capture] Router Linear found: out_features={router_linear.out_features}")
    else:
        log("[capture] Router not found; capturing X only.")

    for it in range(cfg.CAPTURE_ITERS):
        enc = tok(
            cfg.CAPTURE_TEXT,
            return_tensors="pt",
            truncation=True,
            max_length=cfg.CAPTURE_MAX_TOKENS,
            padding="max_length",
        )
        # batch replicate
        for k in enc:
            if enc[k].ndim == 2 and cfg.CAPTURE_BATCH > 1:
                enc[k] = enc[k].repeat(cfg.CAPTURE_BATCH, 1)

        attn_mask_holder["mask"] = enc.get("attention_mask", None)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        with torch.inference_mode():
            _ = model(**enc, use_cache=False)

        if (it + 1) % 4 == 0 or it == 0 or (it + 1) == cfg.CAPTURE_ITERS:
            log(f"[capture] iter {it+1}/{cfg.CAPTURE_ITERS}  nX={coll.nX} nP={coll.nP}")
        if coll.nX >= cfg.CALIB_SAMPLES and (not cfg.RIDGE_WEIGHTED or coll.nP >= cfg.CALIB_SAMPLES):
            break

    h_mlp.remove()
    if h_router is not None:
        h_router.remove()

    if coll.nX == 0:
        raise RuntimeError("Capture failed: got 0 X rows.")

    X = torch.cat(coll.X_chunks, dim=0)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
    save_npz(out_x, X=X)
    log(f"[capture] wrote X -> {out_x}  shape={tuple(X.shape)}")

    p_written = None
    if coll.nP > 0:
        P = torch.cat(coll.P_chunks, dim=0)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
        N = min(P.shape[0], X.shape[0])
        if N < X.shape[0]:
            X = X[:N]
            save_npz(out_x, X=X)
        P = P[:N]
        save_npz(out_p, P=P)
        log(f"[capture] wrote P -> {out_p}  shape={tuple(P.shape)}")
        p_written = out_p
    else:
        log("[capture] Router P not captured.")

    return out_x, p_written

def ensure_calib_router(H: int, E_total: int):
    if not cfg.CALIB_PATH:
        cand = autodetect_calib_path()
        if cand:
            cfg.CALIB_PATH = cand
            log(f"[calib] CALIB_PATH not set -> auto-found {cfg.CALIB_PATH}")
    if not cfg.ROUTER_PATH:
        cand = autodetect_router_path()
        if cand:
            cfg.ROUTER_PATH = cand
            log(f"[router] ROUTER_PATH not set -> auto-found {cfg.ROUTER_PATH}")

    need_capture = cfg.CAPTURE_FORCE or (cfg.CAPTURE_ENABLE and (not cfg.CALIB_PATH))
    if need_capture:
        out_x = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
        out_p = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
        log("[capture] capturing X (and maybe P) via transformers...")
        x_path, p_path = capture_XP_transformers(cfg.MODEL_DIR, cfg.LAYER, H, E_total, out_x, out_p)
        cfg.CALIB_PATH = x_path
        if p_path:
            cfg.ROUTER_PATH = p_path

# ----------------------------
# Ridge build Ws + cache
# ----------------------------
def ws_cache_path() -> str:
    return os.path.join(cfg.OUTPUT_DIR, f"Ws_cache_layer{cfg.LAYER}_E{cfg.MAX_EXPERTS}_ridge_v10_3.npz")

def ws_meta(eids: List[int]) -> dict:
    return dict(
        time=now(),
        model_dir=cfg.MODEL_DIR,
        output_dir=cfg.OUTPUT_DIR,
        layer=cfg.LAYER,
        expert_ids=eids,
        lin_mode="ridge",
        ridge_damp=cfg.RIDGE_DAMP,
        ridge_weighted=cfg.RIDGE_WEIGHTED,
        calib_path=cfg.CALIB_PATH or "",
        calib_samples=cfg.CALIB_SAMPLES,
        router_path=cfg.ROUTER_PATH or "",
        router_global=cfg.ROUTER_EIDS_ARE_GLOBAL,
        normalize_w=cfg.NORMALIZE_W,
        seed=SEED,
        device=str(DEVICE),
    )

@torch.no_grad()
def forward_mlp(X: torch.Tensor, W_gate: torch.Tensor, W_up: torch.Tensor, W_down: torch.Tensor) -> torch.Tensor:
    Xf = X.to(DTYPE_ACC)
    up = Xf @ W_up.to(DTYPE_ACC).t()
    gate = Xf @ W_gate.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    y = hid @ W_down.to(DTYPE_ACC).t()
    return y

def load_or_build_Ws() -> Tuple[List[int], torch.Tensor, torch.Tensor]:
    wm = read_index(cfg.MODEL_DIR)
    all_eids = find_layer_expert_ids(wm, cfg.LAYER)
    if not all_eids:
        raise RuntimeError(f"No experts found at layer={cfg.LAYER}")
    eids = all_eids[:cfg.MAX_EXPERTS]
    log(f"[found] layer={cfg.LAYER} total_experts={len(all_eids)} using={len(eids)} eids={eids}")

    cpath = ws_cache_path()
    if os.path.isfile(cpath):
        z = load_npz(cpath)
        if "meta" in z and "Ws" in z and "expert_ids" in z and "W_scale" in z:
            if _decode_meta(z["meta"]) == ws_meta(eids):
                Ws = torch.from_numpy(z["Ws"]).to(DTYPE_ACC).to(DEVICE)
                W_scale = torch.from_numpy(z["W_scale"]).to(DTYPE_ACC).to(DEVICE)
                log(f"[cache] loaded Ws -> {cpath}  Ws={tuple(Ws.shape)}")
                return [int(x) for x in z["expert_ids"].tolist()], Ws, W_scale
        log("[cache] Ws meta mismatch -> rebuilding.")

    per_e = {}
    need_keys = []
    for eid in eids:
        kk = pick_expert_tensor_keys(wm, cfg.LAYER, eid)
        if not kk:
            raise RuntimeError(f"Expert {eid} missing required tensors in index.")
        per_e[eid] = kk
        need_keys += [kk["up"], kk["down"], kk["gate"]]

    log("[load] reading tensors from shards ...")
    T = load_tensors_from_shards(cfg.MODEL_DIR, wm, sorted(set(need_keys)))

    W_up0 = T[per_e[eids[0]]["up"]]
    dff, H = int(W_up0.shape[0]), int(W_up0.shape[1])
    log(f"[shape] H={H} d_ff={dff}")

    ensure_calib_router(H=H, E_total=len(all_eids))
    if not cfg.CALIB_PATH or (not os.path.isfile(cfg.CALIB_PATH)):
        raise RuntimeError("CALIB_PATH is required for ridge. Set CALIB_PATH or CAPTURE_ENABLE=1.")
    X = load_calib_X(cfg.CALIB_PATH, H)
    log(f"[calib] X loaded: {tuple(X.shape)}  (path={cfg.CALIB_PATH})")

    P = None
    if cfg.RIDGE_WEIGHTED:
        if cfg.ROUTER_PATH and os.path.isfile(cfg.ROUTER_PATH):
            P = load_router_P(cfg.ROUTER_PATH)
            log(f"[router] P loaded: {tuple(P.shape)}  (path={cfg.ROUTER_PATH})")
        else:
            log("[router] RIDGE_WEIGHTED=1 but ROUTER_PATH missing -> forcing RIDGE_WEIGHTED=0")
            cfg.RIDGE_WEIGHTED = False

    Xf = X.to(DTYPE_ACC)
    I = torch.eye(H, dtype=DTYPE_ACC, device=DEVICE)
    XtX = Xf.t() @ Xf
    lam = cfg.RIDGE_DAMP * float(torch.trace(XtX).item()) / float(H)
    cholG = torch.linalg.cholesky(XtX + lam * I)

    Ws_list: List[torch.Tensor] = []
    scales: List[float] = []

    for i, eid in enumerate(tqdm(eids, desc="Build Ws (ridge)")):
        W_up = T[per_e[eid]["up"]].to(DEVICE)
        W_dn = T[per_e[eid]["down"]].to(DEVICE)
        W_gt = T[per_e[eid]["gate"]].to(DEVICE)

        Y = forward_mlp(X, W_gt, W_up, W_dn).to(DTYPE_ACC)

        if cfg.RIDGE_WEIGHTED and (P is not None):
            if cfg.ROUTER_EIDS_ARE_GLOBAL:
                w = torch.from_numpy(P[:X.shape[0], eid]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
            else:
                w = torch.from_numpy(P[:X.shape[0], i]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
            sw = torch.sqrt(w + 1e-12).view(-1, 1)
            Xw = Xf * sw
            Yw = Y * sw
            XtX_e = Xw.t() @ Xw
            lam_e = cfg.RIDGE_DAMP * float(torch.trace(XtX_e).item()) / float(H)
            chol = torch.linalg.cholesky(XtX_e + lam_e * I)
            XtY = Xw.t() @ Yw
            Wt = torch.cholesky_solve(XtY, chol)
            W = Wt.t().contiguous()
        else:
            XtY = Xf.t() @ Y
            Wt = torch.cholesky_solve(XtY, cholG)
            W = Wt.t().contiguous()

        if cfg.NORMALIZE_W:
            s = float(torch.linalg.norm(W, ord="fro").clamp_min(1e-12).item())
            W = (W / s).contiguous()
        else:
            s = 1.0

        Ws_list.append(W)
        scales.append(s)

    Ws = torch.stack(Ws_list, dim=0).to(DTYPE_ACC).to(DEVICE)     # normalized or not
    W_scale = torch.tensor(scales, dtype=DTYPE_ACC, device=DEVICE) # always applies to recover original scale

    save_npz(
        cpath,
        meta=_encode_meta(ws_meta(eids)),
        expert_ids=np.array(eids, dtype=np.int32),
        Ws=Ws.detach().cpu().numpy().astype(np.float32),
        W_scale=W_scale.detach().cpu().numpy().astype(np.float32),
    )
    log(f"[cache] wrote Ws -> {cpath}  size={os.path.getsize(cpath)/1e6:.2f} MB")
    return eids, Ws, W_scale

# ----------------------------
# Clustering helpers (dense_train)
# ----------------------------
@torch.no_grad()
def random_proj_features(Ws: torch.Tensor, d: int) -> torch.Tensor:
    E, n, _ = Ws.shape
    g = torch.Generator(device="cpu").manual_seed(SEED + 17)
    R = (torch.randint(0, 2, (n, d), generator=g, dtype=torch.int8) * 2 - 1).to(DTYPE_ACC).to(DEVICE)
    feats = []
    for e in range(E):
        W = Ws[e]
        row = torch.diag(W @ W.t())
        col = torch.diag(W.t() @ W)
        feats.append(torch.cat([(row @ R), (col @ R)], dim=0).unsqueeze(0))
    X = torch.cat(feats, dim=0)
    X = (X - X.mean(dim=0, keepdim=True)) / (X.std(dim=0, keepdim=True) + 1e-6)
    return X

@torch.no_grad()
def kmeans_torch(X: torch.Tensor, k: int, iters: int, restarts: int) -> torch.Tensor:
    best_lab = None
    best_inertia = float("inf")
    g = torch.Generator(device="cpu").manual_seed(SEED + 999)
    n = X.shape[0]

    for _ in range(max(1, restarts)):
        idx = torch.randperm(n, generator=g)[:k]
        C = X[idx].clone()
        for _ in range(iters):
            dist = torch.cdist(X, C)
            lab = dist.argmin(dim=1)
            # update
            for j in range(k):
                m = (lab == j)
                if m.any():
                    C[j] = X[m].mean(dim=0)
                else:
                    far = dist.min(dim=1).values.argmax().item()
                    C[j] = X[far].clone()
        inertia = float((torch.cdist(X, C).min(dim=1).values ** 2).sum().item())
        if inertia < best_inertia:
            best_inertia = inertia
            best_lab = lab.clone()
    assert best_lab is not None
    return best_lab.to(torch.int64)

@torch.no_grad()
def relabel_contiguous(labels: torch.Tensor) -> torch.Tensor:
    labels = labels.to(torch.int64)
    uniq = torch.unique(labels)
    out = labels.clone()
    for new, old in enumerate(uniq.tolist()):
        out[labels == int(old)] = int(new)
    return out

@torch.no_grad()
def merge_small_clusters(X: torch.Tensor, labels: torch.Tensor, min_size: int) -> torch.Tensor:
    labels = relabel_contiguous(labels)
    if min_size <= 1:
        return labels
    while True:
        K = int(labels.max().item()) + 1
        counts = torch.bincount(labels, minlength=K)
        small = (counts < min_size).nonzero(as_tuple=False).flatten()
        if small.numel() == 0:
            break
        C = torch.stack([X[labels == k].mean(dim=0) for k in range(K)], dim=0)
        for c in small.tolist():
            idxs = (labels == int(c)).nonzero(as_tuple=False).flatten()
            if idxs.numel() == 0:
                continue
            dist = torch.cdist(C[int(c)].unsqueeze(0), C).squeeze(0)
            dist[int(c)] = 1e9
            tgt = int(dist.argmin().item())
            labels[idxs] = tgt
        labels = relabel_contiguous(labels)
    return labels

@torch.no_grad()
def hierarchical_split(X: torch.Tensor, labels: torch.Tensor, max_size: int, max_k: int, split_iters: int) -> torch.Tensor:
    labels = relabel_contiguous(labels)
    if max_size <= 0:
        return labels
    while True:
        K = int(labels.max().item()) + 1
        if K >= max_k:
            break
        counts = torch.bincount(labels, minlength=K)
        biggest = int(torch.argmax(counts).item())
        bigsz = int(counts[biggest].item())
        if bigsz <= max_size:
            break
        idxs = (labels == biggest).nonzero(as_tuple=False).flatten()
        if idxs.numel() < 2:
            break
        sub = X[idxs]
        sub_lab = kmeans_torch(sub, k=2, iters=split_iters, restarts=1)
        a = idxs[sub_lab == 0]
        b = idxs[sub_lab == 1]
        if a.numel() == 0 or b.numel() == 0:
            break
        labels[b] = K
        labels = relabel_contiguous(labels)
    return labels

# ----------------------------
# Dense_train basis training
# ----------------------------
class OrthoParam(nn.Module):
    def __init__(self, n: int, init: str):
        super().__init__()
        if init == "random":
            g = torch.Generator(device="cpu").manual_seed(SEED + 333)
            M = torch.randn(n, n, generator=g, dtype=DTYPE_ACC)
        else:
            M = torch.eye(n, dtype=DTYPE_ACC)
        self.M = nn.Parameter(M.to(DEVICE))

    def orthogonal(self) -> torch.Tensor:
        Q, _ = torch.linalg.qr(self.M)
        return Q

def schedule(step: int, warmup: int, total: int) -> float:
    if total <= 0:
        return 1.0
    if step <= warmup:
        return 0.0
    return float(min(1.0, max(0.0, (step - warmup) / max(1, (total - warmup)))))

# IMPORTANT: This MUST be differentiable (NO @torch.no_grad here!)
def slice_X_batch(Ws_batch: torch.Tensor, U: torch.Tensor, V: torch.Tensor, S: torch.Tensor) -> torch.Tensor:
    U_S = U[:, S]                 # (n,s)
    V_S = V[:, S]                 # (n,s)
    T = Ws_batch @ V_S            # (Eb,n,s)
    Xs = torch.matmul(U_S.t().unsqueeze(0), T)  # (Eb,s,s)
    return Xs

def offdiag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    return (Xs - torch.diag_embed(D)).abs().mean()

def diag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    return torch.diagonal(Xs, dim1=1, dim2=2).abs().mean()

def block_group_sparsity_penalty(Xs: torch.Tensor, block: int) -> torch.Tensor:
    Eb, s, _ = Xs.shape
    b = int(block)
    if b <= 0:
        return torch.zeros((), dtype=DTYPE_ACC, device=Xs.device)
    nb = s // b
    if nb <= 0:
        return torch.zeros((), dtype=DTYPE_ACC, device=Xs.device)
    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()
    Xb = X.view(Eb, nb, b, nb, b).permute(0, 1, 3, 2, 4).contiguous()
    Eblk = (Xb * Xb).sum(dim=(3, 4))  # (Eb,nb,nb)
    P = Eblk.mean(dim=0)
    return torch.sqrt(P + 1e-12).sum() / (P.sum() + 1e-12)

@torch.no_grad()
def make_guidance_mask_from_Xs(Xs: torch.Tensor, block: int, target: float, max_blocks: int) -> Tuple[torch.Tensor, float, int]:
    Eb, s, _ = Xs.shape
    b = int(block)
    if b <= 0:
        return torch.ones(s, s, dtype=DTYPE_ACC, device=Xs.device), 1.0, 0
    nb = s // b
    if nb <= 0:
        return torch.ones(s, s, dtype=DTYPE_ACC, device=Xs.device), 1.0, 0
    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()
    Xb = X.view(Eb, nb, b, nb, b).permute(0, 1, 3, 2, 4).contiguous()
    Eg = (Xb * Xb).sum(dim=(3, 4)).mean(dim=0)  # (nb,nb)
    tot = float((X * X).sum().item()) / max(1, Eb)
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    csum = torch.cumsum(flat[order], dim=0)
    frac = csum / max(tot, 1e-12)
    need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else flat.numel())
    K = min(need, max_blocks, flat.numel())
    pick = order[:K]
    mask = torch.zeros(s2, s2, dtype=DTYPE_ACC, device=Xs.device)
    for idx in pick.tolist():
        bi = idx // nb
        bj = idx % nb
        i0 = bi * b
        j0 = bj * b
        mask[i0:i0 + b, j0:j0 + b] = 1.0
    if s2 < s:
        full = torch.zeros(s, s, dtype=DTYPE_ACC, device=Xs.device)
        full[:s2, :s2] = mask
        mask = full
    ef = float(frac[K - 1].item()) if K > 0 else 0.0
    return mask, ef, int(K)

# ----------------------------
# Block energy + selection
# ----------------------------
@torch.no_grad()
def block_energy_grid(X: torch.Tensor, b: int) -> torch.Tensor:
    n = X.shape[0]
    nb = (n + b - 1) // b
    if (n % b) != 0:
        Xp = torch.zeros(nb * b, nb * b, dtype=X.dtype, device=X.device)
        Xp[:n, :n] = X
        X = Xp
    Xb = X.view(nb, b, nb, b).permute(0, 2, 1, 3).contiguous()
    return (Xb * Xb).sum(dim=(2, 3))

@torch.no_grad()
def pick_top_blocks_from_energy(Eg: torch.Tensor, tot_energy: float, b: int, target: float, max_blocks: int) -> Tuple[List[Tuple[int, int, int, int]], float]:
    nb = Eg.shape[0]
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    csum = torch.cumsum(flat[order], dim=0)
    frac = csum / max(tot_energy, 1e-12)
    need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else flat.numel())
    K = min(need, max_blocks, flat.numel())
    pick = order[:K].tolist()
    blocks = []
    for idx in pick:
        bi = idx // nb
        bj = idx % nb
        blocks.append((bi * b, bj * b, b, b))
    eff = float(frac[K - 1].item()) if K > 0 else 0.0
    return blocks, eff

@torch.no_grad()
def choose_core_blocks(X_list: List[torch.Tensor]) -> dict:
    mode = cfg.CORE_MODE
    agg = cfg.CORE_AGG
    b = int(cfg.CORE_BLOCK)
    target = float(cfg.CORE_TARGET)
    max_blocks = int(cfg.CORE_MAX_BLOCKS)

    if mode == "none":
        return {"mode": "none", "blocks_shared": [], "blocks_per_expert": None, "energy_fracs": []}

    if mode == "blocktopk_perexpert":
        blocks_per = []
        efracs = []
        for X in X_list:
            tot = float((X * X).sum().item())
            Eg = block_energy_grid(X, b).to(DTYPE_ACC)
            blocks, ef = pick_top_blocks_from_energy(Eg, tot, b, target, max_blocks)
            blocks_per.append(blocks)
            efracs.append(ef)
        return {"mode": mode, "blocks_shared": [], "blocks_per_expert": blocks_per, "energy_fracs": efracs}

    # shared modes
    Eg_all = []
    tots = []
    for X in X_list:
        tots.append(float((X * X).sum().item()))
        Eg_all.append(block_energy_grid(X, b).to(DTYPE_ACC))
    tot = float(np.mean(tots))

    if mode == "blockdiag":
        nb = Eg_all[0].shape[0]
        diagE = torch.zeros(nb, dtype=DTYPE_ACC, device=Eg_all[0].device)
        for Eg in Eg_all:
            diagE += torch.diagonal(Eg, 0)
        diagE /= max(1, len(Eg_all))
        order = torch.argsort(diagE, descending=True)
        csum = torch.cumsum(diagE[order], dim=0)
        frac = csum / max(tot, 1e-12)
        need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else nb)
        K = min(need, max_blocks, nb)
        blocks = [(int(bi)*b, int(bi)*b, b, b) for bi in order[:K].tolist()]
        ef = float(frac[K - 1].item()) if K > 0 else 0.0
        return {"mode": mode, "blocks_shared": blocks, "blocks_per_expert": None, "energy_fracs": [ef]}

    if mode == "blocktopk":
        Eg = torch.stack(Eg_all, dim=0).amax(dim=0) if agg == "max" else torch.stack(Eg_all, dim=0).mean(dim=0)
        blocks, ef = pick_top_blocks_from_energy(Eg, tot, b, target, max_blocks)
        return {"mode": mode, "blocks_shared": blocks, "blocks_per_expert": None, "energy_fracs": [ef]}

    raise ValueError("CORE_MODE must be blocktopk_perexpert|blocktopk|blockdiag|none")

@torch.no_grad()
def pick_blocks_values(X: torch.Tensor, blocks: List[Tuple[int, int, int, int]]) -> List[Tuple[int, int, torch.Tensor]]:
    out = []
    n = X.shape[0]
    for (i0, j0, h, w) in blocks:
        h = min(h, n - i0)
        w = min(w, n - j0)
        if h <= 0 or w <= 0:
            continue
        out.append((int(i0), int(j0), X[i0:i0+h, j0:j0+w].clone()))
    return out

# ----------------------------
# Randomized SVD for residual mean
# ----------------------------
@torch.no_grad()
def rand_svd(A: torch.Tensor, r: int, n_iter: int = 2) -> Tuple[torch.Tensor, torch.Tensor]:
    n = A.shape[0]
    r = min(r, n)
    g = torch.Generator(device="cpu").manual_seed(SEED + 777)
    Omega = torch.randn(n, r, generator=g, dtype=DTYPE_ACC, device=A.device)
    Y = A @ Omega
    for _ in range(n_iter):
        Y = A @ (A.t() @ Y)
    Q, _ = torch.linalg.qr(Y)
    B = Q.t() @ A
    Uhat, _, Vh = torch.linalg.svd(B, full_matrices=False)
    U = (Q @ Uhat[:, :r]).contiguous()
    V = (Vh.t()[:, :r]).contiguous()
    return U, V

# ----------------------------
# Payload in-memory + runtime
# ----------------------------
class Payload:
    def __init__(self, E: int):
        self.meta: dict = {}
        self.expert_ids: List[int] = []
        self.cluster_of_pos: List[int] = [0] * E
        self.clusters: List[List[int]] = []

        # basis (dense)
        self.U: List[Optional[torch.Tensor]] = []
        self.V: List[Optional[torch.Tensor]] = []

        # basis (hadamard)
        self.pu: Optional[torch.Tensor] = None
        self.ipu: Optional[torch.Tensor] = None
        self.su: Optional[torch.Tensor] = None
        self.pv: Optional[torch.Tensor] = None
        self.ipv: Optional[torch.Tensor] = None
        self.sv: Optional[torch.Tensor] = None

        # per-cluster low-rank
        self.DL: List[Optional[torch.Tensor]] = []
        self.DR: List[Optional[torch.Tensor]] = []

        # per-expert blocks + gamma
        self.core_blocks: List[List[Tuple[int, int, torch.Tensor]]] = [[] for _ in range(E)]
        self.res_blocks: List[List[Tuple[int, int, torch.Tensor]]] = [[] for _ in range(E)]
        self.gam: List[Optional[torch.Tensor]] = [None for _ in range(E)]

        # per-expert scaling to recover original Ws
        self.W_scale: Optional[torch.Tensor] = None  # (E,)

    @torch.no_grad()
    def apply_expert(self, x: torch.Tensor, pos: int) -> torch.Tensor:
        c = self.cluster_of_pos[pos]
        if cfg.BASIS_MODE == "hadamard_perm":
            z = had_right(x, self.pu, self.su)  # type: ignore[arg-type]
        else:
            U = self.U[c]; V = self.V[c]
            assert U is not None and V is not None
            z = x @ U

        u = torch.zeros_like(z)

        for (i0, j0, Bc) in self.core_blocks[pos]:
            h, w = Bc.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ Bc

        DL = self.DL[c]; DR = self.DR[c]
        assert DL is not None and DR is not None
        g = self.gam[pos]
        assert g is not None
        u += ((z @ DL) * g.view(1, -1)) @ DR.t()

        for (i0, j0, Bb) in self.res_blocks[pos]:
            h, w = Bb.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ Bb

        if cfg.BASIS_MODE == "hadamard_perm":
            y = had_right_T(u, self.ipv, self.sv)  # type: ignore[arg-type]
        else:
            V = self.V[c]
            assert V is not None
            y = u @ V.t()

        # apply original scale if Ws were normalized
        if self.W_scale is not None:
            y = y * float(self.W_scale[pos].item())
        return y

    @torch.no_grad()
    def apply_mixture(self, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
        y = torch.zeros_like(x)
        for a, pos in zip(gates.tolist(), routed):
            y += float(a) * self.apply_expert(x, int(pos))
        return y

# ----------------------------
# Build payload pieces (dense_train)
# ----------------------------
@torch.no_grad()
def build_cluster_dense(Ws: torch.Tensor, idx: List[int], U: torch.Tensor, V: torch.Tensor) -> Tuple[
    List[List[Tuple[int, int, torch.Tensor]]], torch.Tensor, torch.Tensor, List[torch.Tensor], List[List[Tuple[int, int, torch.Tensor]]], dict
]:
    X_list = [(U.t() @ Ws[p] @ V).contiguous() for p in idx]
    core = choose_core_blocks(X_list)

    core_per: List[List[Tuple[int, int, torch.Tensor]]] = []
    if core["blocks_per_expert"] is not None:
        for X, blocks in zip(X_list, core["blocks_per_expert"]):
            core_per.append(pick_blocks_values(X, blocks))
    else:
        shared = core["blocks_shared"]
        for X in X_list:
            core_per.append(pick_blocks_values(X, shared))

    R_list = []
    for X, cb in zip(X_list, core_per):
        Xc = torch.zeros_like(X)
        for (i0, j0, Bc) in cb:
            h, w = Bc.shape
            Xc[i0:i0+h, j0:j0+w] = Bc
        R_list.append((X - Xc).contiguous())

    Rmean = torch.stack(R_list, dim=0).mean(dim=0)
    r = min(cfg.RES_RANK, Rmean.shape[0])
    DL, DR = rand_svd(Rmean, r=r, n_iter=2)

    gam_list: List[torch.Tensor] = []
    res_per: List[List[Tuple[int, int, torch.Tensor]]] = []
    for Rm in R_list:
        g = torch.sum(DL * (Rm @ DR), dim=0).contiguous()
        gam_list.append(g)
        R2 = (Rm - (DL * g.view(1, -1)) @ DR.t()).contiguous()

        b = int(cfg.RES_BSIZE)
        Eg = block_energy_grid(R2, b).to(DTYPE_ACC)
        flat = Eg.reshape(-1)
        order = torch.argsort(flat, descending=True)[:min(cfg.RES_BLOCKS, flat.numel())].tolist()
        nb = Eg.shape[0]
        blocks = []
        for idv in order:
            bi = idv // nb
            bj = idv % nb
            i0 = bi * b
            j0 = bj * b
            hh = min(b, R2.shape[0] - i0)
            ww = min(b, R2.shape[1] - j0)
            if hh > 0 and ww > 0:
                blocks.append((int(i0), int(j0), R2[i0:i0+hh, j0:j0+ww].clone()))
        res_per.append(blocks)

    return core_per, DL.contiguous(), DR.contiguous(), gam_list, res_per, core

@torch.no_grad()
def refine_expert_dense(Ws: torch.Tensor, pos: int, U: torch.Tensor, V: torch.Tensor,
                        core_blocks: List[Tuple[int, int, torch.Tensor]],
                        DL: torch.Tensor, DR: torch.Tensor, g: torch.Tensor,
                        res_blocks: List[Tuple[int, int, torch.Tensor]]) -> List[Tuple[int, int, torch.Tensor]]:
    X = (U.t() @ Ws[pos] @ V).contiguous()

    def reconstruct() -> torch.Tensor:
        Xc = torch.zeros_like(X)
        for (i0, j0, Bc) in core_blocks:
            h, w = Bc.shape
            Xc[i0:i0+h, j0:j0+w] = Bc
        Xlr = (DL * g.view(1, -1)) @ DR.t()
        Xr = torch.zeros_like(X)
        for (i0, j0, Bb) in res_blocks:
            h, w = Bb.shape
            Xr[i0:i0+h, j0:j0+w] += Bb
        return (Xc + Xlr + Xr).contiguous()

    denom = torch.linalg.norm(X, ord="fro").clamp_min(1e-12)
    Xhat = reconstruct()
    err = float((torch.linalg.norm(Xhat - X, ord="fro") / denom).item())
    if err <= cfg.REFINE_ERR_TARGET:
        return res_blocks

    R = (X - Xhat).contiguous()
    added = 0
    b = int(cfg.REFINE_BSIZE)
    while err > cfg.REFINE_ERR_TARGET and added < cfg.REFINE_MAX_EXTRA:
        Eg = block_energy_grid(R, b).to(DTYPE_ACC)
        flat = Eg.reshape(-1)
        if float(flat.max().item()) <= 1e-18:
            break
        nb = Eg.shape[0]
        idx = int(torch.argmax(flat).item())
        bi = idx // nb
        bj = idx % nb
        i0 = bi * b
        j0 = bj * b
        hh = min(b, R.shape[0] - i0)
        ww = min(b, R.shape[1] - j0)
        if hh <= 0 or ww <= 0:
            break
        Bb = R[i0:i0+hh, j0:j0+ww].clone()
        res_blocks.append((int(i0), int(j0), Bb))
        R[i0:i0+hh, j0:j0+ww] -= Bb
        added += 1
        if added % 16 == 0:
            Xhat = reconstruct()
            err = float((torch.linalg.norm(Xhat - X, ord="fro") / denom).item())
    return res_blocks

# ----------------------------
# Eval
# ----------------------------
@torch.no_grad()
def frob(A: torch.Tensor) -> torch.Tensor:
    return torch.linalg.norm(A, ord="fro")

@torch.no_grad()
def eval_payload(payload: Payload, Ws: torch.Tensor, W_scale: torch.Tensor):
    E, n, _ = Ws.shape

    errs = []
    for pos in range(E):
        x = torch.randn(8, n, dtype=DTYPE_ACC, device=DEVICE)
        # reference uses recovered original scale
        y_ref = x @ (Ws[pos] * W_scale[pos])
        y_hat = payload.apply_expert(x, pos)
        err = float((frob(y_hat - y_ref) / (frob(y_ref) + 1e-12)).item())
        errs.append(err)
    log(f"[eval] per-expert rel-error  mean={float(np.mean(errs)):.6f}  p95={float(np.percentile(errs,95)):.6f}  max={float(np.max(errs)):.6f}")

    mix_errs = []
    for _ in range(cfg.EVAL_TRIALS):
        x = torch.randn(cfg.EVAL_BATCH, n, dtype=DTYPE_ACC, device=DEVICE)
        routed = random.sample(range(E), k=min(cfg.ROUTED_K, E))
        gates = torch.rand(len(routed), dtype=DTYPE_ACC, device=DEVICE)
        gates = gates / gates.sum().clamp_min(1e-12)

        Wsum = torch.zeros(n, n, dtype=DTYPE_ACC, device=DEVICE)
        for a, pos in zip(gates, routed):
            Wsum += float(a.item()) * (Ws[int(pos)] * W_scale[int(pos)])
        y_ref = x @ Wsum

        y_hat = payload.apply_mixture(x, routed, gates)
        mix_errs.append(float((frob(y_hat - y_ref) / (frob(y_ref) + 1e-12)).item()))
    log(f"[eval] routed rel-error = {float(np.mean(mix_errs)):.6f} ± {float(np.std(mix_errs)):.6f}  (trials={cfg.EVAL_TRIALS})")

# ----------------------------
# Banner + main
# ----------------------------
def banner():
    log("== DeepSeek KT++-X OFFLINE v10.3 ==")
    log(f"Time:        {now()}")
    log(f"MODEL_DIR:   {cfg.MODEL_DIR}")
    log(f"OUTPUT_DIR:  {cfg.OUTPUT_DIR}")
    log(f"LAYER:       {cfg.LAYER}")
    log(f"MAX_EXPERTS:  {cfg.MAX_EXPERTS}")
    log(f"CALIB_PATH:  {cfg.CALIB_PATH or '(none)'}  CALIB_SAMPLES(cap)={cfg.CALIB_SAMPLES}")
    log(f"ROUTER_PATH: {cfg.ROUTER_PATH or '(none)'}  RIDGE_WEIGHTED={cfg.RIDGE_WEIGHTED}")
    log(f"RIDGE_DAMP:  {cfg.RIDGE_DAMP}  NORMALIZE_W={cfg.NORMALIZE_W}")
    log(f"BASIS_MODE:  {cfg.BASIS_MODE}")
    log(f"CLUSTER:     M0={(cfg.M0 if cfg.M0>0 else '(auto)')} M_MAX={cfg.M_MAX} min_size={cfg.CLUSTER_MIN_SIZE} max_size={cfg.CLUSTER_MAX_SIZE}")
    log(f"TRAIN:       steps={cfg.TRAIN_STEPS} warmup={cfg.TRAIN_WARMUP} lr={cfg.TRAIN_LR} subm={cfg.SUBM} batchE={cfg.BATCH_E}")
    log(f"CORE:        {cfg.CORE_MODE} block={cfg.CORE_BLOCK} target={cfg.CORE_TARGET} max_blocks={cfg.CORE_MAX_BLOCKS}")
    log(f"RESIDUAL:    rank={cfg.RES_RANK} blocks={cfg.RES_BLOCKS} bsize={cfg.RES_BSIZE}")
    log(f"REFINE:      enable={cfg.REFINE_ENABLE} err_target={cfg.REFINE_ERR_TARGET} max_extra={cfg.REFINE_MAX_EXTRA}")
    log(f"DEVICE:      {DEVICE}  Torch={torch.__version__} threads={NTHREADS}")
    log("")

def main():
    banner()

    expert_ids, Ws, W_scale = load_or_build_Ws()
    E, n, _ = Ws.shape
    log(f"[Ws] shape={tuple(Ws.shape)}  (stored normalized={cfg.NORMALIZE_W})")

    payload = Payload(E=E)
    payload.expert_ids = expert_ids
    payload.W_scale = W_scale.detach().clone()

    if cfg.BASIS_MODE == "hadamard_perm":
        if not is_power_of_two(n):
            raise RuntimeError(f"hadamard_perm requires H power-of-two, got H={n}")
        payload.clusters = [list(range(E))]
        payload.cluster_of_pos = [0] * E
        pu, ipu, su = make_perm_sign(n, cfg.HAD_SEED + 1000)
        pv, ipv, sv = make_perm_sign(n, cfg.HAD_SEED + 2000)
        payload.pu, payload.ipu, payload.su = pu, ipu, su
        payload.pv, payload.ipv, payload.sv = pv, ipv, sv
        payload.U = [None]
        payload.V = [None]
        payload.DL = [None]
        payload.DR = [None]

        # Build X in basis (single cluster)
        X_list = [mat_to_basis_hadamard(Ws[pos], pu, ipu, su, pv, ipv, sv) for pos in range(E)]
        core = choose_core_blocks(X_list)

        if core["blocks_per_expert"] is not None:
            for pos, blocks in enumerate(core["blocks_per_expert"]):
                payload.core_blocks[pos] = pick_blocks_values(X_list[pos], blocks)
        else:
            shared = core["blocks_shared"]
            for pos in range(E):
                payload.core_blocks[pos] = pick_blocks_values(X_list[pos], shared)

        R_list = []
        for pos in range(E):
            X = X_list[pos]
            Xc = torch.zeros_like(X)
            for (i0, j0, Bc) in payload.core_blocks[pos]:
                h, w = Bc.shape
                Xc[i0:i0+h, j0:j0+w] = Bc
            R_list.append((X - Xc).contiguous())

        Rmean = torch.stack(R_list, dim=0).mean(dim=0)
        r = min(cfg.RES_RANK, n)
        DL, DR = rand_svd(Rmean, r=r, n_iter=2)
        payload.DL = [DL.contiguous()]
        payload.DR = [DR.contiguous()]

        for pos in range(E):
            Rm = R_list[pos]
            g = torch.sum(DL * (Rm @ DR), dim=0).contiguous()
            payload.gam[pos] = g
            R2 = (Rm - (DL * g.view(1, -1)) @ DR.t()).contiguous()

            b = int(cfg.RES_BSIZE)
            Eg = block_energy_grid(R2, b).to(DTYPE_ACC)
            flat = Eg.reshape(-1)
            order = torch.argsort(flat, descending=True)[:min(cfg.RES_BLOCKS, flat.numel())].tolist()
            nb = Eg.shape[0]
            blocks = []
            for idv in order:
                bi = idv // nb
                bj = idv % nb
                i0 = bi * b
                j0 = bj * b
                hh = min(b, n - i0)
                ww = min(b, n - j0)
                if hh > 0 and ww > 0:
                    blocks.append((int(i0), int(j0), R2[i0:i0+hh, j0:j0+ww].clone()))
            payload.res_blocks[pos] = blocks

        log("[build] hadamard_perm payload built.")
        eval_payload(payload, Ws, W_scale)
        # You can add NPZ serialization here similar to v10.2 if needed.
        log("✅ Done.")
        return

    # Dense_train clustering
    Xfeat = random_proj_features(Ws, d=cfg.CLUSTER_FEAT_D)
    if cfg.M0 > 0:
        M0 = min(cfg.M0, E)
    else:
        M0 = int(round(2.0 * math.sqrt(E)))
        M0 = max(4, min(M0, E))
    M0 = max(2, min(M0, E))

    labels = kmeans_torch(Xfeat, k=M0, iters=cfg.CLUSTER_ITERS, restarts=cfg.CLUSTER_RESTARTS)
    labels = merge_small_clusters(Xfeat, labels, min_size=cfg.CLUSTER_MIN_SIZE)
    labels = hierarchical_split(Xfeat, labels, max_size=cfg.CLUSTER_MAX_SIZE, max_k=min(cfg.M_MAX, E), split_iters=cfg.SPLIT_ITERS)
    labels = merge_small_clusters(Xfeat, labels, min_size=cfg.CLUSTER_MIN_SIZE)
    labels = relabel_contiguous(labels)

    M = int(labels.max().item()) + 1
    clusters = [torch.nonzero(labels == m, as_tuple=False).flatten().tolist() for m in range(M)]
    clusters = [c for c in clusters if len(c) > 0]
    M = len(clusters)
    log(f"[cluster] M={M} sizes={[len(c) for c in clusters]}")
    payload.clusters = clusters
    payload.cluster_of_pos = [0] * E
    for m, idx in enumerate(clusters):
        for pos in idx:
            payload.cluster_of_pos[int(pos)] = m

    # Init parameters
    U_par = [OrthoParam(n, init=cfg.INIT_BASIS) for _ in range(M)]
    V_par = [OrthoParam(n, init=cfg.INIT_BASIS) for _ in range(M)]

    # Train
    if cfg.TRAIN_STEPS > 0:
        params = [p.M for p in U_par] + [p.M for p in V_par]
        opt = torch.optim.Adam(params, lr=cfg.TRAIN_LR)

        guidance_masks: Dict[int, torch.Tensor] = {}
        guidance_stats: Dict[int, Tuple[float, int]] = {}
        t0 = time.perf_counter()

        with torch.enable_grad():
            for step in range(1, cfg.TRAIN_STEPS + 1):
                S = torch.randperm(n, device=DEVICE)[:min(cfg.SUBM, n)]

                # refresh guidance (no_grad OK)
                if cfg.TRAIN_LAM_GUIDE > 0 and (step % max(1, cfg.TRAIN_GUIDE_EVERY) == 0 or step == 1):
                    with torch.no_grad():
                        guidance_masks.clear()
                        guidance_stats.clear()
                        for m, idx in enumerate(clusters):
                            if len(idx) < cfg.TRAIN_MIN_CLUSTER:
                                continue
                            Uo = U_par[m].orthogonal()
                            Vo = V_par[m].orthogonal()
                            if 0 < cfg.BATCH_E < len(idx):
                                pick = torch.randperm(len(idx), device=DEVICE)[:cfg.BATCH_E].tolist()
                                idx_step = [idx[p] for p in pick]
                            else:
                                idx_step = idx
                            Xs_ng = slice_X_batch(Ws[idx_step], Uo, Vo, S).detach()
                            mask, ef, kblk = make_guidance_mask_from_Xs(
                                Xs_ng, block=cfg.CORE_BLOCK,
                                target=cfg.TRAIN_GUIDE_TARGET,
                                max_blocks=cfg.TRAIN_GUIDE_MAX_BLOCKS
                            )
                            guidance_masks[m] = mask
                            guidance_stats[m] = (ef, kblk)

                lam_ramp = schedule(step, cfg.TRAIN_WARMUP, cfg.TRAIN_STEPS)
                lam_block = cfg.TRAIN_LAM_BLOCK * lam_ramp
                lam_guide = cfg.TRAIN_LAM_GUIDE * lam_ramp

                L_total = None
                n_terms = 0

                for m, idx in enumerate(clusters):
                    if len(idx) < cfg.TRAIN_MIN_CLUSTER:
                        continue
                    Uo = U_par[m].orthogonal()
                    Vo = V_par[m].orthogonal()

                    if 0 < cfg.BATCH_E < len(idx):
                        pick = torch.randperm(len(idx), device=DEVICE)[:cfg.BATCH_E].tolist()
                        idx_step = [idx[p] for p in pick]
                    else:
                        idx_step = idx

                    Xs = slice_X_batch(Ws[idx_step], Uo, Vo, S)

                    off = offdiag_abs_mean(Xs)
                    diag = diag_abs_mean(Xs).clamp_min(1e-6)

                    if cfg.TRAIN_OBJ == "ratio":
                        base = off / diag
                    else:
                        base = torch.log(off + 1e-6) - torch.log(diag)

                    if lam_block > 0 and cfg.CORE_MODE.startswith("block"):
                        base = base + lam_block * block_group_sparsity_penalty(Xs, cfg.CORE_BLOCK)

                    if lam_guide > 0 and (m in guidance_masks):
                        Mmask = guidance_masks[m]
                        Etot = (Xs * Xs).mean().clamp_min(1e-12)
                        Eout = ((Xs * (1.0 - Mmask)) ** 2).mean()
                        base = base + lam_guide * (Eout / Etot)

                    L_total = base if (L_total is None) else (L_total + base)
                    n_terms += 1

                if L_total is None:
                    log("[train] no valid clusters >= TRAIN_MIN_CLUSTER")
                    break

                L_total = L_total / max(1, n_terms)

                # Debug: ensure we actually have grads
                if step == 1 and (not L_total.requires_grad):
                    raise RuntimeError("Loss does not require grad. Something is still under no_grad/inference_mode.")

                opt.zero_grad(set_to_none=True)
                L_total.backward()
                if cfg.GRAD_CLIP > 0:
                    torch.nn.utils.clip_grad_norm_(params, max_norm=cfg.GRAD_CLIP)
                opt.step()

                if (step % cfg.REORTHO_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                    with torch.no_grad():
                        for p in U_par:
                            p.M.copy_(p.orthogonal())
                        for p in V_par:
                            p.M.copy_(p.orthogonal())

                if step == 1 or (step % cfg.REPORT_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                    t1 = time.perf_counter()
                    if guidance_stats:
                        ef_mean = float(np.mean([v[0] for v in guidance_stats.values()]))
                        kb_mean = float(np.mean([v[1] for v in guidance_stats.values()]))
                        log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} loss={float(L_total.item()):.4f} "
                            f"lam_block={lam_block:.3f} lam_guide={lam_guide:.3f} "
                            f"guide_energy≈{ef_mean:.3f} guide_blocks≈{kb_mean:.1f} (+{t1-t0:.1f}s)")
                    else:
                        log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} loss={float(L_total.item()):.4f} "
                            f"lam_block={lam_block:.3f} lam_guide={lam_guide:.3f} (+{t1-t0:.1f}s)")
                    t0 = t1

    # Freeze bases
    payload.U = [U_par[m].orthogonal().detach().contiguous() for m in range(M)]
    payload.V = [V_par[m].orthogonal().detach().contiguous() for m in range(M)]
    payload.DL = [None] * M
    payload.DR = [None] * M

    # Build payload
    log("[build] payloads ...")
    for m, idx in enumerate(clusters):
        U = payload.U[m]; V = payload.V[m]
        assert U is not None and V is not None
        core_per, DL, DR, gam_list, res_per, core = build_cluster_dense(Ws, idx, U, V)
        payload.DL[m] = DL
        payload.DR[m] = DR

        if cfg.REFINE_ENABLE:
            for j, pos in enumerate(idx):
                res_per[j] = refine_expert_dense(Ws, pos, U, V, core_per[j], DL, DR, gam_list[j], res_per[j])

        for j, pos in enumerate(idx):
            payload.core_blocks[pos] = core_per[j]
            payload.res_blocks[pos] = res_per[j]
            payload.gam[pos] = gam_list[j].contiguous()

        ef = core["energy_fracs"]
        log(f"  - cluster{m}: E={len(idx)} core_blocks(mean)≈{float(np.mean([len(payload.core_blocks[p]) for p in idx])):.1f} "
            f"core_energy(mean/min)≈{float(np.mean(ef)):.3f}/{float(np.min(ef)):.3f} r={DL.shape[1]}")

    # Eval
    eval_payload(payload, Ws, W_scale)

    log("✅ Done. (If you want NPZ serialization like v10.2, tell me your exact runtime loader expectations.)")

if __name__ == "__main__":
    main()


== DeepSeek KT++-X OFFLINE v10.3 ==
Time:        2026-01-13 11:49:57
MODEL_DIR:   /home/daniyar/deepseek-model
OUTPUT_DIR:  /home/daniyar/moe_ws_outputs
LAYER:       1
MAX_EXPERTS:  16
CALIB_PATH:  (none)  CALIB_SAMPLES(cap)=4096
ROUTER_PATH: (none)  RIDGE_WEIGHTED=False
RIDGE_DAMP:  0.001  NORMALIZE_W=True
BASIS_MODE:  dense_train
CLUSTER:     M0=(auto) M_MAX=16 min_size=2 max_size=4
TRAIN:       steps=24 warmup=6 lr=0.05 subm=256 batchE=4
CORE:        blocktopk_perexpert block=64 target=0.85 max_blocks=256
RESIDUAL:    rank=512 blocks=192 bsize=64
REFINE:      enable=True err_target=0.05 max_extra=256
DEVICE:      cpu  Torch=2.4.1+cpu threads=8

[found] layer=1 total_experts=64 using=16 eids=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
[load] reading tensors from shards ...
[shape] H=2048 d_ff=1408
[calib] CALIB_PATH not set -> auto-found /home/daniyar/moe_ws_outputs/calib_layer1_X.npz
[calib] X loaded: (160, 2048)  (path=/home/daniyar/moe_ws_outputs/calib_layer1_X.npz)


Build Ws (ridge):   0%|          | 0/16 [00:00<?, ?it/s]

[cache] wrote Ws -> /home/daniyar/moe_ws_outputs/Ws_cache_layer1_E16_ridge_v10_3.npz  size=250.24 MB
[Ws] shape=(16, 2048, 2048)  (stored normalized=True)
[cluster] M=6 sizes=[3, 3, 3, 3, 2, 2]
[train] step   1/24 loss=-0.0118 lam_block=0.000 lam_guide=0.000 guide_energy≈0.775 guide_blocks≈11.8 (+7.0s)
[train] step   4/24 loss=-1.2423 lam_block=0.000 lam_guide=0.000 guide_energy≈0.776 guide_blocks≈8.8 (+20.6s)
[train] step   8/24 loss=-1.2404 lam_block=0.011 lam_guide=0.111 guide_energy≈0.774 guide_blocks≈8.2 (+24.9s)
[train] step  12/24 loss=-1.0583 lam_block=0.033 lam_guide=0.333 guide_energy≈0.783 guide_blocks≈4.0 (+24.9s)
[train] step  16/24 loss=-0.3540 lam_block=0.056 lam_guide=0.556 guide_energy≈0.786 guide_blocks≈7.8 (+24.9s)
[train] step  20/24 loss=2.2182 lam_block=0.078 lam_guide=0.778 guide_energy≈0.771 guide_blocks≈9.8 (+24.9s)
[train] step  24/24 loss=1.5800 lam_block=0.100 lam_guide=1.000 guide_energy≈0.779 guide_blocks≈7.5 (+24.9s)
[build] payloads ...
  - cluster0: E=3

In [18]:
#!/usr/bin/env python3
# ============================================================
# DeepSeek KT++-X OFFLINE v10.4  (single-file GIANT runner)
#
# Goal: maximum accuracy + configurable compression for MoE expert linearization.
#
# Pipeline (offline, local shards):
#   0) (Optional) CAPTURE: collect calibration hidden-states X (and router probs P) using transformers
#   1) Build linearized expert matrices Ws via ridge regression:
#        For each expert e:  Y_e = MLP_e(X)    then solve  W_e ≈ argmin ||X W^T - Y||_2^2 (+ ridge)
#        Optionally weighted by router probabilities P[:, eid]
#   2) Cluster experts (optional, for dense_train basis)
#   3) Train orthogonal bases U/V per cluster to promote block sparsity (dense_train)
#   4) Build payload per expert in basis space:
#        X_e = U^T W_e V
#        - CORE blocks: choose blocks to hit CORE_TARGET energy (per-expert or shared)
#        - LOW-RANK residual: DL/DR shared per cluster; coefficients per expert (diag or full)
#        - RES blocks: sparse residual blocks
#        - REFINE: optionally add extra blocks until per-expert basis-space error <= target
#   5) Save payload NPZ + provide runtime loader + eval
#
# IMPORTANT:
#   - Your biggest limiter to "99–100%" is calibration size.
#     If X is only (160,2048), your ridge Ws are fundamentally low-quality.
#     Fix by capturing thousands+ tokens: CALIB_SAMPLES=16384+ (or more).
#   - For 99–100% reconstruction you must raise CORE_TARGET and/or budgets.
#
# Dependencies:
#   - torch, numpy, safetensors, tqdm
#   - transformers only if CAPTURE_ENABLE=1 (or CAPTURE_FORCE=1)
#
# Usage (examples):
#   # Max accuracy profile (big payload):
#   PRESET=maxacc DEVICE=cuda CALIB_SAMPLES=32768 RIDGE_WEIGHTED=1 \
#   CORE_BLOCK=32 CORE_TARGET=0.995 CORE_MAX_BLOCKS=8192 \
#   RES_RANK=2048 RES_COEF=full RES_BLOCKS=4096 \
#   REFINE_ENABLE=1 REFINE_ERR_TARGET=0.01 REFINE_MAX_EXTRA=32768 \
#   CAPTURE_ENABLE=1 CAPTURE_ITERS=64 CAPTURE_MAX_TOKENS=2048 \
#   python3 ktxx_offline_v10_4.py
#
#   # Balanced profile:
#   PRESET=balanced DEVICE=cuda CALIB_SAMPLES=8192 CORE_TARGET=0.97 RES_RANK=1024 python3 ktxx_offline_v10_4.py
#
# ============================================================

import os, re, json, math, time, random, sys
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors import safe_open

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):  # type: ignore
        return x

# ----------------------------
# Robust env parsing
# ----------------------------
def _env_str(k: str, d: str) -> str:
    return os.environ.get(k, d)

def _env_int(k: str, d: int) -> int:
    try:
        return int(os.environ.get(k, str(d)))
    except Exception:
        return d

def _env_float(k: str, d: float) -> float:
    try:
        return float(os.environ.get(k, str(d)))
    except Exception:
        return d

def _env_bool(k: str, d: bool) -> bool:
    v = os.environ.get(k, None)
    if v is None:
        return d
    return v.strip().lower() in ("1", "true", "yes", "y", "on")

# ----------------------------
# Threads / determinism
# ----------------------------
NTHREADS = _env_int("KTXX_THREADS", 8)
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(NTHREADS))
try:
    torch.set_num_threads(NTHREADS)
except Exception:
    pass

SEED = _env_int("SEED", 1234)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(_env_str("DEVICE", "cpu"))

DTYPE_ACC = torch.float32  # compute dtype
DTYPE_STORE_F16 = torch.float16

def now() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")

def log(msg: str):
    print(msg, flush=True)

# ----------------------------
# Config
# ----------------------------
@dataclass
class Cfg:
    # Paths
    MODEL_DIR: str = _env_str("MODEL_DIR", "/home/daniyar/deepseek-model")
    OUTPUT_DIR: str = _env_str("OUTPUT_DIR", "/home/daniyar/moe_ws_outputs")

    # Slice
    LAYER: int = _env_int("LAYER", 1)
    MAX_EXPERTS: int = _env_int("MAX_EXPERTS", 16)

    # Calibration / router
    CALIB_PATH: str = _env_str("CALIB_PATH", "").strip()
    ROUTER_PATH: str = _env_str("ROUTER_PATH", "").strip()
    CALIB_SAMPLES: int = _env_int("CALIB_SAMPLES", 4096)
    RIDGE_WEIGHTED: bool = _env_bool("RIDGE_WEIGHTED", False)
    ROUTER_EIDS_ARE_GLOBAL: bool = _env_bool("ROUTER_EIDS_ARE_GLOBAL", True)

    # Capture (optional)
    CAPTURE_ENABLE: bool = _env_bool("CAPTURE_ENABLE", False)
    CAPTURE_FORCE: bool = _env_bool("CAPTURE_FORCE", False)
    CAPTURE_ITERS: int = _env_int("CAPTURE_ITERS", 32)
    CAPTURE_BATCH: int = _env_int("CAPTURE_BATCH", 1)
    CAPTURE_MAX_TOKENS: int = _env_int("CAPTURE_MAX_TOKENS", 1024)
    CAPTURE_TEXT: str = _env_str(
        "CAPTURE_TEXT",
        ("DeepSeek models use mixture-of-experts layers. "
         "We capture intermediate activations for calibration. " * 256)
    )
    CAPTURE_TEXT_FILE: str = _env_str("CAPTURE_TEXT_FILE", "").strip()
    CAPTURE_KEEP_PAD: bool = _env_bool("CAPTURE_KEEP_PAD", False)

    HF_TRUST_REMOTE_CODE: bool = _env_bool("HF_TRUST_REMOTE_CODE", True)
    HF_LOCAL_FILES_ONLY: bool = _env_bool("HF_LOCAL_FILES_ONLY", True)
    HF_AUTO_PIP: bool = _env_bool("HF_AUTO_PIP", False)

    # Ridge build
    RIDGE_DAMP: float = _env_float("RIDGE_DAMP", 1e-3)
    NORMALIZE_W: bool = _env_bool("NORMALIZE_W", True)  # normalize Ws and store per-expert scales

    # Basis mode
    BASIS_MODE: str = _env_str("BASIS_MODE", "dense_train").strip().lower()  # dense_train|identity|hadamard_perm
    BASIS_STORE_DTYPE: str = _env_str("BASIS_STORE_DTYPE", "float16").strip().lower()  # float16|float32

    # Clustering (dense_train)
    M0: int = _env_int("M0", 0)  # 0 => auto
    M_MAX: int = _env_int("M_MAX", 16)
    CLUSTER_FEAT_D: int = _env_int("CLUSTER_FEAT_D", 64)
    CLUSTER_ITERS: int = _env_int("CLUSTER_ITERS", 60)
    CLUSTER_RESTARTS: int = _env_int("CLUSTER_RESTARTS", 4)
    CLUSTER_MIN_SIZE: int = _env_int("CLUSTER_MIN_SIZE", 2)
    CLUSTER_MAX_SIZE: int = _env_int("CLUSTER_MAX_SIZE", 4)
    SPLIT_ITERS: int = _env_int("SPLIT_ITERS", 50)

    # Training (dense_train)
    TRAIN_STEPS: int = _env_int("TRAIN_STEPS", 24)
    TRAIN_WARMUP: int = _env_int("TRAIN_WARMUP", 6)
    TRAIN_LR: float = _env_float("TRAIN_LR", 5e-2)
    SUBM: int = _env_int("SUBM", 256)          # submatrix size for training objective
    BATCH_E: int = _env_int("BATCH_E", 4)      # per-cluster experts per step
    TRAIN_MIN_CLUSTER: int = _env_int("TRAIN_MIN_CLUSTER", 2)
    REORTHO_EVERY: int = _env_int("REORTHO_EVERY", 4)
    REPORT_EVERY: int = _env_int("REPORT_EVERY", 4)
    GRAD_CLIP: float = _env_float("GRAD_CLIP", 1.0)

    TRAIN_OBJ: str = _env_str("TRAIN_OBJ", "logratio").strip().lower()  # logratio|ratio
    TRAIN_LAM_BLOCK: float = _env_float("TRAIN_LAM_BLOCK", 0.10)
    TRAIN_LAM_GUIDE: float = _env_float("TRAIN_LAM_GUIDE", 1.0)
    TRAIN_GUIDE_EVERY: int = _env_int("TRAIN_GUIDE_EVERY", 2)
    TRAIN_GUIDE_TARGET: float = _env_float("TRAIN_GUIDE_TARGET", 0.80)
    TRAIN_GUIDE_MAX_BLOCKS: int = _env_int("TRAIN_GUIDE_MAX_BLOCKS", 2048)

    # Core selection
    CORE_MODE: str = _env_str("CORE_MODE", "blocktopk_perexpert").strip().lower()  # blocktopk_perexpert|blocktopk|blockdiag|none
    CORE_AGG: str = _env_str("CORE_AGG", "mean").strip().lower()  # mean|max
    CORE_BLOCK: int = _env_int("CORE_BLOCK", 64)
    CORE_TARGET: float = _env_float("CORE_TARGET", 0.85)
    CORE_MAX_BLOCKS: int = _env_int("CORE_MAX_BLOCKS", 256)

    # Residual
    RES_RANK: int = _env_int("RES_RANK", 512)
    RES_COEF: str = _env_str("RES_COEF", "diag").strip().lower()  # diag|full
    RES_BLOCKS: int = _env_int("RES_BLOCKS", 192)
    RES_BSIZE: int = _env_int("RES_BSIZE", 64)

    # Refine
    REFINE_ENABLE: bool = _env_bool("REFINE_ENABLE", True)
    REFINE_ERR_TARGET: float = _env_float("REFINE_ERR_TARGET", 0.05)
    REFINE_MAX_EXTRA: int = _env_int("REFINE_MAX_EXTRA", 256)
    REFINE_BSIZE: int = _env_int("REFINE_BSIZE", 64)
    REFINE_RECHECK_EVERY: int = _env_int("REFINE_RECHECK_EVERY", 32)

    # Quant (for block tensors only)
    QMODE: str = _env_str("QMODE", "none").strip().lower()  # none|float16|int8

    # Eval
    EVAL_TRIALS: int = _env_int("EVAL_TRIALS", 8)
    EVAL_BATCH: int = _env_int("EVAL_BATCH", 2)
    ROUTED_K: int = _env_int("ROUTED_K", 8)

cfg = Cfg()
PRESET = _env_str("PRESET", "").strip().lower()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# ----------------------------
# Presets (override only if user did not set explicitly)
# ----------------------------
def _setdefault_env(k: str, v: str):
    if k not in os.environ:
        os.environ[k] = v

if PRESET == "maxacc":
    _setdefault_env("CALIB_SAMPLES", "32768")
    _setdefault_env("RIDGE_DAMP", "1e-2")          # helps stability on real data
    _setdefault_env("CORE_BLOCK", "32")
    _setdefault_env("CORE_TARGET", "0.995")
    _setdefault_env("CORE_MAX_BLOCKS", "8192")
    _setdefault_env("RES_RANK", "2048")
    _setdefault_env("RES_COEF", "full")
    _setdefault_env("RES_BLOCKS", "4096")
    _setdefault_env("REFINE_ENABLE", "1")
    _setdefault_env("REFINE_ERR_TARGET", "0.01")
    _setdefault_env("REFINE_MAX_EXTRA", "32768")
    _setdefault_env("TRAIN_STEPS", "96")
    _setdefault_env("TRAIN_LR", "0.02")
    _setdefault_env("TRAIN_LAM_GUIDE", "0.5")      # too much guide can hurt exact recon
    _setdefault_env("TRAIN_GUIDE_EVERY", "2")
    cfg = Cfg()

elif PRESET == "balanced":
    _setdefault_env("CALIB_SAMPLES", "8192")
    _setdefault_env("CORE_BLOCK", "32")
    _setdefault_env("CORE_TARGET", "0.97")
    _setdefault_env("CORE_MAX_BLOCKS", "2048")
    _setdefault_env("RES_RANK", "1024")
    _setdefault_env("RES_COEF", "diag")
    _setdefault_env("RES_BLOCKS", "1024")
    _setdefault_env("REFINE_ENABLE", "1")
    _setdefault_env("REFINE_ERR_TARGET", "0.03")
    _setdefault_env("REFINE_MAX_EXTRA", "4096")
    _setdefault_env("TRAIN_STEPS", "64")
    _setdefault_env("TRAIN_LR", "0.03")
    cfg = Cfg()

elif PRESET == "compact":
    _setdefault_env("CALIB_SAMPLES", "4096")
    _setdefault_env("CORE_BLOCK", "64")
    _setdefault_env("CORE_TARGET", "0.90")
    _setdefault_env("CORE_MAX_BLOCKS", "512")
    _setdefault_env("RES_RANK", "512")
    _setdefault_env("RES_COEF", "diag")
    _setdefault_env("RES_BLOCKS", "512")
    _setdefault_env("QMODE", "float16")
    _setdefault_env("REFINE_ENABLE", "0")
    _setdefault_env("TRAIN_STEPS", "24")
    cfg = Cfg()

# ----------------------------
# NPZ helpers
# ----------------------------
def save_npz_compressed(path: str, arrays: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez_compressed(path, **arrays)

def load_npz(path: str) -> Dict[str, np.ndarray]:
    z = np.load(path, allow_pickle=False)
    out = {k: z[k] for k in z.files}
    z.close()
    return out

def _encode_meta(meta: dict) -> np.ndarray:
    b = json.dumps(meta, sort_keys=True).encode("utf-8")
    return np.frombuffer(b, dtype=np.uint8)

def _decode_meta(arr: np.ndarray) -> dict:
    try:
        return json.loads(bytes(arr.tolist()).decode("utf-8"))
    except Exception:
        return {}

# ----------------------------
# Offline shard loading
# ----------------------------
def read_index(model_dir: str) -> Dict[str, str]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    wm = obj.get("weight_map", {})
    if not wm:
        raise RuntimeError("Index JSON has empty weight_map.")
    return wm

def find_layer_expert_ids(weight_map: Dict[str, str], layer: int) -> List[int]:
    pat = re.compile(rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.")
    ids = set()
    for k in weight_map.keys():
        m = pat.match(k)
        if m:
            ids.add(int(m.group(1)))
    return sorted(ids)

def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    prefix = f"model.layers.{layer}.mlp.experts.{eid}."
    def pick(cands: List[str]) -> Optional[str]:
        for suf in cands:
            k = prefix + suf
            if k in weight_map:
                return k
        return None
    up   = pick(["up_proj.weight", "w3.weight", "w1.weight"])
    gate = pick(["gate_proj.weight", "w1.weight", "w3.weight"])
    down = pick(["down_proj.weight", "w2.weight"])
    if up is None or gate is None or down is None:
        return {}
    if up == gate:
        g2 = pick(["gate_proj.weight"])
        if g2:
            gate = g2
    return {"up": up, "gate": gate, "down": down}

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], keys: List[str]) -> Dict[str, torch.Tensor]:
    by_shard: Dict[str, List[str]] = {}
    for k in keys:
        shard = weight_map.get(k, None)
        if shard is None:
            raise KeyError(f"Key not in weight_map: {k}")
        by_shard.setdefault(shard, []).append(k)

    out: Dict[str, torch.Tensor] = {}
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(sp):
            raise FileNotFoundError(f"Missing shard: {sp}")
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks:
                out[k] = f.get_tensor(k)
    return out

# ----------------------------
# Calibration: auto-detect, load, optional capture
# ----------------------------
def autodetect_calib_path() -> Optional[str]:
    cand = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
    return cand if os.path.isfile(cand) else None

def autodetect_router_path() -> Optional[str]:
    cand = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
    return cand if os.path.isfile(cand) else None

def load_calib_X(path: str, H: int) -> torch.Tensor:
    z = np.load(path, allow_pickle=False)
    if "X" not in z.files:
        raise KeyError(f"CALIB npz missing key 'X'. Keys={list(z.files)}")
    Xn = z["X"].astype(np.float32, copy=False)
    z.close()
    X = torch.from_numpy(Xn)
    if X.ndim != 2 or X.shape[1] != H:
        raise RuntimeError(f"Bad X shape {tuple(X.shape)}, expected (*,{H})")
    if X.shape[0] > cfg.CALIB_SAMPLES:
        X = X[:cfg.CALIB_SAMPLES]
    return X.to(device=DEVICE, dtype=DTYPE_ACC)

def load_router_P(path: str) -> np.ndarray:
    z = np.load(path, allow_pickle=False)
    if "P" not in z.files:
        raise KeyError(f"ROUTER npz missing key 'P'. Keys={list(z.files)}")
    P = z["P"].astype(np.float32, copy=False)
    z.close()
    return P

def _maybe_autopip():
    if not cfg.HF_AUTO_PIP:
        return
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", "transformers", "sentencepiece", "tokenizers"])

def _patch_transformers_cache_compat():
    # Some remote-code models require DynamicCache.get_usable_length in some versions.
    try:
        from transformers.cache_utils import DynamicCache  # type: ignore
        if not hasattr(DynamicCache, "get_usable_length"):
            def _get_usable_length(self, seq_length: int):
                return int(seq_length)
            DynamicCache.get_usable_length = _get_usable_length  # type: ignore
            log("[patch] Added DynamicCache.get_usable_length shim.")
    except Exception:
        pass

class _Collector:
    def __init__(self, H: int, E_total: int, max_rows: int):
        self.H = H
        self.E_total = E_total
        self.max_rows = max_rows
        self.X_chunks: List[torch.Tensor] = []
        self.P_chunks: List[torch.Tensor] = []
        self.nX = 0
        self.nP = 0

    def _take_rows(self, flat: torch.Tensor, need: int) -> torch.Tensor:
        if flat.shape[0] > need:
            return flat[:need]
        return flat

    def add_X(self, hs: torch.Tensor, attn_mask: Optional[torch.Tensor]):
        # hs: (B,T,H)
        if hs is None:
            return
        if hs.ndim == 2:
            hs = hs.unsqueeze(0)
        if hs.ndim != 3 or hs.shape[-1] != self.H:
            return

        hs = hs.detach().to(torch.float32).cpu()
        if attn_mask is not None and (not cfg.CAPTURE_KEEP_PAD):
            m = attn_mask.detach().cpu().to(torch.bool)
            flat = hs.reshape(-1, self.H)
            mflat = m.reshape(-1)
            flat = flat[mflat]
        else:
            flat = hs.reshape(-1, self.H)

        if flat.numel() == 0:
            return

        need = self.max_rows - self.nX
        if need <= 0:
            return
        flat = self._take_rows(flat, need)
        self.X_chunks.append(flat)
        self.nX += int(flat.shape[0])

    def add_logits(self, logits: torch.Tensor, attn_mask: Optional[torch.Tensor]):
        # logits: (B,T,E') or (T,E')
        if logits is None:
            return
        if logits.ndim == 2:
            logits = logits.unsqueeze(0)
        if logits.ndim != 3:
            return

        P = torch.softmax(logits.detach().to(torch.float32), dim=-1)
        P = P[..., :self.E_total].cpu()

        if attn_mask is not None and (not cfg.CAPTURE_KEEP_PAD):
            m = attn_mask.detach().cpu().to(torch.bool)
            flat = P.reshape(-1, P.shape[-1])
            mflat = m.reshape(-1)
            flat = flat[mflat]
        else:
            flat = P.reshape(-1, P.shape[-1])

        if flat.numel() == 0:
            return

        need = self.max_rows - self.nP
        if need <= 0:
            return
        flat = self._take_rows(flat, need)
        self.P_chunks.append(flat)
        self.nP += int(flat.shape[0])

def _load_capture_texts() -> List[str]:
    if cfg.CAPTURE_TEXT_FILE and os.path.isfile(cfg.CAPTURE_TEXT_FILE):
        with open(cfg.CAPTURE_TEXT_FILE, "r", encoding="utf-8") as f:
            lines = [ln.strip() for ln in f.readlines()]
        lines = [x for x in lines if x]
        if lines:
            return lines
    return [cfg.CAPTURE_TEXT]

def capture_XP_transformers(model_dir: str, layer_idx: int, H: int, E_total: int,
                            out_x: str, out_p: str) -> Tuple[str, Optional[str]]:
    _maybe_autopip()
    _patch_transformers_cache_compat()
    try:
        from transformers import AutoTokenizer, AutoModelForCausalLM  # type: ignore
    except Exception as e:
        raise RuntimeError("transformers not available; install it or set HF_AUTO_PIP=1.") from e

    log("[capture] Loading tokenizer/model (local_files_only recommended)...")
    tok = AutoTokenizer.from_pretrained(
        model_dir,
        trust_remote_code=cfg.HF_TRUST_REMOTE_CODE,
        local_files_only=cfg.HF_LOCAL_FILES_ONLY,
        use_fast=True,
    )
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token if tok.eos_token is not None else tok.unk_token

    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        trust_remote_code=cfg.HF_TRUST_REMOTE_CODE,
        local_files_only=cfg.HF_LOCAL_FILES_ONLY,
        torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
        device_map=None,
        low_cpu_mem_usage=True,
    )
    model.eval().to(DEVICE)

    # Locate layers
    layers = None
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        layers = list(model.model.layers)
    elif hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        layers = list(model.transformer.h)
    elif hasattr(model, "layers"):
        layers = list(model.layers)
    if layers is None:
        raise RuntimeError("Cannot locate transformer layers list.")
    if layer_idx < 0 or layer_idx >= len(layers):
        raise RuntimeError(f"LAYER={layer_idx} out of range; model has {len(layers)} layers.")
    layer = layers[layer_idx]

    # Find mlp module to hook its input (hidden states X)
    mlp = getattr(layer, "mlp", None)
    if mlp is None:
        for n, m in layer.named_modules():
            if n.lower().endswith("mlp"):
                mlp = m
                break
    if mlp is None:
        raise RuntimeError("Could not find layer.mlp to hook for X capture.")

    # Router discovery (best-effort): look for Linear(H -> >=E_total) with name containing router/gate.
    router_linear: Optional[nn.Linear] = None
    best_score = -1e9
    for name, mod in layer.named_modules():
        if isinstance(mod, nn.Linear) and getattr(mod, "in_features", None) == H and getattr(mod, "out_features", 0) >= E_total:
            nm = name.lower()
            score = 0
            if "router" in nm: score += 10
            if "gate" in nm: score += 6
            if "moe" in nm: score += 3
            if mod.out_features == E_total: score += 6
            score -= 0.01 * float(mod.out_features - E_total)
            if score > best_score:
                best_score = score
                router_linear = mod

    router_weight: Optional[torch.Tensor] = None
    if router_linear is None:
        # try parameters (shape [E, H])
        best = None
        best_score = -1e9
        for pname, p in layer.named_parameters(recurse=True):
            if p.ndim == 2 and p.shape[1] == H and p.shape[0] >= E_total:
                nm = pname.lower()
                score = 0
                if "router" in nm: score += 10
                if "gate" in nm: score += 6
                if p.shape[0] == E_total: score += 6
                score -= 0.01 * float(p.shape[0] - E_total)
                if score > best_score:
                    best_score = score
                    best = p
        if best is not None:
            router_weight = best.detach()

    if router_linear is not None:
        log(f"[capture] Router Linear candidate: in={router_linear.in_features} out={router_linear.out_features}")
    elif router_weight is not None:
        log(f"[capture] Router weight candidate: shape={tuple(router_weight.shape)}")
    else:
        log("[capture] Router not found; will capture X only (no P).")

    coll = _Collector(H=H, E_total=E_total, max_rows=cfg.CALIB_SAMPLES)
    attn_mask_holder = {"mask": None}

    def mlp_pre_hook(_m, inputs):
        hs = inputs[0]
        am = attn_mask_holder["mask"]
        coll.add_X(hs, am)
        # If no router module, compute logits from weight if possible
        if router_linear is None and router_weight is not None and hs is not None:
            hs2 = hs if hs.ndim == 3 else hs.unsqueeze(0)
            W = router_weight.to(hs2.device, dtype=torch.float32)
            logits = torch.matmul(hs2.to(torch.float32), W.t())
            coll.add_logits(logits, am)

    h_mlp = mlp.register_forward_pre_hook(mlp_pre_hook)

    h_router = None
    if router_linear is not None:
        def router_hook(_m, inputs, output):
            out = output[0] if isinstance(output, (tuple, list)) else output
            if torch.is_tensor(out):
                am = attn_mask_holder["mask"]
                out2 = out if out.ndim == 3 else out.unsqueeze(0)
                coll.add_logits(out2, am)
        h_router = router_linear.register_forward_hook(router_hook)

    texts = _load_capture_texts()
    tptr = 0

    for it in range(cfg.CAPTURE_ITERS):
        text = texts[tptr % len(texts)]
        tptr += 1
        enc = tok(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=cfg.CAPTURE_MAX_TOKENS,
            padding="max_length",
        )
        # replicate batch
        for k in list(enc.keys()):
            if enc[k].ndim == 2 and cfg.CAPTURE_BATCH > 1:
                enc[k] = enc[k].repeat(cfg.CAPTURE_BATCH, 1)
        attn_mask_holder["mask"] = enc.get("attention_mask", None)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        with torch.inference_mode():
            _ = model(**enc, use_cache=False)

        if (it + 1) % 4 == 0 or it == 0 or (it + 1) == cfg.CAPTURE_ITERS:
            log(f"[capture] iter {it+1}/{cfg.CAPTURE_ITERS}  nX={coll.nX} nP={coll.nP}")

        if coll.nX >= cfg.CALIB_SAMPLES and (not cfg.RIDGE_WEIGHTED or coll.nP >= cfg.CALIB_SAMPLES):
            break

    h_mlp.remove()
    if h_router is not None:
        h_router.remove()

    if coll.nX == 0:
        raise RuntimeError("Capture failed: collected 0 X rows. Try CAPTURE_MAX_TOKENS↑ and CAPTURE_ITERS↑.")

    X = torch.cat(coll.X_chunks, dim=0)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
    save_npz_compressed(out_x, {"X": X})
    log(f"[capture] wrote X -> {out_x}  shape={tuple(X.shape)}")

    p_written = None
    if coll.nP > 0:
        P = torch.cat(coll.P_chunks, dim=0)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
        N = min(P.shape[0], X.shape[0])
        if N < X.shape[0]:
            X = X[:N]
            save_npz_compressed(out_x, {"X": X})
        P = P[:N]
        save_npz_compressed(out_p, {"P": P})
        log(f"[capture] wrote P -> {out_p}  shape={tuple(P.shape)}")
        p_written = out_p
    else:
        log("[capture] Router P not captured.")

    return out_x, p_written

def ensure_calib_router(H: int, E_total: int):
    # Auto-detect calib/router
    if not cfg.CALIB_PATH:
        cand = autodetect_calib_path()
        if cand:
            cfg.CALIB_PATH = cand
            log(f"[calib] CALIB_PATH not set -> auto-found {cfg.CALIB_PATH}")

    if not cfg.ROUTER_PATH:
        cand = autodetect_router_path()
        if cand:
            cfg.ROUTER_PATH = cand
            log(f"[router] ROUTER_PATH not set -> auto-found {cfg.ROUTER_PATH}")

    need_capture = cfg.CAPTURE_FORCE or (cfg.CAPTURE_ENABLE and (not cfg.CALIB_PATH or not os.path.isfile(cfg.CALIB_PATH)))
    if need_capture:
        out_x = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
        out_p = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
        log("[calib] capturing X (and maybe P) via transformers...")
        x_path, p_path = capture_XP_transformers(cfg.MODEL_DIR, cfg.LAYER, H, E_total, out_x, out_p)
        cfg.CALIB_PATH = x_path
        if p_path:
            cfg.ROUTER_PATH = p_path

# ----------------------------
# Ridge linearization: build Ws
# ----------------------------
@torch.no_grad()
def forward_mlp(X: torch.Tensor, W_gate: torch.Tensor, W_up: torch.Tensor, W_down: torch.Tensor) -> torch.Tensor:
    # X: (N,H); weights are stored as (dff,H) or similar; we use .t() for matmul
    Xf = X.to(DTYPE_ACC)
    up = Xf @ W_up.to(DTYPE_ACC).t()
    gate = Xf @ W_gate.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    y = hid @ W_down.to(DTYPE_ACC).t()
    return y

def ws_cache_path(E: int) -> str:
    return os.path.join(cfg.OUTPUT_DIR, f"Ws_cache_layer{cfg.LAYER}_E{E}_ridge_v10_4.npz")

def ws_meta(eids: List[int]) -> dict:
    return dict(
        script="ktxx_offline_v10_4",
        time=now(),
        model_dir=cfg.MODEL_DIR,
        output_dir=cfg.OUTPUT_DIR,
        layer=cfg.LAYER,
        expert_ids=eids,
        ridge_damp=cfg.RIDGE_DAMP,
        ridge_weighted=cfg.RIDGE_WEIGHTED,
        router_path=cfg.ROUTER_PATH or "",
        calib_path=cfg.CALIB_PATH or "",
        calib_samples=cfg.CALIB_SAMPLES,
        normalize_w=cfg.NORMALIZE_W,
        seed=SEED,
        device=str(DEVICE),
        torch=str(torch.__version__),
    )

@torch.no_grad()
def build_Ws(eids: List[int], wm: Dict[str, str]) -> Tuple[torch.Tensor, torch.Tensor]:
    # Load needed tensors
    per_e = {}
    need_keys = []
    for eid in eids:
        kk = pick_expert_tensor_keys(wm, cfg.LAYER, eid)
        if not kk:
            raise RuntimeError(f"Expert {eid} missing required tensors in index.")
        per_e[eid] = kk
        need_keys += [kk["up"], kk["down"], kk["gate"]]

    log("[load] reading tensors from shards ...")
    T = load_tensors_from_shards(cfg.MODEL_DIR, wm, sorted(set(need_keys)))

    W_up0 = T[per_e[eids[0]]["up"]]
    dff, H = int(W_up0.shape[0]), int(W_up0.shape[1])
    log(f"[shape] H={H} d_ff={dff}")

    ensure_calib_router(H=H, E_total=len(find_layer_expert_ids(wm, cfg.LAYER)))

    if not cfg.CALIB_PATH or not os.path.isfile(cfg.CALIB_PATH):
        raise RuntimeError("CALIB_PATH missing. Set CALIB_PATH or CAPTURE_ENABLE=1.")
    X = load_calib_X(cfg.CALIB_PATH, H)
    log(f"[calib] X loaded: {tuple(X.shape)}  (path={cfg.CALIB_PATH})")

    P = None
    if cfg.RIDGE_WEIGHTED:
        if cfg.ROUTER_PATH and os.path.isfile(cfg.ROUTER_PATH):
            P = load_router_P(cfg.ROUTER_PATH)
            log(f"[router] P loaded: {tuple(P.shape)}  (path={cfg.ROUTER_PATH})")
        else:
            log("[router] RIDGE_WEIGHTED=1 but ROUTER_PATH missing -> forcing RIDGE_WEIGHTED=0")
            cfg.RIDGE_WEIGHTED = False

    # Ridge precompute (unweighted)
    Xf = X.to(DTYPE_ACC)
    Hn = H
    I = torch.eye(Hn, dtype=DTYPE_ACC, device=DEVICE)
    XtX = Xf.t() @ Xf
    lam = cfg.RIDGE_DAMP * float(torch.trace(XtX).item()) / float(Hn)
    cholG = torch.linalg.cholesky(XtX + lam * I)

    Ws_list: List[torch.Tensor] = []
    scales: List[float] = []

    for i, eid in enumerate(tqdm(eids, desc="Build Ws (ridge)")):
        W_up = T[per_e[eid]["up"]].to(DEVICE)
        W_dn = T[per_e[eid]["down"]].to(DEVICE)
        W_gt = T[per_e[eid]["gate"]].to(DEVICE)

        Y = forward_mlp(X, W_gt, W_up, W_dn).to(DTYPE_ACC)

        if cfg.RIDGE_WEIGHTED and (P is not None):
            # weights per sample
            if cfg.ROUTER_EIDS_ARE_GLOBAL:
                if eid >= P.shape[1]:
                    raise RuntimeError(f"P shape {P.shape} cannot index eid={eid}")
                w = torch.from_numpy(P[:X.shape[0], eid]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
            else:
                if i >= P.shape[1]:
                    raise RuntimeError(f"P shape {P.shape} cannot index i={i}")
                w = torch.from_numpy(P[:X.shape[0], i]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
            sw = torch.sqrt(w + 1e-12).view(-1, 1)
            Xw = Xf * sw
            Yw = Y * sw
            XtX_e = Xw.t() @ Xw
            lam_e = cfg.RIDGE_DAMP * float(torch.trace(XtX_e).item()) / float(Hn)
            chol = torch.linalg.cholesky(XtX_e + lam_e * I)
            XtY = Xw.t() @ Yw
            Wt = torch.cholesky_solve(XtY, chol)
            W = Wt.t().contiguous()
        else:
            XtY = Xf.t() @ Y
            Wt = torch.cholesky_solve(XtY, cholG)
            W = Wt.t().contiguous()

        # Normalize, but KEEP scale for true reconstruction
        if cfg.NORMALIZE_W:
            s = float(torch.linalg.norm(W, ord="fro").clamp_min(1e-12).item())
            W = (W / s).contiguous()
        else:
            s = 1.0
        Ws_list.append(W)
        scales.append(s)

    Ws = torch.stack(Ws_list, dim=0).to(DTYPE_ACC).to(DEVICE)       # (E,H,H) normalized if NORMALIZE_W
    Sc = torch.tensor(scales, dtype=DTYPE_ACC, device=DEVICE)       # (E,)
    return Ws, Sc

def load_or_build_Ws() -> Tuple[List[int], torch.Tensor, torch.Tensor]:
    wm = read_index(cfg.MODEL_DIR)
    all_eids = find_layer_expert_ids(wm, cfg.LAYER)
    if not all_eids:
        raise RuntimeError(f"No experts found at layer={cfg.LAYER}")
    eids = all_eids[:cfg.MAX_EXPERTS]
    log(f"[found] layer={cfg.LAYER} total_experts={len(all_eids)} using={len(eids)} eids={eids}")

    cpath = ws_cache_path(len(eids))
    if os.path.isfile(cpath):
        z = load_npz(cpath)
        if "meta" in z and "Ws" in z and "expert_ids" in z and "scales" in z:
            if _decode_meta(z["meta"]) == ws_meta(eids):
                Ws = torch.from_numpy(z["Ws"]).to(DTYPE_ACC).to(DEVICE)
                Sc = torch.from_numpy(z["scales"]).to(DTYPE_ACC).to(DEVICE)
                log(f"[cache] loaded Ws -> {cpath}  Ws={tuple(Ws.shape)}")
                return [int(x) for x in z["expert_ids"].tolist()], Ws, Sc
        log("[cache] Ws meta mismatch -> rebuilding.")

    Ws, Sc = build_Ws(eids, wm)

    save_npz_compressed(cpath, {
        "meta": _encode_meta(ws_meta(eids)),
        "expert_ids": np.array(eids, dtype=np.int32),
        "Ws": Ws.detach().cpu().numpy().astype(np.float32),
        "scales": Sc.detach().cpu().numpy().astype(np.float32),
    })
    log(f"[cache] wrote Ws -> {cpath}  size={os.path.getsize(cpath)/1e6:.2f} MB")
    return eids, Ws, Sc

# ----------------------------
# KMeans clustering helpers
# ----------------------------
@torch.no_grad()
def random_proj_features(Ws: torch.Tensor, d: int) -> torch.Tensor:
    # Features from row/col energy projected into low dimension.
    E, n, _ = Ws.shape
    g = torch.Generator(device="cpu").manual_seed(SEED + 17)
    R = (torch.randint(0, 2, (n, d), generator=g, dtype=torch.int8) * 2 - 1).to(DTYPE_ACC).to(DEVICE)
    feats = []
    for e in range(E):
        W = Ws[e]
        row = torch.diag(W @ W.t())
        col = torch.diag(W.t() @ W)
        feats.append(torch.cat([row @ R, col @ R], dim=0).unsqueeze(0))
    X = torch.cat(feats, dim=0)
    # standardize
    X = (X - X.mean(dim=0, keepdim=True)) / (X.std(dim=0, keepdim=True) + 1e-6)
    return X

@torch.no_grad()
def kmeans_torch(X: torch.Tensor, k: int, iters: int, restarts: int) -> torch.Tensor:
    best_lab = None
    best_inertia = float("inf")
    g = torch.Generator(device="cpu").manual_seed(SEED + 999)

    def init_centers():
        # kmeans++-ish
        n = X.shape[0]
        centers = []
        idx0 = torch.randint(0, n, (1,), generator=g).item()
        centers.append(X[idx0].clone())
        for _ in range(1, k):
            C = torch.stack(centers, dim=0)
            dist2 = torch.cdist(X, C).pow(2).min(dim=1).values
            prob = dist2 / dist2.sum().clamp_min(1e-12)
            idx = torch.multinomial(prob, num_samples=1, generator=g).item()
            centers.append(X[idx].clone())
        return torch.stack(centers, dim=0)

    for _ in range(max(1, restarts)):
        C = init_centers()
        for _ in range(iters):
            dist = torch.cdist(X, C)
            lab = dist.argmin(dim=1)
            for j in range(k):
                m = (lab == j)
                if m.any():
                    C[j] = X[m].mean(dim=0)
                else:
                    # re-seed empty center to farthest point
                    far = dist.min(dim=1).values.argmax().item()
                    C[j] = X[far].clone()
        dist = torch.cdist(X, C)
        inertia = float((dist.min(dim=1).values ** 2).sum().item())
        if inertia < best_inertia:
            best_inertia = inertia
            best_lab = lab.clone()
    assert best_lab is not None
    return best_lab.to(torch.int64)

@torch.no_grad()
def relabel_contiguous(labels: torch.Tensor) -> torch.Tensor:
    labels = labels.to(torch.int64)
    uniq = torch.unique(labels)
    out = labels.clone()
    for new, old in enumerate(uniq.tolist()):
        out[labels == int(old)] = int(new)
    return out

@torch.no_grad()
def merge_small_clusters(X: torch.Tensor, labels: torch.Tensor, min_size: int) -> torch.Tensor:
    labels = relabel_contiguous(labels)
    if min_size <= 1:
        return labels
    while True:
        K = int(labels.max().item()) + 1
        counts = torch.bincount(labels, minlength=K)
        small = (counts < min_size).nonzero(as_tuple=False).flatten()
        if small.numel() == 0:
            break
        C = torch.stack([X[labels == k].mean(dim=0) for k in range(K)], dim=0)
        for c in small.tolist():
            idxs = (labels == int(c)).nonzero(as_tuple=False).flatten()
            if idxs.numel() == 0:
                continue
            dist = torch.cdist(C[int(c)].unsqueeze(0), C).squeeze(0)
            dist[int(c)] = 1e9
            tgt = int(dist.argmin().item())
            labels[idxs] = tgt
        labels = relabel_contiguous(labels)
    return labels

@torch.no_grad()
def hierarchical_split(X: torch.Tensor, labels: torch.Tensor, max_size: int, max_k: int, split_iters: int) -> torch.Tensor:
    labels = relabel_contiguous(labels)
    if max_size <= 0:
        return labels
    while True:
        K = int(labels.max().item()) + 1
        if K >= max_k:
            break
        counts = torch.bincount(labels, minlength=K)
        biggest = int(torch.argmax(counts).item())
        bigsz = int(counts[biggest].item())
        if bigsz <= max_size:
            break
        idxs = (labels == biggest).nonzero(as_tuple=False).flatten()
        if idxs.numel() < 2:
            break
        sub = X[idxs]
        sub_lab = kmeans_torch(sub, k=2, iters=split_iters, restarts=1)
        a = idxs[sub_lab == 0]
        b = idxs[sub_lab == 1]
        if a.numel() == 0 or b.numel() == 0:
            break
        labels[b] = K
        labels = relabel_contiguous(labels)
    return labels

# ----------------------------
# Basis training (dense_train) — NO no_grad on loss path
# ----------------------------
class OrthoParam(nn.Module):
    def __init__(self, n: int, init: str = "identity"):
        super().__init__()
        if init == "random":
            g = torch.Generator(device="cpu").manual_seed(SEED + 333)
            M = torch.randn(n, n, generator=g, dtype=DTYPE_ACC)
        else:
            M = torch.eye(n, dtype=DTYPE_ACC)
        self.M = nn.Parameter(M.to(DEVICE))

    def orthogonal(self) -> torch.Tensor:
        Q, _ = torch.linalg.qr(self.M)
        return Q

def schedule(step: int, warmup: int, total: int) -> float:
    if total <= 0:
        return 1.0
    if step <= warmup:
        return 0.0
    return float(min(1.0, max(0.0, (step - warmup) / max(1, (total - warmup)))))

def slice_X_batch(Ws_batch: torch.Tensor, U: torch.Tensor, V: torch.Tensor, S: torch.Tensor) -> torch.Tensor:
    """
    Differentiable slice for training objective.
    Ws_batch: (Eb, n, n)
    U, V: (n, n)
    S: indices (s,)
    Returns: Xs = U_S^T Ws V_S  (Eb, s, s)
    """
    # Use matmul slices for speed and grad friendliness
    U_S = U[:, S]      # (n, s)
    V_S = V[:, S]      # (n, s)
    T = Ws_batch @ V_S                    # (Eb, n, s)
    Xs = torch.matmul(U_S.t().unsqueeze(0), T)  # (Eb, s, s)
    return Xs

def offdiag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    Xoff = Xs - torch.diag_embed(D)
    return Xoff.abs().mean()

def diag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    return D.abs().mean()

def block_group_sparsity_penalty(Xs: torch.Tensor, block: int) -> torch.Tensor:
    Eb, s, _ = Xs.shape
    b = int(block)
    if b <= 0:
        return torch.zeros((), dtype=DTYPE_ACC, device=Xs.device)
    nb = s // b
    if nb <= 0:
        return torch.zeros((), dtype=DTYPE_ACC, device=Xs.device)
    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()
    Xb = X.view(Eb, nb, b, nb, b).permute(0, 1, 3, 2, 4).contiguous()
    Eblk = (Xb * Xb).sum(dim=(3, 4))     # (Eb, nb, nb)
    P = Eblk.mean(dim=0)
    return torch.sqrt(P + 1e-12).sum() / (P.sum() + 1e-12)

@torch.no_grad()
def make_guidance_mask_from_Xs(Xs: torch.Tensor, block: int, target: float, max_blocks: int) -> Tuple[torch.Tensor, float, int]:
    Eb, s, _ = Xs.shape
    b = int(block)
    if b <= 0:
        return torch.ones(s, s, dtype=DTYPE_ACC, device=Xs.device), 1.0, 0
    nb = s // b
    if nb <= 0:
        return torch.ones(s, s, dtype=DTYPE_ACC, device=Xs.device), 1.0, 0
    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()
    Xb = X.view(Eb, nb, b, nb, b).permute(0, 1, 3, 2, 4).contiguous()
    Eg = (Xb * Xb).sum(dim=(3, 4)).mean(dim=0)  # (nb, nb)
    tot = float((X * X).sum().item()) / max(1, Eb)
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    csum = torch.cumsum(flat[order], dim=0)
    frac = csum / max(tot, 1e-12)
    need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else flat.numel())
    K = min(need, max_blocks, flat.numel())
    pick = order[:K]
    mask = torch.zeros(s2, s2, dtype=DTYPE_ACC, device=Xs.device)
    for idx in pick.tolist():
        bi = idx // nb
        bj = idx % nb
        i0 = bi * b
        j0 = bj * b
        mask[i0:i0 + b, j0:j0 + b] = 1.0
    if s2 < s:
        full = torch.zeros(s, s, dtype=DTYPE_ACC, device=Xs.device)
        full[:s2, :s2] = mask
        mask = full
    ef = float(frac[K - 1].item()) if K > 0 else 0.0
    return mask, ef, int(K)

# ----------------------------
# Block energy + selection
# ----------------------------
@torch.no_grad()
def block_energy_grid(X: torch.Tensor, b: int) -> Tuple[torch.Tensor, float, int]:
    n = X.shape[0]
    nb = (n + b - 1) // b
    if (n % b) != 0:
        Xp = torch.zeros(nb * b, nb * b, dtype=X.dtype, device=X.device)
        Xp[:n, :n] = X
        X = Xp
    Xb = X.view(nb, b, nb, b).permute(0, 2, 1, 3).contiguous()
    Eg = (Xb * Xb).sum(dim=(2, 3))  # (nb, nb)
    tot = float((X * X).sum().item())
    return Eg, tot, nb

@torch.no_grad()
def pick_top_blocks(Eg: torch.Tensor, tot_energy: float, b: int, target: float, max_blocks: int) -> Tuple[List[Tuple[int, int]], float]:
    nb = Eg.shape[0]
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    csum = torch.cumsum(flat[order], dim=0)
    frac = csum / max(tot_energy, 1e-12)
    need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else flat.numel())
    K = min(need, max_blocks, flat.numel())
    pick = order[:K].tolist()
    blocks = [(p // nb, p % nb) for p in pick]
    eff = float(frac[K - 1].item()) if K > 0 else 0.0
    return blocks, eff

@torch.no_grad()
def choose_core_blocks(X_list: List[torch.Tensor]) -> Dict[str, Any]:
    mode = cfg.CORE_MODE
    agg = cfg.CORE_AGG
    b = int(cfg.CORE_BLOCK)

    if mode == "none":
        return {"mode": "none", "shared": None, "per": None, "energy_fracs": []}

    if mode == "blockdiag":
        # keep diagonal blocks only, then choose top diagonal blocks
        Eg_sum = None
        tot = 0.0
        for X in X_list:
            Eg, te, nb = block_energy_grid(X, b)
            tot += te
            diag = torch.diagonal(Eg, 0)
            if Eg_sum is None:
                Eg_sum = torch.zeros_like(Eg)
            Eg_sum += torch.diag(diag)
        Eg_mean = Eg_sum / max(1, len(X_list))
        blocks, eff = pick_top_blocks(Eg_mean, tot / max(1, len(X_list)), b, cfg.CORE_TARGET, cfg.CORE_MAX_BLOCKS)
        return {"mode": "blockdiag", "shared": blocks, "per": None, "energy_fracs": [eff]}

    if mode == "blocktopk":
        Eg_all = []
        tots = []
        for X in X_list:
            Eg, te, nb = block_energy_grid(X, b)
            Eg_all.append(Eg)
            tots.append(te)
        Eg_stack = torch.stack(Eg_all, dim=0)
        Eg = Eg_stack.amax(dim=0) if agg == "max" else Eg_stack.mean(dim=0)
        blocks, eff = pick_top_blocks(Eg, float(np.mean(tots)), b, cfg.CORE_TARGET, cfg.CORE_MAX_BLOCKS)
        return {"mode": "blocktopk", "shared": blocks, "per": None, "energy_fracs": [eff]}

    if mode == "blocktopk_perexpert":
        per = []
        efs = []
        for X in X_list:
            Eg, te, nb = block_energy_grid(X, b)
            blocks, eff = pick_top_blocks(Eg, te, b, cfg.CORE_TARGET, cfg.CORE_MAX_BLOCKS)
            per.append(blocks)
            efs.append(eff)
        return {"mode": "blocktopk_perexpert", "shared": None, "per": per, "energy_fracs": efs}

    raise ValueError("CORE_MODE must be blocktopk_perexpert|blocktopk|blockdiag|none")

@torch.no_grad()
def gather_block(X: torch.Tensor, i0: int, j0: int, b: int) -> torch.Tensor:
    n = X.shape[0]
    i1 = min(n, i0 + b)
    j1 = min(n, j0 + b)
    return X[i0:i1, j0:j1].contiguous()

# ----------------------------
# Low-rank (randomized SVD)
# ----------------------------
@torch.no_grad()
def rand_svd(A: torch.Tensor, r: int, n_iter: int = 2) -> Tuple[torch.Tensor, torch.Tensor]:
    n = A.shape[0]
    r = min(r, n)
    g = torch.Generator(device="cpu").manual_seed(SEED + 777)
    Omega = torch.randn(n, r, generator=g, dtype=DTYPE_ACC, device=A.device)
    Y = A @ Omega
    for _ in range(max(0, n_iter)):
        Y = A @ (A.t() @ Y)
    Q, _ = torch.linalg.qr(Y)
    B = Q.t() @ A
    Uhat, _, Vh = torch.linalg.svd(B, full_matrices=False)
    U = (Q @ Uhat[:, :r]).contiguous()
    V = (Vh.t()[:, :r]).contiguous()
    return U, V

# ----------------------------
# Quant helpers (blocks)
# ----------------------------
@torch.no_grad()
def q_block_int8(B: torch.Tensor) -> Tuple[np.ndarray, np.float16]:
    x = B.detach().cpu().to(torch.float32)
    maxabs = float(x.abs().max().item())
    if maxabs < 1e-12:
        return np.zeros_like(x.numpy(), dtype=np.int8), np.float16(1.0)
    scale = maxabs / 127.0
    q = torch.clamp(torch.round(x / scale), -127, 127).to(torch.int8).cpu().numpy()
    return q, np.float16(scale)

# ----------------------------
# Payload packing (ragged blocks)
# ----------------------------
def pack_blocks_ragged(blocks_per_expert: List[List[Tuple[int, int, torch.Tensor]]], qmode: str) -> Dict[str, np.ndarray]:
    """
    Ragged storage:
      exp_ptr: (E+1,) indices into block arrays
      blk_i0, blk_j0, blk_h, blk_w: (B,)
      blk_ptr: (B+1,) offsets into flattened values
      blk_val (float16) OR blk_q(int8)+blk_scale(float16)
    """
    E = len(blocks_per_expert)
    exp_ptr = [0]
    blk_i0, blk_j0, blk_h, blk_w = [], [], [], []
    blk_ptr = [0]
    vals_f16 = []
    vals_i8 = []
    scales = []

    for e in range(E):
        lst = blocks_per_expert[e]
        for (i0, j0, B) in lst:
            h, w = B.shape
            blk_i0.append(int(i0)); blk_j0.append(int(j0))
            blk_h.append(int(h)); blk_w.append(int(w))
            if qmode == "int8":
                q, sc = q_block_int8(B)
                vals_i8.append(q.reshape(-1))
                scales.append(sc)
                blk_ptr.append(blk_ptr[-1] + q.size)
            else:
                v = B.detach().cpu().to(torch.float16).numpy().reshape(-1)
                vals_f16.append(v)
                blk_ptr.append(blk_ptr[-1] + v.size)
        exp_ptr.append(len(blk_i0))

    out: Dict[str, np.ndarray] = {}
    out["exp_ptr"] = np.array(exp_ptr, dtype=np.int32)
    out["blk_i0"] = np.array(blk_i0, dtype=np.int16)
    out["blk_j0"] = np.array(blk_j0, dtype=np.int16)
    out["blk_h"] = np.array(blk_h, dtype=np.int16)
    out["blk_w"] = np.array(blk_w, dtype=np.int16)
    out["blk_ptr"] = np.array(blk_ptr, dtype=np.int64)

    if qmode == "int8":
        out["blk_q"] = np.concatenate(vals_i8, axis=0).astype(np.int8, copy=False) if vals_i8 else np.zeros((0,), dtype=np.int8)
        out["blk_scale"] = np.array(scales, dtype=np.float16)
    else:
        out["blk_val"] = np.concatenate(vals_f16, axis=0).astype(np.float16, copy=False) if vals_f16 else np.zeros((0,), dtype=np.float16)
    return out

def unpack_blocks_ragged(pack: Dict[str, np.ndarray], qmode: str, device: torch.device) -> List[List[Tuple[int, int, torch.Tensor]]]:
    exp_ptr = pack["exp_ptr"].astype(np.int32)
    blk_i0 = pack["blk_i0"].astype(np.int32)
    blk_j0 = pack["blk_j0"].astype(np.int32)
    blk_h = pack["blk_h"].astype(np.int32)
    blk_w = pack["blk_w"].astype(np.int32)
    blk_ptr = pack["blk_ptr"].astype(np.int64)

    if qmode == "int8":
        blk_q = pack["blk_q"].astype(np.int8)
        blk_scale = pack["blk_scale"].astype(np.float16)
    else:
        blk_val = pack["blk_val"].astype(np.float16)

    E = exp_ptr.shape[0] - 1
    out: List[List[Tuple[int, int, torch.Tensor]]] = []
    for e in range(E):
        b0 = int(exp_ptr[e])
        b1 = int(exp_ptr[e + 1])
        lst = []
        for bi in range(b0, b1):
            i0 = int(blk_i0[bi]); j0 = int(blk_j0[bi])
            h = int(blk_h[bi]); w = int(blk_w[bi])
            v0 = int(blk_ptr[bi]); v1 = int(blk_ptr[bi + 1])
            if qmode == "int8":
                q = blk_q[v0:v1].astype(np.int8, copy=False).astype(np.float32)
                sc = float(blk_scale[bi])
                B = torch.from_numpy((q * sc).reshape(h, w)).to(device=device, dtype=DTYPE_ACC)
            else:
                v = blk_val[v0:v1].astype(np.float16, copy=False).astype(np.float32)
                B = torch.from_numpy(v.reshape(h, w)).to(device=device, dtype=DTYPE_ACC)
            lst.append((i0, j0, B))
        out.append(lst)
    return out

# ----------------------------
# Runtime payload object
# ----------------------------
class PayloadRuntime:
    def __init__(self):
        self.meta: dict = {}
        self.expert_ids: List[int] = []
        self.scales: Optional[torch.Tensor] = None    # (E,) scale to undo normalization
        self.cluster_of_pos: Optional[torch.Tensor] = None  # (E,) -> cluster id
        self.cluster_offsets: Optional[torch.Tensor] = None
        self.cluster_concat: Optional[torch.Tensor] = None

        # Basis per cluster
        self.U: List[torch.Tensor] = []
        self.V: List[torch.Tensor] = []
        self.DL: List[torch.Tensor] = []
        self.DR: List[torch.Tensor] = []

        # Coefs per expert
        self.gam: Optional[torch.Tensor] = None       # (E,r) if diag
        self.Cfull: Optional[torch.Tensor] = None     # (E,r,r) if full

        # Blocks
        self.core_blocks: List[List[Tuple[int, int, torch.Tensor]]] = []
        self.res_blocks: List[List[Tuple[int, int, torch.Tensor]]] = []

        self.qmode: str = "none"
        self.res_coef: str = "diag"

    @torch.no_grad()
    def apply_expert(self, x: torch.Tensor, pos: int) -> torch.Tensor:
        """
        Apply W_hat for expert position `pos` (0..E-1) to batch x: y = x @ W_hat
        """
        c = int(self.cluster_of_pos[pos].item())  # type: ignore[index]
        U = self.U[c]
        V = self.V[c]
        DL = self.DL[c]
        DR = self.DR[c]

        z = x @ U
        u = torch.zeros_like(z)

        # core blocks
        for (i0, j0, B) in self.core_blocks[pos]:
            h, w = B.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ B

        # low-rank residual
        if self.res_coef == "diag":
            g = self.gam[pos]  # type: ignore[index]
            u += ((z @ DL) * g.view(1, -1)) @ DR.t()
        else:
            C = self.Cfull[pos]  # type: ignore[index]
            u += (z @ DL) @ C @ DR.t()

        # sparse residual blocks
        for (i0, j0, B) in self.res_blocks[pos]:
            h, w = B.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ B

        y = u @ V.t()
        # undo normalization
        if self.scales is not None:
            y = y * self.scales[pos]
        return y

    @torch.no_grad()
    def apply_mixture(self, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
        y = torch.zeros_like(x)
        for a, pos in zip(gates.tolist(), routed):
            y += float(a) * self.apply_expert(x, int(pos))
        return y

def load_payload_runtime(path: str, device: torch.device) -> PayloadRuntime:
    z = load_npz(path)
    rt = PayloadRuntime()
    rt.meta = _decode_meta(z["meta"]) if "meta" in z else {}
    rt.qmode = rt.meta.get("qmode", "none")
    rt.res_coef = rt.meta.get("res_coef", "diag")

    rt.expert_ids = [int(x) for x in z["expert_ids"].tolist()]
    rt.scales = torch.from_numpy(z["scales"]).to(device=device, dtype=DTYPE_ACC)
    rt.cluster_of_pos = torch.from_numpy(z["cluster_of_pos"]).to(device=device, dtype=torch.int64)

    M = int(z["n_clusters"][0])
    store_dtype = torch.float16 if rt.meta.get("basis_store_dtype", "float16") == "float16" else torch.float32

    # Load basis + lowrank per cluster
    for m in range(M):
        U = torch.from_numpy(z[f"U_{m}"]).to(device=device, dtype=DTYPE_ACC)
        V = torch.from_numpy(z[f"V_{m}"]).to(device=device, dtype=DTYPE_ACC)
        DL = torch.from_numpy(z[f"DL_{m}"]).to(device=device, dtype=DTYPE_ACC)
        DR = torch.from_numpy(z[f"DR_{m}"]).to(device=device, dtype=DTYPE_ACC)
        rt.U.append(U); rt.V.append(V); rt.DL.append(DL); rt.DR.append(DR)

    # Coefs
    if rt.res_coef == "diag":
        rt.gam = torch.from_numpy(z["gam"]).to(device=device, dtype=DTYPE_ACC)
    else:
        # stored as (E,r,r) float16/float32
        rt.Cfull = torch.from_numpy(z["Cfull"]).to(device=device, dtype=DTYPE_ACC)

    # Blocks
    core_pack = {k.replace("core_", ""): z[k] for k in z.keys() if k.startswith("core_")}
    res_pack = {k.replace("res_", ""): z[k] for k in z.keys() if k.startswith("res_")}
    rt.core_blocks = unpack_blocks_ragged(core_pack, rt.qmode, device)
    rt.res_blocks = unpack_blocks_ragged(res_pack, rt.qmode, device)

    return rt

# ----------------------------
# Build payload (dense_train / identity)
# ----------------------------
@torch.no_grad()
def frob_rel_err(A: torch.Tensor, B: torch.Tensor) -> float:
    # ||A-B||_F / ||B||_F
    num = torch.linalg.norm(A - B, ord="fro")
    den = torch.linalg.norm(B, ord="fro").clamp_min(1e-12)
    return float((num / den).item())

@torch.no_grad()
def build_payload_for_cluster(
    Ws: torch.Tensor,
    Sc: torch.Tensor,
    idx_pos: List[int],
    U: torch.Tensor,
    V: torch.Tensor,
) -> Tuple[
    List[List[Tuple[int,int,torch.Tensor]]],     # core blocks per expert
    torch.Tensor, torch.Tensor,                 # DL, DR
    List[torch.Tensor],                         # coef per expert (diag vector or full matrix)
    List[List[Tuple[int,int,torch.Tensor]]],    # res blocks per expert
    List[float],                                # core energy fractions
]:
    # Work in true W space (undo normalization inside basis space by scaling X_e)
    # X_e = U^T (Sc[e]*Ws[e]) V
    X_list = []
    for pos in idx_pos:
        Wtrue = Ws[pos] * Sc[pos]
        X_list.append((U.t() @ Wtrue @ V).contiguous())

    core = choose_core_blocks(X_list)
    b = int(cfg.CORE_BLOCK)

    core_blocks_per: List[List[Tuple[int,int,torch.Tensor]]] = []
    if core["per"] is not None:
        for X, blocks in zip(X_list, core["per"]):
            lst = []
            for (bi, bj) in blocks:
                i0, j0 = bi*b, bj*b
                B = gather_block(X, i0, j0, b)
                lst.append((i0, j0, B))
            core_blocks_per.append(lst)
    else:
        shared = core["shared"] or []
        for X in X_list:
            lst = []
            for (bi, bj) in shared:
                i0, j0 = bi*b, bj*b
                B = gather_block(X, i0, j0, b)
                lst.append((i0, j0, B))
            core_blocks_per.append(lst)

    # Residual after core
    R_list = []
    for X, cb in zip(X_list, core_blocks_per):
        Xc = torch.zeros_like(X)
        for (i0, j0, Bc) in cb:
            h, w = Bc.shape
            Xc[i0:i0+h, j0:j0+w] = Bc
        R_list.append((X - Xc).contiguous())

    # Shared low-rank basis from mean residual
    Rmean = torch.stack(R_list, dim=0).mean(dim=0)
    r = min(cfg.RES_RANK, Rmean.shape[0])
    DL, DR = rand_svd(Rmean, r=r, n_iter=2)

    # Coefs + residual blocks
    coef_list: List[torch.Tensor] = []
    res_blocks_per: List[List[Tuple[int,int,torch.Tensor]]] = []

    for Rm in R_list:
        if cfg.RES_COEF == "diag":
            g = torch.sum(DL * (Rm @ DR), dim=0).contiguous()      # (r,)
            coef_list.append(g)
            R2 = (Rm - (DL * g.view(1, -1)) @ DR.t()).contiguous()
        else:
            C = (DL.t() @ Rm @ DR).contiguous()                    # (r,r)
            coef_list.append(C)
            R2 = (Rm - (DL @ C @ DR.t())).contiguous()

        # pick top residual blocks
        bb = int(cfg.RES_BSIZE)
        Eg, te, nb = block_energy_grid(R2, bb)
        flat = Eg.reshape(-1)
        order = torch.argsort(flat, descending=True)[:min(cfg.RES_BLOCKS, flat.numel())].tolist()
        blocks = []
        for idx in order:
            bi = idx // nb
            bj = idx % nb
            i0 = bi * bb
            j0 = bj * bb
            B = gather_block(R2, i0, j0, bb)
            blocks.append((i0, j0, B))
        res_blocks_per.append(blocks)

    return core_blocks_per, DL.contiguous(), DR.contiguous(), coef_list, res_blocks_per, core["energy_fracs"]

@torch.no_grad()
def refine_expert_until(
    X: torch.Tensor,
    core_blocks: List[Tuple[int,int,torch.Tensor]],
    DL: torch.Tensor,
    DR: torch.Tensor,
    coef: torch.Tensor,
    res_blocks: List[Tuple[int,int,torch.Tensor]],
) -> List[Tuple[int,int,torch.Tensor]]:
    """
    Refine residual blocks until basis-space Frobenius rel-error <= target or max extra reached.
    """
    n = X.shape[0]

    def reconstruct() -> torch.Tensor:
        Xc = torch.zeros_like(X)
        for (i0, j0, Bc) in core_blocks:
            h, w = Bc.shape
            Xc[i0:i0+h, j0:j0+w] = Bc
        if cfg.RES_COEF == "diag":
            g = coef
            Xlr = (DL * g.view(1, -1)) @ DR.t()
        else:
            C = coef
            Xlr = DL @ C @ DR.t()
        Xr = torch.zeros_like(X)
        for (i0, j0, Bb) in res_blocks:
            h, w = Bb.shape
            Xr[i0:i0+h, j0:j0+w] += Bb
        return (Xc + Xlr + Xr).contiguous()

    Xhat = reconstruct()
    err = frob_rel_err(Xhat, X)
    if err <= cfg.REFINE_ERR_TARGET:
        return res_blocks

    # residual to explain with extra blocks
    R = (X - Xhat).contiguous()
    added = 0
    bb = int(cfg.REFINE_BSIZE)

    # Greedy add highest-energy block repeatedly
    while err > cfg.REFINE_ERR_TARGET and added < cfg.REFINE_MAX_EXTRA:
        Eg, te, nb = block_energy_grid(R, bb)
        flat = Eg.reshape(-1)
        mx = float(flat.max().item())
        if mx <= 1e-18:
            break
        idx = int(torch.argmax(flat).item())
        bi = idx // nb
        bj = idx % nb
        i0 = bi * bb
        j0 = bj * bb
        Bb = gather_block(R, i0, j0, bb)
        res_blocks.append((i0, j0, Bb))
        # remove it from residual
        h, w = Bb.shape
        R[i0:i0+h, j0:j0+w] -= Bb
        added += 1

        if (added % cfg.REFINE_RECHECK_EVERY) == 0:
            Xhat = reconstruct()
            err = frob_rel_err(Xhat, X)

    # final check
    if added > 0:
        Xhat = reconstruct()
        err = frob_rel_err(Xhat, X)
    return res_blocks

# ----------------------------
# Evaluation
# ----------------------------
@torch.no_grad()
def eval_payload(rt: PayloadRuntime, Ws: torch.Tensor, Sc: torch.Tensor):
    E, n, _ = Ws.shape

    # per-expert (operator) rel-error measured by random probes
    errs = []
    for pos in range(E):
        x = torch.randn(8, n, dtype=DTYPE_ACC, device=DEVICE)
        y_hat = rt.apply_expert(x, pos)
        y_ref = x @ (Ws[pos] * Sc[pos])
        num = torch.linalg.norm(y_hat - y_ref)
        den = torch.linalg.norm(y_ref).clamp_min(1e-12)
        errs.append(float((num / den).item()))
    log(f"[eval] per-expert rel-error  mean={float(np.mean(errs)):.6f}  p95={float(np.percentile(errs,95)):.6f}  max={float(np.max(errs)):.6f}")

    # routed mixture
    mix = []
    for _ in range(cfg.EVAL_TRIALS):
        x = torch.randn(cfg.EVAL_BATCH, n, dtype=DTYPE_ACC, device=DEVICE)
        routed = random.sample(range(E), k=min(cfg.ROUTED_K, E))
        gates = torch.rand(len(routed), dtype=DTYPE_ACC, device=DEVICE)
        gates = gates / gates.sum().clamp_min(1e-12)
        y_hat = rt.apply_mixture(x, routed, gates)

        Wsum = torch.zeros(n, n, dtype=DTYPE_ACC, device=DEVICE)
        for a, pos in zip(gates, routed):
            Wsum += float(a.item()) * (Ws[int(pos)] * Sc[int(pos)])
        y_ref = x @ Wsum

        num = torch.linalg.norm(y_hat - y_ref)
        den = torch.linalg.norm(y_ref).clamp_min(1e-12)
        mix.append(float((num / den).item()))
    log(f"[eval] routed rel-error = {float(np.mean(mix)):.6f} ± {float(np.std(mix)):.6f}  (trials={cfg.EVAL_TRIALS})")

# ----------------------------
# Main
# ----------------------------
def banner():
    log(f"== DeepSeek KT++-X OFFLINE v10.4 ==")
    log(f"Time:        {now()}")
    log(f"MODEL_DIR:   {cfg.MODEL_DIR}")
    log(f"OUTPUT_DIR:  {cfg.OUTPUT_DIR}")
    log(f"LAYER:       {cfg.LAYER}")
    log(f"MAX_EXPERTS:  {cfg.MAX_EXPERTS}")
    log(f"CALIB_PATH:  {cfg.CALIB_PATH or '(none)'}  CALIB_SAMPLES(cap)={cfg.CALIB_SAMPLES}")
    log(f"ROUTER_PATH: {cfg.ROUTER_PATH or '(none)'}  RIDGE_WEIGHTED={cfg.RIDGE_WEIGHTED}")
    log(f"RIDGE_DAMP:  {cfg.RIDGE_DAMP}  NORMALIZE_W={cfg.NORMALIZE_W}")
    log(f"BASIS_MODE:  {cfg.BASIS_MODE}")
    log(f"CLUSTER:     M0={(cfg.M0 if cfg.M0>0 else '(auto)')} M_MAX={cfg.M_MAX} min_size={cfg.CLUSTER_MIN_SIZE} max_size={cfg.CLUSTER_MAX_SIZE}")
    log(f"TRAIN:       steps={cfg.TRAIN_STEPS} warmup={cfg.TRAIN_WARMUP} lr={cfg.TRAIN_LR} subm={cfg.SUBM} batchE={cfg.BATCH_E}")
    log(f"CORE:        {cfg.CORE_MODE} block={cfg.CORE_BLOCK} target={cfg.CORE_TARGET} max_blocks={cfg.CORE_MAX_BLOCKS}")
    log(f"RESIDUAL:    rank={cfg.RES_RANK} coef={cfg.RES_COEF} blocks={cfg.RES_BLOCKS} bsize={cfg.RES_BSIZE}")
    log(f"REFINE:      enable={cfg.REFINE_ENABLE} err_target={cfg.REFINE_ERR_TARGET} max_extra={cfg.REFINE_MAX_EXTRA}")
    log(f"QUANT:       QMODE={cfg.QMODE}  BASIS_STORE_DTYPE={cfg.BASIS_STORE_DTYPE}")
    log(f"DEVICE:      {DEVICE}  Torch={torch.__version__} threads={NTHREADS}")
    log("")

def main():
    banner()

    # 1) Ws
    expert_ids, Ws, Sc = load_or_build_Ws()
    E, n, _ = Ws.shape
    log(f"[Ws] shape={tuple(Ws.shape)}  (normalized={cfg.NORMALIZE_W})")

    # 2) Clusters
    if cfg.BASIS_MODE not in ("dense_train", "identity"):
        raise RuntimeError("v10.4 implements BASIS_MODE=dense_train or identity. (hadamard_perm removed for clarity)")

    if cfg.BASIS_MODE == "identity":
        clusters = [list(range(E))]
        cluster_of_pos = [0] * E
        log("[cluster] identity basis: M=1")
    else:
        Xfeat = random_proj_features(Ws, d=cfg.CLUSTER_FEAT_D)
        if cfg.M0 > 0:
            M0 = min(cfg.M0, E)
        else:
            M0 = int(round(2.0 * math.sqrt(E)))
            M0 = max(2, min(M0, E))
        labels = kmeans_torch(Xfeat, k=M0, iters=cfg.CLUSTER_ITERS, restarts=cfg.CLUSTER_RESTARTS)
        labels = merge_small_clusters(Xfeat, labels, min_size=cfg.CLUSTER_MIN_SIZE)
        labels = hierarchical_split(Xfeat, labels, max_size=cfg.CLUSTER_MAX_SIZE, max_k=min(cfg.M_MAX, E), split_iters=cfg.SPLIT_ITERS)
        labels = merge_small_clusters(Xfeat, labels, min_size=cfg.CLUSTER_MIN_SIZE)
        labels = relabel_contiguous(labels)
        M = int(labels.max().item()) + 1
        clusters = [torch.nonzero(labels == m, as_tuple=False).flatten().tolist() for m in range(M)]
        clusters = [c for c in clusters if len(c) > 0]
        log(f"[cluster] M={len(clusters)} sizes={[len(c) for c in clusters]}")
        cluster_of_pos = [0] * E
        for m, idx in enumerate(clusters):
            for pos in idx:
                cluster_of_pos[int(pos)] = m

    # 3) Basis training
    U_list: List[torch.Tensor] = []
    V_list: List[torch.Tensor] = []

    if cfg.BASIS_MODE == "identity":
        U_list = [torch.eye(n, dtype=DTYPE_ACC, device=DEVICE)]
        V_list = [torch.eye(n, dtype=DTYPE_ACC, device=DEVICE)]
    else:
        M = len(clusters)
        U_par = [OrthoParam(n, init="identity") for _ in range(M)]
        V_par = [OrthoParam(n, init="identity") for _ in range(M)]

        if cfg.TRAIN_STEPS > 0:
            params = [p.M for p in U_par] + [p.M for p in V_par]
            opt = torch.optim.Adam(params, lr=cfg.TRAIN_LR)
            guidance_masks: Dict[int, torch.Tensor] = {}
            guidance_stats: Dict[int, Tuple[float, int]] = {}
            t0 = time.perf_counter()

            for step in range(1, cfg.TRAIN_STEPS + 1):
                S = torch.randperm(n, device=DEVICE)[:min(cfg.SUBM, n)]

                # Guidance refresh (no grad)
                if cfg.TRAIN_LAM_GUIDE > 0 and (step == 1 or step % max(1, cfg.TRAIN_GUIDE_EVERY) == 0):
                    with torch.no_grad():
                        guidance_masks.clear()
                        guidance_stats.clear()
                        for m, idx in enumerate(clusters):
                            if len(idx) < cfg.TRAIN_MIN_CLUSTER:
                                continue
                            Uo = U_par[m].orthogonal()
                            Vo = V_par[m].orthogonal()
                            pick = idx
                            if 0 < cfg.BATCH_E < len(idx):
                                pidx = torch.randperm(len(idx), device=DEVICE)[:cfg.BATCH_E].tolist()
                                pick = [idx[i] for i in pidx]
                            Xs_ng = slice_X_batch(Ws[pick] * Sc[pick].view(-1,1,1), Uo, Vo, S).detach()
                            mask, ef, kblk = make_guidance_mask_from_Xs(
                                Xs_ng.to(DTYPE_ACC),
                                block=cfg.CORE_BLOCK,
                                target=cfg.TRAIN_GUIDE_TARGET,
                                max_blocks=cfg.TRAIN_GUIDE_MAX_BLOCKS,
                            )
                            guidance_masks[m] = mask
                            guidance_stats[m] = (ef, kblk)

                lam_ramp = schedule(step, cfg.TRAIN_WARMUP, cfg.TRAIN_STEPS)
                lam_block = cfg.TRAIN_LAM_BLOCK * lam_ramp
                lam_guide = cfg.TRAIN_LAM_GUIDE * lam_ramp

                L_total = None
                n_terms = 0

                for m, idx in enumerate(clusters):
                    if len(idx) < cfg.TRAIN_MIN_CLUSTER:
                        continue

                    Uo = U_par[m].orthogonal()
                    Vo = V_par[m].orthogonal()

                    pick = idx
                    if 0 < cfg.BATCH_E < len(idx):
                        pidx = torch.randperm(len(idx), device=DEVICE)[:cfg.BATCH_E].tolist()
                        pick = [idx[i] for i in pidx]

                    # IMPORTANT: use true W = Ws*Sc, and keep grad
                    Ws_batch = Ws[pick] * Sc[pick].view(-1, 1, 1)
                    Xs = slice_X_batch(Ws_batch, Uo, Vo, S)

                    off = offdiag_abs_mean(Xs)
                    diag = diag_abs_mean(Xs).clamp_min(1e-6)

                    if cfg.TRAIN_OBJ == "ratio":
                        base = off / diag
                    else:
                        base = torch.log(off + 1e-6) - torch.log(diag)

                    if lam_block > 0 and cfg.CORE_MODE.startswith("block"):
                        base = base + lam_block * block_group_sparsity_penalty(Xs, cfg.CORE_BLOCK)

                    if lam_guide > 0 and (m in guidance_masks):
                        Mmask = guidance_masks[m]
                        Etot = (Xs * Xs).mean().clamp_min(1e-12)
                        Eout = ((Xs * (1.0 - Mmask)) ** 2).mean()
                        base = base + lam_guide * (Eout / Etot)

                    L_total = base if (L_total is None) else (L_total + base)
                    n_terms += 1

                if L_total is None:
                    log("[train] skipped (no clusters >= TRAIN_MIN_CLUSTER)")
                    break

                L_total = L_total / max(1, n_terms)
                if not L_total.requires_grad:
                    raise RuntimeError("Loss has no grad: you accidentally detached tensors or used no_grad().")

                opt.zero_grad(set_to_none=True)
                L_total.backward()
                if cfg.GRAD_CLIP > 0:
                    torch.nn.utils.clip_grad_norm_(params, max_norm=cfg.GRAD_CLIP)
                opt.step()

                # Re-orthogonalize to keep bases stable
                if (step % cfg.REORTHO_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                    with torch.no_grad():
                        for p in U_par:
                            p.M.copy_(p.orthogonal())
                        for p in V_par:
                            p.M.copy_(p.orthogonal())

                if step == 1 or (step % cfg.REPORT_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                    t1 = time.perf_counter()
                    if guidance_stats:
                        ef_mean = float(np.mean([v[0] for v in guidance_stats.values()]))
                        kb_mean = float(np.mean([v[1] for v in guidance_stats.values()]))
                        log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} loss={float(L_total.item()):.4f} "
                            f"lam_block={lam_block:.3f} lam_guide={lam_guide:.3f} "
                            f"guide_energy≈{ef_mean:.3f} guide_blocks≈{kb_mean:.1f} (+{t1-t0:.1f}s)")
                    else:
                        log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} loss={float(L_total.item()):.4f} "
                            f"lam_block={lam_block:.3f} lam_guide={lam_guide:.3f} (+{t1-t0:.1f}s)")
                    t0 = t1

        # Freeze U/V
        for m in range(len(clusters)):
            U_list.append(U_par[m].orthogonal().detach().contiguous())
            V_list.append(V_par[m].orthogonal().detach().contiguous())

    # 4) Build payload blocks
    log("[build] payloads ...")
    core_blocks_all: List[List[Tuple[int,int,torch.Tensor]]] = [[] for _ in range(E)]
    res_blocks_all: List[List[Tuple[int,int,torch.Tensor]]] = [[] for _ in range(E)]

    DL_list: List[torch.Tensor] = []
    DR_list: List[torch.Tensor] = []

    # coef storage
    if cfg.RES_COEF == "diag":
        gam = torch.zeros((E, min(cfg.RES_RANK, n)), dtype=DTYPE_ACC, device=DEVICE)
        Cfull = None
    else:
        rmax = min(cfg.RES_RANK, n)
        Cfull = torch.zeros((E, rmax, rmax), dtype=DTYPE_ACC, device=DEVICE)
        gam = None

    core_energy_fracs_per_cluster: List[List[float]] = []

    for m, idx in enumerate(clusters):
        U = U_list[m] if cfg.BASIS_MODE != "identity" else U_list[0]
        V = V_list[m] if cfg.BASIS_MODE != "identity" else V_list[0]

        core_per, DL, DR, coef_list, res_per, efr = build_payload_for_cluster(Ws, Sc, idx, U, V)
        core_energy_fracs_per_cluster.append(efr)

        # refine per expert (basis-space) for accuracy
        if cfg.REFINE_ENABLE:
            for j, pos in enumerate(idx):
                # build true X for refine
                X = (U.t() @ (Ws[pos] * Sc[pos]) @ V).contiguous()
                res_per[j] = refine_expert_until(X, core_per[j], DL, DR, coef_list[j], res_per[j])

        # assign
        for j, pos in enumerate(idx):
            core_blocks_all[pos] = core_per[j]
            res_blocks_all[pos] = res_per[j]
            if cfg.RES_COEF == "diag":
                g = coef_list[j]
                gam[pos, :g.numel()] = g
            else:
                C = coef_list[j]
                r = C.shape[0]
                Cfull[pos, :r, :r] = C

        DL_list.append(DL)
        DR_list.append(DR)

        nbm = float(np.mean([len(core_blocks_all[p]) for p in idx])) if idx else 0.0
        efm = float(np.mean(efr)) if len(efr) else 0.0
        efmin = float(np.min(efr)) if len(efr) else 0.0
        log(f"  - cluster{m}: E={len(idx)} core_blocks(mean)≈{nbm:.1f} core_energy(mean/min)≈{efm:.3f}/{efmin:.3f} r={DL.shape[1]}")

    # 5) Serialize payload
    out_payload = os.path.join(cfg.OUTPUT_DIR, f"ktx_payload_layer{cfg.LAYER}_E{E}_{cfg.BASIS_MODE}_v10_4_q{cfg.QMODE}.npz")

    # Pack blocks
    core_pack = pack_blocks_ragged(core_blocks_all, cfg.QMODE)
    res_pack = pack_blocks_ragged(res_blocks_all, cfg.QMODE)

    # Store basis dtype
    store_dtype = np.float16 if cfg.BASIS_STORE_DTYPE == "float16" else np.float32
    arrays: Dict[str, Any] = {}

    meta = ws_meta(expert_ids)
    meta.update({
        "basis_mode": cfg.BASIS_MODE,
        "qmode": cfg.QMODE,
        "basis_store_dtype": cfg.BASIS_STORE_DTYPE,
        "core_mode": cfg.CORE_MODE,
        "core_block": cfg.CORE_BLOCK,
        "core_target": cfg.CORE_TARGET,
        "core_max_blocks": cfg.CORE_MAX_BLOCKS,
        "res_rank": cfg.RES_RANK,
        "res_coef": cfg.RES_COEF,
        "res_blocks": cfg.RES_BLOCKS,
        "res_bsize": cfg.RES_BSIZE,
        "refine_enable": cfg.REFINE_ENABLE,
        "refine_err_target": cfg.REFINE_ERR_TARGET,
        "refine_max_extra": cfg.REFINE_MAX_EXTRA,
    })

    arrays["meta"] = _encode_meta(meta)
    arrays["expert_ids"] = np.array(expert_ids, dtype=np.int32)
    arrays["scales"] = Sc.detach().cpu().numpy().astype(np.float32)  # undo normalization at runtime
    arrays["cluster_of_pos"] = np.array(cluster_of_pos, dtype=np.int16)
    arrays["n_clusters"] = np.array([len(clusters)], dtype=np.int32)

    # Store U/V/DL/DR per cluster
    for m in range(len(clusters)):
        U = U_list[m] if cfg.BASIS_MODE != "identity" else U_list[0]
        V = V_list[m] if cfg.BASIS_MODE != "identity" else V_list[0]
        arrays[f"U_{m}"] = U.detach().cpu().numpy().astype(store_dtype)
        arrays[f"V_{m}"] = V.detach().cpu().numpy().astype(store_dtype)
        arrays[f"DL_{m}"] = DL_list[m].detach().cpu().numpy().astype(store_dtype)
        arrays[f"DR_{m}"] = DR_list[m].detach().cpu().numpy().astype(store_dtype)

    # Store coefs
    if cfg.RES_COEF == "diag":
        arrays["gam"] = gam.detach().cpu().numpy().astype(store_dtype)
    else:
        arrays["Cfull"] = Cfull.detach().cpu().numpy().astype(store_dtype)

    # Store ragged blocks
    for k, v in core_pack.items():
        arrays["core_" + k] = v
    for k, v in res_pack.items():
        arrays["res_" + k] = v

    save_npz_compressed(out_payload, arrays)
    log(f"[save] payload -> {out_payload}  size={os.path.getsize(out_payload)/1e6:.2f} MB")

    # 6) Load runtime + eval (sanity)
    rt = load_payload_runtime(out_payload, DEVICE)
    eval_payload(rt, Ws, Sc)

    log("✅ Done.")
    log("If you want 99–100% accuracy:")
    log("  - Make sure calib X has THOUSANDS+ rows (CALIB_SAMPLES=16384+ and actually captured).")
    log("  - Raise CORE_TARGET (0.99–0.995), shrink CORE_BLOCK (32/16), increase CORE_MAX_BLOCKS.")
    log("  - Increase RES_RANK and/or switch RES_COEF=full.")
    log("  - Increase RES_BLOCKS and enable REFINE with low REFINE_ERR_TARGET (0.01).")

if __name__ == "__main__":
    main()


== DeepSeek KT++-X OFFLINE v10.4 ==
Time:        2026-01-13 12:16:18
MODEL_DIR:   /home/daniyar/deepseek-model
OUTPUT_DIR:  /home/daniyar/moe_ws_outputs
LAYER:       1
MAX_EXPERTS:  16
CALIB_PATH:  (none)  CALIB_SAMPLES(cap)=4096
ROUTER_PATH: (none)  RIDGE_WEIGHTED=False
RIDGE_DAMP:  0.001  NORMALIZE_W=True
BASIS_MODE:  dense_train
CLUSTER:     M0=(auto) M_MAX=16 min_size=2 max_size=4
TRAIN:       steps=24 warmup=6 lr=0.05 subm=256 batchE=4
CORE:        blocktopk_perexpert block=64 target=0.85 max_blocks=256
RESIDUAL:    rank=512 coef=diag blocks=192 bsize=64
REFINE:      enable=True err_target=0.05 max_extra=256
QUANT:       QMODE=none  BASIS_STORE_DTYPE=float16
DEVICE:      cpu  Torch=2.4.1+cpu threads=8

[found] layer=1 total_experts=64 using=16 eids=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
[load] reading tensors from shards ...
[shape] H=2048 d_ff=1408
[calib] CALIB_PATH not set -> auto-found /home/daniyar/moe_ws_outputs/calib_layer1_X.npz
[calib] X loaded: (160, 2048

Build Ws (ridge):   0%|          | 0/16 [00:00<?, ?it/s]

[cache] wrote Ws -> /home/daniyar/moe_ws_outputs/Ws_cache_layer1_E16_ridge_v10_4.npz  size=250.24 MB
[Ws] shape=(16, 2048, 2048)  (normalized=True)
[cluster] M=4 sizes=[5, 3, 4, 4]
[train] step   1/24 loss=-0.0672 lam_block=0.000 lam_guide=0.000 guide_energy≈0.865 guide_blocks≈10.8 (+4.9s)
[train] step   4/24 loss=-0.8653 lam_block=0.000 lam_guide=0.000 guide_energy≈0.823 guide_blocks≈9.8 (+14.1s)
[train] step   8/24 loss=-0.8761 lam_block=0.011 lam_guide=0.111 guide_energy≈0.822 guide_blocks≈6.0 (+17.0s)
[train] step  12/24 loss=-0.3653 lam_block=0.033 lam_guide=0.333 guide_energy≈0.816 guide_blocks≈10.8 (+17.0s)
[train] step  16/24 loss=-1.5256 lam_block=0.056 lam_guide=0.556 guide_energy≈0.833 guide_blocks≈5.2 (+17.1s)
[train] step  20/24 loss=0.2998 lam_block=0.078 lam_guide=0.778 guide_energy≈0.821 guide_blocks≈9.8 (+16.7s)
[train] step  24/24 loss=1.3998 lam_block=0.100 lam_guide=1.000 guide_energy≈0.817 guide_blocks≈10.5 (+16.8s)
[build] payloads ...
  - cluster0: E=5 core_block

In [20]:
#!/usr/bin/env python3
# ============================================================
# DeepSeek KT++-X OFFLINE v10.5  (single-file GIANT runner)
#
# Fixes vs v10.4:
#   1) SCALE BUG FIX:
#        - Build payload in basis space using NORMALIZED Ws (Wnorm)
#        - Apply per-expert scale Sc ONLY at runtime (y *= Sc[pos])
#        - This fixes huge errors like mean~5 / routed~35.
#
#   2) Stable Ws cache:
#        - ws_meta() no longer includes time() -> cache can be reused.
#
#   3) QMODE=none stores blocks as float32 (not float16).
#
#   4) Better basis initialization:
#        - U/V per cluster initialized from SVD of mean(Ws_cluster).
#
# Pipeline (offline, local shards):
#   (Optional) CAPTURE: collect calibration hidden-states X (and router probs P) using transformers
#   1) Build linearized expert matrices Ws via ridge regression:
#        For each expert e:  Y_e = MLP_e(X)    solve  W_e ≈ argmin ||X W^T - Y||_2^2 (+ ridge)
#        Optionally weighted by router probabilities P[:, eid]
#        Optionally normalize Ws and store scale Sc so W_true = Sc*W_norm
#   2) Cluster experts (optional, for dense_train basis)
#   3) Train orthogonal bases U/V per cluster to promote block sparsity (dense_train)
#   4) Build payload per expert in basis space (using W_norm):
#        X_e = U^T W_norm V
#        - CORE blocks: choose blocks to hit CORE_TARGET energy
#        - LOW-RANK residual: DL/DR shared per cluster; coefficients per expert (diag or full)
#        - RES blocks: sparse residual blocks
#        - REFINE: optionally add extra blocks until basis-space error <= target
#   5) Save payload NPZ + runtime loader + eval
#
# Dependencies:
#   - torch, numpy, safetensors, tqdm
#   - transformers only if CAPTURE_ENABLE=1 (or CAPTURE_FORCE=1)
# ============================================================

import os, re, json, math, time, random, sys
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any, Set

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors import safe_open

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):  # type: ignore
        return x

# ----------------------------
# Robust env parsing
# ----------------------------
def _env_str(k: str, d: str) -> str:
    return os.environ.get(k, d)

def _env_int(k: str, d: int) -> int:
    try:
        return int(os.environ.get(k, str(d)))
    except Exception:
        return d

def _env_float(k: str, d: float) -> float:
    try:
        return float(os.environ.get(k, str(d)))
    except Exception:
        return d

def _env_bool(k: str, d: bool) -> bool:
    v = os.environ.get(k, None)
    if v is None:
        return d
    return v.strip().lower() in ("1", "true", "yes", "y", "on")

# ----------------------------
# Threads / determinism
# ----------------------------
NTHREADS = _env_int("KTXX_THREADS", 8)
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(NTHREADS))
try:
    torch.set_num_threads(NTHREADS)
except Exception:
    pass

SEED = _env_int("SEED", 1234)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(_env_str("DEVICE", "cpu"))

DTYPE_ACC = torch.float32

def now() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")

def log(msg: str):
    print(msg, flush=True)

# ----------------------------
# Config
# ----------------------------
@dataclass
class Cfg:
    # Paths
    MODEL_DIR: str = _env_str("MODEL_DIR", "/home/daniyar/deepseek-model")
    OUTPUT_DIR: str = _env_str("OUTPUT_DIR", "/home/daniyar/moe_ws_outputs")

    # Slice
    LAYER: int = _env_int("LAYER", 1)
    MAX_EXPERTS: int = _env_int("MAX_EXPERTS", 16)

    # Calibration / router
    CALIB_PATH: str = _env_str("CALIB_PATH", "").strip()
    ROUTER_PATH: str = _env_str("ROUTER_PATH", "").strip()
    CALIB_SAMPLES: int = _env_int("CALIB_SAMPLES", 4096)
    RIDGE_WEIGHTED: bool = _env_bool("RIDGE_WEIGHTED", False)
    ROUTER_EIDS_ARE_GLOBAL: bool = _env_bool("ROUTER_EIDS_ARE_GLOBAL", True)

    # Capture (optional)
    CAPTURE_ENABLE: bool = _env_bool("CAPTURE_ENABLE", False)
    CAPTURE_FORCE: bool = _env_bool("CAPTURE_FORCE", False)
    CAPTURE_ITERS: int = _env_int("CAPTURE_ITERS", 32)
    CAPTURE_BATCH: int = _env_int("CAPTURE_BATCH", 1)
    CAPTURE_MAX_TOKENS: int = _env_int("CAPTURE_MAX_TOKENS", 1024)
    CAPTURE_TEXT: str = _env_str(
        "CAPTURE_TEXT",
        ("DeepSeek models use mixture-of-experts layers. "
         "We capture intermediate activations for calibration. " * 256)
    )
    CAPTURE_TEXT_FILE: str = _env_str("CAPTURE_TEXT_FILE", "").strip()
    CAPTURE_KEEP_PAD: bool = _env_bool("CAPTURE_KEEP_PAD", False)

    HF_TRUST_REMOTE_CODE: bool = _env_bool("HF_TRUST_REMOTE_CODE", True)
    HF_LOCAL_FILES_ONLY: bool = _env_bool("HF_LOCAL_FILES_ONLY", True)
    HF_AUTO_PIP: bool = _env_bool("HF_AUTO_PIP", False)

    # Ridge build
    RIDGE_DAMP: float = _env_float("RIDGE_DAMP", 1e-3)
    NORMALIZE_W: bool = _env_bool("NORMALIZE_W", True)  # normalize Ws and store per-expert scales

    # Basis mode
    BASIS_MODE: str = _env_str("BASIS_MODE", "dense_train").strip().lower()  # dense_train|identity
    BASIS_STORE_DTYPE: str = _env_str("BASIS_STORE_DTYPE", "float16").strip().lower()  # float16|float32

    # Clustering (dense_train)
    M0: int = _env_int("M0", 0)  # 0 => auto
    M_MAX: int = _env_int("M_MAX", 16)
    CLUSTER_FEAT_D: int = _env_int("CLUSTER_FEAT_D", 64)
    CLUSTER_ITERS: int = _env_int("CLUSTER_ITERS", 60)
    CLUSTER_RESTARTS: int = _env_int("CLUSTER_RESTARTS", 4)
    CLUSTER_MIN_SIZE: int = _env_int("CLUSTER_MIN_SIZE", 2)
    CLUSTER_MAX_SIZE: int = _env_int("CLUSTER_MAX_SIZE", 4)
    SPLIT_ITERS: int = _env_int("SPLIT_ITERS", 50)

    # Training (dense_train)
    TRAIN_STEPS: int = _env_int("TRAIN_STEPS", 24)
    TRAIN_WARMUP: int = _env_int("TRAIN_WARMUP", 6)
    TRAIN_LR: float = _env_float("TRAIN_LR", 5e-2)
    SUBM: int = _env_int("SUBM", 256)
    BATCH_E: int = _env_int("BATCH_E", 4)
    TRAIN_MIN_CLUSTER: int = _env_int("TRAIN_MIN_CLUSTER", 2)
    REORTHO_EVERY: int = _env_int("REORTHO_EVERY", 4)
    REPORT_EVERY: int = _env_int("REPORT_EVERY", 4)
    GRAD_CLIP: float = _env_float("GRAD_CLIP", 1.0)

    TRAIN_OBJ: str = _env_str("TRAIN_OBJ", "logratio").strip().lower()  # logratio|ratio
    TRAIN_LAM_BLOCK: float = _env_float("TRAIN_LAM_BLOCK", 0.10)
    TRAIN_LAM_GUIDE: float = _env_float("TRAIN_LAM_GUIDE", 1.0)
    TRAIN_GUIDE_EVERY: int = _env_int("TRAIN_GUIDE_EVERY", 2)
    TRAIN_GUIDE_TARGET: float = _env_float("TRAIN_GUIDE_TARGET", 0.80)
    TRAIN_GUIDE_MAX_BLOCKS: int = _env_int("TRAIN_GUIDE_MAX_BLOCKS", 2048)

    # Core selection
    CORE_MODE: str = _env_str("CORE_MODE", "blocktopk_perexpert").strip().lower()
    CORE_AGG: str = _env_str("CORE_AGG", "mean").strip().lower()  # mean|max
    CORE_BLOCK: int = _env_int("CORE_BLOCK", 64)
    CORE_TARGET: float = _env_float("CORE_TARGET", 0.85)
    CORE_MAX_BLOCKS: int = _env_int("CORE_MAX_BLOCKS", 256)

    # Residual
    RES_RANK: int = _env_int("RES_RANK", 512)
    RES_COEF: str = _env_str("RES_COEF", "diag").strip().lower()  # diag|full
    RES_BLOCKS: int = _env_int("RES_BLOCKS", 192)
    RES_BSIZE: int = _env_int("RES_BSIZE", 64)

        # Residual (improved selection)
    RES_TARGET: float = _env_float("RES_TARGET", 0.995)      # energy coverage of residual blocks
    RES_MAX_BLOCKS: int = _env_int("RES_MAX_BLOCKS", 4096)   # cap on residual blocks (replaces fixed-only behavior)

    # Eval on calibration distribution (recommended)
    EVAL_ON_CALIB: bool = _env_bool("EVAL_ON_CALIB", True)
    EVAL_CALIB_ROWS: int = _env_int("EVAL_CALIB_ROWS", 2048)


    # Refine
    REFINE_ENABLE: bool = _env_bool("REFINE_ENABLE", True)
    REFINE_ERR_TARGET: float = _env_float("REFINE_ERR_TARGET", 0.05)
    REFINE_MAX_EXTRA: int = _env_int("REFINE_MAX_EXTRA", 256)
    REFINE_BSIZE: int = _env_int("REFINE_BSIZE", 64)
    REFINE_RECHECK_EVERY: int = _env_int("REFINE_RECHECK_EVERY", 32)

    # Quant (for block tensors only)
    QMODE: str = _env_str("QMODE", "none").strip().lower()  # none|float16|int8

    # Eval
    EVAL_TRIALS: int = _env_int("EVAL_TRIALS", 8)
    EVAL_BATCH: int = _env_int("EVAL_BATCH", 2)
    ROUTED_K: int = _env_int("ROUTED_K", 8)

cfg = Cfg()
PRESET = _env_str("PRESET", "").strip().lower()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

def _setdefault_env(k: str, v: str):
    if k not in os.environ:
        os.environ[k] = v

# Presets
if PRESET == "maxacc":
    _setdefault_env("CALIB_SAMPLES", "32768")
    _setdefault_env("RIDGE_DAMP", "1e-2")
    _setdefault_env("CORE_BLOCK", "32")
    _setdefault_env("CORE_TARGET", "0.995")
    _setdefault_env("CORE_MAX_BLOCKS", "8192")
    _setdefault_env("RES_RANK", "2048")
    _setdefault_env("RES_COEF", "full")
    _setdefault_env("RES_BLOCKS", "4096")
    _setdefault_env("REFINE_ENABLE", "1")
    _setdefault_env("REFINE_ERR_TARGET", "0.01")
    _setdefault_env("REFINE_MAX_EXTRA", "32768")
    _setdefault_env("TRAIN_STEPS", "96")
    _setdefault_env("TRAIN_LR", "0.02")
    _setdefault_env("TRAIN_LAM_GUIDE", "0.5")
    _setdefault_env("TRAIN_GUIDE_EVERY", "2")
    _setdefault_env("BASIS_STORE_DTYPE", "float32")
    cfg = Cfg()
elif PRESET == "balanced":
    _setdefault_env("CALIB_SAMPLES", "8192")
    _setdefault_env("CORE_BLOCK", "32")
    _setdefault_env("CORE_TARGET", "0.97")
    _setdefault_env("CORE_MAX_BLOCKS", "2048")
    _setdefault_env("RES_RANK", "1024")
    _setdefault_env("RES_COEF", "diag")
    _setdefault_env("RES_BLOCKS", "1024")
    _setdefault_env("REFINE_ENABLE", "1")
    _setdefault_env("REFINE_ERR_TARGET", "0.03")
    _setdefault_env("REFINE_MAX_EXTRA", "4096")
    _setdefault_env("TRAIN_STEPS", "64")
    _setdefault_env("TRAIN_LR", "0.03")
    cfg = Cfg()
elif PRESET == "compact":
    _setdefault_env("CALIB_SAMPLES", "4096")
    _setdefault_env("CORE_BLOCK", "64")
    _setdefault_env("CORE_TARGET", "0.90")
    _setdefault_env("CORE_MAX_BLOCKS", "512")
    _setdefault_env("RES_RANK", "512")
    _setdefault_env("RES_COEF", "diag")
    _setdefault_env("RES_BLOCKS", "512")
    _setdefault_env("QMODE", "float16")
    _setdefault_env("REFINE_ENABLE", "0")
    _setdefault_env("TRAIN_STEPS", "24")
    cfg = Cfg()

# ----------------------------
# NPZ helpers
# ----------------------------
def save_npz_compressed(path: str, arrays: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez_compressed(path, **arrays)

def load_npz(path: str) -> Dict[str, np.ndarray]:
    z = np.load(path, allow_pickle=False)
    out = {k: z[k] for k in z.files}
    z.close()
    return out

def _encode_meta(meta: dict) -> np.ndarray:
    b = json.dumps(meta, sort_keys=True).encode("utf-8")
    return np.frombuffer(b, dtype=np.uint8)

def _decode_meta(arr: np.ndarray) -> dict:
    try:
        return json.loads(bytes(arr.tolist()).decode("utf-8"))
    except Exception:
        return {}

# ----------------------------
# Offline shard loading
# ----------------------------
def read_index(model_dir: str) -> Dict[str, str]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    wm = obj.get("weight_map", {})
    if not wm:
        raise RuntimeError("Index JSON has empty weight_map.")
    return wm

def find_layer_expert_ids(weight_map: Dict[str, str], layer: int) -> List[int]:
    pat = re.compile(rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.")
    ids = set()
    for k in weight_map.keys():
        m = pat.match(k)
        if m:
            ids.add(int(m.group(1)))
    return sorted(ids)

def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    prefix = f"model.layers.{layer}.mlp.experts.{eid}."
    def pick(cands: List[str]) -> Optional[str]:
        for suf in cands:
            k = prefix + suf
            if k in weight_map:
                return k
        return None
    up   = pick(["up_proj.weight", "w3.weight", "w1.weight"])
    gate = pick(["gate_proj.weight", "w1.weight", "w3.weight"])
    down = pick(["down_proj.weight", "w2.weight"])
    if up is None or gate is None or down is None:
        return {}
    if up == gate:
        g2 = pick(["gate_proj.weight"])
        if g2:
            gate = g2
    return {"up": up, "gate": gate, "down": down}

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], keys: List[str]) -> Dict[str, torch.Tensor]:
    by_shard: Dict[str, List[str]] = {}
    for k in keys:
        shard = weight_map.get(k, None)
        if shard is None:
            raise KeyError(f"Key not in weight_map: {k}")
        by_shard.setdefault(shard, []).append(k)

    out: Dict[str, torch.Tensor] = {}
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(sp):
            raise FileNotFoundError(f"Missing shard: {sp}")
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks:
                out[k] = f.get_tensor(k)
    return out

# ----------------------------
# Calibration: auto-detect, load, optional capture
# ----------------------------
def autodetect_calib_path() -> Optional[str]:
    cand = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
    return cand if os.path.isfile(cand) else None

def autodetect_router_path() -> Optional[str]:
    cand = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
    return cand if os.path.isfile(cand) else None

def load_calib_X(path: str, H: int) -> torch.Tensor:
    z = np.load(path, allow_pickle=False)
    if "X" not in z.files:
        raise KeyError(f"CALIB npz missing key 'X'. Keys={list(z.files)}")
    Xn = z["X"].astype(np.float32, copy=False)
    z.close()

    if Xn.ndim != 2 or Xn.shape[1] != H:
        raise RuntimeError(f"Bad X shape {tuple(Xn.shape)}, expected (*,{H})")

    nrows = int(Xn.shape[0])
    if nrows < min(1024, cfg.CALIB_SAMPLES):
        log(f"[WARN] Calibration X has only {nrows} rows. "
            f"This will cap accuracy hard. Use CAPTURE_ENABLE=1 and CALIB_SAMPLES=16384+.")

    # cap
    if nrows > cfg.CALIB_SAMPLES:
        Xn = Xn[:cfg.CALIB_SAMPLES]

    X = torch.from_numpy(Xn)
    return X.to(device=DEVICE, dtype=DTYPE_ACC)


def load_router_P(path: str) -> np.ndarray:
    z = np.load(path, allow_pickle=False)
    if "P" not in z.files:
        raise KeyError(f"ROUTER npz missing key 'P'. Keys={list(z.files)}")
    P = z["P"].astype(np.float32, copy=False)
    z.close()
    return P

def _maybe_autopip():
    if not cfg.HF_AUTO_PIP:
        return
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", "transformers", "sentencepiece", "tokenizers"])

def _patch_transformers_cache_compat():
    try:
        from transformers.cache_utils import DynamicCache  # type: ignore
        if not hasattr(DynamicCache, "get_usable_length"):
            def _get_usable_length(self, seq_length: int):
                return int(seq_length)
            DynamicCache.get_usable_length = _get_usable_length  # type: ignore
            log("[patch] Added DynamicCache.get_usable_length shim.")
    except Exception:
        pass

class _Collector:
    def __init__(self, H: int, E_total: int, max_rows: int):
        self.H = H
        self.E_total = E_total
        self.max_rows = max_rows
        self.X_chunks: List[torch.Tensor] = []
        self.P_chunks: List[torch.Tensor] = []
        self.nX = 0
        self.nP = 0

    def _take_rows(self, flat: torch.Tensor, need: int) -> torch.Tensor:
        return flat[:need] if flat.shape[0] > need else flat

    def add_X(self, hs: torch.Tensor, attn_mask: Optional[torch.Tensor]):
        if hs is None:
            return
        if hs.ndim == 2:
            hs = hs.unsqueeze(0)
        if hs.ndim != 3 or hs.shape[-1] != self.H:
            return

        hs = hs.detach().to(torch.float32).cpu()
        if attn_mask is not None and (not cfg.CAPTURE_KEEP_PAD):
            m = attn_mask.detach().cpu().to(torch.bool)
            flat = hs.reshape(-1, self.H)
            mflat = m.reshape(-1)
            flat = flat[mflat]
        else:
            flat = hs.reshape(-1, self.H)

        if flat.numel() == 0:
            return

        need = self.max_rows - self.nX
        if need <= 0:
            return
        flat = self._take_rows(flat, need)
        self.X_chunks.append(flat)
        self.nX += int(flat.shape[0])

    def add_logits(self, logits: torch.Tensor, attn_mask: Optional[torch.Tensor]):
        if logits is None:
            return
        if logits.ndim == 2:
            logits = logits.unsqueeze(0)
        if logits.ndim != 3:
            return

        P = torch.softmax(logits.detach().to(torch.float32), dim=-1)
        P = P[..., :self.E_total].cpu()

        if attn_mask is not None and (not cfg.CAPTURE_KEEP_PAD):
            m = attn_mask.detach().cpu().to(torch.bool)
            flat = P.reshape(-1, P.shape[-1])
            mflat = m.reshape(-1)
            flat = flat[mflat]
        else:
            flat = P.reshape(-1, P.shape[-1])

        if flat.numel() == 0:
            return

        need = self.max_rows - self.nP
        if need <= 0:
            return
        flat = self._take_rows(flat, need)
        self.P_chunks.append(flat)
        self.nP += int(flat.shape[0])

def _load_capture_texts() -> List[str]:
    if cfg.CAPTURE_TEXT_FILE and os.path.isfile(cfg.CAPTURE_TEXT_FILE):
        with open(cfg.CAPTURE_TEXT_FILE, "r", encoding="utf-8") as f:
            lines = [ln.strip() for ln in f.readlines()]
        lines = [x for x in lines if x]
        if lines:
            return lines
    return [cfg.CAPTURE_TEXT]

def capture_XP_transformers(model_dir: str, layer_idx: int, H: int, E_total: int,
                            out_x: str, out_p: str) -> Tuple[str, Optional[str]]:
    _maybe_autopip()
    _patch_transformers_cache_compat()
    try:
        from transformers import AutoTokenizer, AutoModelForCausalLM  # type: ignore
    except Exception as e:
        raise RuntimeError("transformers not available; install it or set HF_AUTO_PIP=1.") from e

    log("[capture] Loading tokenizer/model (local_files_only recommended)...")
    tok = AutoTokenizer.from_pretrained(
        model_dir,
        trust_remote_code=cfg.HF_TRUST_REMOTE_CODE,
        local_files_only=cfg.HF_LOCAL_FILES_ONLY,
        use_fast=True,
    )
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token if tok.eos_token is not None else tok.unk_token

    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        trust_remote_code=cfg.HF_TRUST_REMOTE_CODE,
        local_files_only=cfg.HF_LOCAL_FILES_ONLY,
        torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
        device_map=None,
        low_cpu_mem_usage=True,
    )
    model.eval().to(DEVICE)

    # Locate layers
    layers = None
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        layers = list(model.model.layers)
    elif hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        layers = list(model.transformer.h)
    elif hasattr(model, "layers"):
        layers = list(model.layers)
    if layers is None:
        raise RuntimeError("Cannot locate transformer layers list.")
    if layer_idx < 0 or layer_idx >= len(layers):
        raise RuntimeError(f"LAYER={layer_idx} out of range; model has {len(layers)} layers.")
    layer = layers[layer_idx]

    mlp = getattr(layer, "mlp", None)
    if mlp is None:
        for n, m in layer.named_modules():
            if n.lower().endswith("mlp"):
                mlp = m
                break
    if mlp is None:
        raise RuntimeError("Could not find layer.mlp to hook for X capture.")

    # Router discovery
    router_linear: Optional[nn.Linear] = None
    best_score = -1e9
    for name, mod in layer.named_modules():
        if isinstance(mod, nn.Linear) and getattr(mod, "in_features", None) == H and getattr(mod, "out_features", 0) >= E_total:
            nm = name.lower()
            score = 0
            if "router" in nm: score += 10
            if "gate" in nm: score += 6
            if "moe" in nm: score += 3
            if mod.out_features == E_total: score += 6
            score -= 0.01 * float(mod.out_features - E_total)
            if score > best_score:
                best_score = score
                router_linear = mod

    router_weight: Optional[torch.Tensor] = None
    if router_linear is None:
        best = None
        best_score = -1e9
        for pname, p in layer.named_parameters(recurse=True):
            if p.ndim == 2 and p.shape[1] == H and p.shape[0] >= E_total:
                nm = pname.lower()
                score = 0
                if "router" in nm: score += 10
                if "gate" in nm: score += 6
                if p.shape[0] == E_total: score += 6
                score -= 0.01 * float(p.shape[0] - E_total)
                if score > best_score:
                    best_score = score
                    best = p
        if best is not None:
            router_weight = best.detach()

    if router_linear is not None:
        log(f"[capture] Router Linear candidate: in={router_linear.in_features} out={router_linear.out_features}")
    elif router_weight is not None:
        log(f"[capture] Router weight candidate: shape={tuple(router_weight.shape)}")
    else:
        log("[capture] Router not found; will capture X only (no P).")

    coll = _Collector(H=H, E_total=E_total, max_rows=cfg.CALIB_SAMPLES)
    attn_mask_holder = {"mask": None}

    def mlp_pre_hook(_m, inputs):
        hs = inputs[0]
        am = attn_mask_holder["mask"]
        coll.add_X(hs, am)
        if router_linear is None and router_weight is not None and hs is not None:
            hs2 = hs if hs.ndim == 3 else hs.unsqueeze(0)
            W = router_weight.to(hs2.device, dtype=torch.float32)
            logits = torch.matmul(hs2.to(torch.float32), W.t())
            coll.add_logits(logits, am)

    h_mlp = mlp.register_forward_pre_hook(mlp_pre_hook)

    h_router = None
    if router_linear is not None:
        def router_hook(_m, inputs, output):
            out = output[0] if isinstance(output, (tuple, list)) else output
            if torch.is_tensor(out):
                am = attn_mask_holder["mask"]
                out2 = out if out.ndim == 3 else out.unsqueeze(0)
                coll.add_logits(out2, am)
        h_router = router_linear.register_forward_hook(router_hook)

    texts = _load_capture_texts()
    tptr = 0

    for it in range(cfg.CAPTURE_ITERS):
        text = texts[tptr % len(texts)]
        tptr += 1
        enc = tok(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=cfg.CAPTURE_MAX_TOKENS,
            padding="max_length",
        )
        for k in list(enc.keys()):
            if enc[k].ndim == 2 and cfg.CAPTURE_BATCH > 1:
                enc[k] = enc[k].repeat(cfg.CAPTURE_BATCH, 1)

        attn_mask_holder["mask"] = enc.get("attention_mask", None)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        with torch.inference_mode():
            _ = model(**enc, use_cache=False)

        if (it + 1) % 4 == 0 or it == 0 or (it + 1) == cfg.CAPTURE_ITERS:
            log(f"[capture] iter {it+1}/{cfg.CAPTURE_ITERS}  nX={coll.nX} nP={coll.nP}")

        if coll.nX >= cfg.CALIB_SAMPLES and (not cfg.RIDGE_WEIGHTED or coll.nP >= cfg.CALIB_SAMPLES):
            break

    h_mlp.remove()
    if h_router is not None:
        h_router.remove()

    if coll.nX == 0:
        raise RuntimeError("Capture failed: collected 0 X rows. Try CAPTURE_MAX_TOKENS↑ and CAPTURE_ITERS↑.")

    X = torch.cat(coll.X_chunks, dim=0)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
    save_npz_compressed(out_x, {"X": X})
    log(f"[capture] wrote X -> {out_x}  shape={tuple(X.shape)}")

    p_written = None
    if coll.nP > 0:
        P = torch.cat(coll.P_chunks, dim=0)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
        N = min(P.shape[0], X.shape[0])
        if N < X.shape[0]:
            X = X[:N]
            save_npz_compressed(out_x, {"X": X})
        P = P[:N]
        save_npz_compressed(out_p, {"P": P})
        log(f"[capture] wrote P -> {out_p}  shape={tuple(P.shape)}")
        p_written = out_p
    else:
        log("[capture] Router P not captured.")

    return out_x, p_written

def ensure_calib_router(H: int, E_total: int):
    if not cfg.CALIB_PATH:
        cand = autodetect_calib_path()
        if cand:
            cfg.CALIB_PATH = cand
            log(f"[calib] CALIB_PATH not set -> auto-found {cfg.CALIB_PATH}")

    if not cfg.ROUTER_PATH:
        cand = autodetect_router_path()
        if cand:
            cfg.ROUTER_PATH = cand
            log(f"[router] ROUTER_PATH not set -> auto-found {cfg.ROUTER_PATH}")

    need_capture = cfg.CAPTURE_FORCE or (cfg.CAPTURE_ENABLE and (not cfg.CALIB_PATH or not os.path.isfile(cfg.CALIB_PATH)))
    if need_capture:
        out_x = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
        out_p = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
        log("[calib] capturing X (and maybe P) via transformers...")
        x_path, p_path = capture_XP_transformers(cfg.MODEL_DIR, cfg.LAYER, H, E_total, out_x, out_p)
        cfg.CALIB_PATH = x_path
        if p_path:
            cfg.ROUTER_PATH = p_path

# ----------------------------
# Ridge linearization: build Ws
# ----------------------------
@torch.no_grad()
def forward_mlp(X: torch.Tensor, W_gate: torch.Tensor, W_up: torch.Tensor, W_down: torch.Tensor) -> torch.Tensor:
    Xf = X.to(DTYPE_ACC)
    up = Xf @ W_up.to(DTYPE_ACC).t()
    gate = Xf @ W_gate.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    y = hid @ W_down.to(DTYPE_ACC).t()
    return y

def ws_cache_path(E: int) -> str:
    return os.path.join(cfg.OUTPUT_DIR, f"Ws_cache_layer{cfg.LAYER}_E{E}_ridge_v10_5.npz")

def ws_meta(eids: List[int]) -> dict:
    # IMPORTANT: stable meta (NO time()) so cache can load.
    return dict(
        script="ktxx_offline_v10_5",
        model_dir=cfg.MODEL_DIR,
        layer=cfg.LAYER,
        expert_ids=eids,
        ridge_damp=float(cfg.RIDGE_DAMP),
        ridge_weighted=bool(cfg.RIDGE_WEIGHTED),
        router_eids_are_global=bool(cfg.ROUTER_EIDS_ARE_GLOBAL),
        router_path=(cfg.ROUTER_PATH or ""),
        calib_path=(cfg.CALIB_PATH or ""),
        calib_samples=int(cfg.CALIB_SAMPLES),
        normalize_w=bool(cfg.NORMALIZE_W),
        seed=int(SEED),
        device=str(DEVICE),
    )

@torch.no_grad()
def build_Ws(eids: List[int], wm: Dict[str, str]) -> Tuple[torch.Tensor, torch.Tensor]:
    per_e = {}
    need_keys = []
    for eid in eids:
        kk = pick_expert_tensor_keys(wm, cfg.LAYER, eid)
        if not kk:
            raise RuntimeError(f"Expert {eid} missing required tensors in index.")
        per_e[eid] = kk
        need_keys += [kk["up"], kk["down"], kk["gate"]]

    log("[load] reading tensors from shards ...")
    T = load_tensors_from_shards(cfg.MODEL_DIR, wm, sorted(set(need_keys)))

    W_up0 = T[per_e[eids[0]]["up"]]
    dff, H = int(W_up0.shape[0]), int(W_up0.shape[1])
    log(f"[shape] H={H} d_ff={dff}")

    ensure_calib_router(H=H, E_total=len(find_layer_expert_ids(wm, cfg.LAYER)))

    if not cfg.CALIB_PATH or not os.path.isfile(cfg.CALIB_PATH):
        raise RuntimeError("CALIB_PATH missing. Set CALIB_PATH or CAPTURE_ENABLE=1.")
    X = load_calib_X(cfg.CALIB_PATH, H)
    log(f"[calib] X loaded: {tuple(X.shape)}  (path={cfg.CALIB_PATH})")

    P = None
    if cfg.RIDGE_WEIGHTED:
        if cfg.ROUTER_PATH and os.path.isfile(cfg.ROUTER_PATH):
            P = load_router_P(cfg.ROUTER_PATH)
            log(f"[router] P loaded: {tuple(P.shape)}  (path={cfg.ROUTER_PATH})")
        else:
            log("[router] RIDGE_WEIGHTED=1 but ROUTER_PATH missing -> forcing RIDGE_WEIGHTED=0")
            cfg.RIDGE_WEIGHTED = False

    Xf = X.to(DTYPE_ACC)
    I = torch.eye(H, dtype=DTYPE_ACC, device=DEVICE)
    XtX = Xf.t() @ Xf
    lam = cfg.RIDGE_DAMP * float(torch.trace(XtX).item()) / float(H)
    cholG = torch.linalg.cholesky(XtX + lam * I)

    Ws_list: List[torch.Tensor] = []
    scales: List[float] = []

    for i, eid in enumerate(tqdm(eids, desc="Build Ws (ridge)")):
        W_up = T[per_e[eid]["up"]].to(DEVICE)
        W_dn = T[per_e[eid]["down"]].to(DEVICE)
        W_gt = T[per_e[eid]["gate"]].to(DEVICE)

        Y = forward_mlp(X, W_gt, W_up, W_dn).to(DTYPE_ACC)

        if cfg.RIDGE_WEIGHTED and (P is not None):
            if cfg.ROUTER_EIDS_ARE_GLOBAL:
                if eid >= P.shape[1]:
                    raise RuntimeError(f"P shape {P.shape} cannot index eid={eid}")
                w = torch.from_numpy(P[:X.shape[0], eid]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
            else:
                if i >= P.shape[1]:
                    raise RuntimeError(f"P shape {P.shape} cannot index i={i}")
                w = torch.from_numpy(P[:X.shape[0], i]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
            sw = torch.sqrt(w + 1e-12).view(-1, 1)
            Xw = Xf * sw
            Yw = Y * sw
            XtX_e = Xw.t() @ Xw
            lam_e = cfg.RIDGE_DAMP * float(torch.trace(XtX_e).item()) / float(H)
            chol = torch.linalg.cholesky(XtX_e + lam_e * I)
            XtY = Xw.t() @ Yw
            Wt = torch.cholesky_solve(XtY, chol)
            W = Wt.t().contiguous()
        else:
            XtY = Xf.t() @ Y
            Wt = torch.cholesky_solve(XtY, cholG)
            W = Wt.t().contiguous()

        if cfg.NORMALIZE_W:
            s = float(torch.linalg.norm(W, ord="fro").clamp_min(1e-12).item())
            W = (W / s).contiguous()
        else:
            s = 1.0

        Ws_list.append(W)
        scales.append(s)

    Ws = torch.stack(Ws_list, dim=0).to(DTYPE_ACC).to(DEVICE)  # (E,H,H) normalized if NORMALIZE_W
    Sc = torch.tensor(scales, dtype=DTYPE_ACC, device=DEVICE)  # (E,)
    return Ws, Sc

def load_or_build_Ws() -> Tuple[List[int], torch.Tensor, torch.Tensor]:
    wm = read_index(cfg.MODEL_DIR)
    all_eids = find_layer_expert_ids(wm, cfg.LAYER)
    if not all_eids:
        raise RuntimeError(f"No experts found at layer={cfg.LAYER}")
    eids = all_eids[:cfg.MAX_EXPERTS]
    log(f"[found] layer={cfg.LAYER} total_experts={len(all_eids)} using={len(eids)} eids={eids}")

    # pre-autodetect so meta can match on future runs
    if not cfg.CALIB_PATH:
        c = autodetect_calib_path()
        if c:
            cfg.CALIB_PATH = c
    if not cfg.ROUTER_PATH:
        r = autodetect_router_path()
        if r:
            cfg.ROUTER_PATH = r

    cpath = ws_cache_path(len(eids))
    if os.path.isfile(cpath):
        z = load_npz(cpath)
        if "meta" in z and "Ws" in z and "expert_ids" in z and "scales" in z:
            if _decode_meta(z["meta"]) == ws_meta(eids):
                Ws = torch.from_numpy(z["Ws"]).to(DTYPE_ACC).to(DEVICE)
                Sc = torch.from_numpy(z["scales"]).to(DTYPE_ACC).to(DEVICE)
                log(f"[cache] loaded Ws -> {cpath}  Ws={tuple(Ws.shape)}")
                return [int(x) for x in z["expert_ids"].tolist()], Ws, Sc
        log("[cache] Ws meta mismatch -> rebuilding.")

    Ws, Sc = build_Ws(eids, wm)

    save_npz_compressed(cpath, {
        "meta": _encode_meta(ws_meta(eids)),
        "expert_ids": np.array(eids, dtype=np.int32),
        "Ws": Ws.detach().cpu().numpy().astype(np.float32),
        "scales": Sc.detach().cpu().numpy().astype(np.float32),
    })
    log(f"[cache] wrote Ws -> {cpath}  size={os.path.getsize(cpath)/1e6:.2f} MB")
    return eids, Ws, Sc

# ----------------------------
# KMeans clustering helpers
# ----------------------------
@torch.no_grad()
def random_proj_features(Ws: torch.Tensor, d: int) -> torch.Tensor:
    E, n, _ = Ws.shape
    g = torch.Generator(device="cpu").manual_seed(SEED + 17)
    R = (torch.randint(0, 2, (n, d), generator=g, dtype=torch.int8) * 2 - 1).to(DTYPE_ACC).to(DEVICE)
    feats = []
    for e in range(E):
        W = Ws[e]
        row = torch.diag(W @ W.t())
        col = torch.diag(W.t() @ W)
        feats.append(torch.cat([row @ R, col @ R], dim=0).unsqueeze(0))
    X = torch.cat(feats, dim=0)
    X = (X - X.mean(dim=0, keepdim=True)) / (X.std(dim=0, keepdim=True) + 1e-6)
    return X

@torch.no_grad()
def kmeans_torch(X: torch.Tensor, k: int, iters: int, restarts: int) -> torch.Tensor:
    best_lab = None
    best_inertia = float("inf")
    g = torch.Generator(device="cpu").manual_seed(SEED + 999)

    def init_centers():
        n = X.shape[0]
        centers = []
        idx0 = torch.randint(0, n, (1,), generator=g).item()
        centers.append(X[idx0].clone())
        for _ in range(1, k):
            C = torch.stack(centers, dim=0)
            dist2 = torch.cdist(X, C).pow(2).min(dim=1).values
            prob = dist2 / dist2.sum().clamp_min(1e-12)
            idx = torch.multinomial(prob, num_samples=1, generator=g).item()
            centers.append(X[idx].clone())
        return torch.stack(centers, dim=0)

    for _ in range(max(1, restarts)):
        C = init_centers()
        for _ in range(iters):
            dist = torch.cdist(X, C)
            lab = dist.argmin(dim=1)
            for j in range(k):
                m = (lab == j)
                if m.any():
                    C[j] = X[m].mean(dim=0)
                else:
                    far = dist.min(dim=1).values.argmax().item()
                    C[j] = X[far].clone()
        dist = torch.cdist(X, C)
        inertia = float((dist.min(dim=1).values ** 2).sum().item())
        if inertia < best_inertia:
            best_inertia = inertia
            best_lab = lab.clone()
    assert best_lab is not None
    return best_lab.to(torch.int64)

@torch.no_grad()
def relabel_contiguous(labels: torch.Tensor) -> torch.Tensor:
    labels = labels.to(torch.int64)
    uniq = torch.unique(labels)
    out = labels.clone()
    for new, old in enumerate(uniq.tolist()):
        out[labels == int(old)] = int(new)
    return out

@torch.no_grad()
def merge_small_clusters(X: torch.Tensor, labels: torch.Tensor, min_size: int) -> torch.Tensor:
    labels = relabel_contiguous(labels)
    if min_size <= 1:
        return labels
    while True:
        K = int(labels.max().item()) + 1
        counts = torch.bincount(labels, minlength=K)
        small = (counts < min_size).nonzero(as_tuple=False).flatten()
        if small.numel() == 0:
            break
        C = torch.stack([X[labels == k].mean(dim=0) for k in range(K)], dim=0)
        for c in small.tolist():
            idxs = (labels == int(c)).nonzero(as_tuple=False).flatten()
            if idxs.numel() == 0:
                continue
            dist = torch.cdist(C[int(c)].unsqueeze(0), C).squeeze(0)
            dist[int(c)] = 1e9
            tgt = int(dist.argmin().item())
            labels[idxs] = tgt
        labels = relabel_contiguous(labels)
    return labels

@torch.no_grad()
def hierarchical_split(X: torch.Tensor, labels: torch.Tensor, max_size: int, max_k: int, split_iters: int) -> torch.Tensor:
    labels = relabel_contiguous(labels)
    if max_size <= 0:
        return labels
    while True:
        K = int(labels.max().item()) + 1
        if K >= max_k:
            break
        counts = torch.bincount(labels, minlength=K)
        biggest = int(torch.argmax(counts).item())
        bigsz = int(counts[biggest].item())
        if bigsz <= max_size:
            break
        idxs = (labels == biggest).nonzero(as_tuple=False).flatten()
        if idxs.numel() < 2:
            break
        sub = X[idxs]
        sub_lab = kmeans_torch(sub, k=2, iters=split_iters, restarts=1)
        a = idxs[sub_lab == 0]
        b = idxs[sub_lab == 1]
        if a.numel() == 0 or b.numel() == 0:
            break
        labels[b] = K
        labels = relabel_contiguous(labels)
    return labels

# ----------------------------
# Basis training (dense_train)
# ----------------------------
class OrthoParam(nn.Module):
    def __init__(self, init_mat: torch.Tensor):
        super().__init__()
        self.M = nn.Parameter(init_mat.to(device=DEVICE, dtype=DTYPE_ACC).contiguous())

    def orthogonal(self) -> torch.Tensor:
        Q, _ = torch.linalg.qr(self.M)
        return Q

@torch.no_grad()
def svd_init_from_mean(Wmean: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    U, _, Vh = torch.linalg.svd(Wmean, full_matrices=False)
    return U.to(DTYPE_ACC).contiguous(), Vh.t().to(DTYPE_ACC).contiguous()

def schedule(step: int, warmup: int, total: int) -> float:
    if total <= 0:
        return 1.0
    if step <= warmup:
        return 0.0
    return float(min(1.0, max(0.0, (step - warmup) / max(1, (total - warmup)))))

def slice_X_batch(Ws_batch: torch.Tensor, U: torch.Tensor, V: torch.Tensor, S: torch.Tensor) -> torch.Tensor:
    U_S = U[:, S]      # (n,s)
    V_S = V[:, S]      # (n,s)
    T = Ws_batch @ V_S
    Xs = torch.matmul(U_S.t().unsqueeze(0), T)
    return Xs

def offdiag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    Xoff = Xs - torch.diag_embed(D)
    return Xoff.abs().mean()

def diag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    return D.abs().mean()

def block_group_sparsity_penalty(Xs: torch.Tensor, block: int) -> torch.Tensor:
    Eb, s, _ = Xs.shape
    b = int(block)
    if b <= 0:
        return torch.zeros((), dtype=DTYPE_ACC, device=Xs.device)
    nb = s // b
    if nb <= 0:
        return torch.zeros((), dtype=DTYPE_ACC, device=Xs.device)
    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()
    Xb = X.view(Eb, nb, b, nb, b).permute(0, 1, 3, 2, 4).contiguous()
    Eblk = (Xb * Xb).sum(dim=(3, 4))
    P = Eblk.mean(dim=0)
    return torch.sqrt(P + 1e-12).sum() / (P.sum() + 1e-12)

@torch.no_grad()
def make_guidance_mask_from_Xs(Xs: torch.Tensor, block: int, target: float, max_blocks: int) -> Tuple[torch.Tensor, float, int]:
    Eb, s, _ = Xs.shape
    b = int(block)
    if b <= 0:
        return torch.ones(s, s, dtype=DTYPE_ACC, device=Xs.device), 1.0, 0
    nb = s // b
    if nb <= 0:
        return torch.ones(s, s, dtype=DTYPE_ACC, device=Xs.device), 1.0, 0
    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()
    Xb = X.view(Eb, nb, b, nb, b).permute(0, 1, 3, 2, 4).contiguous()
    Eg = (Xb * Xb).sum(dim=(3, 4)).mean(dim=0)
    tot = float((X * X).sum().item()) / max(1, Eb)
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    csum = torch.cumsum(flat[order], dim=0)
    frac = csum / max(tot, 1e-12)
    need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else flat.numel())
    K = min(need, max_blocks, flat.numel())
    pick = order[:K]
    mask = torch.zeros(s2, s2, dtype=DTYPE_ACC, device=Xs.device)
    for idx in pick.tolist():
        bi = idx // nb
        bj = idx % nb
        i0 = bi * b
        j0 = bj * b
        mask[i0:i0 + b, j0:j0 + b] = 1.0
    if s2 < s:
        full = torch.zeros(s, s, dtype=DTYPE_ACC, device=Xs.device)
        full[:s2, :s2] = mask
        mask = full
    ef = float(frac[K - 1].item()) if K > 0 else 0.0
    return mask, ef, int(K)

# ----------------------------
# Block energy + selection
# ----------------------------
@torch.no_grad()
def block_energy_grid(X: torch.Tensor, b: int) -> Tuple[torch.Tensor, float, int]:
    n = X.shape[0]
    nb = (n + b - 1) // b
    if (n % b) != 0:
        Xp = torch.zeros(nb * b, nb * b, dtype=X.dtype, device=X.device)
        Xp[:n, :n] = X
        X = Xp
    Xb = X.view(nb, b, nb, b).permute(0, 2, 1, 3).contiguous()
    Eg = (Xb * Xb).sum(dim=(2, 3))
    tot = float((X * X).sum().item())
    return Eg, tot, nb



@torch.no_grad()
def pick_blocks_until_target(
    Eg: torch.Tensor,
    tot_energy: float,
    target: float,
    max_blocks: int,
    exclude: Optional[Set[Tuple[int, int]]] = None,
    min_block_energy: float = 0.0,
) -> Tuple[List[Tuple[int, int]], float]:
    """
    Picks (bi,bj) blocks until picked_energy/tot_energy >= target (or max_blocks).
    Optionally excludes block positions.
    """
    nb = Eg.shape[0]
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)

    picked: List[Tuple[int, int]] = []
    eacc = 0.0
    exclude = exclude or set()

    for idx in order.tolist():
        if len(picked) >= max_blocks:
            break
        e = float(flat[idx].item())
        if e <= max(min_block_energy, 1e-18):
            break
        bi = idx // nb
        bj = idx % nb
        pos = (bi, bj)
        if pos in exclude:
            continue
        picked.append(pos)
        eacc += e
        if (eacc / max(tot_energy, 1e-12)) >= target:
            break

    eff = eacc / max(tot_energy, 1e-12)
    return picked, float(eff)



@torch.no_grad()
def pick_top_blocks(Eg: torch.Tensor, tot_energy: float, target: float, max_blocks: int) -> Tuple[List[Tuple[int, int]], float]:
    nb = Eg.shape[0]
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    csum = torch.cumsum(flat[order], dim=0)
    frac = csum / max(tot_energy, 1e-12)
    need = int((torch.nonzero(frac >= target, as_tuple=False)[0].item() + 1) if (frac >= target).any() else flat.numel())
    K = min(need, max_blocks, flat.numel())
    pick = order[:K].tolist()
    blocks = [(p // nb, p % nb) for p in pick]
    eff = float(frac[K - 1].item()) if K > 0 else 0.0
    return blocks, eff

@torch.no_grad()
def choose_core_blocks(X_list: List[torch.Tensor]) -> Dict[str, Any]:
    mode = cfg.CORE_MODE
    agg = cfg.CORE_AGG
    b = int(cfg.CORE_BLOCK)

    if mode == "none":
        return {"mode": "none", "shared": None, "per": None, "energy_fracs": []}

    if mode == "blockdiag":
        Eg_sum = None
        tot = 0.0
        for X in X_list:
            Eg, te, nb = block_energy_grid(X, b)
            tot += te
            diag = torch.diagonal(Eg, 0)
            if Eg_sum is None:
                Eg_sum = torch.zeros_like(Eg)
            Eg_sum += torch.diag(diag)
        Eg_mean = Eg_sum / max(1, len(X_list))
        blocks, eff = pick_top_blocks(Eg_mean, tot / max(1, len(X_list)), cfg.CORE_TARGET, cfg.CORE_MAX_BLOCKS)
        return {"mode": "blockdiag", "shared": blocks, "per": None, "energy_fracs": [eff]}

    if mode == "blocktopk":
        Eg_all = []
        tots = []
        for X in X_list:
            Eg, te, nb = block_energy_grid(X, b)
            Eg_all.append(Eg)
            tots.append(te)
        Eg_stack = torch.stack(Eg_all, dim=0)
        Eg = Eg_stack.amax(dim=0) if agg == "max" else Eg_stack.mean(dim=0)
        blocks, eff = pick_top_blocks(Eg, float(np.mean(tots)), cfg.CORE_TARGET, cfg.CORE_MAX_BLOCKS)
        return {"mode": "blocktopk", "shared": blocks, "per": None, "energy_fracs": [eff]}

    if mode == "blocktopk_perexpert":
        per = []
        efs = []
        for X in X_list:
            Eg, te, nb = block_energy_grid(X, b)
            blocks, eff = pick_top_blocks(Eg, te, cfg.CORE_TARGET, cfg.CORE_MAX_BLOCKS)
            per.append(blocks)
            efs.append(eff)
        return {"mode": "blocktopk_perexpert", "shared": None, "per": per, "energy_fracs": efs}

    raise ValueError("CORE_MODE must be blocktopk_perexpert|blocktopk|blockdiag|none")

@torch.no_grad()
def gather_block(X: torch.Tensor, i0: int, j0: int, b: int) -> torch.Tensor:
    n = X.shape[0]
    i1 = min(n, i0 + b)
    j1 = min(n, j0 + b)
    return X[i0:i1, j0:j1].contiguous()

# ----------------------------
# Low-rank (randomized SVD-like)
# ----------------------------
@torch.no_grad()
def rand_svd_vectors(A: torch.Tensor, r: int, n_iter: int = 2) -> Tuple[torch.Tensor, torch.Tensor]:
    n = A.shape[0]
    r = min(r, n)
    g = torch.Generator(device="cpu").manual_seed(SEED + 777)
    Omega = torch.randn(n, r, generator=g, dtype=DTYPE_ACC, device=A.device)
    Y = A @ Omega
    for _ in range(max(0, n_iter)):
        Y = A @ (A.t() @ Y)
    Q, _ = torch.linalg.qr(Y)
    B = Q.t() @ A
    Uhat, _, Vh = torch.linalg.svd(B, full_matrices=False)
    U = (Q @ Uhat[:, :r]).contiguous()
    V = (Vh.t()[:, :r]).contiguous()
    return U, V

# ----------------------------
# Quant helpers (blocks)
# ----------------------------
@torch.no_grad()
def q_block_int8(B: torch.Tensor) -> Tuple[np.ndarray, np.float16]:
    x = B.detach().cpu().to(torch.float32)
    maxabs = float(x.abs().max().item())
    if maxabs < 1e-12:
        return np.zeros_like(x.numpy(), dtype=np.int8), np.float16(1.0)
    scale = maxabs / 127.0
    q = torch.clamp(torch.round(x / scale), -127, 127).to(torch.int8).cpu().numpy()
    return q, np.float16(scale)

# ----------------------------
# Payload packing (ragged blocks)
# ----------------------------
def _block_store_dtype_for_qmode(qmode: str) -> np.dtype:
    # IMPORTANT: QMODE=none -> float32 (best)
    if qmode == "none":
        return np.float32
    return np.float16  # float16 or int8 modes store float16 values/scales

def pack_blocks_ragged(blocks_per_expert: List[List[Tuple[int, int, torch.Tensor]]], qmode: str) -> Dict[str, np.ndarray]:
    """
    Ragged storage:
      exp_ptr: (E+1,) indices into block arrays
      blk_i0, blk_j0, blk_h, blk_w: (B,)
      blk_ptr: (B+1,) offsets into flattened values
      blk_val (float32/float16) OR blk_q(int8)+blk_scale(float16)
    """
    val_dtype = _block_store_dtype_for_qmode(qmode)

    E = len(blocks_per_expert)
    exp_ptr = [0]
    blk_i0, blk_j0, blk_h, blk_w = [], [], [], []
    blk_ptr = [0]
    vals = []
    vals_i8 = []
    scales = []

    for e in range(E):
        lst = blocks_per_expert[e]
        for (i0, j0, B) in lst:
            h, w = B.shape
            blk_i0.append(int(i0)); blk_j0.append(int(j0))
            blk_h.append(int(h)); blk_w.append(int(w))
            if qmode == "int8":
                q, sc = q_block_int8(B)
                vals_i8.append(q.reshape(-1))
                scales.append(sc)
                blk_ptr.append(blk_ptr[-1] + q.size)
            else:
                v = B.detach().cpu().to(torch.float32).numpy().astype(val_dtype, copy=False).reshape(-1)
                vals.append(v)
                blk_ptr.append(blk_ptr[-1] + v.size)
        exp_ptr.append(len(blk_i0))

    out: Dict[str, np.ndarray] = {}
    out["exp_ptr"] = np.array(exp_ptr, dtype=np.int32)
    out["blk_i0"] = np.array(blk_i0, dtype=np.int16)
    out["blk_j0"] = np.array(blk_j0, dtype=np.int16)
    out["blk_h"] = np.array(blk_h, dtype=np.int16)
    out["blk_w"] = np.array(blk_w, dtype=np.int16)
    out["blk_ptr"] = np.array(blk_ptr, dtype=np.int64)

    if qmode == "int8":
        out["blk_q"] = np.concatenate(vals_i8, axis=0).astype(np.int8, copy=False) if vals_i8 else np.zeros((0,), dtype=np.int8)
        out["blk_scale"] = np.array(scales, dtype=np.float16)
    else:
        out["blk_val"] = np.concatenate(vals, axis=0) if vals else np.zeros((0,), dtype=val_dtype)
    return out

def unpack_blocks_ragged(pack: Dict[str, np.ndarray], qmode: str, device: torch.device) -> List[List[Tuple[int, int, torch.Tensor]]]:
    exp_ptr = pack["exp_ptr"].astype(np.int32)
    blk_i0 = pack["blk_i0"].astype(np.int32)
    blk_j0 = pack["blk_j0"].astype(np.int32)
    blk_h = pack["blk_h"].astype(np.int32)
    blk_w = pack["blk_w"].astype(np.int32)
    blk_ptr = pack["blk_ptr"].astype(np.int64)

    if qmode == "int8":
        blk_q = pack["blk_q"].astype(np.int8)
        blk_scale = pack["blk_scale"].astype(np.float16)
        blk_val = None
    else:
        blk_val = pack["blk_val"]  # may be float16 or float32 depending on qmode
        blk_q = None
        blk_scale = None

    E = exp_ptr.shape[0] - 1
    out: List[List[Tuple[int, int, torch.Tensor]]] = []
    for e in range(E):
        b0 = int(exp_ptr[e])
        b1 = int(exp_ptr[e + 1])
        lst = []
        for bi in range(b0, b1):
            i0 = int(blk_i0[bi]); j0 = int(blk_j0[bi])
            h = int(blk_h[bi]); w = int(blk_w[bi])
            v0 = int(blk_ptr[bi]); v1 = int(blk_ptr[bi + 1])
            if qmode == "int8":
                q = blk_q[v0:v1].astype(np.int8, copy=False).astype(np.float32)
                sc = float(blk_scale[bi])
                B = torch.from_numpy((q * sc).reshape(h, w)).to(device=device, dtype=DTYPE_ACC)
            else:
                v = blk_val[v0:v1].astype(np.float32, copy=False)
                B = torch.from_numpy(v.reshape(h, w)).to(device=device, dtype=DTYPE_ACC)
            lst.append((i0, j0, B))
        out.append(lst)
    return out

# ----------------------------
# Runtime payload object
# ----------------------------
class PayloadRuntime:
    def __init__(self):
        self.meta: dict = {}
        self.expert_ids: List[int] = []
        self.scales: Optional[torch.Tensor] = None       # (E,) Sc to undo normalization
        self.cluster_of_pos: Optional[torch.Tensor] = None

        self.U: List[torch.Tensor] = []
        self.V: List[torch.Tensor] = []
        self.DL: List[torch.Tensor] = []
        self.DR: List[torch.Tensor] = []

        self.gam: Optional[torch.Tensor] = None          # (E,r) if diag
        self.Cfull: Optional[torch.Tensor] = None        # (E,r,r) if full

        self.core_blocks: List[List[Tuple[int, int, torch.Tensor]]] = []
        self.res_blocks: List[List[Tuple[int, int, torch.Tensor]]] = []

        self.qmode: str = "none"
        self.res_coef: str = "diag"

    @torch.no_grad()
    def apply_expert(self, x: torch.Tensor, pos: int) -> torch.Tensor:
        c = int(self.cluster_of_pos[pos].item())  # type: ignore[index]
        U = self.U[c]
        V = self.V[c]
        DL = self.DL[c]
        DR = self.DR[c]

        z = x @ U
        u = torch.zeros_like(z)

        for (i0, j0, B) in self.core_blocks[pos]:
            h, w = B.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ B

        if self.res_coef == "diag":
            g = self.gam[pos]  # type: ignore[index]
            u += ((z @ DL) * g.view(1, -1)) @ DR.t()
        else:
            C = self.Cfull[pos]  # type: ignore[index]
            u += (z @ DL) @ C @ DR.t()

        for (i0, j0, B) in self.res_blocks[pos]:
            h, w = B.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ B

        y = u @ V.t()

        # IMPORTANT: apply Sc exactly once (payload built on W_norm)
        if self.scales is not None:
            y = y * self.scales[pos]
        return y

    @torch.no_grad()
    def apply_mixture(self, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
        y = torch.zeros_like(x)
        for a, pos in zip(gates.tolist(), routed):
            y += float(a) * self.apply_expert(x, int(pos))
        return y

def load_payload_runtime(path: str, device: torch.device) -> PayloadRuntime:
    z = load_npz(path)
    rt = PayloadRuntime()
    rt.meta = _decode_meta(z["meta"]) if "meta" in z else {}
    rt.qmode = rt.meta.get("qmode", "none")
    rt.res_coef = rt.meta.get("res_coef", "diag")

    rt.expert_ids = [int(x) for x in z["expert_ids"].tolist()]
    rt.scales = torch.from_numpy(z["scales"]).to(device=device, dtype=DTYPE_ACC)
    rt.cluster_of_pos = torch.from_numpy(z["cluster_of_pos"]).to(device=device, dtype=torch.int64)

    M = int(z["n_clusters"][0])

    for m in range(M):
        rt.U.append(torch.from_numpy(z[f"U_{m}"]).to(device=device, dtype=DTYPE_ACC))
        rt.V.append(torch.from_numpy(z[f"V_{m}"]).to(device=device, dtype=DTYPE_ACC))
        rt.DL.append(torch.from_numpy(z[f"DL_{m}"]).to(device=device, dtype=DTYPE_ACC))
        rt.DR.append(torch.from_numpy(z[f"DR_{m}"]).to(device=device, dtype=DTYPE_ACC))

    if rt.res_coef == "diag":
        rt.gam = torch.from_numpy(z["gam"]).to(device=device, dtype=DTYPE_ACC)
    else:
        rt.Cfull = torch.from_numpy(z["Cfull"]).to(device=device, dtype=DTYPE_ACC)

    core_pack = {k.replace("core_", ""): z[k] for k in z.keys() if k.startswith("core_")}
    res_pack = {k.replace("res_", ""): z[k] for k in z.keys() if k.startswith("res_")}
    rt.core_blocks = unpack_blocks_ragged(core_pack, rt.qmode, device)
    rt.res_blocks = unpack_blocks_ragged(res_pack, rt.qmode, device)

    return rt

# ----------------------------
# Build payload (dense_train / identity)
# ----------------------------
@torch.no_grad()
def frob_rel_err(A: torch.Tensor, B: torch.Tensor) -> float:
    num = torch.linalg.norm(A - B, ord="fro")
    den = torch.linalg.norm(B, ord="fro").clamp_min(1e-12)
    return float((num / den).item())

@torch.no_grad()
def build_payload_for_cluster(
    Ws: torch.Tensor,                 # normalized Ws
    idx_pos: List[int],
    U: torch.Tensor,
    V: torch.Tensor,
) -> Tuple[
    List[List[Tuple[int,int,torch.Tensor]]],   # core blocks per expert
    torch.Tensor, torch.Tensor,               # DL, DR
    List[torch.Tensor],                       # coef per expert (diag vector or full matrix)
    List[List[Tuple[int,int,torch.Tensor]]],  # res blocks per expert
    List[float],                              # core energy fractions
]:
    # IMPORTANT: basis-space uses NORMALIZED W (no Sc here)
    X_list = [(U.t() @ Ws[pos] @ V).contiguous() for pos in idx_pos]

    core = choose_core_blocks(X_list)
    b = int(cfg.CORE_BLOCK)

    core_blocks_per: List[List[Tuple[int,int,torch.Tensor]]] = []
    if core["per"] is not None:
        for X, blocks in zip(X_list, core["per"]):
            lst = []
            for (bi, bj) in blocks:
                i0, j0 = bi*b, bj*b
                lst.append((i0, j0, gather_block(X, i0, j0, b)))
            core_blocks_per.append(lst)
    else:
        shared = core["shared"] or []
        for X in X_list:
            lst = []
            for (bi, bj) in shared:
                i0, j0 = bi*b, bj*b
                lst.append((i0, j0, gather_block(X, i0, j0, b)))
            core_blocks_per.append(lst)

    # residual after core
    R_list = []
    for X, cb in zip(X_list, core_blocks_per):
        Xc = torch.zeros_like(X)
        for (i0, j0, Bc) in cb:
            h, w = Bc.shape
            Xc[i0:i0+h, j0:j0+w] = Bc
        R_list.append((X - Xc).contiguous())

    # shared low-rank from mean residual
    Rmean = torch.stack(R_list, dim=0).mean(dim=0)
    r = min(cfg.RES_RANK, Rmean.shape[0])
    DL, DR = rand_svd_vectors(Rmean, r=r, n_iter=2)

    coef_list: List[torch.Tensor] = []
    res_blocks_per: List[List[Tuple[int,int,torch.Tensor]]] = []

    bb = int(cfg.RES_BSIZE)
    for Rm in R_list:
        if cfg.RES_COEF == "diag":
            g = torch.sum(DL * (Rm @ DR), dim=0).contiguous()
            coef_list.append(g)
            R2 = (Rm - (DL * g.view(1, -1)) @ DR.t()).contiguous()
        else:
            C = (DL.t() @ Rm @ DR).contiguous()
            coef_list.append(C)
            R2 = (Rm - (DL @ C @ DR.t())).contiguous()

        # --- residual sparse blocks: pick by energy target, excluding CORE positions ---
        Eg, te, nb = block_energy_grid(R2, bb)

        # exclude any block that is already in CORE for this expert
        core_excl: Set[Tuple[int, int]] = set()
        for (ci0, cj0, _Bc) in core_blocks_per[len(res_blocks_per)]:  # current expert index
            core_excl.add((ci0 // bb, cj0 // bb))  # convert to bb-grid

        # energy-target pick (Pareto)
        maxb = int(getattr(cfg, "RES_MAX_BLOCKS", cfg.RES_BLOCKS))
        tgt = float(getattr(cfg, "RES_TARGET", 0.0))
        if tgt <= 0.0:
            tgt = 1.0  # behave like "take as much as possible"

        picks, _eff = pick_blocks_until_target(
            Eg=Eg,
            tot_energy=te,
            target=tgt,
            max_blocks=maxb,
            exclude=core_excl,
            min_block_energy=0.0,
        )

        blocks = []
        for (bi, bj) in picks:
            i0 = bi * bb
            j0 = bj * bb
            blocks.append((i0, j0, gather_block(R2, i0, j0, bb)))

        res_blocks_per.append(blocks)


    return core_blocks_per, DL.contiguous(), DR.contiguous(), coef_list, res_blocks_per, core["energy_fracs"]

@torch.no_grad()
def refine_expert_until(
    X: torch.Tensor,
    core_blocks: List[Tuple[int,int,torch.Tensor]],
    DL: torch.Tensor,
    DR: torch.Tensor,
    coef: torch.Tensor,
    res_blocks: List[Tuple[int,int,torch.Tensor]],
) -> List[Tuple[int,int,torch.Tensor]]:
    bb = int(cfg.REFINE_BSIZE)

    core_pos = {(i0, j0) for (i0, j0, _) in core_blocks}
    res_pos  = {(i0, j0) for (i0, j0, _) in res_blocks}

    def reconstruct() -> torch.Tensor:
        Xc = torch.zeros_like(X)
        for (i0, j0, Bc) in core_blocks:
            h, w = Bc.shape
            Xc[i0:i0+h, j0:j0+w] = Bc
        if cfg.RES_COEF == "diag":
            g = coef
            Xlr = (DL * g.view(1, -1)) @ DR.t()
        else:
            C = coef
            Xlr = DL @ C @ DR.t()
        Xr = torch.zeros_like(X)
        for (i0, j0, Bb) in res_blocks:
            h, w = Bb.shape
            Xr[i0:i0+h, j0:j0+w] += Bb
        return (Xc + Xlr + Xr).contiguous()

    Xhat = reconstruct()
    err = frob_rel_err(Xhat, X)
    if err <= cfg.REFINE_ERR_TARGET:
        return res_blocks

    added = 0
    while err > cfg.REFINE_ERR_TARGET and added < cfg.REFINE_MAX_EXTRA:
        R = (X - Xhat).contiguous()
        Eg, te, nb = block_energy_grid(R, bb)
        flat = Eg.reshape(-1)
        if float(flat.max().item()) <= 1e-18:
            break

        order = torch.argsort(flat, descending=True)
        found = False
        for idx in order.tolist():
            if float(flat[idx].item()) <= 1e-18:
                break
            bi = idx // nb
            bj = idx % nb
            i0 = bi * bb
            j0 = bj * bb

            # skip overlaps, but KEEP SEARCHING
            if (i0, j0) in core_pos or (i0, j0) in res_pos:
                continue

            Bb = gather_block(R, i0, j0, bb)
            res_blocks.append((i0, j0, Bb))
            res_pos.add((i0, j0))
            added += 1
            found = True
            break

        if not found:
            break

        if (added % cfg.REFINE_RECHECK_EVERY) == 0:
            Xhat = reconstruct()
            err = frob_rel_err(Xhat, X)

    # final check
    Xhat = reconstruct()
    err = frob_rel_err(Xhat, X)
    return res_blocks

# ----------------------------
# Evaluation
# ----------------------------
@torch.no_grad()
def eval_payload(rt: PayloadRuntime, Ws: torch.Tensor, Sc: torch.Tensor):
    E, n, _ = Ws.shape
    errs = []
    for pos in range(E):
        x = torch.randn(8, n, dtype=DTYPE_ACC, device=DEVICE)
        y_hat = rt.apply_expert(x, pos)
        y_ref = x @ (Ws[pos] * Sc[pos])
        num = torch.linalg.norm(y_hat - y_ref)
        den = torch.linalg.norm(y_ref).clamp_min(1e-12)
        errs.append(float((num / den).item()))
    log(f"[eval] per-expert rel-error  mean={float(np.mean(errs)):.6f}  p95={float(np.percentile(errs,95)):.6f}  max={float(np.max(errs)):.6f}")

    mix = []
    for _ in range(cfg.EVAL_TRIALS):
        x = torch.randn(cfg.EVAL_BATCH, n, dtype=DTYPE_ACC, device=DEVICE)
        routed = random.sample(range(E), k=min(cfg.ROUTED_K, E))
        gates = torch.rand(len(routed), dtype=DTYPE_ACC, device=DEVICE)
        gates = gates / gates.sum().clamp_min(1e-12)
        y_hat = rt.apply_mixture(x, routed, gates)

        Wsum = torch.zeros(n, n, dtype=DTYPE_ACC, device=DEVICE)
        for a, pos in zip(gates, routed):
            Wsum += float(a.item()) * (Ws[int(pos)] * Sc[int(pos)])
        y_ref = x @ Wsum

        num = torch.linalg.norm(y_hat - y_ref)
        den = torch.linalg.norm(y_ref).clamp_min(1e-12)
        mix.append(float((num / den).item()))
    log(f"[eval] routed rel-error = {float(np.mean(mix)):.6f} ± {float(np.std(mix)):.6f}  (trials={cfg.EVAL_TRIALS})")


        # Optional: eval on calibration distribution (more meaningful than random Gaussian)
    if cfg.EVAL_ON_CALIB and cfg.CALIB_PATH and os.path.isfile(cfg.CALIB_PATH):
        try:
            Xcal = load_calib_X(cfg.CALIB_PATH, n).to(device=DEVICE, dtype=DTYPE_ACC)
            Xcal = Xcal[:min(int(cfg.EVAL_CALIB_ROWS), Xcal.shape[0])]
            if Xcal.shape[0] >= 32:
                errs_cal = []
                for pos in range(E):
                    xb = Xcal[torch.randperm(Xcal.shape[0], device=DEVICE)[:min(256, Xcal.shape[0])]]
                    y_hat = rt.apply_expert(xb, pos)
                    y_ref = xb @ (Ws[pos] * Sc[pos])
                    num = torch.linalg.norm(y_hat - y_ref)
                    den = torch.linalg.norm(y_ref).clamp_min(1e-12)
                    errs_cal.append(float((num / den).item()))
                log(f"[eval] calib-X rel-error  mean={float(np.mean(errs_cal)):.6f}  "
                    f"p95={float(np.percentile(errs_cal,95)):.6f}  max={float(np.max(errs_cal)):.6f}")
        except Exception as e:
            log(f"[eval] calib-X eval skipped: {e}")


# ----------------------------
# Main
# ----------------------------
def banner():
    log(f"== DeepSeek KT++-X OFFLINE v10.5 ==")
    log(f"Time:        {now()}")
    log(f"MODEL_DIR:   {cfg.MODEL_DIR}")
    log(f"OUTPUT_DIR:  {cfg.OUTPUT_DIR}")
    log(f"LAYER:       {cfg.LAYER}")
    log(f"MAX_EXPERTS:  {cfg.MAX_EXPERTS}")
    log(f"CALIB_PATH:  {cfg.CALIB_PATH or '(none)'}  CALIB_SAMPLES(cap)={cfg.CALIB_SAMPLES}")
    log(f"ROUTER_PATH: {cfg.ROUTER_PATH or '(none)'}  RIDGE_WEIGHTED={cfg.RIDGE_WEIGHTED}")
    log(f"RIDGE_DAMP:  {cfg.RIDGE_DAMP}  NORMALIZE_W={cfg.NORMALIZE_W}")
    log(f"BASIS_MODE:  {cfg.BASIS_MODE}")
    log(f"CLUSTER:     M0={(cfg.M0 if cfg.M0>0 else '(auto)')} M_MAX={cfg.M_MAX} min_size={cfg.CLUSTER_MIN_SIZE} max_size={cfg.CLUSTER_MAX_SIZE}")
    log(f"TRAIN:       steps={cfg.TRAIN_STEPS} warmup={cfg.TRAIN_WARMUP} lr={cfg.TRAIN_LR} subm={cfg.SUBM} batchE={cfg.BATCH_E}")
    log(f"CORE:        {cfg.CORE_MODE} block={cfg.CORE_BLOCK} target={cfg.CORE_TARGET} max_blocks={cfg.CORE_MAX_BLOCKS}")
    log(f"RESIDUAL:    rank={cfg.RES_RANK} coef={cfg.RES_COEF} blocks={cfg.RES_BLOCKS} bsize={cfg.RES_BSIZE}")
    log(f"REFINE:      enable={cfg.REFINE_ENABLE} err_target={cfg.REFINE_ERR_TARGET} max_extra={cfg.REFINE_MAX_EXTRA}")
    log(f"QUANT:       QMODE={cfg.QMODE}  BASIS_STORE_DTYPE={cfg.BASIS_STORE_DTYPE}")
    log(f"DEVICE:      {DEVICE}  Torch={torch.__version__} threads={NTHREADS}")
    log("")

def main():
    banner()

    expert_ids, Ws, Sc = load_or_build_Ws()
    E, n, _ = Ws.shape
    log(f"[Ws] shape={tuple(Ws.shape)}  (normalized={cfg.NORMALIZE_W})")

    if cfg.BASIS_MODE not in ("dense_train", "identity"):
        raise RuntimeError("v10.5 implements BASIS_MODE=dense_train or identity.")

    # Clusters
    if cfg.BASIS_MODE == "identity":
        clusters = [list(range(E))]
        cluster_of_pos = [0] * E
        log("[cluster] identity basis: M=1")
    else:
        Xfeat = random_proj_features(Ws, d=cfg.CLUSTER_FEAT_D)
        if cfg.M0 > 0:
            M0 = min(cfg.M0, E)
        else:
            M0 = int(round(2.0 * math.sqrt(E)))
            M0 = max(2, min(M0, E))
        labels = kmeans_torch(Xfeat, k=M0, iters=cfg.CLUSTER_ITERS, restarts=cfg.CLUSTER_RESTARTS)
        labels = merge_small_clusters(Xfeat, labels, min_size=cfg.CLUSTER_MIN_SIZE)
        labels = hierarchical_split(Xfeat, labels, max_size=cfg.CLUSTER_MAX_SIZE, max_k=min(cfg.M_MAX, E), split_iters=cfg.SPLIT_ITERS)
        labels = merge_small_clusters(Xfeat, labels, min_size=cfg.CLUSTER_MIN_SIZE)
        labels = relabel_contiguous(labels)
        M = int(labels.max().item()) + 1
        clusters = [torch.nonzero(labels == m, as_tuple=False).flatten().tolist() for m in range(M)]
        clusters = [c for c in clusters if len(c) > 0]
        log(f"[cluster] M={len(clusters)} sizes={[len(c) for c in clusters]}")
        cluster_of_pos = [0] * E
        for m, idx in enumerate(clusters):
            for pos in idx:
                cluster_of_pos[int(pos)] = m

    # Basis init (SVD mean per cluster)
    U_list: List[torch.Tensor] = []
    V_list: List[torch.Tensor] = []
    if cfg.BASIS_MODE == "identity":
        U_list = [torch.eye(n, dtype=DTYPE_ACC, device=DEVICE)]
        V_list = [torch.eye(n, dtype=DTYPE_ACC, device=DEVICE)]
    else:
        M = len(clusters)
        U_par: List[OrthoParam] = []
        V_par: List[OrthoParam] = []
        with torch.no_grad():
            for idx in clusters:
                Wm = Ws[idx].mean(dim=0)
                U0, V0 = svd_init_from_mean(Wm)
                U_par.append(OrthoParam(U0))
                V_par.append(OrthoParam(V0))

        # Train dense bases (on NORMALIZED Ws)
        if cfg.TRAIN_STEPS > 0:
            params = [p.M for p in U_par] + [p.M for p in V_par]
            opt = torch.optim.Adam(params, lr=cfg.TRAIN_LR)

            guidance_masks: Dict[int, torch.Tensor] = {}
            guidance_stats: Dict[int, Tuple[float, int]] = {}
            t0 = time.perf_counter()

            for step in range(1, cfg.TRAIN_STEPS + 1):
                S = torch.randperm(n, device=DEVICE)[:min(cfg.SUBM, n)]

                if cfg.TRAIN_LAM_GUIDE > 0 and (step == 1 or step % max(1, cfg.TRAIN_GUIDE_EVERY) == 0):
                    with torch.no_grad():
                        guidance_masks.clear()
                        guidance_stats.clear()
                        for m, idx in enumerate(clusters):
                            if len(idx) < cfg.TRAIN_MIN_CLUSTER:
                                continue
                            Uo = U_par[m].orthogonal()
                            Vo = V_par[m].orthogonal()
                            pick = idx
                            if 0 < cfg.BATCH_E < len(idx):
                                pidx = torch.randperm(len(idx), device=DEVICE)[:cfg.BATCH_E].tolist()
                                pick = [idx[i] for i in pidx]
                            Xs_ng = slice_X_batch(Ws[pick], Uo, Vo, S).detach()
                            mask, ef, kblk = make_guidance_mask_from_Xs(
                                Xs_ng.to(DTYPE_ACC),
                                block=cfg.CORE_BLOCK,
                                target=cfg.TRAIN_GUIDE_TARGET,
                                max_blocks=cfg.TRAIN_GUIDE_MAX_BLOCKS,
                            )
                            guidance_masks[m] = mask
                            guidance_stats[m] = (ef, kblk)

                lam_ramp = schedule(step, cfg.TRAIN_WARMUP, cfg.TRAIN_STEPS)
                lam_block = cfg.TRAIN_LAM_BLOCK * lam_ramp
                lam_guide = cfg.TRAIN_LAM_GUIDE * lam_ramp

                L_total = None
                n_terms = 0
                for m, idx in enumerate(clusters):
                    if len(idx) < cfg.TRAIN_MIN_CLUSTER:
                        continue

                    Uo = U_par[m].orthogonal()
                    Vo = V_par[m].orthogonal()

                    pick = idx
                    if 0 < cfg.BATCH_E < len(idx):
                        pidx = torch.randperm(len(idx), device=DEVICE)[:cfg.BATCH_E].tolist()
                        pick = [idx[i] for i in pidx]

                    Xs = slice_X_batch(Ws[pick], Uo, Vo, S)
                    off = offdiag_abs_mean(Xs)
                    diag = diag_abs_mean(Xs).clamp_min(1e-6)

                    if cfg.TRAIN_OBJ == "ratio":
                        base = off / diag
                    else:
                        base = torch.log(off + 1e-6) - torch.log(diag)

                    if lam_block > 0 and cfg.CORE_MODE.startswith("block"):
                        base = base + lam_block * block_group_sparsity_penalty(Xs, cfg.CORE_BLOCK)

                    if lam_guide > 0 and (m in guidance_masks):
                        Mmask = guidance_masks[m]
                        Etot = (Xs * Xs).mean().clamp_min(1e-12)
                        Eout = ((Xs * (1.0 - Mmask)) ** 2).mean()
                        base = base + lam_guide * (Eout / Etot)

                    L_total = base if (L_total is None) else (L_total + base)
                    n_terms += 1

                if L_total is None:
                    log("[train] skipped (no clusters >= TRAIN_MIN_CLUSTER)")
                    break

                L_total = L_total / max(1, n_terms)

                opt.zero_grad(set_to_none=True)
                L_total.backward()
                if cfg.GRAD_CLIP > 0:
                    torch.nn.utils.clip_grad_norm_(params, max_norm=cfg.GRAD_CLIP)
                opt.step()

                if (step % cfg.REORTHO_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                    with torch.no_grad():
                        for p in U_par:
                            p.M.copy_(p.orthogonal())
                        for p in V_par:
                            p.M.copy_(p.orthogonal())

                if step == 1 or (step % cfg.REPORT_EVERY) == 0 or step == cfg.TRAIN_STEPS:
                    t1 = time.perf_counter()
                    if guidance_stats:
                        ef_mean = float(np.mean([v[0] for v in guidance_stats.values()]))
                        kb_mean = float(np.mean([v[1] for v in guidance_stats.values()]))
                        log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} loss={float(L_total.item()):.4f} "
                            f"lam_block={lam_block:.3f} lam_guide={lam_guide:.3f} "
                            f"guide_energy≈{ef_mean:.3f} guide_blocks≈{kb_mean:.1f} (+{t1-t0:.1f}s)")
                    else:
                        log(f"[train] step {step:3d}/{cfg.TRAIN_STEPS} loss={float(L_total.item()):.4f} "
                            f"lam_block={lam_block:.3f} lam_guide={lam_guide:.3f} (+{t1-t0:.1f}s)")
                    t0 = t1

        for m in range(len(clusters)):
            U_list.append(U_par[m].orthogonal().detach().contiguous())
            V_list.append(V_par[m].orthogonal().detach().contiguous())

    # Build payload blocks
    log("[build] payloads ...")
    core_blocks_all: List[List[Tuple[int,int,torch.Tensor]]] = [[] for _ in range(E)]
    res_blocks_all: List[List[Tuple[int,int,torch.Tensor]]] = [[] for _ in range(E)]

    DL_list: List[torch.Tensor] = []
    DR_list: List[torch.Tensor] = []

    rmax = min(cfg.RES_RANK, n)
    if cfg.RES_COEF == "diag":
        gam = torch.zeros((E, rmax), dtype=DTYPE_ACC, device=DEVICE)
        Cfull = None
    else:
        gam = None
        Cfull = torch.zeros((E, rmax, rmax), dtype=DTYPE_ACC, device=DEVICE)

    for m, idx in enumerate(clusters):
        U = U_list[m] if cfg.BASIS_MODE != "identity" else U_list[0]
        V = V_list[m] if cfg.BASIS_MODE != "identity" else V_list[0]

        core_per, DL, DR, coef_list, res_per, efr = build_payload_for_cluster(Ws, idx, U, V)

        if cfg.REFINE_ENABLE:
            for j, pos in enumerate(idx):
                X = (U.t() @ Ws[pos] @ V).contiguous()
                res_per[j] = refine_expert_until(X, core_per[j], DL, DR, coef_list[j], res_per[j])

        for j, pos in enumerate(idx):
            core_blocks_all[pos] = core_per[j]
            res_blocks_all[pos] = res_per[j]
            if cfg.RES_COEF == "diag":
                g = coef_list[j]
                gam[pos, :g.numel()] = g
            else:
                C = coef_list[j]
                r = C.shape[0]
                Cfull[pos, :r, :r] = C

        DL_list.append(DL)
        DR_list.append(DR)

        nbm = float(np.mean([len(core_blocks_all[p]) for p in idx])) if idx else 0.0
        efm = float(np.mean(efr)) if len(efr) else 0.0
        efmin = float(np.min(efr)) if len(efr) else 0.0
        log(f"  - cluster{m}: E={len(idx)} core_blocks(mean)≈{nbm:.1f} core_energy(mean/min)≈{efm:.3f}/{efmin:.3f} r={DL.shape[1]}")

    # Serialize payload
    out_payload = os.path.join(cfg.OUTPUT_DIR, f"ktx_payload_layer{cfg.LAYER}_E{E}_{cfg.BASIS_MODE}_v10_5_q{cfg.QMODE}.npz")

    core_pack = pack_blocks_ragged(core_blocks_all, cfg.QMODE)
    res_pack = pack_blocks_ragged(res_blocks_all, cfg.QMODE)

    store_dtype = np.float16 if cfg.BASIS_STORE_DTYPE == "float16" else np.float32
    arrays: Dict[str, Any] = {}

    meta = ws_meta(expert_ids)
    meta.update({
        "time": now(),
        "basis_mode": cfg.BASIS_MODE,
        "qmode": cfg.QMODE,
        "basis_store_dtype": cfg.BASIS_STORE_DTYPE,
        "core_mode": cfg.CORE_MODE,
        "core_block": cfg.CORE_BLOCK,
        "core_target": cfg.CORE_TARGET,
        "core_max_blocks": cfg.CORE_MAX_BLOCKS,
        "res_rank": cfg.RES_RANK,
        "res_coef": cfg.RES_COEF,
        "res_blocks": cfg.RES_BLOCKS,
        "res_bsize": cfg.RES_BSIZE,
        "refine_enable": cfg.REFINE_ENABLE,
        "refine_err_target": cfg.REFINE_ERR_TARGET,
        "refine_max_extra": cfg.REFINE_MAX_EXTRA,
    })

    arrays["meta"] = _encode_meta(meta)
    arrays["expert_ids"] = np.array(expert_ids, dtype=np.int32)
    arrays["scales"] = Sc.detach().cpu().numpy().astype(np.float32)  # runtime applies Sc once
    arrays["cluster_of_pos"] = np.array(cluster_of_pos, dtype=np.int16)
    arrays["n_clusters"] = np.array([len(clusters)], dtype=np.int32)

    for m in range(len(clusters)):
        U = U_list[m] if cfg.BASIS_MODE != "identity" else U_list[0]
        V = V_list[m] if cfg.BASIS_MODE != "identity" else V_list[0]
        arrays[f"U_{m}"] = U.detach().cpu().numpy().astype(store_dtype)
        arrays[f"V_{m}"] = V.detach().cpu().numpy().astype(store_dtype)
        arrays[f"DL_{m}"] = DL_list[m].detach().cpu().numpy().astype(store_dtype)
        arrays[f"DR_{m}"] = DR_list[m].detach().cpu().numpy().astype(store_dtype)

    if cfg.RES_COEF == "diag":
        arrays["gam"] = gam.detach().cpu().numpy().astype(store_dtype)
    else:
        arrays["Cfull"] = Cfull.detach().cpu().numpy().astype(store_dtype)

    for k, v in core_pack.items():
        arrays["core_" + k] = v
    for k, v in res_pack.items():
        arrays["res_" + k] = v

    save_npz_compressed(out_payload, arrays)
    log(f"[save] payload -> {out_payload}  size={os.path.getsize(out_payload)/1e6:.2f} MB")

    # Load runtime + eval
    rt = load_payload_runtime(out_payload, DEVICE)
    eval_payload(rt, Ws, Sc)

    log("✅ Done.")
    log("Key fix in v10.5: payload built on W_norm, runtime multiplies by Sc ONCE (no Sc^2).")
    log("For 99–100% accuracy:")
    log("  - CALIB_SAMPLES=16384+ with real captured tokens (not just 160 rows).")
    log("  - CORE_TARGET 0.99–0.995, CORE_BLOCK 32/16, increase CORE_MAX_BLOCKS.")
    log("  - RES_RANK up, RES_COEF=full, RES_BLOCKS up, REFINE_ERR_TARGET ~0.01 with larger REFINE_MAX_EXTRA.")
    log("If you want v9-like behavior, try: CLUSTER_MIN_SIZE=1 and CLUSTER_MAX_SIZE=3.")

if __name__ == "__main__":
    main()


== DeepSeek KT++-X OFFLINE v10.5 ==
Time:        2026-01-13 12:48:46
MODEL_DIR:   /home/daniyar/deepseek-model
OUTPUT_DIR:  /home/daniyar/moe_ws_outputs
LAYER:       1
MAX_EXPERTS:  16
CALIB_PATH:  (none)  CALIB_SAMPLES(cap)=4096
ROUTER_PATH: (none)  RIDGE_WEIGHTED=False
RIDGE_DAMP:  0.001  NORMALIZE_W=True
BASIS_MODE:  dense_train
CLUSTER:     M0=(auto) M_MAX=16 min_size=2 max_size=4
TRAIN:       steps=24 warmup=6 lr=0.05 subm=256 batchE=4
CORE:        blocktopk_perexpert block=64 target=0.85 max_blocks=256
RESIDUAL:    rank=512 coef=diag blocks=192 bsize=64
REFINE:      enable=True err_target=0.05 max_extra=256
QUANT:       QMODE=none  BASIS_STORE_DTYPE=float16
DEVICE:      cpu  Torch=2.4.1+cpu threads=8

[found] layer=1 total_experts=64 using=16 eids=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
[cache] loaded Ws -> /home/daniyar/moe_ws_outputs/Ws_cache_layer1_E16_ridge_v10_5.npz  Ws=(16, 2048, 2048)
[Ws] shape=(16, 2048, 2048)  (normalized=True)
[cluster] M=4 sizes=[5, 3, 

In [1]:
#!/usr/bin/env python3
# ============================================================
# KTX++ HyperBlocks v0 (CRAZY MODE)
#
# Goal:
#   Compress your linearized expert matrices W_e (from ridge) MUCH harder
#   by not storing blocks directly, but generating them via a small hypernetwork:
#
#     block(e, bi, bj) = HyperNet( z_e , emb_i[bi], emb_j[bj] )
#
#   Then optionally store a tiny int8 residual for the worst blocks.
#
# What it still "uses" from your method:
#   - You still linearize each expert MLP into W_e (ridge Ws cache).
#   - You still approximate expert action y = x @ W_e.
#   - We only replace the storage format of W_e (block-implicit).
#
# File inputs:
#   - Ws_cache_layer{LAYER}_E{E}_ridge_v10_5.npz
#     keys: Ws (E,H,H float32), scales (E float32), expert_ids (E int32)
#
# Outputs:
#   - hyper_payload_layer{LAYER}_E{E}_b{B}.pt   (torch state: hypernet + latents)
#   - hyper_payload_layer{LAYER}_E{E}_b{B}.npz  (block indices + int8 residuals)
#
# Evaluate:
#   - per-expert rel error vs x @ (Ws[e] * Sc[e])
#   - routed mixture rel error
#
# Notes:
#   - This is intentionally "weird": implicit weight representation (INR-ish),
#     but for block matrices. It can compress insanely well if the structure exists.
#   - If it underfits, increase hidden size, latent dim, or add residual.
# ============================================================

import os, math, random, time, json
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------------------------
# Determinism / threads
# ----------------------------
def _env_int(k: str, d: int) -> int:
    try: return int(os.environ.get(k, d))
    except: return d

def _env_float(k: str, d: float) -> float:
    try: return float(os.environ.get(k, d))
    except: return d

def _env_str(k: str, d: str) -> str:
    return str(os.environ.get(k, d))

def _env_bool(k: str, d: bool) -> bool:
    v = os.environ.get(k, None)
    if v is None: return d
    return v.strip().lower() in ("1","true","yes","y","on")

SEED = _env_int("SEED", 1234)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

NTHREADS = _env_int("KTXX_THREADS", 8)
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
try:
    torch.set_num_threads(NTHREADS)
except:
    pass

DEVICE = torch.device(_env_str("DEVICE", "cuda" if torch.cuda.is_available() else "cpu"))
DTYPE = torch.float32

def log(msg: str):
    print(msg, flush=True)

def now():
    return time.strftime("%Y-%m-%d %H:%M:%S")

# ----------------------------
# Config
# ----------------------------
@dataclass
class Cfg:
    OUTPUT_DIR: str = _env_str("OUTPUT_DIR", "/home/daniyar/moe_ws_outputs")
    LAYER: int = _env_int("LAYER", 1)
    E: int = _env_int("MAX_EXPERTS", 16)  # must match the cache you created
    WS_CACHE: str = _env_str("WS_CACHE", "")  # optional explicit path

    # Blocking
    BLOCK: int = _env_int("BLOCK", 32)          # try 16/32
    CORE_TARGET: float = _env_float("CORE_TARGET", 0.98)  # energy coverage per expert
    MAX_BLOCKS_PER_EXPERT: int = _env_int("MAX_BLOCKS_PER_EXPERT", 1024)

    # HyperNet
    LATENT_D: int = _env_int("LATENT_D", 64)
    EMB_D: int = _env_int("EMB_D", 64)
    HIDDEN: int = _env_int("HIDDEN", 512)
    DEPTH: int = _env_int("DEPTH", 3)          # 2..5
    DROPOUT: float = _env_float("DROPOUT", 0.0)

    # Training
    STEPS: int = _env_int("STEPS", 20000)
    BATCH: int = _env_int("BATCH", 256)
    LR: float = _env_float("LR", 2e-4)
    WARMUP: int = _env_int("WARMUP", 1000)
    WEIGHT_DECAY: float = _env_float("WEIGHT_DECAY", 0.0)
    GRAD_CLIP: float = _env_float("GRAD_CLIP", 1.0)

    # Loss shaping
    ENERGY_WEIGHT_POWER: float = _env_float("ENERGY_WEIGHT_POWER", 0.5)  # sqrt-energy weight
    ENERGY_WEIGHT_CLIP: float = _env_float("ENERGY_WEIGHT_CLIP", 10.0)

    # Residual add-on (tiny int8)
    RESID_ENABLE: bool = _env_bool("RESID_ENABLE", True)
    RESID_KEEP_FRAC: float = _env_float("RESID_KEEP_FRAC", 0.05)  # keep worst 5% blocks residual
    RESID_MAX_PER_EXPERT: int = _env_int("RESID_MAX_PER_EXPERT", 64)
    RESID_QUANT: str = _env_str("RESID_QUANT", "int8")  # int8 only in this script

    # Eval
    EVAL_EVERY: int = _env_int("EVAL_EVERY", 2000)
    EVAL_TRIALS: int = _env_int("EVAL_TRIALS", 8)
    EVAL_BATCH: int = _env_int("EVAL_BATCH", 4)
    ROUTED_K: int = _env_int("ROUTED_K", 8)

cfg = Cfg()

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# ----------------------------
# Load Ws cache
# ----------------------------
def autodetect_ws_cache() -> str:
    cand = os.path.join(cfg.OUTPUT_DIR, f"Ws_cache_layer{cfg.LAYER}_E{cfg.E}_ridge_v10_5.npz")
    if os.path.isfile(cand):
        return cand
    raise FileNotFoundError("WS cache not found. Set WS_CACHE=... or put it in OUTPUT_DIR.")

def load_ws_cache(path: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    z = np.load(path, allow_pickle=False)
    need = ("Ws","scales","expert_ids")
    for k in need:
        if k not in z.files:
            raise KeyError(f"Missing {k} in {path}. keys={list(z.files)}")
    Ws = z["Ws"].astype(np.float32, copy=False)       # (E,H,H)
    Sc = z["scales"].astype(np.float32, copy=False)   # (E,)
    eids = z["expert_ids"].astype(np.int32, copy=False)
    z.close()
    return Ws, Sc, eids

ws_cache_path = cfg.WS_CACHE.strip() or autodetect_ws_cache()
log(f"[load] Ws cache: {ws_cache_path}")
Ws_np, Sc_np, eids_np = load_ws_cache(ws_cache_path)

E, H, H2 = Ws_np.shape
assert H == H2
assert E == cfg.E, f"Cache has E={E} but cfg.E={cfg.E}"
log(f"[Ws] E={E} H={H} normalized=True (cache)")

Ws = torch.from_numpy(Ws_np).to(device=DEVICE, dtype=DTYPE)  # normalized
Sc = torch.from_numpy(Sc_np).to(device=DEVICE, dtype=DTYPE)  # scales
expert_ids = eids_np.tolist()

# ----------------------------
# Block utilities
# ----------------------------
@torch.no_grad()
def block_energy_grid(W: torch.Tensor, b: int) -> Tuple[torch.Tensor, float, int]:
    n = W.shape[0]
    nb = (n + b - 1) // b
    if n % b != 0:
        Wp = torch.zeros(nb*b, nb*b, dtype=W.dtype, device=W.device)
        Wp[:n, :n] = W
        W = Wp
    Wb = W.view(nb, b, nb, b).permute(0,2,1,3).contiguous()  # (nb,nb,b,b)
    Eg = (Wb * Wb).sum(dim=(2,3))  # (nb,nb)
    tot = float((W * W).sum().item())
    return Eg, tot, nb

@torch.no_grad()
def pick_blocks_by_energy(Eg: torch.Tensor, tot: float, target: float, max_blocks: int) -> List[Tuple[int,int,float]]:
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    picked = []
    acc = 0.0
    nb = Eg.shape[0]
    for idx in order.tolist():
        if len(picked) >= max_blocks:
            break
        e = float(flat[idx].item())
        if e <= 1e-18:
            break
        bi = idx // nb
        bj = idx % nb
        picked.append((bi, bj, e))
        acc += e
        if acc / max(tot, 1e-12) >= target:
            break
    return picked

@torch.no_grad()
def gather_block(W: torch.Tensor, bi: int, bj: int, b: int) -> torch.Tensor:
    i0 = bi * b
    j0 = bj * b
    return W[i0:i0+b, j0:j0+b].contiguous()

# ----------------------------
# Build dataset: (eid, bi, bj) -> target block (flatten)
# ----------------------------
log("[data] selecting blocks per expert by energy ...")
b = cfg.BLOCK
blocks_e: List[List[Tuple[int,int,float]]] = []
nb_ref = None

for e in range(E):
    Eg, tot, nb = block_energy_grid(Ws[e], b)
    if nb_ref is None:
        nb_ref = nb
    picked = pick_blocks_by_energy(Eg, tot, cfg.CORE_TARGET, cfg.MAX_BLOCKS_PER_EXPERT)
    blocks_e.append(picked)

sizes = [len(x) for x in blocks_e]
log(f"[data] nb={nb_ref} block={b}  blocks/expert min/mean/max = {min(sizes)}/{sum(sizes)/len(sizes):.1f}/{max(sizes)}")

# Flatten dataset
eid_list = []
bi_list = []
bj_list = []
w_list  = []
tgt_list = []

log("[data] materializing target blocks (this can be a few seconds) ...")
with torch.no_grad():
    for e in range(E):
        for (bi, bj, energy) in blocks_e[e]:
            B = gather_block(Ws[e], bi, bj, b)                 # (b,b)
            tgt_list.append(B.reshape(-1).cpu().numpy())       # (b*b,)
            eid_list.append(e)
            bi_list.append(bi)
            bj_list.append(bj)
            w_list.append(energy)

tgt = torch.from_numpy(np.stack(tgt_list, axis=0)).to(device=DEVICE, dtype=DTYPE)  # (N, D)
eid_t = torch.tensor(eid_list, device=DEVICE, dtype=torch.long)
bi_t  = torch.tensor(bi_list, device=DEVICE, dtype=torch.long)
bj_t  = torch.tensor(bj_list, device=DEVICE, dtype=torch.long)

w = torch.tensor(w_list, device=DEVICE, dtype=DTYPE)
w = (w / (w.mean().clamp_min(1e-12))) ** cfg.ENERGY_WEIGHT_POWER
w = torch.clamp(w, 0.0, cfg.ENERGY_WEIGHT_CLIP).detach()  # (N,)

N, D = tgt.shape
log(f"[data] Nblocks={N}  D={D} (=b*b)")

# ----------------------------
# HyperNet definition
# ----------------------------
class HyperBlockNet(nn.Module):
    """
    block_vec = f( z_e , emb_i[bi], emb_j[bj] ) -> R^(b*b)
    """
    def __init__(self, E: int, nb: int, latent_d: int, emb_d: int, hidden: int, depth: int, out_d: int, dropout: float):
        super().__init__()
        self.latents = nn.Embedding(E, latent_d)
        self.emb_i   = nn.Embedding(nb, emb_d)
        self.emb_j   = nn.Embedding(nb, emb_d)

        in_d = latent_d + 2*emb_d
        layers = []
        dcur = in_d
        for k in range(max(1, depth)):
            layers.append(nn.Linear(dcur, hidden))
            layers.append(nn.GELU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            dcur = hidden
        layers.append(nn.Linear(dcur, out_d))
        self.mlp = nn.Sequential(*layers)

        # init: small output to stabilize early training
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
        nn.init.normal_(self.latents.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.emb_i.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.emb_j.weight, mean=0.0, std=0.02)

    def forward(self, eid: torch.Tensor, bi: torch.Tensor, bj: torch.Tensor) -> torch.Tensor:
        z = self.latents(eid)      # (B, latent_d)
        ei = self.emb_i(bi)        # (B, emb_d)
        ej = self.emb_j(bj)        # (B, emb_d)
        x = torch.cat([z, ei, ej], dim=-1)
        return self.mlp(x)         # (B, out_d)

nb = int(nb_ref)
net = HyperBlockNet(
    E=E, nb=nb,
    latent_d=cfg.LATENT_D,
    emb_d=cfg.EMB_D,
    hidden=cfg.HIDDEN,
    depth=cfg.DEPTH,
    out_d=D,
    dropout=cfg.DROPOUT,
).to(device=DEVICE, dtype=DTYPE)

# ----------------------------
# Training
# ----------------------------
log("[train] starting ...")
opt = torch.optim.AdamW(net.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)

def lr_schedule(step: int) -> float:
    if step <= cfg.WARMUP:
        return float(step) / float(max(1, cfg.WARMUP))
    return 1.0

@torch.no_grad()
def eval_operator_errors(net: HyperBlockNet, take_resid: bool, resid_pack: Optional[dict]) -> Tuple[float,float,float,float]:
    """
    Evaluate per-expert and routed mixture error vs true W_true = Ws*Sc.
    We reconstruct only the selected blocks (plus residual if enabled).
    Everything else is treated as zero -> this matches what we store.
    """
    # Build sparse block dict per expert
    # dict[e] = list of (i0,j0,B) on device
    sparse = [[] for _ in range(E)]
    net.eval()

    # Precompute residual access
    resid_map = None
    if take_resid and resid_pack is not None:
        resid_map = resid_pack  # contains arrays and ptrs

    # materialize predicted blocks
    BATCH_GEN = 4096
    idx = torch.arange(N, device=DEVICE)
    for s in range(0, N, BATCH_GEN):
        t = idx[s:s+BATCH_GEN]
        pred = net(eid_t[t], bi_t[t], bj_t[t]).view(-1, b, b)
        # add residual if available
        if resid_map is not None:
            # residual is stored only for a subset; we add it if present
            # We'll handle residual later in apply, not here (simpler).
            pass
        # push into sparse lists
        for k in range(pred.shape[0]):
            e = int(eid_t[t[k]].item())
            bi = int(bi_t[t[k]].item())
            bj = int(bj_t[t[k]].item())
            i0 = bi*b
            j0 = bj*b
            sparse[e].append((i0, j0, pred[k].detach()))

    # helper: get residual block if stored
    def get_resid_block(e: int, bi: int, bj: int) -> Optional[torch.Tensor]:
        if resid_map is None:
            return None
        # resid stored as per-expert ragged list with (bi,bj,q,scale)
        # arrays: exp_ptr, bi, bj, ptr, q, scale
        ep = resid_map["exp_ptr"]
        bbi = resid_map["bi"]
        bbj = resid_map["bj"]
        ptr = resid_map["ptr"]
        q = resid_map["q"]
        sc = resid_map["scale"]
        a0 = int(ep[e]); a1 = int(ep[e+1])
        # linear scan (small by design)
        for ii in range(a0, a1):
            if int(bbi[ii]) == bi and int(bbj[ii]) == bj:
                v0 = int(ptr[ii]); v1 = int(ptr[ii+1])
                qq = torch.from_numpy(q[v0:v1].astype(np.float32, copy=False)).to(DEVICE)
                scc = float(sc[ii])
                return (qq * scc).view(b, b)
        return None

    # apply sparse operator
    @torch.no_grad()
    def apply_sparse(x: torch.Tensor, e: int) -> torch.Tensor:
        # y = x @ W_hat where W_hat is sparse sum of predicted blocks (+ residual)
        y = torch.zeros_like(x)
        for (i0, j0, Bblk) in sparse[e]:
            # x[:, i0:i0+b] @ B -> add to y[:, j0:j0+b]
            y[:, j0:j0+b] += x[:, i0:i0+b] @ Bblk
            if resid_map is not None:
                bi = i0 // b
                bj = j0 // b
                Rb = get_resid_block(e, bi, bj)
                if Rb is not None:
                    y[:, j0:j0+b] += x[:, i0:i0+b] @ Rb
        # apply scale once to undo normalization
        y = y * Sc[e]
        return y

    # Per-expert error on random x
    per = []
    for e in range(E):
        x = torch.randn(8, H, device=DEVICE, dtype=DTYPE)
        y_hat = apply_sparse(x, e)
        y_ref = x @ (Ws[e] * Sc[e])  # true linearized op
        num = torch.linalg.norm(y_hat - y_ref)
        den = torch.linalg.norm(y_ref).clamp_min(1e-12)
        per.append(float((num/den).item()))
    per_mean = float(np.mean(per))
    per_p95  = float(np.percentile(per, 95))
    per_max  = float(np.max(per))

    # Routed mixture error
    mix = []
    for _ in range(cfg.EVAL_TRIALS):
        x = torch.randn(cfg.EVAL_BATCH, H, device=DEVICE, dtype=DTYPE)
        routed = random.sample(range(E), k=min(cfg.ROUTED_K, E))
        gates = torch.rand(len(routed), device=DEVICE, dtype=DTYPE)
        gates = gates / gates.sum().clamp_min(1e-12)

        y_hat = torch.zeros_like(x)
        for a, e in zip(gates.tolist(), routed):
            y_hat += float(a) * apply_sparse(x, int(e))

        Wsum = torch.zeros(H, H, device=DEVICE, dtype=DTYPE)
        for a, e in zip(gates, routed):
            Wsum += float(a.item()) * (Ws[int(e)] * Sc[int(e)])
        y_ref = x @ Wsum

        num = torch.linalg.norm(y_hat - y_ref)
        den = torch.linalg.norm(y_ref).clamp_min(1e-12)
        mix.append(float((num/den).item()))
    mix_mean = float(np.mean(mix))
    mix_std  = float(np.std(mix))
    return per_mean, per_p95, per_max, mix_mean  # (std omitted in print)

# Training loop
net.train()
best_loss = float("inf")
t0 = time.time()

for step in range(1, cfg.STEPS + 1):
    net.train()
    # mini-batch
    idx = torch.randint(0, N, (cfg.BATCH,), device=DEVICE)
    pred = net(eid_t[idx], bi_t[idx], bj_t[idx])  # (B, D)
    err = pred - tgt[idx]
    loss = ((err * err).mean(dim=1) * w[idx]).mean()

    # LR schedule
    lr_mult = lr_schedule(step)
    for pg in opt.param_groups:
        pg["lr"] = cfg.LR * lr_mult

    opt.zero_grad(set_to_none=True)
    loss.backward()
    if cfg.GRAD_CLIP > 0:
        torch.nn.utils.clip_grad_norm_(net.parameters(), cfg.GRAD_CLIP)
    opt.step()

    l = float(loss.item())
    if l < best_loss:
        best_loss = l

    if step == 1 or step % 200 == 0:
        dt = time.time() - t0
        log(f"[train] step {step:6d}/{cfg.STEPS} loss={l:.6e} best={best_loss:.6e} lr={cfg.LR*lr_mult:.2e} (+{dt:.1f}s)")
        t0 = time.time()

    if step % cfg.EVAL_EVERY == 0 or step == cfg.STEPS:
        # quick operator eval without residual first (residual built after training)
        per_mean, per_p95, per_max, mix_mean = eval_operator_errors(net, take_resid=False, resid_pack=None)
        log(f"[eval@{step}] per-expert relerr mean={per_mean:.6f} p95={per_p95:.6f} max={per_max:.6f} | routed mean={mix_mean:.6f}")

log("[train] done.")

# ----------------------------
# Build optional int8 residual (tiny) for worst blocks
# ----------------------------
def quant_int8_block(B: torch.Tensor) -> Tuple[np.ndarray, np.float16]:
    x = B.detach().cpu().to(torch.float32)
    maxabs = float(x.abs().max().item())
    if maxabs < 1e-12:
        q = np.zeros((x.numel(),), dtype=np.int8)
        return q, np.float16(1.0)
    scale = maxabs / 127.0
    q = torch.clamp(torch.round(x / scale), -127, 127).to(torch.int8).cpu().numpy().reshape(-1)
    return q.astype(np.int8, copy=False), np.float16(scale)

log("[pack] generating predictions for all selected blocks ...")
net.eval()
with torch.no_grad():
    pred_all = []
    BATCH_GEN = 4096
    for s in range(0, N, BATCH_GEN):
        t = slice(s, min(N, s + BATCH_GEN))
        pred = net(eid_t[t], bi_t[t], bj_t[t])
        pred_all.append(pred.detach())
    pred_all = torch.cat(pred_all, dim=0)  # (N, D)

resid_pack = None
if cfg.RESID_ENABLE:
    log("[resid] building tiny int8 residual for worst blocks ...")
    # residual per block vector
    resid = (tgt - pred_all).view(N, b, b)
    # measure per-block relative contribution using weighted norm
    rnorm = torch.sqrt((resid * resid).sum(dim=(1,2))).detach()  # (N,)
    # select worst per expert, capped
    # We'll create ragged arrays per expert.
    exp_ptr = [0]
    bi_keep = []
    bj_keep = []
    q_flat = []
    ptr = [0]
    scale_keep = []

    # group indices by expert
    idx_by_e: List[List[int]] = [[] for _ in range(E)]
    for i in range(N):
        idx_by_e[int(eid_list[i])].append(i)

    for e in range(E):
        ii = idx_by_e[e]
        if not ii:
            exp_ptr.append(exp_ptr[-1])
            continue
        rn = rnorm[ii]
        # number to keep
        k0 = int(math.ceil(len(ii) * cfg.RESID_KEEP_FRAC))
        k0 = max(0, min(k0, cfg.RESID_MAX_PER_EXPERT))
        if k0 <= 0:
            exp_ptr.append(exp_ptr[-1])
            continue
        top = torch.topk(rn, k=k0, largest=True).indices.tolist()
        keep_idx = [ii[t] for t in top]

        for j in keep_idx:
            q, sc = quant_int8_block(resid[j])
            bi_keep.append(int(bi_list[j]))
            bj_keep.append(int(bj_list[j]))
            q_flat.append(q)
            scale_keep.append(sc)
            ptr.append(ptr[-1] + q.size)

        exp_ptr.append(len(bi_keep))

    resid_pack = {
        "exp_ptr": np.array(exp_ptr, dtype=np.int32),   # (E+1,)
        "bi": np.array(bi_keep, dtype=np.int16),        # (M,)
        "bj": np.array(bj_keep, dtype=np.int16),        # (M,)
        "ptr": np.array(ptr, dtype=np.int64),           # (M+1,)
        "q": (np.concatenate(q_flat, axis=0).astype(np.int8, copy=False) if q_flat else np.zeros((0,), dtype=np.int8)),
        "scale": np.array(scale_keep, dtype=np.float16) # (M,)
    }

    log(f"[resid] kept blocks total={len(bi_keep)} (int8)  ptr_end={ptr[-1]} elems")
else:
    log("[resid] disabled.")

# ----------------------------
# Final evaluation with residual
# ----------------------------
per_mean, per_p95, per_max, mix_mean = eval_operator_errors(net, take_resid=True, resid_pack=resid_pack)
log(f"[final] per-expert relerr mean={per_mean:.6f} p95={per_p95:.6f} max={per_max:.6f} | routed mean={mix_mean:.6f}")

# ----------------------------
# Save payload
# ----------------------------
tag = f"layer{cfg.LAYER}_E{E}_b{b}_t{cfg.CORE_TARGET}_hd{cfg.HIDDEN}_ld{cfg.LATENT_D}"
pt_path  = os.path.join(cfg.OUTPUT_DIR, f"hyper_payload_{tag}.pt")
npz_path = os.path.join(cfg.OUTPUT_DIR, f"hyper_payload_{tag}.npz")

state = {
    "meta": {
        "time": now(),
        "seed": SEED,
        "layer": cfg.LAYER,
        "E": E,
        "H": H,
        "block": b,
        "core_target": cfg.CORE_TARGET,
        "max_blocks_per_expert": cfg.MAX_BLOCKS_PER_EXPERT,
        "latent_d": cfg.LATENT_D,
        "emb_d": cfg.EMB_D,
        "hidden": cfg.HIDDEN,
        "depth": cfg.DEPTH,
        "resid_enable": cfg.RESID_ENABLE,
        "resid_keep_frac": cfg.RESID_KEEP_FRAC,
        "resid_max_per_expert": cfg.RESID_MAX_PER_EXPERT,
        "ws_cache": ws_cache_path,
    },
    "expert_ids": expert_ids,
    "Sc": Sc.detach().cpu(),  # scales needed to undo normalization once
    "net": net.state_dict(),
}

torch.save(state, pt_path)
log(f"[save] {pt_path}  (torch payload)")

# save indices + targets coverage lists + residual pack
npz_dict = {
    "expert_ids": np.array(expert_ids, dtype=np.int32),
    "Sc": Sc.detach().cpu().numpy().astype(np.float32),
    "block": np.array([b], dtype=np.int32),
    "nb": np.array([nb], dtype=np.int32),
    "N": np.array([N], dtype=np.int64),
    "eid": np.array(eid_list, dtype=np.int16),
    "bi": np.array(bi_list, dtype=np.int16),
    "bj": np.array(bj_list, dtype=np.int16),
    "w": w.detach().cpu().numpy().astype(np.float32),
}
if resid_pack is not None:
    for k,v in resid_pack.items():
        npz_dict["resid_"+k] = v

np.savez_compressed(npz_path, **npz_dict)
log(f"[save] {npz_path}  (npz indices+residual) size={os.path.getsize(npz_path)/1e6:.2f} MB")

# ----------------------------
# Runtime loader (example usage)
# ----------------------------
RUNTIME_CODE = r"""
# Example runtime snippet:

import numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F

def load_hyper_payload(pt_path, npz_path, device="cuda"):
    st = torch.load(pt_path, map_location="cpu")
    z = np.load(npz_path, allow_pickle=False)
    meta = st["meta"]
    Sc = torch.from_numpy(z["Sc"]).to(device=device, dtype=torch.float32)
    block = int(z["block"][0])
    nb = int(z["nb"][0])
    eid = torch.from_numpy(z["eid"].astype(np.int64)).to(device=device)
    bi  = torch.from_numpy(z["bi"].astype(np.int64)).to(device=device)
    bj  = torch.from_numpy(z["bj"].astype(np.int64)).to(device=device)

    # Residual pack (optional)
    resid = None
    if "resid_exp_ptr" in z.files:
        resid = {
            "exp_ptr": z["resid_exp_ptr"],
            "bi": z["resid_bi"],
            "bj": z["resid_bj"],
            "ptr": z["resid_ptr"],
            "q": z["resid_q"],
            "scale": z["resid_scale"],
        }

    # Rebuild net
    class HyperBlockNet(nn.Module):
        def __init__(self, E, nb, latent_d, emb_d, hidden, depth, out_d):
            super().__init__()
            self.latents = nn.Embedding(E, latent_d)
            self.emb_i   = nn.Embedding(nb, emb_d)
            self.emb_j   = nn.Embedding(nb, emb_d)
            in_d = latent_d + 2*emb_d
            layers = []
            dcur = in_d
            for _ in range(max(1, depth)):
                layers += [nn.Linear(dcur, hidden), nn.GELU()]
                dcur = hidden
            layers += [nn.Linear(dcur, out_d)]
            self.mlp = nn.Sequential(*layers)
        def forward(self, eid, bi, bj):
            z = self.latents(eid)
            ei = self.emb_i(bi)
            ej = self.emb_j(bj)
            x = torch.cat([z,ei,ej], dim=-1)
            return self.mlp(x)

    E = meta["E"]
    out_d = block*block
    net = HyperBlockNet(E, nb, meta["latent_d"], meta["emb_d"], meta["hidden"], meta["depth"], out_d).to(device)
    net.load_state_dict(st["net"], strict=True)
    net.eval()

    return net, Sc, (eid,bi,bj), resid, block, nb

"""
log("[info] runtime loader snippet saved in RUNTIME_CODE string (see end of file).")
log("✅ Done.")
log("Try knobs:")
log("  BLOCK=16 CORE_TARGET=0.995 HIDDEN=768 DEPTH=4 LATENT_D=128 STEPS=50000")
log("  RESID_KEEP_FRAC=0.02 RESID_MAX_PER_EXPERT=128")
log("And crucially: train on REAL calibration Ws (16384+ rows in ridge stage).")


[load] Ws cache: /home/daniyar/moe_ws_outputs/Ws_cache_layer1_E16_ridge_v10_5.npz
[Ws] E=16 H=2048 normalized=True (cache)
[data] selecting blocks per expert by energy ...
[data] nb=64 block=32  blocks/expert min/mean/max = 228/974.2/1024
[data] materializing target blocks (this can be a few seconds) ...
[data] Nblocks=15588  D=1024 (=b*b)
[train] starting ...
[train] step      1/20000 loss=8.091168e-07 best=8.091168e-07 lr=2.00e-07 (+0.2s)
[train] step    200/20000 loss=7.620780e-07 best=3.240469e-07 lr=4.00e-05 (+2.7s)
[train] step    400/20000 loss=4.343422e-07 best=3.240469e-07 lr=8.00e-05 (+2.6s)
[train] step    600/20000 loss=5.276195e-07 best=3.240469e-07 lr=1.20e-04 (+2.5s)
[train] step    800/20000 loss=7.845034e-07 best=3.182174e-07 lr=1.60e-04 (+2.5s)
[train] step   1000/20000 loss=3.746913e-07 best=3.182174e-07 lr=2.00e-04 (+2.5s)
[train] step   1200/20000 loss=8.878997e-07 best=3.182174e-07 lr=2.00e-04 (+2.5s)
[train] step   1400/20000 loss=7.145345e-07 best=3.182174e-07 l

In [1]:
#!/usr/bin/env python3
# ============================================================
# GS-KT++ v0 : Graph-Spectral KT++ for MoE Expert Compression
#
# Key idea:
#   Compress across experts first using a graph Fourier transform
#   (Laplacian eigenvectors of an expert-similarity graph),
#   then compress only K spectral component matrices via KT++-style
#   block-sparse core + shared low-rank residual,
#   plus tiny per-expert sparse "patch" residuals.
#
# This is a research prototype for "break-the-limits" experiments.
# ============================================================

import os, re, json, math, time, random, sys
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any, Set

import numpy as np
import torch
import torch.nn.functional as F
from safetensors import safe_open

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):  # type: ignore
        return x

# ----------------------------
# Env helpers
# ----------------------------
def _env_str(k: str, d: str) -> str:
    return os.environ.get(k, d)

def _env_int(k: str, d: int) -> int:
    try:
        return int(os.environ.get(k, str(d)))
    except Exception:
        return d

def _env_float(k: str, d: float) -> float:
    try:
        return float(os.environ.get(k, str(d)))
    except Exception:
        return d

def _env_bool(k: str, d: bool) -> bool:
    v = os.environ.get(k, None)
    if v is None:
        return d
    return v.strip().lower() in ("1", "true", "yes", "y", "on")

# ----------------------------
# Threads / determinism
# ----------------------------
NTHREADS = _env_int("KTXX_THREADS", 8)
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(NTHREADS))
try:
    torch.set_num_threads(NTHREADS)
except Exception:
    pass

SEED = _env_int("SEED", 1234)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(_env_str("DEVICE", "cpu"))
DTYPE_ACC = torch.float32

def now() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")

def log(msg: str):
    print(msg, flush=True)

# ----------------------------
# Config
# ----------------------------
@dataclass
class Cfg:
    MODEL_DIR: str = _env_str("MODEL_DIR", "/home/daniyar/deepseek-model")
    OUTPUT_DIR: str = _env_str("OUTPUT_DIR", "/home/daniyar/moe_ws_outputs")

    LAYER: int = _env_int("LAYER", 1)
    MAX_EXPERTS: int = _env_int("MAX_EXPERTS", 16)

    # Calibration / router
    CALIB_PATH: str = _env_str("CALIB_PATH", "").strip()
    ROUTER_PATH: str = _env_str("ROUTER_PATH", "").strip()
    CALIB_SAMPLES: int = _env_int("CALIB_SAMPLES", 4096)
    RIDGE_DAMP: float = _env_float("RIDGE_DAMP", 1e-3)
    RIDGE_WEIGHTED: bool = _env_bool("RIDGE_WEIGHTED", False)
    ROUTER_EIDS_ARE_GLOBAL: bool = _env_bool("ROUTER_EIDS_ARE_GLOBAL", True)
    NORMALIZE_W: bool = _env_bool("NORMALIZE_W", True)

    # Capture
    CAPTURE_ENABLE: bool = _env_bool("CAPTURE_ENABLE", False)
    CAPTURE_FORCE: bool = _env_bool("CAPTURE_FORCE", False)
    CAPTURE_ITERS: int = _env_int("CAPTURE_ITERS", 32)
    CAPTURE_BATCH: int = _env_int("CAPTURE_BATCH", 1)
    CAPTURE_MAX_TOKENS: int = _env_int("CAPTURE_MAX_TOKENS", 1024)
    CAPTURE_TEXT: str = _env_str("CAPTURE_TEXT", "MoE calibration text. " * 4096)
    HF_TRUST_REMOTE_CODE: bool = _env_bool("HF_TRUST_REMOTE_CODE", True)
    HF_LOCAL_FILES_ONLY: bool = _env_bool("HF_LOCAL_FILES_ONLY", True)

    # Graph-spectral
    GRAPH_K: int = _env_int("GRAPH_K", 8)                 # keep K low-frequency components
    GRAPH_ALPHA: float = _env_float("GRAPH_ALPHA", 0.6)   # weight_sim vs router_sim mix
    GRAPH_KNN: int = _env_int("GRAPH_KNN", 8)             # sparsify adjacency (top-k per node); 0=full
    GRAPH_EPS: float = _env_float("GRAPH_EPS", 1e-6)

    # Component compression (KT++-style)
    BASIS_TRAIN: bool = _env_bool("BASIS_TRAIN", True)
    TRAIN_STEPS: int = _env_int("TRAIN_STEPS", 64)
    TRAIN_WARMUP: int = _env_int("TRAIN_WARMUP", 8)
    TRAIN_LR: float = _env_float("TRAIN_LR", 3e-2)
    SUBM: int = _env_int("SUBM", 256)
    GRAD_CLIP: float = _env_float("GRAD_CLIP", 1.0)
    CORE_BLOCK: int = _env_int("CORE_BLOCK", 64)
    CORE_TARGET: float = _env_float("CORE_TARGET", 0.90)
    CORE_MAX_BLOCKS: int = _env_int("CORE_MAX_BLOCKS", 512)

    RES_RANK: int = _env_int("RES_RANK", 512)
    RES_COEF: str = _env_str("RES_COEF", "diag").strip().lower()  # diag|full
    RES_BSIZE: int = _env_int("RES_BSIZE", 64)
    RES_TARGET: float = _env_float("RES_TARGET", 0.995)
    RES_MAX_BLOCKS: int = _env_int("RES_MAX_BLOCKS", 4096)

    REFINE_ENABLE: bool = _env_bool("REFINE_ENABLE", True)
    REFINE_ERR_TARGET: float = _env_float("REFINE_ERR_TARGET", 0.03)
    REFINE_MAX_EXTRA: int = _env_int("REFINE_MAX_EXTRA", 4096)
    REFINE_BSIZE: int = _env_int("REFINE_BSIZE", 64)
    REFINE_RECHECK_EVERY: int = _env_int("REFINE_RECHECK_EVERY", 64)

    # Patch (per expert residual after truncation to K comps)
    PATCH_BLOCK: int = _env_int("PATCH_BLOCK", 64)
    PATCH_TARGET: float = _env_float("PATCH_TARGET", 0.98)
    PATCH_MAX_BLOCKS: int = _env_int("PATCH_MAX_BLOCKS", 256)
    PATCH_MIN_BLOCK_ENERGY: float = _env_float("PATCH_MIN_BLOCK_ENERGY", 0.0)

    # Storage / quant
    QMODE: str = _env_str("QMODE", "none").strip().lower()  # none|float16|int8

    # Eval
    EVAL_TRIALS: int = _env_int("EVAL_TRIALS", 8)
    EVAL_BATCH: int = _env_int("EVAL_BATCH", 2)
    ROUTED_K: int = _env_int("ROUTED_K", 8)

cfg = Cfg()
PRESET = _env_str("PRESET", "").strip().lower()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

def _setdefault_env(k: str, v: str):
    if k not in os.environ:
        os.environ[k] = v

if PRESET == "maxacc":
    _setdefault_env("CALIB_SAMPLES", "32768")
    _setdefault_env("RIDGE_DAMP", "1e-2")
    _setdefault_env("GRAPH_K", "8")
    _setdefault_env("CORE_BLOCK", "32")
    _setdefault_env("CORE_TARGET", "0.995")
    _setdefault_env("CORE_MAX_BLOCKS", "8192")
    _setdefault_env("RES_RANK", "2048")
    _setdefault_env("RES_COEF", "full")
    _setdefault_env("RES_TARGET", "0.999")
    _setdefault_env("RES_MAX_BLOCKS", "32768")
    _setdefault_env("REFINE_ENABLE", "1")
    _setdefault_env("REFINE_ERR_TARGET", "0.01")
    _setdefault_env("REFINE_MAX_EXTRA", "65536")
    _setdefault_env("PATCH_TARGET", "0.995")
    _setdefault_env("PATCH_MAX_BLOCKS", "1024")
    _setdefault_env("TRAIN_STEPS", "128")
    _setdefault_env("TRAIN_LR", "0.02")
    cfg = Cfg()
elif PRESET == "compact":
    _setdefault_env("GRAPH_K", "6")
    _setdefault_env("CORE_BLOCK", "64")
    _setdefault_env("CORE_TARGET", "0.90")
    _setdefault_env("CORE_MAX_BLOCKS", "512")
    _setdefault_env("RES_RANK", "512")
    _setdefault_env("RES_COEF", "diag")
    _setdefault_env("RES_TARGET", "0.99")
    _setdefault_env("RES_MAX_BLOCKS", "4096")
    _setdefault_env("REFINE_ENABLE", "0")
    _setdefault_env("QMODE", "float16")
    _setdefault_env("PATCH_TARGET", "0.95")
    _setdefault_env("PATCH_MAX_BLOCKS", "128")
    cfg = Cfg()

# ----------------------------
# NPZ helpers
# ----------------------------
def save_npz_compressed(path: str, arrays: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez_compressed(path, **arrays)

def load_npz(path: str) -> Dict[str, np.ndarray]:
    z = np.load(path, allow_pickle=False)
    out = {k: z[k] for k in z.files}
    z.close()
    return out

def _encode_meta(meta: dict) -> np.ndarray:
    b = json.dumps(meta, sort_keys=True).encode("utf-8")
    return np.frombuffer(b, dtype=np.uint8)

def _decode_meta(arr: np.ndarray) -> dict:
    try:
        return json.loads(bytes(arr.tolist()).decode("utf-8"))
    except Exception:
        return {}

# ----------------------------
# Offline shard loading
# ----------------------------
def read_index(model_dir: str) -> Dict[str, str]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    wm = obj.get("weight_map", {})
    if not wm:
        raise RuntimeError("Index JSON has empty weight_map.")
    return wm

def find_layer_expert_ids(weight_map: Dict[str, str], layer: int) -> List[int]:
    pat = re.compile(rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.")
    ids = set()
    for k in weight_map.keys():
        m = pat.match(k)
        if m:
            ids.add(int(m.group(1)))
    return sorted(ids)

def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    prefix = f"model.layers.{layer}.mlp.experts.{eid}."
    def pick(cands: List[str]) -> Optional[str]:
        for suf in cands:
            k = prefix + suf
            if k in weight_map:
                return k
        return None
    up   = pick(["up_proj.weight", "w3.weight", "w1.weight"])
    gate = pick(["gate_proj.weight", "w1.weight", "w3.weight"])
    down = pick(["down_proj.weight", "w2.weight"])
    if up is None or gate is None or down is None:
        return {}
    if up == gate:
        g2 = pick(["gate_proj.weight"])
        if g2:
            gate = g2
    return {"up": up, "gate": gate, "down": down}

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], keys: List[str]) -> Dict[str, torch.Tensor]:
    by_shard: Dict[str, List[str]] = {}
    for k in keys:
        shard = weight_map.get(k, None)
        if shard is None:
            raise KeyError(f"Key not in weight_map: {k}")
        by_shard.setdefault(shard, []).append(k)

    out: Dict[str, torch.Tensor] = {}
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(sp):
            raise FileNotFoundError(f"Missing shard: {sp}")
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks:
                out[k] = f.get_tensor(k)
    return out

# ----------------------------
# Calibration load (optionally capture)
# ----------------------------
def autodetect_calib_path() -> Optional[str]:
    cand = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
    return cand if os.path.isfile(cand) else None

def autodetect_router_path() -> Optional[str]:
    cand = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
    return cand if os.path.isfile(cand) else None

def load_calib_X(path: str, H: int) -> torch.Tensor:
    z = np.load(path, allow_pickle=False)
    Xn = z["X"].astype(np.float32, copy=False)
    z.close()
    if Xn.ndim != 2 or Xn.shape[1] != H:
        raise RuntimeError(f"Bad X shape {tuple(Xn.shape)}, expected (*,{H})")
    if Xn.shape[0] > cfg.CALIB_SAMPLES:
        Xn = Xn[:cfg.CALIB_SAMPLES]
    return torch.from_numpy(Xn).to(device=DEVICE, dtype=DTYPE_ACC)

def load_router_P(path: str) -> np.ndarray:
    z = np.load(path, allow_pickle=False)
    P = z["P"].astype(np.float32, copy=False)
    z.close()
    return P

def capture_XP_transformers(model_dir: str, layer_idx: int, H: int, E_total: int,
                            out_x: str, out_p: str) -> Tuple[str, Optional[str]]:
    # minimal capture (hooks)
    from transformers import AutoTokenizer, AutoModelForCausalLM  # type: ignore

    tok = AutoTokenizer.from_pretrained(
        model_dir,
        trust_remote_code=cfg.HF_TRUST_REMOTE_CODE,
        local_files_only=cfg.HF_LOCAL_FILES_ONLY,
        use_fast=True,
    )
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token if tok.eos_token is not None else tok.unk_token

    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        trust_remote_code=cfg.HF_TRUST_REMOTE_CODE,
        local_files_only=cfg.HF_LOCAL_FILES_ONLY,
        torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
        device_map=None,
        low_cpu_mem_usage=True,
    ).to(DEVICE).eval()

    # find layers list
    layers = None
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        layers = list(model.model.layers)
    elif hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        layers = list(model.transformer.h)
    elif hasattr(model, "layers"):
        layers = list(model.layers)
    if layers is None:
        raise RuntimeError("Cannot locate transformer layers list.")
    layer = layers[layer_idx]

    mlp = getattr(layer, "mlp", None)
    if mlp is None:
        raise RuntimeError("Cannot find layer.mlp for capture.")

    # attempt to find router Linear
    router_linear = None
    for name, mod in layer.named_modules():
        if isinstance(mod, torch.nn.Linear) and getattr(mod, "in_features", None) == H and getattr(mod, "out_features", 0) >= E_total:
            if ("router" in name.lower()) or ("gate" in name.lower()) or ("moe" in name.lower()):
                router_linear = mod
                break

    X_buf = []
    P_buf = []
    nX = 0
    nP = 0
    attn_mask_holder = {"mask": None}

    def mlp_pre_hook(_m, inputs):
        nonlocal nX
        hs = inputs[0]
        if hs is None:
            return
        if hs.ndim == 2:
            hs = hs.unsqueeze(0)
        flat = hs.detach().to(torch.float32).reshape(-1, H).cpu()
        need = cfg.CALIB_SAMPLES - nX
        if need > 0:
            flat = flat[:need]
            X_buf.append(flat)
            nX += flat.shape[0]

    h1 = mlp.register_forward_pre_hook(mlp_pre_hook)

    h2 = None
    if router_linear is not None:
        def router_hook(_m, inputs, output):
            nonlocal nP
            out = output[0] if isinstance(output, (tuple, list)) else output
            if not torch.is_tensor(out):
                return
            if out.ndim == 2:
                out = out.unsqueeze(0)
            prob = torch.softmax(out.detach().to(torch.float32), dim=-1)[..., :E_total].reshape(-1, E_total).cpu()
            need = cfg.CALIB_SAMPLES - nP
            if need > 0:
                prob = prob[:need]
                P_buf.append(prob)
                nP += prob.shape[0]
        h2 = router_linear.register_forward_hook(router_hook)

    text = cfg.CAPTURE_TEXT
    for it in range(cfg.CAPTURE_ITERS):
        enc = tok(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=cfg.CAPTURE_MAX_TOKENS,
            padding="max_length",
        )
        for k in list(enc.keys()):
            if enc[k].ndim == 2 and cfg.CAPTURE_BATCH > 1:
                enc[k] = enc[k].repeat(cfg.CAPTURE_BATCH, 1)
        attn_mask_holder["mask"] = enc.get("attention_mask", None)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.inference_mode():
            _ = model(**enc, use_cache=False)
        log(f"[capture] iter {it+1}/{cfg.CAPTURE_ITERS} nX={nX} nP={nP}")
        if nX >= cfg.CALIB_SAMPLES and (not cfg.RIDGE_WEIGHTED or nP >= cfg.CALIB_SAMPLES):
            break

    h1.remove()
    if h2 is not None:
        h2.remove()

    if nX == 0:
        raise RuntimeError("Capture produced 0 rows.")

    X = torch.cat(X_buf, dim=0)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
    save_npz_compressed(out_x, {"X": X})
    p_written = None
    if nP > 0:
        P = torch.cat(P_buf, dim=0)[:min(nP, X.shape[0])].numpy().astype(np.float32)
        save_npz_compressed(out_p, {"P": P})
        p_written = out_p
    return out_x, p_written

def ensure_calib_router(H: int, E_total: int):
    if not cfg.CALIB_PATH:
        c = autodetect_calib_path()
        if c:
            cfg.CALIB_PATH = c
            log(f"[calib] auto-found {cfg.CALIB_PATH}")
    if not cfg.ROUTER_PATH:
        r = autodetect_router_path()
        if r:
            cfg.ROUTER_PATH = r
            log(f"[router] auto-found {cfg.ROUTER_PATH}")

    need_capture = cfg.CAPTURE_FORCE or (cfg.CAPTURE_ENABLE and (not cfg.CALIB_PATH or not os.path.isfile(cfg.CALIB_PATH)))
    if need_capture:
        out_x = os.path.join(cfg.OUTPUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
        out_p = os.path.join(cfg.OUTPUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
        log("[calib] capturing via transformers...")
        x_path, p_path = capture_XP_transformers(cfg.MODEL_DIR, cfg.LAYER, H, E_total, out_x, out_p)
        cfg.CALIB_PATH = x_path
        if p_path:
            cfg.ROUTER_PATH = p_path

# ----------------------------
# Ridge linearization (build Ws)
# ----------------------------
@torch.no_grad()
def forward_mlp(X: torch.Tensor, W_gate: torch.Tensor, W_up: torch.Tensor, W_down: torch.Tensor) -> torch.Tensor:
    Xf = X.to(DTYPE_ACC)
    up = Xf @ W_up.to(DTYPE_ACC).t()
    gate = Xf @ W_gate.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    y = hid @ W_down.to(DTYPE_ACC).t()
    return y

def ws_cache_path(E: int) -> str:
    return os.path.join(cfg.OUTPUT_DIR, f"Ws_cache_layer{cfg.LAYER}_E{E}_ridge_gsktpp_v0.npz")

def ws_meta(eids: List[int]) -> dict:
    return dict(
        script="gs_ktpp_v0",
        model_dir=cfg.MODEL_DIR,
        layer=cfg.LAYER,
        expert_ids=eids,
        ridge_damp=float(cfg.RIDGE_DAMP),
        ridge_weighted=bool(cfg.RIDGE_WEIGHTED),
        router_eids_are_global=bool(cfg.ROUTER_EIDS_ARE_GLOBAL),
        router_path=(cfg.ROUTER_PATH or ""),
        calib_path=(cfg.CALIB_PATH or ""),
        calib_samples=int(cfg.CALIB_SAMPLES),
        normalize_w=bool(cfg.NORMALIZE_W),
        seed=int(SEED),
        device=str(DEVICE),
    )

@torch.no_grad()
def build_Ws(eids: List[int], wm: Dict[str, str]) -> Tuple[torch.Tensor, torch.Tensor]:
    per_e = {}
    need_keys = []
    for eid in eids:
        kk = pick_expert_tensor_keys(wm, cfg.LAYER, eid)
        if not kk:
            raise RuntimeError(f"Expert {eid} missing tensors.")
        per_e[eid] = kk
        need_keys += [kk["up"], kk["down"], kk["gate"]]

    log("[load] reading tensors from shards ...")
    T = load_tensors_from_shards(cfg.MODEL_DIR, wm, sorted(set(need_keys)))

    W_up0 = T[per_e[eids[0]]["up"]]
    dff, H = int(W_up0.shape[0]), int(W_up0.shape[1])
    log(f"[shape] H={H} d_ff={dff}")

    ensure_calib_router(H=H, E_total=len(find_layer_expert_ids(wm, cfg.LAYER)))
    if not cfg.CALIB_PATH or not os.path.isfile(cfg.CALIB_PATH):
        raise RuntimeError("CALIB_PATH missing. Set CALIB_PATH or CAPTURE_ENABLE=1.")
    X = load_calib_X(cfg.CALIB_PATH, H)
    log(f"[calib] X: {tuple(X.shape)}  path={cfg.CALIB_PATH}")

    P = None
    if cfg.RIDGE_WEIGHTED:
        if cfg.ROUTER_PATH and os.path.isfile(cfg.ROUTER_PATH):
            P = load_router_P(cfg.ROUTER_PATH)
            log(f"[router] P: {tuple(P.shape)} path={cfg.ROUTER_PATH}")
        else:
            log("[router] RIDGE_WEIGHTED=1 but ROUTER_PATH missing -> disabling.")
            cfg.RIDGE_WEIGHTED = False

    Xf = X.to(DTYPE_ACC)
    I = torch.eye(H, dtype=DTYPE_ACC, device=DEVICE)
    XtX = Xf.t() @ Xf
    lam = cfg.RIDGE_DAMP * float(torch.trace(XtX).item()) / float(H)
    cholG = torch.linalg.cholesky(XtX + lam * I)

    Ws_list: List[torch.Tensor] = []
    scales: List[float] = []

    for i, eid in enumerate(tqdm(eids, desc="Build Ws (ridge)")):
        W_up = T[per_e[eid]["up"]].to(DEVICE)
        W_dn = T[per_e[eid]["down"]].to(DEVICE)
        W_gt = T[per_e[eid]["gate"]].to(DEVICE)

        Y = forward_mlp(X, W_gt, W_up, W_dn).to(DTYPE_ACC)

        if cfg.RIDGE_WEIGHTED and (P is not None):
            if cfg.ROUTER_EIDS_ARE_GLOBAL:
                w = torch.from_numpy(P[:X.shape[0], eid]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
            else:
                w = torch.from_numpy(P[:X.shape[0], i]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
            sw = torch.sqrt(w + 1e-12).view(-1, 1)
            Xw = Xf * sw
            Yw = Y * sw
            XtX_e = Xw.t() @ Xw
            lam_e = cfg.RIDGE_DAMP * float(torch.trace(XtX_e).item()) / float(H)
            chol = torch.linalg.cholesky(XtX_e + lam_e * I)
            XtY = Xw.t() @ Yw
            Wt = torch.cholesky_solve(XtY, chol)
            W = Wt.t().contiguous()
        else:
            XtY = Xf.t() @ Y
            Wt = torch.cholesky_solve(XtY, cholG)
            W = Wt.t().contiguous()

        if cfg.NORMALIZE_W:
            s = float(torch.linalg.norm(W, ord="fro").clamp_min(1e-12).item())
            W = (W / s).contiguous()
        else:
            s = 1.0

        Ws_list.append(W)
        scales.append(s)

    Ws = torch.stack(Ws_list, dim=0).to(DTYPE_ACC).to(DEVICE)  # (E,H,H) normalized
    Sc = torch.tensor(scales, dtype=DTYPE_ACC, device=DEVICE)  # (E,)
    return Ws, Sc

def load_or_build_Ws() -> Tuple[List[int], torch.Tensor, torch.Tensor]:
    wm = read_index(cfg.MODEL_DIR)
    all_eids = find_layer_expert_ids(wm, cfg.LAYER)
    if not all_eids:
        raise RuntimeError(f"No experts found at layer={cfg.LAYER}")
    eids = all_eids[:cfg.MAX_EXPERTS]
    log(f"[found] layer={cfg.LAYER} total={len(all_eids)} using={len(eids)} eids={eids}")

    if not cfg.CALIB_PATH:
        c = autodetect_calib_path()
        if c:
            cfg.CALIB_PATH = c
    if not cfg.ROUTER_PATH:
        r = autodetect_router_path()
        if r:
            cfg.ROUTER_PATH = r

    cpath = ws_cache_path(len(eids))
    if os.path.isfile(cpath):
        z = load_npz(cpath)
        if {"meta","Ws","expert_ids","scales"} <= set(z.keys()):
            if _decode_meta(z["meta"]) == ws_meta(eids):
                Ws = torch.from_numpy(z["Ws"]).to(DTYPE_ACC).to(DEVICE)
                Sc = torch.from_numpy(z["scales"]).to(DTYPE_ACC).to(DEVICE)
                log(f"[cache] loaded Ws -> {cpath} shape={tuple(Ws.shape)}")
                return [int(x) for x in z["expert_ids"].tolist()], Ws, Sc
        log("[cache] meta mismatch -> rebuild.")

    Ws, Sc = build_Ws(eids, wm)
    save_npz_compressed(cpath, {
        "meta": _encode_meta(ws_meta(eids)),
        "expert_ids": np.array(eids, dtype=np.int32),
        "Ws": Ws.detach().cpu().numpy().astype(np.float32),
        "scales": Sc.detach().cpu().numpy().astype(np.float32),
    })
    log(f"[cache] wrote Ws -> {cpath} size={os.path.getsize(cpath)/1e6:.2f} MB")
    return eids, Ws, Sc

# ----------------------------
# Expert similarity graph + Laplacian eigenvectors
# ----------------------------
@torch.no_grad()
def random_proj_features(Ws: torch.Tensor, d: int = 64) -> torch.Tensor:
    # Cheap feature: row/col energy profiles projected
    E, n, _ = Ws.shape
    g = torch.Generator(device="cpu").manual_seed(SEED + 17)
    R = (torch.randint(0, 2, (n, d), generator=g, dtype=torch.int8) * 2 - 1).to(DTYPE_ACC).to(DEVICE)
    feats = []
    for e in range(E):
        W = Ws[e]
        row = torch.diag(W @ W.t())
        col = torch.diag(W.t() @ W)
        feats.append(torch.cat([row @ R, col @ R], dim=0).unsqueeze(0))
    X = torch.cat(feats, dim=0)
    X = (X - X.mean(dim=0, keepdim=True)) / (X.std(dim=0, keepdim=True) + 1e-6)
    return X

@torch.no_grad()
def build_expert_adjacency(Ws: torch.Tensor, eids: List[int]) -> torch.Tensor:
    E = Ws.shape[0]
    # weight similarity
    Xf = random_proj_features(Ws, d=64)  # (E, 128)
    Xn = Xf / (torch.linalg.norm(Xf, dim=1, keepdim=True).clamp_min(cfg.GRAPH_EPS))
    A_w = (Xn @ Xn.t()).clamp_min(0.0)
    A_w.fill_diagonal_(0.0)

    # router similarity (optional)
    A_r = None
    if cfg.ROUTER_PATH and os.path.isfile(cfg.ROUTER_PATH):
        try:
            P = load_router_P(cfg.ROUTER_PATH)  # (N, E_total)
            # select columns for these eids
            cols = np.array(eids, dtype=np.int64)
            if cols.max() < P.shape[1]:
                Ps = P[:min(P.shape[0], cfg.CALIB_SAMPLES), :][:, cols]  # (N, E)
                C = Ps.T @ Ps
                # cosine-normalize
                d = np.sqrt(np.diag(C) + 1e-12)
                Cn = (C / (d[:, None] * d[None, :])).astype(np.float32)
                A_r = torch.from_numpy(Cn).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
                A_r.fill_diagonal_(0.0)
        except Exception as e:
            log(f"[graph] router sim skipped: {e}")

    if A_r is None:
        A = A_w
        log("[graph] adjacency = weight_sim only")
    else:
        a = float(cfg.GRAPH_ALPHA)
        A = (a * A_w + (1.0 - a) * A_r).clamp_min(0.0)
        A.fill_diagonal_(0.0)
        log(f"[graph] adjacency mix: alpha={a:.3f} weight_sim + router_sim")

    # optional KNN sparsify
    if cfg.GRAPH_KNN > 0 and cfg.GRAPH_KNN < E:
        K = int(cfg.GRAPH_KNN)
        A2 = torch.zeros_like(A)
        for i in range(E):
            vals = A[i]
            topk = torch.topk(vals, k=K, largest=True).indices
            A2[i, topk] = vals[topk]
        A = torch.maximum(A2, A2.t())  # symmetrize
        A.fill_diagonal_(0.0)
        log(f"[graph] KNN sparsify: K={K}")

    return A

@torch.no_grad()
def graph_fourier_basis(A: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    # Laplacian L = D - A
    D = torch.diag(A.sum(dim=1))
    L = D - A
    evals, evecs = torch.linalg.eigh(L)  # ascending
    return evals, evecs  # evecs columns are eigenvectors

# ----------------------------
# Block selection / packing
# ----------------------------
@torch.no_grad()
def block_energy_grid(X: torch.Tensor, b: int) -> Tuple[torch.Tensor, float, int]:
    n = X.shape[0]
    nb = (n + b - 1) // b
    if (n % b) != 0:
        Xp = torch.zeros(nb * b, nb * b, dtype=X.dtype, device=X.device)
        Xp[:n, :n] = X
        X = Xp
    Xb = X.view(nb, b, nb, b).permute(0, 2, 1, 3).contiguous()
    Eg = (Xb * Xb).sum(dim=(2, 3))
    tot = float((X * X).sum().item())
    return Eg, tot, nb

@torch.no_grad()
def pick_blocks_until_target(
    Eg: torch.Tensor,
    tot_energy: float,
    target: float,
    max_blocks: int,
    exclude: Optional[Set[Tuple[int, int]]] = None,
    min_block_energy: float = 0.0,
) -> Tuple[List[Tuple[int, int]], float]:
    nb = Eg.shape[0]
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)
    picked: List[Tuple[int, int]] = []
    eacc = 0.0
    exclude = exclude or set()

    for idx in order.tolist():
        if len(picked) >= max_blocks:
            break
        e = float(flat[idx].item())
        if e <= max(min_block_energy, 1e-18):
            break
        bi = idx // nb
        bj = idx % nb
        pos = (bi, bj)
        if pos in exclude:
            continue
        picked.append(pos)
        eacc += e
        if (eacc / max(tot_energy, 1e-12)) >= target:
            break

    eff = eacc / max(tot_energy, 1e-12)
    return picked, float(eff)

@torch.no_grad()
def gather_block(X: torch.Tensor, i0: int, j0: int, b: int) -> torch.Tensor:
    n = X.shape[0]
    i1 = min(n, i0 + b)
    j1 = min(n, j0 + b)
    return X[i0:i1, j0:j1].contiguous()

@torch.no_grad()
def frob_rel_err(A: torch.Tensor, B: torch.Tensor) -> float:
    num = torch.linalg.norm(A - B, ord="fro")
    den = torch.linalg.norm(B, ord="fro").clamp_min(1e-12)
    return float((num / den).item())

@torch.no_grad()
def q_block_int8(B: torch.Tensor) -> Tuple[np.ndarray, np.float16]:
    x = B.detach().cpu().to(torch.float32)
    maxabs = float(x.abs().max().item())
    if maxabs < 1e-12:
        return np.zeros_like(x.numpy(), dtype=np.int8), np.float16(1.0)
    scale = maxabs / 127.0
    q = torch.clamp(torch.round(x / scale), -127, 127).to(torch.int8).cpu().numpy()
    return q, np.float16(scale)

def _block_store_dtype_for_qmode(qmode: str) -> np.dtype:
    if qmode == "none":
        return np.float32
    return np.float16

def pack_blocks_ragged(blocks_per_item: List[List[Tuple[int, int, torch.Tensor]]], qmode: str) -> Dict[str, np.ndarray]:
    val_dtype = _block_store_dtype_for_qmode(qmode)
    N = len(blocks_per_item)
    item_ptr = [0]
    blk_i0, blk_j0, blk_h, blk_w = [], [], [], []
    blk_ptr = [0]
    vals = []
    vals_i8 = []
    scales = []

    for i in range(N):
        for (i0, j0, B) in blocks_per_item[i]:
            h, w = B.shape
            blk_i0.append(int(i0)); blk_j0.append(int(j0))
            blk_h.append(int(h)); blk_w.append(int(w))
            if qmode == "int8":
                q, sc = q_block_int8(B)
                vals_i8.append(q.reshape(-1))
                scales.append(sc)
                blk_ptr.append(blk_ptr[-1] + q.size)
            else:
                v = B.detach().cpu().to(torch.float32).numpy().astype(val_dtype, copy=False).reshape(-1)
                vals.append(v)
                blk_ptr.append(blk_ptr[-1] + v.size)
        item_ptr.append(len(blk_i0))

    out: Dict[str, np.ndarray] = {}
    out["item_ptr"] = np.array(item_ptr, dtype=np.int32)
    out["blk_i0"] = np.array(blk_i0, dtype=np.int16)
    out["blk_j0"] = np.array(blk_j0, dtype=np.int16)
    out["blk_h"]  = np.array(blk_h, dtype=np.int16)
    out["blk_w"]  = np.array(blk_w, dtype=np.int16)
    out["blk_ptr"] = np.array(blk_ptr, dtype=np.int64)

    if qmode == "int8":
        out["blk_q"] = np.concatenate(vals_i8, axis=0).astype(np.int8, copy=False) if vals_i8 else np.zeros((0,), dtype=np.int8)
        out["blk_scale"] = np.array(scales, dtype=np.float16)
    else:
        out["blk_val"] = np.concatenate(vals, axis=0) if vals else np.zeros((0,), dtype=val_dtype)
    return out

def unpack_blocks_ragged(pack: Dict[str, np.ndarray], qmode: str, device: torch.device) -> List[List[Tuple[int, int, torch.Tensor]]]:
    item_ptr = pack["item_ptr"].astype(np.int32)
    blk_i0 = pack["blk_i0"].astype(np.int32)
    blk_j0 = pack["blk_j0"].astype(np.int32)
    blk_h  = pack["blk_h"].astype(np.int32)
    blk_w  = pack["blk_w"].astype(np.int32)
    blk_ptr = pack["blk_ptr"].astype(np.int64)

    if qmode == "int8":
        blk_q = pack["blk_q"].astype(np.int8)
        blk_scale = pack["blk_scale"].astype(np.float16)
        blk_val = None
    else:
        blk_val = pack["blk_val"]
        blk_q = None
        blk_scale = None

    N = item_ptr.shape[0] - 1
    out: List[List[Tuple[int, int, torch.Tensor]]] = []
    for i in range(N):
        b0 = int(item_ptr[i]); b1 = int(item_ptr[i+1])
        lst = []
        for bi in range(b0, b1):
            i0 = int(blk_i0[bi]); j0 = int(blk_j0[bi])
            h = int(blk_h[bi]); w = int(blk_w[bi])
            v0 = int(blk_ptr[bi]); v1 = int(blk_ptr[bi+1])
            if qmode == "int8":
                q = blk_q[v0:v1].astype(np.int8, copy=False).astype(np.float32)
                sc = float(blk_scale[bi])
                B = torch.from_numpy((q * sc).reshape(h, w)).to(device=device, dtype=DTYPE_ACC)
            else:
                v = blk_val[v0:v1].astype(np.float32, copy=False)
                B = torch.from_numpy(v.reshape(h, w)).to(device=device, dtype=DTYPE_ACC)
            lst.append((i0, j0, B))
        out.append(lst)
    return out

# ----------------------------
# Shared low-rank helper
# ----------------------------
@torch.no_grad()
def rand_svd_vectors(A: torch.Tensor, r: int, n_iter: int = 2) -> Tuple[torch.Tensor, torch.Tensor]:
    n = A.shape[0]
    r = min(r, n)
    g = torch.Generator(device="cpu").manual_seed(SEED + 777)
    Omega = torch.randn(n, r, generator=g, dtype=DTYPE_ACC, device=A.device)
    Y = A @ Omega
    for _ in range(max(0, n_iter)):
        Y = A @ (A.t() @ Y)
    Q, _ = torch.linalg.qr(Y)
    B = Q.t() @ A
    Uhat, _, Vh = torch.linalg.svd(B, full_matrices=False)
    U = (Q @ Uhat[:, :r]).contiguous()
    V = (Vh.t()[:, :r]).contiguous()
    return U, V

# ----------------------------
# Basis training for components (one shared U,V)
# ----------------------------
class OrthoParam(torch.nn.Module):
    def __init__(self, init_mat: torch.Tensor):
        super().__init__()
        self.M = torch.nn.Parameter(init_mat.to(device=DEVICE, dtype=DTYPE_ACC).contiguous())

    def orthogonal(self) -> torch.Tensor:
        Q, _ = torch.linalg.qr(self.M)
        return Q

@torch.no_grad()
def svd_init_from_mean(Wmean: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    U, _, Vh = torch.linalg.svd(Wmean, full_matrices=False)
    return U.to(DTYPE_ACC).contiguous(), Vh.t().to(DTYPE_ACC).contiguous()

def schedule(step: int, warmup: int, total: int) -> float:
    if total <= 0:
        return 1.0
    if step <= warmup:
        return 0.0
    return float(min(1.0, max(0.0, (step - warmup) / max(1, (total - warmup)))))

def train_basis(mats: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    # mats: (K, n, n) in normalized space
    K, n, _ = mats.shape
    Wm = mats.mean(dim=0)
    U0, V0 = svd_init_from_mean(Wm)
    Up = OrthoParam(U0)
    Vp = OrthoParam(V0)

    if not cfg.BASIS_TRAIN or cfg.TRAIN_STEPS <= 0:
        return Up.orthogonal().detach(), Vp.orthogonal().detach()

    opt = torch.optim.Adam([Up.M, Vp.M], lr=cfg.TRAIN_LR)
    t0 = time.perf_counter()

    for step in range(1, cfg.TRAIN_STEPS + 1):
        S = torch.randperm(n, device=DEVICE)[:min(cfg.SUBM, n)]
        U = Up.orthogonal()
        V = Vp.orthogonal()
        # slice: compute U_S^T (W V_S)
        U_S = U[:, S]
        V_S = V[:, S]
        T = torch.matmul(mats, V_S)              # (K,n,s)
        Xs = torch.matmul(U_S.t().unsqueeze(0), T)  # (K,s,s)
        D = torch.diagonal(Xs, dim1=1, dim2=2)
        off = (Xs - torch.diag_embed(D)).abs().mean()
        diag = D.abs().mean().clamp_min(1e-6)
        base = torch.log(off + 1e-6) - torch.log(diag)

        lam = schedule(step, cfg.TRAIN_WARMUP, cfg.TRAIN_STEPS)
        loss = base * (1.0 + 0.0 * lam)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        if cfg.GRAD_CLIP > 0:
            torch.nn.utils.clip_grad_norm_([Up.M, Vp.M], max_norm=cfg.GRAD_CLIP)
        opt.step()

        if step % 8 == 0 or step == 1 or step == cfg.TRAIN_STEPS:
            t1 = time.perf_counter()
            log(f"[basis] step {step:4d}/{cfg.TRAIN_STEPS} loss={float(loss.item()):.4f} (+{t1-t0:.1f}s)")
            t0 = t1

        if step % 16 == 0 or step == cfg.TRAIN_STEPS:
            with torch.no_grad():
                Up.M.copy_(Up.orthogonal())
                Vp.M.copy_(Vp.orthogonal())

    return Up.orthogonal().detach(), Vp.orthogonal().detach()

# ----------------------------
# Compress component matrices with KT++-style core + shared low-rank + residual blocks
# ----------------------------
@torch.no_grad()
def compress_components(mats: torch.Tensor, U: torch.Tensor, V: torch.Tensor) -> Dict[str, Any]:
    # mats: (K,n,n) in normalized space
    K, n, _ = mats.shape
    Xs = torch.matmul(torch.matmul(U.t().unsqueeze(0), mats), V)  # (K,n,n)

    # core blocks per component
    b = int(cfg.CORE_BLOCK)
    core_blocks: List[List[Tuple[int,int,torch.Tensor]]] = []
    core_energy = []
    for k in range(K):
        Eg, te, nb = block_energy_grid(Xs[k], b)
        picks, eff = pick_blocks_until_target(Eg, te, cfg.CORE_TARGET, cfg.CORE_MAX_BLOCKS)
        blocks = []
        for (bi, bj) in picks:
            i0, j0 = bi*b, bj*b
            blocks.append((i0, j0, gather_block(Xs[k], i0, j0, b)))
        core_blocks.append(blocks)
        core_energy.append(eff)

    # residual after core
    R = []
    for k in range(K):
        Xc = torch.zeros_like(Xs[k])
        for (i0,j0,Bc) in core_blocks[k]:
            h,w = Bc.shape
            Xc[i0:i0+h, j0:j0+w] = Bc
        R.append((Xs[k] - Xc).contiguous())
    R = torch.stack(R, dim=0)

    # shared low-rank from mean residual
    Rmean = R.mean(dim=0)
    r = min(int(cfg.RES_RANK), n)
    DL, DR = rand_svd_vectors(Rmean, r=r, n_iter=2)

    coef_list = []
    R2_list = []
    for k in range(K):
        if cfg.RES_COEF == "diag":
            g = torch.sum(DL * (R[k] @ DR), dim=0).contiguous()
            coef_list.append(g)
            R2 = (R[k] - (DL * g.view(1, -1)) @ DR.t()).contiguous()
        else:
            C = (DL.t() @ R[k] @ DR).contiguous()
            coef_list.append(C)
            R2 = (R[k] - (DL @ C @ DR.t())).contiguous()
        R2_list.append(R2)
    R2 = torch.stack(R2_list, dim=0)

    # residual sparse blocks per component
    bb = int(cfg.RES_BSIZE)
    res_blocks: List[List[Tuple[int,int,torch.Tensor]]] = []
    res_energy = []
    for k in range(K):
        Eg, te, nb = block_energy_grid(R2[k], bb)
        exclude = set()
        # exclude blocks already used in core (mapped to bb grid)
        for (ci0, cj0, _Bc) in core_blocks[k]:
            exclude.add((ci0 // bb, cj0 // bb))
        picks, eff = pick_blocks_until_target(
            Eg, te, cfg.RES_TARGET, cfg.RES_MAX_BLOCKS,
            exclude=exclude, min_block_energy=0.0
        )
        blocks = []
        for (bi, bj) in picks:
            i0, j0 = bi*bb, bj*bb
            blocks.append((i0, j0, gather_block(R2[k], i0, j0, bb)))
        res_blocks.append(blocks)
        res_energy.append(eff)

    # optional refine: add blocks until err target in basis-space
    if cfg.REFINE_ENABLE:
        rb = int(cfg.REFINE_BSIZE)
        for k in range(K):
            def reconstruct() -> torch.Tensor:
                Xc = torch.zeros_like(Xs[k])
                for (i0,j0,Bc) in core_blocks[k]:
                    h,w = Bc.shape
                    Xc[i0:i0+h, j0:j0+w] = Bc
                if cfg.RES_COEF == "diag":
                    g = coef_list[k]
                    Xlr = (DL * g.view(1, -1)) @ DR.t()
                else:
                    C = coef_list[k]
                    Xlr = DL @ C @ DR.t()
                Xr = torch.zeros_like(Xs[k])
                for (i0,j0,Bb) in res_blocks[k]:
                    h,w = Bb.shape
                    Xr[i0:i0+h, j0:j0+w] += Bb
                return (Xc + Xlr + Xr).contiguous()

            Xhat = reconstruct()
            err = frob_rel_err(Xhat, Xs[k])
            added = 0
            core_pos = {(i0,j0) for (i0,j0,_) in core_blocks[k]}
            res_pos = {(i0,j0) for (i0,j0,_) in res_blocks[k]}

            while err > cfg.REFINE_ERR_TARGET and added < cfg.REFINE_MAX_EXTRA:
                Rerr = (Xs[k] - Xhat).contiguous()
                Eg, te, nb = block_energy_grid(Rerr, rb)
                flat = Eg.reshape(-1)
                if float(flat.max().item()) <= 1e-18:
                    break
                order = torch.argsort(flat, descending=True)
                found = False
                for idx in order.tolist():
                    bi = idx // nb; bj = idx % nb
                    i0 = bi*rb; j0 = bj*rb
                    if (i0,j0) in core_pos or (i0,j0) in res_pos:
                        continue
                    Bb = gather_block(Rerr, i0, j0, rb)
                    res_blocks[k].append((i0,j0,Bb))
                    res_pos.add((i0,j0))
                    added += 1
                    found = True
                    break
                if not found:
                    break
                if added % cfg.REFINE_RECHECK_EVERY == 0:
                    Xhat = reconstruct()
                    err = frob_rel_err(Xhat, Xs[k])
            Xhat = reconstruct()
            err = frob_rel_err(Xhat, Xs[k])
            log(f"[refine] comp{k} err={err:.6f} blocks_core={len(core_blocks[k])} blocks_res={len(res_blocks[k])}")

    return {
        "U": U, "V": V, "DL": DL, "DR": DR,
        "core_blocks": core_blocks,
        "res_blocks": res_blocks,
        "coef_list": coef_list,
        "core_energy": core_energy,
        "res_energy": res_energy,
    }

# ----------------------------
# Patch blocks per expert (residual after truncation)
# ----------------------------
@torch.no_grad()
def build_patches(Ws: torch.Tensor, Qk: torch.Tensor, comps: torch.Tensor, U: torch.Tensor, V: torch.Tensor) -> List[List[Tuple[int,int,torch.Tensor]]]:
    # Ws: (E,n,n) normalized
    # comps: (K,n,n) normalized in original space (not basis space)
    # Qk: (E,K)
    E, n, _ = Ws.shape
    K = comps.shape[0]
    # reconstruct low-freq approx in original space: What_low[e] = sum_k Qk[e,k] * comps[k]
    What = torch.einsum("ek,knm->enm", Qk, comps)  # (E,n,n)
    R = (Ws - What).contiguous()

    # move to basis space for block picking
    Xr = torch.matmul(torch.matmul(U.t().unsqueeze(0), R), V)  # (E,n,n)

    bb = int(cfg.PATCH_BLOCK)
    patches: List[List[Tuple[int,int,torch.Tensor]]] = []
    for e in range(E):
        Eg, te, nb = block_energy_grid(Xr[e], bb)
        picks, eff = pick_blocks_until_target(
            Eg, te, cfg.PATCH_TARGET, cfg.PATCH_MAX_BLOCKS,
            exclude=None,
            min_block_energy=cfg.PATCH_MIN_BLOCK_ENERGY
        )
        blocks = []
        for (bi,bj) in picks:
            i0, j0 = bi*bb, bj*bb
            blocks.append((i0,j0,gather_block(Xr[e], i0, j0, bb)))
        patches.append(blocks)
    return patches

# ----------------------------
# Runtime
# ----------------------------
class PayloadRuntime:
    def __init__(self):
        self.meta: dict = {}
        self.expert_ids: List[int] = []
        self.scales: Optional[torch.Tensor] = None  # (E,)
        self.Qk: Optional[torch.Tensor] = None      # (E,K)

        self.U: Optional[torch.Tensor] = None
        self.V: Optional[torch.Tensor] = None
        self.DL: Optional[torch.Tensor] = None
        self.DR: Optional[torch.Tensor] = None
        self.res_coef: str = "diag"

        self.gam: Optional[torch.Tensor] = None     # (K,r)
        self.Cfull: Optional[torch.Tensor] = None   # (K,r,r)

        self.core_blocks: List[List[Tuple[int,int,torch.Tensor]]] = []  # K items
        self.res_blocks: List[List[Tuple[int,int,torch.Tensor]]] = []   # K items
        self.patch_blocks: List[List[Tuple[int,int,torch.Tensor]]] = [] # E items

        self.qmode: str = "none"

    @torch.no_grad()
    def apply_components(self, x: torch.Tensor) -> torch.Tensor:
        # returns Ycomp: (K, B, n) in original space (normalized)
        assert self.U is not None and self.V is not None and self.DL is not None and self.DR is not None
        U, V, DL, DR = self.U, self.V, self.DL, self.DR
        K = len(self.core_blocks)
        z = x @ U  # (B,n)
        Y = []
        for k in range(K):
            u = torch.zeros_like(z)
            for (i0,j0,Bb) in self.core_blocks[k]:
                h,w = Bb.shape
                u[:, j0:j0+w] += z[:, i0:i0+h] @ Bb
            if self.res_coef == "diag":
                g = self.gam[k]  # type: ignore[index]
                u += ((z @ DL) * g.view(1, -1)) @ DR.t()
            else:
                C = self.Cfull[k]  # type: ignore[index]
                u += (z @ DL) @ C @ DR.t()
            for (i0,j0,Bb) in self.res_blocks[k]:
                h,w = Bb.shape
                u[:, j0:j0+w] += z[:, i0:i0+h] @ Bb
            yk = u @ V.t()
            Y.append(yk)
        return torch.stack(Y, dim=0)  # (K,B,n)

    @torch.no_grad()
    def apply_patch(self, x: torch.Tensor, epos: int) -> torch.Tensor:
        # patch is stored in basis space, so apply similarly with only blocks
        assert self.U is not None and self.V is not None
        z = x @ self.U
        u = torch.zeros_like(z)
        for (i0,j0,Bb) in self.patch_blocks[epos]:
            h,w = Bb.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ Bb
        return u @ self.V.t()

    @torch.no_grad()
    def apply_expert(self, x: torch.Tensor, epos: int, Ycomp: Optional[torch.Tensor] = None) -> torch.Tensor:
        assert self.Qk is not None and self.scales is not None
        if Ycomp is None:
            Ycomp = self.apply_components(x)  # (K,B,n)
        qe = self.Qk[epos]  # (K,)
        # combine components
        y = torch.einsum("k,kbn->bn", qe, Ycomp)
        # add patch
        y += self.apply_patch(x, epos)
        # undo normalization scale
        y = y * self.scales[epos]
        return y

    @torch.no_grad()
    def apply_mixture(self, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
        Ycomp = self.apply_components(x)
        y = torch.zeros_like(x)
        for a, epos in zip(gates.tolist(), routed):
            y += float(a) * self.apply_expert(x, int(epos), Ycomp=Ycomp)
        return y

def load_payload_runtime(path: str, device: torch.device) -> PayloadRuntime:
    z = load_npz(path)
    rt = PayloadRuntime()
    rt.meta = _decode_meta(z["meta"])
    rt.qmode = rt.meta.get("qmode", "none")
    rt.res_coef = rt.meta.get("res_coef", "diag")

    rt.expert_ids = [int(x) for x in z["expert_ids"].tolist()]
    rt.scales = torch.from_numpy(z["scales"]).to(device=device, dtype=DTYPE_ACC)
    rt.Qk = torch.from_numpy(z["Qk"]).to(device=device, dtype=DTYPE_ACC)

    rt.U = torch.from_numpy(z["U"]).to(device=device, dtype=DTYPE_ACC)
    rt.V = torch.from_numpy(z["V"]).to(device=device, dtype=DTYPE_ACC)
    rt.DL = torch.from_numpy(z["DL"]).to(device=device, dtype=DTYPE_ACC)
    rt.DR = torch.from_numpy(z["DR"]).to(device=device, dtype=DTYPE_ACC)

    if rt.res_coef == "diag":
        rt.gam = torch.from_numpy(z["gam"]).to(device=device, dtype=DTYPE_ACC)
    else:
        rt.Cfull = torch.from_numpy(z["Cfull"]).to(device=device, dtype=DTYPE_ACC)

    # blocks
    core_pack = {k.replace("core_", ""): z[k] for k in z.keys() if k.startswith("core_")}
    res_pack  = {k.replace("res_", ""): z[k] for k in z.keys() if k.startswith("res_")}
    pat_pack  = {k.replace("pat_", ""): z[k] for k in z.keys() if k.startswith("pat_")}

    rt.core_blocks = unpack_blocks_ragged(core_pack, rt.qmode, device)
    rt.res_blocks  = unpack_blocks_ragged(res_pack, rt.qmode, device)
    rt.patch_blocks= unpack_blocks_ragged(pat_pack, rt.qmode, device)

    return rt

# ----------------------------
# Eval
# ----------------------------
@torch.no_grad()
def eval_payload(rt: PayloadRuntime, Ws: torch.Tensor, Sc: torch.Tensor):
    E, n, _ = Ws.shape
    errs = []
    Ycomp = None
    for e in range(E):
        x = torch.randn(8, n, dtype=DTYPE_ACC, device=DEVICE)
        if Ycomp is None:
            Ycomp = rt.apply_components(x)
        y_hat = rt.apply_expert(x, e, Ycomp=Ycomp)
        y_ref = x @ (Ws[e] * Sc[e])
        num = torch.linalg.norm(y_hat - y_ref)
        den = torch.linalg.norm(y_ref).clamp_min(1e-12)
        errs.append(float((num / den).item()))
    log(f"[eval] per-expert rel-error mean={float(np.mean(errs)):.6f} p95={float(np.percentile(errs,95)):.6f} max={float(np.max(errs)):.6f}")

    mix = []
    for _ in range(cfg.EVAL_TRIALS):
        x = torch.randn(cfg.EVAL_BATCH, n, dtype=DTYPE_ACC, device=DEVICE)
        routed = random.sample(range(E), k=min(cfg.ROUTED_K, E))
        gates = torch.rand(len(routed), dtype=DTYPE_ACC, device=DEVICE)
        gates = gates / gates.sum().clamp_min(1e-12)
        y_hat = rt.apply_mixture(x, routed, gates)

        Wsum = torch.zeros(n, n, dtype=DTYPE_ACC, device=DEVICE)
        for a, pos in zip(gates, routed):
            Wsum += float(a.item()) * (Ws[int(pos)] * Sc[int(pos)])
        y_ref = x @ Wsum
        num = torch.linalg.norm(y_hat - y_ref)
        den = torch.linalg.norm(y_ref).clamp_min(1e-12)
        mix.append(float((num / den).item()))
    log(f"[eval] routed rel-error mean={float(np.mean(mix)):.6f} ± {float(np.std(mix)):.6f} trials={cfg.EVAL_TRIALS}")

# ----------------------------
# Main
# ----------------------------
def banner():
    log("== GS-KT++ v0 ==")
    log(f"Time:      {now()}")
    log(f"MODEL_DIR: {cfg.MODEL_DIR}")
    log(f"OUT_DIR:   {cfg.OUTPUT_DIR}")
    log(f"LAYER:     {cfg.LAYER}")
    log(f"EXPERTS:   {cfg.MAX_EXPERTS}")
    log(f"CALIB:     {cfg.CALIB_PATH or '(none)'} cap={cfg.CALIB_SAMPLES} capture={cfg.CAPTURE_ENABLE}")
    log(f"GRAPH:     K={cfg.GRAPH_K} alpha={cfg.GRAPH_ALPHA} knn={cfg.GRAPH_KNN}")
    log(f"CORE:      block={cfg.CORE_BLOCK} target={cfg.CORE_TARGET} max_blocks={cfg.CORE_MAX_BLOCKS}")
    log(f"RES:       rank={cfg.RES_RANK} coef={cfg.RES_COEF} target={cfg.RES_TARGET} max_blocks={cfg.RES_MAX_BLOCKS} bsize={cfg.RES_BSIZE}")
    log(f"PATCH:     block={cfg.PATCH_BLOCK} target={cfg.PATCH_TARGET} max_blocks={cfg.PATCH_MAX_BLOCKS}")
    log(f"REFINE:    enable={cfg.REFINE_ENABLE} err={cfg.REFINE_ERR_TARGET} max_extra={cfg.REFINE_MAX_EXTRA}")
    log(f"QMODE:     {cfg.QMODE}")
    log(f"DEVICE:    {DEVICE} torch={torch.__version__} threads={NTHREADS}")
    log("")

def main():
    banner()
    expert_ids, Ws, Sc = load_or_build_Ws()
    E, n, _ = Ws.shape
    log(f"[Ws] shape={tuple(Ws.shape)} normalized={cfg.NORMALIZE_W}")

    # Build expert graph
    A = build_expert_adjacency(Ws, expert_ids)
    evals, Q = graph_fourier_basis(A)  # Q: (E,E)
    # Keep K low-frequency components (smallest eigenvalues)
    K = max(1, min(int(cfg.GRAPH_K), E))
    Qk = Q[:, :K].contiguous()
    log(f"[graph] kept K={K} low-freq components; evals[0..K-1]={evals[:K].detach().cpu().numpy()}")

    # Graph transform -> component matrices
    # comps[k] = sum_e Q[e,k] * Ws[e]
    comps = torch.einsum("ek,enm->knm", Qk, Ws).contiguous()  # (K,n,n)

    # Train shared basis U,V on comps (optional)
    log("[basis] training shared basis on spectral components ...")
    U, V = train_basis(comps)

    # Compress spectral components
    log("[compress] compressing spectral components ...")
    comp_payload = compress_components(comps, U, V)

    # Build patches
    log("[patch] building per-expert patches (truncate->residual) ...")
    patches = build_patches(Ws, Qk, comps, comp_payload["U"], comp_payload["V"])

    # Pack blocks
    core_pack = pack_blocks_ragged(comp_payload["core_blocks"], cfg.QMODE)  # K items
    res_pack  = pack_blocks_ragged(comp_payload["res_blocks"], cfg.QMODE)   # K items
    pat_pack  = pack_blocks_ragged(patches, cfg.QMODE)                     # E items

    # Coefs
    if cfg.RES_COEF == "diag":
        r = comp_payload["DL"].shape[1]
        gam = torch.stack([g[:r] for g in comp_payload["coef_list"]], dim=0)  # (K,r)
        Cfull = None
    else:
        r = comp_payload["DL"].shape[1]
        Cfull = torch.stack([C[:r, :r] for C in comp_payload["coef_list"]], dim=0)  # (K,r,r)
        gam = None

    # Save payload
    out_path = os.path.join(cfg.OUTPUT_DIR, f"gsktpp_payload_layer{cfg.LAYER}_E{E}_K{K}_q{cfg.QMODE}.npz")
    meta = ws_meta(expert_ids)
    meta.update({
        "time": now(),
        "graph_k": K,
        "graph_alpha": float(cfg.GRAPH_ALPHA),
        "graph_knn": int(cfg.GRAPH_KNN),
        "qmode": cfg.QMODE,
        "res_coef": cfg.RES_COEF,
        "core_block": cfg.CORE_BLOCK,
        "core_target": cfg.CORE_TARGET,
        "res_rank": cfg.RES_RANK,
        "res_target": cfg.RES_TARGET,
        "patch_target": cfg.PATCH_TARGET,
    })

    arrays: Dict[str, Any] = {}
    arrays["meta"] = _encode_meta(meta)
    arrays["expert_ids"] = np.array(expert_ids, dtype=np.int32)
    arrays["scales"] = Sc.detach().cpu().numpy().astype(np.float32)
    arrays["Qk"] = Qk.detach().cpu().numpy().astype(np.float32)

    arrays["U"] = comp_payload["U"].detach().cpu().numpy().astype(np.float32)
    arrays["V"] = comp_payload["V"].detach().cpu().numpy().astype(np.float32)
    arrays["DL"] = comp_payload["DL"].detach().cpu().numpy().astype(np.float32)
    arrays["DR"] = comp_payload["DR"].detach().cpu().numpy().astype(np.float32)

    if cfg.RES_COEF == "diag":
        arrays["gam"] = gam.detach().cpu().numpy().astype(np.float32)
    else:
        arrays["Cfull"] = Cfull.detach().cpu().numpy().astype(np.float32)

    for k,v in core_pack.items():
        arrays["core_"+k] = v
    for k,v in res_pack.items():
        arrays["res_"+k] = v
    for k,v in pat_pack.items():
        arrays["pat_"+k] = v

    save_npz_compressed(out_path, arrays)
    log(f"[save] payload -> {out_path} size={os.path.getsize(out_path)/1e6:.2f} MB")

    # Load runtime + eval
    rt = load_payload_runtime(out_path, DEVICE)
    eval_payload(rt, Ws, Sc)

    log("✅ Done.")
    log("Tip: If patches get big, increase GRAPH_K (more spectral comps) or improve the expert graph (router P helps).")
    log("Tip: Your accuracy is still capped hard if CALIB_SAMPLES is tiny (capture real tokens).")

if __name__ == "__main__":
    main()


== GS-KT++ v0 ==
Time:      2026-01-13 16:45:34
MODEL_DIR: /home/daniyar/deepseek-model
OUT_DIR:   /home/daniyar/moe_ws_outputs
LAYER:     1
EXPERTS:   16
CALIB:     (none) cap=4096 capture=False
GRAPH:     K=8 alpha=0.6 knn=8
CORE:      block=64 target=0.9 max_blocks=512
RES:       rank=512 coef=diag target=0.995 max_blocks=4096 bsize=64
PATCH:     block=64 target=0.98 max_blocks=256
REFINE:    enable=True err=0.03 max_extra=4096
QMODE:     none
DEVICE:    cpu torch=2.4.1+cpu threads=8

[found] layer=1 total=64 using=16 eids=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
[cache] loaded Ws -> /home/daniyar/moe_ws_outputs/Ws_cache_layer1_E16_ridge_gsktpp_v0.npz shape=(16, 2048, 2048)
[Ws] shape=(16, 2048, 2048) normalized=True
[graph] adjacency = weight_sim only
[graph] KNN sparsify: K=8
[graph] kept K=8 low-freq components; evals[0..K-1]=[-3.2938825e-08  5.3920445e-07  8.1101739e-01  1.5921074e+00
  1.9759872e+00  2.6913233e+00  3.3007066e+00  3.5826781e+00]
[basis] training sh

In [7]:
#!/usr/bin/env python3
# ============================================================
# GS-KT++ v1  (Graph-Spectral KT++ for MoE expert compression)
#
# What this script does (single-file, offline):
#   0) (Optional) CAPTURE calibration X (and router probs P) via transformers
#   1) Build linearized expert matrices Ws via ridge regression:
#        For each expert e:  Y_e = MLP_e(X)    solve  W_e ≈ argmin ||X W^T - Y||_2^2 (+ ridge)
#        Optional weighting by router probabilities P[:, eid]
#        Optional normalize each W_e and store scale Sc so W_true = Sc[e] * W_norm
#   2) Build an expert similarity graph (weight-sim and/or router-sim)
#   3) Graph spectral decomposition => keep K low-frequency eigenvectors C (E x K)
#   4) Solve for K component matrices B_k (H x H) in least-squares:
#        minimize ||Ws - C B||_F  =>  B = (C^T C + λI)^-1 C^T Ws
#      => each component B_k is a weighted sum of Ws (efficient)
#   5) Train a shared orthogonal basis (U,V) to make {B_k} (and optionally Ws) sparse in basis space
#   6) Compress each component B_k in basis space using KT++ blocks + low-rank + residual + refine
#   7) Reconstruct B̂_k, then build per-expert residual R_e = Ws[e] - sum_k C[e,k] * B̂_k
#   8) Compress per-expert patches (also KT++ style) using a shared low-rank patch basis
#   9) Save a single NPZ payload + runtime loader + evaluation
#
# IMPORTANT FIX (vs your GS-KT++ v0 log):
#   Your per-expert eval was wrong because it reused component outputs from a different x.
#   This script computes components for the current x in every per-expert trial.
#
# Dependencies:
#   - torch, numpy, safetensors, tqdm
#   - transformers only if CAPTURE_ENABLE=1 (or CAPTURE_FORCE=1)
# ============================================================

import os, re, json, math, time, random, sys
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any, Set

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors import safe_open

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):  # type: ignore
        return x

# ----------------------------
# Robust env parsing
# ----------------------------
def _env_str(k: str, d: str) -> str:
    return os.environ.get(k, d)

def _env_int(k: str, d: int) -> int:
    try:
        return int(os.environ.get(k, str(d)))
    except Exception:
        return d

def _env_float(k: str, d: float) -> float:
    try:
        return float(os.environ.get(k, str(d)))
    except Exception:
        return d

def _env_bool(k: str, d: bool) -> bool:
    v = os.environ.get(k, None)
    if v is None:
        return d
    return v.strip().lower() in ("1", "true", "yes", "y", "on")

# ----------------------------
# Threads / determinism
# ----------------------------
NTHREADS = _env_int("KTXX_THREADS", 8)
os.environ.setdefault("OMP_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(NTHREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(NTHREADS))
try:
    torch.set_num_threads(NTHREADS)
except Exception:
    pass

SEED = _env_int("SEED", 1234)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(_env_str("DEVICE", "cpu"))
DTYPE_ACC = torch.float32

def now() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")

def log(msg: str):
    print(msg, flush=True)

# ----------------------------
# Config
# ----------------------------
@dataclass
class Cfg:
    # Paths
    MODEL_DIR: str = _env_str("MODEL_DIR", "/home/daniyar/deepseek-model")
    OUT_DIR: str = _env_str("OUT_DIR", "/home/daniyar/moe_ws_outputs")

    # Slice
    LAYER: int = _env_int("LAYER", 1)
    MAX_EXPERTS: int = _env_int("EXPERTS", 16)

    # Calibration / router
    CALIB_PATH: str = _env_str("CALIB_PATH", "").strip()
    ROUTER_PATH: str = _env_str("ROUTER_PATH", "").strip()
    CALIB_SAMPLES: int = _env_int("CALIB_SAMPLES", 4096)
    RIDGE_WEIGHTED: bool = _env_bool("RIDGE_WEIGHTED", False)
    ROUTER_EIDS_ARE_GLOBAL: bool = _env_bool("ROUTER_EIDS_ARE_GLOBAL", True)
    RIDGE_DAMP: float = _env_float("RIDGE_DAMP", 1e-3)
    NORMALIZE_W: bool = _env_bool("NORMALIZE_W", True)
    FORCE_REBUILD_WS: bool = _env_bool("FORCE_REBUILD_WS", False)

    # Capture (optional)
    CAPTURE_ENABLE: bool = _env_bool("CAPTURE_ENABLE", False)
    CAPTURE_FORCE: bool = _env_bool("CAPTURE_FORCE", False)
    CAPTURE_ITERS: int = _env_int("CAPTURE_ITERS", 32)
    CAPTURE_BATCH: int = _env_int("CAPTURE_BATCH", 1)
    CAPTURE_MAX_TOKENS: int = _env_int("CAPTURE_MAX_TOKENS", 1024)
    CAPTURE_TEXT: str = _env_str(
        "CAPTURE_TEXT",
        ("DeepSeek models use mixture-of-experts layers. "
         "We capture intermediate activations for calibration. " * 256)
    )
    CAPTURE_TEXT_FILE: str = _env_str("CAPTURE_TEXT_FILE", "").strip()
    CAPTURE_KEEP_PAD: bool = _env_bool("CAPTURE_KEEP_PAD", False)
    HF_TRUST_REMOTE_CODE: bool = _env_bool("HF_TRUST_REMOTE_CODE", True)
    HF_LOCAL_FILES_ONLY: bool = _env_bool("HF_LOCAL_FILES_ONLY", True)
    HF_AUTO_PIP: bool = _env_bool("HF_AUTO_PIP", False)

    # Graph
    GRAPH_K: int = _env_int("GRAPH_K", 8)
    GRAPH_ALPHA: float = _env_float("GRAPH_ALPHA", 0.6)     # blend weight_sim vs router_sim
    GRAPH_KNN: int = _env_int("GRAPH_KNN", 8)               # KNN sparsify adjacency
    GRAPH_USE_ROUTER: bool = _env_bool("GRAPH_USE_ROUTER", True)
    GRAPH_USE_WEIGHT: bool = _env_bool("GRAPH_USE_WEIGHT", True)

    # Basis training (shared U,V)
    BASIS_STEPS: int = _env_int("BASIS_STEPS", 64)
    BASIS_LR: float = _env_float("BASIS_LR", 3e-2)
    BASIS_SUBM: int = _env_int("BASIS_SUBM", 256)          # column subset size
    BASIS_BATCH_M: int = _env_int("BASIS_BATCH_M", 8)      # number of matrices per step
    BASIS_REORTHO_EVERY: int = _env_int("BASIS_REORTHO_EVERY", 4)
    BASIS_REPORT_EVERY: int = _env_int("BASIS_REPORT_EVERY", 8)
    BASIS_OBJ: str = _env_str("BASIS_OBJ", "logratio").strip().lower()  # logratio|ratio
    BASIS_LAM_BLOCK: float = _env_float("BASIS_LAM_BLOCK", 0.10)
    BASIS_GRAD_CLIP: float = _env_float("BASIS_GRAD_CLIP", 0.0)  # 0 disables grad clip


    # Component compression (B_k)
    CORE_BLOCK: int = _env_int("CORE_BLOCK", 64)
    CORE_TARGET: float = _env_float("CORE_TARGET", 0.90)
    CORE_MAX_BLOCKS: int = _env_int("CORE_MAX_BLOCKS", 512)

    RES_RANK: int = _env_int("RES_RANK", 512)
    RES_TARGET: float = _env_float("RES_TARGET", 0.995)
    RES_MAX_BLOCKS: int = _env_int("RES_MAX_BLOCKS", 4096)
    RES_BSIZE: int = _env_int("RES_BSIZE", 64)

    REFINE_ENABLE: bool = _env_bool("REFINE_ENABLE", True)
    REFINE_ERR_TARGET: float = _env_float("REFINE_ERR_TARGET", 0.03)
    REFINE_MAX_EXTRA: int = _env_int("REFINE_MAX_EXTRA", 4096)
    REFINE_BSIZE: int = _env_int("REFINE_BSIZE", 64)
    REFINE_RECHECK_EVERY: int = _env_int("REFINE_RECHECK_EVERY", 32)

    # Patch compression (per expert residual)
    PATCH_CORE_BLOCK: int = _env_int("PATCH_BLOCK", 64)
    PATCH_CORE_TARGET: float = _env_float("PATCH_TARGET", 0.98)
    PATCH_CORE_MAX_BLOCKS: int = _env_int("PATCH_MAX_BLOCKS", 256)

    PATCH_RANK: int = _env_int("PATCH_RANK", 512)          # shared low-rank basis for patches
    PATCH_TARGET: float = _env_float("PATCH_RES_TARGET", 0.995)
    PATCH_MAX_BLOCKS: int = _env_int("PATCH_RES_MAX_BLOCKS", 4096)
    PATCH_BSIZE: int = _env_int("PATCH_BSIZE", 64)

    PATCH_REFINE_ENABLE: bool = _env_bool("PATCH_REFINE_ENABLE", True)
    PATCH_REFINE_ERR: float = _env_float("PATCH_REFINE_ERR", 0.03)
    PATCH_REFINE_MAX_EXTRA: int = _env_int("PATCH_REFINE_MAX_EXTRA", 4096)
    PATCH_REFINE_RECHECK_EVERY: int = _env_int("PATCH_REFINE_RECHECK_EVERY", 32)

    # Storage / quant
    QMODE: str = _env_str("QMODE", "none").strip().lower()          # none|float16|int8
    STORE_DTYPE: str = _env_str("STORE_DTYPE", "float16").strip().lower()  # float16|float32

    # Eval
    EVAL_TRIALS: int = _env_int("EVAL_TRIALS", 8)
    EVAL_BATCH: int = _env_int("EVAL_BATCH", 2)
    ROUTED_K: int = _env_int("ROUTED_K", 8)
    EVAL_ON_CALIB: bool = _env_bool("EVAL_ON_CALIB", True)
    EVAL_CALIB_ROWS: int = _env_int("EVAL_CALIB_ROWS", 2048)

cfg = Cfg()
os.makedirs(cfg.OUT_DIR, exist_ok=True)

# ----------------------------
# NPZ helpers
# ----------------------------
def save_npz_compressed(path: str, arrays: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez_compressed(path, **arrays)

def load_npz(path: str) -> Dict[str, np.ndarray]:
    z = np.load(path, allow_pickle=False)
    out = {k: z[k] for k in z.files}
    z.close()
    return out

def _encode_meta(meta: dict) -> np.ndarray:
    b = json.dumps(meta, sort_keys=True).encode("utf-8")
    return np.frombuffer(b, dtype=np.uint8)

def _decode_meta(arr: np.ndarray) -> dict:
    try:
        return json.loads(bytes(arr.tolist()).decode("utf-8"))
    except Exception:
        return {}

# ----------------------------
# Offline shard loading
# ----------------------------
def read_index(model_dir: str) -> Dict[str, str]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        raise FileNotFoundError(f"Missing index: {idx_path}")
    with open(idx_path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    wm = obj.get("weight_map", {})
    if not wm:
        raise RuntimeError("Index JSON has empty weight_map.")
    return wm

def find_layer_expert_ids(weight_map: Dict[str, str], layer: int) -> List[int]:
    pat = re.compile(rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.")
    ids = set()
    for k in weight_map.keys():
        m = pat.match(k)
        if m:
            ids.add(int(m.group(1)))
    return sorted(ids)

def pick_expert_tensor_keys(weight_map: Dict[str, str], layer: int, eid: int) -> Dict[str, str]:
    prefix = f"model.layers.{layer}.mlp.experts.{eid}."
    def pick(cands: List[str]) -> Optional[str]:
        for suf in cands:
            k = prefix + suf
            if k in weight_map:
                return k
        return None
    up   = pick(["up_proj.weight", "w3.weight", "w1.weight"])
    gate = pick(["gate_proj.weight", "w1.weight", "w3.weight"])
    down = pick(["down_proj.weight", "w2.weight"])
    if up is None or gate is None or down is None:
        return {}
    if up == gate:
        g2 = pick(["gate_proj.weight"])
        if g2:
            gate = g2
    return {"up": up, "gate": gate, "down": down}

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], keys: List[str]) -> Dict[str, torch.Tensor]:
    by_shard: Dict[str, List[str]] = {}
    for k in keys:
        shard = weight_map.get(k, None)
        if shard is None:
            raise KeyError(f"Key not in weight_map: {k}")
        by_shard.setdefault(shard, []).append(k)

    out: Dict[str, torch.Tensor] = {}
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(sp):
            raise FileNotFoundError(f"Missing shard: {sp}")
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks:
                out[k] = f.get_tensor(k)
    return out

# ----------------------------
# Calibration: load, optional capture
# ----------------------------
def autodetect_calib_path() -> Optional[str]:
    cand = os.path.join(cfg.OUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
    return cand if os.path.isfile(cand) else None

def autodetect_router_path() -> Optional[str]:
    cand = os.path.join(cfg.OUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
    return cand if os.path.isfile(cand) else None

def load_calib_X(path: str, H: int) -> torch.Tensor:
    z = np.load(path, allow_pickle=False)
    if "X" not in z.files:
        raise KeyError(f"CALIB npz missing key 'X'. Keys={list(z.files)}")
    Xn = z["X"].astype(np.float32, copy=False)
    z.close()

    if Xn.ndim != 2 or Xn.shape[1] != H:
        raise RuntimeError(f"Bad X shape {tuple(Xn.shape)}, expected (*,{H})")

    nrows = int(Xn.shape[0])
    if nrows < min(1024, cfg.CALIB_SAMPLES):
        log(f"[WARN] Calibration X has only {nrows} rows. "
            f"This caps accuracy hard. Use CAPTURE_ENABLE=1 and CALIB_SAMPLES=16384+.")

    if nrows > cfg.CALIB_SAMPLES:
        Xn = Xn[:cfg.CALIB_SAMPLES]

    X = torch.from_numpy(Xn)
    return X.to(device=DEVICE, dtype=DTYPE_ACC)

def load_router_P(path: str) -> np.ndarray:
    z = np.load(path, allow_pickle=False)
    if "P" not in z.files:
        raise KeyError(f"ROUTER npz missing key 'P'. Keys={list(z.files)}")
    P = z["P"].astype(np.float32, copy=False)
    z.close()
    return P

def _maybe_autopip():
    if not cfg.HF_AUTO_PIP:
        return
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", "transformers", "sentencepiece", "tokenizers"])

def _patch_transformers_cache_compat():
    try:
        from transformers.cache_utils import DynamicCache  # type: ignore
        if not hasattr(DynamicCache, "get_usable_length"):
            def _get_usable_length(self, seq_length: int):
                return int(seq_length)
            DynamicCache.get_usable_length = _get_usable_length  # type: ignore
            log("[patch] Added DynamicCache.get_usable_length shim.")
    except Exception:
        pass

class _Collector:
    def __init__(self, H: int, E_total: int, max_rows: int):
        self.H = H
        self.E_total = E_total
        self.max_rows = max_rows
        self.X_chunks: List[torch.Tensor] = []
        self.P_chunks: List[torch.Tensor] = []
        self.nX = 0
        self.nP = 0

    def _take_rows(self, flat: torch.Tensor, need: int) -> torch.Tensor:
        return flat[:need] if flat.shape[0] > need else flat

    def add_X(self, hs: torch.Tensor, attn_mask: Optional[torch.Tensor]):
        if hs is None:
            return
        if hs.ndim == 2:
            hs = hs.unsqueeze(0)
        if hs.ndim != 3 or hs.shape[-1] != self.H:
            return

        hs = hs.detach().to(torch.float32).cpu()
        if attn_mask is not None and (not cfg.CAPTURE_KEEP_PAD):
            m = attn_mask.detach().cpu().to(torch.bool)
            flat = hs.reshape(-1, self.H)
            mflat = m.reshape(-1)
            flat = flat[mflat]
        else:
            flat = hs.reshape(-1, self.H)

        if flat.numel() == 0:
            return

        need = self.max_rows - self.nX
        if need <= 0:
            return
        flat = self._take_rows(flat, need)
        self.X_chunks.append(flat)
        self.nX += int(flat.shape[0])

    def add_logits(self, logits: torch.Tensor, attn_mask: Optional[torch.Tensor]):
        if logits is None:
            return
        if logits.ndim == 2:
            logits = logits.unsqueeze(0)
        if logits.ndim != 3:
            return

        P = torch.softmax(logits.detach().to(torch.float32), dim=-1)
        P = P[..., :self.E_total].cpu()

        if attn_mask is not None and (not cfg.CAPTURE_KEEP_PAD):
            m = attn_mask.detach().cpu().to(torch.bool)
            flat = P.reshape(-1, P.shape[-1])
            mflat = m.reshape(-1)
            flat = flat[mflat]
        else:
            flat = P.reshape(-1, P.shape[-1])

        if flat.numel() == 0:
            return

        need = self.max_rows - self.nP
        if need <= 0:
            return
        flat = self._take_rows(flat, need)
        self.P_chunks.append(flat)
        self.nP += int(flat.shape[0])

def _load_capture_texts() -> List[str]:
    if cfg.CAPTURE_TEXT_FILE and os.path.isfile(cfg.CAPTURE_TEXT_FILE):
        with open(cfg.CAPTURE_TEXT_FILE, "r", encoding="utf-8") as f:
            lines = [ln.strip() for ln in f.readlines()]
        lines = [x for x in lines if x]
        if lines:
            return lines
    return [cfg.CAPTURE_TEXT]

def capture_XP_transformers(model_dir: str, layer_idx: int, H: int, E_total: int,
                            out_x: str, out_p: str) -> Tuple[str, Optional[str]]:
    _maybe_autopip()
    _patch_transformers_cache_compat()
    try:
        from transformers import AutoTokenizer, AutoModelForCausalLM  # type: ignore
    except Exception as e:
        raise RuntimeError("transformers not available; install it or set HF_AUTO_PIP=1.") from e

    log("[capture] Loading tokenizer/model (local_files_only recommended)...")
    tok = AutoTokenizer.from_pretrained(
        model_dir,
        trust_remote_code=cfg.HF_TRUST_REMOTE_CODE,
        local_files_only=cfg.HF_LOCAL_FILES_ONLY,
        use_fast=True,
    )
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token if tok.eos_token is not None else tok.unk_token

    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        trust_remote_code=cfg.HF_TRUST_REMOTE_CODE,
        local_files_only=cfg.HF_LOCAL_FILES_ONLY,
        torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
        device_map=None,
        low_cpu_mem_usage=True,
    )
    model.eval().to(DEVICE)

    # Locate layers
    layers = None
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        layers = list(model.model.layers)
    elif hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        layers = list(model.transformer.h)
    elif hasattr(model, "layers"):
        layers = list(model.layers)
    if layers is None:
        raise RuntimeError("Cannot locate transformer layers list.")
    if layer_idx < 0 or layer_idx >= len(layers):
        raise RuntimeError(f"LAYER={layer_idx} out of range; model has {len(layers)} layers.")
    layer = layers[layer_idx]

    mlp = getattr(layer, "mlp", None)
    if mlp is None:
        for n, m in layer.named_modules():
            if n.lower().endswith("mlp"):
                mlp = m
                break
    if mlp is None:
        raise RuntimeError("Could not find layer.mlp to hook for X capture.")

    # Router discovery (best-effort)
    router_linear: Optional[nn.Linear] = None
    best_score = -1e9
    for name, mod in layer.named_modules():
        if isinstance(mod, nn.Linear) and getattr(mod, "in_features", None) == H and getattr(mod, "out_features", 0) >= E_total:
            nm = name.lower()
            score = 0
            if "router" in nm: score += 10
            if "gate" in nm: score += 6
            if "moe" in nm: score += 3
            if mod.out_features == E_total: score += 6
            score -= 0.01 * float(mod.out_features - E_total)
            if score > best_score:
                best_score = score
                router_linear = mod

    router_weight: Optional[torch.Tensor] = None
    if router_linear is None:
        best = None
        best_score = -1e9
        for pname, p in layer.named_parameters(recurse=True):
            if p.ndim == 2 and p.shape[1] == H and p.shape[0] >= E_total:
                nm = pname.lower()
                score = 0
                if "router" in nm: score += 10
                if "gate" in nm: score += 6
                if p.shape[0] == E_total: score += 6
                score -= 0.01 * float(p.shape[0] - E_total)
                if score > best_score:
                    best_score = score
                    best = p
        if best is not None:
            router_weight = best.detach()

    if router_linear is not None:
        log(f"[capture] Router Linear candidate: in={router_linear.in_features} out={router_linear.out_features}")
    elif router_weight is not None:
        log(f"[capture] Router weight candidate: shape={tuple(router_weight.shape)}")
    else:
        log("[capture] Router not found; will capture X only (no P).")

    coll = _Collector(H=H, E_total=E_total, max_rows=cfg.CALIB_SAMPLES)
    attn_mask_holder = {"mask": None}

    def mlp_pre_hook(_m, inputs):
        hs = inputs[0]
        am = attn_mask_holder["mask"]
        coll.add_X(hs, am)
        if router_linear is None and router_weight is not None and hs is not None:
            hs2 = hs if hs.ndim == 3 else hs.unsqueeze(0)
            W = router_weight.to(hs2.device, dtype=torch.float32)
            logits = torch.matmul(hs2.to(torch.float32), W.t())
            coll.add_logits(logits, am)

    h_mlp = mlp.register_forward_pre_hook(mlp_pre_hook)

    h_router = None
    if router_linear is not None:
        def router_hook(_m, inputs, output):
            out = output[0] if isinstance(output, (tuple, list)) else output
            if torch.is_tensor(out):
                am = attn_mask_holder["mask"]
                out2 = out if out.ndim == 3 else out.unsqueeze(0)
                coll.add_logits(out2, am)
        h_router = router_linear.register_forward_hook(router_hook)

    texts = _load_capture_texts()
    tptr = 0
    for it in range(cfg.CAPTURE_ITERS):
        text = texts[tptr % len(texts)]
        tptr += 1
        enc = tok(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=cfg.CAPTURE_MAX_TOKENS,
            padding="max_length",
        )
        for k in list(enc.keys()):
            if enc[k].ndim == 2 and cfg.CAPTURE_BATCH > 1:
                enc[k] = enc[k].repeat(cfg.CAPTURE_BATCH, 1)

        attn_mask_holder["mask"] = enc.get("attention_mask", None)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        with torch.inference_mode():
            _ = model(**enc, use_cache=False)

        if (it + 1) % 4 == 0 or it == 0 or (it + 1) == cfg.CAPTURE_ITERS:
            log(f"[capture] iter {it+1}/{cfg.CAPTURE_ITERS}  nX={coll.nX} nP={coll.nP}")

        if coll.nX >= cfg.CALIB_SAMPLES and (not cfg.RIDGE_WEIGHTED or coll.nP >= cfg.CALIB_SAMPLES):
            break

    h_mlp.remove()
    if h_router is not None:
        h_router.remove()

    if coll.nX == 0:
        raise RuntimeError("Capture failed: collected 0 X rows. Try CAPTURE_MAX_TOKENS↑ and CAPTURE_ITERS↑.")

    X = torch.cat(coll.X_chunks, dim=0)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
    save_npz_compressed(out_x, {"X": X})
    log(f"[capture] wrote X -> {out_x}  shape={tuple(X.shape)}")

    p_written = None
    if coll.nP > 0:
        P = torch.cat(coll.P_chunks, dim=0)[:cfg.CALIB_SAMPLES].numpy().astype(np.float32)
        N = min(P.shape[0], X.shape[0])
        if N < X.shape[0]:
            X = X[:N]
            save_npz_compressed(out_x, {"X": X})
        P = P[:N]
        save_npz_compressed(out_p, {"P": P})
        log(f"[capture] wrote P -> {out_p}  shape={tuple(P.shape)}")
        p_written = out_p
    else:
        log("[capture] Router P not captured.")

    return out_x, p_written

def ensure_calib_router(H: int, E_total: int):
    if not cfg.CALIB_PATH:
        cand = autodetect_calib_path()
        if cand:
            cfg.CALIB_PATH = cand
            log(f"[calib] CALIB_PATH not set -> auto-found {cfg.CALIB_PATH}")

    if not cfg.ROUTER_PATH:
        cand = autodetect_router_path()
        if cand:
            cfg.ROUTER_PATH = cand
            log(f"[router] ROUTER_PATH not set -> auto-found {cfg.ROUTER_PATH}")

    need_capture = cfg.CAPTURE_FORCE or (cfg.CAPTURE_ENABLE and (not cfg.CALIB_PATH or not os.path.isfile(cfg.CALIB_PATH)))
    if need_capture:
        out_x = os.path.join(cfg.OUT_DIR, f"calib_layer{cfg.LAYER}_X.npz")
        out_p = os.path.join(cfg.OUT_DIR, f"router_layer{cfg.LAYER}_P.npz")
        log("[calib] capturing X (and maybe P) via transformers...")
        x_path, p_path = capture_XP_transformers(cfg.MODEL_DIR, cfg.LAYER, H, E_total, out_x, out_p)
        cfg.CALIB_PATH = x_path
        if p_path:
            cfg.ROUTER_PATH = p_path

# ----------------------------
# Ridge linearization: build Ws
# ----------------------------
@torch.no_grad()
def forward_mlp(X: torch.Tensor, W_gate: torch.Tensor, W_up: torch.Tensor, W_down: torch.Tensor) -> torch.Tensor:
    Xf = X.to(DTYPE_ACC)
    up = Xf @ W_up.to(DTYPE_ACC).t()
    gate = Xf @ W_gate.to(DTYPE_ACC).t()
    hid = F.silu(gate) * up
    y = hid @ W_down.to(DTYPE_ACC).t()
    return y

def ws_cache_path(E: int) -> str:
    return os.path.join(cfg.OUT_DIR, f"Ws_cache_layer{cfg.LAYER}_E{E}_ridge_gsktpp_v1.npz")

def ws_meta(eids: List[int]) -> dict:
    return dict(
        script="gs_ktpp_v1",
        model_dir=cfg.MODEL_DIR,
        layer=cfg.LAYER,
        expert_ids=eids,
        ridge_damp=float(cfg.RIDGE_DAMP),
        ridge_weighted=bool(cfg.RIDGE_WEIGHTED),
        router_eids_are_global=bool(cfg.ROUTER_EIDS_ARE_GLOBAL),
        router_path=(cfg.ROUTER_PATH or ""),
        calib_path=(cfg.CALIB_PATH or ""),
        calib_samples=int(cfg.CALIB_SAMPLES),
        normalize_w=bool(cfg.NORMALIZE_W),
        seed=int(SEED),
        device=str(DEVICE),
    )

@torch.no_grad()
def build_Ws(eids: List[int], wm: Dict[str, str]) -> Tuple[torch.Tensor, torch.Tensor, Optional[np.ndarray]]:
    per_e = {}
    need_keys = []
    for eid in eids:
        kk = pick_expert_tensor_keys(wm, cfg.LAYER, eid)
        if not kk:
            raise RuntimeError(f"Expert {eid} missing required tensors in index.")
        per_e[eid] = kk
        need_keys += [kk["up"], kk["down"], kk["gate"]]

    log("[load] reading tensors from shards ...")
    T = load_tensors_from_shards(cfg.MODEL_DIR, wm, sorted(set(need_keys)))

    W_up0 = T[per_e[eids[0]]["up"]]
    dff, H = int(W_up0.shape[0]), int(W_up0.shape[1])
    log(f"[shape] H={H} d_ff={dff}")

    E_total = len(find_layer_expert_ids(wm, cfg.LAYER))
    ensure_calib_router(H=H, E_total=E_total)

    if not cfg.CALIB_PATH or not os.path.isfile(cfg.CALIB_PATH):
        raise RuntimeError("CALIB_PATH missing. Set CALIB_PATH or CAPTURE_ENABLE=1.")
    X = load_calib_X(cfg.CALIB_PATH, H)
    log(f"[calib] X loaded: {tuple(X.shape)}  (path={cfg.CALIB_PATH})")

    P = None
    if cfg.RIDGE_WEIGHTED:
        if cfg.ROUTER_PATH and os.path.isfile(cfg.ROUTER_PATH):
            P = load_router_P(cfg.ROUTER_PATH)
            log(f"[router] P loaded: {tuple(P.shape)}  (path={cfg.ROUTER_PATH})")
        else:
            log("[router] RIDGE_WEIGHTED=1 but ROUTER_PATH missing -> forcing RIDGE_WEIGHTED=0")
            cfg.RIDGE_WEIGHTED = False

    Xf = X.to(DTYPE_ACC)
    I = torch.eye(H, dtype=DTYPE_ACC, device=DEVICE)
    XtX = Xf.t() @ Xf
    lam = cfg.RIDGE_DAMP * float(torch.trace(XtX).item()) / float(H)
    cholG = torch.linalg.cholesky(XtX + lam * I)

    Ws_list: List[torch.Tensor] = []
    scales: List[float] = []

    for i, eid in enumerate(tqdm(eids, desc="Build Ws (ridge)")):
        W_up = T[per_e[eid]["up"]].to(DEVICE)
        W_dn = T[per_e[eid]["down"]].to(DEVICE)
        W_gt = T[per_e[eid]["gate"]].to(DEVICE)

        Y = forward_mlp(X, W_gt, W_up, W_dn).to(DTYPE_ACC)

        if cfg.RIDGE_WEIGHTED and (P is not None):
            if cfg.ROUTER_EIDS_ARE_GLOBAL:
                if eid >= P.shape[1]:
                    raise RuntimeError(f"P shape {P.shape} cannot index eid={eid}")
                w = torch.from_numpy(P[:X.shape[0], eid]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
            else:
                if i >= P.shape[1]:
                    raise RuntimeError(f"P shape {P.shape} cannot index i={i}")
                w = torch.from_numpy(P[:X.shape[0], i]).to(DTYPE_ACC).to(DEVICE).clamp_min(0.0)
            sw = torch.sqrt(w + 1e-12).view(-1, 1)
            Xw = Xf * sw
            Yw = Y * sw
            XtX_e = Xw.t() @ Xw
            lam_e = cfg.RIDGE_DAMP * float(torch.trace(XtX_e).item()) / float(H)
            chol = torch.linalg.cholesky(XtX_e + lam_e * I)
            XtY = Xw.t() @ Yw
            Wt = torch.cholesky_solve(XtY, chol)
            W = Wt.t().contiguous()
        else:
            XtY = Xf.t() @ Y
            Wt = torch.cholesky_solve(XtY, cholG)
            W = Wt.t().contiguous()

        if cfg.NORMALIZE_W:
            s = float(torch.linalg.norm(W, ord="fro").clamp_min(1e-12).item())
            W = (W / s).contiguous()
        else:
            s = 1.0

        Ws_list.append(W)
        scales.append(s)

    Ws = torch.stack(Ws_list, dim=0).to(DTYPE_ACC).to(DEVICE)  # (E,H,H)
    Sc = torch.tensor(scales, dtype=DTYPE_ACC, device=DEVICE)  # (E,)

    # Return router probs for just these eids (optional use)
    Psub = None
    if P is not None:
        Psub = P[:X.shape[0], :].copy()  # keep full columns; graph step will slice by eids
    return Ws, Sc, Psub

def load_or_build_Ws() -> Tuple[List[int], torch.Tensor, torch.Tensor, Optional[np.ndarray]]:
    wm = read_index(cfg.MODEL_DIR)
    all_eids = find_layer_expert_ids(wm, cfg.LAYER)
    if not all_eids:
        raise RuntimeError(f"No experts found at layer={cfg.LAYER}")
    eids = all_eids[:cfg.MAX_EXPERTS]
    log(f"[found] layer={cfg.LAYER} total={len(all_eids)} using={len(eids)} eids={eids}")

    if not cfg.CALIB_PATH:
        c = autodetect_calib_path()
        if c:
            cfg.CALIB_PATH = c
    if not cfg.ROUTER_PATH:
        r = autodetect_router_path()
        if r:
            cfg.ROUTER_PATH = r

    cpath = ws_cache_path(len(eids))
    if (not cfg.FORCE_REBUILD_WS) and os.path.isfile(cpath):
        z = load_npz(cpath)
        if "meta" in z and "Ws" in z and "expert_ids" in z and "scales" in z:
            if _decode_meta(z["meta"]) == ws_meta(eids):
                Ws = torch.from_numpy(z["Ws"]).to(DTYPE_ACC).to(DEVICE)
                Sc = torch.from_numpy(z["scales"]).to(DTYPE_ACC).to(DEVICE)
                Psub = z["P"] if "P" in z else None
                log(f"[cache] loaded Ws -> {cpath} shape={tuple(Ws.shape)}")
                return [int(x) for x in z["expert_ids"].tolist()], Ws, Sc, Psub
        log("[cache] Ws meta mismatch -> rebuilding.")

    Ws, Sc, Psub = build_Ws(eids, wm)
    arrays = {
        "meta": _encode_meta(ws_meta(eids)),
        "expert_ids": np.array(eids, dtype=np.int32),
        "Ws": Ws.detach().cpu().numpy().astype(np.float32),
        "scales": Sc.detach().cpu().numpy().astype(np.float32),
    }
    if Psub is not None:
        arrays["P"] = Psub.astype(np.float32, copy=False)
    save_npz_compressed(cpath, arrays)
    log(f"[cache] wrote Ws -> {cpath}  size={os.path.getsize(cpath)/1e6:.2f} MB")
    return eids, Ws, Sc, Psub

# ----------------------------
# Graph + spectral components
# ----------------------------
@torch.no_grad()
def cosine_sim_matrix_from_features(Fm: torch.Tensor) -> torch.Tensor:
    # Fm: (E,d)
    Fm = Fm / (torch.linalg.norm(Fm, dim=1, keepdim=True).clamp_min(1e-12))
    return (Fm @ Fm.t()).clamp(-1, 1)

@torch.no_grad()
def weight_features(Ws: torch.Tensor, d: int = 128) -> torch.Tensor:
    # cheap random projection features for each expert
    E, n, _ = Ws.shape
    g = torch.Generator(device="cpu").manual_seed(SEED + 17)
    R = (torch.randint(0, 2, (n, d), generator=g, dtype=torch.int8) * 2 - 1).to(DTYPE_ACC).to(DEVICE)
    feats = []
    for e in range(E):
        W = Ws[e]
        row = torch.diag(W @ W.t())
        col = torch.diag(W.t() @ W)
        feats.append(torch.cat([row @ R, col @ R], dim=0).unsqueeze(0))
    X = torch.cat(feats, dim=0)
    X = (X - X.mean(dim=0, keepdim=True)) / (X.std(dim=0, keepdim=True) + 1e-6)
    return X

@torch.no_grad()
def router_features(P: np.ndarray, eids: List[int]) -> torch.Tensor:
    # P: (N, E_total) -> slice to these eids => (N, E)
    cols = np.array(eids, dtype=np.int64)
    Psub = P[:, cols].astype(np.float32, copy=False)
    # use mean, variance, and a few quantiles as features per expert
    # (keep it cheap + robust)
    m = Psub.mean(axis=0)
    v = Psub.var(axis=0)
    q1 = np.quantile(Psub, 0.25, axis=0)
    q5 = np.quantile(Psub, 0.50, axis=0)
    q9 = np.quantile(Psub, 0.75, axis=0)
    Fm = np.stack([m, v, q1, q5, q9], axis=1)  # (E,5)
    X = torch.from_numpy(Fm).to(DTYPE_ACC).to(DEVICE)
    X = (X - X.mean(dim=0, keepdim=True)) / (X.std(dim=0, keepdim=True) + 1e-6)
    return X

@torch.no_grad()
def knn_sparsify(A: torch.Tensor, k: int) -> torch.Tensor:
    # Keep top-k neighbors per row (symmetric)
    E = A.shape[0]
    if k <= 0 or k >= E:
        return A
    A2 = A.clone()
    A2.fill_diagonal_(0.0)
    keep = torch.zeros_like(A2, dtype=torch.bool)
    for i in range(E):
        vals = A2[i]
        topk = torch.topk(vals, k=min(k, E-1), largest=True).indices
        keep[i, topk] = True
    keep = keep | keep.t()
    A2 = torch.where(keep, A2, torch.zeros_like(A2))
    # restore diagonal as 0
    A2.fill_diagonal_(0.0)
    return A2

@torch.no_grad()
def build_graph_adjacency(Ws: torch.Tensor, eids: List[int], Psub: Optional[np.ndarray]) -> torch.Tensor:
    E = Ws.shape[0]
    A = torch.zeros((E, E), dtype=DTYPE_ACC, device=DEVICE)

    parts = []
    if cfg.GRAPH_USE_WEIGHT:
        Fw = weight_features(Ws, d=128)
        Sw = cosine_sim_matrix_from_features(Fw)
        # map [-1,1] -> [0,1]
        Sw = (Sw + 1.0) * 0.5
        parts.append(("weight_sim", Sw))

    if cfg.GRAPH_USE_ROUTER and (Psub is not None) and (cfg.ROUTER_PATH or cfg.RIDGE_WEIGHTED or True):
        try:
            Fr = router_features(Psub, eids)
            Sr = cosine_sim_matrix_from_features(Fr)
            Sr = (Sr + 1.0) * 0.5
            parts.append(("router_sim", Sr))
        except Exception as e:
            log(f"[graph] router features skipped: {e}")

    if not parts:
        raise RuntimeError("Graph has no similarity sources. Enable GRAPH_USE_WEIGHT or provide router P and enable GRAPH_USE_ROUTER.")

    if len(parts) == 1:
        name, S = parts[0]
        A = S
        log(f"[graph] adjacency = {name} only")
    else:
        # blend first two (weight, router)
        S0 = parts[0][1]
        S1 = parts[1][1]
        a = float(cfg.GRAPH_ALPHA)
        A = a * S0 + (1.0 - a) * S1
        log(f"[graph] adjacency = blend({parts[0][0]}:{a:.2f}, {parts[1][0]}:{1-a:.2f})")

    # remove diagonal
    A = A.clone()
    A.fill_diagonal_(0.0)

    # KNN sparsify
    if cfg.GRAPH_KNN > 0:
        A = knn_sparsify(A, cfg.GRAPH_KNN)
        log(f"[graph] KNN sparsify: K={cfg.GRAPH_KNN}")

    # ensure nonnegative
    A = A.clamp_min(0.0)
    return A

@torch.no_grad()
def spectral_components_from_adjacency(A: torch.Tensor, K: int) -> Tuple[torch.Tensor, torch.Tensor]:
    # Graph Laplacian L = D - A
    D = torch.diag(A.sum(dim=1))
    L = (D - A).to(torch.float64).cpu().numpy()  # E x E
    # E is small; full eigendecomp is fine
    evals, evecs = np.linalg.eigh(L)
    order = np.argsort(evals)
    evals = evals[order]
    evecs = evecs[:, order]
    K = max(1, min(int(K), evecs.shape[1]))
    C = torch.from_numpy(evecs[:, :K].astype(np.float32)).to(DTYPE_ACC).to(DEVICE)  # (E,K)
    ev = torch.from_numpy(evals[:K].astype(np.float32)).to(DTYPE_ACC).to(DEVICE)
    return C, ev

# ----------------------------
# Shared basis training (U,V)
# ----------------------------
class OrthoParam(nn.Module):
    def __init__(self, init_mat: torch.Tensor):
        super().__init__()
        self.M = nn.Parameter(init_mat.contiguous())

    def orthogonal(self) -> torch.Tensor:
        # QR is differentiable in PyTorch (don’t detach!)
        Q, _ = torch.linalg.qr(self.M)
        return Q

def offdiag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    Xoff = Xs - torch.diag_embed(D)
    return Xoff.abs().mean()

def diag_abs_mean(Xs: torch.Tensor) -> torch.Tensor:
    D = torch.diagonal(Xs, dim1=1, dim2=2)
    return D.abs().mean()

def block_group_sparsity_penalty(Xs: torch.Tensor, block: int) -> torch.Tensor:
    Eb, s, _ = Xs.shape
    b = int(block)
    if b <= 0:
        return torch.zeros((), dtype=DTYPE_ACC, device=Xs.device)
    nb = s // b
    if nb <= 0:
        return torch.zeros((), dtype=DTYPE_ACC, device=Xs.device)
    s2 = nb * b
    X = Xs[:, :s2, :s2].contiguous()
    Xb = X.view(Eb, nb, b, nb, b).permute(0, 1, 3, 2, 4).contiguous()
    Eblk = (Xb * Xb).sum(dim=(3, 4))
    P = Eblk.mean(dim=0)
    return torch.sqrt(P + 1e-12).sum() / (P.sum() + 1e-12)

@torch.no_grad()
def slice_basis_batch(A_batch: torch.Tensor, U: torch.Tensor, V: torch.Tensor, S: torch.Tensor) -> torch.Tensor:
    # A_batch: (B,n,n)
    U_S = U[:, S]     # (n,s)
    V_S = V[:, S]     # (n,s)
    T = A_batch @ V_S
    Xs = torch.matmul(U_S.t().unsqueeze(0), T)
    return Xs  # (B,s,s)

def train_shared_basis(mats: torch.Tensor):
    """
    mats: (N, n, n) tensor on DEVICE, float32
    returns: U,V (n,n) orthogonal
    """
    mats = mats.to(device=DEVICE, dtype=DTYPE_ACC)
    n = mats.shape[-1]

    # init from SVD of mean (better than identity)
    with torch.no_grad():
        Wm = mats.mean(dim=0)
        U0, _, Vh = torch.linalg.svd(Wm, full_matrices=False)
        V0 = Vh.t()

    U_par = OrthoParam(U0.to(device=DEVICE, dtype=DTYPE_ACC))
    V_par = OrthoParam(V0.to(device=DEVICE, dtype=DTYPE_ACC))

    opt = torch.optim.Adam([U_par.M, V_par.M], lr=cfg.BASIS_LR)

    t0 = time.perf_counter()
    for step in range(1, cfg.BASIS_STEPS + 1):
        # submatrix indices
        S = torch.randperm(n, device=DEVICE)[:min(cfg.BASIS_SUBM, n)]

        # IMPORTANT: keep gradients!
        Uo = U_par.orthogonal()
        Vo = V_par.orthogonal()

        # slice: build Xs for a mini-batch of matrices
        # pick a subset of mats to speed up
        if cfg.BASIS_BATCH_M > 0 and cfg.BASIS_BATCH_M < mats.shape[0]:
            midx = torch.randperm(mats.shape[0], device=DEVICE)[:cfg.BASIS_BATCH_M]
            Mb = mats[midx]
        else:
            Mb = mats

        # Differentiable slice:
        # Xs = U_S^T * (Mb @ V_S)
        U_S = Uo[:, S]                      # (n,s)
        V_S = Vo[:, S]                      # (n,s)
        T = Mb @ V_S                        # (B,n,s)
        Xs = torch.matmul(U_S.t(), T)       # (B,s,s)

        # objective
        D = torch.diagonal(Xs, dim1=1, dim2=2)
        Xoff = Xs - torch.diag_embed(D)
        off = Xoff.abs().mean()
        diag = D.abs().mean().clamp_min(1e-6)

        loss = (torch.log(off + 1e-6) - torch.log(diag))

        if cfg.BASIS_LAM_BLOCK > 0:
            loss = loss + cfg.BASIS_LAM_BLOCK * block_group_sparsity_penalty(Xs, cfg.CORE_BLOCK)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        if cfg.BASIS_GRAD_CLIP > 0:
            torch.nn.utils.clip_grad_norm_([U_par.M, V_par.M], cfg.BASIS_GRAD_CLIP)
        opt.step()

        # optional re-orthonormalize (NO GRAD here)
        if cfg.BASIS_REORTHO_EVERY > 0 and (step % cfg.BASIS_REORTHO_EVERY == 0 or step == cfg.BASIS_STEPS):
            with torch.no_grad():
                U_par.M.copy_(U_par.orthogonal())
                V_par.M.copy_(V_par.orthogonal())

        if step == 1 or step % 8 == 0 or step == cfg.BASIS_STEPS:
            t1 = time.perf_counter()
            log(f"[basis] step {step:4d}/{cfg.BASIS_STEPS} loss={float(loss.item()):.4f} (+{t1-t0:.1f}s)")
            t0 = t1

    # return frozen orthogonal bases
    with torch.no_grad():
        U = U_par.orthogonal().detach().contiguous()
        V = V_par.orthogonal().detach().contiguous()
    return U, V

# ----------------------------
# Block energy + selection
# ----------------------------
@torch.no_grad()
def block_energy_grid(X: torch.Tensor, b: int) -> Tuple[torch.Tensor, float, int, int]:
    # returns (Eg, total_energy, nb, padded_n)
    n = X.shape[0]
    nb = (n + b - 1) // b
    Np = nb * b
    if Np != n:
        Xp = torch.zeros(Np, Np, dtype=X.dtype, device=X.device)
        Xp[:n, :n] = X
        X = Xp
    Xb = X.view(nb, b, nb, b).permute(0, 2, 1, 3).contiguous()
    Eg = (Xb * Xb).sum(dim=(2, 3))
    tot = float((X * X).sum().item())
    return Eg, tot, nb, Np

@torch.no_grad()
def pick_blocks_until_target(
    Eg: torch.Tensor,
    tot_energy: float,
    target: float,
    max_blocks: int,
    exclude: Optional[Set[Tuple[int, int]]] = None,
    min_block_energy: float = 0.0,
) -> Tuple[List[Tuple[int, int]], float]:
    nb = Eg.shape[0]
    flat = Eg.reshape(-1)
    order = torch.argsort(flat, descending=True)

    picked: List[Tuple[int, int]] = []
    eacc = 0.0
    exclude = exclude or set()

    for idx in order.tolist():
        if len(picked) >= max_blocks:
            break
        e = float(flat[idx].item())
        if e <= max(min_block_energy, 1e-18):
            break
        bi = idx // nb
        bj = idx % nb
        pos = (bi, bj)
        if pos in exclude:
            continue
        picked.append(pos)
        eacc += e
        if (eacc / max(tot_energy, 1e-12)) >= target:
            break

    eff = eacc / max(tot_energy, 1e-12)
    return picked, float(eff)

@torch.no_grad()
def gather_block(X: torch.Tensor, i0: int, j0: int, b: int) -> torch.Tensor:
    n = X.shape[0]
    i1 = min(n, i0 + b)
    j1 = min(n, j0 + b)
    return X[i0:i1, j0:j1].contiguous()

# ----------------------------
# Randomized SVD low-rank
# ----------------------------
@torch.no_grad()
def rand_svd(A: torch.Tensor, r: int, n_iter: int = 2) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    # returns U, s, V  such that A ≈ U diag(s) V^T
    n = A.shape[0]
    r = min(r, n)
    g = torch.Generator(device="cpu").manual_seed(SEED + 777)
    Omega = torch.randn(n, r, generator=g, dtype=DTYPE_ACC, device=A.device)
    Y = A @ Omega
    for _ in range(max(0, n_iter)):
        Y = A @ (A.t() @ Y)
    Q, _ = torch.linalg.qr(Y)
    B = Q.t() @ A
    Uh, s, Vh = torch.linalg.svd(B, full_matrices=False)
    U = (Q @ Uh[:, :r]).contiguous()
    s = s[:r].contiguous()
    V = (Vh.t()[:, :r]).contiguous()
    return U, s, V

# ----------------------------
# Quant helpers + ragged pack
# ----------------------------
@torch.no_grad()
def q_block_int8(B: torch.Tensor) -> Tuple[np.ndarray, np.float16]:
    x = B.detach().cpu().to(torch.float32)
    maxabs = float(x.abs().max().item())
    if maxabs < 1e-12:
        return np.zeros_like(x.numpy(), dtype=np.int8), np.float16(1.0)
    scale = maxabs / 127.0
    q = torch.clamp(torch.round(x / scale), -127, 127).to(torch.int8).cpu().numpy()
    return q, np.float16(scale)

def _block_store_dtype_for_qmode(qmode: str) -> np.dtype:
    if qmode == "none":
        return np.float32
    return np.float16

def pack_blocks_ragged(blocks_per_item: List[List[Tuple[int, int, torch.Tensor]]], qmode: str) -> Dict[str, np.ndarray]:
    val_dtype = _block_store_dtype_for_qmode(qmode)

    M = len(blocks_per_item)
    item_ptr = [0]
    blk_i0, blk_j0, blk_h, blk_w = [], [], [], []
    blk_ptr = [0]
    vals = []
    vals_i8 = []
    scales = []

    for m in range(M):
        lst = blocks_per_item[m]
        for (i0, j0, B) in lst:
            h, w = B.shape
            blk_i0.append(int(i0)); blk_j0.append(int(j0))
            blk_h.append(int(h)); blk_w.append(int(w))
            if qmode == "int8":
                q, sc = q_block_int8(B)
                vals_i8.append(q.reshape(-1))
                scales.append(sc)
                blk_ptr.append(blk_ptr[-1] + q.size)
            else:
                v = B.detach().cpu().to(torch.float32).numpy().astype(val_dtype, copy=False).reshape(-1)
                vals.append(v)
                blk_ptr.append(blk_ptr[-1] + v.size)
        item_ptr.append(len(blk_i0))

    out: Dict[str, np.ndarray] = {}
    out["item_ptr"] = np.array(item_ptr, dtype=np.int32)
    out["blk_i0"] = np.array(blk_i0, dtype=np.int16)
    out["blk_j0"] = np.array(blk_j0, dtype=np.int16)
    out["blk_h"]  = np.array(blk_h, dtype=np.int16)
    out["blk_w"]  = np.array(blk_w, dtype=np.int16)
    out["blk_ptr"] = np.array(blk_ptr, dtype=np.int64)

    if qmode == "int8":
        out["blk_q"] = np.concatenate(vals_i8, axis=0).astype(np.int8, copy=False) if vals_i8 else np.zeros((0,), dtype=np.int8)
        out["blk_scale"] = np.array(scales, dtype=np.float16)
    else:
        out["blk_val"] = np.concatenate(vals, axis=0) if vals else np.zeros((0,), dtype=val_dtype)

    return out

def unpack_blocks_ragged(pack: Dict[str, np.ndarray], qmode: str, device: torch.device) -> List[List[Tuple[int, int, torch.Tensor]]]:
    item_ptr = pack["item_ptr"].astype(np.int32)
    blk_i0 = pack["blk_i0"].astype(np.int32)
    blk_j0 = pack["blk_j0"].astype(np.int32)
    blk_h  = pack["blk_h"].astype(np.int32)
    blk_w  = pack["blk_w"].astype(np.int32)
    blk_ptr = pack["blk_ptr"].astype(np.int64)

    if qmode == "int8":
        blk_q = pack["blk_q"].astype(np.int8)
        blk_scale = pack["blk_scale"].astype(np.float16)
        blk_val = None
    else:
        blk_val = pack["blk_val"]
        blk_q = None
        blk_scale = None

    M = item_ptr.shape[0] - 1
    out: List[List[Tuple[int, int, torch.Tensor]]] = []
    for m in range(M):
        b0 = int(item_ptr[m])
        b1 = int(item_ptr[m + 1])
        lst = []
        for bi in range(b0, b1):
            i0 = int(blk_i0[bi]); j0 = int(blk_j0[bi])
            h = int(blk_h[bi]); w = int(blk_w[bi])
            v0 = int(blk_ptr[bi]); v1 = int(blk_ptr[bi + 1])
            if qmode == "int8":
                q = blk_q[v0:v1].astype(np.int8, copy=False).astype(np.float32)
                sc = float(blk_scale[bi])
                B = torch.from_numpy((q * sc).reshape(h, w)).to(device=device, dtype=DTYPE_ACC)
            else:
                v = blk_val[v0:v1].astype(np.float32, copy=False)
                B = torch.from_numpy(v.reshape(h, w)).to(device=device, dtype=DTYPE_ACC)
            lst.append((i0, j0, B))
        out.append(lst)
    return out

# ----------------------------
# KT++ compress (for one matrix in basis space)
# ----------------------------
@torch.no_grad()
def reconstruct_from_parts(
    Xshape_n: int,
    core_blocks: List[Tuple[int,int,torch.Tensor]],
    U_lr: Optional[torch.Tensor],
    s_lr: Optional[torch.Tensor],
    V_lr: Optional[torch.Tensor],
    res_blocks: List[Tuple[int,int,torch.Tensor]],
) -> torch.Tensor:
    Xhat = torch.zeros((Xshape_n, Xshape_n), dtype=DTYPE_ACC, device=DEVICE)
    for (i0, j0, B) in core_blocks:
        h, w = B.shape
        Xhat[i0:i0+h, j0:j0+w] = B
    if U_lr is not None and s_lr is not None and V_lr is not None and s_lr.numel() > 0:
        Xhat = Xhat + (U_lr * s_lr.view(1, -1)) @ V_lr.t()
    for (i0, j0, B) in res_blocks:
        h, w = B.shape
        Xhat[i0:i0+h, j0:j0+w] += B
    return Xhat

@torch.no_grad()
def frob_rel_err(A: torch.Tensor, B: torch.Tensor) -> float:
    num = torch.linalg.norm(A - B, ord="fro")
    den = torch.linalg.norm(B, ord="fro").clamp_min(1e-12)
    return float((num / den).item())

@torch.no_grad()
def compress_matrix_in_basis(
    X: torch.Tensor,
    core_block: int,
    core_target: float,
    core_max_blocks: int,
    res_rank: int,
    res_target: float,
    res_max_blocks: int,
    res_bsize: int,
    refine_enable: bool,
    refine_err_target: float,
    refine_max_extra: int,
    refine_bsize: int,
    refine_recheck_every: int,
) -> Tuple[
    List[Tuple[int,int,torch.Tensor]],
    torch.Tensor, torch.Tensor, torch.Tensor,    # U_lr, s_lr, V_lr
    List[Tuple[int,int,torch.Tensor]],
    float
]:
    # core blocks
    b = int(core_block)
    Eg, te, nb, Np = block_energy_grid(X, b)
    core_pos, core_eff = pick_blocks_until_target(Eg, te, core_target, core_max_blocks)
    core_blocks: List[Tuple[int,int,torch.Tensor]] = []
    for (bi, bj) in core_pos:
        i0 = bi * b; j0 = bj * b
        core_blocks.append((i0, j0, gather_block(X, i0, j0, b)))

    # residual after core
    Xc = torch.zeros_like(X)
    for (i0, j0, Bc) in core_blocks:
        h, w = Bc.shape
        Xc[i0:i0+h, j0:j0+w] = Bc
    R = (X - Xc).contiguous()

    # low-rank residual
    r = min(int(res_rank), X.shape[0])
    if r <= 0:
        U_lr = torch.zeros((X.shape[0], 0), dtype=DTYPE_ACC, device=DEVICE)
        s_lr = torch.zeros((0,), dtype=DTYPE_ACC, device=DEVICE)
        V_lr = torch.zeros((X.shape[0], 0), dtype=DTYPE_ACC, device=DEVICE)
        R2 = R
    else:
        U_lr, s_lr, V_lr = rand_svd(R, r=r, n_iter=2)
        R2 = (R - (U_lr * s_lr.view(1, -1)) @ V_lr.t()).contiguous()

    # residual blocks
    bb = int(res_bsize)
    Eg2, te2, nb2, _ = block_energy_grid(R2, bb)
    exclude: Set[Tuple[int,int]] = set()
    # exclude core blocks (map to bb grid)
    for (i0, j0, _Bc) in core_blocks:
        exclude.add((i0 // bb, j0 // bb))

    picks, _ = pick_blocks_until_target(Eg2, te2, res_target, res_max_blocks, exclude=exclude)
    res_blocks: List[Tuple[int,int,torch.Tensor]] = []
    for (bi, bj) in picks:
        i0 = bi * bb; j0 = bj * bb
        res_blocks.append((i0, j0, gather_block(R2, i0, j0, bb)))

    # refine
    if refine_enable:
        core_pos0 = {(i0, j0) for (i0, j0, _) in core_blocks}
        res_pos0  = {(i0, j0) for (i0, j0, _) in res_blocks}
        added = 0
        Xhat = reconstruct_from_parts(X.shape[0], core_blocks, U_lr, s_lr, V_lr, res_blocks)
        err = frob_rel_err(Xhat, X)

        while err > refine_err_target and added < refine_max_extra:
            Rr = (X - Xhat).contiguous()
            Eg3, te3, nb3, _ = block_energy_grid(Rr, refine_bsize)
            flat = Eg3.reshape(-1)
            if float(flat.max().item()) <= 1e-18:
                break
            order = torch.argsort(flat, descending=True)

            found = False
            for idx in order.tolist():
                if float(flat[idx].item()) <= 1e-18:
                    break
                bi = idx // nb3
                bj = idx % nb3
                i0 = bi * refine_bsize
                j0 = bj * refine_bsize
                if (i0, j0) in core_pos0 or (i0, j0) in res_pos0:
                    continue
                Bb = gather_block(Rr, i0, j0, refine_bsize)
                res_blocks.append((i0, j0, Bb))
                res_pos0.add((i0, j0))
                added += 1
                found = True
                break

            if not found:
                break

            if (added % max(1, refine_recheck_every)) == 0:
                Xhat = reconstruct_from_parts(X.shape[0], core_blocks, U_lr, s_lr, V_lr, res_blocks)
                err = frob_rel_err(Xhat, X)

        Xhat = reconstruct_from_parts(X.shape[0], core_blocks, U_lr, s_lr, V_lr, res_blocks)
        err = frob_rel_err(Xhat, X)
        core_eff = core_eff  # unchanged

    return core_blocks, U_lr, s_lr, V_lr, res_blocks, float(core_eff)

# ----------------------------
# Payload runtime
# ----------------------------
class PayloadRuntime:
    def __init__(self):
        self.meta: dict = {}
        self.expert_ids: List[int] = []
        self.scales: Optional[torch.Tensor] = None  # (E,)
        self.C: Optional[torch.Tensor] = None       # (E,K) spectral coeffs

        self.U: Optional[torch.Tensor] = None       # (n,n)
        self.V: Optional[torch.Tensor] = None       # (n,n)

        # Components (K)
        self.comp_Ulr: Optional[torch.Tensor] = None   # (K,n,rk) stored as ragged? we store max-rank dense with padding
        self.comp_slr: Optional[torch.Tensor] = None   # (K,rk)
        self.comp_Vlr: Optional[torch.Tensor] = None   # (K,n,rk)
        self.comp_core: List[List[Tuple[int,int,torch.Tensor]]] = []
        self.comp_res:  List[List[Tuple[int,int,torch.Tensor]]] = []

        # Patch shared low-rank basis (n,rp)
        self.patch_Ulr: Optional[torch.Tensor] = None
        self.patch_Vlr: Optional[torch.Tensor] = None
        self.patch_slr: Optional[torch.Tensor] = None   # (E,rp) diag coeffs per expert

        self.patch_core: List[List[Tuple[int,int,torch.Tensor]]] = []
        self.patch_res:  List[List[Tuple[int,int,torch.Tensor]]] = []

        self.qmode: str = "none"

    @torch.no_grad()
    def _apply_blocks(self, z: torch.Tensor, blocks: List[Tuple[int,int,torch.Tensor]]) -> torch.Tensor:
        u = torch.zeros_like(z)
        for (i0, j0, B) in blocks:
            h, w = B.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ B
        return u

    @torch.no_grad()
    def apply_components(self, x: torch.Tensor) -> torch.Tensor:
        # returns Ycomp: (K, batch, n)
        assert self.U is not None and self.V is not None
        z = x @ self.U
        K = len(self.comp_core)
        Y = []
        for k in range(K):
            u = self._apply_blocks(z, self.comp_core[k])
            # low-rank
            if self.comp_Ulr is not None and self.comp_slr is not None and self.comp_Vlr is not None:
                Uk = self.comp_Ulr[k]
                sk = self.comp_slr[k]
                Vk = self.comp_Vlr[k]
                if sk.numel() > 0:
                    u = u + (z @ Uk) * sk.view(1, -1) @ Vk.t()
            # residual blocks
            u = u + self._apply_blocks(z, self.comp_res[k])
            yk = u @ self.V.t()
            Y.append(yk)
        return torch.stack(Y, dim=0)

    @torch.no_grad()
    def apply_expert(self, x: torch.Tensor, pos: int, Ycomp: Optional[torch.Tensor] = None) -> torch.Tensor:
        assert self.C is not None and self.scales is not None
        assert self.U is not None and self.V is not None
        if Ycomp is None:
            Ycomp = self.apply_components(x)

        # combine spectral components
        c = self.C[pos]  # (K,)
        y_norm = torch.einsum("k,kbn->bn", c, Ycomp)

        # patch in basis
        z = x @ self.U
        u = self._apply_blocks(z, self.patch_core[pos])
        if self.patch_Ulr is not None and self.patch_Vlr is not None and self.patch_slr is not None:
            g = self.patch_slr[pos]  # (rp,)
            if g.numel() > 0:
                u = u + ((z @ self.patch_Ulr) * g.view(1, -1)) @ self.patch_Vlr.t()
        u = u + self._apply_blocks(z, self.patch_res[pos])
        y_patch = u @ self.V.t()

        y = (y_norm + y_patch) * self.scales[pos]   # apply Sc ONCE (we work in W_norm space)
        return y

    @torch.no_grad()
    def apply_mixture(self, x: torch.Tensor, routed: List[int], gates: torch.Tensor) -> torch.Tensor:
        Ycomp = self.apply_components(x)
        y = torch.zeros_like(x)
        for a, pos in zip(gates.tolist(), routed):
            y += float(a) * self.apply_expert(x, int(pos), Ycomp=Ycomp)
        return y

def load_payload_runtime(path: str, device: torch.device) -> PayloadRuntime:
    z = load_npz(path)
    rt = PayloadRuntime()
    rt.meta = _decode_meta(z["meta"]) if "meta" in z else {}
    rt.qmode = rt.meta.get("qmode", "none")

    rt.expert_ids = [int(x) for x in z["expert_ids"].tolist()]
    rt.scales = torch.from_numpy(z["scales"]).to(device=device, dtype=DTYPE_ACC)
    rt.C = torch.from_numpy(z["C"]).to(device=device, dtype=DTYPE_ACC)

    rt.U = torch.from_numpy(z["U"]).to(device=device, dtype=DTYPE_ACC)
    rt.V = torch.from_numpy(z["V"]).to(device=device, dtype=DTYPE_ACC)

    # components
    K = int(z["K"][0])
    rk = int(z["comp_rk"][0])

    rt.comp_slr = torch.from_numpy(z["comp_s"]).to(device=device, dtype=DTYPE_ACC)
    rt.comp_Ulr = torch.from_numpy(z["comp_U"]).to(device=device, dtype=DTYPE_ACC)
    rt.comp_Vlr = torch.from_numpy(z["comp_V"]).to(device=device, dtype=DTYPE_ACC)

    comp_core_pack = {k.replace("comp_core_", ""): z[k] for k in z.keys() if k.startswith("comp_core_")}
    comp_res_pack  = {k.replace("comp_res_", ""): z[k] for k in z.keys() if k.startswith("comp_res_")}
    rt.comp_core = unpack_blocks_ragged(comp_core_pack, rt.qmode, device)
    rt.comp_res  = unpack_blocks_ragged(comp_res_pack, rt.qmode, device)

    # patches
    rp = int(z["patch_rp"][0])
    rt.patch_Ulr = torch.from_numpy(z["patch_U"]).to(device=device, dtype=DTYPE_ACC)
    rt.patch_Vlr = torch.from_numpy(z["patch_V"]).to(device=device, dtype=DTYPE_ACC)
    rt.patch_slr = torch.from_numpy(z["patch_g"]).to(device=device, dtype=DTYPE_ACC)

    patch_core_pack = {k.replace("patch_core_", ""): z[k] for k in z.keys() if k.startswith("patch_core_")}
    patch_res_pack  = {k.replace("patch_res_", ""): z[k] for k in z.keys() if k.startswith("patch_res_")}
    rt.patch_core = unpack_blocks_ragged(patch_core_pack, rt.qmode, device)
    rt.patch_res  = unpack_blocks_ragged(patch_res_pack, rt.qmode, device)

    return rt

# ----------------------------
# Evaluation (FIXED per-expert component usage)
# ----------------------------
@torch.no_grad()
def eval_payload(rt: PayloadRuntime, Ws: torch.Tensor, Sc: torch.Tensor):
    E, n, _ = Ws.shape

    # Per-expert (FIX: recompute components for current x)
    errs = []
    for pos in range(E):
        x = torch.randn(8, n, dtype=DTYPE_ACC, device=DEVICE)
        Ycomp = rt.apply_components(x)  # recompute for this x
        y_hat = rt.apply_expert(x, pos, Ycomp=Ycomp)
        y_ref = x @ (Ws[pos] * Sc[pos])
        num = torch.linalg.norm(y_hat - y_ref)
        den = torch.linalg.norm(y_ref).clamp_min(1e-12)
        errs.append(float((num / den).item()))
    log(f"[eval] per-expert rel-error mean={float(np.mean(errs)):.6f} p95={float(np.percentile(errs,95)):.6f} max={float(np.max(errs)):.6f}")

    # Routed mixture
    mix = []
    for _ in range(cfg.EVAL_TRIALS):
        x = torch.randn(cfg.EVAL_BATCH, n, dtype=DTYPE_ACC, device=DEVICE)
        routed = random.sample(range(E), k=min(cfg.ROUTED_K, E))
        gates = torch.rand(len(routed), dtype=DTYPE_ACC, device=DEVICE)
        gates = gates / gates.sum().clamp_min(1e-12)
        y_hat = rt.apply_mixture(x, routed, gates)

        Wsum = torch.zeros(n, n, dtype=DTYPE_ACC, device=DEVICE)
        for a, pos in zip(gates, routed):
            Wsum += float(a.item()) * (Ws[int(pos)] * Sc[int(pos)])
        y_ref = x @ Wsum

        num = torch.linalg.norm(y_hat - y_ref)
        den = torch.linalg.norm(y_ref).clamp_min(1e-12)
        mix.append(float((num / den).item()))
    log(f"[eval] routed rel-error mean={float(np.mean(mix)):.6f} ± {float(np.std(mix)):.6f} trials={cfg.EVAL_TRIALS}")

    # Calibration-distribution eval (more meaningful)
    if cfg.EVAL_ON_CALIB and cfg.CALIB_PATH and os.path.isfile(cfg.CALIB_PATH):
        try:
            Xcal = load_calib_X(cfg.CALIB_PATH, n).to(device=DEVICE, dtype=DTYPE_ACC)
            Xcal = Xcal[:min(int(cfg.EVAL_CALIB_ROWS), Xcal.shape[0])]
            if Xcal.shape[0] >= 32:
                errs_cal = []
                for pos in range(E):
                    xb = Xcal[torch.randperm(Xcal.shape[0], device=DEVICE)[:min(256, Xcal.shape[0])]]
                    Ycomp = rt.apply_components(xb)
                    y_hat = rt.apply_expert(xb, pos, Ycomp=Ycomp)
                    y_ref = xb @ (Ws[pos] * Sc[pos])
                    num = torch.linalg.norm(y_hat - y_ref)
                    den = torch.linalg.norm(y_ref).clamp_min(1e-12)
                    errs_cal.append(float((num / den).item()))
                log(f"[eval] calib-X rel-error mean={float(np.mean(errs_cal)):.6f} p95={float(np.percentile(errs_cal,95)):.6f} max={float(np.max(errs_cal)):.6f}")
        except Exception as e:
            log(f"[eval] calib-X eval skipped: {e}")

# ----------------------------
# Banner + main
# ----------------------------
def banner():
    log(f"== GS-KT++ v1 ==")
    log(f"Time:      {now()}")
    log(f"MODEL_DIR: {cfg.MODEL_DIR}")
    log(f"OUT_DIR:   {cfg.OUT_DIR}")
    log(f"LAYER:     {cfg.LAYER}")
    log(f"EXPERTS:   {cfg.MAX_EXPERTS}")
    log(f"CALIB:     {cfg.CALIB_PATH or '(none)'} cap={cfg.CALIB_SAMPLES} capture={cfg.CAPTURE_ENABLE or cfg.CAPTURE_FORCE}")
    log(f"GRAPH:     K={cfg.GRAPH_K} alpha={cfg.GRAPH_ALPHA} knn={cfg.GRAPH_KNN} (use_weight={cfg.GRAPH_USE_WEIGHT} use_router={cfg.GRAPH_USE_ROUTER})")
    log(f"BASIS:     steps={cfg.BASIS_STEPS} lr={cfg.BASIS_LR} subm={cfg.BASIS_SUBM} batchM={cfg.BASIS_BATCH_M}")
    log(f"CORE:      block={cfg.CORE_BLOCK} target={cfg.CORE_TARGET} max_blocks={cfg.CORE_MAX_BLOCKS}")
    log(f"RES:       rank={cfg.RES_RANK} target={cfg.RES_TARGET} max_blocks={cfg.RES_MAX_BLOCKS} bsize={cfg.RES_BSIZE}")
    log(f"PATCH:     block={cfg.PATCH_CORE_BLOCK} target={cfg.PATCH_CORE_TARGET} max_blocks={cfg.PATCH_CORE_MAX_BLOCKS} rank={cfg.PATCH_RANK}")
    log(f"REFINE:    enable={cfg.REFINE_ENABLE} err={cfg.REFINE_ERR_TARGET} max_extra={cfg.REFINE_MAX_EXTRA}")
    log(f"QMODE:     {cfg.QMODE} STORE_DTYPE={cfg.STORE_DTYPE}")
    log(f"DEVICE:    {DEVICE} torch={torch.__version__} threads={NTHREADS}")
    log("")

def main():
    banner()

    expert_ids, Ws, Sc, Psub = load_or_build_Ws()
    E, n, _ = Ws.shape
    log(f"[Ws] shape={tuple(Ws.shape)} normalized={cfg.NORMALIZE_W}")

    # Graph adjacency and spectral components
    A = build_graph_adjacency(Ws, expert_ids, Psub)
    C, evals = spectral_components_from_adjacency(A, cfg.GRAPH_K)
    K = C.shape[1]
    log(f"[graph] kept K={K} low-freq components; evals[0..K-1]={evals.detach().cpu().numpy()}")

    # Solve for component matrices B_k via least squares in expert space:
    # B = (C^T C + λI)^-1 C^T Ws  => B_k = sum_e M[k,e] Ws[e]
    CtC = (C.t() @ C)
    lam = 1e-6 * torch.trace(CtC).clamp_min(1e-12) / max(1, K)
    M = torch.linalg.solve(CtC + lam * torch.eye(K, device=DEVICE, dtype=DTYPE_ACC), C.t())  # (K,E)

    # B: (K,n,n)
    B = torch.einsum("ke,eij->kij", M, Ws).contiguous()


    # Basis training set: components + a few raw experts (helps basis generalize)
    # (feel free to increase BASIS_BATCH_M and BASIS_STEPS if you have compute)
    train_set = torch.cat([B, Ws], dim=0)  # (K+E,n,n)
    log("[basis] training shared basis on (components + experts) ...")
    U, V = train_shared_basis(train_set)

    # Compress components
    log("[compress] compressing spectral components ...")
    comp_core_blocks: List[List[Tuple[int,int,torch.Tensor]]] = []
    comp_res_blocks:  List[List[Tuple[int,int,torch.Tensor]]] = []

    # store component low-rank as dense padded tensors (K,n,rk)
    rk = min(cfg.RES_RANK, n)
    comp_U = torch.zeros((K, n, rk), dtype=DTYPE_ACC, device=DEVICE)
    comp_V = torch.zeros((K, n, rk), dtype=DTYPE_ACC, device=DEVICE)
    comp_s = torch.zeros((K, rk), dtype=DTYPE_ACC, device=DEVICE)

    # Also reconstruct Bhat_k for patch building
    Bhat = torch.zeros_like(B)

    for k in range(K):
        X = (U.t() @ B[k] @ V).contiguous()
        core, Ulr, slr, Vlr, res, eff = compress_matrix_in_basis(
            X,
            core_block=cfg.CORE_BLOCK,
            core_target=cfg.CORE_TARGET,
            core_max_blocks=cfg.CORE_MAX_BLOCKS,
            res_rank=cfg.RES_RANK,
            res_target=cfg.RES_TARGET,
            res_max_blocks=cfg.RES_MAX_BLOCKS,
            res_bsize=cfg.RES_BSIZE,
            refine_enable=cfg.REFINE_ENABLE,
            refine_err_target=cfg.REFINE_ERR_TARGET,
            refine_max_extra=cfg.REFINE_MAX_EXTRA,
            refine_bsize=cfg.REFINE_BSIZE,
            refine_recheck_every=cfg.REFINE_RECHECK_EVERY,
        )
        comp_core_blocks.append(core)
        comp_res_blocks.append(res)

        rkk = min(rk, slr.numel())
        if rkk > 0:
            comp_U[k, :, :rkk] = Ulr[:, :rkk]
            comp_V[k, :, :rkk] = Vlr[:, :rkk]
            comp_s[k, :rkk] = slr[:rkk]

        # reconstruct Xhat -> Bhat
        Xhat = reconstruct_from_parts(n, core, Ulr, slr, Vlr, res)
        Bhat[k] = (U @ Xhat @ V.t()).contiguous()

    # Build per-expert residuals in normalized space:
    # Ws_hat[e] = sum_k C[e,k] * Bhat[k]
    Ws_hat = torch.einsum("ek,kij->eij", C, Bhat).contiguous()

    R = (Ws - Ws_hat).contiguous()  # (E,n,n)

    # Patch compression:
    #   - Patch core blocks per expert
    #   - Shared low-rank basis from mean residual after core
    #   - Residual blocks per expert + refine
    log("[patch] building per-expert patches ...")
    patch_core_blocks: List[List[Tuple[int,int,torch.Tensor]]] = [[] for _ in range(E)]
    patch_res_blocks:  List[List[Tuple[int,int,torch.Tensor]]] = [[] for _ in range(E)]

    # Step 1: patch core blocks + residual matrices after core, in basis space
    Xres_list: List[torch.Tensor] = []
    for e in range(E):
        X = (U.t() @ R[e] @ V).contiguous()
        b = int(cfg.PATCH_CORE_BLOCK)
        Eg, te, nb, _ = block_energy_grid(X, b)
        picks, _ = pick_blocks_until_target(Eg, te, cfg.PATCH_CORE_TARGET, cfg.PATCH_CORE_MAX_BLOCKS)
        core = []
        for (bi, bj) in picks:
            i0 = bi * b; j0 = bj * b
            core.append((i0, j0, gather_block(X, i0, j0, b)))
        patch_core_blocks[e] = core

        Xc = torch.zeros_like(X)
        for (i0, j0, Bc) in core:
            h, w = Bc.shape
            Xc[i0:i0+h, j0:j0+w] = Bc
        Xres_list.append((X - Xc).contiguous())

    # Step 2: shared low-rank basis on mean residual
    rp = min(cfg.PATCH_RANK, n)
    Rmean = torch.stack(Xres_list, dim=0).mean(dim=0)
    if rp > 0:
        patch_Ulr, patch_s, patch_Vlr = rand_svd(Rmean, r=rp, n_iter=2)  # U,s,V
    else:
        patch_Ulr = torch.zeros((n, 0), dtype=DTYPE_ACC, device=DEVICE)
        patch_s = torch.zeros((0,), dtype=DTYPE_ACC, device=DEVICE)
        patch_Vlr = torch.zeros((n, 0), dtype=DTYPE_ACC, device=DEVICE)

    # Step 3: per-expert diag coeff g so that low-rank = (U * (s*g)) V^T
    # We'll absorb patch_s into per-expert g for better numeric conditioning
    patch_g = torch.zeros((E, patch_s.numel()), dtype=DTYPE_ACC, device=DEVICE)

    # Step 4: residual blocks + refine per expert
    for e in range(E):
        Xr = Xres_list[e]
        if patch_s.numel() > 0:
            # coefficient vector g_e approximates projection onto shared low-rank basis
            # g_e ~= diag( U^T X V ) / s  (stable clamp)
            proj = torch.sum(patch_Ulr * (Xr @ patch_Vlr), dim=0)  # (rp,)
            g = proj / patch_s.clamp_min(1e-12)                    # (rp,)
            sg = patch_s * g                                       # (rp,)  <-- absorb s
            patch_g[e, :sg.numel()] = sg                           # store absorbed coeffs
            
            X2 = (Xr - (patch_Ulr * sg.view(1, -1)) @ patch_Vlr.t()).contiguous()
        else:
            X2 = Xr

        # residual blocks in patch
        bb = int(cfg.PATCH_BSIZE)
        Eg2, te2, nb2, _ = block_energy_grid(X2, bb)
        exclude: Set[Tuple[int,int]] = set()
        for (i0, j0, _Bc) in patch_core_blocks[e]:
            exclude.add((i0 // bb, j0 // bb))
        picks, _ = pick_blocks_until_target(Eg2, te2, cfg.PATCH_TARGET, cfg.PATCH_MAX_BLOCKS, exclude=exclude)

        res = []
        for (bi, bj) in picks:
            i0 = bi * bb; j0 = bj * bb
            res.append((i0, j0, gather_block(X2, i0, j0, bb)))
        patch_res_blocks[e] = res

        # refine patch (basis-space error)
        if cfg.PATCH_REFINE_ENABLE:
            core_pos0 = {(i0, j0) for (i0, j0, _) in patch_core_blocks[e]}
            res_pos0  = {(i0, j0) for (i0, j0, _) in patch_res_blocks[e]}
            added = 0

            # build Xhat for patch
            Xhat = torch.zeros_like(X2)
            for (i0, j0, Bc) in patch_core_blocks[e]:
                h, w = Bc.shape
                Xhat[i0:i0+h, j0:j0+w] = Bc
            if patch_s.numel() > 0:
                sg = patch_g[e]
                Xhat = Xhat + (patch_Ulr * sg.view(1, -1)) @ patch_Vlr.t()
            for (i0, j0, Bb) in patch_res_blocks[e]:
                h, w = Bb.shape
                Xhat[i0:i0+h, j0:j0+w] += Bb

            err = frob_rel_err(Xhat, (U.t() @ R[e] @ V))
            while err > cfg.PATCH_REFINE_ERR and added < cfg.PATCH_REFINE_MAX_EXTRA:
                Rr = ((U.t() @ R[e] @ V) - Xhat).contiguous()
                Eg3, te3, nb3, _ = block_energy_grid(Rr, cfg.REFINE_BSIZE)
                flat = Eg3.reshape(-1)
                if float(flat.max().item()) <= 1e-18:
                    break
                order = torch.argsort(flat, descending=True)

                found = False
                for idx in order.tolist():
                    if float(flat[idx].item()) <= 1e-18:
                        break
                    bi = idx // nb3
                    bj = idx % nb3
                    i0 = bi * cfg.REFINE_BSIZE
                    j0 = bj * cfg.REFINE_BSIZE
                    if (i0, j0) in core_pos0 or (i0, j0) in res_pos0:
                        continue
                    Bb = gather_block(Rr, i0, j0, cfg.REFINE_BSIZE)
                    patch_res_blocks[e].append((i0, j0, Bb))
                    res_pos0.add((i0, j0))
                    added += 1
                    found = True
                    break
                if not found:
                    break

                if (added % max(1, cfg.PATCH_REFINE_RECHECK_EVERY)) == 0:
                    # recompute Xhat
                    Xhat = torch.zeros_like(X2)
                    for (i0, j0, Bc) in patch_core_blocks[e]:
                        h, w = Bc.shape
                        Xhat[i0:i0+h, j0:j0+w] = Bc
                    if patch_s.numel() > 0:
                        g = patch_g[e]
                        Xhat = Xhat + (patch_Ulr * (patch_s * g).view(1, -1)) @ patch_Vlr.t()
                    for (i0, j0, Bb) in patch_res_blocks[e]:
                        h, w = Bb.shape
                        Xhat[i0:i0+h, j0:j0+w] += Bb
                    err = frob_rel_err(Xhat, (U.t() @ R[e] @ V))

    # Serialize payload
    out_payload = os.path.join(cfg.OUT_DIR, f"gsktpp_payload_layer{cfg.LAYER}_E{E}_K{K}_v1_q{cfg.QMODE}.npz")

    comp_core_pack = pack_blocks_ragged(comp_core_blocks, cfg.QMODE)
    comp_res_pack  = pack_blocks_ragged(comp_res_blocks, cfg.QMODE)
    patch_core_pack = pack_blocks_ragged(patch_core_blocks, cfg.QMODE)
    patch_res_pack  = pack_blocks_ragged(patch_res_blocks, cfg.QMODE)

    store_dtype = np.float16 if cfg.STORE_DTYPE == "float16" else np.float32

    meta = ws_meta(expert_ids)
    meta.update({
        "time": now(),
        "script": "gs_ktpp_v1",
        "qmode": cfg.QMODE,
        "store_dtype": cfg.STORE_DTYPE,
        "graph_k": cfg.GRAPH_K,
        "graph_alpha": cfg.GRAPH_ALPHA,
        "graph_knn": cfg.GRAPH_KNN,
        "core_block": cfg.CORE_BLOCK,
        "core_target": cfg.CORE_TARGET,
        "res_rank": cfg.RES_RANK,
        "res_target": cfg.RES_TARGET,
        "patch_block": cfg.PATCH_CORE_BLOCK,
        "patch_target": cfg.PATCH_CORE_TARGET,
        "patch_rank": cfg.PATCH_RANK,
    })

    arrays: Dict[str, Any] = {
        "meta": _encode_meta(meta),
        "expert_ids": np.array(expert_ids, dtype=np.int32),
        "scales": Sc.detach().cpu().numpy().astype(np.float32),
        "C": C.detach().cpu().numpy().astype(np.float32),
        "K": np.array([K], dtype=np.int32),

        "U": U.detach().cpu().numpy().astype(store_dtype),
        "V": V.detach().cpu().numpy().astype(store_dtype),

        # component low-rank
        "comp_rk": np.array([rk], dtype=np.int32),
        "comp_U": comp_U.detach().cpu().numpy().astype(store_dtype),
        "comp_V": comp_V.detach().cpu().numpy().astype(store_dtype),
        "comp_s": comp_s.detach().cpu().numpy().astype(store_dtype),

        # patch low-rank shared basis + per-expert diag g
        "patch_rp": np.array([patch_s.numel()], dtype=np.int32),
        "patch_U": patch_Ulr.detach().cpu().numpy().astype(store_dtype),
        "patch_V": patch_Vlr.detach().cpu().numpy().astype(store_dtype),
        "patch_g": patch_g.detach().cpu().numpy().astype(store_dtype),
    }

    for k, v in comp_core_pack.items():
        arrays["comp_core_" + k] = v
    for k, v in comp_res_pack.items():
        arrays["comp_res_" + k] = v
    for k, v in patch_core_pack.items():
        arrays["patch_core_" + k] = v
    for k, v in patch_res_pack.items():
        arrays["patch_res_" + k] = v

    save_npz_compressed(out_payload, arrays)
    log(f"[save] payload -> {out_payload} size={os.path.getsize(out_payload)/1e6:.2f} MB")

    # Load runtime + eval
    rt = load_payload_runtime(out_payload, DEVICE)
    eval_payload(rt, Ws, Sc)

    log("✅ Done.")
    log("Notes:")
    log("  - If CALIB is tiny, your Ws are noisy; capture real tokens for CALIB_SAMPLES=16384–65536.")
    log("  - If routed error is good but per-expert is bad, it often means patch budget is too small or K too small.")
    log("  - If payload is huge, reduce PATCH_* first, then RES_MAX_BLOCKS, then increase GRAPH_K to reduce patches.")

if __name__ == "__main__":
    main()


== GS-KT++ v1 ==
Time:      2026-01-13 17:30:59
MODEL_DIR: /home/daniyar/deepseek-model
OUT_DIR:   /home/daniyar/moe_ws_outputs
LAYER:     1
EXPERTS:   16
CALIB:     (none) cap=4096 capture=False
GRAPH:     K=8 alpha=0.6 knn=8 (use_weight=True use_router=True)
BASIS:     steps=64 lr=0.03 subm=256 batchM=8
CORE:      block=64 target=0.9 max_blocks=512
RES:       rank=512 target=0.995 max_blocks=4096 bsize=64
PATCH:     block=64 target=0.98 max_blocks=256 rank=512
REFINE:    enable=True err=0.03 max_extra=4096
QMODE:     none STORE_DTYPE=float16
DEVICE:    cpu torch=2.4.1+cpu threads=8

[found] layer=1 total=64 using=16 eids=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
[cache] loaded Ws -> /home/daniyar/moe_ws_outputs/Ws_cache_layer1_E16_ridge_gsktpp_v1.npz shape=(16, 2048, 2048)
[Ws] shape=(16, 2048, 2048) normalized=True
[graph] adjacency = weight_sim only
[graph] KNN sparsify: K=8
[graph] kept K=8 low-freq components; evals[0..K-1]=[7.0780516e-08 2.2488794e+00 3.5282357e+00 

In [8]:
#!/usr/bin/env python3
import os, re, json
from safetensors import safe_open

MODEL_DIR = "/home/daniyar/deepseek-model"
LAYER = 1
MAX_EXPERTS = 16
PAYLOAD = "/home/daniyar/moe_ws_outputs/gsktpp_payload_layer1_E16_K8_v1_qnone.npz"

def read_index(model_dir: str):
    p = os.path.join(model_dir, "model.safetensors.index.json")
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)["weight_map"]

def find_layer_expert_ids(weight_map, layer: int):
    pat = re.compile(rf"^model\.layers\.{layer}\.mlp\.experts\.(\d+)\.")
    ids = set()
    for k in weight_map.keys():
        m = pat.match(k)
        if m:
            ids.add(int(m.group(1)))
    return sorted(ids)

def pick_expert_tensor_keys(weight_map, layer: int, eid: int):
    prefix = f"model.layers.{layer}.mlp.experts.{eid}."
    def pick(cands):
        for suf in cands:
            k = prefix + suf
            if k in weight_map:
                return k
        return None
    up   = pick(["up_proj.weight", "w3.weight", "w1.weight"])
    gate = pick(["gate_proj.weight", "w1.weight", "w3.weight"])
    down = pick(["down_proj.weight", "w2.weight"])
    if up is None or gate is None or down is None:
        return None
    return [up, gate, down]

def total_tensor_bytes(weight_map, keys):
    by_shard = {}
    for k in keys:
        by_shard.setdefault(weight_map[k], []).append(k)

    total = 0
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(MODEL_DIR, shard_fn)
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks:
                t = f.get_tensor(k)
                total += t.numel() * t.element_size()
    return total

wm = read_index(MODEL_DIR)
eids = find_layer_expert_ids(wm, LAYER)[:MAX_EXPERTS]
keys = []
for eid in eids:
    kk = pick_expert_tensor_keys(wm, LAYER, eid)
    if kk is None:
        raise RuntimeError(f"missing expert tensors for eid={eid}")
    keys += kk

orig_bytes = total_tensor_bytes(wm, keys)
payload_bytes = os.path.getsize(PAYLOAD)

def fmt(b):
    return f"{b/1024/1024:.2f} MiB"

print("Experts:", eids)
print("Original expert tensors:", fmt(orig_bytes))
print("Payload:", fmt(payload_bytes))

ratio = payload_bytes / max(1, orig_bytes)
comp_percent = (1.0 - ratio) * 100.0
print(f"Payload/original = {ratio:.4f}")
print(f"Compression = {comp_percent:.2f}%  (negative means expansion)")


Experts: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
Original expert tensors: 264.00 MiB
Payload: 401.21 MiB
Payload/original = 1.5197
Compression = -51.97%  (negative means expansion)


In [18]:
# ============================================================
# ONE GIANT NOTEBOOK CELL: GS-KT++ end-to-end production-style sanity bench
#
# What it does (in THIS one cell):
#   1) Loads your DeepSeek MoE model + tokenizer
#   2) Loads GS-KT++ payload(s) (.npz OR dir OR glob)
#   3) Computes compression % (payload bytes vs original expert weights on disk, if index exists)
#   4) Patches experts (router unchanged) and runs:
#        - fidelity: routed-MLP output rel-L2 vs baseline
#        - ppl: baseline vs patched perplexity (local text)
#        - speed: baseline vs patched tokens/s
#   5) Writes a small “method card” JSON (NOT a proof of global novelty)
#
# You only need to edit the CONFIG section below and run the cell.
# ============================================================

import os, re, json, math, time, glob, contextlib, random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any, Callable

import numpy as np
import torch
import torch.nn as nn
from safetensors import safe_open

# ----------------------------
# CONFIG: EDIT THESE ONLY
# ----------------------------
CONFIG = dict(
    MODEL_DIR="/home/daniyar/deepseek-model",
    PAYLOAD="/home/daniyar/moe_ws_outputs/gsktpp_payload_layer1_E16_K8_v1_qnone.npz",  # .npz OR directory OR glob like ".../gsktpp_payload_layer*_*.npz"

    # Optional: restrict to only these layers. None = use all layers found in payload(s).
    LAYERS=None,   # e.g. [1] or [1,2,3]

    # Runtime device/dtype for benchmarking
    DEVICE="cuda" if torch.cuda.is_available() else "cpu",
    MODEL_DTYPE="float16",  # "float16" | "bfloat16" | "float32"

    # Fidelity test settings
    FID_ROWS=4096,          # number of token-rows captured per tested layer
    FID_MAX_LEN=1024,
    FID_TEXT=("DeepSeek MoE calibration text. " * 512),

    # PPL test settings (local text only)
    PPL_TEXT=("Perplexity evaluation text. " * 4096),
    PPL_SEQ_LEN=1024,
    PPL_STRIDE=512,
    PPL_MAX_BATCHES=64,     # 0 = no limit

    # Speed test settings
    SPEED_BATCH=1,
    SPEED_SEQ=512,
    SPEED_WARMUP=5,
    SPEED_ITERS=20,

    # Output artifacts
    OUT_JSON="gsktpp_method_card.json",

    # Repro
    SEED=1234,
)

# ============================================================
# Internal helpers
# ============================================================

def log(msg: str):
    print(msg, flush=True)

def now() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")

def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def torch_dtype_from_str(s: str):
    s = s.lower()
    if s == "float16": return torch.float16
    if s == "bfloat16": return torch.bfloat16
    if s == "float32": return torch.float32
    return torch.float16

# ============================================================
# Payload runtime
# ============================================================

def load_npz(path: str) -> Dict[str, np.ndarray]:
    z = np.load(path, allow_pickle=False)
    out = {k: z[k] for k in z.files}
    z.close()
    return out

def _decode_meta(arr: np.ndarray) -> dict:
    try:
        return json.loads(bytes(arr.tolist()).decode("utf-8"))
    except Exception:
        return {}

def unpack_blocks_ragged(pack: Dict[str, np.ndarray], qmode: str, device: torch.device, rt_dtype: torch.dtype) -> List[List[Tuple[int, int, torch.Tensor]]]:
    item_ptr = pack["item_ptr"].astype(np.int32)
    blk_i0 = pack["blk_i0"].astype(np.int32)
    blk_j0 = pack["blk_j0"].astype(np.int32)
    blk_h  = pack["blk_h"].astype(np.int32)
    blk_w  = pack["blk_w"].astype(np.int32)
    blk_ptr = pack["blk_ptr"].astype(np.int64)

    if qmode == "int8":
        blk_q = pack["blk_q"].astype(np.int8)
        blk_scale = pack["blk_scale"].astype(np.float16)
        blk_val = None
    else:
        blk_val = pack["blk_val"]
        blk_q = None
        blk_scale = None

    M = item_ptr.shape[0] - 1
    out: List[List[Tuple[int, int, torch.Tensor]]] = []
    for m in range(M):
        b0 = int(item_ptr[m])
        b1 = int(item_ptr[m + 1])
        lst = []
        for bi in range(b0, b1):
            i0 = int(blk_i0[bi]); j0 = int(blk_j0[bi])
            h = int(blk_h[bi]); w = int(blk_w[bi])
            v0 = int(blk_ptr[bi]); v1 = int(blk_ptr[bi + 1])
            if qmode == "int8":
                q = blk_q[v0:v1].astype(np.int8, copy=False).astype(np.float32)
                sc = float(blk_scale[bi])
                B = torch.from_numpy((q * sc).reshape(h, w)).to(device=device, dtype=rt_dtype)
            else:
                v = blk_val[v0:v1].astype(np.float32, copy=False)
                B = torch.from_numpy(v.reshape(h, w)).to(device=device, dtype=rt_dtype)
            lst.append((i0, j0, B))
        out.append(lst)
    return out

class PayloadRuntime:
    def __init__(self):
        self.meta: dict = {}
        self.layer: Optional[int] = None
        self.qmode: str = "none"

        self.expert_ids: List[int] = []
        self.scales: Optional[torch.Tensor] = None  # (E,)
        self.C: Optional[torch.Tensor] = None       # (E,K)
        self.U: Optional[torch.Tensor] = None       # (n,n)
        self.V: Optional[torch.Tensor] = None       # (n,n)

        self.comp_Ulr: Optional[torch.Tensor] = None   # (K,n,rk)
        self.comp_slr: Optional[torch.Tensor] = None   # (K,rk)
        self.comp_Vlr: Optional[torch.Tensor] = None   # (K,n,rk)
        self.comp_core: List[List[Tuple[int,int,torch.Tensor]]] = []
        self.comp_res:  List[List[Tuple[int,int,torch.Tensor]]] = []

        self.patch_Ulr: Optional[torch.Tensor] = None  # (n,rp)
        self.patch_Vlr: Optional[torch.Tensor] = None  # (n,rp)
        self.patch_slr: Optional[torch.Tensor] = None  # (E,rp)
        self.patch_core: List[List[Tuple[int,int,torch.Tensor]]] = []
        self.patch_res:  List[List[Tuple[int,int,torch.Tensor]]] = []

    @torch.no_grad()
    def _apply_blocks(self, z: torch.Tensor, blocks: List[Tuple[int,int,torch.Tensor]]) -> torch.Tensor:
        u = torch.zeros_like(z)
        for (i0, j0, B) in blocks:
            h, w = B.shape
            u[:, j0:j0+w] += z[:, i0:i0+h] @ B
        return u

    @torch.no_grad()
    def apply_components(self, x: torch.Tensor) -> torch.Tensor:
        assert self.U is not None and self.V is not None
        z = x @ self.U
        K = len(self.comp_core)
        Y = []
        for k in range(K):
            u = self._apply_blocks(z, self.comp_core[k])
            if self.comp_Ulr is not None and self.comp_slr is not None and self.comp_Vlr is not None:
                Uk = self.comp_Ulr[k]
                sk = self.comp_slr[k]
                Vk = self.comp_Vlr[k]
                if sk.numel() > 0:
                    u = u + ((z @ Uk) * sk.view(1, -1)) @ Vk.t()
            u = u + self._apply_blocks(z, self.comp_res[k])
            yk = u @ self.V.t()
            Y.append(yk)
        return torch.stack(Y, dim=0)  # (K,B,n)

    @torch.no_grad()
    def apply_expert(self, x: torch.Tensor, pos: int, Ycomp: Optional[torch.Tensor]=None) -> torch.Tensor:
        assert self.C is not None and self.scales is not None
        assert self.U is not None and self.V is not None

        if Ycomp is None:
            Ycomp = self.apply_components(x)

        c = self.C[pos]  # (K,)
        y_norm = torch.einsum("k,kbn->bn", c, Ycomp)

        z = x @ self.U
        u = self._apply_blocks(z, self.patch_core[pos])
        if self.patch_Ulr is not None and self.patch_Vlr is not None and self.patch_slr is not None:
            g = self.patch_slr[pos]
            if g.numel() > 0:
                u = u + ((z @ self.patch_Ulr) * g.view(1, -1)) @ self.patch_Vlr.t()
        u = u + self._apply_blocks(z, self.patch_res[pos])
        y_patch = u @ self.V.t()

        return (y_norm + y_patch) * self.scales[pos]

def load_payload_runtime(path: str, device: torch.device, rt_dtype: torch.dtype) -> PayloadRuntime:
    z = load_npz(path)
    rt = PayloadRuntime()

    rt.meta = _decode_meta(z["meta"]) if "meta" in z else {}
    rt.qmode = rt.meta.get("qmode", "none")
    rt.layer = rt.meta.get("layer", None)

    rt.expert_ids = [int(x) for x in z["expert_ids"].tolist()]
    rt.scales = torch.from_numpy(z["scales"]).to(device=device, dtype=rt_dtype)
    rt.C = torch.from_numpy(z["C"]).to(device=device, dtype=rt_dtype)
    rt.U = torch.from_numpy(z["U"]).to(device=device, dtype=rt_dtype)
    rt.V = torch.from_numpy(z["V"]).to(device=device, dtype=rt_dtype)

    rt.comp_slr = torch.from_numpy(z["comp_s"]).to(device=device, dtype=rt_dtype)
    rt.comp_Ulr = torch.from_numpy(z["comp_U"]).to(device=device, dtype=rt_dtype)
    rt.comp_Vlr = torch.from_numpy(z["comp_V"]).to(device=device, dtype=rt_dtype)

    comp_core_pack = {k.replace("comp_core_", ""): z[k] for k in z.keys() if k.startswith("comp_core_")}
    comp_res_pack  = {k.replace("comp_res_", ""): z[k] for k in z.keys() if k.startswith("comp_res_")}
    rt.comp_core = unpack_blocks_ragged(comp_core_pack, rt.qmode, device, rt_dtype)
    rt.comp_res  = unpack_blocks_ragged(comp_res_pack, rt.qmode, device, rt_dtype)

    rt.patch_Ulr = torch.from_numpy(z["patch_U"]).to(device=device, dtype=rt_dtype)
    rt.patch_Vlr = torch.from_numpy(z["patch_V"]).to(device=device, dtype=rt_dtype)
    patch_g_key = "patch_g" if "patch_g" in z else ("patch_slr" if "patch_slr" in z else None)
    if patch_g_key is None:
        raise KeyError("Payload missing patch_g / patch_slr")
    rt.patch_slr = torch.from_numpy(z[patch_g_key]).to(device=device, dtype=rt_dtype)

    patch_core_pack = {k.replace("patch_core_", ""): z[k] for k in z.keys() if k.startswith("patch_core_")}
    patch_res_pack  = {k.replace("patch_res_", ""): z[k] for k in z.keys() if k.startswith("patch_res_")}
    rt.patch_core = unpack_blocks_ragged(patch_core_pack, rt.qmode, device, rt_dtype)
    rt.patch_res  = unpack_blocks_ragged(patch_res_pack, rt.qmode, device, rt_dtype)

    return rt

def load_payload_bank(payload_arg: str, device: torch.device, rt_dtype: torch.dtype) -> Dict[Any, Any]:
    paths: List[str] = []
    if os.path.isdir(payload_arg):
        paths = sorted(glob.glob(os.path.join(payload_arg, "*.npz")))
    elif any(ch in payload_arg for ch in ["*", "?", "["]):
        paths = sorted(glob.glob(payload_arg))
    else:
        paths = [payload_arg]
    if not paths:
        raise FileNotFoundError(f"No payload files found for: {payload_arg}")

    bank: Dict[Any, Any] = {"_paths": paths}
    for p in paths:
        rt = load_payload_runtime(p, device=device, rt_dtype=rt_dtype)
        li = rt.layer
        if li is None:
            m = re.search(r"layer(\d+)", os.path.basename(p))
            li = int(m.group(1)) if m else -1
        bank[int(li)] = rt
    return bank

# ============================================================
# Model utilities
# ============================================================

def load_model_and_tokenizer(model_dir: str, device: str, dtype: str):
    from transformers import AutoTokenizer, AutoModelForCausalLM
    torch_dtype = torch_dtype_from_str(dtype)

    tok = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token if tok.eos_token is not None else tok.unk_token

    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        trust_remote_code=True,
        torch_dtype=torch_dtype,
        low_cpu_mem_usage=True,
    )
    model.eval()
    model.to(torch.device(device))
    return model, tok

def get_layers_list(model) -> List[nn.Module]:
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return list(model.model.layers)
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return list(model.transformer.h)
    if hasattr(model, "layers"):
        return list(model.layers)
    raise RuntimeError("Cannot locate transformer layers list on this model.")

def discover_moe_experts(layer: nn.Module) -> Optional[List[nn.Module]]:
    mlp = getattr(layer, "mlp", None)
    if mlp is None:
        return None
    ex = getattr(mlp, "experts", None)
    if ex is not None and isinstance(ex, (list, nn.ModuleList)):
        return list(ex)
    moe = getattr(mlp, "moe", None)
    if moe is not None:
        ex2 = getattr(moe, "experts", None)
        if ex2 is not None and isinstance(ex2, (list, nn.ModuleList)):
            return list(ex2)
    for name, mod in mlp.named_modules():
        if name.endswith("experts") and isinstance(mod, (list, nn.ModuleList)):
            return list(mod)
    return None

# ============================================================
# Expert patcher
# ============================================================

@dataclass
class PatchRecord:
    module: nn.Module
    old_forward: Callable

class ExpertPatcher:
    def __init__(self, model: nn.Module, layers: List[nn.Module], bank: Dict[int, PayloadRuntime], device: torch.device, rt_dtype: torch.dtype):
        self.model = model
        self.layers = layers
        self.bank = bank
        self.device = device
        self.rt_dtype = rt_dtype
        self.records: List[PatchRecord] = []

    def patch(self, layer_ids: Optional[List[int]] = None):
        targets = layer_ids if layer_ids is not None else sorted([k for k in self.bank.keys() if k != "_paths"])
        patched = 0

        for li in targets:
            if li < 0 or li >= len(self.layers):
                continue
            rt = self.bank.get(li, None)
            if rt is None:
                continue
            experts = discover_moe_experts(self.layers[li])
            if experts is None:
                log(f"[patch] layer {li}: no experts found (skipping)")
                continue

            for pos, eid in enumerate(rt.expert_ids):
                if eid < 0 or eid >= len(experts):
                    continue
                exp_mod = experts[eid]
                old = exp_mod.forward

                def make_forward(rt_: PayloadRuntime, pos_: int):
                    @torch.no_grad()
                    def _f(hidden_states, *args, **kwargs):
                        x = hidden_states
                        orig_shape = x.shape
                        h = orig_shape[-1]

                        # payload expects last dim == n
                        n = int(rt_.U.shape[0])  # type: ignore
                        if h != n:
                            raise RuntimeError(
                                f"Shape mismatch: hidden dim={h} but payload n={n}. "
                                f"This payload can't be applied to this expert forward directly."
                            )

                        x2 = x.reshape(-1, h).to(device=self.device, dtype=self.rt_dtype)
                        y2 = rt_.apply_expert(x2, pos_)  # (tokens, h)
                        y = y2.reshape(*orig_shape[:-1], h).to(dtype=hidden_states.dtype, device=hidden_states.device)
                        return y
                    return _f

                exp_mod.forward = make_forward(rt, pos)  # type: ignore
                self.records.append(PatchRecord(exp_mod, old))
                patched += 1

        log(f"[patch] patched {patched} expert modules")

    def restore(self):
        for r in self.records:
            r.module.forward = r.old_forward  # type: ignore
        log(f"[patch] restored {len(self.records)} expert modules")
        self.records.clear()

@contextlib.contextmanager
def patched_experts(model, bank: Dict[int, PayloadRuntime], layer_ids: Optional[List[int]], device: torch.device, rt_dtype: torch.dtype):
    layers = get_layers_list(model)
    p = ExpertPatcher(model, layers, bank, device, rt_dtype)
    try:
        p.patch(layer_ids)
        yield
    finally:
        p.restore()

# ============================================================
# Compression stats (disk-based if safetensors index exists)
# ============================================================

def read_index(model_dir: str) -> Optional[Dict[str, str]]:
    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if not os.path.isfile(idx_path):
        return None
    with open(idx_path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    wm = obj.get("weight_map", {})
    return wm if wm else None

def bytes_of_tensor(t: torch.Tensor) -> int:
    return int(t.numel() * t.element_size())

def load_tensors_from_shards(model_dir: str, weight_map: Dict[str, str], keys: List[str]) -> Dict[str, torch.Tensor]:
    by_shard: Dict[str, List[str]] = {}
    for k in keys:
        shard = weight_map.get(k, None)
        if shard is None:
            continue
        by_shard.setdefault(shard, []).append(k)

    out: Dict[str, torch.Tensor] = {}
    for shard_fn, ks in by_shard.items():
        sp = os.path.join(model_dir, shard_fn)
        if not os.path.isfile(sp):
            continue
        with safe_open(sp, framework="pt", device="cpu") as f:
            for k in ks:
                out[k] = f.get_tensor(k)
    return out

def disk_bytes_for_expert_prefix(weight_map: Dict[str,str], model_dir: str, layer: int, eid: int) -> int:
    # sum ALL tensors under model.layers.{layer}.mlp.experts.{eid}.*
    prefix = f"model.layers.{layer}.mlp.experts.{eid}."
    keys = [k for k in weight_map.keys() if k.startswith(prefix)]
    if not keys:
        return 0
    T = load_tensors_from_shards(model_dir, weight_map, keys)
    return sum(bytes_of_tensor(T[k]) for k in T.keys())

def run_stats(model_dir: str, bank: Dict[Any,Any]) -> dict:
    payload_paths = bank["_paths"]
    payload_total = sum(os.path.getsize(p) for p in payload_paths)

    wm = read_index(model_dir)
    disk_total = 0
    per_layer = []

    if wm is not None:
        for layer_id, rt in bank.items():
            if layer_id == "_paths":
                continue
            layer_bytes = 0
            for eid in rt.expert_ids:
                layer_bytes += disk_bytes_for_expert_prefix(wm, model_dir, int(layer_id), int(eid))
            if layer_bytes > 0:
                disk_total += layer_bytes
                per_layer.append((int(layer_id), layer_bytes, len(rt.expert_ids)))
    else:
        log("[stats] NOTE: model.safetensors.index.json not found -> disk-based expert bytes unavailable.")

    stats = dict(
        time=now(),
        payload_files=len(payload_paths),
        payload_total_bytes=payload_total,
        payload_total_mb=payload_total/1e6,
        disk_expert_total_bytes=disk_total,
        disk_expert_total_mb=disk_total/1e6,
        per_layer=[dict(layer=l, disk_mb=b/1e6, experts=e) for (l,b,e) in sorted(per_layer)]
    )
    if disk_total > 0:
        ratio = payload_total / disk_total
        stats["disk_payload_over_experts_ratio"] = ratio
        stats["disk_space_saved_pct"] = (1.0 - ratio) * 100.0
        stats["disk_compression_factor_x_smaller"] = 1.0 / max(ratio, 1e-12)
    return stats

# ============================================================
# Fidelity
# ============================================================

@torch.no_grad()
def collect_layer_inputs(model, tok, text: str, layer_ids: List[int], max_rows: int, device: torch.device, max_len: int):
    layers = get_layers_list(model)
    got: Dict[int, List[torch.Tensor]] = {li: [] for li in layer_ids}

    hooks = []
    def make_hook(li: int):
        def _pre_hook(mod, inputs):
            hs = inputs[0]
            if hs is None:
                return
            x = hs.detach()
            x2 = x.reshape(-1, x.shape[-1])
            need = max_rows - sum(t.shape[0] for t in got[li])
            if need <= 0:
                return
            got[li].append(x2[:need].to("cpu", dtype=torch.float32))
        return _pre_hook

    for li in layer_ids:
        if li < 0 or li >= len(layers):
            continue
        mlp = getattr(layers[li], "mlp", None)
        if mlp is None:
            continue
        hooks.append(mlp.register_forward_pre_hook(make_hook(li)))

    enc = tok(text, return_tensors="pt", truncation=True, max_length=max_len, padding="max_length")
    enc = {k: v.to(device) for k, v in enc.items()}

    model.eval()
    _ = model(**enc, use_cache=False)

    for h in hooks:
        h.remove()

    out = {}
    for li in layer_ids:
        if got[li]:
            out[li] = torch.cat(got[li], dim=0)[:max_rows]
    return out

@torch.no_grad()
def try_mlp_forward(layer, hs: torch.Tensor):
    mlp = getattr(layer, "mlp", None)
    if mlp is None:
        raise RuntimeError("layer has no .mlp")
    try:
        return mlp(hs)
    except TypeError:
        return mlp(hs, None)

def rel_l2(a: torch.Tensor, b: torch.Tensor) -> float:
    num = torch.linalg.norm(a - b)
    den = torch.linalg.norm(b).clamp_min(1e-12)
    return float((num / den).item())

def run_fidelity(model, tok, bank, layer_ids: List[int], device: torch.device, model_dtype: torch.dtype, rt_dtype: torch.dtype,
                 text: str, rows: int, max_len: int) -> dict:
    Xs = collect_layer_inputs(model, tok, text, layer_ids, max_rows=rows, device=device, max_len=max_len)
    if not Xs:
        raise RuntimeError("Could not capture any layer inputs (check layer IDs / architecture).")

    layers = get_layers_list(model)
    out = {"time": now(), "layers": []}

    for li, Xcpu in Xs.items():
        X = Xcpu.to(device, dtype=model_dtype)
        layer = layers[li]

        with torch.inference_mode():
            y_ref = try_mlp_forward(layer, X)

        with patched_experts(model, bank, layer_ids=[li], device=device, rt_dtype=rt_dtype):
            with torch.inference_mode():
                y_hat = try_mlp_forward(layer, X)

        err = rel_l2(y_hat.float(), y_ref.float())
        out["layers"].append({"layer": int(li), "rows": int(X.shape[0]), "routed_mlp_rel_l2": err})
        log(f"[fidelity] layer {li:3d}: routed-MLP relL2={err:.6f} (rows={X.shape[0]})")

    return out

# ============================================================
# PPL
# ============================================================

@torch.no_grad()
def ppl_on_text(model, tok, text: str, device: torch.device, seq_len: int, stride: int, max_batches: int) -> float:
    enc = tok(text, return_tensors="pt")
    input_ids = enc["input_ids"][0]
    n = input_ids.numel()
    if n < 2:
        return float("inf")

    nlls = []
    total_tokens = 0
    model.eval()

    for start in range(0, max(1, n - 1), stride):
        end = min(start + seq_len, n)
        if end - start < 2:
            break
        chunk = input_ids[start:end]
        if chunk.numel() < seq_len:
            pad = torch.full((seq_len - chunk.numel(),), tok.pad_token_id, dtype=chunk.dtype)
            chunk = torch.cat([chunk, pad], dim=0)

        chunk = chunk.unsqueeze(0).to(device)
        labels = chunk.clone()
        labels[labels == tok.pad_token_id] = -100

        out = model(input_ids=chunk, labels=labels, use_cache=False)
        loss = out.loss
        nlls.append(float(loss.item()) * (labels != -100).sum().item())
        total_tokens += int((labels != -100).sum().item())

        if max_batches > 0 and len(nlls) >= max_batches:
            break

    return math.exp(sum(nlls) / max(1, total_tokens))

def run_ppl(model, tok, bank, device: torch.device, rt_dtype: torch.dtype, layer_ids: Optional[List[int]],
            text: str, seq_len: int, stride: int, max_batches: int) -> dict:
    log("[ppl] computing baseline...")
    with torch.inference_mode():
        ppl_ref = ppl_on_text(model, tok, text, device, seq_len, stride, max_batches)
    log(f"[ppl] baseline ppl: {ppl_ref:.4f}")

    log("[ppl] computing patched...")
    with patched_experts(model, bank, layer_ids=layer_ids, device=device, rt_dtype=rt_dtype):
        with torch.inference_mode():
            ppl_hat = ppl_on_text(model, tok, text, device, seq_len, stride, max_batches)
    log(f"[ppl] patched  ppl: {ppl_hat:.4f}")

    return dict(time=now(), baseline_ppl=ppl_ref, patched_ppl=ppl_hat, delta=ppl_hat - ppl_ref)

# ============================================================
# Speed
# ============================================================

@torch.no_grad()
def speed_bench(model, tok, device: torch.device, batch_size: int, seq_len: int, iters: int, warmup: int) -> float:
    vocab = int(getattr(tok, "vocab_size", 32000))
    x = torch.randint(0, vocab, (batch_size, seq_len), device=device)
    attn = torch.ones_like(x, device=device)

    for _ in range(max(0, warmup)):
        _ = model(input_ids=x, attention_mask=attn, use_cache=False)

    if device.type == "cuda":
        torch.cuda.synchronize()

    t0 = time.perf_counter()
    for _ in range(iters):
        _ = model(input_ids=x, attention_mask=attn, use_cache=False)
    if device.type == "cuda":
        torch.cuda.synchronize()
    t1 = time.perf_counter()

    toks = batch_size * seq_len * iters
    return toks / max(1e-9, (t1 - t0))

def run_speed(model, tok, bank, device: torch.device, rt_dtype: torch.dtype, layer_ids: Optional[List[int]],
              batch_size: int, seq_len: int, warmup: int, iters: int) -> dict:
    log("[speed] running baseline...")
    with torch.inference_mode():
        base_tps = speed_bench(model, tok, device, batch_size, seq_len, iters, warmup)
    log(f"[speed] baseline tokens/s: {base_tps:.2f}")

    log("[speed] running patched...")
    with patched_experts(model, bank, layer_ids=layer_ids, device=device, rt_dtype=rt_dtype):
        with torch.inference_mode():
            patch_tps = speed_bench(model, tok, device, batch_size, seq_len, iters, warmup)
    log(f"[speed] patched  tokens/s: {patch_tps:.2f}")

    return dict(time=now(), baseline_toks_per_s=base_tps, patched_toks_per_s=patch_tps, speedup_x=patch_tps/max(base_tps,1e-9))

# ============================================================
# Method card (NOT a proof of novelty / “outperforms all research”)
# ============================================================

def write_method_card(path: str, stats: dict, fidelity: dict, ppl: dict, speed: dict, bank: Dict[Any,Any]):
    layers = sorted([int(k) for k in bank.keys() if k != "_paths"])
    card = dict(
        time=now(),
        payload_layers=layers,
        what_we_measured=[
            "Disk compression ratio (payload bytes vs on-disk expert tensor bytes, when index exists).",
            "Routed-MLP output fidelity rel-L2 for selected layers (router unchanged).",
            "End-to-end perplexity on local text (baseline vs patched).",
            "Throughput (tokens/s) on random input (baseline vs patched).",
        ],
        important_notes=[
            "This script does NOT prove global novelty. It only creates reproducible measurements.",
            "Speed numbers here include Python overhead; production speed requires fused kernels / integration.",
            "If payload targets a different sub-operator than 'expert forward', patching strategy must be adapted.",
        ],
        stats=stats,
        fidelity=fidelity,
        ppl=ppl,
        speed=speed,
    )
    with open(path, "w", encoding="utf-8") as f:
        json.dump(card, f, indent=2)
    log(f"[write] method card -> {path}")

# ============================================================
# RUN EVERYTHING
# ============================================================

seed_all(CONFIG["SEED"])
device = torch.device(CONFIG["DEVICE"])
model_dtype = torch_dtype_from_str(CONFIG["MODEL_DTYPE"])

# runtime dtype: use model dtype on GPU for speed; on CPU prefer float32 for stability
if device.type == "cuda":
    rt_dtype = model_dtype
else:
    rt_dtype = torch.float32

log("============================================================")
log("GS-KT++ ONE-CELL BENCH")
log(f"time:      {now()}")
log(f"device:    {device}   (rt_dtype={rt_dtype}, model_dtype={model_dtype})")
log(f"model_dir: {CONFIG['MODEL_DIR']}")
log(f"payload:   {CONFIG['PAYLOAD']}")
log("============================================================\n")

log("[1/5] loading model + tokenizer...")
model, tok = load_model_and_tokenizer(CONFIG["MODEL_DIR"], CONFIG["DEVICE"], CONFIG["MODEL_DTYPE"])

log("[2/5] loading payload bank...")
bank = load_payload_bank(CONFIG["PAYLOAD"], device=device, rt_dtype=rt_dtype)
payload_paths = bank["_paths"]
payload_layers = sorted([k for k in bank.keys() if k != "_paths"])
log(f"  payload files: {len(payload_paths)}")
log(f"  payload layers: {payload_layers}\n")

# choose layers to test
if CONFIG["LAYERS"] is None:
    test_layers = [int(x) for x in payload_layers]
else:
    test_layers = [int(x) for x in CONFIG["LAYERS"]]
test_layers = sorted(set(test_layers))

log("[3/5] compression stats...")
stats = run_stats(CONFIG["MODEL_DIR"], bank)
log(f"  payload size: {stats['payload_total_mb']:.2f} MB")
if stats.get("disk_expert_total_bytes", 0) > 0:
    log(f"  expert bytes (disk, matched): {stats['disk_expert_total_mb']:.2f} MB")
    log(f"  payload/original ratio: {stats['disk_payload_over_experts_ratio']:.6f}")
    log(f"  space saved: {stats['disk_space_saved_pct']:.2f}%")
    log(f"  compression factor: {stats['disk_compression_factor_x_smaller']:.2f}x smaller")
else:
    log("  (disk expert bytes unavailable; missing index or unmatched names)")
log("")

log("[4/5] fidelity (routed-MLP output)...")
fidelity = run_fidelity(
    model, tok, bank,
    layer_ids=test_layers,
    device=device,
    model_dtype=model_dtype,
    rt_dtype=rt_dtype,
    text=CONFIG["FID_TEXT"],
    rows=CONFIG["FID_ROWS"],
    max_len=CONFIG["FID_MAX_LEN"],
)
log("")

log("[5/5] ppl + speed (end-to-end)...")
ppl = run_ppl(
    model, tok, bank,
    device=device,
    rt_dtype=rt_dtype,
    layer_ids=(test_layers if CONFIG["LAYERS"] is not None else None),
    text=CONFIG["PPL_TEXT"],
    seq_len=CONFIG["PPL_SEQ_LEN"],
    stride=CONFIG["PPL_STRIDE"],
    max_batches=CONFIG["PPL_MAX_BATCHES"],
)
log("")
speed = run_speed(
    model, tok, bank,
    device=device,
    rt_dtype=rt_dtype,
    layer_ids=(test_layers if CONFIG["LAYERS"] is not None else None),
    batch_size=CONFIG["SPEED_BATCH"],
    seq_len=CONFIG["SPEED_SEQ"],
    warmup=CONFIG["SPEED_WARMUP"],
    iters=CONFIG["SPEED_ITERS"],
)
log("")

write_method_card(CONFIG["OUT_JSON"], stats, fidelity, ppl, speed, bank)

log("\n✅ DONE. Key results:")
if stats.get("disk_expert_total_bytes", 0) > 0:
    log(f"  Compression saved: {stats['disk_space_saved_pct']:.2f}%  (disk)")
log(f"  Fidelity: layers tested = {len(fidelity['layers'])} (see prints above)")
log(f"  PPL delta: {ppl['delta']:+.4f}")
log(f"  Speedup:  {speed['speedup_x']:.3f}x")
log(f"  Report:   {CONFIG['OUT_JSON']}")


GS-KT++ ONE-CELL BENCH
time:      2026-01-13 18:45:38
device:    cpu   (rt_dtype=torch.float32, model_dtype=torch.float16)
model_dir: /home/daniyar/deepseek-model
payload:   /home/daniyar/moe_ws_outputs/gsktpp_payload_layer1_E16_K8_v1_qnone.npz

[1/5] loading model + tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

[2/5] loading payload bank...
  payload files: 1
  payload layers: [1]

[3/5] compression stats...
  payload size: 420.70 MB
  expert bytes (disk, matched): 276.82 MB
  payload/original ratio: 1.519730
  space saved: -51.97%
  compression factor: 0.66x smaller

[4/5] fidelity (routed-MLP output)...


ValueError: not enough values to unpack (expected 3, got 2)

ModuleNotFoundError: No module named 'gsktpp_prod_bench'